In [ ]:
SEED = 123

# ===== Reproducibilidad (ANTES de importar TensorFlow) =====
import os
os.environ.setdefault('TF_DETERMINISTIC_OPS', '1')
os.environ.setdefault('TF_CUDNN_DETERMINISTIC', '1')
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '')  # fuerza CPU para determinismo
os.environ["PYTHONHASHSEED"] = str(SEED)

In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
import igraph as ig
import random

random.seed(SEED)
np.random.seed(SEED)

import tensorflow as tf
tf.random.set_seed(SEED)
tf.config.experimental.enable_op_determinism()
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize
from sklearn.preprocessing import OneHotEncoder
from tensorflow.keras import losses, layers, Model
import pickle
from collections import defaultdict
from sklearn.metrics import normalized_mutual_info_score

import glob
from pathlib import Path
import re

In [ ]:
############ CSEA ########################


# Step 1: K-truss Algorithm to identify core structures
def k_truss(graph, k):
    """
    Identifies the k-truss structure of the graph.
    Args:
        graph (nx.Graph): Input graph.
        k (int): The value of k for k-truss.
    Returns:
        subgraph (nx.Graph): k-truss subgraph.
    """
    G = graph.copy()
    while True:
        edges_to_remove = []
        for u, v in G.edges():
            common_neighbors = set(G.neighbors(u)).intersection(G.neighbors(v))
            if len(common_neighbors) < k - 2:
                edges_to_remove.append((u, v))
        if not edges_to_remove:
            break
        G.remove_edges_from(edges_to_remove)
    return G

# Step 2: Variational Autoencoder (VAE) for dimension reduction
class VariationalAutoencoder(Model):
    def __init__(self, original_dim, latent_dim, num_clusters):
        super(VariationalAutoencoder, self).__init__()
        self.encoder = tf.keras.Sequential([
            layers.InputLayer(shape=(original_dim,)),  # Cambiado a 'shape'
            layers.Dense(128, activation='relu'),
            layers.Dense(64, activation='relu'),
            layers.Dense(latent_dim + latent_dim)
        ])
        self.decoder_reconstruction = tf.keras.Sequential([  # Decoder para reconstrucción
            layers.InputLayer(shape=(latent_dim,)),  # Cambiado a 'shape'
            layers.Dense(64, activation='relu'),
            layers.Dense(128, activation='relu'),
            layers.Dense(original_dim, activation='sigmoid')
        ])
        self.decoder_classification = tf.keras.Sequential([  # Decoder para clasificación
            layers.InputLayer(shape=(latent_dim,)),  # Cambiado a 'shape'
            layers.Dense(num_clusters, activation='softmax')
        ])

    def call(self, x):
        mean, logvar = tf.split(self.encoder(x), num_or_size_splits=2, axis=1)
        z = self.reparameterize(mean, logvar)
        reconstructed = self.decoder_reconstruction(z)  # Salida para reconstrucción
        classification = self.decoder_classification(z)  # Salida para clasificación
        return reconstructed, classification, mean, logvar  # Devolver ambas salidas



    @staticmethod
    def reparameterize(mean, logvar):
        eps = tf.random.normal(shape=tf.shape(mean))
        return eps * tf.exp(logvar * 0.5) + mean

def vae_loss(x, reconstructed, classification, mean, logvar, true_labels_onehot):
    reconstruction_loss = tf.reduce_mean(tf.keras.losses.binary_crossentropy(x, reconstructed))
    kl_divergence = -0.5 * tf.reduce_mean(1 + logvar - tf.square(mean) - tf.exp(logvar))
    classification_loss = losses.CategoricalCrossentropy()(true_labels_onehot, classification)  # Usar salida de clasificación
    return reconstruction_loss + kl_divergence + classification_loss

## escupe tambien el embeding
def csea_algorithm(graph, latent_dim=10, num_epochs=50, true_labels=None, seed=SEED, node_list=None):
    """
    Implementa el algoritmo CSEA, modificado para usar true_labels en el entrenamiento.

    Args:
        graph (nx.Graph): El grafo de entrada.
        latent_dim (int): La dimensión latente para el VAE. Se ajustará según num_clusters.
        num_epochs (int): El número de épocas de entrenamiento para el VAE.
        true_labels (np.ndarray, optional): Las etiquetas reales para el entrenamiento semi-supervisado.

    Returns:
        cluster_labels (np.ndarray): Las etiquetas de cluster para los nodos.
    """
    # Obtener las etiquetas reales usando el algoritmo de Fast Greedy con pesos en las aristas si no se proporcionan
    if true_labels is None:
        # Mantener correspondencia entre nodos de networkx e índices de igraph
        node_list = list(graph.nodes)
        node_to_idx = {node: i for i, node in enumerate(node_list)}

        edges = [(node_to_idx[u], node_to_idx[v]) for u, v in graph.edges()]
        weights = [graph[u][v].get("weight", 1.0) for u, v in graph.edges()]

        g_ig = ig.Graph(directed=False)
        g_ig.add_vertices(len(node_list))
        g_ig.add_edges(edges)
        g_ig.es["weight"] = weights

        # Fast Greedy devuelve un dendrograma; lo convertimos a clustering
        partition = g_ig.community_fastgreedy(weights=g_ig.es["weight"]).as_clustering()

        true_labels = np.array(partition.membership)

    # Determinar num_clusters a partir de las true_labels
    num_clusters = len(np.unique(true_labels))

    # Ajustar latent_dim según num_clusters (puedes ajustar esta relación)
    latent_dim = min(latent_dim, num_clusters * 2)  # Ejemplo: latent_dim como el doble de num_clusters

    # Paso 1: Calcular la matriz de similitud usando K-truss
    k_truss_graph = k_truss(graph, k=3)
    adjacency_matrix = nx.to_numpy_array(k_truss_graph)
    similarity_matrix = normalize(adjacency_matrix + np.eye(len(graph.nodes)))

    # Convertir true_labels a one-hot encoding
    encoder = OneHotEncoder(sparse_output=False)
   # true_labels_onehot = encoder.fit_transform(true_labels.reshape(-1, 1)) esto es lo que habia antes
    true_labels_onehot = encoder.fit_transform(np.array(true_labels).reshape(-1, 1))

    # Paso 2: Entrenar el Autoencoder Variacional
    original_dim = similarity_matrix.shape[1]
    vae = VariationalAutoencoder(original_dim, latent_dim, num_clusters)
    optimizer = tf.keras.optimizers.Adam()

    for epoch in range(num_epochs):
        with tf.GradientTape() as tape:
            reconstructed, classification, mean, logvar = vae(similarity_matrix)
            loss = vae_loss(similarity_matrix, reconstructed, classification, mean, logvar, true_labels_onehot)
        gradients = tape.gradient(loss, vae.trainable_variables)
        optimizer.apply_gradients(zip(gradients, vae.trainable_variables))

    # Obtener las características de baja dimensión
    low_dim_features = vae.encoder(similarity_matrix)

    # Paso 3: Clustering usando K-means

    from sklearn.cluster import KMeans

    kmeans = KMeans(n_clusters=num_clusters, random_state=SEED, n_init=10)  # Puedes ajustar random_state
    cluster_labels = kmeans.fit_predict(low_dim_features)
    return cluster_labels, low_dim_features

In [ ]:
# Esto es para transformar una particion en un vector donde me diga cada nodo a qué cluster está

def transform_partitions(partitions, num_nodes):
  """Transforms partitions into true_labels format.

  Args:
    partitions: A list of sets representing the partitions.
    num_nodes: The total number of nodes in the graph.

  Returns:
    A list representing the true_labels.
  """
  # Obtener el nodo con el ID máximo
  max_node_id = max(node for community in partitions for node in community)

  # Inicializar true_labels con el tamaño suficiente
  true_labels = [0] * max_node_id


  #true_labels = [0] * num_nodes  # Initialize with all zeros

  for cluster_id, cluster_nodes in enumerate(partitions):
    for node in cluster_nodes:
      true_labels[node - 1] = cluster_id +1  # Assign cluster_id to node

  return true_labels


def partition_from_labels(labels):
    """
    Convierte una lista de etiquetas de partición en una lista de conjuntos de comunidades.
    Args:
        labels: Lista donde labels[i] indica la comunidad a la que pertenece el nodo i.
    Returns:
        Lista de conjuntos, cada conjunto contiene los nodos de una comunidad.
    """
    communities = defaultdict(set)
    for node, community in enumerate(labels):
        communities[community].add(node + 1)  # Los nodos comienzan desde 1
    return list(communities.values())


def partition_from_dict(partition_dict, all_nodes):
    """
    Transforma un diccionario de partición en una lista de conjuntos de comunidades.

    Args:
        partition_dict: Diccionario donde las claves son los nodos y los valores son las comunidades.
        all_nodes: Lista de todos los nodos.

    Returns:
        Lista de conjuntos, donde cada conjunto representa una comunidad.
    """

    communities = defaultdict(set)  # Usamos defaultdict para facilitar la agrupación

    # Iteramos sobre los nodos y sus comunidades
    for node, community in partition_dict.items():
        communities[community].add(node + 1)  # Agregamos el nodo a su comunidad, sumando 1 para que empiece desde 1

    # Agregamos los nodos que no están en el diccionario a la comunidad 0 (o la comunidad que elijas)
    missing_nodes = set(range(1, len(all_nodes) + 1)) - set(partition_dict.keys())
    communities[0] = communities[0].union(missing_nodes)

    return list(communities.values())  # Convertimos el defaultdict a una lista

In [ ]:
# FUNCIÓN DE MODULARIDAD:

def calculate_modularity_igraph(G_nx, partition):
    """
    Calcula la modularidad de un grafo usando igraph.

    Args:
        G_nx: Grafo de NetworkX.
        partition: Lista de conjuntos que representan la partición.

    Returns:
        Modularidad del grafo.
    """

    # Crear un mapeo de los nodos originales a enteros consecutivos desde 0
    node_map = {node: i for i, node in enumerate(G_nx.nodes)}

    # Reetiquetar los nodos del grafo usando el mapeo
    G_nx = nx.relabel_nodes(G_nx, node_map)

    # Ajustar la partición para que use los nuevos IDs de los nodos
    remapped_partition = [
        {node_map[node] for node in community} for community in partition
    ]

    # Convertir el grafo de NetworkX a igraph (ahora con IDs consecutivos desde 0)
    G_ig = ig.Graph(directed=True)
    G_ig.add_vertices(len(G_nx.nodes))  # Agregar el número correcto de vértices
    edges_ig = [(u, v) for u, v in G_nx.edges]  # No restar 1, ya que los IDs comienzan en 0
    weights = [d["weight"] for _, _, d in G_nx.edges(data=True)]
    G_ig.add_edges(edges_ig)
    G_ig.es["weight"] = weights

    # Mapear nodos a comunidades en forma de lista (usando los nuevos IDs)
    node_mapping = {node: idx for idx, community in enumerate(remapped_partition) for node in community}
    membership = [node_mapping[node] for node in sorted(G_nx.nodes)]

    # Calcular la modularidad dirigida
    modularity = G_ig.modularity(membership, weights="weight")

    return modularity

In [ ]:
# ===== Wrapper: ejecutar CSEA entrenando con partición 'teacher' (Fast Greedy) =====
def run_csea_with_teacher(graph, teacher_partition, latent_dim=10, num_epochs=50, seed=SEED):
    """Entrena CSEA usando como etiquetas 'teacher' la partición de Fast Greedy y devuelve modularidad y partición estimada."""
    # Reset de estado para reproducibilidad en ejecuciones repetidas
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass

    # Orden estable de nodos
    node_order = sorted(graph.nodes())

    # Canonizar partición teacher (orden estable de comunidades)
    def _canonicalize_partition(partition):
        comms = []
        for comm in partition:
            comm_list = sorted(list(comm))
            if comm_list:
                comms.append(comm_list)
        comms.sort(key=lambda c: (c[0], len(c)))
        return comms

    teacher_partition = _canonicalize_partition(teacher_partition)

    # Mapear nodo -> id de comunidad (0..)
    node_to_cluster = {}
    for cid, comm in enumerate(teacher_partition):
        for node in comm:
            node_to_cluster[node] = cid

    true_labels = [node_to_cluster.get(node, -1) for node in node_order]

    # Si faltan nodos en la partición, los metemos en un cluster extra
    if -1 in true_labels:
        extra = max([x for x in true_labels if x != -1] + [-1]) + 1
        true_labels = [extra if x == -1 else x for x in true_labels]

    # Ejecutar CSEA (la función base ya existe en el notebook: csea_algorithm)
    cluster_labels, low_dim_features = csea_algorithm(
        graph,
        latent_dim=latent_dim,
        num_epochs=num_epochs,
        true_labels=true_labels,
        seed=seed,
        node_list=node_order
    )

    # Convertir etiquetas predichas a partición (lista de sets)
    comms = defaultdict(set)
    for node, lab in zip(node_order, cluster_labels):
        comms[int(lab)].add(node)
    pred_partition = [comms[k] for k in sorted(comms.keys())]

    # Modularidad igraph
    mod_csea = calculate_modularity_igraph(graph, pred_partition)

    return mod_csea, pred_partition, cluster_labels, low_dim_features

In [ ]:
#karate
G1 = nx.DiGraph()
G1.add_edge(1,2,weight=1)
G1.add_edge(1,3,weight=1)
G1.add_edge(2,3,weight=1)
G1.add_edge(1,4,weight=1)
G1.add_edge(2,4,weight=1)
G1.add_edge(3,4,weight=1)
G1.add_edge(1,5,weight=1)
G1.add_edge(1,6,weight=1)
G1.add_edge(1,7,weight=1)
G1.add_edge(5,7,weight=1)
G1.add_edge(6,7,weight=1)
G1.add_edge(1,8,weight=1)
G1.add_edge(2,8,weight=1)
G1.add_edge(3,8,weight=1)
G1.add_edge(4,8,weight=1)
G1.add_edge(1,9,weight=1)
G1.add_edge(3,9,weight=1)
G1.add_edge(3,10,weight=1)
G1.add_edge(1,11,weight=1)
G1.add_edge(5,11,weight=1)
G1.add_edge(6,11,weight=1)
G1.add_edge(1,12,weight=1)
G1.add_edge(1,13,weight=1)
G1.add_edge(4,13,weight=1)
G1.add_edge(1,14,weight=1)
G1.add_edge(2,14,weight=1)
G1.add_edge(3,14,weight=1)
G1.add_edge(4,14,weight=1)
G1.add_edge(6,17,weight=1)
G1.add_edge(7,17,weight=1)
G1.add_edge(1,18,weight=1)
G1.add_edge(2,18,weight=1)
G1.add_edge(1,20,weight=1)
G1.add_edge(2,20,weight=1)
G1.add_edge(1,22,weight=1)
G1.add_edge(2,22,weight=1)
G1.add_edge(24,26,weight=1)
G1.add_edge(25,26,weight=1)
G1.add_edge(3,28,weight=1)
G1.add_edge(24,28,weight=1)
G1.add_edge(25,28,weight=1)
G1.add_edge(3,29,weight=1)
G1.add_edge(24,30,weight=1)
G1.add_edge(27,30,weight=1)
G1.add_edge(2,31,weight=1)
G1.add_edge(9,31,weight=1)
G1.add_edge(1,32,weight=1)
G1.add_edge(25,32,weight=1)
G1.add_edge(26,32,weight=1)
G1.add_edge(29,32,weight=1)
G1.add_edge(3,33,weight=1)
G1.add_edge(9,33,weight=1)
G1.add_edge(15,33,weight=1)
G1.add_edge(16,33,weight=1)
G1.add_edge(19,33,weight=1)
G1.add_edge(21,33,weight=1)
G1.add_edge(23,33,weight=1)
G1.add_edge(24,33,weight=1)
G1.add_edge(30,33,weight=1)
G1.add_edge(31,33,weight=1)
G1.add_edge(32,33,weight=1)
G1.add_edge(9,34,weight=1)
G1.add_edge(10,34,weight=1)
G1.add_edge(14,34,weight=1)
G1.add_edge(15,34,weight=1)
G1.add_edge(16,34,weight=1)
G1.add_edge(19,34,weight=1)
G1.add_edge(20,34,weight=1)
G1.add_edge(21,34,weight=1)
G1.add_edge(23,34,weight=1)
G1.add_edge(24,34,weight=1)
G1.add_edge(27,34,weight=1)
G1.add_edge(28,34,weight=1)
G1.add_edge(29,34,weight=1)
G1.add_edge(30,34,weight=1)
G1.add_edge(31,34,weight=1)
G1.add_edge(32,34,weight=1)
G1.add_edge(33,34,weight=1)


#Delfines
G2 = nx.DiGraph()
G2.add_edge(9,4,weight=1)
G2.add_edge(10,6,weight=1)
G2.add_edge(10,7,weight=1)
G2.add_edge(11,1,weight=1)
G2.add_edge(11,3,weight=1)
G2.add_edge(14,6,weight=1)
G2.add_edge(14,7,weight=1)
G2.add_edge(14,10,weight=1)
G2.add_edge(15,1,weight=1)
G2.add_edge(15,4,weight=1)
G2.add_edge(16,1,weight=1)
G2.add_edge(17,15,weight=1)
G2.add_edge(18,2,weight=1)
G2.add_edge(18,7,weight=1)
G2.add_edge(18,10,weight=1)
G2.add_edge(18,14,weight=1)
G2.add_edge(19,16,weight=1)
G2.add_edge(20,2,weight=1)
G2.add_edge(20,8,weight=1)
G2.add_edge(21,9,weight=1)
G2.add_edge(21,17,weight=1)
G2.add_edge(21,19,weight=1)
G2.add_edge(22,19,weight=1)
G2.add_edge(23,18,weight=1)
G2.add_edge(25,15,weight=1)
G2.add_edge(25,16,weight=1)
G2.add_edge(25,19,weight=1)
G2.add_edge(26,18,weight=1)
G2.add_edge(27,2,weight=1)
G2.add_edge(27,26,weight=1)
G2.add_edge(28,2,weight=1)
G2.add_edge(28,8,weight=1)
G2.add_edge(28,18,weight=1)
G2.add_edge(28,26,weight=1)
G2.add_edge(28,27,weight=1)
G2.add_edge(29,2,weight=1)
G2.add_edge(29,9,weight=1)
G2.add_edge(29,21,weight=1)
G2.add_edge(30,11,weight=1)
G2.add_edge(30,19,weight=1)
G2.add_edge(30,22,weight=1)
G2.add_edge(30,25,weight=1)
G2.add_edge(31,8,weight=1)
G2.add_edge(31,20,weight=1)
G2.add_edge(31,29,weight=1)
G2.add_edge(32,18,weight=1)
G2.add_edge(33,10,weight=1)
G2.add_edge(33,14,weight=1)
G2.add_edge(34,13,weight=1)
G2.add_edge(34,15,weight=1)
G2.add_edge(34,17,weight=1)
G2.add_edge(34,22,weight=1)
G2.add_edge(35,15,weight=1)
G2.add_edge(35,34,weight=1)
G2.add_edge(36,30,weight=1)
G2.add_edge(37,2,weight=1)
G2.add_edge(37,21,weight=1)
G2.add_edge(37,24,weight=1)
G2.add_edge(38,9,weight=1)
G2.add_edge(38,15,weight=1)
G2.add_edge(38,17,weight=1)
G2.add_edge(38,22,weight=1)
G2.add_edge(38,34,weight=1)
G2.add_edge(38,35,weight=1)
G2.add_edge(38,37,weight=1)
G2.add_edge(39,15,weight=1)
G2.add_edge(39,17,weight=1)
G2.add_edge(39,21,weight=1)
G2.add_edge(39,34,weight=1)
G2.add_edge(40,37,weight=1)
G2.add_edge(41,1,weight=1)
G2.add_edge(41,8,weight=1)
G2.add_edge(41,15,weight=1)
G2.add_edge(41,16,weight=1)
G2.add_edge(41,34,weight=1)
G2.add_edge(41,37,weight=1)
G2.add_edge(41,38,weight=1)
G2.add_edge(42,2,weight=1)
G2.add_edge(42,10,weight=1)
G2.add_edge(42,14,weight=1)
G2.add_edge(43,1,weight=1)
G2.add_edge(43,3,weight=1)
G2.add_edge(43,11,weight=1)
G2.add_edge(43,31,weight=1)
G2.add_edge(44,15,weight=1)
G2.add_edge(44,30,weight=1)
G2.add_edge(44,34,weight=1)
G2.add_edge(44,38,weight=1)
G2.add_edge(44,39,weight=1)
G2.add_edge(45,3,weight=1)
G2.add_edge(45,21,weight=1)
G2.add_edge(45,35,weight=1)
G2.add_edge(45,39,weight=1)
G2.add_edge(46,9,weight=1)
G2.add_edge(46,16,weight=1)
G2.add_edge(46,19,weight=1)
G2.add_edge(46,22,weight=1)
G2.add_edge(46,24,weight=1)
G2.add_edge(46,25,weight=1)
G2.add_edge(46,30,weight=1)
G2.add_edge(46,38,weight=1)
G2.add_edge(47,44,weight=1)
G2.add_edge(48,1,weight=1)
G2.add_edge(48,11,weight=1)
G2.add_edge(48,21,weight=1)
G2.add_edge(48,29,weight=1)
G2.add_edge(48,31,weight=1)
G2.add_edge(48,43,weight=1)
G2.add_edge(50,35,weight=1)
G2.add_edge(50,47,weight=1)
G2.add_edge(51,15,weight=1)
G2.add_edge(51,17,weight=1)
G2.add_edge(51,21,weight=1)
G2.add_edge(51,34,weight=1)
G2.add_edge(51,43,weight=1)
G2.add_edge(51,46,weight=1)
G2.add_edge(52,5,weight=1)
G2.add_edge(52,12,weight=1)
G2.add_edge(52,19,weight=1)
G2.add_edge(52,22,weight=1)
G2.add_edge(52,24,weight=1)
G2.add_edge(52,25,weight=1)
G2.add_edge(52,30,weight=1)
G2.add_edge(52,46,weight=1)
G2.add_edge(52,51,weight=1)
G2.add_edge(53,15,weight=1)
G2.add_edge(53,30,weight=1)
G2.add_edge(53,39,weight=1)
G2.add_edge(53,41,weight=1)
G2.add_edge(54,44,weight=1)
G2.add_edge(55,2,weight=1)
G2.add_edge(55,7,weight=1)
G2.add_edge(55,8,weight=1)
G2.add_edge(55,14,weight=1)
G2.add_edge(55,20,weight=1)
G2.add_edge(55,42,weight=1)
G2.add_edge(56,16,weight=1)
G2.add_edge(56,52,weight=1)
G2.add_edge(57,6,weight=1)
G2.add_edge(57,7,weight=1)
G2.add_edge(58,6,weight=1)
G2.add_edge(58,7,weight=1)
G2.add_edge(58,10,weight=1)
G2.add_edge(58,14,weight=1)
G2.add_edge(58,18,weight=1)
G2.add_edge(58,40,weight=1)
G2.add_edge(58,42,weight=1)
G2.add_edge(58,49,weight=1)
G2.add_edge(58,55,weight=1)
G2.add_edge(59,39,weight=1)
G2.add_edge(60,4,weight=1)
G2.add_edge(60,9,weight=1)
G2.add_edge(60,16,weight=1)
G2.add_edge(60,37,weight=1)
G2.add_edge(60,46,weight=1)
G2.add_edge(61,33,weight=1)
G2.add_edge(62,3,weight=1)
G2.add_edge(62,38,weight=1)
G2.add_edge(62,54,weight=1)

#Los miserables
G3 = nx.DiGraph()
G3.add_edge(1,2,weight=1)
G3.add_edge(2,3,weight=8)
G3.add_edge(2,4,weight=10)
G3.add_edge(2,5,weight=1)
G3.add_edge(2,6,weight=1)
G3.add_edge(2,7,weight=1)
G3.add_edge(2,8,weight=1)
G3.add_edge(2,9,weight=2)
G3.add_edge(2,10,weight=1)
G3.add_edge(2,11,weight=5)
G3.add_edge(3,4,weight=6)
G3.add_edge(3,11,weight=3)
G3.add_edge(4,11,weight=3)
G3.add_edge(11,12,weight=1)
G3.add_edge(11,13,weight=1)
G3.add_edge(11,14,weight=1)
G3.add_edge(11,15,weight=1)
G3.add_edge(11,16,weight=1)
G3.add_edge(11,24,weight=9)
G3.add_edge(11,25,weight=7)
G3.add_edge(11,26,weight=12)
G3.add_edge(11,27,weight=31)
G3.add_edge(11,28,weight=17)
G3.add_edge(11,29,weight=8)
G3.add_edge(11,30,weight=2)
G3.add_edge(11,32,weight=3)
G3.add_edge(11,33,weight=1)
G3.add_edge(11,34,weight=2)
G3.add_edge(11,35,weight=3)
G3.add_edge(11,36,weight=3)
G3.add_edge(11,37,weight=2)
G3.add_edge(11,38,weight=2)
G3.add_edge(11,39,weight=2)
G3.add_edge(11,44,weight=3)
G3.add_edge(11,45,weight=1)
G3.add_edge(11,49,weight=1)
G3.add_edge(11,50,weight=2)
G3.add_edge(11,52,weight=2)
G3.add_edge(11,56,weight=19)
G3.add_edge(11,59,weight=4)
G3.add_edge(11,65,weight=1)
G3.add_edge(11,69,weight=1)
G3.add_edge(11,70,weight=1)
G3.add_edge(11,71,weight=1)
G3.add_edge(11,72,weight=1)
G3.add_edge(11,73,weight=1)
G3.add_edge(13,24,weight=2)
G3.add_edge(17,18,weight=4)
G3.add_edge(17,19,weight=4)
G3.add_edge(17,20,weight=4)
G3.add_edge(17,21,weight=3)
G3.add_edge(17,22,weight=3)
G3.add_edge(17,23,weight=3)
G3.add_edge(17,24,weight=3)
G3.add_edge(18,19,weight=4)
G3.add_edge(18,20,weight=4)
G3.add_edge(18,21,weight=3)
G3.add_edge(18,22,weight=3)
G3.add_edge(18,23,weight=3)
G3.add_edge(18,24,weight=3)
G3.add_edge(18,27,weight=1)
G3.add_edge(18,56,weight=1)
G3.add_edge(19,20,weight=4)
G3.add_edge(19,21,weight=3)
G3.add_edge(19,22,weight=3)
G3.add_edge(19,23,weight=3)
G3.add_edge(19,24,weight=3)
G3.add_edge(20,21,weight=4)
G3.add_edge(20,22,weight=3)
G3.add_edge(20,23,weight=3)
G3.add_edge(20,24,weight=3)
G3.add_edge(21,22,weight=5)
G3.add_edge(21,23,weight=4)
G3.add_edge(21,24,weight=4)
G3.add_edge(22,23,weight=4)
G3.add_edge(22,24,weight=4)
G3.add_edge(23,24,weight=4)
G3.add_edge(24,25,weight=2)
G3.add_edge(24,26,weight=1)
G3.add_edge(24,28,weight=5)
G3.add_edge(24,30,weight=1)
G3.add_edge(24,31,weight=1)
G3.add_edge(24,32,weight=2)
G3.add_edge(25,26,weight=13)
G3.add_edge(25,27,weight=4)
G3.add_edge(25,28,weight=1)
G3.add_edge(25,42,weight=2)
G3.add_edge(25,43,weight=1)
G3.add_edge(25,51,weight=1)
G3.add_edge(25,69,weight=1)
G3.add_edge(25,70,weight=1)
G3.add_edge(25,71,weight=1)
G3.add_edge(26,27,weight=1)
G3.add_edge(26,28,weight=5)
G3.add_edge(26,40,weight=1)
G3.add_edge(26,41,weight=1)
G3.add_edge(26,42,weight=3)
G3.add_edge(26,43,weight=2)
G3.add_edge(26,49,weight=1)
G3.add_edge(26,56,weight=2)
G3.add_edge(26,69,weight=5)
G3.add_edge(26,70,weight=6)
G3.add_edge(26,71,weight=4)
G3.add_edge(26,72,weight=1)
G3.add_edge(26,76,weight=3)
G3.add_edge(27,28,weight=1)
G3.add_edge(27,44,weight=1)
G3.add_edge(27,50,weight=3)
G3.add_edge(27,52,weight=2)
G3.add_edge(27,55,weight=1)
G3.add_edge(27,56,weight=21)
G3.add_edge(27,73,weight=2)
G3.add_edge(28,29,weight=1)
G3.add_edge(28,30,weight=1)
G3.add_edge(28,32,weight=1)
G3.add_edge(28,34,weight=1)
G3.add_edge(28,44,weight=1)
G3.add_edge(28,49,weight=1)
G3.add_edge(28,59,weight=6)
G3.add_edge(28,69,weight=1)
G3.add_edge(28,70,weight=2)
G3.add_edge(28,71,weight=1)
G3.add_edge(28,72,weight=1)
G3.add_edge(28,73,weight=1)
G3.add_edge(29,45,weight=3)
G3.add_edge(29,46,weight=2)
G3.add_edge(30,35,weight=2)
G3.add_edge(30,36,weight=2)
G3.add_edge(30,37,weight=1)
G3.add_edge(30,38,weight=1)
G3.add_edge(30,39,weight=1)
G3.add_edge(31,32,weight=2)
G3.add_edge(35,36,weight=3)
G3.add_edge(35,37,weight=2)
G3.add_edge(35,38,weight=2)
G3.add_edge(35,39,weight=2)
G3.add_edge(36,37,weight=2)
G3.add_edge(36,38,weight=2)
G3.add_edge(36,39,weight=2)
G3.add_edge(37,38,weight=2)
G3.add_edge(37,39,weight=2)
G3.add_edge(38,39,weight=2)
G3.add_edge(40,53,weight=1)
G3.add_edge(40,56,weight=1)
G3.add_edge(42,43,weight=2)
G3.add_edge(42,56,weight=5)
G3.add_edge(42,58,weight=1)
G3.add_edge(42,63,weight=1)
G3.add_edge(42,69,weight=1)
G3.add_edge(42,70,weight=1)
G3.add_edge(42,71,weight=1)
G3.add_edge(42,72,weight=1)
G3.add_edge(42,76,weight=1)
G3.add_edge(47,48,weight=1)
G3.add_edge(47,49,weight=2)
G3.add_edge(49,56,weight=4)
G3.add_edge(49,58,weight=1)
G3.add_edge(49,59,weight=7)
G3.add_edge(49,60,weight=6)
G3.add_edge(49,61,weight=1)
G3.add_edge(49,62,weight=2)
G3.add_edge(49,63,weight=7)
G3.add_edge(49,64,weight=5)
G3.add_edge(49,65,weight=5)
G3.add_edge(49,66,weight=3)
G3.add_edge(49,67,weight=1)
G3.add_edge(49,69,weight=1)
G3.add_edge(49,70,weight=1)
G3.add_edge(49,72,weight=1)
G3.add_edge(49,74,weight=2)
G3.add_edge(49,75,weight=2)
G3.add_edge(49,76,weight=1)
G3.add_edge(49,77,weight=1)
G3.add_edge(50,51,weight=1)
G3.add_edge(50,52,weight=9)
G3.add_edge(50,55,weight=1)
G3.add_edge(50,56,weight=12)
G3.add_edge(50,57,weight=1)
G3.add_edge(52,53,weight=1)
G3.add_edge(52,54,weight=1)
G3.add_edge(52,55,weight=2)
G3.add_edge(52,56,weight=6)
G3.add_edge(55,56,weight=1)
G3.add_edge(56,57,weight=1)
G3.add_edge(56,58,weight=1)
G3.add_edge(56,59,weight=7)
G3.add_edge(56,60,weight=5)
G3.add_edge(56,62,weight=1)
G3.add_edge(56,63,weight=9)
G3.add_edge(56,64,weight=1)
G3.add_edge(56,65,weight=5)
G3.add_edge(56,66,weight=2)
G3.add_edge(58,59,weight=1)
G3.add_edge(58,60,weight=2)
G3.add_edge(58,62,weight=1)
G3.add_edge(58,63,weight=2)
G3.add_edge(58,64,weight=2)
G3.add_edge(58,65,weight=1)
G3.add_edge(58,66,weight=1)
G3.add_edge(58,68,weight=3)
G3.add_edge(59,60,weight=15)
G3.add_edge(59,61,weight=4)
G3.add_edge(59,62,weight=6)
G3.add_edge(59,63,weight=17)
G3.add_edge(59,64,weight=4)
G3.add_edge(59,65,weight=10)
G3.add_edge(59,66,weight=5)
G3.add_edge(59,67,weight=3)
G3.add_edge(59,71,weight=1)
G3.add_edge(59,77,weight=1)
G3.add_edge(60,61,weight=2)
G3.add_edge(60,62,weight=5)
G3.add_edge(60,63,weight=13)
G3.add_edge(60,64,weight=5)
G3.add_edge(60,65,weight=9)
G3.add_edge(60,66,weight=5)
G3.add_edge(60,67,weight=1)
G3.add_edge(61,62,weight=2)
G3.add_edge(61,63,weight=3)
G3.add_edge(61,64,weight=2)
G3.add_edge(61,65,weight=2)
G3.add_edge(61,66,weight=2)
G3.add_edge(61,67,weight=1)
G3.add_edge(62,63,weight=6)
G3.add_edge(62,64,weight=3)
G3.add_edge(62,65,weight=6)
G3.add_edge(62,66,weight=5)
G3.add_edge(62,67,weight=1)
G3.add_edge(63,64,weight=6)
G3.add_edge(63,65,weight=12)
G3.add_edge(63,66,weight=5)
G3.add_edge(63,67,weight=2)
G3.add_edge(63,77,weight=1)
G3.add_edge(64,65,weight=4)
G3.add_edge(64,66,weight=5)
G3.add_edge(64,67,weight=1)
G3.add_edge(64,77,weight=1)
G3.add_edge(65,66,weight=7)
G3.add_edge(65,67,weight=3)
G3.add_edge(65,77,weight=1)
G3.add_edge(66,67,weight=2)
G3.add_edge(66,77,weight=1)
G3.add_edge(67,77,weight=1)
G3.add_edge(69,70,weight=6)
G3.add_edge(69,71,weight=4)
G3.add_edge(69,72,weight=2)
G3.add_edge(69,76,weight=3)
G3.add_edge(70,71,weight=4)
G3.add_edge(70,72,weight=2)
G3.add_edge(70,76,weight=3)
G3.add_edge(71,72,weight=2)
G3.add_edge(71,76,weight=1)
G3.add_edge(72,76,weight=1)
G3.add_edge(74,75,weight=3)


#LibrosPoliticos
G4 = nx.DiGraph()
G4.add_edge(1,2,weight=1)
G4.add_edge(1,3,weight=1)
G4.add_edge(1,4,weight=1)
G4.add_edge(1,5,weight=1)
G4.add_edge(1,6,weight=1)
G4.add_edge(1,7,weight=1)
G4.add_edge(2,4,weight=1)
G4.add_edge(2,6,weight=1)
G4.add_edge(2,7,weight=1)
G4.add_edge(3,5,weight=1)
G4.add_edge(3,6,weight=1)
G4.add_edge(3,8,weight=1)
G4.add_edge(4,6,weight=1)
G4.add_edge(4,9,weight=1)
G4.add_edge(4,10,weight=1)
G4.add_edge(4,11,weight=1)
G4.add_edge(4,12,weight=1)
G4.add_edge(4,13,weight=1)
G4.add_edge(4,14,weight=1)
G4.add_edge(4,15,weight=1)
G4.add_edge(4,16,weight=1)
G4.add_edge(4,17,weight=1)
G4.add_edge(4,18,weight=1)
G4.add_edge(4,19,weight=1)
G4.add_edge(4,20,weight=1)
G4.add_edge(4,21,weight=1)
G4.add_edge(4,22,weight=1)
G4.add_edge(4,23,weight=1)
G4.add_edge(4,24,weight=1)
G4.add_edge(4,25,weight=1)
G4.add_edge(4,26,weight=1)
G4.add_edge(4,27,weight=1)
G4.add_edge(4,28,weight=1)
G4.add_edge(5,6,weight=1)
G4.add_edge(5,7,weight=1)
G4.add_edge(5,29,weight=1)
G4.add_edge(5,30,weight=1)
G4.add_edge(5,31,weight=1)
G4.add_edge(5,32,weight=1)
G4.add_edge(6,7,weight=1)
G4.add_edge(6,8,weight=1)
G4.add_edge(7,8,weight=1)
G4.add_edge(7,11,weight=1)
G4.add_edge(7,13,weight=1)
G4.add_edge(7,19,weight=1)
G4.add_edge(7,23,weight=1)
G4.add_edge(7,26,weight=1)
G4.add_edge(7,30,weight=1)
G4.add_edge(8,15,weight=1)
G4.add_edge(8,31,weight=1)
G4.add_edge(8,33,weight=1)
G4.add_edge(8,34,weight=1)
G4.add_edge(8,35,weight=1)
G4.add_edge(9,10,weight=1)
G4.add_edge(9,11,weight=1)
G4.add_edge(9,12,weight=1)
G4.add_edge(9,13,weight=1)
G4.add_edge(9,14,weight=1)
G4.add_edge(9,15,weight=1)
G4.add_edge(9,21,weight=1)
G4.add_edge(9,22,weight=1)
G4.add_edge(9,23,weight=1)
G4.add_edge(9,24,weight=1)
G4.add_edge(9,25,weight=1)
G4.add_edge(9,27,weight=1)
G4.add_edge(9,28,weight=1)
G4.add_edge(9,36,weight=1)
G4.add_edge(9,37,weight=1)
G4.add_edge(9,38,weight=1)
G4.add_edge(9,39,weight=1)
G4.add_edge(9,40,weight=1)
G4.add_edge(9,41,weight=1)
G4.add_edge(9,42,weight=1)
G4.add_edge(9,43,weight=1)
G4.add_edge(9,44,weight=1)
G4.add_edge(9,45,weight=1)
G4.add_edge(9,46,weight=1)
G4.add_edge(10,12,weight=1)
G4.add_edge(10,13,weight=1)
G4.add_edge(10,15,weight=1)
G4.add_edge(10,21,weight=1)
G4.add_edge(10,25,weight=1)
G4.add_edge(10,28,weight=1)
G4.add_edge(10,41,weight=1)
G4.add_edge(10,45,weight=1)
G4.add_edge(10,47,weight=1)
G4.add_edge(10,48,weight=1)
G4.add_edge(10,49,weight=1)
G4.add_edge(10,50,weight=1)
G4.add_edge(10,51,weight=1)
G4.add_edge(10,52,weight=1)
G4.add_edge(11,12,weight=1)
G4.add_edge(11,13,weight=1)
G4.add_edge(11,16,weight=1)
G4.add_edge(11,17,weight=1)
G4.add_edge(11,20,weight=1)
G4.add_edge(11,22,weight=1)
G4.add_edge(11,37,weight=1)
G4.add_edge(11,38,weight=1)
G4.add_edge(11,39,weight=1)
G4.add_edge(11,53,weight=1)
G4.add_edge(11,54,weight=1)
G4.add_edge(11,55,weight=1)
G4.add_edge(12,13,weight=1)
G4.add_edge(12,14,weight=1)
G4.add_edge(12,15,weight=1)
G4.add_edge(12,18,weight=1)
G4.add_edge(12,21,weight=1)
G4.add_edge(12,22,weight=1)
G4.add_edge(12,23,weight=1)
G4.add_edge(12,27,weight=1)
G4.add_edge(12,28,weight=1)
G4.add_edge(12,30,weight=1)
G4.add_edge(12,45,weight=1)
G4.add_edge(12,47,weight=1)
G4.add_edge(12,50,weight=1)
G4.add_edge(12,56,weight=1)
G4.add_edge(13,14,weight=1)
G4.add_edge(13,15,weight=1)
G4.add_edge(13,16,weight=1)
G4.add_edge(13,18,weight=1)
G4.add_edge(13,19,weight=1)
G4.add_edge(13,24,weight=1)
G4.add_edge(13,25,weight=1)
G4.add_edge(13,36,weight=1)
G4.add_edge(13,37,weight=1)
G4.add_edge(13,57,weight=1)
G4.add_edge(13,53,weight=1)
G4.add_edge(13,54,weight=1)
G4.add_edge(13,40,weight=1)
G4.add_edge(13,41,weight=1)
G4.add_edge(13,44,weight=1)
G4.add_edge(13,46,weight=1)
G4.add_edge(13,47,weight=1)
G4.add_edge(13,58,weight=1)
G4.add_edge(13,55,weight=1)
G4.add_edge(14,18,weight=1)
G4.add_edge(14,30,weight=1)
G4.add_edge(14,36,weight=1)
G4.add_edge(14,40,weight=1)
G4.add_edge(14,42,weight=1)
G4.add_edge(14,43,weight=1)
G4.add_edge(14,44,weight=1)
G4.add_edge(14,47,weight=1)
G4.add_edge(14,59,weight=1)
G4.add_edge(15,26,weight=1)
G4.add_edge(15,27,weight=1)
G4.add_edge(15,33,weight=1)
G4.add_edge(16,17,weight=1)
G4.add_edge(16,55,weight=1)
G4.add_edge(18,47,weight=1)
G4.add_edge(20,55,weight=1)
G4.add_edge(20,56,weight=1)
G4.add_edge(20,60,weight=1)
G4.add_edge(21,25,weight=1)
G4.add_edge(21,40,weight=1)
G4.add_edge(21,48,weight=1)
G4.add_edge(21,49,weight=1)
G4.add_edge(21,61,weight=1)
G4.add_edge(21,59,weight=1)
G4.add_edge(22,24,weight=1)
G4.add_edge(23,26,weight=1)
G4.add_edge(23,40,weight=1)
G4.add_edge(23,52,weight=1)
G4.add_edge(24,28,weight=1)
G4.add_edge(24,36,weight=1)
G4.add_edge(24,37,weight=1)
G4.add_edge(24,47,weight=1)
G4.add_edge(24,58,weight=1)
G4.add_edge(25,27,weight=1)
G4.add_edge(25,40,weight=1)
G4.add_edge(25,47,weight=1)
G4.add_edge(25,61,weight=1)
G4.add_edge(26,40,weight=1)
G4.add_edge(27,40,weight=1)
G4.add_edge(27,45,weight=1)
G4.add_edge(27,47,weight=1)
G4.add_edge(27,61,weight=1)
G4.add_edge(28,40,weight=1)
G4.add_edge(28,41,weight=1)
G4.add_edge(28,47,weight=1)
G4.add_edge(28,58,weight=1)
G4.add_edge(29,62,weight=1)
G4.add_edge(29,63,weight=1)
G4.add_edge(31,32,weight=1)
G4.add_edge(31,33,weight=1)
G4.add_edge(31,62,weight=1)
G4.add_edge(31,64,weight=1)
G4.add_edge(31,65,weight=1)
G4.add_edge(31,66,weight=1)
G4.add_edge(31,67,weight=1)
G4.add_edge(31,68,weight=1)
G4.add_edge(31,69,weight=1)
G4.add_edge(31,60,weight=1)
G4.add_edge(31,70,weight=1)
G4.add_edge(31,71,weight=1)
G4.add_edge(31,72,weight=1)
G4.add_edge(31,73,weight=1)
G4.add_edge(31,74,weight=1)
G4.add_edge(31,75,weight=1)
G4.add_edge(31,76,weight=1)
G4.add_edge(31,77,weight=1)
G4.add_edge(32,49,weight=1)
G4.add_edge(32,66,weight=1)
G4.add_edge(32,67,weight=1)
G4.add_edge(32,68,weight=1)
G4.add_edge(32,69,weight=1)
G4.add_edge(32,60,weight=1)
G4.add_edge(32,78,weight=1)
G4.add_edge(32,72,weight=1)
G4.add_edge(32,79,weight=1)
G4.add_edge(36,37,weight=1)
G4.add_edge(37,39,weight=1)
G4.add_edge(37,53,weight=1)
G4.add_edge(37,54,weight=1)
G4.add_edge(37,47,weight=1)
G4.add_edge(80,38,weight=1)
G4.add_edge(80,57,weight=1)
G4.add_edge(80,39,weight=1)
G4.add_edge(80,53,weight=1)
G4.add_edge(80,54,weight=1)
G4.add_edge(38,57,weight=1)
G4.add_edge(38,39,weight=1)
G4.add_edge(38,53,weight=1)
G4.add_edge(38,54,weight=1)
G4.add_edge(38,40,weight=1)
G4.add_edge(38,43,weight=1)
G4.add_edge(38,44,weight=1)
G4.add_edge(57,41,weight=1)
G4.add_edge(57,47,weight=1)
G4.add_edge(39,53,weight=1)
G4.add_edge(39,47,weight=1)
G4.add_edge(53,54,weight=1)
G4.add_edge(54,40,weight=1)
G4.add_edge(54,42,weight=1)
G4.add_edge(40,41,weight=1)
G4.add_edge(40,42,weight=1)
G4.add_edge(40,44,weight=1)
G4.add_edge(40,45,weight=1)
G4.add_edge(40,47,weight=1)
G4.add_edge(40,61,weight=1)
G4.add_edge(40,58,weight=1)
G4.add_edge(41,47,weight=1)
G4.add_edge(41,58,weight=1)
G4.add_edge(42,43,weight=1)
G4.add_edge(42,47,weight=1)
G4.add_edge(43,56,weight=1)
G4.add_edge(45,47,weight=1)
G4.add_edge(46,47,weight=1)
G4.add_edge(46,81,weight=1)
G4.add_edge(47,58,weight=1)
G4.add_edge(48,49,weight=1)
G4.add_edge(48,59,weight=1)
G4.add_edge(49,59,weight=1)
G4.add_edge(49,33,weight=1)
G4.add_edge(49,63,weight=1)
G4.add_edge(49,69,weight=1)
G4.add_edge(50,33,weight=1)
G4.add_edge(51,52,weight=1)
G4.add_edge(51,33,weight=1)
G4.add_edge(51,82,weight=1)
G4.add_edge(51,83,weight=1)
G4.add_edge(51,84,weight=1)
G4.add_edge(52,33,weight=1)
G4.add_edge(52,82,weight=1)
G4.add_edge(61,69,weight=1)
G4.add_edge(56,59,weight=1)
G4.add_edge(33,82,weight=1)
G4.add_edge(33,83,weight=1)
G4.add_edge(33,85,weight=1)
G4.add_edge(33,84,weight=1)
G4.add_edge(33,60,weight=1)
G4.add_edge(33,35,weight=1)
G4.add_edge(86,87,weight=1)
G4.add_edge(86,88,weight=1)
G4.add_edge(86,89,weight=1)
G4.add_edge(86,90,weight=1)
G4.add_edge(86,77,weight=1)
G4.add_edge(87,89,weight=1)
G4.add_edge(87,90,weight=1)
G4.add_edge(87,74,weight=1)
G4.add_edge(87,75,weight=1)
G4.add_edge(87,77,weight=1)
G4.add_edge(88,75,weight=1)
G4.add_edge(88,91,weight=1)
G4.add_edge(88,92,weight=1)
G4.add_edge(89,90,weight=1)
G4.add_edge(89,74,weight=1)
G4.add_edge(89,77,weight=1)
G4.add_edge(89,93,weight=1)
G4.add_edge(90,77,weight=1)
G4.add_edge(82,83,weight=1)
G4.add_edge(82,62,weight=1)
G4.add_edge(82,64,weight=1)
G4.add_edge(82,85,weight=1)
G4.add_edge(82,84,weight=1)
G4.add_edge(82,65,weight=1)
G4.add_edge(83,64,weight=1)
G4.add_edge(83,85,weight=1)
G4.add_edge(83,84,weight=1)
G4.add_edge(83,35,weight=1)
G4.add_edge(62,64,weight=1)
G4.add_edge(62,65,weight=1)
G4.add_edge(62,63,weight=1)
G4.add_edge(62,66,weight=1)
G4.add_edge(62,67,weight=1)
G4.add_edge(62,69,weight=1)
G4.add_edge(62,71,weight=1)
G4.add_edge(62,74,weight=1)
G4.add_edge(62,35,weight=1)
G4.add_edge(62,75,weight=1)
G4.add_edge(62,94,weight=1)
G4.add_edge(62,95,weight=1)
G4.add_edge(62,96,weight=1)
G4.add_edge(62,76,weight=1)
G4.add_edge(62,97,weight=1)
G4.add_edge(62,98,weight=1)
G4.add_edge(62,77,weight=1)
G4.add_edge(62,93,weight=1)
G4.add_edge(64,99,weight=1)
G4.add_edge(64,100,weight=1)
G4.add_edge(85,34,weight=1)
G4.add_edge(84,100,weight=1)
G4.add_edge(65,34,weight=1)
G4.add_edge(65,63,weight=1)
G4.add_edge(65,68,weight=1)
G4.add_edge(65,96,weight=1)
G4.add_edge(34,63,weight=1)
G4.add_edge(34,66,weight=1)
G4.add_edge(34,67,weight=1)
G4.add_edge(34,68,weight=1)
G4.add_edge(34,69,weight=1)
G4.add_edge(34,60,weight=1)
G4.add_edge(34,78,weight=1)
G4.add_edge(34,70,weight=1)
G4.add_edge(34,71,weight=1)
G4.add_edge(34,101,weight=1)
G4.add_edge(34,72,weight=1)
G4.add_edge(34,73,weight=1)
G4.add_edge(63,66,weight=1)
G4.add_edge(63,67,weight=1)
G4.add_edge(63,68,weight=1)
G4.add_edge(63,69,weight=1)
G4.add_edge(63,78,weight=1)
G4.add_edge(63,70,weight=1)
G4.add_edge(63,71,weight=1)
G4.add_edge(63,72,weight=1)
G4.add_edge(63,74,weight=1)
G4.add_edge(63,35,weight=1)
G4.add_edge(63,75,weight=1)
G4.add_edge(63,102,weight=1)
G4.add_edge(63,94,weight=1)
G4.add_edge(63,95,weight=1)
G4.add_edge(63,96,weight=1)
G4.add_edge(63,79,weight=1)
G4.add_edge(63,103,weight=1)
G4.add_edge(66,67,weight=1)
G4.add_edge(66,68,weight=1)
G4.add_edge(66,72,weight=1)
G4.add_edge(66,73,weight=1)
G4.add_edge(66,74,weight=1)
G4.add_edge(66,75,weight=1)
G4.add_edge(66,95,weight=1)
G4.add_edge(66,103,weight=1)
G4.add_edge(66,76,weight=1)
G4.add_edge(66,104,weight=1)
G4.add_edge(66,91,weight=1)
G4.add_edge(66,97,weight=1)
G4.add_edge(66,98,weight=1)
G4.add_edge(66,105,weight=1)
G4.add_edge(66,77,weight=1)
G4.add_edge(66,93,weight=1)
G4.add_edge(67,68,weight=1)
G4.add_edge(67,78,weight=1)
G4.add_edge(67,70,weight=1)
G4.add_edge(67,72,weight=1)
G4.add_edge(67,74,weight=1)
G4.add_edge(67,102,weight=1)
G4.add_edge(67,94,weight=1)
G4.add_edge(67,79,weight=1)
G4.add_edge(67,105,weight=1)
G4.add_edge(67,77,weight=1)
G4.add_edge(68,69,weight=1)
G4.add_edge(68,60,weight=1)
G4.add_edge(68,78,weight=1)
G4.add_edge(68,70,weight=1)
G4.add_edge(68,72,weight=1)
G4.add_edge(68,73,weight=1)
G4.add_edge(68,74,weight=1)
G4.add_edge(68,79,weight=1)
G4.add_edge(68,103,weight=1)
G4.add_edge(69,60,weight=1)
G4.add_edge(69,72,weight=1)
G4.add_edge(69,73,weight=1)
G4.add_edge(69,74,weight=1)
G4.add_edge(69,75,weight=1)
G4.add_edge(70,74,weight=1)
G4.add_edge(70,79,weight=1)
G4.add_edge(70,93,weight=1)
G4.add_edge(101,74,weight=1)
G4.add_edge(101,75,weight=1)
G4.add_edge(101,98,weight=1)
G4.add_edge(72,74,weight=1)
G4.add_edge(73,74,weight=1)
G4.add_edge(73,102,weight=1)
G4.add_edge(73,93,weight=1)
G4.add_edge(74,75,weight=1)
G4.add_edge(74,102,weight=1)
G4.add_edge(74,94,weight=1)
G4.add_edge(74,95,weight=1)
G4.add_edge(74,104,weight=1)
G4.add_edge(74,97,weight=1)
G4.add_edge(74,98,weight=1)
G4.add_edge(74,77,weight=1)
G4.add_edge(74,93,weight=1)
G4.add_edge(74,92,weight=1)
G4.add_edge(75,95,weight=1)
G4.add_edge(75,76,weight=1)
G4.add_edge(75,98,weight=1)
G4.add_edge(75,93,weight=1)
G4.add_edge(75,92,weight=1)
G4.add_edge(102,105,weight=1)
G4.add_edge(94,95,weight=1)
G4.add_edge(96,79,weight=1)
G4.add_edge(96,77,weight=1)
G4.add_edge(79,105,weight=1)
G4.add_edge(79,93,weight=1)
G4.add_edge(76,104,weight=1)
G4.add_edge(76,77,weight=1)
G4.add_edge(76,81,weight=1)
G4.add_edge(104,91,weight=1)
G4.add_edge(104,97,weight=1)
G4.add_edge(104,92,weight=1)
G4.add_edge(104,81,weight=1)
G4.add_edge(91,81,weight=1)
G4.add_edge(97,98,weight=1)
G4.add_edge(97,93,weight=1)
G4.add_edge(105,93,weight=1)
G4.add_edge(77,93,weight=1)
G4.add_edge(93,92,weight=1)
G4.add_edge(99,100,weight=1)

#Football
G5 = nx.DiGraph()
G5.add_edge(1,2,weight=1)
G5.add_edge(1,3,weight=1)
G5.add_edge(1,4,weight=1)
G5.add_edge(1,5,weight=1)
G5.add_edge(1,6,weight=1)
G5.add_edge(1,7,weight=1)
G5.add_edge(1,8,weight=1)
G5.add_edge(1,9,weight=1)
G5.add_edge(1,10,weight=1)
G5.add_edge(1,11,weight=1)
G5.add_edge(1,12,weight=1)
G5.add_edge(1,13,weight=1)
G5.add_edge(2,14,weight=1)
G5.add_edge(2,15,weight=1)
G5.add_edge(2,7,weight=1)
G5.add_edge(2,16,weight=1)
G5.add_edge(2,17,weight=1)
G5.add_edge(2,18,weight=1)
G5.add_edge(2,19,weight=1)
G5.add_edge(2,20,weight=1)
G5.add_edge(2,21,weight=1)
G5.add_edge(2,22,weight=1)
G5.add_edge(2,23,weight=1)
G5.add_edge(24,25,weight=1)
G5.add_edge(24,26,weight=1)
G5.add_edge(24,27,weight=1)
G5.add_edge(24,28,weight=1)
G5.add_edge(24,29,weight=1)
G5.add_edge(24,30,weight=1)
G5.add_edge(24,31,weight=1)
G5.add_edge(24,32,weight=1)
G5.add_edge(24,33,weight=1)
G5.add_edge(24,34,weight=1)
G5.add_edge(24,35,weight=1)
G5.add_edge(24,36,weight=1)
G5.add_edge(25,37,weight=1)
G5.add_edge(25,38,weight=1)
G5.add_edge(25,39,weight=1)
G5.add_edge(25,40,weight=1)
G5.add_edge(25,41,weight=1)
G5.add_edge(25,42,weight=1)
G5.add_edge(25,33,weight=1)
G5.add_edge(25,34,weight=1)
G5.add_edge(25,43,weight=1)
G5.add_edge(25,44,weight=1)
G5.add_edge(25,45,weight=1)
G5.add_edge(3,37,weight=1)
G5.add_edge(3,4,weight=1)
G5.add_edge(3,5,weight=1)
G5.add_edge(3,6,weight=1)
G5.add_edge(3,46,weight=1)
G5.add_edge(3,9,weight=1)
G5.add_edge(3,47,weight=1)
G5.add_edge(3,12,weight=1)
G5.add_edge(3,13,weight=1)
G5.add_edge(3,48,weight=1)
G5.add_edge(37,49,weight=1)
G5.add_edge(37,38,weight=1)
G5.add_edge(37,41,weight=1)
G5.add_edge(37,34,weight=1)
G5.add_edge(37,43,weight=1)
G5.add_edge(37,44,weight=1)
G5.add_edge(37,11,weight=1)
G5.add_edge(37,50,weight=1)
G5.add_edge(37,51,weight=1)
G5.add_edge(37,52,weight=1)
G5.add_edge(26,53,weight=1)
G5.add_edge(26,54,weight=1)
G5.add_edge(26,55,weight=1)
G5.add_edge(26,30,weight=1)
G5.add_edge(26,56,weight=1)
G5.add_edge(26,42,weight=1)
G5.add_edge(26,31,weight=1)
G5.add_edge(26,32,weight=1)
G5.add_edge(26,57,weight=1)
G5.add_edge(26,35,weight=1)
G5.add_edge(26,36,weight=1)
G5.add_edge(53,58,weight=1)
G5.add_edge(53,59,weight=1)
G5.add_edge(53,60,weight=1)
G5.add_edge(53,40,weight=1)
G5.add_edge(53,61,weight=1)
G5.add_edge(53,62,weight=1)
G5.add_edge(53,63,weight=1)
G5.add_edge(53,64,weight=1)
G5.add_edge(53,65,weight=1)
G5.add_edge(53,48,weight=1)
G5.add_edge(53,66,weight=1)
G5.add_edge(58,4,weight=1)
G5.add_edge(58,59,weight=1)
G5.add_edge(58,60,weight=1)
G5.add_edge(58,9,weight=1)
G5.add_edge(58,67,weight=1)
G5.add_edge(58,61,weight=1)
G5.add_edge(58,63,weight=1)
G5.add_edge(58,64,weight=1)
G5.add_edge(58,11,weight=1)
G5.add_edge(58,66,weight=1)
G5.add_edge(4,5,weight=1)
G5.add_edge(4,60,weight=1)
G5.add_edge(4,6,weight=1)
G5.add_edge(4,9,weight=1)
G5.add_edge(4,32,weight=1)
G5.add_edge(4,12,weight=1)
G5.add_edge(4,13,weight=1)
G5.add_edge(4,48,weight=1)
G5.add_edge(49,38,weight=1)
G5.add_edge(49,31,weight=1)
G5.add_edge(49,33,weight=1)
G5.add_edge(49,34,weight=1)
G5.add_edge(49,43,weight=1)
G5.add_edge(49,44,weight=1)
G5.add_edge(49,51,weight=1)
G5.add_edge(49,45,weight=1)
G5.add_edge(49,52,weight=1)
G5.add_edge(38,68,weight=1)
G5.add_edge(38,46,weight=1)
G5.add_edge(38,69,weight=1)
G5.add_edge(38,47,weight=1)
G5.add_edge(38,11,weight=1)
G5.add_edge(38,50,weight=1)
G5.add_edge(38,13,weight=1)
G5.add_edge(70,27,weight=1)
G5.add_edge(70,28,weight=1)
G5.add_edge(70,71,weight=1)
G5.add_edge(70,72,weight=1)
G5.add_edge(70,39,weight=1)
G5.add_edge(70,73,weight=1)
G5.add_edge(70,74,weight=1)
G5.add_edge(70,75,weight=1)
G5.add_edge(70,76,weight=1)
G5.add_edge(70,57,weight=1)
G5.add_edge(27,29,weight=1)
G5.add_edge(27,54,weight=1)
G5.add_edge(27,55,weight=1)
G5.add_edge(27,17,weight=1)
G5.add_edge(27,31,weight=1)
G5.add_edge(27,32,weight=1)
G5.add_edge(27,35,weight=1)
G5.add_edge(27,36,weight=1)
G5.add_edge(27,77,weight=1)
G5.add_edge(28,29,weight=1)
G5.add_edge(28,39,weight=1)
G5.add_edge(28,75,weight=1)
G5.add_edge(28,76,weight=1)
G5.add_edge(28,78,weight=1)
G5.add_edge(28,79,weight=1)
G5.add_edge(28,57,weight=1)
G5.add_edge(28,80,weight=1)
G5.add_edge(29,54,weight=1)
G5.add_edge(29,55,weight=1)
G5.add_edge(29,30,weight=1)
G5.add_edge(29,31,weight=1)
G5.add_edge(29,61,weight=1)
G5.add_edge(29,81,weight=1)
G5.add_edge(29,35,weight=1)
G5.add_edge(29,36,weight=1)
G5.add_edge(29,82,weight=1)
G5.add_edge(5,71,weight=1)
G5.add_edge(5,6,weight=1)
G5.add_edge(5,75,weight=1)
G5.add_edge(5,9,weight=1)
G5.add_edge(5,83,weight=1)
G5.add_edge(5,43,weight=1)
G5.add_edge(5,12,weight=1)
G5.add_edge(5,13,weight=1)
G5.add_edge(71,84,weight=1)
G5.add_edge(71,15,weight=1)
G5.add_edge(71,42,weight=1)
G5.add_edge(71,85,weight=1)
G5.add_edge(71,10,weight=1)
G5.add_edge(71,86,weight=1)
G5.add_edge(71,87,weight=1)
G5.add_edge(71,88,weight=1)
G5.add_edge(71,89,weight=1)
G5.add_edge(72,90,weight=1)
G5.add_edge(72,91,weight=1)
G5.add_edge(72,73,weight=1)
G5.add_edge(72,74,weight=1)
G5.add_edge(72,75,weight=1)
G5.add_edge(72,92,weight=1)
G5.add_edge(72,78,weight=1)
G5.add_edge(72,93,weight=1)
G5.add_edge(72,79,weight=1)
G5.add_edge(72,80,weight=1)
G5.add_edge(90,94,weight=1)
G5.add_edge(90,95,weight=1)
G5.add_edge(90,7,weight=1)
G5.add_edge(90,8,weight=1)
G5.add_edge(90,74,weight=1)
G5.add_edge(90,96,weight=1)
G5.add_edge(90,56,weight=1)
G5.add_edge(90,97,weight=1)
G5.add_edge(90,98,weight=1)
G5.add_edge(90,20,weight=1)
G5.add_edge(84,59,weight=1)
G5.add_edge(84,74,weight=1)
G5.add_edge(84,85,weight=1)
G5.add_edge(84,10,weight=1)
G5.add_edge(84,99,weight=1)
G5.add_edge(84,100,weight=1)
G5.add_edge(84,101,weight=1)
G5.add_edge(84,86,weight=1)
G5.add_edge(84,88,weight=1)
G5.add_edge(84,89,weight=1)
G5.add_edge(59,60,weight=1)
G5.add_edge(59,54,weight=1)
G5.add_edge(59,102,weight=1)
G5.add_edge(59,67,weight=1)
G5.add_edge(59,61,weight=1)
G5.add_edge(59,63,weight=1)
G5.add_edge(59,48,weight=1)
G5.add_edge(59,66,weight=1)
G5.add_edge(60,6,weight=1)
G5.add_edge(60,30,weight=1)
G5.add_edge(60,67,weight=1)
G5.add_edge(60,61,weight=1)
G5.add_edge(60,63,weight=1)
G5.add_edge(60,64,weight=1)
G5.add_edge(60,48,weight=1)
G5.add_edge(6,9,weight=1)
G5.add_edge(6,64,weight=1)
G5.add_edge(6,11,weight=1)
G5.add_edge(6,12,weight=1)
G5.add_edge(6,13,weight=1)
G5.add_edge(6,66,weight=1)
G5.add_edge(68,14,weight=1)
G5.add_edge(68,46,weight=1)
G5.add_edge(68,69,weight=1)
G5.add_edge(68,103,weight=1)
G5.add_edge(68,47,weight=1)
G5.add_edge(68,44,weight=1)
G5.add_edge(68,86,weight=1)
G5.add_edge(68,11,weight=1)
G5.add_edge(68,77,weight=1)
G5.add_edge(14,7,weight=1)
G5.add_edge(14,16,weight=1)
G5.add_edge(14,17,weight=1)
G5.add_edge(14,104,weight=1)
G5.add_edge(14,19,weight=1)
G5.add_edge(14,21,weight=1)
G5.add_edge(14,22,weight=1)
G5.add_edge(14,36,weight=1)
G5.add_edge(14,23,weight=1)
G5.add_edge(39,15,weight=1)
G5.add_edge(39,73,weight=1)
G5.add_edge(39,75,weight=1)
G5.add_edge(39,92,weight=1)
G5.add_edge(39,76,weight=1)
G5.add_edge(39,93,weight=1)
G5.add_edge(39,57,weight=1)
G5.add_edge(15,105,weight=1)
G5.add_edge(15,85,weight=1)
G5.add_edge(15,106,weight=1)
G5.add_edge(15,10,weight=1)
G5.add_edge(15,99,weight=1)
G5.add_edge(15,101,weight=1)
G5.add_edge(15,87,weight=1)
G5.add_edge(15,88,weight=1)
G5.add_edge(46,75,weight=1)
G5.add_edge(46,69,weight=1)
G5.add_edge(46,47,weight=1)
G5.add_edge(46,64,weight=1)
G5.add_edge(46,11,weight=1)
G5.add_edge(46,89,weight=1)
G5.add_edge(94,95,weight=1)
G5.add_edge(94,8,weight=1)
G5.add_edge(94,92,weight=1)
G5.add_edge(94,56,weight=1)
G5.add_edge(94,97,weight=1)
G5.add_edge(94,107,weight=1)
G5.add_edge(94,65,weight=1)
G5.add_edge(94,108,weight=1)
G5.add_edge(94,98,weight=1)
G5.add_edge(94,20,weight=1)
G5.add_edge(95,8,weight=1)
G5.add_edge(95,96,weight=1)
G5.add_edge(95,69,weight=1)
G5.add_edge(95,56,weight=1)
G5.add_edge(95,97,weight=1)
G5.add_edge(95,65,weight=1)
G5.add_edge(95,98,weight=1)
G5.add_edge(95,20,weight=1)
G5.add_edge(95,23,weight=1)
G5.add_edge(91,54,weight=1)
G5.add_edge(91,73,weight=1)
G5.add_edge(91,76,weight=1)
G5.add_edge(91,78,weight=1)
G5.add_edge(91,56,weight=1)
G5.add_edge(91,93,weight=1)
G5.add_edge(91,79,weight=1)
G5.add_edge(91,97,weight=1)
G5.add_edge(91,57,weight=1)
G5.add_edge(91,80,weight=1)
G5.add_edge(54,55,weight=1)
G5.add_edge(54,30,weight=1)
G5.add_edge(54,109,weight=1)
G5.add_edge(54,32,weight=1)
G5.add_edge(54,35,weight=1)
G5.add_edge(54,36,weight=1)
G5.add_edge(7,16,weight=1)
G5.add_edge(7,17,weight=1)
G5.add_edge(7,19,weight=1)
G5.add_edge(7,21,weight=1)
G5.add_edge(7,22,weight=1)
G5.add_edge(7,23,weight=1)
G5.add_edge(73,8,weight=1)
G5.add_edge(73,92,weight=1)
G5.add_edge(73,78,weight=1)
G5.add_edge(73,93,weight=1)
G5.add_edge(73,79,weight=1)
G5.add_edge(73,98,weight=1)
G5.add_edge(73,80,weight=1)
G5.add_edge(8,96,weight=1)
G5.add_edge(8,56,weight=1)
G5.add_edge(8,97,weight=1)
G5.add_edge(8,81,weight=1)
G5.add_edge(8,98,weight=1)
G5.add_edge(8,20,weight=1)
G5.add_edge(74,16,weight=1)
G5.add_edge(74,76,weight=1)
G5.add_edge(74,42,weight=1)
G5.add_edge(74,110,weight=1)
G5.add_edge(16,17,weight=1)
G5.add_edge(16,107,weight=1)
G5.add_edge(16,19,weight=1)
G5.add_edge(16,87,weight=1)
G5.add_edge(16,21,weight=1)
G5.add_edge(16,22,weight=1)
G5.add_edge(16,23,weight=1)
G5.add_edge(75,55,weight=1)
G5.add_edge(75,76,weight=1)
G5.add_edge(75,78,weight=1)
G5.add_edge(75,79,weight=1)
G5.add_edge(75,57,weight=1)
G5.add_edge(55,30,weight=1)
G5.add_edge(55,78,weight=1)
G5.add_edge(55,31,weight=1)
G5.add_edge(55,65,weight=1)
G5.add_edge(55,35,weight=1)
G5.add_edge(55,36,weight=1)
G5.add_edge(40,9,weight=1)
G5.add_edge(40,67,weight=1)
G5.add_edge(40,41,weight=1)
G5.add_edge(40,33,weight=1)
G5.add_edge(40,34,weight=1)
G5.add_edge(40,43,weight=1)
G5.add_edge(40,51,weight=1)
G5.add_edge(40,45,weight=1)
G5.add_edge(40,52,weight=1)
G5.add_edge(9,83,weight=1)
G5.add_edge(9,12,weight=1)
G5.add_edge(9,13,weight=1)
G5.add_edge(92,76,weight=1)
G5.add_edge(92,18,weight=1)
G5.add_edge(92,106,weight=1)
G5.add_edge(76,93,weight=1)
G5.add_edge(76,99,weight=1)
G5.add_edge(76,97,weight=1)
G5.add_edge(76,57,weight=1)
G5.add_edge(96,17,weight=1)
G5.add_edge(96,111,weight=1)
G5.add_edge(96,18,weight=1)
G5.add_edge(96,103,weight=1)
G5.add_edge(96,100,weight=1)
G5.add_edge(96,112,weight=1)
G5.add_edge(96,108,weight=1)
G5.add_edge(96,113,weight=1)
G5.add_edge(17,85,weight=1)
G5.add_edge(17,19,weight=1)
G5.add_edge(17,21,weight=1)
G5.add_edge(17,22,weight=1)
G5.add_edge(17,23,weight=1)
G5.add_edge(102,30,weight=1)
G5.add_edge(102,109,weight=1)
G5.add_edge(102,104,weight=1)
G5.add_edge(102,83,weight=1)
G5.add_edge(102,62,weight=1)
G5.add_edge(102,114,weight=1)
G5.add_edge(102,115,weight=1)
G5.add_edge(102,77,weight=1)
G5.add_edge(102,66,weight=1)
G5.add_edge(102,82,weight=1)
G5.add_edge(30,31,weight=1)
G5.add_edge(30,93,weight=1)
G5.add_edge(30,32,weight=1)
G5.add_edge(30,35,weight=1)
G5.add_edge(111,109,weight=1)
G5.add_edge(111,104,weight=1)
G5.add_edge(111,18,weight=1)
G5.add_edge(111,103,weight=1)
G5.add_edge(111,100,weight=1)
G5.add_edge(111,112,weight=1)
G5.add_edge(111,108,weight=1)
G5.add_edge(111,81,weight=1)
G5.add_edge(111,88,weight=1)
G5.add_edge(111,51,weight=1)
G5.add_edge(109,104,weight=1)
G5.add_edge(109,83,weight=1)
G5.add_edge(109,62,weight=1)
G5.add_edge(109,114,weight=1)
G5.add_edge(109,44,weight=1)
G5.add_edge(109,115,weight=1)
G5.add_edge(109,77,weight=1)
G5.add_edge(109,82,weight=1)
G5.add_edge(69,67,weight=1)
G5.add_edge(69,61,weight=1)
G5.add_edge(69,47,weight=1)
G5.add_edge(69,64,weight=1)
G5.add_edge(69,11,weight=1)
G5.add_edge(67,61,weight=1)
G5.add_edge(67,63,weight=1)
G5.add_edge(67,64,weight=1)
G5.add_edge(67,20,weight=1)
G5.add_edge(67,48,weight=1)
G5.add_edge(67,66,weight=1)
G5.add_edge(41,104,weight=1)
G5.add_edge(41,33,weight=1)
G5.add_edge(41,34,weight=1)
G5.add_edge(41,44,weight=1)
G5.add_edge(41,51,weight=1)
G5.add_edge(41,45,weight=1)
G5.add_edge(41,113,weight=1)
G5.add_edge(104,83,weight=1)
G5.add_edge(104,62,weight=1)
G5.add_edge(104,114,weight=1)
G5.add_edge(104,112,weight=1)
G5.add_edge(104,115,weight=1)
G5.add_edge(104,77,weight=1)
G5.add_edge(104,82,weight=1)
G5.add_edge(78,56,weight=1)
G5.add_edge(78,93,weight=1)
G5.add_edge(78,79,weight=1)
G5.add_edge(78,80,weight=1)
G5.add_edge(56,97,weight=1)
G5.add_edge(56,19,weight=1)
G5.add_edge(56,98,weight=1)
G5.add_edge(56,20,weight=1)
G5.add_edge(105,18,weight=1)
G5.add_edge(105,85,weight=1)
G5.add_edge(105,10,weight=1)
G5.add_edge(105,99,weight=1)
G5.add_edge(105,101,weight=1)
G5.add_edge(105,86,weight=1)
G5.add_edge(105,87,weight=1)
G5.add_edge(105,88,weight=1)
G5.add_edge(105,36,weight=1)
G5.add_edge(18,100,weight=1)
G5.add_edge(18,112,weight=1)
G5.add_edge(18,108,weight=1)
G5.add_edge(18,81,weight=1)
G5.add_edge(18,113,weight=1)
G5.add_edge(42,110,weight=1)
G5.add_edge(42,106,weight=1)
G5.add_edge(42,115,weight=1)
G5.add_edge(42,50,weight=1)
G5.add_edge(42,20,weight=1)
G5.add_edge(42,82,weight=1)
G5.add_edge(110,31,weight=1)
G5.add_edge(110,106,weight=1)
G5.add_edge(110,103,weight=1)
G5.add_edge(110,101,weight=1)
G5.add_edge(110,50,weight=1)
G5.add_edge(110,89,weight=1)
G5.add_edge(31,32,weight=1)
G5.add_edge(31,79,weight=1)
G5.add_edge(31,36,weight=1)
G5.add_edge(93,85,weight=1)
G5.add_edge(93,79,weight=1)
G5.add_edge(93,81,weight=1)
G5.add_edge(93,80,weight=1)
G5.add_edge(85,99,weight=1)
G5.add_edge(85,101,weight=1)
G5.add_edge(85,86,weight=1)
G5.add_edge(85,87,weight=1)
G5.add_edge(85,22,weight=1)
G5.add_edge(106,32,weight=1)
G5.add_edge(106,10,weight=1)
G5.add_edge(106,50,weight=1)
G5.add_edge(106,23,weight=1)
G5.add_edge(106,113,weight=1)
G5.add_edge(32,35,weight=1)
G5.add_edge(32,36,weight=1)
G5.add_edge(32,66,weight=1)
G5.add_edge(10,103,weight=1)
G5.add_edge(10,99,weight=1)
G5.add_edge(10,86,weight=1)
G5.add_edge(10,88,weight=1)
G5.add_edge(10,89,weight=1)
G5.add_edge(103,100,weight=1)
G5.add_edge(103,101,weight=1)
G5.add_edge(103,112,weight=1)
G5.add_edge(103,108,weight=1)
G5.add_edge(103,81,weight=1)
G5.add_edge(103,113,weight=1)
G5.add_edge(83,61,weight=1)
G5.add_edge(83,62,weight=1)
G5.add_edge(83,114,weight=1)
G5.add_edge(83,115,weight=1)
G5.add_edge(83,13,weight=1)
G5.add_edge(83,77,weight=1)
G5.add_edge(83,82,weight=1)
G5.add_edge(61,64,weight=1)
G5.add_edge(61,48,weight=1)
G5.add_edge(61,66,weight=1)
G5.add_edge(47,99,weight=1)
G5.add_edge(47,114,weight=1)
G5.add_edge(47,115,weight=1)
G5.add_edge(47,11,weight=1)
G5.add_edge(47,108,weight=1)
G5.add_edge(47,87,weight=1)
G5.add_edge(99,101,weight=1)
G5.add_edge(99,87,weight=1)
G5.add_edge(99,21,weight=1)
G5.add_edge(99,89,weight=1)
G5.add_edge(79,33,weight=1)
G5.add_edge(79,80,weight=1)
G5.add_edge(33,34,weight=1)
G5.add_edge(33,43,weight=1)
G5.add_edge(33,45,weight=1)
G5.add_edge(33,13,weight=1)
G5.add_edge(33,52,weight=1)
G5.add_edge(62,34,weight=1)
G5.add_edge(62,63,weight=1)
G5.add_edge(62,114,weight=1)
G5.add_edge(62,115,weight=1)
G5.add_edge(62,77,weight=1)
G5.add_edge(62,82,weight=1)
G5.add_edge(34,65,weight=1)
G5.add_edge(34,44,weight=1)
G5.add_edge(34,45,weight=1)
G5.add_edge(100,101,weight=1)
G5.add_edge(100,112,weight=1)
G5.add_edge(100,81,weight=1)
G5.add_edge(100,52,weight=1)
G5.add_edge(100,113,weight=1)
G5.add_edge(101,87,weight=1)
G5.add_edge(101,88,weight=1)
G5.add_edge(101,89,weight=1)
G5.add_edge(63,64,weight=1)
G5.add_edge(63,65,weight=1)
G5.add_edge(63,51,weight=1)
G5.add_edge(63,48,weight=1)
G5.add_edge(63,66,weight=1)
G5.add_edge(64,48,weight=1)
G5.add_edge(64,66,weight=1)
G5.add_edge(97,107,weight=1)
G5.add_edge(97,98,weight=1)
G5.add_edge(97,20,weight=1)
G5.add_edge(97,23,weight=1)
G5.add_edge(107,65,weight=1)
G5.add_edge(107,57,weight=1)
G5.add_edge(107,112,weight=1)
G5.add_edge(107,108,weight=1)
G5.add_edge(107,12,weight=1)
G5.add_edge(107,98,weight=1)
G5.add_edge(107,22,weight=1)
G5.add_edge(107,77,weight=1)
G5.add_edge(43,65,weight=1)
G5.add_edge(43,114,weight=1)
G5.add_edge(43,44,weight=1)
G5.add_edge(43,51,weight=1)
G5.add_edge(43,52,weight=1)
G5.add_edge(65,12,weight=1)
G5.add_edge(65,98,weight=1)
G5.add_edge(65,35,weight=1)
G5.add_edge(114,44,weight=1)
G5.add_edge(114,115,weight=1)
G5.add_edge(114,77,weight=1)
G5.add_edge(114,82,weight=1)
G5.add_edge(44,51,weight=1)
G5.add_edge(44,52,weight=1)
G5.add_edge(57,80,weight=1)
G5.add_edge(112,86,weight=1)
G5.add_edge(112,108,weight=1)
G5.add_edge(112,81,weight=1)
G5.add_edge(112,50,weight=1)
G5.add_edge(86,87,weight=1)
G5.add_edge(86,88,weight=1)
G5.add_edge(86,13,weight=1)
G5.add_edge(86,89,weight=1)
G5.add_edge(115,19,weight=1)
G5.add_edge(115,52,weight=1)
G5.add_edge(115,77,weight=1)
G5.add_edge(115,82,weight=1)
G5.add_edge(19,80,weight=1)
G5.add_edge(19,21,weight=1)
G5.add_edge(19,22,weight=1)
G5.add_edge(19,23,weight=1)
G5.add_edge(108,81,weight=1)
G5.add_edge(108,12,weight=1)
G5.add_edge(108,113,weight=1)
G5.add_edge(81,36,weight=1)
G5.add_edge(81,113,weight=1)
G5.add_edge(12,13,weight=1)
G5.add_edge(98,20,weight=1)
G5.add_edge(87,89,weight=1)
G5.add_edge(88,113,weight=1)
G5.add_edge(88,89,weight=1)
G5.add_edge(50,51,weight=1)
G5.add_edge(50,113,weight=1)
G5.add_edge(51,45,weight=1)
G5.add_edge(51,52,weight=1)
G5.add_edge(80,35,weight=1)
G5.add_edge(35,45,weight=1)
G5.add_edge(45,21,weight=1)
G5.add_edge(45,52,weight=1)
G5.add_edge(21,22,weight=1)
G5.add_edge(21,23,weight=1)
G5.add_edge(13,82,weight=1)
G5.add_edge(22,23,weight=1)
G5.add_edge(48,66,weight=1)
G5.add_edge(77,82,weight=1)



#Jass
G6 = nx.DiGraph()
G6.add_edge(1,10,weight=1)
G6.add_edge(1,11,weight=1)
G6.add_edge(1,12,weight=1)
G6.add_edge(1,13,weight=1)
G6.add_edge(1,14,weight=1)
G6.add_edge(1,15,weight=1)
G6.add_edge(1,16,weight=1)
G6.add_edge(1,17,weight=1)
G6.add_edge(1,18,weight=1)
G6.add_edge(1,19,weight=1)
G6.add_edge(1,2,weight=1)
G6.add_edge(1,20,weight=1)
G6.add_edge(1,21,weight=1)
G6.add_edge(1,22,weight=1)
G6.add_edge(1,23,weight=1)
G6.add_edge(1,24,weight=1)
G6.add_edge(1,3,weight=1)
G6.add_edge(1,4,weight=1)
G6.add_edge(1,5,weight=1)
G6.add_edge(1,6,weight=1)
G6.add_edge(1,7,weight=1)
G6.add_edge(1,8,weight=1)
G6.add_edge(1,9,weight=1)
G6.add_edge(2,11,weight=1)
G6.add_edge(2,12,weight=1)
G6.add_edge(2,120,weight=1)
G6.add_edge(2,121,weight=1)
G6.add_edge(2,122,weight=1)
G6.add_edge(2,123,weight=1)
G6.add_edge(2,124,weight=1)
G6.add_edge(2,125,weight=1)
G6.add_edge(2,13,weight=1)
G6.add_edge(2,14,weight=1)
G6.add_edge(2,15,weight=1)
G6.add_edge(2,19,weight=1)
G6.add_edge(2,20,weight=1)
G6.add_edge(2,21,weight=1)
G6.add_edge(2,23,weight=1)
G6.add_edge(2,24,weight=1)
G6.add_edge(2,4,weight=1)
G6.add_edge(2,6,weight=1)
G6.add_edge(2,67,weight=1)
G6.add_edge(2,7,weight=1)
G6.add_edge(3,10,weight=1)
G6.add_edge(3,125,weight=1)
G6.add_edge(3,13,weight=1)
G6.add_edge(3,130,weight=1)
G6.add_edge(3,14,weight=1)
G6.add_edge(3,145,weight=1)
G6.add_edge(3,146,weight=1)
G6.add_edge(3,147,weight=1)
G6.add_edge(3,148,weight=1)
G6.add_edge(3,149,weight=1)
G6.add_edge(3,150,weight=1)
G6.add_edge(3,157,weight=1)
G6.add_edge(3,159,weight=1)
G6.add_edge(3,16,weight=1)
G6.add_edge(3,160,weight=1)
G6.add_edge(3,17,weight=1)
G6.add_edge(3,18,weight=1)
G6.add_edge(3,20,weight=1)
G6.add_edge(3,21,weight=1)
G6.add_edge(3,23,weight=1)
G6.add_edge(3,24,weight=1)
G6.add_edge(3,4,weight=1)
G6.add_edge(3,5,weight=1)
G6.add_edge(3,6,weight=1)
G6.add_edge(3,67,weight=1)
G6.add_edge(3,7,weight=1)
G6.add_edge(3,8,weight=1)
G6.add_edge(3,9,weight=1)
G6.add_edge(4,101,weight=1)
G6.add_edge(4,11,weight=1)
G6.add_edge(4,112,weight=1)
G6.add_edge(4,12,weight=1)
G6.add_edge(4,121,weight=1)
G6.add_edge(4,128,weight=1)
G6.add_edge(4,13,weight=1)
G6.add_edge(4,133,weight=1)
G6.add_edge(4,137,weight=1)
G6.add_edge(4,14,weight=1)
G6.add_edge(4,149,weight=1)
G6.add_edge(4,15,weight=1)
G6.add_edge(4,150,weight=1)
G6.add_edge(4,152,weight=1)
G6.add_edge(4,16,weight=1)
G6.add_edge(4,164,weight=1)
G6.add_edge(4,165,weight=1)
G6.add_edge(4,166,weight=1)
G6.add_edge(4,167,weight=1)
G6.add_edge(4,168,weight=1)
G6.add_edge(4,169,weight=1)
G6.add_edge(4,17,weight=1)
G6.add_edge(4,170,weight=1)
G6.add_edge(4,171,weight=1)
G6.add_edge(4,172,weight=1)
G6.add_edge(4,173,weight=1)
G6.add_edge(4,174,weight=1)
G6.add_edge(4,177,weight=1)
G6.add_edge(4,178,weight=1)
G6.add_edge(4,179,weight=1)
G6.add_edge(4,18,weight=1)
G6.add_edge(4,19,weight=1)
G6.add_edge(4,20,weight=1)
G6.add_edge(4,21,weight=1)
G6.add_edge(4,23,weight=1)
G6.add_edge(4,24,weight=1)
G6.add_edge(4,6,weight=1)
G6.add_edge(4,7,weight=1)
G6.add_edge(4,8,weight=1)
G6.add_edge(4,9,weight=1)
G6.add_edge(5,10,weight=1)
G6.add_edge(5,153,weight=1)
G6.add_edge(5,155,weight=1)
G6.add_edge(5,160,weight=1)
G6.add_edge(5,164,weight=1)
G6.add_edge(5,165,weight=1)
G6.add_edge(5,17,weight=1)
G6.add_edge(5,22,weight=1)
G6.add_edge(5,6,weight=1)
G6.add_edge(5,67,weight=1)
G6.add_edge(6,10,weight=1)
G6.add_edge(6,11,weight=1)
G6.add_edge(6,12,weight=1)
G6.add_edge(6,13,weight=1)
G6.add_edge(6,14,weight=1)
G6.add_edge(6,15,weight=1)
G6.add_edge(6,153,weight=1)
G6.add_edge(6,155,weight=1)
G6.add_edge(6,164,weight=1)
G6.add_edge(6,165,weight=1)
G6.add_edge(6,17,weight=1)
G6.add_edge(6,19,weight=1)
G6.add_edge(6,20,weight=1)
G6.add_edge(6,21,weight=1)
G6.add_edge(6,22,weight=1)
G6.add_edge(6,23,weight=1)
G6.add_edge(6,24,weight=1)
G6.add_edge(6,7,weight=1)
G6.add_edge(7,10,weight=1)
G6.add_edge(7,100,weight=1)
G6.add_edge(7,101,weight=1)
G6.add_edge(7,102,weight=1)
G6.add_edge(7,103,weight=1)
G6.add_edge(7,104,weight=1)
G6.add_edge(7,105,weight=1)
G6.add_edge(7,106,weight=1)
G6.add_edge(7,107,weight=1)
G6.add_edge(7,108,weight=1)
G6.add_edge(7,109,weight=1)
G6.add_edge(7,11,weight=1)
G6.add_edge(7,110,weight=1)
G6.add_edge(7,111,weight=1)
G6.add_edge(7,116,weight=1)
G6.add_edge(7,117,weight=1)
G6.add_edge(7,118,weight=1)
G6.add_edge(7,119,weight=1)
G6.add_edge(7,12,weight=1)
G6.add_edge(7,120,weight=1)
G6.add_edge(7,121,weight=1)
G6.add_edge(7,122,weight=1)
G6.add_edge(7,123,weight=1)
G6.add_edge(7,124,weight=1)
G6.add_edge(7,125,weight=1)
G6.add_edge(7,127,weight=1)
G6.add_edge(7,128,weight=1)
G6.add_edge(7,13,weight=1)
G6.add_edge(7,130,weight=1)
G6.add_edge(7,133,weight=1)
G6.add_edge(7,137,weight=1)
G6.add_edge(7,138,weight=1)
G6.add_edge(7,139,weight=1)
G6.add_edge(7,14,weight=1)
G6.add_edge(7,140,weight=1)
G6.add_edge(7,148,weight=1)
G6.add_edge(7,149,weight=1)
G6.add_edge(7,15,weight=1)
G6.add_edge(7,150,weight=1)
G6.add_edge(7,152,weight=1)
G6.add_edge(7,153,weight=1)
G6.add_edge(7,154,weight=1)
G6.add_edge(7,155,weight=1)
G6.add_edge(7,157,weight=1)
G6.add_edge(7,164,weight=1)
G6.add_edge(7,165,weight=1)
G6.add_edge(7,166,weight=1)
G6.add_edge(7,167,weight=1)
G6.add_edge(7,168,weight=1)
G6.add_edge(7,169,weight=1)
G6.add_edge(7,170,weight=1)
G6.add_edge(7,171,weight=1)
G6.add_edge(7,172,weight=1)
G6.add_edge(7,173,weight=1)
G6.add_edge(7,174,weight=1)
G6.add_edge(7,177,weight=1)
G6.add_edge(7,178,weight=1)
G6.add_edge(7,179,weight=1)
G6.add_edge(7,18,weight=1)
G6.add_edge(7,19,weight=1)
G6.add_edge(7,191,weight=1)
G6.add_edge(7,20,weight=1)
G6.add_edge(7,21,weight=1)
G6.add_edge(7,23,weight=1)
G6.add_edge(7,24,weight=1)
G6.add_edge(7,26,weight=1)
G6.add_edge(7,27,weight=1)
G6.add_edge(7,28,weight=1)
G6.add_edge(7,48,weight=1)
G6.add_edge(7,54,weight=1)
G6.add_edge(7,55,weight=1)
G6.add_edge(7,67,weight=1)
G6.add_edge(7,74,weight=1)
G6.add_edge(7,75,weight=1)
G6.add_edge(7,76,weight=1)
G6.add_edge(7,80,weight=1)
G6.add_edge(7,81,weight=1)
G6.add_edge(7,83,weight=1)
G6.add_edge(7,84,weight=1)
G6.add_edge(7,85,weight=1)
G6.add_edge(7,86,weight=1)
G6.add_edge(7,87,weight=1)
G6.add_edge(7,89,weight=1)
G6.add_edge(7,9,weight=1)
G6.add_edge(7,90,weight=1)
G6.add_edge(7,92,weight=1)
G6.add_edge(7,93,weight=1)
G6.add_edge(7,95,weight=1)
G6.add_edge(7,96,weight=1)
G6.add_edge(7,97,weight=1)
G6.add_edge(7,98,weight=1)
G6.add_edge(8,100,weight=1)
G6.add_edge(8,110,weight=1)
G6.add_edge(8,111,weight=1)
G6.add_edge(8,16,weight=1)
G6.add_edge(8,17,weight=1)
G6.add_edge(8,18,weight=1)
G6.add_edge(8,21,weight=1)
G6.add_edge(8,23,weight=1)
G6.add_edge(8,24,weight=1)
G6.add_edge(8,28,weight=1)
G6.add_edge(8,80,weight=1)
G6.add_edge(8,81,weight=1)
G6.add_edge(8,9,weight=1)
G6.add_edge(8,91,weight=1)
G6.add_edge(8,92,weight=1)
G6.add_edge(8,94,weight=1)
G6.add_edge(8,98,weight=1)
G6.add_edge(9,109,weight=1)
G6.add_edge(9,11,weight=1)
G6.add_edge(9,125,weight=1)
G6.add_edge(9,13,weight=1)
G6.add_edge(9,130,weight=1)
G6.add_edge(9,134,weight=1)
G6.add_edge(9,137,weight=1)
G6.add_edge(9,14,weight=1)
G6.add_edge(9,146,weight=1)
G6.add_edge(9,148,weight=1)
G6.add_edge(9,157,weight=1)
G6.add_edge(9,159,weight=1)
G6.add_edge(9,16,weight=1)
G6.add_edge(9,160,weight=1)
G6.add_edge(9,17,weight=1)
G6.add_edge(9,18,weight=1)
G6.add_edge(9,192,weight=1)
G6.add_edge(9,21,weight=1)
G6.add_edge(9,23,weight=1)
G6.add_edge(9,24,weight=1)
G6.add_edge(9,67,weight=1)
G6.add_edge(10,109,weight=1)
G6.add_edge(10,111,weight=1)
G6.add_edge(10,112,weight=1)
G6.add_edge(10,114,weight=1)
G6.add_edge(10,119,weight=1)
G6.add_edge(10,12,weight=1)
G6.add_edge(10,125,weight=1)
G6.add_edge(10,13,weight=1)
G6.add_edge(10,14,weight=1)
G6.add_edge(10,145,weight=1)
G6.add_edge(10,146,weight=1)
G6.add_edge(10,147,weight=1)
G6.add_edge(10,149,weight=1)
G6.add_edge(10,15,weight=1)
G6.add_edge(10,153,weight=1)
G6.add_edge(10,155,weight=1)
G6.add_edge(10,158,weight=1)
G6.add_edge(10,159,weight=1)
G6.add_edge(10,160,weight=1)
G6.add_edge(10,164,weight=1)
G6.add_edge(10,165,weight=1)
G6.add_edge(10,17,weight=1)
G6.add_edge(10,18,weight=1)
G6.add_edge(10,19,weight=1)
G6.add_edge(10,20,weight=1)
G6.add_edge(10,28,weight=1)
G6.add_edge(10,54,weight=1)
G6.add_edge(10,67,weight=1)
G6.add_edge(10,74,weight=1)
G6.add_edge(10,75,weight=1)
G6.add_edge(10,76,weight=1)
G6.add_edge(10,80,weight=1)
G6.add_edge(10,83,weight=1)
G6.add_edge(10,85,weight=1)
G6.add_edge(10,86,weight=1)
G6.add_edge(10,88,weight=1)
G6.add_edge(10,93,weight=1)
G6.add_edge(11,101,weight=1)
G6.add_edge(11,102,weight=1)
G6.add_edge(11,103,weight=1)
G6.add_edge(11,106,weight=1)
G6.add_edge(11,107,weight=1)
G6.add_edge(11,109,weight=1)
G6.add_edge(11,116,weight=1)
G6.add_edge(11,117,weight=1)
G6.add_edge(11,118,weight=1)
G6.add_edge(11,12,weight=1)
G6.add_edge(11,120,weight=1)
G6.add_edge(11,121,weight=1)
G6.add_edge(11,124,weight=1)
G6.add_edge(11,125,weight=1)
G6.add_edge(11,13,weight=1)
G6.add_edge(11,130,weight=1)
G6.add_edge(11,133,weight=1)
G6.add_edge(11,137,weight=1)
G6.add_edge(11,14,weight=1)
G6.add_edge(11,148,weight=1)
G6.add_edge(11,15,weight=1)
G6.add_edge(11,18,weight=1)
G6.add_edge(11,19,weight=1)
G6.add_edge(11,20,weight=1)
G6.add_edge(11,21,weight=1)
G6.add_edge(11,23,weight=1)
G6.add_edge(11,24,weight=1)
G6.add_edge(11,67,weight=1)
G6.add_edge(11,80,weight=1)
G6.add_edge(11,89,weight=1)
G6.add_edge(11,90,weight=1)
G6.add_edge(11,95,weight=1)
G6.add_edge(11,96,weight=1)
G6.add_edge(11,97,weight=1)
G6.add_edge(12,101,weight=1)
G6.add_edge(12,111,weight=1)
G6.add_edge(12,112,weight=1)
G6.add_edge(12,114,weight=1)
G6.add_edge(12,121,weight=1)
G6.add_edge(12,125,weight=1)
G6.add_edge(12,128,weight=1)
G6.add_edge(12,13,weight=1)
G6.add_edge(12,133,weight=1)
G6.add_edge(12,137,weight=1)
G6.add_edge(12,14,weight=1)
G6.add_edge(12,149,weight=1)
G6.add_edge(12,15,weight=1)
G6.add_edge(12,150,weight=1)
G6.add_edge(12,152,weight=1)
G6.add_edge(12,158,weight=1)
G6.add_edge(12,159,weight=1)
G6.add_edge(12,160,weight=1)
G6.add_edge(12,164,weight=1)
G6.add_edge(12,165,weight=1)
G6.add_edge(12,166,weight=1)
G6.add_edge(12,167,weight=1)
G6.add_edge(12,168,weight=1)
G6.add_edge(12,169,weight=1)
G6.add_edge(12,170,weight=1)
G6.add_edge(12,171,weight=1)
G6.add_edge(12,172,weight=1)
G6.add_edge(12,173,weight=1)
G6.add_edge(12,174,weight=1)
G6.add_edge(12,18,weight=1)
G6.add_edge(12,19,weight=1)
G6.add_edge(12,20,weight=1)
G6.add_edge(12,21,weight=1)
G6.add_edge(12,23,weight=1)
G6.add_edge(12,24,weight=1)
G6.add_edge(12,67,weight=1)
G6.add_edge(12,74,weight=1)
G6.add_edge(12,76,weight=1)
G6.add_edge(12,93,weight=1)
G6.add_edge(13,101,weight=1)
G6.add_edge(13,111,weight=1)
G6.add_edge(13,112,weight=1)
G6.add_edge(13,114,weight=1)
G6.add_edge(13,121,weight=1)
G6.add_edge(13,122,weight=1)
G6.add_edge(13,123,weight=1)
G6.add_edge(13,125,weight=1)
G6.add_edge(13,128,weight=1)
G6.add_edge(13,133,weight=1)
G6.add_edge(13,137,weight=1)
G6.add_edge(13,14,weight=1)
G6.add_edge(13,149,weight=1)
G6.add_edge(13,15,weight=1)
G6.add_edge(13,150,weight=1)
G6.add_edge(13,152,weight=1)
G6.add_edge(13,154,weight=1)
G6.add_edge(13,158,weight=1)
G6.add_edge(13,159,weight=1)
G6.add_edge(13,160,weight=1)
G6.add_edge(13,164,weight=1)
G6.add_edge(13,165,weight=1)
G6.add_edge(13,166,weight=1)
G6.add_edge(13,167,weight=1)
G6.add_edge(13,168,weight=1)
G6.add_edge(13,169,weight=1)
G6.add_edge(13,170,weight=1)
G6.add_edge(13,171,weight=1)
G6.add_edge(13,172,weight=1)
G6.add_edge(13,173,weight=1)
G6.add_edge(13,174,weight=1)
G6.add_edge(13,177,weight=1)
G6.add_edge(13,178,weight=1)
G6.add_edge(13,179,weight=1)
G6.add_edge(13,18,weight=1)
G6.add_edge(13,187,weight=1)
G6.add_edge(13,188,weight=1)
G6.add_edge(13,189,weight=1)
G6.add_edge(13,19,weight=1)
G6.add_edge(13,20,weight=1)
G6.add_edge(13,21,weight=1)
G6.add_edge(13,23,weight=1)
G6.add_edge(13,24,weight=1)
G6.add_edge(13,56,weight=1)
G6.add_edge(13,67,weight=1)
G6.add_edge(13,74,weight=1)
G6.add_edge(13,76,weight=1)
G6.add_edge(13,89,weight=1)
G6.add_edge(13,90,weight=1)
G6.add_edge(13,93,weight=1)
G6.add_edge(14,101,weight=1)
G6.add_edge(14,111,weight=1)
G6.add_edge(14,112,weight=1)
G6.add_edge(14,114,weight=1)
G6.add_edge(14,121,weight=1)
G6.add_edge(14,125,weight=1)
G6.add_edge(14,128,weight=1)
G6.add_edge(14,133,weight=1)
G6.add_edge(14,137,weight=1)
G6.add_edge(14,149,weight=1)
G6.add_edge(14,15,weight=1)
G6.add_edge(14,150,weight=1)
G6.add_edge(14,152,weight=1)
G6.add_edge(14,158,weight=1)
G6.add_edge(14,159,weight=1)
G6.add_edge(14,160,weight=1)
G6.add_edge(14,164,weight=1)
G6.add_edge(14,165,weight=1)
G6.add_edge(14,166,weight=1)
G6.add_edge(14,167,weight=1)
G6.add_edge(14,168,weight=1)
G6.add_edge(14,169,weight=1)
G6.add_edge(14,170,weight=1)
G6.add_edge(14,171,weight=1)
G6.add_edge(14,172,weight=1)
G6.add_edge(14,173,weight=1)
G6.add_edge(14,174,weight=1)
G6.add_edge(14,18,weight=1)
G6.add_edge(14,187,weight=1)
G6.add_edge(14,19,weight=1)
G6.add_edge(14,20,weight=1)
G6.add_edge(14,21,weight=1)
G6.add_edge(14,23,weight=1)
G6.add_edge(14,24,weight=1)
G6.add_edge(14,67,weight=1)
G6.add_edge(14,74,weight=1)
G6.add_edge(14,76,weight=1)
G6.add_edge(14,93,weight=1)
G6.add_edge(15,101,weight=1)
G6.add_edge(15,111,weight=1)
G6.add_edge(15,112,weight=1)
G6.add_edge(15,114,weight=1)
G6.add_edge(15,121,weight=1)
G6.add_edge(15,125,weight=1)
G6.add_edge(15,128,weight=1)
G6.add_edge(15,133,weight=1)
G6.add_edge(15,137,weight=1)
G6.add_edge(15,149,weight=1)
G6.add_edge(15,150,weight=1)
G6.add_edge(15,152,weight=1)
G6.add_edge(15,158,weight=1)
G6.add_edge(15,159,weight=1)
G6.add_edge(15,160,weight=1)
G6.add_edge(15,164,weight=1)
G6.add_edge(15,165,weight=1)
G6.add_edge(15,166,weight=1)
G6.add_edge(15,167,weight=1)
G6.add_edge(15,168,weight=1)
G6.add_edge(15,169,weight=1)
G6.add_edge(15,170,weight=1)
G6.add_edge(15,171,weight=1)
G6.add_edge(15,172,weight=1)
G6.add_edge(15,173,weight=1)
G6.add_edge(15,174,weight=1)
G6.add_edge(15,177,weight=1)
G6.add_edge(15,178,weight=1)
G6.add_edge(15,179,weight=1)
G6.add_edge(15,18,weight=1)
G6.add_edge(15,19,weight=1)
G6.add_edge(15,20,weight=1)
G6.add_edge(15,21,weight=1)
G6.add_edge(15,23,weight=1)
G6.add_edge(15,24,weight=1)
G6.add_edge(15,67,weight=1)
G6.add_edge(15,74,weight=1)
G6.add_edge(15,76,weight=1)
G6.add_edge(15,93,weight=1)
G6.add_edge(16,100,weight=1)
G6.add_edge(16,110,weight=1)
G6.add_edge(16,111,weight=1)
G6.add_edge(16,114,weight=1)
G6.add_edge(16,17,weight=1)
G6.add_edge(16,18,weight=1)
G6.add_edge(16,21,weight=1)
G6.add_edge(16,23,weight=1)
G6.add_edge(16,24,weight=1)
G6.add_edge(16,28,weight=1)
G6.add_edge(16,80,weight=1)
G6.add_edge(16,81,weight=1)
G6.add_edge(16,88,weight=1)
G6.add_edge(16,91,weight=1)
G6.add_edge(16,92,weight=1)
G6.add_edge(16,93,weight=1)
G6.add_edge(16,94,weight=1)
G6.add_edge(16,98,weight=1)
G6.add_edge(17,145,weight=1)
G6.add_edge(17,146,weight=1)
G6.add_edge(17,147,weight=1)
G6.add_edge(17,148,weight=1)
G6.add_edge(17,149,weight=1)
G6.add_edge(17,150,weight=1)
G6.add_edge(17,18,weight=1)
G6.add_edge(17,21,weight=1)
G6.add_edge(17,22,weight=1)
G6.add_edge(17,23,weight=1)
G6.add_edge(17,24,weight=1)
G6.add_edge(18,101,weight=1)
G6.add_edge(18,102,weight=1)
G6.add_edge(18,107,weight=1)
G6.add_edge(18,111,weight=1)
G6.add_edge(18,112,weight=1)
G6.add_edge(18,114,weight=1)
G6.add_edge(18,116,weight=1)
G6.add_edge(18,117,weight=1)
G6.add_edge(18,118,weight=1)
G6.add_edge(18,121,weight=1)
G6.add_edge(18,125,weight=1)
G6.add_edge(18,128,weight=1)
G6.add_edge(18,133,weight=1)
G6.add_edge(18,137,weight=1)
G6.add_edge(18,149,weight=1)
G6.add_edge(18,150,weight=1)
G6.add_edge(18,152,weight=1)
G6.add_edge(18,158,weight=1)
G6.add_edge(18,159,weight=1)
G6.add_edge(18,160,weight=1)
G6.add_edge(18,164,weight=1)
G6.add_edge(18,165,weight=1)
G6.add_edge(18,166,weight=1)
G6.add_edge(18,167,weight=1)
G6.add_edge(18,168,weight=1)
G6.add_edge(18,169,weight=1)
G6.add_edge(18,170,weight=1)
G6.add_edge(18,171,weight=1)
G6.add_edge(18,172,weight=1)
G6.add_edge(18,173,weight=1)
G6.add_edge(18,174,weight=1)
G6.add_edge(18,19,weight=1)
G6.add_edge(18,20,weight=1)
G6.add_edge(18,21,weight=1)
G6.add_edge(18,23,weight=1)
G6.add_edge(18,24,weight=1)
G6.add_edge(18,67,weight=1)
G6.add_edge(18,74,weight=1)
G6.add_edge(18,76,weight=1)
G6.add_edge(18,80,weight=1)
G6.add_edge(18,89,weight=1)
G6.add_edge(18,90,weight=1)
G6.add_edge(18,93,weight=1)
G6.add_edge(18,95,weight=1)
G6.add_edge(18,96,weight=1)
G6.add_edge(18,97,weight=1)
G6.add_edge(19,100,weight=1)
G6.add_edge(19,101,weight=1)
G6.add_edge(19,109,weight=1)
G6.add_edge(19,111,weight=1)
G6.add_edge(19,112,weight=1)
G6.add_edge(19,114,weight=1)
G6.add_edge(19,121,weight=1)
G6.add_edge(19,125,weight=1)
G6.add_edge(19,128,weight=1)
G6.add_edge(19,133,weight=1)
G6.add_edge(19,135,weight=1)
G6.add_edge(19,137,weight=1)
G6.add_edge(19,139,weight=1)
G6.add_edge(19,149,weight=1)
G6.add_edge(19,150,weight=1)
G6.add_edge(19,152,weight=1)
G6.add_edge(19,153,weight=1)
G6.add_edge(19,154,weight=1)
G6.add_edge(19,155,weight=1)
G6.add_edge(19,158,weight=1)
G6.add_edge(19,159,weight=1)
G6.add_edge(19,160,weight=1)
G6.add_edge(19,164,weight=1)
G6.add_edge(19,165,weight=1)
G6.add_edge(19,166,weight=1)
G6.add_edge(19,167,weight=1)
G6.add_edge(19,168,weight=1)
G6.add_edge(19,169,weight=1)
G6.add_edge(19,170,weight=1)
G6.add_edge(19,171,weight=1)
G6.add_edge(19,172,weight=1)
G6.add_edge(19,173,weight=1)
G6.add_edge(19,174,weight=1)
G6.add_edge(19,20,weight=1)
G6.add_edge(19,21,weight=1)
G6.add_edge(19,23,weight=1)
G6.add_edge(19,24,weight=1)
G6.add_edge(19,54,weight=1)
G6.add_edge(19,67,weight=1)
G6.add_edge(19,74,weight=1)
G6.add_edge(19,75,weight=1)
G6.add_edge(19,76,weight=1)
G6.add_edge(19,86,weight=1)
G6.add_edge(19,93,weight=1)
G6.add_edge(20,100,weight=1)
G6.add_edge(20,101,weight=1)
G6.add_edge(20,102,weight=1)
G6.add_edge(20,106,weight=1)
G6.add_edge(20,107,weight=1)
G6.add_edge(20,109,weight=1)
G6.add_edge(20,111,weight=1)
G6.add_edge(20,112,weight=1)
G6.add_edge(20,114,weight=1)
G6.add_edge(20,116,weight=1)
G6.add_edge(20,117,weight=1)
G6.add_edge(20,118,weight=1)
G6.add_edge(20,120,weight=1)
G6.add_edge(20,121,weight=1)
G6.add_edge(20,122,weight=1)
G6.add_edge(20,123,weight=1)
G6.add_edge(20,124,weight=1)
G6.add_edge(20,125,weight=1)
G6.add_edge(20,128,weight=1)
G6.add_edge(20,133,weight=1)
G6.add_edge(20,137,weight=1)
G6.add_edge(20,139,weight=1)
G6.add_edge(20,149,weight=1)
G6.add_edge(20,150,weight=1)
G6.add_edge(20,152,weight=1)
G6.add_edge(20,153,weight=1)
G6.add_edge(20,154,weight=1)
G6.add_edge(20,155,weight=1)
G6.add_edge(20,158,weight=1)
G6.add_edge(20,159,weight=1)
G6.add_edge(20,160,weight=1)
G6.add_edge(20,164,weight=1)
G6.add_edge(20,165,weight=1)
G6.add_edge(20,166,weight=1)
G6.add_edge(20,167,weight=1)
G6.add_edge(20,168,weight=1)
G6.add_edge(20,169,weight=1)
G6.add_edge(20,170,weight=1)
G6.add_edge(20,171,weight=1)
G6.add_edge(20,172,weight=1)
G6.add_edge(20,173,weight=1)
G6.add_edge(20,174,weight=1)
G6.add_edge(20,177,weight=1)
G6.add_edge(20,178,weight=1)
G6.add_edge(20,179,weight=1)
G6.add_edge(20,21,weight=1)
G6.add_edge(20,23,weight=1)
G6.add_edge(20,24,weight=1)
G6.add_edge(20,54,weight=1)
G6.add_edge(20,67,weight=1)
G6.add_edge(20,74,weight=1)
G6.add_edge(20,75,weight=1)
G6.add_edge(20,76,weight=1)
G6.add_edge(20,80,weight=1)
G6.add_edge(20,86,weight=1)
G6.add_edge(20,89,weight=1)
G6.add_edge(20,90,weight=1)
G6.add_edge(20,93,weight=1)
G6.add_edge(20,95,weight=1)
G6.add_edge(20,96,weight=1)
G6.add_edge(20,97,weight=1)
G6.add_edge(21,101,weight=1)
G6.add_edge(21,112,weight=1)
G6.add_edge(21,121,weight=1)
G6.add_edge(21,128,weight=1)
G6.add_edge(21,133,weight=1)
G6.add_edge(21,137,weight=1)
G6.add_edge(21,149,weight=1)
G6.add_edge(21,150,weight=1)
G6.add_edge(21,152,weight=1)
G6.add_edge(21,164,weight=1)
G6.add_edge(21,165,weight=1)
G6.add_edge(21,166,weight=1)
G6.add_edge(21,167,weight=1)
G6.add_edge(21,168,weight=1)
G6.add_edge(21,169,weight=1)
G6.add_edge(21,170,weight=1)
G6.add_edge(21,171,weight=1)
G6.add_edge(21,172,weight=1)
G6.add_edge(21,173,weight=1)
G6.add_edge(21,174,weight=1)
G6.add_edge(21,177,weight=1)
G6.add_edge(21,178,weight=1)
G6.add_edge(21,179,weight=1)
G6.add_edge(21,23,weight=1)
G6.add_edge(21,24,weight=1)
G6.add_edge(22,119,weight=1)
G6.add_edge(22,133,weight=1)
G6.add_edge(22,176,weight=1)
G6.add_edge(22,193,weight=1)
G6.add_edge(22,88,weight=1)
G6.add_edge(22,93,weight=1)
G6.add_edge(23,101,weight=1)
G6.add_edge(23,102,weight=1)
G6.add_edge(23,107,weight=1)
G6.add_edge(23,109,weight=1)
G6.add_edge(23,112,weight=1)
G6.add_edge(23,116,weight=1)
G6.add_edge(23,117,weight=1)
G6.add_edge(23,118,weight=1)
G6.add_edge(23,119,weight=1)
G6.add_edge(23,121,weight=1)
G6.add_edge(23,122,weight=1)
G6.add_edge(23,123,weight=1)
G6.add_edge(23,124,weight=1)
G6.add_edge(23,126,weight=1)
G6.add_edge(23,127,weight=1)
G6.add_edge(23,128,weight=1)
G6.add_edge(23,130,weight=1)
G6.add_edge(23,131,weight=1)
G6.add_edge(23,132,weight=1)
G6.add_edge(23,133,weight=1)
G6.add_edge(23,134,weight=1)
G6.add_edge(23,137,weight=1)
G6.add_edge(23,148,weight=1)
G6.add_edge(23,149,weight=1)
G6.add_edge(23,150,weight=1)
G6.add_edge(23,152,weight=1)
G6.add_edge(23,153,weight=1)
G6.add_edge(23,157,weight=1)
G6.add_edge(23,164,weight=1)
G6.add_edge(23,165,weight=1)
G6.add_edge(23,166,weight=1)
G6.add_edge(23,167,weight=1)
G6.add_edge(23,168,weight=1)
G6.add_edge(23,169,weight=1)
G6.add_edge(23,170,weight=1)
G6.add_edge(23,171,weight=1)
G6.add_edge(23,172,weight=1)
G6.add_edge(23,173,weight=1)
G6.add_edge(23,174,weight=1)
G6.add_edge(23,177,weight=1)
G6.add_edge(23,178,weight=1)
G6.add_edge(23,179,weight=1)
G6.add_edge(23,24,weight=1)
G6.add_edge(23,27,weight=1)
G6.add_edge(23,31,weight=1)
G6.add_edge(23,67,weight=1)
G6.add_edge(23,74,weight=1)
G6.add_edge(23,80,weight=1)
G6.add_edge(23,89,weight=1)
G6.add_edge(23,90,weight=1)
G6.add_edge(23,91,weight=1)
G6.add_edge(23,93,weight=1)
G6.add_edge(23,95,weight=1)
G6.add_edge(23,96,weight=1)
G6.add_edge(23,97,weight=1)
G6.add_edge(24,101,weight=1)
G6.add_edge(24,102,weight=1)
G6.add_edge(24,103,weight=1)
G6.add_edge(24,106,weight=1)
G6.add_edge(24,107,weight=1)
G6.add_edge(24,109,weight=1)
G6.add_edge(24,116,weight=1)
G6.add_edge(24,117,weight=1)
G6.add_edge(24,118,weight=1)
G6.add_edge(24,120,weight=1)
G6.add_edge(24,121,weight=1)
G6.add_edge(24,124,weight=1)
G6.add_edge(24,125,weight=1)
G6.add_edge(24,133,weight=1)
G6.add_edge(24,168,weight=1)
G6.add_edge(24,177,weight=1)
G6.add_edge(24,178,weight=1)
G6.add_edge(24,179,weight=1)
G6.add_edge(24,67,weight=1)
G6.add_edge(24,80,weight=1)
G6.add_edge(24,89,weight=1)
G6.add_edge(24,90,weight=1)
G6.add_edge(24,95,weight=1)
G6.add_edge(24,96,weight=1)
G6.add_edge(24,97,weight=1)
G6.add_edge(25,26,weight=1)
G6.add_edge(25,27,weight=1)
G6.add_edge(25,28,weight=1)
G6.add_edge(26,100,weight=1)
G6.add_edge(26,103,weight=1)
G6.add_edge(26,105,weight=1)
G6.add_edge(26,106,weight=1)
G6.add_edge(26,108,weight=1)
G6.add_edge(26,109,weight=1)
G6.add_edge(26,110,weight=1)
G6.add_edge(26,111,weight=1)
G6.add_edge(26,115,weight=1)
G6.add_edge(26,117,weight=1)
G6.add_edge(26,119,weight=1)
G6.add_edge(26,144,weight=1)
G6.add_edge(26,27,weight=1)
G6.add_edge(26,28,weight=1)
G6.add_edge(26,62,weight=1)
G6.add_edge(26,74,weight=1)
G6.add_edge(26,80,weight=1)
G6.add_edge(26,81,weight=1)
G6.add_edge(26,87,weight=1)
G6.add_edge(26,89,weight=1)
G6.add_edge(26,90,weight=1)
G6.add_edge(26,94,weight=1)
G6.add_edge(26,98,weight=1)
G6.add_edge(27,102,weight=1)
G6.add_edge(27,104,weight=1)
G6.add_edge(27,105,weight=1)
G6.add_edge(27,107,weight=1)
G6.add_edge(27,109,weight=1)
G6.add_edge(27,110,weight=1)
G6.add_edge(27,111,weight=1)
G6.add_edge(27,112,weight=1)
G6.add_edge(27,113,weight=1)
G6.add_edge(27,114,weight=1)
G6.add_edge(27,116,weight=1)
G6.add_edge(27,117,weight=1)
G6.add_edge(27,118,weight=1)
G6.add_edge(27,119,weight=1)
G6.add_edge(27,125,weight=1)
G6.add_edge(27,138,weight=1)
G6.add_edge(27,139,weight=1)
G6.add_edge(27,140,weight=1)
G6.add_edge(27,144,weight=1)
G6.add_edge(27,28,weight=1)
G6.add_edge(27,35,weight=1)
G6.add_edge(27,48,weight=1)
G6.add_edge(27,54,weight=1)
G6.add_edge(27,55,weight=1)
G6.add_edge(27,62,weight=1)
G6.add_edge(27,70,weight=1)
G6.add_edge(27,74,weight=1)
G6.add_edge(27,75,weight=1)
G6.add_edge(27,76,weight=1)
G6.add_edge(27,83,weight=1)
G6.add_edge(27,84,weight=1)
G6.add_edge(27,85,weight=1)
G6.add_edge(27,86,weight=1)
G6.add_edge(27,87,weight=1)
G6.add_edge(27,88,weight=1)
G6.add_edge(27,89,weight=1)
G6.add_edge(27,90,weight=1)
G6.add_edge(27,93,weight=1)
G6.add_edge(27,94,weight=1)
G6.add_edge(27,95,weight=1)
G6.add_edge(27,96,weight=1)
G6.add_edge(27,97,weight=1)
G6.add_edge(28,103,weight=1)
G6.add_edge(28,105,weight=1)
G6.add_edge(28,108,weight=1)
G6.add_edge(28,109,weight=1)
G6.add_edge(28,110,weight=1)
G6.add_edge(28,111,weight=1)
G6.add_edge(28,112,weight=1)
G6.add_edge(28,114,weight=1)
G6.add_edge(28,115,weight=1)
G6.add_edge(28,117,weight=1)
G6.add_edge(28,119,weight=1)
G6.add_edge(28,124,weight=1)
G6.add_edge(28,144,weight=1)
G6.add_edge(28,54,weight=1)
G6.add_edge(28,62,weight=1)
G6.add_edge(28,67,weight=1)
G6.add_edge(28,70,weight=1)
G6.add_edge(28,74,weight=1)
G6.add_edge(28,75,weight=1)
G6.add_edge(28,80,weight=1)
G6.add_edge(28,83,weight=1)
G6.add_edge(28,84,weight=1)
G6.add_edge(28,85,weight=1)
G6.add_edge(28,86,weight=1)
G6.add_edge(28,88,weight=1)
G6.add_edge(28,89,weight=1)
G6.add_edge(28,90,weight=1)
G6.add_edge(28,91,weight=1)
G6.add_edge(28,92,weight=1)
G6.add_edge(28,93,weight=1)
G6.add_edge(28,94,weight=1)
G6.add_edge(28,98,weight=1)
G6.add_edge(29,30,weight=1)
G6.add_edge(29,31,weight=1)
G6.add_edge(29,32,weight=1)
G6.add_edge(29,33,weight=1)
G6.add_edge(30,143,weight=1)
G6.add_edge(30,31,weight=1)
G6.add_edge(30,32,weight=1)
G6.add_edge(30,33,weight=1)
G6.add_edge(30,78,weight=1)
G6.add_edge(30,79,weight=1)
G6.add_edge(31,101,weight=1)
G6.add_edge(31,122,weight=1)
G6.add_edge(31,123,weight=1)
G6.add_edge(31,126,weight=1)
G6.add_edge(31,131,weight=1)
G6.add_edge(31,132,weight=1)
G6.add_edge(31,134,weight=1)
G6.add_edge(31,141,weight=1)
G6.add_edge(31,142,weight=1)
G6.add_edge(31,143,weight=1)
G6.add_edge(31,161,weight=1)
G6.add_edge(31,180,weight=1)
G6.add_edge(31,183,weight=1)
G6.add_edge(31,184,weight=1)
G6.add_edge(31,32,weight=1)
G6.add_edge(31,33,weight=1)
G6.add_edge(31,34,weight=1)
G6.add_edge(31,35,weight=1)
G6.add_edge(31,36,weight=1)
G6.add_edge(31,37,weight=1)
G6.add_edge(31,42,weight=1)
G6.add_edge(31,50,weight=1)
G6.add_edge(31,51,weight=1)
G6.add_edge(31,52,weight=1)
G6.add_edge(31,53,weight=1)
G6.add_edge(31,58,weight=1)
G6.add_edge(31,60,weight=1)
G6.add_edge(31,61,weight=1)
G6.add_edge(31,62,weight=1)
G6.add_edge(31,64,weight=1)
G6.add_edge(31,65,weight=1)
G6.add_edge(31,66,weight=1)
G6.add_edge(31,68,weight=1)
G6.add_edge(31,69,weight=1)
G6.add_edge(31,70,weight=1)
G6.add_edge(31,72,weight=1)
G6.add_edge(31,91,weight=1)
G6.add_edge(31,97,weight=1)
G6.add_edge(32,139,weight=1)
G6.add_edge(32,154,weight=1)
G6.add_edge(32,156,weight=1)
G6.add_edge(32,161,weight=1)
G6.add_edge(32,181,weight=1)
G6.add_edge(32,190,weight=1)
G6.add_edge(32,196,weight=1)
G6.add_edge(32,33,weight=1)
G6.add_edge(32,34,weight=1)
G6.add_edge(32,35,weight=1)
G6.add_edge(32,36,weight=1)
G6.add_edge(32,37,weight=1)
G6.add_edge(32,38,weight=1)
G6.add_edge(32,39,weight=1)
G6.add_edge(32,40,weight=1)
G6.add_edge(32,41,weight=1)
G6.add_edge(32,42,weight=1)
G6.add_edge(32,43,weight=1)
G6.add_edge(32,44,weight=1)
G6.add_edge(32,45,weight=1)
G6.add_edge(32,46,weight=1)
G6.add_edge(32,47,weight=1)
G6.add_edge(32,50,weight=1)
G6.add_edge(32,51,weight=1)
G6.add_edge(32,52,weight=1)
G6.add_edge(32,53,weight=1)
G6.add_edge(32,56,weight=1)
G6.add_edge(32,57,weight=1)
G6.add_edge(32,59,weight=1)
G6.add_edge(32,60,weight=1)
G6.add_edge(32,61,weight=1)
G6.add_edge(32,62,weight=1)
G6.add_edge(32,63,weight=1)
G6.add_edge(32,65,weight=1)
G6.add_edge(32,66,weight=1)
G6.add_edge(32,67,weight=1)
G6.add_edge(32,68,weight=1)
G6.add_edge(32,70,weight=1)
G6.add_edge(32,71,weight=1)
G6.add_edge(32,72,weight=1)
G6.add_edge(32,73,weight=1)
G6.add_edge(32,90,weight=1)
G6.add_edge(33,139,weight=1)
G6.add_edge(33,154,weight=1)
G6.add_edge(33,156,weight=1)
G6.add_edge(33,34,weight=1)
G6.add_edge(33,35,weight=1)
G6.add_edge(33,36,weight=1)
G6.add_edge(33,37,weight=1)
G6.add_edge(33,38,weight=1)
G6.add_edge(33,40,weight=1)
G6.add_edge(33,41,weight=1)
G6.add_edge(33,42,weight=1)
G6.add_edge(33,43,weight=1)
G6.add_edge(33,44,weight=1)
G6.add_edge(33,45,weight=1)
G6.add_edge(33,46,weight=1)
G6.add_edge(33,47,weight=1)
G6.add_edge(33,50,weight=1)
G6.add_edge(33,51,weight=1)
G6.add_edge(33,52,weight=1)
G6.add_edge(33,53,weight=1)
G6.add_edge(33,56,weight=1)
G6.add_edge(33,57,weight=1)
G6.add_edge(33,58,weight=1)
G6.add_edge(33,59,weight=1)
G6.add_edge(33,60,weight=1)
G6.add_edge(33,61,weight=1)
G6.add_edge(33,62,weight=1)
G6.add_edge(33,63,weight=1)
G6.add_edge(33,65,weight=1)
G6.add_edge(33,66,weight=1)
G6.add_edge(33,67,weight=1)
G6.add_edge(33,68,weight=1)
G6.add_edge(33,69,weight=1)
G6.add_edge(33,70,weight=1)
G6.add_edge(33,72,weight=1)
G6.add_edge(33,73,weight=1)
G6.add_edge(34,35,weight=1)
G6.add_edge(34,36,weight=1)
G6.add_edge(34,37,weight=1)
G6.add_edge(34,38,weight=1)
G6.add_edge(34,39,weight=1)
G6.add_edge(34,40,weight=1)
G6.add_edge(34,41,weight=1)
G6.add_edge(34,42,weight=1)
G6.add_edge(34,43,weight=1)
G6.add_edge(34,44,weight=1)
G6.add_edge(34,45,weight=1)
G6.add_edge(34,46,weight=1)
G6.add_edge(34,47,weight=1)
G6.add_edge(35,36,weight=1)
G6.add_edge(35,37,weight=1)
G6.add_edge(35,38,weight=1)
G6.add_edge(35,39,weight=1)
G6.add_edge(35,40,weight=1)
G6.add_edge(35,41,weight=1)
G6.add_edge(35,42,weight=1)
G6.add_edge(35,43,weight=1)
G6.add_edge(35,44,weight=1)
G6.add_edge(35,45,weight=1)
G6.add_edge(35,46,weight=1)
G6.add_edge(35,47,weight=1)
G6.add_edge(35,48,weight=1)
G6.add_edge(35,49,weight=1)
G6.add_edge(35,50,weight=1)
G6.add_edge(35,51,weight=1)
G6.add_edge(35,52,weight=1)
G6.add_edge(35,53,weight=1)
G6.add_edge(35,54,weight=1)
G6.add_edge(35,55,weight=1)
G6.add_edge(35,56,weight=1)
G6.add_edge(35,57,weight=1)
G6.add_edge(35,58,weight=1)
G6.add_edge(35,59,weight=1)
G6.add_edge(35,60,weight=1)
G6.add_edge(35,61,weight=1)
G6.add_edge(35,62,weight=1)
G6.add_edge(35,63,weight=1)
G6.add_edge(35,64,weight=1)
G6.add_edge(35,65,weight=1)
G6.add_edge(35,66,weight=1)
G6.add_edge(35,67,weight=1)
G6.add_edge(35,68,weight=1)
G6.add_edge(35,69,weight=1)
G6.add_edge(35,70,weight=1)
G6.add_edge(35,71,weight=1)
G6.add_edge(35,72,weight=1)
G6.add_edge(35,73,weight=1)
G6.add_edge(35,74,weight=1)
G6.add_edge(35,75,weight=1)
G6.add_edge(35,76,weight=1)
G6.add_edge(36,141,weight=1)
G6.add_edge(36,142,weight=1)
G6.add_edge(36,143,weight=1)
G6.add_edge(36,37,weight=1)
G6.add_edge(36,39,weight=1)
G6.add_edge(36,40,weight=1)
G6.add_edge(36,42,weight=1)
G6.add_edge(36,43,weight=1)
G6.add_edge(36,44,weight=1)
G6.add_edge(36,45,weight=1)
G6.add_edge(36,46,weight=1)
G6.add_edge(36,47,weight=1)
G6.add_edge(36,51,weight=1)
G6.add_edge(36,56,weight=1)
G6.add_edge(36,60,weight=1)
G6.add_edge(36,66,weight=1)
G6.add_edge(36,68,weight=1)
G6.add_edge(36,70,weight=1)
G6.add_edge(36,71,weight=1)
G6.add_edge(36,73,weight=1)
G6.add_edge(37,114,weight=1)
G6.add_edge(37,156,weight=1)
G6.add_edge(37,161,weight=1)
G6.add_edge(37,38,weight=1)
G6.add_edge(37,39,weight=1)
G6.add_edge(37,40,weight=1)
G6.add_edge(37,41,weight=1)
G6.add_edge(37,42,weight=1)
G6.add_edge(37,43,weight=1)
G6.add_edge(37,44,weight=1)
G6.add_edge(37,45,weight=1)
G6.add_edge(37,46,weight=1)
G6.add_edge(37,47,weight=1)
G6.add_edge(37,51,weight=1)
G6.add_edge(37,52,weight=1)
G6.add_edge(37,56,weight=1)
G6.add_edge(37,60,weight=1)
G6.add_edge(37,61,weight=1)
G6.add_edge(37,62,weight=1)
G6.add_edge(37,64,weight=1)
G6.add_edge(37,66,weight=1)
G6.add_edge(37,68,weight=1)
G6.add_edge(37,70,weight=1)
G6.add_edge(37,71,weight=1)
G6.add_edge(37,73,weight=1)
G6.add_edge(38,139,weight=1)
G6.add_edge(38,154,weight=1)
G6.add_edge(38,156,weight=1)
G6.add_edge(38,185,weight=1)
G6.add_edge(38,186,weight=1)
G6.add_edge(38,39,weight=1)
G6.add_edge(38,40,weight=1)
G6.add_edge(38,41,weight=1)
G6.add_edge(38,42,weight=1)
G6.add_edge(38,44,weight=1)
G6.add_edge(38,45,weight=1)
G6.add_edge(38,51,weight=1)
G6.add_edge(38,52,weight=1)
G6.add_edge(38,56,weight=1)
G6.add_edge(38,60,weight=1)
G6.add_edge(38,61,weight=1)
G6.add_edge(38,62,weight=1)
G6.add_edge(38,63,weight=1)
G6.add_edge(38,64,weight=1)
G6.add_edge(38,66,weight=1)
G6.add_edge(38,67,weight=1)
G6.add_edge(38,68,weight=1)
G6.add_edge(38,70,weight=1)
G6.add_edge(38,73,weight=1)
G6.add_edge(39,40,weight=1)
G6.add_edge(39,42,weight=1)
G6.add_edge(39,43,weight=1)
G6.add_edge(39,44,weight=1)
G6.add_edge(39,47,weight=1)
G6.add_edge(39,51,weight=1)
G6.add_edge(39,56,weight=1)
G6.add_edge(39,60,weight=1)
G6.add_edge(39,66,weight=1)
G6.add_edge(39,68,weight=1)
G6.add_edge(39,70,weight=1)
G6.add_edge(39,71,weight=1)
G6.add_edge(39,73,weight=1)
G6.add_edge(40,41,weight=1)
G6.add_edge(40,42,weight=1)
G6.add_edge(40,43,weight=1)
G6.add_edge(40,44,weight=1)
G6.add_edge(40,45,weight=1)
G6.add_edge(40,46,weight=1)
G6.add_edge(40,47,weight=1)
G6.add_edge(40,51,weight=1)
G6.add_edge(40,60,weight=1)
G6.add_edge(40,66,weight=1)
G6.add_edge(40,68,weight=1)
G6.add_edge(41,114,weight=1)
G6.add_edge(41,196,weight=1)
G6.add_edge(41,42,weight=1)
G6.add_edge(41,44,weight=1)
G6.add_edge(41,45,weight=1)
G6.add_edge(41,51,weight=1)
G6.add_edge(41,52,weight=1)
G6.add_edge(41,60,weight=1)
G6.add_edge(41,61,weight=1)
G6.add_edge(41,62,weight=1)
G6.add_edge(41,63,weight=1)
G6.add_edge(41,64,weight=1)
G6.add_edge(41,66,weight=1)
G6.add_edge(41,67,weight=1)
G6.add_edge(41,70,weight=1)
G6.add_edge(41,71,weight=1)
G6.add_edge(42,125,weight=1)
G6.add_edge(42,139,weight=1)
G6.add_edge(42,154,weight=1)
G6.add_edge(42,156,weight=1)
G6.add_edge(42,43,weight=1)
G6.add_edge(42,44,weight=1)
G6.add_edge(42,45,weight=1)
G6.add_edge(42,46,weight=1)
G6.add_edge(42,47,weight=1)
G6.add_edge(42,50,weight=1)
G6.add_edge(42,51,weight=1)
G6.add_edge(42,52,weight=1)
G6.add_edge(42,53,weight=1)
G6.add_edge(42,56,weight=1)
G6.add_edge(42,57,weight=1)
G6.add_edge(42,58,weight=1)
G6.add_edge(42,59,weight=1)
G6.add_edge(42,60,weight=1)
G6.add_edge(42,61,weight=1)
G6.add_edge(42,62,weight=1)
G6.add_edge(42,63,weight=1)
G6.add_edge(42,65,weight=1)
G6.add_edge(42,66,weight=1)
G6.add_edge(42,67,weight=1)
G6.add_edge(42,68,weight=1)
G6.add_edge(42,69,weight=1)
G6.add_edge(42,70,weight=1)
G6.add_edge(42,73,weight=1)
G6.add_edge(43,181,weight=1)
G6.add_edge(43,44,weight=1)
G6.add_edge(43,45,weight=1)
G6.add_edge(43,46,weight=1)
G6.add_edge(43,47,weight=1)
G6.add_edge(43,51,weight=1)
G6.add_edge(43,56,weight=1)
G6.add_edge(43,60,weight=1)
G6.add_edge(43,66,weight=1)
G6.add_edge(43,68,weight=1)
G6.add_edge(43,70,weight=1)
G6.add_edge(43,71,weight=1)
G6.add_edge(43,73,weight=1)
G6.add_edge(44,45,weight=1)
G6.add_edge(44,46,weight=1)
G6.add_edge(44,47,weight=1)
G6.add_edge(44,52,weight=1)
G6.add_edge(44,60,weight=1)
G6.add_edge(44,61,weight=1)
G6.add_edge(44,62,weight=1)
G6.add_edge(44,63,weight=1)
G6.add_edge(44,66,weight=1)
G6.add_edge(44,68,weight=1)
G6.add_edge(44,70,weight=1)
G6.add_edge(45,156,weight=1)
G6.add_edge(45,46,weight=1)
G6.add_edge(45,47,weight=1)
G6.add_edge(45,51,weight=1)
G6.add_edge(45,60,weight=1)
G6.add_edge(45,61,weight=1)
G6.add_edge(45,62,weight=1)
G6.add_edge(45,66,weight=1)
G6.add_edge(45,68,weight=1)
G6.add_edge(45,70,weight=1)
G6.add_edge(45,72,weight=1)
G6.add_edge(45,73,weight=1)
G6.add_edge(46,117,weight=1)
G6.add_edge(46,156,weight=1)
G6.add_edge(46,161,weight=1)
G6.add_edge(46,47,weight=1)
G6.add_edge(46,51,weight=1)
G6.add_edge(46,52,weight=1)
G6.add_edge(46,60,weight=1)
G6.add_edge(46,61,weight=1)
G6.add_edge(46,62,weight=1)
G6.add_edge(46,64,weight=1)
G6.add_edge(46,66,weight=1)
G6.add_edge(46,68,weight=1)
G6.add_edge(46,70,weight=1)
G6.add_edge(47,126,weight=1)
G6.add_edge(47,127,weight=1)
G6.add_edge(47,128,weight=1)
G6.add_edge(47,129,weight=1)
G6.add_edge(47,130,weight=1)
G6.add_edge(47,133,weight=1)
G6.add_edge(47,135,weight=1)
G6.add_edge(47,136,weight=1)
G6.add_edge(47,137,weight=1)
G6.add_edge(47,51,weight=1)
G6.add_edge(47,56,weight=1)
G6.add_edge(47,60,weight=1)
G6.add_edge(47,66,weight=1)
G6.add_edge(47,68,weight=1)
G6.add_edge(47,70,weight=1)
G6.add_edge(47,71,weight=1)
G6.add_edge(47,73,weight=1)
G6.add_edge(47,82,weight=1)
G6.add_edge(48,104,weight=1)
G6.add_edge(48,105,weight=1)
G6.add_edge(48,115,weight=1)
G6.add_edge(48,117,weight=1)
G6.add_edge(48,119,weight=1)
G6.add_edge(48,138,weight=1)
G6.add_edge(48,139,weight=1)
G6.add_edge(48,140,weight=1)
G6.add_edge(48,157,weight=1)
G6.add_edge(48,54,weight=1)
G6.add_edge(48,55,weight=1)
G6.add_edge(48,67,weight=1)
G6.add_edge(48,74,weight=1)
G6.add_edge(48,75,weight=1)
G6.add_edge(48,76,weight=1)
G6.add_edge(48,80,weight=1)
G6.add_edge(48,84,weight=1)
G6.add_edge(48,85,weight=1)
G6.add_edge(48,86,weight=1)
G6.add_edge(48,87,weight=1)
G6.add_edge(48,90,weight=1)
G6.add_edge(50,52,weight=1)
G6.add_edge(50,53,weight=1)
G6.add_edge(50,58,weight=1)
G6.add_edge(50,62,weight=1)
G6.add_edge(50,65,weight=1)
G6.add_edge(50,66,weight=1)
G6.add_edge(50,69,weight=1)
G6.add_edge(50,70,weight=1)
G6.add_edge(51,156,weight=1)
G6.add_edge(51,161,weight=1)
G6.add_edge(51,52,weight=1)
G6.add_edge(51,56,weight=1)
G6.add_edge(51,60,weight=1)
G6.add_edge(51,61,weight=1)
G6.add_edge(51,62,weight=1)
G6.add_edge(51,64,weight=1)
G6.add_edge(51,66,weight=1)
G6.add_edge(51,68,weight=1)
G6.add_edge(51,70,weight=1)
G6.add_edge(51,71,weight=1)
G6.add_edge(51,73,weight=1)
G6.add_edge(52,139,weight=1)
G6.add_edge(52,154,weight=1)
G6.add_edge(52,156,weight=1)
G6.add_edge(52,161,weight=1)
G6.add_edge(52,53,weight=1)
G6.add_edge(52,56,weight=1)
G6.add_edge(52,58,weight=1)
G6.add_edge(52,60,weight=1)
G6.add_edge(52,61,weight=1)
G6.add_edge(52,62,weight=1)
G6.add_edge(52,63,weight=1)
G6.add_edge(52,64,weight=1)
G6.add_edge(52,65,weight=1)
G6.add_edge(52,66,weight=1)
G6.add_edge(52,67,weight=1)
G6.add_edge(52,68,weight=1)
G6.add_edge(52,69,weight=1)
G6.add_edge(52,70,weight=1)
G6.add_edge(52,73,weight=1)
G6.add_edge(53,125,weight=1)
G6.add_edge(53,183,weight=1)
G6.add_edge(53,184,weight=1)
G6.add_edge(53,57,weight=1)
G6.add_edge(53,58,weight=1)
G6.add_edge(53,59,weight=1)
G6.add_edge(53,62,weight=1)
G6.add_edge(53,65,weight=1)
G6.add_edge(53,66,weight=1)
G6.add_edge(53,67,weight=1)
G6.add_edge(53,69,weight=1)
G6.add_edge(53,70,weight=1)
G6.add_edge(54,100,weight=1)
G6.add_edge(54,104,weight=1)
G6.add_edge(54,109,weight=1)
G6.add_edge(54,110,weight=1)
G6.add_edge(54,111,weight=1)
G6.add_edge(54,112,weight=1)
G6.add_edge(54,114,weight=1)
G6.add_edge(54,119,weight=1)
G6.add_edge(54,138,weight=1)
G6.add_edge(54,139,weight=1)
G6.add_edge(54,140,weight=1)
G6.add_edge(54,153,weight=1)
G6.add_edge(54,154,weight=1)
G6.add_edge(54,155,weight=1)
G6.add_edge(54,191,weight=1)
G6.add_edge(54,55,weight=1)
G6.add_edge(54,70,weight=1)
G6.add_edge(54,74,weight=1)
G6.add_edge(54,75,weight=1)
G6.add_edge(54,76,weight=1)
G6.add_edge(54,80,weight=1)
G6.add_edge(54,81,weight=1)
G6.add_edge(54,82,weight=1)
G6.add_edge(54,83,weight=1)
G6.add_edge(54,84,weight=1)
G6.add_edge(54,85,weight=1)
G6.add_edge(54,86,weight=1)
G6.add_edge(54,87,weight=1)
G6.add_edge(54,88,weight=1)
G6.add_edge(54,91,weight=1)
G6.add_edge(54,93,weight=1)
G6.add_edge(54,97,weight=1)
G6.add_edge(54,98,weight=1)
G6.add_edge(54,99,weight=1)
G6.add_edge(55,100,weight=1)
G6.add_edge(55,104,weight=1)
G6.add_edge(55,105,weight=1)
G6.add_edge(55,109,weight=1)
G6.add_edge(55,110,weight=1)
G6.add_edge(55,111,weight=1)
G6.add_edge(55,115,weight=1)
G6.add_edge(55,117,weight=1)
G6.add_edge(55,119,weight=1)
G6.add_edge(55,120,weight=1)
G6.add_edge(55,123,weight=1)
G6.add_edge(55,124,weight=1)
G6.add_edge(55,138,weight=1)
G6.add_edge(55,139,weight=1)
G6.add_edge(55,140,weight=1)
G6.add_edge(55,67,weight=1)
G6.add_edge(55,74,weight=1)
G6.add_edge(55,75,weight=1)
G6.add_edge(55,76,weight=1)
G6.add_edge(55,80,weight=1)
G6.add_edge(55,81,weight=1)
G6.add_edge(55,82,weight=1)
G6.add_edge(55,84,weight=1)
G6.add_edge(55,85,weight=1)
G6.add_edge(55,86,weight=1)
G6.add_edge(55,87,weight=1)
G6.add_edge(55,90,weight=1)
G6.add_edge(55,91,weight=1)
G6.add_edge(55,93,weight=1)
G6.add_edge(55,97,weight=1)
G6.add_edge(55,98,weight=1)
G6.add_edge(55,99,weight=1)
G6.add_edge(56,121,weight=1)
G6.add_edge(56,125,weight=1)
G6.add_edge(56,138,weight=1)
G6.add_edge(56,139,weight=1)
G6.add_edge(56,154,weight=1)
G6.add_edge(56,156,weight=1)
G6.add_edge(56,189,weight=1)
G6.add_edge(56,60,weight=1)
G6.add_edge(56,61,weight=1)
G6.add_edge(56,62,weight=1)
G6.add_edge(56,66,weight=1)
G6.add_edge(56,67,weight=1)
G6.add_edge(56,68,weight=1)
G6.add_edge(56,70,weight=1)
G6.add_edge(56,71,weight=1)
G6.add_edge(56,73,weight=1)
G6.add_edge(56,89,weight=1)
G6.add_edge(56,90,weight=1)
G6.add_edge(57,125,weight=1)
G6.add_edge(57,161,weight=1)
G6.add_edge(57,58,weight=1)
G6.add_edge(57,59,weight=1)
G6.add_edge(57,60,weight=1)
G6.add_edge(57,62,weight=1)
G6.add_edge(57,65,weight=1)
G6.add_edge(57,66,weight=1)
G6.add_edge(57,67,weight=1)
G6.add_edge(57,69,weight=1)
G6.add_edge(57,70,weight=1)
G6.add_edge(57,72,weight=1)
G6.add_edge(58,125,weight=1)
G6.add_edge(58,59,weight=1)
G6.add_edge(58,62,weight=1)
G6.add_edge(58,65,weight=1)
G6.add_edge(58,66,weight=1)
G6.add_edge(58,67,weight=1)
G6.add_edge(58,69,weight=1)
G6.add_edge(58,70,weight=1)
G6.add_edge(59,122,weight=1)
G6.add_edge(59,124,weight=1)
G6.add_edge(59,125,weight=1)
G6.add_edge(59,189,weight=1)
G6.add_edge(59,60,weight=1)
G6.add_edge(59,62,weight=1)
G6.add_edge(59,65,weight=1)
G6.add_edge(59,66,weight=1)
G6.add_edge(59,67,weight=1)
G6.add_edge(59,69,weight=1)
G6.add_edge(59,70,weight=1)
G6.add_edge(59,72,weight=1)
G6.add_edge(59,84,weight=1)
G6.add_edge(60,139,weight=1)
G6.add_edge(60,154,weight=1)
G6.add_edge(60,156,weight=1)
G6.add_edge(60,161,weight=1)
G6.add_edge(60,185,weight=1)
G6.add_edge(60,186,weight=1)
G6.add_edge(60,61,weight=1)
G6.add_edge(60,62,weight=1)
G6.add_edge(60,63,weight=1)
G6.add_edge(60,64,weight=1)
G6.add_edge(60,66,weight=1)
G6.add_edge(60,67,weight=1)
G6.add_edge(60,68,weight=1)
G6.add_edge(60,70,weight=1)
G6.add_edge(60,71,weight=1)
G6.add_edge(60,72,weight=1)
G6.add_edge(60,73,weight=1)
G6.add_edge(60,89,weight=1)
G6.add_edge(60,90,weight=1)
G6.add_edge(61,102,weight=1)
G6.add_edge(61,139,weight=1)
G6.add_edge(61,154,weight=1)
G6.add_edge(61,156,weight=1)
G6.add_edge(61,161,weight=1)
G6.add_edge(61,180,weight=1)
G6.add_edge(61,181,weight=1)
G6.add_edge(61,62,weight=1)
G6.add_edge(61,63,weight=1)
G6.add_edge(61,64,weight=1)
G6.add_edge(61,66,weight=1)
G6.add_edge(61,67,weight=1)
G6.add_edge(61,68,weight=1)
G6.add_edge(61,70,weight=1)
G6.add_edge(61,73,weight=1)
G6.add_edge(61,84,weight=1)
G6.add_edge(61,89,weight=1)
G6.add_edge(61,93,weight=1)
G6.add_edge(62,102,weight=1)
G6.add_edge(62,105,weight=1)
G6.add_edge(62,109,weight=1)
G6.add_edge(62,110,weight=1)
G6.add_edge(62,111,weight=1)
G6.add_edge(62,117,weight=1)
G6.add_edge(62,124,weight=1)
G6.add_edge(62,139,weight=1)
G6.add_edge(62,144,weight=1)
G6.add_edge(62,154,weight=1)
G6.add_edge(62,156,weight=1)
G6.add_edge(62,161,weight=1)
G6.add_edge(62,185,weight=1)
G6.add_edge(62,186,weight=1)
G6.add_edge(62,63,weight=1)
G6.add_edge(62,64,weight=1)
G6.add_edge(62,65,weight=1)
G6.add_edge(62,66,weight=1)
G6.add_edge(62,67,weight=1)
G6.add_edge(62,68,weight=1)
G6.add_edge(62,69,weight=1)
G6.add_edge(62,70,weight=1)
G6.add_edge(62,72,weight=1)
G6.add_edge(62,73,weight=1)
G6.add_edge(62,74,weight=1)
G6.add_edge(62,89,weight=1)
G6.add_edge(62,90,weight=1)
G6.add_edge(62,94,weight=1)
G6.add_edge(63,154,weight=1)
G6.add_edge(63,185,weight=1)
G6.add_edge(63,186,weight=1)
G6.add_edge(63,64,weight=1)
G6.add_edge(63,66,weight=1)
G6.add_edge(63,67,weight=1)
G6.add_edge(63,70,weight=1)
G6.add_edge(64,156,weight=1)
G6.add_edge(64,161,weight=1)
G6.add_edge(64,66,weight=1)
G6.add_edge(64,68,weight=1)
G6.add_edge(65,125,weight=1)
G6.add_edge(65,66,weight=1)
G6.add_edge(65,67,weight=1)
G6.add_edge(65,69,weight=1)
G6.add_edge(65,70,weight=1)
G6.add_edge(66,117,weight=1)
G6.add_edge(66,139,weight=1)
G6.add_edge(66,154,weight=1)
G6.add_edge(66,156,weight=1)
G6.add_edge(66,67,weight=1)
G6.add_edge(66,68,weight=1)
G6.add_edge(66,69,weight=1)
G6.add_edge(66,70,weight=1)
G6.add_edge(66,71,weight=1)
G6.add_edge(66,73,weight=1)
G6.add_edge(67,100,weight=1)
G6.add_edge(67,101,weight=1)
G6.add_edge(67,102,weight=1)
G6.add_edge(67,103,weight=1)
G6.add_edge(67,104,weight=1)
G6.add_edge(67,105,weight=1)
G6.add_edge(67,106,weight=1)
G6.add_edge(67,107,weight=1)
G6.add_edge(67,108,weight=1)
G6.add_edge(67,109,weight=1)
G6.add_edge(67,110,weight=1)
G6.add_edge(67,111,weight=1)
G6.add_edge(67,112,weight=1)
G6.add_edge(67,113,weight=1)
G6.add_edge(67,114,weight=1)
G6.add_edge(67,115,weight=1)
G6.add_edge(67,116,weight=1)
G6.add_edge(67,117,weight=1)
G6.add_edge(67,118,weight=1)
G6.add_edge(67,120,weight=1)
G6.add_edge(67,121,weight=1)
G6.add_edge(67,122,weight=1)
G6.add_edge(67,123,weight=1)
G6.add_edge(67,124,weight=1)
G6.add_edge(67,125,weight=1)
G6.add_edge(67,134,weight=1)
G6.add_edge(67,136,weight=1)
G6.add_edge(67,139,weight=1)
G6.add_edge(67,149,weight=1)
G6.add_edge(67,151,weight=1)
G6.add_edge(67,154,weight=1)
G6.add_edge(67,155,weight=1)
G6.add_edge(67,156,weight=1)
G6.add_edge(67,157,weight=1)
G6.add_edge(67,158,weight=1)
G6.add_edge(67,159,weight=1)
G6.add_edge(67,160,weight=1)
G6.add_edge(67,172,weight=1)
G6.add_edge(67,173,weight=1)
G6.add_edge(67,185,weight=1)
G6.add_edge(67,186,weight=1)
G6.add_edge(67,187,weight=1)
G6.add_edge(67,188,weight=1)
G6.add_edge(67,189,weight=1)
G6.add_edge(67,192,weight=1)
G6.add_edge(67,193,weight=1)
G6.add_edge(67,194,weight=1)
G6.add_edge(67,68,weight=1)
G6.add_edge(67,69,weight=1)
G6.add_edge(67,70,weight=1)
G6.add_edge(67,72,weight=1)
G6.add_edge(67,73,weight=1)
G6.add_edge(67,74,weight=1)
G6.add_edge(67,75,weight=1)
G6.add_edge(67,76,weight=1)
G6.add_edge(67,80,weight=1)
G6.add_edge(67,84,weight=1)
G6.add_edge(67,88,weight=1)
G6.add_edge(67,89,weight=1)
G6.add_edge(67,90,weight=1)
G6.add_edge(67,93,weight=1)
G6.add_edge(67,95,weight=1)
G6.add_edge(67,96,weight=1)
G6.add_edge(68,139,weight=1)
G6.add_edge(68,154,weight=1)
G6.add_edge(68,156,weight=1)
G6.add_edge(68,161,weight=1)
G6.add_edge(68,70,weight=1)
G6.add_edge(68,71,weight=1)
G6.add_edge(68,73,weight=1)
G6.add_edge(69,125,weight=1)
G6.add_edge(69,161,weight=1)
G6.add_edge(69,70,weight=1)
G6.add_edge(70,114,weight=1)
G6.add_edge(70,119,weight=1)
G6.add_edge(70,120,weight=1)
G6.add_edge(70,124,weight=1)
G6.add_edge(70,125,weight=1)
G6.add_edge(70,139,weight=1)
G6.add_edge(70,141,weight=1)
G6.add_edge(70,154,weight=1)
G6.add_edge(70,156,weight=1)
G6.add_edge(70,183,weight=1)
G6.add_edge(70,184,weight=1)
G6.add_edge(70,197,weight=1)
G6.add_edge(70,71,weight=1)
G6.add_edge(70,72,weight=1)
G6.add_edge(70,73,weight=1)
G6.add_edge(70,74,weight=1)
G6.add_edge(70,84,weight=1)
G6.add_edge(70,88,weight=1)
G6.add_edge(70,93,weight=1)
G6.add_edge(71,114,weight=1)
G6.add_edge(71,180,weight=1)
G6.add_edge(71,181,weight=1)
G6.add_edge(71,182,weight=1)
G6.add_edge(71,190,weight=1)
G6.add_edge(71,73,weight=1)
G6.add_edge(72,142,weight=1)
G6.add_edge(73,139,weight=1)
G6.add_edge(73,154,weight=1)
G6.add_edge(73,156,weight=1)
G6.add_edge(74,104,weight=1)
G6.add_edge(74,105,weight=1)
G6.add_edge(74,109,weight=1)
G6.add_edge(74,110,weight=1)
G6.add_edge(74,111,weight=1)
G6.add_edge(74,112,weight=1)
G6.add_edge(74,114,weight=1)
G6.add_edge(74,117,weight=1)
G6.add_edge(74,119,weight=1)
G6.add_edge(74,122,weight=1)
G6.add_edge(74,125,weight=1)
G6.add_edge(74,138,weight=1)
G6.add_edge(74,139,weight=1)
G6.add_edge(74,140,weight=1)
G6.add_edge(74,144,weight=1)
G6.add_edge(74,149,weight=1)
G6.add_edge(74,155,weight=1)
G6.add_edge(74,158,weight=1)
G6.add_edge(74,159,weight=1)
G6.add_edge(74,160,weight=1)
G6.add_edge(74,193,weight=1)
G6.add_edge(74,194,weight=1)
G6.add_edge(74,75,weight=1)
G6.add_edge(74,76,weight=1)
G6.add_edge(74,80,weight=1)
G6.add_edge(74,83,weight=1)
G6.add_edge(74,84,weight=1)
G6.add_edge(74,85,weight=1)
G6.add_edge(74,86,weight=1)
G6.add_edge(74,87,weight=1)
G6.add_edge(74,88,weight=1)
G6.add_edge(74,89,weight=1)
G6.add_edge(74,90,weight=1)
G6.add_edge(74,93,weight=1)
G6.add_edge(74,94,weight=1)
G6.add_edge(74,97,weight=1)
G6.add_edge(75,100,weight=1)
G6.add_edge(75,104,weight=1)
G6.add_edge(75,109,weight=1)
G6.add_edge(75,110,weight=1)
G6.add_edge(75,111,weight=1)
G6.add_edge(75,112,weight=1)
G6.add_edge(75,114,weight=1)
G6.add_edge(75,119,weight=1)
G6.add_edge(75,120,weight=1)
G6.add_edge(75,123,weight=1)
G6.add_edge(75,124,weight=1)
G6.add_edge(75,138,weight=1)
G6.add_edge(75,139,weight=1)
G6.add_edge(75,140,weight=1)
G6.add_edge(75,153,weight=1)
G6.add_edge(75,154,weight=1)
G6.add_edge(75,155,weight=1)
G6.add_edge(75,191,weight=1)
G6.add_edge(75,76,weight=1)
G6.add_edge(75,80,weight=1)
G6.add_edge(75,81,weight=1)
G6.add_edge(75,82,weight=1)
G6.add_edge(75,83,weight=1)
G6.add_edge(75,84,weight=1)
G6.add_edge(75,85,weight=1)
G6.add_edge(75,86,weight=1)
G6.add_edge(75,87,weight=1)
G6.add_edge(75,88,weight=1)
G6.add_edge(75,91,weight=1)
G6.add_edge(75,93,weight=1)
G6.add_edge(75,97,weight=1)
G6.add_edge(75,98,weight=1)
G6.add_edge(75,99,weight=1)
G6.add_edge(76,104,weight=1)
G6.add_edge(76,111,weight=1)
G6.add_edge(76,112,weight=1)
G6.add_edge(76,114,weight=1)
G6.add_edge(76,119,weight=1)
G6.add_edge(76,125,weight=1)
G6.add_edge(76,138,weight=1)
G6.add_edge(76,139,weight=1)
G6.add_edge(76,140,weight=1)
G6.add_edge(76,149,weight=1)
G6.add_edge(76,158,weight=1)
G6.add_edge(76,159,weight=1)
G6.add_edge(76,160,weight=1)
G6.add_edge(76,80,weight=1)
G6.add_edge(76,83,weight=1)
G6.add_edge(76,85,weight=1)
G6.add_edge(76,86,weight=1)
G6.add_edge(76,87,weight=1)
G6.add_edge(76,93,weight=1)
G6.add_edge(77,78,weight=1)
G6.add_edge(77,79,weight=1)
G6.add_edge(78,79,weight=1)
G6.add_edge(79,182,weight=1)
G6.add_edge(80,100,weight=1)
G6.add_edge(80,101,weight=1)
G6.add_edge(80,102,weight=1)
G6.add_edge(80,103,weight=1)
G6.add_edge(80,104,weight=1)
G6.add_edge(80,105,weight=1)
G6.add_edge(80,106,weight=1)
G6.add_edge(80,107,weight=1)
G6.add_edge(80,108,weight=1)
G6.add_edge(80,109,weight=1)
G6.add_edge(80,110,weight=1)
G6.add_edge(80,111,weight=1)
G6.add_edge(80,112,weight=1)
G6.add_edge(80,113,weight=1)
G6.add_edge(80,114,weight=1)
G6.add_edge(80,115,weight=1)
G6.add_edge(80,116,weight=1)
G6.add_edge(80,117,weight=1)
G6.add_edge(80,118,weight=1)
G6.add_edge(80,119,weight=1)
G6.add_edge(80,81,weight=1)
G6.add_edge(80,82,weight=1)
G6.add_edge(80,83,weight=1)
G6.add_edge(80,84,weight=1)
G6.add_edge(80,85,weight=1)
G6.add_edge(80,86,weight=1)
G6.add_edge(80,87,weight=1)
G6.add_edge(80,88,weight=1)
G6.add_edge(80,89,weight=1)
G6.add_edge(80,90,weight=1)
G6.add_edge(80,91,weight=1)
G6.add_edge(80,92,weight=1)
G6.add_edge(80,93,weight=1)
G6.add_edge(80,94,weight=1)
G6.add_edge(80,95,weight=1)
G6.add_edge(80,96,weight=1)
G6.add_edge(80,97,weight=1)
G6.add_edge(80,98,weight=1)
G6.add_edge(80,99,weight=1)
G6.add_edge(81,100,weight=1)
G6.add_edge(81,103,weight=1)
G6.add_edge(81,105,weight=1)
G6.add_edge(81,106,weight=1)
G6.add_edge(81,108,weight=1)
G6.add_edge(81,109,weight=1)
G6.add_edge(81,110,weight=1)
G6.add_edge(81,111,weight=1)
G6.add_edge(81,117,weight=1)
G6.add_edge(81,119,weight=1)
G6.add_edge(81,82,weight=1)
G6.add_edge(81,84,weight=1)
G6.add_edge(81,87,weight=1)
G6.add_edge(81,89,weight=1)
G6.add_edge(81,90,weight=1)
G6.add_edge(81,91,weight=1)
G6.add_edge(81,94,weight=1)
G6.add_edge(81,98,weight=1)
G6.add_edge(81,99,weight=1)
G6.add_edge(82,100,weight=1)
G6.add_edge(82,109,weight=1)
G6.add_edge(82,110,weight=1)
G6.add_edge(82,111,weight=1)
G6.add_edge(82,126,weight=1)
G6.add_edge(82,127,weight=1)
G6.add_edge(82,128,weight=1)
G6.add_edge(82,129,weight=1)
G6.add_edge(82,130,weight=1)
G6.add_edge(82,133,weight=1)
G6.add_edge(82,135,weight=1)
G6.add_edge(82,136,weight=1)
G6.add_edge(82,137,weight=1)
G6.add_edge(82,84,weight=1)
G6.add_edge(82,91,weight=1)
G6.add_edge(82,98,weight=1)
G6.add_edge(82,99,weight=1)
G6.add_edge(83,104,weight=1)
G6.add_edge(83,109,weight=1)
G6.add_edge(83,110,weight=1)
G6.add_edge(83,111,weight=1)
G6.add_edge(83,112,weight=1)
G6.add_edge(83,114,weight=1)
G6.add_edge(83,117,weight=1)
G6.add_edge(83,119,weight=1)
G6.add_edge(83,125,weight=1)
G6.add_edge(83,84,weight=1)
G6.add_edge(83,85,weight=1)
G6.add_edge(83,86,weight=1)
G6.add_edge(83,88,weight=1)
G6.add_edge(83,90,weight=1)
G6.add_edge(83,92,weight=1)
G6.add_edge(83,93,weight=1)
G6.add_edge(83,94,weight=1)
G6.add_edge(84,100,weight=1)
G6.add_edge(84,102,weight=1)
G6.add_edge(84,103,weight=1)
G6.add_edge(84,105,weight=1)
G6.add_edge(84,109,weight=1)
G6.add_edge(84,110,weight=1)
G6.add_edge(84,111,weight=1)
G6.add_edge(84,114,weight=1)
G6.add_edge(84,115,weight=1)
G6.add_edge(84,117,weight=1)
G6.add_edge(84,119,weight=1)
G6.add_edge(84,122,weight=1)
G6.add_edge(84,123,weight=1)
G6.add_edge(84,124,weight=1)
G6.add_edge(84,125,weight=1)
G6.add_edge(84,136,weight=1)
G6.add_edge(84,138,weight=1)
G6.add_edge(84,189,weight=1)
G6.add_edge(84,191,weight=1)
G6.add_edge(84,88,weight=1)
G6.add_edge(84,89,weight=1)
G6.add_edge(84,90,weight=1)
G6.add_edge(84,91,weight=1)
G6.add_edge(84,93,weight=1)
G6.add_edge(84,98,weight=1)
G6.add_edge(84,99,weight=1)
G6.add_edge(85,104,weight=1)
G6.add_edge(85,109,weight=1)
G6.add_edge(85,111,weight=1)
G6.add_edge(85,112,weight=1)
G6.add_edge(85,114,weight=1)
G6.add_edge(85,119,weight=1)
G6.add_edge(85,138,weight=1)
G6.add_edge(85,139,weight=1)
G6.add_edge(85,140,weight=1)
G6.add_edge(85,86,weight=1)
G6.add_edge(85,87,weight=1)
G6.add_edge(85,88,weight=1)
G6.add_edge(85,93,weight=1)
G6.add_edge(86,100,weight=1)
G6.add_edge(86,104,weight=1)
G6.add_edge(86,109,weight=1)
G6.add_edge(86,111,weight=1)
G6.add_edge(86,112,weight=1)
G6.add_edge(86,114,weight=1)
G6.add_edge(86,119,weight=1)
G6.add_edge(86,138,weight=1)
G6.add_edge(86,139,weight=1)
G6.add_edge(86,140,weight=1)
G6.add_edge(86,153,weight=1)
G6.add_edge(86,154,weight=1)
G6.add_edge(86,155,weight=1)
G6.add_edge(86,87,weight=1)
G6.add_edge(86,88,weight=1)
G6.add_edge(86,93,weight=1)
G6.add_edge(86,97,weight=1)
G6.add_edge(87,100,weight=1)
G6.add_edge(87,103,weight=1)
G6.add_edge(87,104,weight=1)
G6.add_edge(87,105,weight=1)
G6.add_edge(87,106,weight=1)
G6.add_edge(87,108,weight=1)
G6.add_edge(87,109,weight=1)
G6.add_edge(87,113,weight=1)
G6.add_edge(87,117,weight=1)
G6.add_edge(87,119,weight=1)
G6.add_edge(87,128,weight=1)
G6.add_edge(87,138,weight=1)
G6.add_edge(87,139,weight=1)
G6.add_edge(87,140,weight=1)
G6.add_edge(87,164,weight=1)
G6.add_edge(87,165,weight=1)
G6.add_edge(87,89,weight=1)
G6.add_edge(87,90,weight=1)
G6.add_edge(87,97,weight=1)
G6.add_edge(87,98,weight=1)
G6.add_edge(88,103,weight=1)
G6.add_edge(88,109,weight=1)
G6.add_edge(88,111,weight=1)
G6.add_edge(88,112,weight=1)
G6.add_edge(88,114,weight=1)
G6.add_edge(88,117,weight=1)
G6.add_edge(88,119,weight=1)
G6.add_edge(88,125,weight=1)
G6.add_edge(88,133,weight=1)
G6.add_edge(88,176,weight=1)
G6.add_edge(88,193,weight=1)
G6.add_edge(88,90,weight=1)
G6.add_edge(88,93,weight=1)
G6.add_edge(88,94,weight=1)
G6.add_edge(89,100,weight=1)
G6.add_edge(89,101,weight=1)
G6.add_edge(89,102,weight=1)
G6.add_edge(89,103,weight=1)
G6.add_edge(89,105,weight=1)
G6.add_edge(89,106,weight=1)
G6.add_edge(89,107,weight=1)
G6.add_edge(89,108,weight=1)
G6.add_edge(89,109,weight=1)
G6.add_edge(89,110,weight=1)
G6.add_edge(89,111,weight=1)
G6.add_edge(89,113,weight=1)
G6.add_edge(89,116,weight=1)
G6.add_edge(89,117,weight=1)
G6.add_edge(89,118,weight=1)
G6.add_edge(89,119,weight=1)
G6.add_edge(89,121,weight=1)
G6.add_edge(89,122,weight=1)
G6.add_edge(89,123,weight=1)
G6.add_edge(89,124,weight=1)
G6.add_edge(89,125,weight=1)
G6.add_edge(89,144,weight=1)
G6.add_edge(89,154,weight=1)
G6.add_edge(89,155,weight=1)
G6.add_edge(89,174,weight=1)
G6.add_edge(89,189,weight=1)
G6.add_edge(89,194,weight=1)
G6.add_edge(89,90,weight=1)
G6.add_edge(89,94,weight=1)
G6.add_edge(89,95,weight=1)
G6.add_edge(89,96,weight=1)
G6.add_edge(89,97,weight=1)
G6.add_edge(89,98,weight=1)
G6.add_edge(90,100,weight=1)
G6.add_edge(90,101,weight=1)
G6.add_edge(90,102,weight=1)
G6.add_edge(90,103,weight=1)
G6.add_edge(90,105,weight=1)
G6.add_edge(90,106,weight=1)
G6.add_edge(90,107,weight=1)
G6.add_edge(90,108,weight=1)
G6.add_edge(90,109,weight=1)
G6.add_edge(90,110,weight=1)
G6.add_edge(90,111,weight=1)
G6.add_edge(90,112,weight=1)
G6.add_edge(90,113,weight=1)
G6.add_edge(90,114,weight=1)
G6.add_edge(90,115,weight=1)
G6.add_edge(90,116,weight=1)
G6.add_edge(90,117,weight=1)
G6.add_edge(90,118,weight=1)
G6.add_edge(90,119,weight=1)
G6.add_edge(90,121,weight=1)
G6.add_edge(90,122,weight=1)
G6.add_edge(90,123,weight=1)
G6.add_edge(90,124,weight=1)
G6.add_edge(90,125,weight=1)
G6.add_edge(90,144,weight=1)
G6.add_edge(90,154,weight=1)
G6.add_edge(90,155,weight=1)
G6.add_edge(90,157,weight=1)
G6.add_edge(90,163,weight=1)
G6.add_edge(90,189,weight=1)
G6.add_edge(90,194,weight=1)
G6.add_edge(90,93,weight=1)
G6.add_edge(90,94,weight=1)
G6.add_edge(90,95,weight=1)
G6.add_edge(90,96,weight=1)
G6.add_edge(90,97,weight=1)
G6.add_edge(90,98,weight=1)
G6.add_edge(91,100,weight=1)
G6.add_edge(91,101,weight=1)
G6.add_edge(91,109,weight=1)
G6.add_edge(91,110,weight=1)
G6.add_edge(91,111,weight=1)
G6.add_edge(91,119,weight=1)
G6.add_edge(91,122,weight=1)
G6.add_edge(91,123,weight=1)
G6.add_edge(91,126,weight=1)
G6.add_edge(91,131,weight=1)
G6.add_edge(91,132,weight=1)
G6.add_edge(91,134,weight=1)
G6.add_edge(91,138,weight=1)
G6.add_edge(91,92,weight=1)
G6.add_edge(91,93,weight=1)
G6.add_edge(91,97,weight=1)
G6.add_edge(91,98,weight=1)
G6.add_edge(91,99,weight=1)
G6.add_edge(92,110,weight=1)
G6.add_edge(92,111,weight=1)
G6.add_edge(93,100,weight=1)
G6.add_edge(93,103,weight=1)
G6.add_edge(93,106,weight=1)
G6.add_edge(93,109,weight=1)
G6.add_edge(93,111,weight=1)
G6.add_edge(93,112,weight=1)
G6.add_edge(93,114,weight=1)
G6.add_edge(93,115,weight=1)
G6.add_edge(93,117,weight=1)
G6.add_edge(93,119,weight=1)
G6.add_edge(93,120,weight=1)
G6.add_edge(93,122,weight=1)
G6.add_edge(93,123,weight=1)
G6.add_edge(93,124,weight=1)
G6.add_edge(93,125,weight=1)
G6.add_edge(93,133,weight=1)
G6.add_edge(93,134,weight=1)
G6.add_edge(93,138,weight=1)
G6.add_edge(93,149,weight=1)
G6.add_edge(93,158,weight=1)
G6.add_edge(93,159,weight=1)
G6.add_edge(93,160,weight=1)
G6.add_edge(93,163,weight=1)
G6.add_edge(93,174,weight=1)
G6.add_edge(93,176,weight=1)
G6.add_edge(93,191,weight=1)
G6.add_edge(93,193,weight=1)
G6.add_edge(93,94,weight=1)
G6.add_edge(93,97,weight=1)
G6.add_edge(94,100,weight=1)
G6.add_edge(94,105,weight=1)
G6.add_edge(94,109,weight=1)
G6.add_edge(94,110,weight=1)
G6.add_edge(94,111,weight=1)
G6.add_edge(94,117,weight=1)
G6.add_edge(94,119,weight=1)
G6.add_edge(94,144,weight=1)
G6.add_edge(94,157,weight=1)
G6.add_edge(94,98,weight=1)
G6.add_edge(95,100,weight=1)
G6.add_edge(95,101,weight=1)
G6.add_edge(95,102,weight=1)
G6.add_edge(95,103,weight=1)
G6.add_edge(95,105,weight=1)
G6.add_edge(95,107,weight=1)
G6.add_edge(95,113,weight=1)
G6.add_edge(95,116,weight=1)
G6.add_edge(95,117,weight=1)
G6.add_edge(95,118,weight=1)
G6.add_edge(95,122,weight=1)
G6.add_edge(95,123,weight=1)
G6.add_edge(95,155,weight=1)
G6.add_edge(95,194,weight=1)
G6.add_edge(95,96,weight=1)
G6.add_edge(95,97,weight=1)
G6.add_edge(96,100,weight=1)
G6.add_edge(96,101,weight=1)
G6.add_edge(96,102,weight=1)
G6.add_edge(96,105,weight=1)
G6.add_edge(96,107,weight=1)
G6.add_edge(96,113,weight=1)
G6.add_edge(96,116,weight=1)
G6.add_edge(96,117,weight=1)
G6.add_edge(96,118,weight=1)
G6.add_edge(96,122,weight=1)
G6.add_edge(96,123,weight=1)
G6.add_edge(96,155,weight=1)
G6.add_edge(96,194,weight=1)
G6.add_edge(96,97,weight=1)
G6.add_edge(97,101,weight=1)
G6.add_edge(97,102,weight=1)
G6.add_edge(97,107,weight=1)
G6.add_edge(97,112,weight=1)
G6.add_edge(97,116,weight=1)
G6.add_edge(97,117,weight=1)
G6.add_edge(97,118,weight=1)
G6.add_edge(97,119,weight=1)
G6.add_edge(97,122,weight=1)
G6.add_edge(97,123,weight=1)
G6.add_edge(97,126,weight=1)
G6.add_edge(97,127,weight=1)
G6.add_edge(97,128,weight=1)
G6.add_edge(97,131,weight=1)
G6.add_edge(97,132,weight=1)
G6.add_edge(97,134,weight=1)
G6.add_edge(97,140,weight=1)
G6.add_edge(97,153,weight=1)
G6.add_edge(97,164,weight=1)
G6.add_edge(97,165,weight=1)
G6.add_edge(98,100,weight=1)
G6.add_edge(98,103,weight=1)
G6.add_edge(98,105,weight=1)
G6.add_edge(98,106,weight=1)
G6.add_edge(98,108,weight=1)
G6.add_edge(98,109,weight=1)
G6.add_edge(98,110,weight=1)
G6.add_edge(98,111,weight=1)
G6.add_edge(98,117,weight=1)
G6.add_edge(98,119,weight=1)
G6.add_edge(98,99,weight=1)
G6.add_edge(99,100,weight=1)
G6.add_edge(99,109,weight=1)
G6.add_edge(99,110,weight=1)
G6.add_edge(99,111,weight=1)
G6.add_edge(100,102,weight=1)
G6.add_edge(100,103,weight=1)
G6.add_edge(100,105,weight=1)
G6.add_edge(100,106,weight=1)
G6.add_edge(100,107,weight=1)
G6.add_edge(100,108,weight=1)
G6.add_edge(100,109,weight=1)
G6.add_edge(100,110,weight=1)
G6.add_edge(100,111,weight=1)
G6.add_edge(100,116,weight=1)
G6.add_edge(100,117,weight=1)
G6.add_edge(100,118,weight=1)
G6.add_edge(100,119,weight=1)
G6.add_edge(100,124,weight=1)
G6.add_edge(100,138,weight=1)
G6.add_edge(100,139,weight=1)
G6.add_edge(100,153,weight=1)
G6.add_edge(100,154,weight=1)
G6.add_edge(100,155,weight=1)
G6.add_edge(101,102,weight=1)
G6.add_edge(101,107,weight=1)
G6.add_edge(101,116,weight=1)
G6.add_edge(101,117,weight=1)
G6.add_edge(101,118,weight=1)
G6.add_edge(101,121,weight=1)
G6.add_edge(101,122,weight=1)
G6.add_edge(101,123,weight=1)
G6.add_edge(101,126,weight=1)
G6.add_edge(101,128,weight=1)
G6.add_edge(101,131,weight=1)
G6.add_edge(101,132,weight=1)
G6.add_edge(101,133,weight=1)
G6.add_edge(101,134,weight=1)
G6.add_edge(101,137,weight=1)
G6.add_edge(101,149,weight=1)
G6.add_edge(101,150,weight=1)
G6.add_edge(101,152,weight=1)
G6.add_edge(101,159,weight=1)
G6.add_edge(101,164,weight=1)
G6.add_edge(101,165,weight=1)
G6.add_edge(101,166,weight=1)
G6.add_edge(101,167,weight=1)
G6.add_edge(101,168,weight=1)
G6.add_edge(101,169,weight=1)
G6.add_edge(101,170,weight=1)
G6.add_edge(101,171,weight=1)
G6.add_edge(101,172,weight=1)
G6.add_edge(101,173,weight=1)
G6.add_edge(101,174,weight=1)
G6.add_edge(101,187,weight=1)
G6.add_edge(101,188,weight=1)
G6.add_edge(101,189,weight=1)
G6.add_edge(102,105,weight=1)
G6.add_edge(102,107,weight=1)
G6.add_edge(102,108,weight=1)
G6.add_edge(102,113,weight=1)
G6.add_edge(102,116,weight=1)
G6.add_edge(102,117,weight=1)
G6.add_edge(102,118,weight=1)
G6.add_edge(102,122,weight=1)
G6.add_edge(102,123,weight=1)
G6.add_edge(102,124,weight=1)
G6.add_edge(102,155,weight=1)
G6.add_edge(102,194,weight=1)
G6.add_edge(103,105,weight=1)
G6.add_edge(103,106,weight=1)
G6.add_edge(103,108,weight=1)
G6.add_edge(103,109,weight=1)
G6.add_edge(103,111,weight=1)
G6.add_edge(103,114,weight=1)
G6.add_edge(103,115,weight=1)
G6.add_edge(103,117,weight=1)
G6.add_edge(103,119,weight=1)
G6.add_edge(103,120,weight=1)
G6.add_edge(103,124,weight=1)
G6.add_edge(103,125,weight=1)
G6.add_edge(104,112,weight=1)
G6.add_edge(104,119,weight=1)
G6.add_edge(104,122,weight=1)
G6.add_edge(104,138,weight=1)
G6.add_edge(104,139,weight=1)
G6.add_edge(104,140,weight=1)
G6.add_edge(104,155,weight=1)
G6.add_edge(104,193,weight=1)
G6.add_edge(104,194,weight=1)
G6.add_edge(105,106,weight=1)
G6.add_edge(105,107,weight=1)
G6.add_edge(105,108,weight=1)
G6.add_edge(105,109,weight=1)
G6.add_edge(105,110,weight=1)
G6.add_edge(105,111,weight=1)
G6.add_edge(105,113,weight=1)
G6.add_edge(105,115,weight=1)
G6.add_edge(105,116,weight=1)
G6.add_edge(105,117,weight=1)
G6.add_edge(105,118,weight=1)
G6.add_edge(105,119,weight=1)
G6.add_edge(105,144,weight=1)
G6.add_edge(106,108,weight=1)
G6.add_edge(106,109,weight=1)
G6.add_edge(106,117,weight=1)
G6.add_edge(106,119,weight=1)
G6.add_edge(106,120,weight=1)
G6.add_edge(106,124,weight=1)
G6.add_edge(106,125,weight=1)
G6.add_edge(107,108,weight=1)
G6.add_edge(107,113,weight=1)
G6.add_edge(107,116,weight=1)
G6.add_edge(107,117,weight=1)
G6.add_edge(107,118,weight=1)
G6.add_edge(107,122,weight=1)
G6.add_edge(107,123,weight=1)
G6.add_edge(107,155,weight=1)
G6.add_edge(107,194,weight=1)
G6.add_edge(108,109,weight=1)
G6.add_edge(108,113,weight=1)
G6.add_edge(108,115,weight=1)
G6.add_edge(108,116,weight=1)
G6.add_edge(108,117,weight=1)
G6.add_edge(108,118,weight=1)
G6.add_edge(108,119,weight=1)
G6.add_edge(109,110,weight=1)
G6.add_edge(109,111,weight=1)
G6.add_edge(109,112,weight=1)
G6.add_edge(109,114,weight=1)
G6.add_edge(109,117,weight=1)
G6.add_edge(109,119,weight=1)
G6.add_edge(109,120,weight=1)
G6.add_edge(109,123,weight=1)
G6.add_edge(109,124,weight=1)
G6.add_edge(109,125,weight=1)
G6.add_edge(109,134,weight=1)
G6.add_edge(109,139,weight=1)
G6.add_edge(109,144,weight=1)
G6.add_edge(109,153,weight=1)
G6.add_edge(109,154,weight=1)
G6.add_edge(109,155,weight=1)
G6.add_edge(109,157,weight=1)
G6.add_edge(109,159,weight=1)
G6.add_edge(109,187,weight=1)
G6.add_edge(109,192,weight=1)
G6.add_edge(109,193,weight=1)
G6.add_edge(110,111,weight=1)
G6.add_edge(110,117,weight=1)
G6.add_edge(110,124,weight=1)
G6.add_edge(110,144,weight=1)
G6.add_edge(111,112,weight=1)
G6.add_edge(111,114,weight=1)
G6.add_edge(111,117,weight=1)
G6.add_edge(111,119,weight=1)
G6.add_edge(111,124,weight=1)
G6.add_edge(111,125,weight=1)
G6.add_edge(111,144,weight=1)
G6.add_edge(111,149,weight=1)
G6.add_edge(111,158,weight=1)
G6.add_edge(111,159,weight=1)
G6.add_edge(111,160,weight=1)
G6.add_edge(112,114,weight=1)
G6.add_edge(112,117,weight=1)
G6.add_edge(112,119,weight=1)
G6.add_edge(112,121,weight=1)
G6.add_edge(112,122,weight=1)
G6.add_edge(112,125,weight=1)
G6.add_edge(112,138,weight=1)
G6.add_edge(112,140,weight=1)
G6.add_edge(112,149,weight=1)
G6.add_edge(112,155,weight=1)
G6.add_edge(112,158,weight=1)
G6.add_edge(112,159,weight=1)
G6.add_edge(112,160,weight=1)
G6.add_edge(112,171,weight=1)
G6.add_edge(112,172,weight=1)
G6.add_edge(112,173,weight=1)
G6.add_edge(112,174,weight=1)
G6.add_edge(112,193,weight=1)
G6.add_edge(112,194,weight=1)
G6.add_edge(113,116,weight=1)
G6.add_edge(113,117,weight=1)
G6.add_edge(113,118,weight=1)
G6.add_edge(113,122,weight=1)
G6.add_edge(113,123,weight=1)
G6.add_edge(113,155,weight=1)
G6.add_edge(113,174,weight=1)
G6.add_edge(113,194,weight=1)
G6.add_edge(114,117,weight=1)
G6.add_edge(114,119,weight=1)
G6.add_edge(114,125,weight=1)
G6.add_edge(114,149,weight=1)
G6.add_edge(114,158,weight=1)
G6.add_edge(114,159,weight=1)
G6.add_edge(114,160,weight=1)
G6.add_edge(115,117,weight=1)
G6.add_edge(115,124,weight=1)
G6.add_edge(115,198,weight=1)
G6.add_edge(116,117,weight=1)
G6.add_edge(116,118,weight=1)
G6.add_edge(116,122,weight=1)
G6.add_edge(116,123,weight=1)
G6.add_edge(116,155,weight=1)
G6.add_edge(116,194,weight=1)
G6.add_edge(117,118,weight=1)
G6.add_edge(117,119,weight=1)
G6.add_edge(117,122,weight=1)
G6.add_edge(117,123,weight=1)
G6.add_edge(117,125,weight=1)
G6.add_edge(117,144,weight=1)
G6.add_edge(117,155,weight=1)
G6.add_edge(117,163,weight=1)
G6.add_edge(117,194,weight=1)
G6.add_edge(118,122,weight=1)
G6.add_edge(118,123,weight=1)
G6.add_edge(118,124,weight=1)
G6.add_edge(118,155,weight=1)
G6.add_edge(118,194,weight=1)
G6.add_edge(119,133,weight=1)
G6.add_edge(119,138,weight=1)
G6.add_edge(119,139,weight=1)
G6.add_edge(119,140,weight=1)
G6.add_edge(119,176,weight=1)
G6.add_edge(119,193,weight=1)
G6.add_edge(120,122,weight=1)
G6.add_edge(120,123,weight=1)
G6.add_edge(120,124,weight=1)
G6.add_edge(120,125,weight=1)
G6.add_edge(121,125,weight=1)
G6.add_edge(121,128,weight=1)
G6.add_edge(121,133,weight=1)
G6.add_edge(121,135,weight=1)
G6.add_edge(121,137,weight=1)
G6.add_edge(121,149,weight=1)
G6.add_edge(121,150,weight=1)
G6.add_edge(121,152,weight=1)
G6.add_edge(121,153,weight=1)
G6.add_edge(121,154,weight=1)
G6.add_edge(121,160,weight=1)
G6.add_edge(121,164,weight=1)
G6.add_edge(121,165,weight=1)
G6.add_edge(121,166,weight=1)
G6.add_edge(121,167,weight=1)
G6.add_edge(121,168,weight=1)
G6.add_edge(121,169,weight=1)
G6.add_edge(121,170,weight=1)
G6.add_edge(121,171,weight=1)
G6.add_edge(121,172,weight=1)
G6.add_edge(121,173,weight=1)
G6.add_edge(121,174,weight=1)
G6.add_edge(121,189,weight=1)
G6.add_edge(122,123,weight=1)
G6.add_edge(122,124,weight=1)
G6.add_edge(122,125,weight=1)
G6.add_edge(122,126,weight=1)
G6.add_edge(122,131,weight=1)
G6.add_edge(122,132,weight=1)
G6.add_edge(122,134,weight=1)
G6.add_edge(122,155,weight=1)
G6.add_edge(122,159,weight=1)
G6.add_edge(122,187,weight=1)
G6.add_edge(122,188,weight=1)
G6.add_edge(122,189,weight=1)
G6.add_edge(122,193,weight=1)
G6.add_edge(122,194,weight=1)
G6.add_edge(123,124,weight=1)
G6.add_edge(123,125,weight=1)
G6.add_edge(123,126,weight=1)
G6.add_edge(123,131,weight=1)
G6.add_edge(123,132,weight=1)
G6.add_edge(123,134,weight=1)
G6.add_edge(123,155,weight=1)
G6.add_edge(123,159,weight=1)
G6.add_edge(123,187,weight=1)
G6.add_edge(123,188,weight=1)
G6.add_edge(123,189,weight=1)
G6.add_edge(123,194,weight=1)
G6.add_edge(124,125,weight=1)
G6.add_edge(124,189,weight=1)
G6.add_edge(125,149,weight=1)
G6.add_edge(125,154,weight=1)
G6.add_edge(125,158,weight=1)
G6.add_edge(125,159,weight=1)
G6.add_edge(125,160,weight=1)
G6.add_edge(125,187,weight=1)
G6.add_edge(125,189,weight=1)
G6.add_edge(126,127,weight=1)
G6.add_edge(126,128,weight=1)
G6.add_edge(126,129,weight=1)
G6.add_edge(126,130,weight=1)
G6.add_edge(126,131,weight=1)
G6.add_edge(126,132,weight=1)
G6.add_edge(126,133,weight=1)
G6.add_edge(126,134,weight=1)
G6.add_edge(126,135,weight=1)
G6.add_edge(126,136,weight=1)
G6.add_edge(126,137,weight=1)
G6.add_edge(127,128,weight=1)
G6.add_edge(127,129,weight=1)
G6.add_edge(127,130,weight=1)
G6.add_edge(127,133,weight=1)
G6.add_edge(127,135,weight=1)
G6.add_edge(127,136,weight=1)
G6.add_edge(127,137,weight=1)
G6.add_edge(128,129,weight=1)
G6.add_edge(128,130,weight=1)
G6.add_edge(128,133,weight=1)
G6.add_edge(128,135,weight=1)
G6.add_edge(128,136,weight=1)
G6.add_edge(128,137,weight=1)
G6.add_edge(128,140,weight=1)
G6.add_edge(128,148,weight=1)
G6.add_edge(128,149,weight=1)
G6.add_edge(128,150,weight=1)
G6.add_edge(128,152,weight=1)
G6.add_edge(128,164,weight=1)
G6.add_edge(128,165,weight=1)
G6.add_edge(128,166,weight=1)
G6.add_edge(128,167,weight=1)
G6.add_edge(128,168,weight=1)
G6.add_edge(128,169,weight=1)
G6.add_edge(128,170,weight=1)
G6.add_edge(128,171,weight=1)
G6.add_edge(128,172,weight=1)
G6.add_edge(128,173,weight=1)
G6.add_edge(128,174,weight=1)
G6.add_edge(129,130,weight=1)
G6.add_edge(129,131,weight=1)
G6.add_edge(129,133,weight=1)
G6.add_edge(129,135,weight=1)
G6.add_edge(129,136,weight=1)
G6.add_edge(129,137,weight=1)
G6.add_edge(129,148,weight=1)
G6.add_edge(129,149,weight=1)
G6.add_edge(129,166,weight=1)
G6.add_edge(129,167,weight=1)
G6.add_edge(129,168,weight=1)
G6.add_edge(129,170,weight=1)
G6.add_edge(129,175,weight=1)
G6.add_edge(129,176,weight=1)
G6.add_edge(130,133,weight=1)
G6.add_edge(130,135,weight=1)
G6.add_edge(130,136,weight=1)
G6.add_edge(130,137,weight=1)
G6.add_edge(130,148,weight=1)
G6.add_edge(130,157,weight=1)
G6.add_edge(130,183,weight=1)
G6.add_edge(130,184,weight=1)
G6.add_edge(131,132,weight=1)
G6.add_edge(131,133,weight=1)
G6.add_edge(131,134,weight=1)
G6.add_edge(131,137,weight=1)
G6.add_edge(131,148,weight=1)
G6.add_edge(131,166,weight=1)
G6.add_edge(131,167,weight=1)
G6.add_edge(131,175,weight=1)
G6.add_edge(131,176,weight=1)
G6.add_edge(131,183,weight=1)
G6.add_edge(131,184,weight=1)
G6.add_edge(132,134,weight=1)
G6.add_edge(132,137,weight=1)
G6.add_edge(132,141,weight=1)
G6.add_edge(132,155,weight=1)
G6.add_edge(133,135,weight=1)
G6.add_edge(133,136,weight=1)
G6.add_edge(133,137,weight=1)
G6.add_edge(133,148,weight=1)
G6.add_edge(133,149,weight=1)
G6.add_edge(133,150,weight=1)
G6.add_edge(133,152,weight=1)
G6.add_edge(133,164,weight=1)
G6.add_edge(133,165,weight=1)
G6.add_edge(133,166,weight=1)
G6.add_edge(133,167,weight=1)
G6.add_edge(133,168,weight=1)
G6.add_edge(133,169,weight=1)
G6.add_edge(133,170,weight=1)
G6.add_edge(133,171,weight=1)
G6.add_edge(133,172,weight=1)
G6.add_edge(133,173,weight=1)
G6.add_edge(133,174,weight=1)
G6.add_edge(133,175,weight=1)
G6.add_edge(133,176,weight=1)
G6.add_edge(133,177,weight=1)
G6.add_edge(133,178,weight=1)
G6.add_edge(133,179,weight=1)
G6.add_edge(133,193,weight=1)
G6.add_edge(134,157,weight=1)
G6.add_edge(134,192,weight=1)
G6.add_edge(135,136,weight=1)
G6.add_edge(135,137,weight=1)
G6.add_edge(135,148,weight=1)
G6.add_edge(135,153,weight=1)
G6.add_edge(135,168,weight=1)
G6.add_edge(135,171,weight=1)
G6.add_edge(135,172,weight=1)
G6.add_edge(135,173,weight=1)
G6.add_edge(136,137,weight=1)
G6.add_edge(136,150,weight=1)
G6.add_edge(136,159,weight=1)
G6.add_edge(136,165,weight=1)
G6.add_edge(136,169,weight=1)
G6.add_edge(136,179,weight=1)
G6.add_edge(136,187,weight=1)
G6.add_edge(136,192,weight=1)
G6.add_edge(137,148,weight=1)
G6.add_edge(137,149,weight=1)
G6.add_edge(137,150,weight=1)
G6.add_edge(137,152,weight=1)
G6.add_edge(137,153,weight=1)
G6.add_edge(137,155,weight=1)
G6.add_edge(137,164,weight=1)
G6.add_edge(137,165,weight=1)
G6.add_edge(137,166,weight=1)
G6.add_edge(137,167,weight=1)
G6.add_edge(137,168,weight=1)
G6.add_edge(137,169,weight=1)
G6.add_edge(137,170,weight=1)
G6.add_edge(137,171,weight=1)
G6.add_edge(137,172,weight=1)
G6.add_edge(137,173,weight=1)
G6.add_edge(137,174,weight=1)
G6.add_edge(137,175,weight=1)
G6.add_edge(137,176,weight=1)
G6.add_edge(138,139,weight=1)
G6.add_edge(138,140,weight=1)
G6.add_edge(139,140,weight=1)
G6.add_edge(139,153,weight=1)
G6.add_edge(139,154,weight=1)
G6.add_edge(139,155,weight=1)
G6.add_edge(139,156,weight=1)
G6.add_edge(140,164,weight=1)
G6.add_edge(140,165,weight=1)
G6.add_edge(141,142,weight=1)
G6.add_edge(141,143,weight=1)
G6.add_edge(142,143,weight=1)
G6.add_edge(145,146,weight=1)
G6.add_edge(145,147,weight=1)
G6.add_edge(145,148,weight=1)
G6.add_edge(145,149,weight=1)
G6.add_edge(145,150,weight=1)
G6.add_edge(146,147,weight=1)
G6.add_edge(146,148,weight=1)
G6.add_edge(146,149,weight=1)
G6.add_edge(146,150,weight=1)
G6.add_edge(147,148,weight=1)
G6.add_edge(147,149,weight=1)
G6.add_edge(147,150,weight=1)
G6.add_edge(148,149,weight=1)
G6.add_edge(148,150,weight=1)
G6.add_edge(148,166,weight=1)
G6.add_edge(148,167,weight=1)
G6.add_edge(148,168,weight=1)
G6.add_edge(148,170,weight=1)
G6.add_edge(148,175,weight=1)
G6.add_edge(148,176,weight=1)
G6.add_edge(149,150,weight=1)
G6.add_edge(149,152,weight=1)
G6.add_edge(149,158,weight=1)
G6.add_edge(149,159,weight=1)
G6.add_edge(149,160,weight=1)
G6.add_edge(149,164,weight=1)
G6.add_edge(149,165,weight=1)
G6.add_edge(149,166,weight=1)
G6.add_edge(149,167,weight=1)
G6.add_edge(149,168,weight=1)
G6.add_edge(149,169,weight=1)
G6.add_edge(149,170,weight=1)
G6.add_edge(149,171,weight=1)
G6.add_edge(149,172,weight=1)
G6.add_edge(149,173,weight=1)
G6.add_edge(149,174,weight=1)
G6.add_edge(150,152,weight=1)
G6.add_edge(150,164,weight=1)
G6.add_edge(150,165,weight=1)
G6.add_edge(150,166,weight=1)
G6.add_edge(150,167,weight=1)
G6.add_edge(150,168,weight=1)
G6.add_edge(150,169,weight=1)
G6.add_edge(150,170,weight=1)
G6.add_edge(150,171,weight=1)
G6.add_edge(150,172,weight=1)
G6.add_edge(150,173,weight=1)
G6.add_edge(150,174,weight=1)
G6.add_edge(150,179,weight=1)
G6.add_edge(150,192,weight=1)
G6.add_edge(151,152,weight=1)
G6.add_edge(152,164,weight=1)
G6.add_edge(152,165,weight=1)
G6.add_edge(152,166,weight=1)
G6.add_edge(152,167,weight=1)
G6.add_edge(152,168,weight=1)
G6.add_edge(152,169,weight=1)
G6.add_edge(152,170,weight=1)
G6.add_edge(152,171,weight=1)
G6.add_edge(152,172,weight=1)
G6.add_edge(152,173,weight=1)
G6.add_edge(152,174,weight=1)
G6.add_edge(153,154,weight=1)
G6.add_edge(153,155,weight=1)
G6.add_edge(153,164,weight=1)
G6.add_edge(153,165,weight=1)
G6.add_edge(153,168,weight=1)
G6.add_edge(153,171,weight=1)
G6.add_edge(153,172,weight=1)
G6.add_edge(153,173,weight=1)
G6.add_edge(153,177,weight=1)
G6.add_edge(153,178,weight=1)
G6.add_edge(154,155,weight=1)
G6.add_edge(154,156,weight=1)
G6.add_edge(154,185,weight=1)
G6.add_edge(154,186,weight=1)
G6.add_edge(154,189,weight=1)
G6.add_edge(154,191,weight=1)
G6.add_edge(155,160,weight=1)
G6.add_edge(155,164,weight=1)
G6.add_edge(155,165,weight=1)
G6.add_edge(155,177,weight=1)
G6.add_edge(155,178,weight=1)
G6.add_edge(155,193,weight=1)
G6.add_edge(155,194,weight=1)
G6.add_edge(156,161,weight=1)
G6.add_edge(158,159,weight=1)
G6.add_edge(158,160,weight=1)
G6.add_edge(159,160,weight=1)
G6.add_edge(159,187,weight=1)
G6.add_edge(159,188,weight=1)
G6.add_edge(159,189,weight=1)
G6.add_edge(160,173,weight=1)
G6.add_edge(162,163,weight=1)
G6.add_edge(164,165,weight=1)
G6.add_edge(164,166,weight=1)
G6.add_edge(164,167,weight=1)
G6.add_edge(164,168,weight=1)
G6.add_edge(164,169,weight=1)
G6.add_edge(164,170,weight=1)
G6.add_edge(164,171,weight=1)
G6.add_edge(164,172,weight=1)
G6.add_edge(164,173,weight=1)
G6.add_edge(164,174,weight=1)
G6.add_edge(164,179,weight=1)
G6.add_edge(164,192,weight=1)
G6.add_edge(165,166,weight=1)
G6.add_edge(165,167,weight=1)
G6.add_edge(165,168,weight=1)
G6.add_edge(165,169,weight=1)
G6.add_edge(165,170,weight=1)
G6.add_edge(165,171,weight=1)
G6.add_edge(165,172,weight=1)
G6.add_edge(165,173,weight=1)
G6.add_edge(165,174,weight=1)
G6.add_edge(165,179,weight=1)
G6.add_edge(165,192,weight=1)
G6.add_edge(166,167,weight=1)
G6.add_edge(166,168,weight=1)
G6.add_edge(166,169,weight=1)
G6.add_edge(166,170,weight=1)
G6.add_edge(166,171,weight=1)
G6.add_edge(166,172,weight=1)
G6.add_edge(166,173,weight=1)
G6.add_edge(166,174,weight=1)
G6.add_edge(166,175,weight=1)
G6.add_edge(166,176,weight=1)
G6.add_edge(167,168,weight=1)
G6.add_edge(167,169,weight=1)
G6.add_edge(167,170,weight=1)
G6.add_edge(167,171,weight=1)
G6.add_edge(167,172,weight=1)
G6.add_edge(167,173,weight=1)
G6.add_edge(167,174,weight=1)
G6.add_edge(167,175,weight=1)
G6.add_edge(167,176,weight=1)
G6.add_edge(168,169,weight=1)
G6.add_edge(168,170,weight=1)
G6.add_edge(168,171,weight=1)
G6.add_edge(168,172,weight=1)
G6.add_edge(168,173,weight=1)
G6.add_edge(168,174,weight=1)
G6.add_edge(168,177,weight=1)
G6.add_edge(168,178,weight=1)
G6.add_edge(168,179,weight=1)
G6.add_edge(169,170,weight=1)
G6.add_edge(169,171,weight=1)
G6.add_edge(169,172,weight=1)
G6.add_edge(169,173,weight=1)
G6.add_edge(169,174,weight=1)
G6.add_edge(169,179,weight=1)
G6.add_edge(169,192,weight=1)
G6.add_edge(170,171,weight=1)
G6.add_edge(170,172,weight=1)
G6.add_edge(170,173,weight=1)
G6.add_edge(170,174,weight=1)
G6.add_edge(171,172,weight=1)
G6.add_edge(171,173,weight=1)
G6.add_edge(171,174,weight=1)
G6.add_edge(172,173,weight=1)
G6.add_edge(172,174,weight=1)
G6.add_edge(173,174,weight=1)
G6.add_edge(175,176,weight=1)
G6.add_edge(175,195,weight=1)
G6.add_edge(176,193,weight=1)
G6.add_edge(177,178,weight=1)
G6.add_edge(177,179,weight=1)
G6.add_edge(178,179,weight=1)
G6.add_edge(179,192,weight=1)
G6.add_edge(180,181,weight=1)
G6.add_edge(180,182,weight=1)
G6.add_edge(181,182,weight=1)
G6.add_edge(182,190,weight=1)
G6.add_edge(183,184,weight=1)
G6.add_edge(185,186,weight=1)
G6.add_edge(187,188,weight=1)
G6.add_edge(187,189,weight=1)
G6.add_edge(188,189,weight=1)
G6.add_edge(193,194,weight=1)




#Metabolico
G7 = nx.DiGraph()
G7.add_edge(1,2,weight=1)
G7.add_edge(1,3,weight=1)
G7.add_edge(1,4,weight=1)
G7.add_edge(1,5,weight=1)
G7.add_edge(1,6,weight=1)
G7.add_edge(1,7,weight=1)
G7.add_edge(1,8,weight=1)
G7.add_edge(9,10,weight=1)
G7.add_edge(9,3,weight=1)
G7.add_edge(9,11,weight=1)
G7.add_edge(10,3,weight=1)
G7.add_edge(10,11,weight=1)
G7.add_edge(10,12,weight=1)
G7.add_edge(10,13,weight=1)
G7.add_edge(10,14,weight=1)
G7.add_edge(15,16,weight=1)
G7.add_edge(15,17,weight=1)
G7.add_edge(15,18,weight=1)
G7.add_edge(15,13,weight=1)
G7.add_edge(15,19,weight=1)
G7.add_edge(15,20,weight=1)
G7.add_edge(21,22,weight=1)
G7.add_edge(21,3,weight=1)
G7.add_edge(19,23,weight=1)
G7.add_edge(19,24,weight=1)
G7.add_edge(19,25,weight=1)
G7.add_edge(19,26,weight=1)
G7.add_edge(19,27,weight=1)
G7.add_edge(19,28,weight=1)
G7.add_edge(19,29,weight=1)
G7.add_edge(19,30,weight=1)
G7.add_edge(19,12,weight=1)
G7.add_edge(19,31,weight=1)
G7.add_edge(19,13,weight=1)
G7.add_edge(19,32,weight=1)
G7.add_edge(19,20,weight=1)
G7.add_edge(19,33,weight=1)
G7.add_edge(19,34,weight=1)
G7.add_edge(19,6,weight=1)
G7.add_edge(19,11,weight=1)
G7.add_edge(19,35,weight=1)
G7.add_edge(19,4,weight=1)
G7.add_edge(19,36,weight=1)
G7.add_edge(19,37,weight=1)
G7.add_edge(19,38,weight=1)
G7.add_edge(12,13,weight=1)
G7.add_edge(12,14,weight=1)
G7.add_edge(12,6,weight=1)
G7.add_edge(12,39,weight=1)
G7.add_edge(12,4,weight=1)
G7.add_edge(12,16,weight=1)
G7.add_edge(12,40,weight=1)
G7.add_edge(12,18,weight=1)
G7.add_edge(41,42,weight=1)
G7.add_edge(41,43,weight=1)
G7.add_edge(41,44,weight=1)
G7.add_edge(41,45,weight=1)
G7.add_edge(46,47,weight=1)
G7.add_edge(46,48,weight=1)
G7.add_edge(46,49,weight=1)
G7.add_edge(46,50,weight=1)
G7.add_edge(51,52,weight=1)
G7.add_edge(51,11,weight=1)
G7.add_edge(51,53,weight=1)
G7.add_edge(51,54,weight=1)
G7.add_edge(51,55,weight=1)
G7.add_edge(51,56,weight=1)
G7.add_edge(51,57,weight=1)
G7.add_edge(58,55,weight=1)
G7.add_edge(58,11,weight=1)
G7.add_edge(58,54,weight=1)
G7.add_edge(58,59,weight=1)
G7.add_edge(58,60,weight=1)
G7.add_edge(47,48,weight=1)
G7.add_edge(47,61,weight=1)
G7.add_edge(47,49,weight=1)
G7.add_edge(62,63,weight=1)
G7.add_edge(62,64,weight=1)
G7.add_edge(62,65,weight=1)
G7.add_edge(62,66,weight=1)
G7.add_edge(62,3,weight=1)
G7.add_edge(62,67,weight=1)
G7.add_edge(62,68,weight=1)
G7.add_edge(69,16,weight=1)
G7.add_edge(69,70,weight=1)
G7.add_edge(69,65,weight=1)
G7.add_edge(69,18,weight=1)
G7.add_edge(69,11,weight=1)
G7.add_edge(69,71,weight=1)
G7.add_edge(69,72,weight=1)
G7.add_edge(45,43,weight=1)
G7.add_edge(45,44,weight=1)
G7.add_edge(45,73,weight=1)
G7.add_edge(50,48,weight=1)
G7.add_edge(50,49,weight=1)
G7.add_edge(50,74,weight=1)
G7.add_edge(74,48,weight=1)
G7.add_edge(74,49,weight=1)
G7.add_edge(74,75,weight=1)
G7.add_edge(75,76,weight=1)
G7.add_edge(75,77,weight=1)
G7.add_edge(75,44,weight=1)
G7.add_edge(75,48,weight=1)
G7.add_edge(75,49,weight=1)
G7.add_edge(77,76,weight=1)
G7.add_edge(77,44,weight=1)
G7.add_edge(77,78,weight=1)
G7.add_edge(78,76,weight=1)
G7.add_edge(78,44,weight=1)
G7.add_edge(78,79,weight=1)
G7.add_edge(79,76,weight=1)
G7.add_edge(79,44,weight=1)
G7.add_edge(79,80,weight=1)
G7.add_edge(80,76,weight=1)
G7.add_edge(80,44,weight=1)
G7.add_edge(80,42,weight=1)
G7.add_edge(42,76,weight=1)
G7.add_edge(42,44,weight=1)
G7.add_edge(42,43,weight=1)
G7.add_edge(81,3,weight=1)
G7.add_edge(81,82,weight=1)
G7.add_edge(81,6,weight=1)
G7.add_edge(81,83,weight=1)
G7.add_edge(81,4,weight=1)
G7.add_edge(84,3,weight=1)
G7.add_edge(84,85,weight=1)
G7.add_edge(84,6,weight=1)
G7.add_edge(84,86,weight=1)
G7.add_edge(84,4,weight=1)
G7.add_edge(87,3,weight=1)
G7.add_edge(87,88,weight=1)
G7.add_edge(87,6,weight=1)
G7.add_edge(87,89,weight=1)
G7.add_edge(87,4,weight=1)
G7.add_edge(90,91,weight=1)
G7.add_edge(90,7,weight=1)
G7.add_edge(90,8,weight=1)
G7.add_edge(90,3,weight=1)
G7.add_edge(90,92,weight=1)
G7.add_edge(90,4,weight=1)
G7.add_edge(90,6,weight=1)
G7.add_edge(93,94,weight=1)
G7.add_edge(93,7,weight=1)
G7.add_edge(93,8,weight=1)
G7.add_edge(93,4,weight=1)
G7.add_edge(93,6,weight=1)
G7.add_edge(93,95,weight=1)
G7.add_edge(93,3,weight=1)
G7.add_edge(96,6,weight=1)
G7.add_edge(96,97,weight=1)
G7.add_edge(96,4,weight=1)
G7.add_edge(96,98,weight=1)
G7.add_edge(96,3,weight=1)
G7.add_edge(97,99,weight=1)
G7.add_edge(97,67,weight=1)
G7.add_edge(97,6,weight=1)
G7.add_edge(97,4,weight=1)
G7.add_edge(95,65,weight=1)
G7.add_edge(95,63,weight=1)
G7.add_edge(95,100,weight=1)
G7.add_edge(95,3,weight=1)
G7.add_edge(101,3,weight=1)
G7.add_edge(101,102,weight=1)
G7.add_edge(101,92,weight=1)
G7.add_edge(101,103,weight=1)
G7.add_edge(101,98,weight=1)
G7.add_edge(104,105,weight=1)
G7.add_edge(104,16,weight=1)
G7.add_edge(104,18,weight=1)
G7.add_edge(104,67,weight=1)
G7.add_edge(104,106,weight=1)
G7.add_edge(104,11,weight=1)
G7.add_edge(107,108,weight=1)
G7.add_edge(107,3,weight=1)
G7.add_edge(107,109,weight=1)
G7.add_edge(107,72,weight=1)
G7.add_edge(110,111,weight=1)
G7.add_edge(110,112,weight=1)
G7.add_edge(113,114,weight=1)
G7.add_edge(113,7,weight=1)
G7.add_edge(113,8,weight=1)
G7.add_edge(113,16,weight=1)
G7.add_edge(113,105,weight=1)
G7.add_edge(113,18,weight=1)
G7.add_edge(105,16,weight=1)
G7.add_edge(105,18,weight=1)
G7.add_edge(115,94,weight=1)
G7.add_edge(115,116,weight=1)
G7.add_edge(115,117,weight=1)
G7.add_edge(115,67,weight=1)
G7.add_edge(91,117,weight=1)
G7.add_edge(91,67,weight=1)
G7.add_edge(91,7,weight=1)
G7.add_edge(91,8,weight=1)
G7.add_edge(91,4,weight=1)
G7.add_edge(91,6,weight=1)
G7.add_edge(118,119,weight=1)
G7.add_edge(118,120,weight=1)
G7.add_edge(118,3,weight=1)
G7.add_edge(118,121,weight=1)
G7.add_edge(118,122,weight=1)
G7.add_edge(123,124,weight=1)
G7.add_edge(123,3,weight=1)
G7.add_edge(123,6,weight=1)
G7.add_edge(123,125,weight=1)
G7.add_edge(123,4,weight=1)
G7.add_edge(126,127,weight=1)
G7.add_edge(126,3,weight=1)
G7.add_edge(126,103,weight=1)
G7.add_edge(126,6,weight=1)
G7.add_edge(126,128,weight=1)
G7.add_edge(126,4,weight=1)
G7.add_edge(127,3,weight=1)
G7.add_edge(127,103,weight=1)
G7.add_edge(127,129,weight=1)
G7.add_edge(94,7,weight=1)
G7.add_edge(94,8,weight=1)
G7.add_edge(94,4,weight=1)
G7.add_edge(94,6,weight=1)
G7.add_edge(114,7,weight=1)
G7.add_edge(114,8,weight=1)
G7.add_edge(114,3,weight=1)
G7.add_edge(114,130,weight=1)
G7.add_edge(114,102,weight=1)
G7.add_edge(114,103,weight=1)
G7.add_edge(131,3,weight=1)
G7.add_edge(131,132,weight=1)
G7.add_edge(131,6,weight=1)
G7.add_edge(131,133,weight=1)
G7.add_edge(131,4,weight=1)
G7.add_edge(134,6,weight=1)
G7.add_edge(134,135,weight=1)
G7.add_edge(134,4,weight=1)
G7.add_edge(134,3,weight=1)
G7.add_edge(134,136,weight=1)
G7.add_edge(137,3,weight=1)
G7.add_edge(137,138,weight=1)
G7.add_edge(137,6,weight=1)
G7.add_edge(137,139,weight=1)
G7.add_edge(137,4,weight=1)
G7.add_edge(128,6,weight=1)
G7.add_edge(128,4,weight=1)
G7.add_edge(128,103,weight=1)
G7.add_edge(128,67,weight=1)
G7.add_edge(128,140,weight=1)
G7.add_edge(141,142,weight=1)
G7.add_edge(141,3,weight=1)
G7.add_edge(141,6,weight=1)
G7.add_edge(141,4,weight=1)
G7.add_edge(141,143,weight=1)
G7.add_edge(144,3,weight=1)
G7.add_edge(144,145,weight=1)
G7.add_edge(144,146,weight=1)
G7.add_edge(144,147,weight=1)
G7.add_edge(148,3,weight=1)
G7.add_edge(148,149,weight=1)
G7.add_edge(148,150,weight=1)
G7.add_edge(148,151,weight=1)
G7.add_edge(148,152,weight=1)
G7.add_edge(153,154,weight=1)
G7.add_edge(153,16,weight=1)
G7.add_edge(153,18,weight=1)
G7.add_edge(153,155,weight=1)
G7.add_edge(153,156,weight=1)
G7.add_edge(153,32,weight=1)
G7.add_edge(154,16,weight=1)
G7.add_edge(154,18,weight=1)
G7.add_edge(157,155,weight=1)
G7.add_edge(157,156,weight=1)
G7.add_edge(157,32,weight=1)
G7.add_edge(158,159,weight=1)
G7.add_edge(158,67,weight=1)
G7.add_edge(158,160,weight=1)
G7.add_edge(158,161,weight=1)
G7.add_edge(158,162,weight=1)
G7.add_edge(163,23,weight=1)
G7.add_edge(163,67,weight=1)
G7.add_edge(163,3,weight=1)
G7.add_edge(163,164,weight=1)
G7.add_edge(165,166,weight=1)
G7.add_edge(165,67,weight=1)
G7.add_edge(165,16,weight=1)
G7.add_edge(165,167,weight=1)
G7.add_edge(165,168,weight=1)
G7.add_edge(165,18,weight=1)
G7.add_edge(165,11,weight=1)
G7.add_edge(166,67,weight=1)
G7.add_edge(166,169,weight=1)
G7.add_edge(166,16,weight=1)
G7.add_edge(166,18,weight=1)
G7.add_edge(166,11,weight=1)
G7.add_edge(168,16,weight=1)
G7.add_edge(168,167,weight=1)
G7.add_edge(168,18,weight=1)
G7.add_edge(168,11,weight=1)
G7.add_edge(168,170,weight=1)
G7.add_edge(168,171,weight=1)
G7.add_edge(23,67,weight=1)
G7.add_edge(23,3,weight=1)
G7.add_edge(23,24,weight=1)
G7.add_edge(23,25,weight=1)
G7.add_edge(172,16,weight=1)
G7.add_edge(172,65,weight=1)
G7.add_edge(172,173,weight=1)
G7.add_edge(172,18,weight=1)
G7.add_edge(172,11,weight=1)
G7.add_edge(172,170,weight=1)
G7.add_edge(172,174,weight=1)
G7.add_edge(172,175,weight=1)
G7.add_edge(172,176,weight=1)
G7.add_edge(172,177,weight=1)
G7.add_edge(173,16,weight=1)
G7.add_edge(173,65,weight=1)
G7.add_edge(173,18,weight=1)
G7.add_edge(173,11,weight=1)
G7.add_edge(173,178,weight=1)
G7.add_edge(179,180,weight=1)
G7.add_edge(179,6,weight=1)
G7.add_edge(179,4,weight=1)
G7.add_edge(179,3,weight=1)
G7.add_edge(179,181,weight=1)
G7.add_edge(180,6,weight=1)
G7.add_edge(180,4,weight=1)
G7.add_edge(182,3,weight=1)
G7.add_edge(182,183,weight=1)
G7.add_edge(182,11,weight=1)
G7.add_edge(182,184,weight=1)
G7.add_edge(183,3,weight=1)
G7.add_edge(183,11,weight=1)
G7.add_edge(40,6,weight=1)
G7.add_edge(40,185,weight=1)
G7.add_edge(40,4,weight=1)
G7.add_edge(40,8,weight=1)
G7.add_edge(40,7,weight=1)
G7.add_edge(40,35,weight=1)
G7.add_edge(40,18,weight=1)
G7.add_edge(40,16,weight=1)
G7.add_edge(40,186,weight=1)
G7.add_edge(150,3,weight=1)
G7.add_edge(150,149,weight=1)
G7.add_edge(150,151,weight=1)
G7.add_edge(150,152,weight=1)
G7.add_edge(169,16,weight=1)
G7.add_edge(169,18,weight=1)
G7.add_edge(169,11,weight=1)
G7.add_edge(169,177,weight=1)
G7.add_edge(169,3,weight=1)
G7.add_edge(169,187,weight=1)
G7.add_edge(169,65,weight=1)
G7.add_edge(152,3,weight=1)
G7.add_edge(152,151,weight=1)
G7.add_edge(152,188,weight=1)
G7.add_edge(147,3,weight=1)
G7.add_edge(147,146,weight=1)
G7.add_edge(189,190,weight=1)
G7.add_edge(189,191,weight=1)
G7.add_edge(189,32,weight=1)
G7.add_edge(189,16,weight=1)
G7.add_edge(189,192,weight=1)
G7.add_edge(189,193,weight=1)
G7.add_edge(194,195,weight=1)
G7.add_edge(194,11,weight=1)
G7.add_edge(194,60,weight=1)
G7.add_edge(194,196,weight=1)
G7.add_edge(194,197,weight=1)
G7.add_edge(194,57,weight=1)
G7.add_edge(194,198,weight=1)
G7.add_edge(199,3,weight=1)
G7.add_edge(199,102,weight=1)
G7.add_edge(199,92,weight=1)
G7.add_edge(199,103,weight=1)
G7.add_edge(199,200,weight=1)
G7.add_edge(98,3,weight=1)
G7.add_edge(125,6,weight=1)
G7.add_edge(125,4,weight=1)
G7.add_edge(125,103,weight=1)
G7.add_edge(125,102,weight=1)
G7.add_edge(125,140,weight=1)
G7.add_edge(201,120,weight=1)
G7.add_edge(201,124,weight=1)
G7.add_edge(201,121,weight=1)
G7.add_edge(201,103,weight=1)
G7.add_edge(201,202,weight=1)
G7.add_edge(201,203,weight=1)
G7.add_edge(124,120,weight=1)
G7.add_edge(124,121,weight=1)
G7.add_edge(124,3,weight=1)
G7.add_edge(204,205,weight=1)
G7.add_edge(204,7,weight=1)
G7.add_edge(204,206,weight=1)
G7.add_edge(204,3,weight=1)
G7.add_edge(204,8,weight=1)
G7.add_edge(204,207,weight=1)
G7.add_edge(204,208,weight=1)
G7.add_edge(204,209,weight=1)
G7.add_edge(205,7,weight=1)
G7.add_edge(205,206,weight=1)
G7.add_edge(205,3,weight=1)
G7.add_edge(205,8,weight=1)
G7.add_edge(205,210,weight=1)
G7.add_edge(205,207,weight=1)
G7.add_edge(205,209,weight=1)
G7.add_edge(211,212,weight=1)
G7.add_edge(211,7,weight=1)
G7.add_edge(211,206,weight=1)
G7.add_edge(211,3,weight=1)
G7.add_edge(211,8,weight=1)
G7.add_edge(211,207,weight=1)
G7.add_edge(211,213,weight=1)
G7.add_edge(211,209,weight=1)
G7.add_edge(210,213,weight=1)
G7.add_edge(210,7,weight=1)
G7.add_edge(210,206,weight=1)
G7.add_edge(210,3,weight=1)
G7.add_edge(210,8,weight=1)
G7.add_edge(210,207,weight=1)
G7.add_edge(210,209,weight=1)
G7.add_edge(213,7,weight=1)
G7.add_edge(213,206,weight=1)
G7.add_edge(213,3,weight=1)
G7.add_edge(213,8,weight=1)
G7.add_edge(213,207,weight=1)
G7.add_edge(213,209,weight=1)
G7.add_edge(212,7,weight=1)
G7.add_edge(212,206,weight=1)
G7.add_edge(212,3,weight=1)
G7.add_edge(212,8,weight=1)
G7.add_edge(212,214,weight=1)
G7.add_edge(212,67,weight=1)
G7.add_edge(215,216,weight=1)
G7.add_edge(215,63,weight=1)
G7.add_edge(215,217,weight=1)
G7.add_edge(215,218,weight=1)
G7.add_edge(215,6,weight=1)
G7.add_edge(215,4,weight=1)
G7.add_edge(219,99,weight=1)
G7.add_edge(219,67,weight=1)
G7.add_edge(219,200,weight=1)
G7.add_edge(219,6,weight=1)
G7.add_edge(219,4,weight=1)
G7.add_edge(116,117,weight=1)
G7.add_edge(116,67,weight=1)
G7.add_edge(116,3,weight=1)
G7.add_edge(116,220,weight=1)
G7.add_edge(116,68,weight=1)
G7.add_edge(116,221,weight=1)
G7.add_edge(116,70,weight=1)
G7.add_edge(116,222,weight=1)
G7.add_edge(63,65,weight=1)
G7.add_edge(63,100,weight=1)
G7.add_edge(63,187,weight=1)
G7.add_edge(63,4,weight=1)
G7.add_edge(63,6,weight=1)
G7.add_edge(63,68,weight=1)
G7.add_edge(63,3,weight=1)
G7.add_edge(63,7,weight=1)
G7.add_edge(63,8,weight=1)
G7.add_edge(63,185,weight=1)
G7.add_edge(63,223,weight=1)
G7.add_edge(63,224,weight=1)
G7.add_edge(63,225,weight=1)
G7.add_edge(63,226,weight=1)
G7.add_edge(63,227,weight=1)
G7.add_edge(63,228,weight=1)
G7.add_edge(63,67,weight=1)
G7.add_edge(63,229,weight=1)
G7.add_edge(63,230,weight=1)
G7.add_edge(63,159,weight=1)
G7.add_edge(63,2,weight=1)
G7.add_edge(63,167,weight=1)
G7.add_edge(63,231,weight=1)
G7.add_edge(63,64,weight=1)
G7.add_edge(63,232,weight=1)
G7.add_edge(63,233,weight=1)
G7.add_edge(63,216,weight=1)
G7.add_edge(63,217,weight=1)
G7.add_edge(63,99,weight=1)
G7.add_edge(63,234,weight=1)
G7.add_edge(63,92,weight=1)
G7.add_edge(63,235,weight=1)
G7.add_edge(99,67,weight=1)
G7.add_edge(99,65,weight=1)
G7.add_edge(99,234,weight=1)
G7.add_edge(186,3,weight=1)
G7.add_edge(186,236,weight=1)
G7.add_edge(237,16,weight=1)
G7.add_edge(237,11,weight=1)
G7.add_edge(237,3,weight=1)
G7.add_edge(237,193,weight=1)
G7.add_edge(238,239,weight=1)
G7.add_edge(238,240,weight=1)
G7.add_edge(238,241,weight=1)
G7.add_edge(238,242,weight=1)
G7.add_edge(238,16,weight=1)
G7.add_edge(238,243,weight=1)
G7.add_edge(238,18,weight=1)
G7.add_edge(244,3,weight=1)
G7.add_edge(244,245,weight=1)
G7.add_edge(244,11,weight=1)
G7.add_edge(244,156,weight=1)
G7.add_edge(244,246,weight=1)
G7.add_edge(244,247,weight=1)
G7.add_edge(245,3,weight=1)
G7.add_edge(245,11,weight=1)
G7.add_edge(248,24,weight=1)
G7.add_edge(248,249,weight=1)
G7.add_edge(248,67,weight=1)
G7.add_edge(248,103,weight=1)
G7.add_edge(248,7,weight=1)
G7.add_edge(248,250,weight=1)
G7.add_edge(248,8,weight=1)
G7.add_edge(251,252,weight=1)
G7.add_edge(251,3,weight=1)
G7.add_edge(251,253,weight=1)
G7.add_edge(251,11,weight=1)
G7.add_edge(252,3,weight=1)
G7.add_edge(252,7,weight=1)
G7.add_edge(252,8,weight=1)
G7.add_edge(252,254,weight=1)
G7.add_edge(39,6,weight=1)
G7.add_edge(39,4,weight=1)
G7.add_edge(39,24,weight=1)
G7.add_edge(39,117,weight=1)
G7.add_edge(39,64,weight=1)
G7.add_edge(200,3,weight=1)
G7.add_edge(200,6,weight=1)
G7.add_edge(200,4,weight=1)
G7.add_edge(214,255,weight=1)
G7.add_edge(214,256,weight=1)
G7.add_edge(214,32,weight=1)
G7.add_edge(214,67,weight=1)
G7.add_edge(83,6,weight=1)
G7.add_edge(83,4,weight=1)
G7.add_edge(83,103,weight=1)
G7.add_edge(83,102,weight=1)
G7.add_edge(83,257,weight=1)
G7.add_edge(86,6,weight=1)
G7.add_edge(86,4,weight=1)
G7.add_edge(86,103,weight=1)
G7.add_edge(86,102,weight=1)
G7.add_edge(86,258,weight=1)
G7.add_edge(133,6,weight=1)
G7.add_edge(133,4,weight=1)
G7.add_edge(133,103,weight=1)
G7.add_edge(133,102,weight=1)
G7.add_edge(133,259,weight=1)
G7.add_edge(89,6,weight=1)
G7.add_edge(89,4,weight=1)
G7.add_edge(89,103,weight=1)
G7.add_edge(89,260,weight=1)
G7.add_edge(89,102,weight=1)
G7.add_edge(139,6,weight=1)
G7.add_edge(139,4,weight=1)
G7.add_edge(139,103,weight=1)
G7.add_edge(139,102,weight=1)
G7.add_edge(139,261,weight=1)
G7.add_edge(135,6,weight=1)
G7.add_edge(135,4,weight=1)
G7.add_edge(135,103,weight=1)
G7.add_edge(135,102,weight=1)
G7.add_edge(135,262,weight=1)
G7.add_edge(35,6,weight=1)
G7.add_edge(35,11,weight=1)
G7.add_edge(35,4,weight=1)
G7.add_edge(35,18,weight=1)
G7.add_edge(35,16,weight=1)
G7.add_edge(35,37,weight=1)
G7.add_edge(35,38,weight=1)
G7.add_edge(185,65,weight=1)
G7.add_edge(185,223,weight=1)
G7.add_edge(185,6,weight=1)
G7.add_edge(185,4,weight=1)
G7.add_edge(185,8,weight=1)
G7.add_edge(185,7,weight=1)
G7.add_edge(263,16,weight=1)
G7.add_edge(263,254,weight=1)
G7.add_edge(263,18,weight=1)
G7.add_edge(263,236,weight=1)
G7.add_edge(263,264,weight=1)
G7.add_edge(263,11,weight=1)
G7.add_edge(265,3,weight=1)
G7.add_edge(265,67,weight=1)
G7.add_edge(265,266,weight=1)
G7.add_edge(265,68,weight=1)
G7.add_edge(265,267,weight=1)
G7.add_edge(66,268,weight=1)
G7.add_edge(66,3,weight=1)
G7.add_edge(66,67,weight=1)
G7.add_edge(66,68,weight=1)
G7.add_edge(269,3,weight=1)
G7.add_edge(269,270,weight=1)
G7.add_edge(269,271,weight=1)
G7.add_edge(224,65,weight=1)
G7.add_edge(224,225,weight=1)
G7.add_edge(224,67,weight=1)
G7.add_edge(272,273,weight=1)
G7.add_edge(272,3,weight=1)
G7.add_edge(272,274,weight=1)
G7.add_edge(272,171,weight=1)
G7.add_edge(226,122,weight=1)
G7.add_edge(226,3,weight=1)
G7.add_edge(226,4,weight=1)
G7.add_edge(226,6,weight=1)
G7.add_edge(226,227,weight=1)
G7.add_edge(226,65,weight=1)
G7.add_edge(122,3,weight=1)
G7.add_edge(122,4,weight=1)
G7.add_edge(122,6,weight=1)
G7.add_edge(119,120,weight=1)
G7.add_edge(119,3,weight=1)
G7.add_edge(119,121,weight=1)
G7.add_edge(255,256,weight=1)
G7.add_edge(255,32,weight=1)
G7.add_edge(255,275,weight=1)
G7.add_edge(255,117,weight=1)
G7.add_edge(273,206,weight=1)
G7.add_edge(273,276,weight=1)
G7.add_edge(218,29,weight=1)
G7.add_edge(218,6,weight=1)
G7.add_edge(218,4,weight=1)
G7.add_edge(277,7,weight=1)
G7.add_edge(277,278,weight=1)
G7.add_edge(277,8,weight=1)
G7.add_edge(277,11,weight=1)
G7.add_edge(277,16,weight=1)
G7.add_edge(277,167,weight=1)
G7.add_edge(277,18,weight=1)
G7.add_edge(161,160,weight=1)
G7.add_edge(161,162,weight=1)
G7.add_edge(161,279,weight=1)
G7.add_edge(170,280,weight=1)
G7.add_edge(170,281,weight=1)
G7.add_edge(170,32,weight=1)
G7.add_edge(170,171,weight=1)
G7.add_edge(170,174,weight=1)
G7.add_edge(170,175,weight=1)
G7.add_edge(177,16,weight=1)
G7.add_edge(177,3,weight=1)
G7.add_edge(177,187,weight=1)
G7.add_edge(177,18,weight=1)
G7.add_edge(177,65,weight=1)
G7.add_edge(177,11,weight=1)
G7.add_edge(177,176,weight=1)
G7.add_edge(177,174,weight=1)
G7.add_edge(282,283,weight=1)
G7.add_edge(282,16,weight=1)
G7.add_edge(282,193,weight=1)
G7.add_edge(282,11,weight=1)
G7.add_edge(284,3,weight=1)
G7.add_edge(284,71,weight=1)
G7.add_edge(284,174,weight=1)
G7.add_edge(284,24,weight=1)
G7.add_edge(284,285,weight=1)
G7.add_edge(284,286,weight=1)
G7.add_edge(284,287,weight=1)
G7.add_edge(284,288,weight=1)
G7.add_edge(284,68,weight=1)
G7.add_edge(284,289,weight=1)
G7.add_edge(174,286,weight=1)
G7.add_edge(174,7,weight=1)
G7.add_edge(174,8,weight=1)
G7.add_edge(174,16,weight=1)
G7.add_edge(174,65,weight=1)
G7.add_edge(174,18,weight=1)
G7.add_edge(174,11,weight=1)
G7.add_edge(174,290,weight=1)
G7.add_edge(174,3,weight=1)
G7.add_edge(174,71,weight=1)
G7.add_edge(174,24,weight=1)
G7.add_edge(174,288,weight=1)
G7.add_edge(174,68,weight=1)
G7.add_edge(174,289,weight=1)
G7.add_edge(174,175,weight=1)
G7.add_edge(174,176,weight=1)
G7.add_edge(174,291,weight=1)
G7.add_edge(174,292,weight=1)
G7.add_edge(174,293,weight=1)
G7.add_edge(267,7,weight=1)
G7.add_edge(267,197,weight=1)
G7.add_edge(267,8,weight=1)
G7.add_edge(267,3,weight=1)
G7.add_edge(268,7,weight=1)
G7.add_edge(268,57,weight=1)
G7.add_edge(268,8,weight=1)
G7.add_edge(268,3,weight=1)
G7.add_edge(280,281,weight=1)
G7.add_edge(280,32,weight=1)
G7.add_edge(175,3,weight=1)
G7.add_edge(175,294,weight=1)
G7.add_edge(283,16,weight=1)
G7.add_edge(283,193,weight=1)
G7.add_edge(283,11,weight=1)
G7.add_edge(291,292,weight=1)
G7.add_edge(291,293,weight=1)
G7.add_edge(281,295,weight=1)
G7.add_edge(281,164,weight=1)
G7.add_edge(281,32,weight=1)
G7.add_edge(281,16,weight=1)
G7.add_edge(281,27,weight=1)
G7.add_edge(281,193,weight=1)
G7.add_edge(281,143,weight=1)
G7.add_edge(281,296,weight=1)
G7.add_edge(281,54,weight=1)
G7.add_edge(281,3,weight=1)
G7.add_edge(281,187,weight=1)
G7.add_edge(281,297,weight=1)
G7.add_edge(281,65,weight=1)
G7.add_edge(281,60,weight=1)
G7.add_edge(281,298,weight=1)
G7.add_edge(297,3,weight=1)
G7.add_edge(297,187,weight=1)
G7.add_edge(297,65,weight=1)
G7.add_edge(297,32,weight=1)
G7.add_edge(297,16,weight=1)
G7.add_edge(297,71,weight=1)
G7.add_edge(297,18,weight=1)
G7.add_edge(297,176,weight=1)
G7.add_edge(297,11,weight=1)
G7.add_edge(299,300,weight=1)
G7.add_edge(299,3,weight=1)
G7.add_edge(299,65,weight=1)
G7.add_edge(299,301,weight=1)
G7.add_edge(299,302,weight=1)
G7.add_edge(303,304,weight=1)
G7.add_edge(305,301,weight=1)
G7.add_edge(305,7,weight=1)
G7.add_edge(305,8,weight=1)
G7.add_edge(305,306,weight=1)
G7.add_edge(307,308,weight=1)
G7.add_edge(307,3,weight=1)
G7.add_edge(307,8,weight=1)
G7.add_edge(307,67,weight=1)
G7.add_edge(307,309,weight=1)
G7.add_edge(307,7,weight=1)
G7.add_edge(301,3,weight=1)
G7.add_edge(301,302,weight=1)
G7.add_edge(301,7,weight=1)
G7.add_edge(301,8,weight=1)
G7.add_edge(306,7,weight=1)
G7.add_edge(306,8,weight=1)
G7.add_edge(286,16,weight=1)
G7.add_edge(286,65,weight=1)
G7.add_edge(286,191,weight=1)
G7.add_edge(286,18,weight=1)
G7.add_edge(286,11,weight=1)
G7.add_edge(286,7,weight=1)
G7.add_edge(286,8,weight=1)
G7.add_edge(286,285,weight=1)
G7.add_edge(286,287,weight=1)
G7.add_edge(253,29,weight=1)
G7.add_edge(253,3,weight=1)
G7.add_edge(253,236,weight=1)
G7.add_edge(253,11,weight=1)
G7.add_edge(18,16,weight=1)
G7.add_edge(18,67,weight=1)
G7.add_edge(18,106,weight=1)
G7.add_edge(18,11,weight=1)
G7.add_edge(18,55,weight=1)
G7.add_edge(18,193,weight=1)
G7.add_edge(18,65,weight=1)
G7.add_edge(18,191,weight=1)
G7.add_edge(18,290,weight=1)
G7.add_edge(18,300,weight=1)
G7.add_edge(18,49,weight=1)
G7.add_edge(18,167,weight=1)
G7.add_edge(18,220,weight=1)
G7.add_edge(18,310,weight=1)
G7.add_edge(18,311,weight=1)
G7.add_edge(18,155,weight=1)
G7.add_edge(18,312,weight=1)
G7.add_edge(18,313,weight=1)
G7.add_edge(18,247,weight=1)
G7.add_edge(18,3,weight=1)
G7.add_edge(18,68,weight=1)
G7.add_edge(18,314,weight=1)
G7.add_edge(18,315,weight=1)
G7.add_edge(18,316,weight=1)
G7.add_edge(18,317,weight=1)
G7.add_edge(18,318,weight=1)
G7.add_edge(18,319,weight=1)
G7.add_edge(18,198,weight=1)
G7.add_edge(18,285,weight=1)
G7.add_edge(18,320,weight=1)
G7.add_edge(18,321,weight=1)
G7.add_edge(18,322,weight=1)
G7.add_edge(18,323,weight=1)
G7.add_edge(18,324,weight=1)
G7.add_edge(18,196,weight=1)
G7.add_edge(18,287,weight=1)
G7.add_edge(18,325,weight=1)
G7.add_edge(18,326,weight=1)
G7.add_edge(18,254,weight=1)
G7.add_edge(18,178,weight=1)
G7.add_edge(18,17,weight=1)
G7.add_edge(18,30,weight=1)
G7.add_edge(18,33,weight=1)
G7.add_edge(18,236,weight=1)
G7.add_edge(18,117,weight=1)
G7.add_edge(18,327,weight=1)
G7.add_edge(18,184,weight=1)
G7.add_edge(18,328,weight=1)
G7.add_edge(18,188,weight=1)
G7.add_edge(18,246,weight=1)
G7.add_edge(18,187,weight=1)
G7.add_edge(18,70,weight=1)
G7.add_edge(18,71,weight=1)
G7.add_edge(18,72,weight=1)
G7.add_edge(18,329,weight=1)
G7.add_edge(18,298,weight=1)
G7.add_edge(18,103,weight=1)
G7.add_edge(18,330,weight=1)
G7.add_edge(18,331,weight=1)
G7.add_edge(18,332,weight=1)
G7.add_edge(18,333,weight=1)
G7.add_edge(18,140,weight=1)
G7.add_edge(18,111,weight=1)
G7.add_edge(18,334,weight=1)
G7.add_edge(18,335,weight=1)
G7.add_edge(18,336,weight=1)
G7.add_edge(18,337,weight=1)
G7.add_edge(18,176,weight=1)
G7.add_edge(18,231,weight=1)
G7.add_edge(18,338,weight=1)
G7.add_edge(18,339,weight=1)
G7.add_edge(18,340,weight=1)
G7.add_edge(18,27,weight=1)
G7.add_edge(18,243,weight=1)
G7.add_edge(18,341,weight=1)
G7.add_edge(18,342,weight=1)
G7.add_edge(18,343,weight=1)
G7.add_edge(18,14,weight=1)
G7.add_edge(18,31,weight=1)
G7.add_edge(18,344,weight=1)
G7.add_edge(18,56,weight=1)
G7.add_edge(193,16,weight=1)
G7.add_edge(193,11,weight=1)
G7.add_edge(193,103,weight=1)
G7.add_edge(193,181,weight=1)
G7.add_edge(193,249,weight=1)
G7.add_edge(193,32,weight=1)
G7.add_edge(193,345,weight=1)
G7.add_edge(193,102,weight=1)
G7.add_edge(193,55,weight=1)
G7.add_edge(193,192,weight=1)
G7.add_edge(193,300,weight=1)
G7.add_edge(193,49,weight=1)
G7.add_edge(193,3,weight=1)
G7.add_edge(193,294,weight=1)
G7.add_edge(193,68,weight=1)
G7.add_edge(193,346,weight=1)
G7.add_edge(193,167,weight=1)
G7.add_edge(193,187,weight=1)
G7.add_edge(193,347,weight=1)
G7.add_edge(193,65,weight=1)
G7.add_edge(193,348,weight=1)
G7.add_edge(193,349,weight=1)
G7.add_edge(193,27,weight=1)
G7.add_edge(193,30,weight=1)
G7.add_edge(193,33,weight=1)
G7.add_edge(193,328,weight=1)
G7.add_edge(193,184,weight=1)
G7.add_edge(193,350,weight=1)
G7.add_edge(193,171,weight=1)
G7.add_edge(193,351,weight=1)
G7.add_edge(193,298,weight=1)
G7.add_edge(193,54,weight=1)
G7.add_edge(16,11,weight=1)
G7.add_edge(16,103,weight=1)
G7.add_edge(16,181,weight=1)
G7.add_edge(16,249,weight=1)
G7.add_edge(16,32,weight=1)
G7.add_edge(16,345,weight=1)
G7.add_edge(16,102,weight=1)
G7.add_edge(16,67,weight=1)
G7.add_edge(16,106,weight=1)
G7.add_edge(16,55,weight=1)
G7.add_edge(16,65,weight=1)
G7.add_edge(16,191,weight=1)
G7.add_edge(16,290,weight=1)
G7.add_edge(16,192,weight=1)
G7.add_edge(16,300,weight=1)
G7.add_edge(16,49,weight=1)
G7.add_edge(16,3,weight=1)
G7.add_edge(16,346,weight=1)
G7.add_edge(16,167,weight=1)
G7.add_edge(16,187,weight=1)
G7.add_edge(16,347,weight=1)
G7.add_edge(16,220,weight=1)
G7.add_edge(16,310,weight=1)
G7.add_edge(16,311,weight=1)
G7.add_edge(16,155,weight=1)
G7.add_edge(16,312,weight=1)
G7.add_edge(16,313,weight=1)
G7.add_edge(16,247,weight=1)
G7.add_edge(16,68,weight=1)
G7.add_edge(16,314,weight=1)
G7.add_edge(16,348,weight=1)
G7.add_edge(16,349,weight=1)
G7.add_edge(16,315,weight=1)
G7.add_edge(16,316,weight=1)
G7.add_edge(16,317,weight=1)
G7.add_edge(16,318,weight=1)
G7.add_edge(16,319,weight=1)
G7.add_edge(16,198,weight=1)
G7.add_edge(16,285,weight=1)
G7.add_edge(16,320,weight=1)
G7.add_edge(16,321,weight=1)
G7.add_edge(16,322,weight=1)
G7.add_edge(16,323,weight=1)
G7.add_edge(16,324,weight=1)
G7.add_edge(16,196,weight=1)
G7.add_edge(16,287,weight=1)
G7.add_edge(16,325,weight=1)
G7.add_edge(16,326,weight=1)
G7.add_edge(16,254,weight=1)
G7.add_edge(16,27,weight=1)
G7.add_edge(16,178,weight=1)
G7.add_edge(16,17,weight=1)
G7.add_edge(16,30,weight=1)
G7.add_edge(16,33,weight=1)
G7.add_edge(16,236,weight=1)
G7.add_edge(16,117,weight=1)
G7.add_edge(16,327,weight=1)
G7.add_edge(16,184,weight=1)
G7.add_edge(16,188,weight=1)
G7.add_edge(16,246,weight=1)
G7.add_edge(16,70,weight=1)
G7.add_edge(16,71,weight=1)
G7.add_edge(16,72,weight=1)
G7.add_edge(16,329,weight=1)
G7.add_edge(16,298,weight=1)
G7.add_edge(16,330,weight=1)
G7.add_edge(16,331,weight=1)
G7.add_edge(16,332,weight=1)
G7.add_edge(16,333,weight=1)
G7.add_edge(16,140,weight=1)
G7.add_edge(16,111,weight=1)
G7.add_edge(16,351,weight=1)
G7.add_edge(16,293,weight=1)
G7.add_edge(16,207,weight=1)
G7.add_edge(16,334,weight=1)
G7.add_edge(16,335,weight=1)
G7.add_edge(16,336,weight=1)
G7.add_edge(16,337,weight=1)
G7.add_edge(16,176,weight=1)
G7.add_edge(16,231,weight=1)
G7.add_edge(16,338,weight=1)
G7.add_edge(16,339,weight=1)
G7.add_edge(16,120,weight=1)
G7.add_edge(16,340,weight=1)
G7.add_edge(16,352,weight=1)
G7.add_edge(16,243,weight=1)
G7.add_edge(16,341,weight=1)
G7.add_edge(16,342,weight=1)
G7.add_edge(16,343,weight=1)
G7.add_edge(16,14,weight=1)
G7.add_edge(16,31,weight=1)
G7.add_edge(16,344,weight=1)
G7.add_edge(16,56,weight=1)
G7.add_edge(22,3,weight=1)
G7.add_edge(311,155,weight=1)
G7.add_edge(311,353,weight=1)
G7.add_edge(311,3,weight=1)
G7.add_edge(311,312,weight=1)
G7.add_edge(311,241,weight=1)
G7.add_edge(311,247,weight=1)
G7.add_edge(311,44,weight=1)
G7.add_edge(311,354,weight=1)
G7.add_edge(156,155,weight=1)
G7.add_edge(156,32,weight=1)
G7.add_edge(156,246,weight=1)
G7.add_edge(156,247,weight=1)
G7.add_edge(355,155,weight=1)
G7.add_edge(355,246,weight=1)
G7.add_edge(355,32,weight=1)
G7.add_edge(247,315,weight=1)
G7.add_edge(247,246,weight=1)
G7.add_edge(67,117,weight=1)
G7.add_edge(67,206,weight=1)
G7.add_edge(67,232,weight=1)
G7.add_edge(67,276,weight=1)
G7.add_edge(67,228,weight=1)
G7.add_edge(67,229,weight=1)
G7.add_edge(67,6,weight=1)
G7.add_edge(67,230,weight=1)
G7.add_edge(67,4,weight=1)
G7.add_edge(67,106,weight=1)
G7.add_edge(67,11,weight=1)
G7.add_edge(67,3,weight=1)
G7.add_edge(67,356,weight=1)
G7.add_edge(67,333,weight=1)
G7.add_edge(67,68,weight=1)
G7.add_edge(67,314,weight=1)
G7.add_edge(67,8,weight=1)
G7.add_edge(67,309,weight=1)
G7.add_edge(67,7,weight=1)
G7.add_edge(67,187,weight=1)
G7.add_edge(67,65,weight=1)
G7.add_edge(67,296,weight=1)
G7.add_edge(67,344,weight=1)
G7.add_edge(67,120,weight=1)
G7.add_edge(67,331,weight=1)
G7.add_edge(67,121,weight=1)
G7.add_edge(67,357,weight=1)
G7.add_edge(67,358,weight=1)
G7.add_edge(67,202,weight=1)
G7.add_edge(67,159,weight=1)
G7.add_edge(67,359,weight=1)
G7.add_edge(67,360,weight=1)
G7.add_edge(67,361,weight=1)
G7.add_edge(67,24,weight=1)
G7.add_edge(67,249,weight=1)
G7.add_edge(67,103,weight=1)
G7.add_edge(67,266,weight=1)
G7.add_edge(67,92,weight=1)
G7.add_edge(67,362,weight=1)
G7.add_edge(67,140,weight=1)
G7.add_edge(155,44,weight=1)
G7.add_edge(155,354,weight=1)
G7.add_edge(155,246,weight=1)
G7.add_edge(155,32,weight=1)
G7.add_edge(155,187,weight=1)
G7.add_edge(155,31,weight=1)
G7.add_edge(155,65,weight=1)
G7.add_edge(155,11,weight=1)
G7.add_edge(155,68,weight=1)
G7.add_edge(103,102,weight=1)
G7.add_edge(103,262,weight=1)
G7.add_edge(103,260,weight=1)
G7.add_edge(103,181,weight=1)
G7.add_edge(103,249,weight=1)
G7.add_edge(103,32,weight=1)
G7.add_edge(103,130,weight=1)
G7.add_edge(103,261,weight=1)
G7.add_edge(103,259,weight=1)
G7.add_edge(103,257,weight=1)
G7.add_edge(103,258,weight=1)
G7.add_edge(103,345,weight=1)
G7.add_edge(103,229,weight=1)
G7.add_edge(103,203,weight=1)
G7.add_edge(103,112,weight=1)
G7.add_edge(103,49,weight=1)
G7.add_edge(103,11,weight=1)
G7.add_edge(103,300,weight=1)
G7.add_edge(103,363,weight=1)
G7.add_edge(103,3,weight=1)
G7.add_edge(103,231,weight=1)
G7.add_edge(103,364,weight=1)
G7.add_edge(103,365,weight=1)
G7.add_edge(103,366,weight=1)
G7.add_edge(103,330,weight=1)
G7.add_edge(103,331,weight=1)
G7.add_edge(103,202,weight=1)
G7.add_edge(103,140,weight=1)
G7.add_edge(103,92,weight=1)
G7.add_edge(103,361,weight=1)
G7.add_edge(103,24,weight=1)
G7.add_edge(103,367,weight=1)
G7.add_edge(103,368,weight=1)
G7.add_edge(103,369,weight=1)
G7.add_edge(103,6,weight=1)
G7.add_edge(103,4,weight=1)
G7.add_edge(103,362,weight=1)
G7.add_edge(103,370,weight=1)
G7.add_edge(208,207,weight=1)
G7.add_edge(208,209,weight=1)
G7.add_edge(29,3,weight=1)
G7.add_edge(29,236,weight=1)
G7.add_edge(29,11,weight=1)
G7.add_edge(29,6,weight=1)
G7.add_edge(29,4,weight=1)
G7.add_edge(29,30,weight=1)
G7.add_edge(29,26,weight=1)
G7.add_edge(29,28,weight=1)
G7.add_edge(33,30,weight=1)
G7.add_edge(33,34,weight=1)
G7.add_edge(33,36,weight=1)
G7.add_edge(30,26,weight=1)
G7.add_edge(30,28,weight=1)
G7.add_edge(30,184,weight=1)
G7.add_edge(30,371,weight=1)
G7.add_edge(30,3,weight=1)
G7.add_edge(30,68,weight=1)
G7.add_edge(308,3,weight=1)
G7.add_edge(308,184,weight=1)
G7.add_edge(308,8,weight=1)
G7.add_edge(308,7,weight=1)
G7.add_edge(371,3,weight=1)
G7.add_edge(371,68,weight=1)
G7.add_edge(371,335,weight=1)
G7.add_edge(371,345,weight=1)
G7.add_edge(327,184,weight=1)
G7.add_edge(327,3,weight=1)
G7.add_edge(327,372,weight=1)
G7.add_edge(327,17,weight=1)
G7.add_edge(373,31,weight=1)
G7.add_edge(373,14,weight=1)
G7.add_edge(373,13,weight=1)
G7.add_edge(373,32,weight=1)
G7.add_edge(184,8,weight=1)
G7.add_edge(184,7,weight=1)
G7.add_edge(184,328,weight=1)
G7.add_edge(216,217,weight=1)
G7.add_edge(109,374,weight=1)
G7.add_edge(109,375,weight=1)
G7.add_edge(109,117,weight=1)
G7.add_edge(109,3,weight=1)
G7.add_edge(109,72,weight=1)
G7.add_edge(27,26,weight=1)
G7.add_edge(27,28,weight=1)
G7.add_edge(27,309,weight=1)
G7.add_edge(27,340,weight=1)
G7.add_edge(309,28,weight=1)
G7.add_edge(309,8,weight=1)
G7.add_edge(309,7,weight=1)
G7.add_edge(26,28,weight=1)
G7.add_edge(227,65,weight=1)
G7.add_edge(227,365,weight=1)
G7.add_edge(227,117,weight=1)
G7.add_edge(250,120,weight=1)
G7.add_edge(250,121,weight=1)
G7.add_edge(250,368,weight=1)
G7.add_edge(250,7,weight=1)
G7.add_edge(250,8,weight=1)
G7.add_edge(120,249,weight=1)
G7.add_edge(120,121,weight=1)
G7.add_edge(120,138,weight=1)
G7.add_edge(120,261,weight=1)
G7.add_edge(120,136,weight=1)
G7.add_edge(120,262,weight=1)
G7.add_edge(120,132,weight=1)
G7.add_edge(120,259,weight=1)
G7.add_edge(120,82,weight=1)
G7.add_edge(120,257,weight=1)
G7.add_edge(120,88,weight=1)
G7.add_edge(120,260,weight=1)
G7.add_edge(120,85,weight=1)
G7.add_edge(120,258,weight=1)
G7.add_edge(120,357,weight=1)
G7.add_edge(120,3,weight=1)
G7.add_edge(120,363,weight=1)
G7.add_edge(120,171,weight=1)
G7.add_edge(120,246,weight=1)
G7.add_edge(120,34,weight=1)
G7.add_edge(120,331,weight=1)
G7.add_edge(120,339,weight=1)
G7.add_edge(120,32,weight=1)
G7.add_edge(120,368,weight=1)
G7.add_edge(120,370,weight=1)
G7.add_edge(120,129,weight=1)
G7.add_edge(121,249,weight=1)
G7.add_edge(121,138,weight=1)
G7.add_edge(121,261,weight=1)
G7.add_edge(121,136,weight=1)
G7.add_edge(121,262,weight=1)
G7.add_edge(121,132,weight=1)
G7.add_edge(121,259,weight=1)
G7.add_edge(121,82,weight=1)
G7.add_edge(121,257,weight=1)
G7.add_edge(121,88,weight=1)
G7.add_edge(121,260,weight=1)
G7.add_edge(121,85,weight=1)
G7.add_edge(121,258,weight=1)
G7.add_edge(121,357,weight=1)
G7.add_edge(121,3,weight=1)
G7.add_edge(121,363,weight=1)
G7.add_edge(121,171,weight=1)
G7.add_edge(121,246,weight=1)
G7.add_edge(121,34,weight=1)
G7.add_edge(121,331,weight=1)
G7.add_edge(121,368,weight=1)
G7.add_edge(121,370,weight=1)
G7.add_edge(121,129,weight=1)
G7.add_edge(339,338,weight=1)
G7.add_edge(339,32,weight=1)
G7.add_edge(49,11,weight=1)
G7.add_edge(49,112,weight=1)
G7.add_edge(49,300,weight=1)
G7.add_edge(49,363,weight=1)
G7.add_edge(49,48,weight=1)
G7.add_edge(49,61,weight=1)
G7.add_edge(49,345,weight=1)
G7.add_edge(49,102,weight=1)
G7.add_edge(49,298,weight=1)
G7.add_edge(49,294,weight=1)
G7.add_edge(49,167,weight=1)
G7.add_edge(49,350,weight=1)
G7.add_edge(49,3,weight=1)
G7.add_edge(49,376,weight=1)
G7.add_edge(49,32,weight=1)
G7.add_edge(377,378,weight=1)
G7.add_edge(377,7,weight=1)
G7.add_edge(377,379,weight=1)
G7.add_edge(377,8,weight=1)
G7.add_edge(380,7,weight=1)
G7.add_edge(380,381,weight=1)
G7.add_edge(380,8,weight=1)
G7.add_edge(380,382,weight=1)
G7.add_edge(382,48,weight=1)
G7.add_edge(382,3,weight=1)
G7.add_edge(379,7,weight=1)
G7.add_edge(379,8,weight=1)
G7.add_edge(381,7,weight=1)
G7.add_edge(381,8,weight=1)
G7.add_edge(48,61,weight=1)
G7.add_edge(48,3,weight=1)
G7.add_edge(298,3,weight=1)
G7.add_edge(298,383,weight=1)
G7.add_edge(298,7,weight=1)
G7.add_edge(298,294,weight=1)
G7.add_edge(298,8,weight=1)
G7.add_edge(298,68,weight=1)
G7.add_edge(298,187,weight=1)
G7.add_edge(298,351,weight=1)
G7.add_edge(298,65,weight=1)
G7.add_edge(298,32,weight=1)
G7.add_edge(298,60,weight=1)
G7.add_edge(300,11,weight=1)
G7.add_edge(300,112,weight=1)
G7.add_edge(300,363,weight=1)
G7.add_edge(300,345,weight=1)
G7.add_edge(300,102,weight=1)
G7.add_edge(300,383,weight=1)
G7.add_edge(300,3,weight=1)
G7.add_edge(300,65,weight=1)
G7.add_edge(300,384,weight=1)
G7.add_edge(300,294,weight=1)
G7.add_edge(300,167,weight=1)
G7.add_edge(300,350,weight=1)
G7.add_edge(332,333,weight=1)
G7.add_edge(332,3,weight=1)
G7.add_edge(332,11,weight=1)
G7.add_edge(332,385,weight=1)
G7.add_edge(332,386,weight=1)
G7.add_edge(332,206,weight=1)
G7.add_edge(332,387,weight=1)
G7.add_edge(3,85,weight=1)
G7.add_edge(3,82,weight=1)
G7.add_edge(3,88,weight=1)
G7.add_edge(3,6,weight=1)
G7.add_edge(3,4,weight=1)
G7.add_edge(3,181,weight=1)
G7.add_edge(3,132,weight=1)
G7.add_edge(3,138,weight=1)
G7.add_edge(3,136,weight=1)
G7.add_edge(3,357,weight=1)
G7.add_edge(3,388,weight=1)
G7.add_edge(3,68,weight=1)
G7.add_edge(3,65,weight=1)
G7.add_edge(3,7,weight=1)
G7.add_edge(3,8,weight=1)
G7.add_edge(3,223,weight=1)
G7.add_edge(3,24,weight=1)
G7.add_edge(3,11,weight=1)
G7.add_edge(3,225,weight=1)
G7.add_edge(3,363,weight=1)
G7.add_edge(3,274,weight=1)
G7.add_edge(3,171,weight=1)
G7.add_edge(3,389,weight=1)
G7.add_edge(3,390,weight=1)
G7.add_edge(3,391,weight=1)
G7.add_edge(3,345,weight=1)
G7.add_edge(3,366,weight=1)
G7.add_edge(3,102,weight=1)
G7.add_edge(3,231,weight=1)
G7.add_edge(3,364,weight=1)
G7.add_edge(3,392,weight=1)
G7.add_edge(3,230,weight=1)
G7.add_edge(3,130,weight=1)
G7.add_edge(3,333,weight=1)
G7.add_edge(3,55,weight=1)
G7.add_edge(3,52,weight=1)
G7.add_edge(3,54,weight=1)
G7.add_edge(3,340,weight=1)
G7.add_edge(3,393,weight=1)
G7.add_edge(3,294,weight=1)
G7.add_edge(3,346,weight=1)
G7.add_edge(3,2,weight=1)
G7.add_edge(3,394,weight=1)
G7.add_edge(3,159,weight=1)
G7.add_edge(3,395,weight=1)
G7.add_edge(3,206,weight=1)
G7.add_edge(3,396,weight=1)
G7.add_edge(3,397,weight=1)
G7.add_edge(3,347,weight=1)
G7.add_edge(3,167,weight=1)
G7.add_edge(3,310,weight=1)
G7.add_edge(3,222,weight=1)
G7.add_edge(3,71,weight=1)
G7.add_edge(3,398,weight=1)
G7.add_edge(3,246,weight=1)
G7.add_edge(3,399,weight=1)
G7.add_edge(3,188,weight=1)
G7.add_edge(3,353,weight=1)
G7.add_edge(3,312,weight=1)
G7.add_edge(3,241,weight=1)
G7.add_edge(3,383,weight=1)
G7.add_edge(3,400,weight=1)
G7.add_edge(3,401,weight=1)
G7.add_edge(3,402,weight=1)
G7.add_edge(3,403,weight=1)
G7.add_edge(3,352,weight=1)
G7.add_edge(3,314,weight=1)
G7.add_edge(3,404,weight=1)
G7.add_edge(3,117,weight=1)
G7.add_edge(3,405,weight=1)
G7.add_edge(3,406,weight=1)
G7.add_edge(3,57,weight=1)
G7.add_edge(3,70,weight=1)
G7.add_edge(3,407,weight=1)
G7.add_edge(3,315,weight=1)
G7.add_edge(3,56,weight=1)
G7.add_edge(3,408,weight=1)
G7.add_edge(3,409,weight=1)
G7.add_edge(3,319,weight=1)
G7.add_edge(3,285,weight=1)
G7.add_edge(3,410,weight=1)
G7.add_edge(3,198,weight=1)
G7.add_edge(3,411,weight=1)
G7.add_edge(3,378,weight=1)
G7.add_edge(3,326,weight=1)
G7.add_edge(3,32,weight=1)
G7.add_edge(3,236,weight=1)
G7.add_edge(3,13,weight=1)
G7.add_edge(3,412,weight=1)
G7.add_edge(3,187,weight=1)
G7.add_edge(3,142,weight=1)
G7.add_edge(3,302,weight=1)
G7.add_edge(3,384,weight=1)
G7.add_edge(3,413,weight=1)
G7.add_edge(3,414,weight=1)
G7.add_edge(3,59,weight=1)
G7.add_edge(3,60,weight=1)
G7.add_edge(3,386,weight=1)
G7.add_edge(3,72,weight=1)
G7.add_edge(3,415,weight=1)
G7.add_edge(3,220,weight=1)
G7.add_edge(3,292,weight=1)
G7.add_edge(3,221,weight=1)
G7.add_edge(3,270,weight=1)
G7.add_edge(3,271,weight=1)
G7.add_edge(3,350,weight=1)
G7.add_edge(3,351,weight=1)
G7.add_edge(3,53,weight=1)
G7.add_edge(3,365,weight=1)
G7.add_edge(3,372,weight=1)
G7.add_edge(3,17,weight=1)
G7.add_edge(3,293,weight=1)
G7.add_edge(3,207,weight=1)
G7.add_edge(3,416,weight=1)
G7.add_edge(3,417,weight=1)
G7.add_edge(3,418,weight=1)
G7.add_edge(3,419,weight=1)
G7.add_edge(3,420,weight=1)
G7.add_edge(3,421,weight=1)
G7.add_edge(3,422,weight=1)
G7.add_edge(3,423,weight=1)
G7.add_edge(3,335,weight=1)
G7.add_edge(3,92,weight=1)
G7.add_edge(3,242,weight=1)
G7.add_edge(3,239,weight=1)
G7.add_edge(3,424,weight=1)
G7.add_edge(3,425,weight=1)
G7.add_edge(3,376,weight=1)
G7.add_edge(3,209,weight=1)
G7.add_edge(3,426,weight=1)
G7.add_edge(3,427,weight=1)
G7.add_edge(3,149,weight=1)
G7.add_edge(3,151,weight=1)
G7.add_edge(3,145,weight=1)
G7.add_edge(3,146,weight=1)
G7.add_edge(3,266,weight=1)
G7.add_edge(3,428,weight=1)
G7.add_edge(3,429,weight=1)
G7.add_edge(3,14,weight=1)
G7.add_edge(3,430,weight=1)
G7.add_edge(3,325,weight=1)
G7.add_edge(3,431,weight=1)
G7.add_edge(3,129,weight=1)
G7.add_edge(3,432,weight=1)
G7.add_edge(3,433,weight=1)
G7.add_edge(386,206,weight=1)
G7.add_edge(386,337,weight=1)
G7.add_edge(386,434,weight=1)
G7.add_edge(386,72,weight=1)
G7.add_edge(386,415,weight=1)
G7.add_edge(386,425,weight=1)
G7.add_edge(386,365,weight=1)
G7.add_edge(386,426,weight=1)
G7.add_edge(386,427,weight=1)
G7.add_edge(386,71,weight=1)
G7.add_edge(386,385,weight=1)
G7.add_edge(294,68,weight=1)
G7.add_edge(294,7,weight=1)
G7.add_edge(294,8,weight=1)
G7.add_edge(294,167,weight=1)
G7.add_edge(294,350,weight=1)
G7.add_edge(294,11,weight=1)
G7.add_edge(294,6,weight=1)
G7.add_edge(294,4,weight=1)
G7.add_edge(294,351,weight=1)
G7.add_edge(435,374,weight=1)
G7.add_edge(435,375,weight=1)
G7.add_edge(435,117,weight=1)
G7.add_edge(266,68,weight=1)
G7.add_edge(388,6,weight=1)
G7.add_edge(388,4,weight=1)
G7.add_edge(388,130,weight=1)
G7.add_edge(388,357,weight=1)
G7.add_edge(436,391,weight=1)
G7.add_edge(436,71,weight=1)
G7.add_edge(64,65,weight=1)
G7.add_edge(64,365,weight=1)
G7.add_edge(64,71,weight=1)
G7.add_edge(64,117,weight=1)
G7.add_edge(64,24,weight=1)
G7.add_edge(64,100,weight=1)
G7.add_edge(64,358,weight=1)
G7.add_edge(64,235,weight=1)
G7.add_edge(64,92,weight=1)
G7.add_edge(417,416,weight=1)
G7.add_edge(417,418,weight=1)
G7.add_edge(394,159,weight=1)
G7.add_edge(394,395,weight=1)
G7.add_edge(394,349,weight=1)
G7.add_edge(394,171,weight=1)
G7.add_edge(349,167,weight=1)
G7.add_edge(349,348,weight=1)
G7.add_edge(349,32,weight=1)
G7.add_edge(349,171,weight=1)
G7.add_edge(347,167,weight=1)
G7.add_edge(347,68,weight=1)
G7.add_edge(347,187,weight=1)
G7.add_edge(347,65,weight=1)
G7.add_edge(347,32,weight=1)
G7.add_edge(167,68,weight=1)
G7.add_edge(167,65,weight=1)
G7.add_edge(167,231,weight=1)
G7.add_edge(167,187,weight=1)
G7.add_edge(167,32,weight=1)
G7.add_edge(167,348,weight=1)
G7.add_edge(167,314,weight=1)
G7.add_edge(167,142,weight=1)
G7.add_edge(167,11,weight=1)
G7.add_edge(167,171,weight=1)
G7.add_edge(167,350,weight=1)
G7.add_edge(278,7,weight=1)
G7.add_edge(278,8,weight=1)
G7.add_edge(278,11,weight=1)
G7.add_edge(278,220,weight=1)
G7.add_edge(278,4,weight=1)
G7.add_edge(278,6,weight=1)
G7.add_edge(348,159,weight=1)
G7.add_edge(348,314,weight=1)
G7.add_edge(348,11,weight=1)
G7.add_edge(348,32,weight=1)
G7.add_edge(221,292,weight=1)
G7.add_edge(221,24,weight=1)
G7.add_edge(221,70,weight=1)
G7.add_edge(221,68,weight=1)
G7.add_edge(70,68,weight=1)
G7.add_edge(70,407,weight=1)
G7.add_edge(70,405,weight=1)
G7.add_edge(70,117,weight=1)
G7.add_edge(70,65,weight=1)
G7.add_edge(70,11,weight=1)
G7.add_edge(70,367,weight=1)
G7.add_edge(70,345,weight=1)
G7.add_edge(70,24,weight=1)
G7.add_edge(404,68,weight=1)
G7.add_edge(404,117,weight=1)
G7.add_edge(404,405,weight=1)
G7.add_edge(428,25,weight=1)
G7.add_edge(428,206,weight=1)
G7.add_edge(428,32,weight=1)
G7.add_edge(428,65,weight=1)
G7.add_edge(428,429,weight=1)
G7.add_edge(65,100,weight=1)
G7.add_edge(65,187,weight=1)
G7.add_edge(65,4,weight=1)
G7.add_edge(65,6,weight=1)
G7.add_edge(65,68,weight=1)
G7.add_edge(65,7,weight=1)
G7.add_edge(65,8,weight=1)
G7.add_edge(65,223,weight=1)
G7.add_edge(65,225,weight=1)
G7.add_edge(65,191,weight=1)
G7.add_edge(65,11,weight=1)
G7.add_edge(65,290,weight=1)
G7.add_edge(65,159,weight=1)
G7.add_edge(65,2,weight=1)
G7.add_edge(65,231,weight=1)
G7.add_edge(65,32,weight=1)
G7.add_edge(65,275,weight=1)
G7.add_edge(65,295,weight=1)
G7.add_edge(65,117,weight=1)
G7.add_edge(65,232,weight=1)
G7.add_edge(65,233,weight=1)
G7.add_edge(65,178,weight=1)
G7.add_edge(65,314,weight=1)
G7.add_edge(65,437,weight=1)
G7.add_edge(65,438,weight=1)
G7.add_edge(65,329,weight=1)
G7.add_edge(65,384,weight=1)
G7.add_edge(65,351,weight=1)
G7.add_edge(65,234,weight=1)
G7.add_edge(65,92,weight=1)
G7.add_edge(65,235,weight=1)
G7.add_edge(65,429,weight=1)
G7.add_edge(65,31,weight=1)
G7.add_edge(329,7,weight=1)
G7.add_edge(329,2,weight=1)
G7.add_edge(329,8,weight=1)
G7.add_edge(329,11,weight=1)
G7.add_edge(2,159,weight=1)
G7.add_edge(2,6,weight=1)
G7.add_edge(2,4,weight=1)
G7.add_edge(2,7,weight=1)
G7.add_edge(2,8,weight=1)
G7.add_edge(2,11,weight=1)
G7.add_edge(187,4,weight=1)
G7.add_edge(187,6,weight=1)
G7.add_edge(187,32,weight=1)
G7.add_edge(187,275,weight=1)
G7.add_edge(187,295,weight=1)
G7.add_edge(187,117,weight=1)
G7.add_edge(187,314,weight=1)
G7.add_edge(187,11,weight=1)
G7.add_edge(187,68,weight=1)
G7.add_edge(187,351,weight=1)
G7.add_edge(187,31,weight=1)
G7.add_edge(439,68,weight=1)
G7.add_edge(439,270,weight=1)
G7.add_edge(292,24,weight=1)
G7.add_edge(292,209,weight=1)
G7.add_edge(292,55,weight=1)
G7.add_edge(292,293,weight=1)
G7.add_edge(220,7,weight=1)
G7.add_edge(220,8,weight=1)
G7.add_edge(220,4,weight=1)
G7.add_edge(220,6,weight=1)
G7.add_edge(220,310,weight=1)
G7.add_edge(220,68,weight=1)
G7.add_edge(100,117,weight=1)
G7.add_edge(100,358,weight=1)
G7.add_edge(366,171,weight=1)
G7.add_edge(366,6,weight=1)
G7.add_edge(366,4,weight=1)
G7.add_edge(366,231,weight=1)
G7.add_edge(366,102,weight=1)
G7.add_edge(366,365,weight=1)
G7.add_edge(293,207,weight=1)
G7.add_edge(293,11,weight=1)
G7.add_edge(293,32,weight=1)
G7.add_edge(159,395,weight=1)
G7.add_edge(159,314,weight=1)
G7.add_edge(159,11,weight=1)
G7.add_edge(5,4,weight=1)
G7.add_edge(5,6,weight=1)
G7.add_edge(5,7,weight=1)
G7.add_edge(5,8,weight=1)
G7.add_edge(5,304,weight=1)
G7.add_edge(418,416,weight=1)
G7.add_edge(24,223,weight=1)
G7.add_edge(24,11,weight=1)
G7.add_edge(24,71,weight=1)
G7.add_edge(24,25,weight=1)
G7.add_edge(24,117,weight=1)
G7.add_edge(24,102,weight=1)
G7.add_edge(24,367,weight=1)
G7.add_edge(24,249,weight=1)
G7.add_edge(24,68,weight=1)
G7.add_edge(24,407,weight=1)
G7.add_edge(222,310,weight=1)
G7.add_edge(222,11,weight=1)
G7.add_edge(222,391,weight=1)
G7.add_edge(222,71,weight=1)
G7.add_edge(222,68,weight=1)
G7.add_edge(25,206,weight=1)
G7.add_edge(25,32,weight=1)
G7.add_edge(233,232,weight=1)
G7.add_edge(235,92,weight=1)
G7.add_edge(235,117,weight=1)
G7.add_edge(279,160,weight=1)
G7.add_edge(279,162,weight=1)
G7.add_edge(426,206,weight=1)
G7.add_edge(426,427,weight=1)
G7.add_edge(426,71,weight=1)
G7.add_edge(6,4,weight=1)
G7.add_edge(6,181,weight=1)
G7.add_edge(6,130,weight=1)
G7.add_edge(6,68,weight=1)
G7.add_edge(6,225,weight=1)
G7.add_edge(6,363,weight=1)
G7.add_edge(6,231,weight=1)
G7.add_edge(6,230,weight=1)
G7.add_edge(6,356,weight=1)
G7.add_edge(6,232,weight=1)
G7.add_edge(6,391,weight=1)
G7.add_edge(6,345,weight=1)
G7.add_edge(6,440,weight=1)
G7.add_edge(6,13,weight=1)
G7.add_edge(6,412,weight=1)
G7.add_edge(6,425,weight=1)
G7.add_edge(6,365,weight=1)
G7.add_edge(6,11,weight=1)
G7.add_edge(6,246,weight=1)
G7.add_edge(6,34,weight=1)
G7.add_edge(6,143,weight=1)
G7.add_edge(6,351,weight=1)
G7.add_edge(6,140,weight=1)
G7.add_edge(4,181,weight=1)
G7.add_edge(4,130,weight=1)
G7.add_edge(4,68,weight=1)
G7.add_edge(4,225,weight=1)
G7.add_edge(4,363,weight=1)
G7.add_edge(4,231,weight=1)
G7.add_edge(4,230,weight=1)
G7.add_edge(4,356,weight=1)
G7.add_edge(4,232,weight=1)
G7.add_edge(4,391,weight=1)
G7.add_edge(4,345,weight=1)
G7.add_edge(4,440,weight=1)
G7.add_edge(4,13,weight=1)
G7.add_edge(4,412,weight=1)
G7.add_edge(4,425,weight=1)
G7.add_edge(4,365,weight=1)
G7.add_edge(4,11,weight=1)
G7.add_edge(4,246,weight=1)
G7.add_edge(4,34,weight=1)
G7.add_edge(4,143,weight=1)
G7.add_edge(4,351,weight=1)
G7.add_edge(4,140,weight=1)
G7.add_edge(8,7,weight=1)
G7.add_edge(8,68,weight=1)
G7.add_edge(8,225,weight=1)
G7.add_edge(8,363,weight=1)
G7.add_edge(8,391,weight=1)
G7.add_edge(8,345,weight=1)
G7.add_edge(8,11,weight=1)
G7.add_edge(8,206,weight=1)
G7.add_edge(8,254,weight=1)
G7.add_edge(8,246,weight=1)
G7.add_edge(8,34,weight=1)
G7.add_edge(8,230,weight=1)
G7.add_edge(8,197,weight=1)
G7.add_edge(8,57,weight=1)
G7.add_edge(7,68,weight=1)
G7.add_edge(7,225,weight=1)
G7.add_edge(7,363,weight=1)
G7.add_edge(7,391,weight=1)
G7.add_edge(7,345,weight=1)
G7.add_edge(7,11,weight=1)
G7.add_edge(7,206,weight=1)
G7.add_edge(7,254,weight=1)
G7.add_edge(7,246,weight=1)
G7.add_edge(7,34,weight=1)
G7.add_edge(7,230,weight=1)
G7.add_edge(7,197,weight=1)
G7.add_edge(7,57,weight=1)
G7.add_edge(68,55,weight=1)
G7.add_edge(68,52,weight=1)
G7.add_edge(68,393,weight=1)
G7.add_edge(68,345,weight=1)
G7.add_edge(68,275,weight=1)
G7.add_edge(68,295,weight=1)
G7.add_edge(68,117,weight=1)
G7.add_edge(68,314,weight=1)
G7.add_edge(68,11,weight=1)
G7.add_edge(68,405,weight=1)
G7.add_edge(68,406,weight=1)
G7.add_edge(68,57,weight=1)
G7.add_edge(68,407,weight=1)
G7.add_edge(68,315,weight=1)
G7.add_edge(68,56,weight=1)
G7.add_edge(68,408,weight=1)
G7.add_edge(68,409,weight=1)
G7.add_edge(68,319,weight=1)
G7.add_edge(68,285,weight=1)
G7.add_edge(68,410,weight=1)
G7.add_edge(68,198,weight=1)
G7.add_edge(68,171,weight=1)
G7.add_edge(68,437,weight=1)
G7.add_edge(68,438,weight=1)
G7.add_edge(68,288,weight=1)
G7.add_edge(68,289,weight=1)
G7.add_edge(68,270,weight=1)
G7.add_edge(68,31,weight=1)
G7.add_edge(206,276,weight=1)
G7.add_edge(206,232,weight=1)
G7.add_edge(206,396,weight=1)
G7.add_edge(206,397,weight=1)
G7.add_edge(206,337,weight=1)
G7.add_edge(206,434,weight=1)
G7.add_edge(206,425,weight=1)
G7.add_edge(206,365,weight=1)
G7.add_edge(206,427,weight=1)
G7.add_edge(206,71,weight=1)
G7.add_edge(206,385,weight=1)
G7.add_edge(206,32,weight=1)
G7.add_edge(341,342,weight=1)
G7.add_edge(341,343,weight=1)
G7.add_edge(343,196,weight=1)
G7.add_edge(14,441,weight=1)
G7.add_edge(14,442,weight=1)
G7.add_edge(14,61,weight=1)
G7.add_edge(14,31,weight=1)
G7.add_edge(14,13,weight=1)
G7.add_edge(14,32,weight=1)
G7.add_edge(14,443,weight=1)
G7.add_edge(14,444,weight=1)
G7.add_edge(14,430,weight=1)
G7.add_edge(14,325,weight=1)
G7.add_edge(14,431,weight=1)
G7.add_edge(14,239,weight=1)
G7.add_edge(14,241,weight=1)
G7.add_edge(14,353,weight=1)
G7.add_edge(14,344,weight=1)
G7.add_edge(412,445,weight=1)
G7.add_edge(412,13,weight=1)
G7.add_edge(441,354,weight=1)
G7.add_edge(441,344,weight=1)
G7.add_edge(441,442,weight=1)
G7.add_edge(441,61,weight=1)
G7.add_edge(20,13,weight=1)
G7.add_edge(13,31,weight=1)
G7.add_edge(13,32,weight=1)
G7.add_edge(13,443,weight=1)
G7.add_edge(13,444,weight=1)
G7.add_edge(344,354,weight=1)
G7.add_edge(344,442,weight=1)
G7.add_edge(344,296,weight=1)
G7.add_edge(344,56,weight=1)
G7.add_edge(31,32,weight=1)
G7.add_edge(31,11,weight=1)
G7.add_edge(351,32,weight=1)
G7.add_edge(351,432,weight=1)
G7.add_edge(351,433,weight=1)
G7.add_edge(433,432,weight=1)
G7.add_edge(176,71,weight=1)
G7.add_edge(176,11,weight=1)
G7.add_edge(61,442,weight=1)
G7.add_edge(164,295,weight=1)
G7.add_edge(164,32,weight=1)
G7.add_edge(422,419,weight=1)
G7.add_edge(422,423,weight=1)
G7.add_edge(420,419,weight=1)
G7.add_edge(420,421,weight=1)
G7.add_edge(421,419,weight=1)
G7.add_edge(423,419,weight=1)
G7.add_edge(401,400,weight=1)
G7.add_edge(401,402,weight=1)
G7.add_edge(442,354,weight=1)
G7.add_edge(334,335,weight=1)
G7.add_edge(335,345,weight=1)
G7.add_edge(142,314,weight=1)
G7.add_edge(142,11,weight=1)
G7.add_edge(438,437,weight=1)
G7.add_edge(369,249,weight=1)
G7.add_edge(369,368,weight=1)
G7.add_edge(359,360,weight=1)
G7.add_edge(264,236,weight=1)
G7.add_edge(264,11,weight=1)
G7.add_edge(264,275,weight=1)
G7.add_edge(367,102,weight=1)
G7.add_edge(367,407,weight=1)
G7.add_edge(367,345,weight=1)
G7.add_edge(223,11,weight=1)
G7.add_edge(202,358,weight=1)
G7.add_edge(202,228,weight=1)
G7.add_edge(202,203,weight=1)
G7.add_edge(361,228,weight=1)
G7.add_edge(361,117,weight=1)
G7.add_edge(361,102,weight=1)
G7.add_edge(361,203,weight=1)
G7.add_edge(209,207,weight=1)
G7.add_edge(209,55,weight=1)
G7.add_edge(207,11,weight=1)
G7.add_edge(207,32,weight=1)
G7.add_edge(160,162,weight=1)
G7.add_edge(288,289,weight=1)
G7.add_edge(362,92,weight=1)
G7.add_edge(362,228,weight=1)
G7.add_edge(362,203,weight=1)
G7.add_edge(362,370,weight=1)
G7.add_edge(229,203,weight=1)
G7.add_edge(229,112,weight=1)
G7.add_edge(229,228,weight=1)
G7.add_edge(256,32,weight=1)
G7.add_edge(392,364,weight=1)
G7.add_edge(392,230,weight=1)
G7.add_edge(260,85,weight=1)
G7.add_edge(260,102,weight=1)
G7.add_edge(190,191,weight=1)
G7.add_edge(190,32,weight=1)
G7.add_edge(232,276,weight=1)
G7.add_edge(232,356,weight=1)
G7.add_edge(390,389,weight=1)
G7.add_edge(390,11,weight=1)
G7.add_edge(389,11,weight=1)
G7.add_edge(246,398,weight=1)
G7.add_edge(246,399,weight=1)
G7.add_edge(246,188,weight=1)
G7.add_edge(246,32,weight=1)
G7.add_edge(246,34,weight=1)
G7.add_edge(357,258,weight=1)
G7.add_edge(357,331,weight=1)
G7.add_edge(82,259,weight=1)
G7.add_edge(132,262,weight=1)
G7.add_edge(138,249,weight=1)
G7.add_edge(136,261,weight=1)
G7.add_edge(88,257,weight=1)
G7.add_edge(391,345,weight=1)
G7.add_edge(391,71,weight=1)
G7.add_edge(391,440,weight=1)
G7.add_edge(393,345,weight=1)
G7.add_edge(345,102,weight=1)
G7.add_edge(345,32,weight=1)
G7.add_edge(345,11,weight=1)
G7.add_edge(345,407,weight=1)
G7.add_edge(274,171,weight=1)
G7.add_edge(130,102,weight=1)
G7.add_edge(102,262,weight=1)
G7.add_edge(102,261,weight=1)
G7.add_edge(102,259,weight=1)
G7.add_edge(102,257,weight=1)
G7.add_edge(102,258,weight=1)
G7.add_edge(102,32,weight=1)
G7.add_edge(102,231,weight=1)
G7.add_edge(102,364,weight=1)
G7.add_edge(102,365,weight=1)
G7.add_edge(102,11,weight=1)
G7.add_edge(102,140,weight=1)
G7.add_edge(102,92,weight=1)
G7.add_edge(102,203,weight=1)
G7.add_edge(54,55,weight=1)
G7.add_edge(54,11,weight=1)
G7.add_edge(54,340,weight=1)
G7.add_edge(54,32,weight=1)
G7.add_edge(55,52,weight=1)
G7.add_edge(55,11,weight=1)
G7.add_edge(55,340,weight=1)
G7.add_edge(240,239,weight=1)
G7.add_edge(240,241,weight=1)
G7.add_edge(240,242,weight=1)
G7.add_edge(350,171,weight=1)
G7.add_edge(350,11,weight=1)
G7.add_edge(243,352,weight=1)
G7.add_edge(243,32,weight=1)
G7.add_edge(295,275,weight=1)
G7.add_edge(295,117,weight=1)
G7.add_edge(295,32,weight=1)
G7.add_edge(396,397,weight=1)
G7.add_edge(333,140,weight=1)
G7.add_edge(333,111,weight=1)
G7.add_edge(333,11,weight=1)
G7.add_edge(333,117,weight=1)
G7.add_edge(333,231,weight=1)
G7.add_edge(398,399,weight=1)
G7.add_edge(383,11,weight=1)
G7.add_edge(314,11,weight=1)
G7.add_edge(151,149,weight=1)
G7.add_edge(151,188,weight=1)
G7.add_edge(402,400,weight=1)
G7.add_edge(400,403,weight=1)
G7.add_edge(400,352,weight=1)
G7.add_edge(403,352,weight=1)
G7.add_edge(275,117,weight=1)
G7.add_edge(275,356,weight=1)
G7.add_edge(275,11,weight=1)
G7.add_edge(364,231,weight=1)
G7.add_edge(315,56,weight=1)
G7.add_edge(406,57,weight=1)
G7.add_edge(316,317,weight=1)
G7.add_edge(316,318,weight=1)
G7.add_edge(312,353,weight=1)
G7.add_edge(312,241,weight=1)
G7.add_edge(312,313,weight=1)
G7.add_edge(312,319,weight=1)
G7.add_edge(319,285,weight=1)
G7.add_edge(320,321,weight=1)
G7.add_edge(320,322,weight=1)
G7.add_edge(323,324,weight=1)
G7.add_edge(323,287,weight=1)
G7.add_edge(378,411,weight=1)
G7.add_edge(287,196,weight=1)
G7.add_edge(287,285,weight=1)
G7.add_edge(325,326,weight=1)
G7.add_edge(325,285,weight=1)
G7.add_edge(325,430,weight=1)
G7.add_edge(325,431,weight=1)
G7.add_edge(325,239,weight=1)
G7.add_edge(325,241,weight=1)
G7.add_edge(325,353,weight=1)
G7.add_edge(285,198,weight=1)
G7.add_edge(285,326,weight=1)
G7.add_edge(285,32,weight=1)
G7.add_edge(326,32,weight=1)
G7.add_edge(408,409,weight=1)
G7.add_edge(410,198,weight=1)
G7.add_edge(195,11,weight=1)
G7.add_edge(195,60,weight=1)
G7.add_edge(198,57,weight=1)
G7.add_edge(198,11,weight=1)
G7.add_edge(203,112,weight=1)
G7.add_edge(203,370,weight=1)
G7.add_edge(414,413,weight=1)
G7.add_edge(414,11,weight=1)
G7.add_edge(414,192,weight=1)
G7.add_edge(414,446,weight=1)
G7.add_edge(413,384,weight=1)
G7.add_edge(413,32,weight=1)
G7.add_edge(413,11,weight=1)
G7.add_edge(384,32,weight=1)
G7.add_edge(192,446,weight=1)
G7.add_edge(191,11,weight=1)
G7.add_edge(191,32,weight=1)
G7.add_edge(447,106,weight=1)
G7.add_edge(447,448,weight=1)
G7.add_edge(447,32,weight=1)
G7.add_edge(44,354,weight=1)
G7.add_edge(44,43,weight=1)
G7.add_edge(44,73,weight=1)
G7.add_edge(44,76,weight=1)
G7.add_edge(43,73,weight=1)
G7.add_edge(449,32,weight=1)
G7.add_edge(146,145,weight=1)
G7.add_edge(37,11,weight=1)
G7.add_edge(37,38,weight=1)
G7.add_edge(374,375,weight=1)
G7.add_edge(374,117,weight=1)
G7.add_edge(375,117,weight=1)
G7.add_edge(427,71,weight=1)
G7.add_edge(178,11,weight=1)
G7.add_edge(171,363,weight=1)
G7.add_edge(448,106,weight=1)
G7.add_edge(448,32,weight=1)
G7.add_edge(450,106,weight=1)
G7.add_edge(450,451,weight=1)
G7.add_edge(330,331,weight=1)
G7.add_edge(330,11,weight=1)
G7.add_edge(430,431,weight=1)
G7.add_edge(331,11,weight=1)
G7.add_edge(72,71,weight=1)
G7.add_edge(72,11,weight=1)
G7.add_edge(72,415,weight=1)
G7.add_edge(108,452,weight=1)
G7.add_edge(188,399,weight=1)
G7.add_edge(71,365,weight=1)
G7.add_edge(71,117,weight=1)
G7.add_edge(71,11,weight=1)
G7.add_edge(453,444,weight=1)
G7.add_edge(443,444,weight=1)
G7.add_edge(425,365,weight=1)
G7.add_edge(425,424,weight=1)
G7.add_edge(425,11,weight=1)
G7.add_edge(365,117,weight=1)
G7.add_edge(365,230,weight=1)
G7.add_edge(365,363,weight=1)
G7.add_edge(60,11,weight=1)
G7.add_edge(60,59,weight=1)
G7.add_edge(60,340,weight=1)
G7.add_edge(60,32,weight=1)
G7.add_edge(59,340,weight=1)
G7.add_edge(59,11,weight=1)
G7.add_edge(53,52,weight=1)
G7.add_edge(53,11,weight=1)
G7.add_edge(53,340,weight=1)
G7.add_edge(52,11,weight=1)
G7.add_edge(52,340,weight=1)
G7.add_edge(370,129,weight=1)
G7.add_edge(230,363,weight=1)
G7.add_edge(106,11,weight=1)
G7.add_edge(106,32,weight=1)
G7.add_edge(106,451,weight=1)
G7.add_edge(358,117,weight=1)
G7.add_edge(358,228,weight=1)
G7.add_edge(92,117,weight=1)
G7.add_edge(92,228,weight=1)
G7.add_edge(372,17,weight=1)
G7.add_edge(228,117,weight=1)
G7.add_edge(111,140,weight=1)
G7.add_edge(111,11,weight=1)
G7.add_edge(143,296,weight=1)
G7.add_edge(143,32,weight=1)
G7.add_edge(296,32,weight=1)
G7.add_edge(11,112,weight=1)
G7.add_edge(11,363,weight=1)
G7.add_edge(11,290,weight=1)
G7.add_edge(11,310,weight=1)
G7.add_edge(11,56,weight=1)
G7.add_edge(11,57,weight=1)
G7.add_edge(11,236,weight=1)
G7.add_edge(11,38,weight=1)
G7.add_edge(11,140,weight=1)
G7.add_edge(11,32,weight=1)
G7.add_edge(11,424,weight=1)
G7.add_edge(11,117,weight=1)
G7.add_edge(11,231,weight=1)
G7.add_edge(11,196,weight=1)
G7.add_edge(11,197,weight=1)
G7.add_edge(231,117,weight=1)
G7.add_edge(241,353,weight=1)
G7.add_edge(241,239,weight=1)
G7.add_edge(241,242,weight=1)
G7.add_edge(241,407,weight=1)
G7.add_edge(181,249,weight=1)
G7.add_edge(181,32,weight=1)
G7.add_edge(249,32,weight=1)
G7.add_edge(249,368,weight=1)
G7.add_edge(236,117,weight=1)
G7.add_edge(376,32,weight=1)
G7.add_edge(434,337,weight=1)
G7.add_edge(336,337,weight=1)
G7.add_edge(32,352,weight=1)
G7.add_edge(117,405,weight=1)
G7.add_edge(117,407,weight=1)
G7.add_edge(363,225,weight=1)
G7.add_edge(363,112,weight=1)
G7.add_edge(407,405,weight=1)
G7.add_edge(407,242,weight=1)
G7.add_edge(407,239,weight=1)
G7.add_edge(242,239,weight=1)
G7.add_edge(196,197,weight=1)
G7.add_edge(57,56,weight=1)




#Cora
G8 = nx.DiGraph()
G8.add_edge(164,403,weight=1)
G8.add_edge(164,660,weight=1)
G8.add_edge(164,1697,weight=1)
G8.add_edge(164,2296,weight=1)
G8.add_edge(164,1275,weight=1)
G8.add_edge(164,1287,weight=1)
G8.add_edge(164,1545,weight=1)
G8.add_edge(164,2601,weight=1)
G8.add_edge(164,2364,weight=1)
G8.add_edge(164,1906,weight=1)
G8.add_edge(164,1612,weight=1)
G8.add_edge(164,142,weight=1)
G8.add_edge(164,1808,weight=1)
G8.add_edge(164,1111,weight=1)
G8.add_edge(164,175,weight=1)
G8.add_edge(164,2522,weight=1)
G8.add_edge(164,1793,weight=1)
G8.add_edge(164,1676,weight=1)
G8.add_edge(164,1335,weight=1)
G8.add_edge(164,814,weight=1)
G8.add_edge(164,1800,weight=1)
G8.add_edge(164,1944,weight=1)
G8.add_edge(164,2078,weight=1)
G8.add_edge(164,766,weight=1)
G8.add_edge(164,770,weight=1)
G8.add_edge(164,782,weight=1)
G8.add_edge(164,941,weight=1)
G8.add_edge(164,943,weight=1)
G8.add_edge(164,1591,weight=1)
G8.add_edge(164,1735,weight=1)
G8.add_edge(164,1873,weight=1)
G8.add_edge(164,2287,weight=1)
G8.add_edge(164,391,weight=1)
G8.add_edge(164,1718,weight=1)
G8.add_edge(164,1031,weight=1)
G8.add_edge(164,2275,weight=1)
G8.add_edge(164,2519,weight=1)
G8.add_edge(164,607,weight=1)
G8.add_edge(164,801,weight=1)
G8.add_edge(164,1576,weight=1)
G8.add_edge(164,547,weight=1)
G8.add_edge(164,1071,weight=1)
G8.add_edge(164,310,weight=1)
G8.add_edge(164,936,weight=1)
G8.add_edge(164,1206,weight=1)
G8.add_edge(164,1572,weight=1)
G8.add_edge(164,1972,weight=1)
G8.add_edge(164,1128,weight=1)
G8.add_edge(164,531,weight=1)
G8.add_edge(164,857,weight=1)
G8.add_edge(164,2605,weight=1)
G8.add_edge(164,911,weight=1)
G8.add_edge(164,2174,weight=1)
G8.add_edge(164,192,weight=1)
G8.add_edge(164,1254,weight=1)
G8.add_edge(164,1729,weight=1)
G8.add_edge(164,1730,weight=1)
G8.add_edge(164,1207,weight=1)
G8.add_edge(164,2178,weight=1)
G8.add_edge(164,1137,weight=1)
G8.add_edge(164,1458,weight=1)
G8.add_edge(164,2266,weight=1)
G8.add_edge(164,1226,weight=1)
G8.add_edge(164,2564,weight=1)
G8.add_edge(164,1690,weight=1)
G8.add_edge(164,1499,weight=1)
G8.add_edge(164,564,weight=1)
G8.add_edge(164,2397,weight=1)
G8.add_edge(164,718,weight=1)
G8.add_edge(164,1891,weight=1)
G8.add_edge(164,189,weight=1)
G8.add_edge(164,983,weight=1)
G8.add_edge(164,1131,weight=1)
G8.add_edge(164,56,weight=1)
G8.add_edge(164,347,weight=1)
G8.add_edge(164,2674,weight=1)
G8.add_edge(164,1380,weight=1)
G8.add_edge(164,1381,weight=1)
G8.add_edge(164,1208,weight=1)
G8.add_edge(164,2452,weight=1)
G8.add_edge(164,962,weight=1)
G8.add_edge(164,43,weight=1)
G8.add_edge(164,625,weight=1)
G8.add_edge(164,1574,weight=1)
G8.add_edge(164,2233,weight=1)
G8.add_edge(164,146,weight=1)
G8.add_edge(164,2362,weight=1)
G8.add_edge(164,2660,weight=1)
G8.add_edge(164,744,weight=1)
G8.add_edge(164,745,weight=1)
G8.add_edge(164,1140,weight=1)
G8.add_edge(164,1573,weight=1)
G8.add_edge(164,1304,weight=1)
G8.add_edge(164,1306,weight=1)
G8.add_edge(164,130,weight=1)
G8.add_edge(164,416,weight=1)
G8.add_edge(164,2205,weight=1)
G8.add_edge(164,1225,weight=1)
G8.add_edge(164,220,weight=1)
G8.add_edge(164,2668,weight=1)
G8.add_edge(164,728,weight=1)
G8.add_edge(164,449,weight=1)
G8.add_edge(164,238,weight=1)
G8.add_edge(164,2558,weight=1)
G8.add_edge(164,2201,weight=1)
G8.add_edge(164,23,weight=1)
G8.add_edge(164,1219,weight=1)
G8.add_edge(164,2203,weight=1)
G8.add_edge(164,1070,weight=1)
G8.add_edge(164,1179,weight=1)
G8.add_edge(164,2176,weight=1)
G8.add_edge(164,967,weight=1)
G8.add_edge(164,1220,weight=1)
G8.add_edge(164,758,weight=1)
G8.add_edge(164,1686,weight=1)
G8.add_edge(164,1692,weight=1)
G8.add_edge(164,1851,weight=1)
G8.add_edge(164,1160,weight=1)
G8.add_edge(164,1411,weight=1)
G8.add_edge(164,381,weight=1)
G8.add_edge(164,1537,weight=1)
G8.add_edge(164,1651,weight=1)
G8.add_edge(164,1559,weight=1)
G8.add_edge(164,2565,weight=1)
G8.add_edge(164,2299,weight=1)
G8.add_edge(164,1363,weight=1)
G8.add_edge(164,1785,weight=1)
G8.add_edge(164,1832,weight=1)
G8.add_edge(164,1066,weight=1)
G8.add_edge(164,291,weight=1)
G8.add_edge(164,1278,weight=1)
G8.add_edge(164,396,weight=1)
G8.add_edge(164,1099,weight=1)
G8.add_edge(164,1405,weight=1)
G8.add_edge(164,1100,weight=1)
G8.add_edge(164,2197,weight=1)
G8.add_edge(164,2555,weight=1)
G8.add_edge(164,1716,weight=1)
G8.add_edge(164,2249,weight=1)
G8.add_edge(164,2252,weight=1)
G8.add_edge(164,2281,weight=1)
G8.add_edge(164,2297,weight=1)
G8.add_edge(164,1114,weight=1)
G8.add_edge(164,715,weight=1)
G8.add_edge(164,2639,weight=1)
G8.add_edge(164,603,weight=1)
G8.add_edge(164,1531,weight=1)
G8.add_edge(164,1258,weight=1)
G8.add_edge(164,2260,weight=1)
G8.add_edge(164,690,weight=1)
G8.add_edge(164,1061,weight=1)
G8.add_edge(164,2040,weight=1)
G8.add_edge(164,1595,weight=1)
G8.add_edge(164,2117,weight=1)
G8.add_edge(164,1776,weight=1)
G8.add_edge(164,423,weight=1)
G8.add_edge(164,2317,weight=1)
G8.add_edge(164,267,weight=1)
G8.add_edge(164,659,weight=1)
G8.add_edge(164,1334,weight=1)
G8.add_edge(164,1154,weight=1)
G8.add_edge(164,1516,weight=1)
G8.add_edge(164,1115,weight=1)
G8.add_edge(164,524,weight=1)
G8.add_edge(164,1107,weight=1)
G8.add_edge(164,1468,weight=1)
G8.add_edge(169,476,weight=1)
G8.add_edge(169,1757,weight=1)
G8.add_edge(169,2605,weight=1)
G8.add_edge(553,549,weight=1)
G8.add_edge(553,375,weight=1)
G8.add_edge(553,452,weight=1)
G8.add_edge(553,473,weight=1)
G8.add_edge(553,3,weight=1)
G8.add_edge(553,306,weight=1)
G8.add_edge(553,240,weight=1)
G8.add_edge(553,447,weight=1)
G8.add_edge(553,645,weight=1)
G8.add_edge(553,221,weight=1)
G8.add_edge(553,312,weight=1)
G8.add_edge(553,654,weight=1)
G8.add_edge(553,17,weight=1)
G8.add_edge(553,174,weight=1)
G8.add_edge(553,555,weight=1)
G8.add_edge(553,560,weight=1)
G8.add_edge(553,413,weight=1)
G8.add_edge(553,724,weight=1)
G8.add_edge(553,86,weight=1)
G8.add_edge(553,689,weight=1)
G8.add_edge(553,622,weight=1)
G8.add_edge(553,637,weight=1)
G8.add_edge(553,739,weight=1)
G8.add_edge(553,750,weight=1)
G8.add_edge(553,58,weight=1)
G8.add_edge(553,61,weight=1)
G8.add_edge(553,211,weight=1)
G8.add_edge(553,672,weight=1)
G8.add_edge(553,168,weight=1)
G8.add_edge(553,93,weight=1)
G8.add_edge(553,692,weight=1)
G8.add_edge(553,368,weight=1)
G8.add_edge(553,467,weight=1)
G8.add_edge(553,474,weight=1)
G8.add_edge(553,484,weight=1)
G8.add_edge(553,576,weight=1)
G8.add_edge(553,456,weight=1)
G8.add_edge(553,299,weight=1)
G8.add_edge(553,602,weight=1)
G8.add_edge(553,371,weight=1)
G8.add_edge(553,307,weight=1)
G8.add_edge(553,278,weight=1)
G8.add_edge(1460,2079,weight=1)
G8.add_edge(1460,2222,weight=1)
G8.add_edge(1460,1688,weight=1)
G8.add_edge(1460,2405,weight=1)
G8.add_edge(1460,776,weight=1)
G8.add_edge(1460,1541,weight=1)
G8.add_edge(1460,634,weight=1)
G8.add_edge(1460,750,weight=1)
G8.add_edge(1460,58,weight=1)
G8.add_edge(1460,46,weight=1)
G8.add_edge(1460,1138,weight=1)
G8.add_edge(1460,211,weight=1)
G8.add_edge(1460,672,weight=1)
G8.add_edge(1460,168,weight=1)
G8.add_edge(1460,474,weight=1)
G8.add_edge(1460,567,weight=1)
G8.add_edge(1460,805,weight=1)
G8.add_edge(555,447,weight=1)
G8.add_edge(555,1860,weight=1)
G8.add_edge(555,168,weight=1)
G8.add_edge(555,307,weight=1)
G8.add_edge(560,35,weight=1)
G8.add_edge(560,1817,weight=1)
G8.add_edge(560,168,weight=1)
G8.add_edge(683,1792,weight=1)
G8.add_edge(683,2431,weight=1)
G8.add_edge(88,384,weight=1)
G8.add_edge(88,48,weight=1)
G8.add_edge(88,2246,weight=1)
G8.add_edge(93,2439,weight=1)
G8.add_edge(1893,2132,weight=1)
G8.add_edge(1995,2098,weight=1)
G8.add_edge(1995,2242,weight=1)
G8.add_edge(1995,1996,weight=1)
G8.add_edge(1995,1801,weight=1)
G8.add_edge(1996,2098,weight=1)
G8.add_edge(1996,1908,weight=1)
G8.add_edge(1996,1801,weight=1)
G8.add_edge(524,479,weight=1)
G8.add_edge(524,871,weight=1)
G8.add_edge(524,2443,weight=1)
G8.add_edge(524,2601,weight=1)
G8.add_edge(524,1617,weight=1)
G8.add_edge(524,2552,weight=1)
G8.add_edge(524,911,weight=1)
G8.add_edge(524,883,weight=1)
G8.add_edge(524,284,weight=1)
G8.add_edge(524,1215,weight=1)
G8.add_edge(524,1688,weight=1)
G8.add_edge(524,1932,weight=1)
G8.add_edge(524,2409,weight=1)
G8.add_edge(524,962,weight=1)
G8.add_edge(524,55,weight=1)
G8.add_edge(524,2643,weight=1)
G8.add_edge(524,1677,weight=1)
G8.add_edge(524,509,weight=1)
G8.add_edge(524,541,weight=1)
G8.add_edge(524,416,weight=1)
G8.add_edge(524,1791,weight=1)
G8.add_edge(524,466,weight=1)
G8.add_edge(524,566,weight=1)
G8.add_edge(524,2481,weight=1)
G8.add_edge(524,552,weight=1)
G8.add_edge(524,1922,weight=1)
G8.add_edge(524,278,weight=1)
G8.add_edge(609,395,weight=1)
G8.add_edge(609,2231,weight=1)
G8.add_edge(609,681,weight=1)
G8.add_edge(609,976,weight=1)
G8.add_edge(609,180,weight=1)
G8.add_edge(609,134,weight=1)
G8.add_edge(609,612,weight=1)
G8.add_edge(612,900,weight=1)
G8.add_edge(612,2231,weight=1)
G8.add_edge(612,2502,weight=1)
G8.add_edge(612,1104,weight=1)
G8.add_edge(612,2341,weight=1)
G8.add_edge(612,2353,weight=1)
G8.add_edge(612,719,weight=1)
G8.add_edge(612,865,weight=1)
G8.add_edge(612,2506,weight=1)
G8.add_edge(612,1756,weight=1)
G8.add_edge(612,1764,weight=1)
G8.add_edge(612,2051,weight=1)
G8.add_edge(612,387,weight=1)
G8.add_edge(612,180,weight=1)
G8.add_edge(612,717,weight=1)
G8.add_edge(612,1733,weight=1)
G8.add_edge(612,512,weight=1)
G8.add_edge(612,2541,weight=1)
G8.add_edge(612,2677,weight=1)
G8.add_edge(612,2386,weight=1)
G8.add_edge(612,1669,weight=1)
G8.add_edge(612,29,weight=1)
G8.add_edge(612,634,weight=1)
G8.add_edge(612,1353,weight=1)
G8.add_edge(612,580,weight=1)
G8.add_edge(612,1494,weight=1)
G8.add_edge(612,134,weight=1)
G8.add_edge(612,1117,weight=1)
G8.add_edge(612,1565,weight=1)
G8.add_edge(612,369,weight=1)
G8.add_edge(612,1489,weight=1)
G8.add_edge(612,2379,weight=1)
G8.add_edge(612,939,weight=1)
G8.add_edge(612,1266,weight=1)
G8.add_edge(612,1936,weight=1)
G8.add_edge(612,1500,weight=1)
G8.add_edge(612,2261,weight=1)
G8.add_edge(612,1316,weight=1)
G8.add_edge(612,2326,weight=1)
G8.add_edge(612,602,weight=1)
G8.add_edge(612,414,weight=1)
G8.add_edge(1535,1598,weight=1)
G8.add_edge(1535,1164,weight=1)
G8.add_edge(1535,1557,weight=1)
G8.add_edge(1535,1499,weight=1)
G8.add_edge(1535,1279,weight=1)
G8.add_edge(1535,1123,weight=1)
G8.add_edge(1535,290,weight=1)
G8.add_edge(1535,1149,weight=1)
G8.add_edge(1536,1349,weight=1)
G8.add_edge(1536,630,weight=1)
G8.add_edge(1536,290,weight=1)
G8.add_edge(1538,1557,weight=1)
G8.add_edge(1538,1349,weight=1)
G8.add_edge(1538,630,weight=1)
G8.add_edge(1538,290,weight=1)
G8.add_edge(1538,1536,weight=1)
G8.add_edge(613,2569,weight=1)
G8.add_edge(613,290,weight=1)
G8.add_edge(613,1047,weight=1)
G8.add_edge(2303,404,weight=1)
G8.add_edge(2303,1354,weight=1)
G8.add_edge(2303,2112,weight=1)
G8.add_edge(2303,1881,weight=1)
G8.add_edge(2303,2105,weight=1)
G8.add_edge(403,404,weight=1)
G8.add_edge(403,682,weight=1)
G8.add_edge(404,2105,weight=1)
G8.add_edge(2310,404,weight=1)
G8.add_edge(2310,682,weight=1)
G8.add_edge(2310,1326,weight=1)
G8.add_edge(2528,1957,weight=1)
G8.add_edge(2528,2101,weight=1)
G8.add_edge(2528,1962,weight=1)
G8.add_edge(2538,1847,weight=1)
G8.add_edge(2538,2199,weight=1)
G8.add_edge(2538,452,weight=1)
G8.add_edge(2538,1928,weight=1)
G8.add_edge(1561,2199,weight=1)
G8.add_edge(1561,536,weight=1)
G8.add_edge(1561,792,weight=1)
G8.add_edge(1561,1162,weight=1)
G8.add_edge(1561,457,weight=1)
G8.add_edge(639,2466,weight=1)
G8.add_edge(639,2147,weight=1)
G8.add_edge(639,702,weight=1)
G8.add_edge(639,863,weight=1)
G8.add_edge(639,723,weight=1)
G8.add_edge(639,1809,weight=1)
G8.add_edge(639,1190,weight=1)
G8.add_edge(639,384,weight=1)
G8.add_edge(639,619,weight=1)
G8.add_edge(639,1461,weight=1)
G8.add_edge(639,1313,weight=1)
G8.add_edge(639,1469,weight=1)
G8.add_edge(639,1896,weight=1)
G8.add_edge(639,2674,weight=1)
G8.add_edge(639,2172,weight=1)
G8.add_edge(639,442,weight=1)
G8.add_edge(639,228,weight=1)
G8.add_edge(639,297,weight=1)
G8.add_edge(639,2542,weight=1)
G8.add_edge(639,2046,weight=1)
G8.add_edge(639,1328,weight=1)
G8.add_edge(639,265,weight=1)
G8.add_edge(639,780,weight=1)
G8.add_edge(639,349,weight=1)
G8.add_edge(639,90,weight=1)
G8.add_edge(639,1028,weight=1)
G8.add_edge(639,522,weight=1)
G8.add_edge(639,2427,weight=1)
G8.add_edge(639,376,weight=1)
G8.add_edge(639,1540,weight=1)
G8.add_edge(639,1920,weight=1)
G8.add_edge(639,450,weight=1)
G8.add_edge(748,84,weight=1)
G8.add_edge(748,337,weight=1)
G8.add_edge(748,74,weight=1)
G8.add_edge(748,601,weight=1)
G8.add_edge(748,204,weight=1)
G8.add_edge(748,350,weight=1)
G8.add_edge(748,113,weight=1)
G8.add_edge(748,235,weight=1)
G8.add_edge(748,459,weight=1)
G8.add_edge(748,681,weight=1)
G8.add_edge(748,696,weight=1)
G8.add_edge(748,199,weight=1)
G8.add_edge(748,503,weight=1)
G8.add_edge(748,725,weight=1)
G8.add_edge(748,329,weight=1)
G8.add_edge(748,398,weight=1)
G8.add_edge(748,738,weight=1)
G8.add_edge(748,600,weight=1)
G8.add_edge(748,666,weight=1)
G8.add_edge(748,546,weight=1)
G8.add_edge(748,187,weight=1)
G8.add_edge(748,495,weight=1)
G8.add_edge(748,610,weight=1)
G8.add_edge(748,611,weight=1)
G8.add_edge(748,316,weight=1)
G8.add_edge(748,331,weight=1)
G8.add_edge(748,387,weight=1)
G8.add_edge(748,621,weight=1)
G8.add_edge(748,230,weight=1)
G8.add_edge(748,236,weight=1)
G8.add_edge(748,701,weight=1)
G8.add_edge(748,629,weight=1)
G8.add_edge(748,527,weight=1)
G8.add_edge(748,528,weight=1)
G8.add_edge(748,764,weight=1)
G8.add_edge(748,147,weight=1)
G8.add_edge(748,284,weight=1)
G8.add_edge(748,105,weight=1)
G8.add_edge(748,772,weight=1)
G8.add_edge(748,573,weight=1)
G8.add_edge(748,700,weight=1)
G8.add_edge(748,218,weight=1)
G8.add_edge(748,111,weight=1)
G8.add_edge(748,203,weight=1)
G8.add_edge(748,205,weight=1)
G8.add_edge(748,44,weight=1)
G8.add_edge(748,656,weight=1)
G8.add_edge(748,73,weight=1)
G8.add_edge(748,580,weight=1)
G8.add_edge(748,27,weight=1)
G8.add_edge(748,686,weight=1)
G8.add_edge(748,369,weight=1)
G8.add_edge(748,19,weight=1)
G8.add_edge(748,780,weight=1)
G8.add_edge(748,543,weight=1)
G8.add_edge(748,589,weight=1)
G8.add_edge(748,704,weight=1)
G8.add_edge(748,742,weight=1)
G8.add_edge(748,311,weight=1)
G8.add_edge(748,125,weight=1)
G8.add_edge(748,128,weight=1)
G8.add_edge(748,720,weight=1)
G8.add_edge(748,154,weight=1)
G8.add_edge(748,206,weight=1)
G8.add_edge(748,208,weight=1)
G8.add_edge(748,114,weight=1)
G8.add_edge(748,697,weight=1)
G8.add_edge(748,490,weight=1)
G8.add_edge(748,562,weight=1)
G8.add_edge(748,370,weight=1)
G8.add_edge(748,176,weight=1)
G8.add_edge(748,354,weight=1)
G8.add_edge(748,92,weight=1)
G8.add_edge(748,363,weight=1)
G8.add_edge(67,2360,weight=1)
G8.add_edge(67,1249,weight=1)
G8.add_edge(67,2295,weight=1)
G8.add_edge(67,70,weight=1)
G8.add_edge(67,216,weight=1)
G8.add_edge(67,2311,weight=1)
G8.add_edge(67,1532,weight=1)
G8.add_edge(67,2442,weight=1)
G8.add_edge(67,2123,weight=1)
G8.add_edge(67,184,weight=1)
G8.add_edge(67,35,weight=1)
G8.add_edge(67,1699,weight=1)
G8.add_edge(67,150,weight=1)
G8.add_edge(67,2521,weight=1)
G8.add_edge(67,289,weight=1)
G8.add_edge(67,2158,weight=1)
G8.add_edge(1137,1287,weight=1)
G8.add_edge(1137,1031,weight=1)
G8.add_edge(1137,801,weight=1)
G8.add_edge(1137,1741,weight=1)
G8.add_edge(1137,1576,weight=1)
G8.add_edge(1137,936,weight=1)
G8.add_edge(1137,1064,weight=1)
G8.add_edge(1137,1143,weight=1)
G8.add_edge(1137,1019,weight=1)
G8.add_edge(1137,728,weight=1)
G8.add_edge(1137,1559,weight=1)
G8.add_edge(1137,1517,weight=1)
G8.add_edge(1137,1061,weight=1)
G8.add_edge(1137,1595,weight=1)
G8.add_edge(1137,1334,weight=1)
G8.add_edge(1143,1019,weight=1)
G8.add_edge(346,118,weight=1)
G8.add_edge(346,2390,weight=1)
G8.add_edge(346,2267,weight=1)
G8.add_edge(346,1925,weight=1)
G8.add_edge(346,262,weight=1)
G8.add_edge(1257,882,weight=1)
G8.add_edge(1352,901,weight=1)
G8.add_edge(2475,2426,weight=1)
G8.add_edge(2475,2079,weight=1)
G8.add_edge(581,923,weight=1)
G8.add_edge(581,1213,weight=1)
G8.add_edge(1497,1126,weight=1)
G8.add_edge(1498,902,weight=1)
G8.add_edge(1498,1164,weight=1)
G8.add_edge(1498,581,weight=1)
G8.add_edge(1498,1213,weight=1)
G8.add_edge(582,223,weight=1)
G8.add_edge(582,50,weight=1)
G8.add_edge(582,51,weight=1)
G8.add_edge(582,581,weight=1)
G8.add_edge(583,819,weight=1)
G8.add_edge(583,260,weight=1)
G8.add_edge(583,1793,weight=1)
G8.add_edge(583,1499,weight=1)
G8.add_edge(1499,2399,weight=1)
G8.add_edge(1499,1164,weight=1)
G8.add_edge(1499,2159,weight=1)
G8.add_edge(1499,1050,weight=1)
G8.add_edge(1499,1072,weight=1)
G8.add_edge(1499,1352,weight=1)
G8.add_edge(1499,2451,weight=1)
G8.add_edge(1499,1484,weight=1)
G8.add_edge(584,2399,weight=1)
G8.add_edge(584,2569,weight=1)
G8.add_edge(584,51,weight=1)
G8.add_edge(584,1215,weight=1)
G8.add_edge(584,1279,weight=1)
G8.add_edge(584,1127,weight=1)
G8.add_edge(584,65,weight=1)
G8.add_edge(584,285,weight=1)
G8.add_edge(584,644,weight=1)
G8.add_edge(584,2372,weight=1)
G8.add_edge(1513,1236,weight=1)
G8.add_edge(1513,1672,weight=1)
G8.add_edge(1513,950,weight=1)
G8.add_edge(1513,1088,weight=1)
G8.add_edge(1513,1000,weight=1)
G8.add_edge(1513,1464,weight=1)
G8.add_edge(1513,1336,weight=1)
G8.add_edge(1513,1268,weight=1)
G8.add_edge(1513,1609,weight=1)
G8.add_edge(2509,1236,weight=1)
G8.add_edge(2509,2578,weight=1)
G8.add_edge(2509,2138,weight=1)
G8.add_edge(2509,633,weight=1)
G8.add_edge(2509,323,weight=1)
G8.add_edge(252,2561,weight=1)
G8.add_edge(252,7,weight=1)
G8.add_edge(252,933,weight=1)
G8.add_edge(252,1878,weight=1)
G8.add_edge(252,483,weight=1)
G8.add_edge(252,89,weight=1)
G8.add_edge(252,1506,weight=1)
G8.add_edge(345,1623,weight=1)
G8.add_edge(345,2,weight=1)
G8.add_edge(345,2618,weight=1)
G8.add_edge(345,2610,weight=1)
G8.add_edge(345,1672,weight=1)
G8.add_edge(345,336,weight=1)
G8.add_edge(345,1882,weight=1)
G8.add_edge(345,2340,weight=1)
G8.add_edge(345,2442,weight=1)
G8.add_edge(345,2230,weight=1)
G8.add_edge(345,964,weight=1)
G8.add_edge(345,317,weight=1)
G8.add_edge(345,1443,weight=1)
G8.add_edge(345,1446,weight=1)
G8.add_edge(345,2590,weight=1)
G8.add_edge(345,1313,weight=1)
G8.add_edge(345,1628,weight=1)
G8.add_edge(345,2033,weight=1)
G8.add_edge(345,2172,weight=1)
G8.add_edge(345,1268,weight=1)
G8.add_edge(345,1276,weight=1)
G8.add_edge(345,2291,weight=1)
G8.add_edge(345,2292,weight=1)
G8.add_edge(345,1261,weight=1)
G8.add_edge(345,349,weight=1)
G8.add_edge(345,1245,weight=1)
G8.add_edge(345,676,weight=1)
G8.add_edge(345,522,weight=1)
G8.add_edge(345,2427,weight=1)
G8.add_edge(345,1008,weight=1)
G8.add_edge(570,1182,weight=1)
G8.add_edge(570,768,weight=1)
G8.add_edge(570,551,weight=1)
G8.add_edge(570,1274,weight=1)
G8.add_edge(570,1093,weight=1)
G8.add_edge(570,1713,weight=1)
G8.add_edge(570,256,weight=1)
G8.add_edge(570,1523,weight=1)
G8.add_edge(570,1987,weight=1)
G8.add_edge(570,1458,weight=1)
G8.add_edge(570,1431,weight=1)
G8.add_edge(570,2100,weight=1)
G8.add_edge(570,515,weight=1)
G8.add_edge(570,711,weight=1)
G8.add_edge(570,1270,weight=1)
G8.add_edge(570,1001,weight=1)
G8.add_edge(570,457,weight=1)
G8.add_edge(570,1026,weight=1)
G8.add_edge(1485,1522,weight=1)
G8.add_edge(1485,1058,weight=1)
G8.add_edge(1485,1693,weight=1)
G8.add_edge(1485,1221,weight=1)
G8.add_edge(1485,1093,weight=1)
G8.add_edge(1485,932,weight=1)
G8.add_edge(1485,823,weight=1)
G8.add_edge(1485,1523,weight=1)
G8.add_edge(1485,1431,weight=1)
G8.add_edge(1485,1483,weight=1)
G8.add_edge(1485,1198,weight=1)
G8.add_edge(1485,1501,weight=1)
G8.add_edge(1485,1238,weight=1)
G8.add_edge(571,213,weight=1)
G8.add_edge(571,2027,weight=1)
G8.add_edge(571,1259,weight=1)
G8.add_edge(571,1382,weight=1)
G8.add_edge(571,1201,weight=1)
G8.add_edge(571,1244,weight=1)
G8.add_edge(571,1247,weight=1)
G8.add_edge(571,1525,weight=1)
G8.add_edge(571,1710,weight=1)
G8.add_edge(571,170,weight=1)
G8.add_edge(571,1923,weight=1)
G8.add_edge(571,1815,weight=1)
G8.add_edge(571,873,weight=1)
G8.add_edge(571,60,weight=1)
G8.add_edge(571,875,weight=1)
G8.add_edge(571,1246,weight=1)
G8.add_edge(571,1068,weight=1)
G8.add_edge(2487,1971,weight=1)
G8.add_edge(577,606,weight=1)
G8.add_edge(577,864,weight=1)
G8.add_edge(577,213,weight=1)
G8.add_edge(577,294,weight=1)
G8.add_edge(577,2027,weight=1)
G8.add_edge(577,1259,weight=1)
G8.add_edge(577,1382,weight=1)
G8.add_edge(577,1527,weight=1)
G8.add_edge(577,1201,weight=1)
G8.add_edge(577,331,weight=1)
G8.add_edge(577,355,weight=1)
G8.add_edge(577,1525,weight=1)
G8.add_edge(577,1711,weight=1)
G8.add_edge(577,571,weight=1)
G8.add_edge(577,1923,weight=1)
G8.add_edge(577,727,weight=1)
G8.add_edge(577,1802,weight=1)
G8.add_edge(577,835,weight=1)
G8.add_edge(577,840,weight=1)
G8.add_edge(577,1815,weight=1)
G8.add_edge(577,1015,weight=1)
G8.add_edge(577,2553,weight=1)
G8.add_edge(577,1933,weight=1)
G8.add_edge(577,60,weight=1)
G8.add_edge(577,875,weight=1)
G8.add_edge(577,1246,weight=1)
G8.add_edge(577,356,weight=1)
G8.add_edge(577,1068,weight=1)
G8.add_edge(2499,2148,weight=1)
G8.add_edge(2499,2568,weight=1)
G8.add_edge(2499,2501,weight=1)
G8.add_edge(2499,2323,weight=1)
G8.add_edge(2500,204,weight=1)
G8.add_edge(2500,2148,weight=1)
G8.add_edge(2500,864,weight=1)
G8.add_edge(2500,2377,weight=1)
G8.add_edge(2500,1788,weight=1)
G8.add_edge(2500,2506,weight=1)
G8.add_edge(2500,1872,weight=1)
G8.add_edge(2500,2358,weight=1)
G8.add_edge(2500,2386,weight=1)
G8.add_edge(2500,2423,weight=1)
G8.add_edge(2500,2499,weight=1)
G8.add_edge(2500,2501,weight=1)
G8.add_edge(2500,2323,weight=1)
G8.add_edge(2500,1419,weight=1)
G8.add_edge(2500,360,weight=1)
G8.add_edge(2500,1936,weight=1)
G8.add_edge(2500,1069,weight=1)
G8.add_edge(2501,2377,weight=1)
G8.add_edge(2501,2423,weight=1)
G8.add_edge(2501,2499,weight=1)
G8.add_edge(2501,2323,weight=1)
G8.add_edge(2501,1069,weight=1)
G8.add_edge(2571,481,weight=1)
G8.add_edge(2571,668,weight=1)
G8.add_edge(2571,1929,weight=1)
G8.add_edge(2571,1807,weight=1)
G8.add_edge(1123,800,weight=1)
G8.add_edge(1123,1195,weight=1)
G8.add_edge(1123,595,weight=1)
G8.add_edge(1124,540,weight=1)
G8.add_edge(1124,542,weight=1)
G8.add_edge(1124,1189,weight=1)
G8.add_edge(1124,1177,weight=1)
G8.add_edge(1124,76,weight=1)
G8.add_edge(1124,1484,weight=1)
G8.add_edge(1126,818,weight=1)
G8.add_edge(1126,819,weight=1)
G8.add_edge(1126,858,weight=1)
G8.add_edge(1126,1484,weight=1)
G8.add_edge(1127,2490,weight=1)
G8.add_edge(1127,924,weight=1)
G8.add_edge(1127,581,weight=1)
G8.add_edge(1127,1498,weight=1)
G8.add_edge(1127,1213,weight=1)
G8.add_edge(1213,818,weight=1)
G8.add_edge(1213,819,weight=1)
G8.add_edge(1213,820,weight=1)
G8.add_edge(1213,1553,weight=1)
G8.add_edge(1216,1700,weight=1)
G8.add_edge(1216,819,weight=1)
G8.add_edge(1216,820,weight=1)
G8.add_edge(1216,1210,weight=1)
G8.add_edge(1216,581,weight=1)
G8.add_edge(1216,1708,weight=1)
G8.add_edge(1216,825,weight=1)
G8.add_edge(1216,826,weight=1)
G8.add_edge(348,714,weight=1)
G8.add_edge(348,999,weight=1)
G8.add_edge(348,1000,weight=1)
G8.add_edge(348,453,weight=1)
G8.add_edge(348,85,weight=1)
G8.add_edge(1241,913,weight=1)
G8.add_edge(1241,1202,weight=1)
G8.add_edge(1241,842,weight=1)
G8.add_edge(1241,898,weight=1)
G8.add_edge(1241,999,weight=1)
G8.add_edge(1241,1000,weight=1)
G8.add_edge(1241,988,weight=1)
G8.add_edge(1241,1087,weight=1)
G8.add_edge(1241,743,weight=1)
G8.add_edge(1241,1142,weight=1)
G8.add_edge(1241,1242,weight=1)
G8.add_edge(1241,951,weight=1)
G8.add_edge(1241,85,weight=1)
G8.add_edge(1241,1348,weight=1)
G8.add_edge(1241,1148,weight=1)
G8.add_edge(1242,913,weight=1)
G8.add_edge(1242,999,weight=1)
G8.add_edge(1242,1142,weight=1)
G8.add_edge(1242,1241,weight=1)
G8.add_edge(1242,951,weight=1)
G8.add_edge(1242,85,weight=1)
G8.add_edge(1242,1348,weight=1)
G8.add_edge(1242,1148,weight=1)
G8.add_edge(1323,1623,weight=1)
G8.add_edge(1323,1021,weight=1)
G8.add_edge(1323,2540,weight=1)
G8.add_edge(1323,1028,weight=1)
G8.add_edge(1323,324,weight=1)
G8.add_edge(427,30,weight=1)
G8.add_edge(427,336,weight=1)
G8.add_edge(427,2370,weight=1)
G8.add_edge(427,1577,weight=1)
G8.add_edge(427,837,weight=1)
G8.add_edge(427,1188,weight=1)
G8.add_edge(427,2127,weight=1)
G8.add_edge(427,1487,weight=1)
G8.add_edge(427,938,weight=1)
G8.add_edge(427,304,weight=1)
G8.add_edge(427,1276,weight=1)
G8.add_edge(427,1062,weight=1)
G8.add_edge(427,1529,weight=1)
G8.add_edge(427,319,weight=1)
G8.add_edge(427,2432,weight=1)
G8.add_edge(1329,1424,weight=1)
G8.add_edge(1329,1585,weight=1)
G8.add_edge(1329,1102,weight=1)
G8.add_edge(1329,1032,weight=1)
G8.add_edge(1329,1514,weight=1)
G8.add_edge(1329,928,weight=1)
G8.add_edge(1329,1413,weight=1)
G8.add_edge(1329,1156,weight=1)
G8.add_edge(1329,1021,weight=1)
G8.add_edge(1329,1110,weight=1)
G8.add_edge(1329,1112,weight=1)
G8.add_edge(1329,1569,weight=1)
G8.add_edge(1329,948,weight=1)
G8.add_edge(1329,784,weight=1)
G8.add_edge(1329,786,weight=1)
G8.add_edge(1329,1174,weight=1)
G8.add_edge(1329,1108,weight=1)
G8.add_edge(2339,1025,weight=1)
G8.add_edge(2339,1479,weight=1)
G8.add_edge(2339,2678,weight=1)
G8.add_edge(2339,2128,weight=1)
G8.add_edge(2339,1987,weight=1)
G8.add_edge(2339,2444,weight=1)
G8.add_edge(1330,2128,weight=1)
G8.add_edge(1330,1050,weight=1)
G8.add_edge(1330,1072,weight=1)
G8.add_edge(1330,1987,weight=1)
G8.add_edge(1330,1021,weight=1)
G8.add_edge(428,2045,weight=1)
G8.add_edge(428,2296,weight=1)
G8.add_edge(428,1275,weight=1)
G8.add_edge(428,1767,weight=1)
G8.add_edge(428,1915,weight=1)
G8.add_edge(428,2256,weight=1)
G8.add_edge(428,1104,weight=1)
G8.add_edge(428,1812,weight=1)
G8.add_edge(428,2384,weight=1)
G8.add_edge(428,494,weight=1)
G8.add_edge(428,2454,weight=1)
G8.add_edge(428,1761,weight=1)
G8.add_edge(428,666,weight=1)
G8.add_edge(428,2575,weight=1)
G8.add_edge(428,2389,weight=1)
G8.add_edge(428,1400,weight=1)
G8.add_edge(428,1406,weight=1)
G8.add_edge(428,495,weight=1)
G8.add_edge(428,611,weight=1)
G8.add_edge(428,588,weight=1)
G8.add_edge(428,2259,weight=1)
G8.add_edge(428,824,weight=1)
G8.add_edge(428,1319,weight=1)
G8.add_edge(428,527,weight=1)
G8.add_edge(428,1548,weight=1)
G8.add_edge(428,2413,weight=1)
G8.add_edge(428,1038,weight=1)
G8.add_edge(428,147,weight=1)
G8.add_edge(428,2307,weight=1)
G8.add_edge(428,313,weight=1)
G8.add_edge(428,203,weight=1)
G8.add_edge(428,1284,weight=1)
G8.add_edge(428,1285,weight=1)
G8.add_edge(428,231,weight=1)
G8.add_edge(428,2659,weight=1)
G8.add_edge(428,400,weight=1)
G8.add_edge(428,699,weight=1)
G8.add_edge(428,407,weight=1)
G8.add_edge(428,1976,weight=1)
G8.add_edge(428,251,weight=1)
G8.add_edge(428,585,weight=1)
G8.add_edge(428,305,weight=1)
G8.add_edge(428,2449,weight=1)
G8.add_edge(428,2684,weight=1)
G8.add_edge(428,1489,weight=1)
G8.add_edge(428,19,weight=1)
G8.add_edge(428,955,weight=1)
G8.add_edge(428,2438,weight=1)
G8.add_edge(428,917,weight=1)
G8.add_edge(428,2000,weight=1)
G8.add_edge(428,675,weight=1)
G8.add_edge(428,947,weight=1)
G8.add_edge(428,1578,weight=1)
G8.add_edge(428,659,weight=1)
G8.add_edge(428,206,weight=1)
G8.add_edge(428,697,weight=1)
G8.add_edge(428,462,weight=1)
G8.add_edge(428,1341,weight=1)
G8.add_edge(428,562,weight=1)
G8.add_edge(428,354,weight=1)
G8.add_edge(428,392,weight=1)
G8.add_edge(431,2488,weight=1)
G8.add_edge(431,1618,weight=1)
G8.add_edge(431,2594,weight=1)
G8.add_edge(431,1638,weight=1)
G8.add_edge(431,684,weight=1)
G8.add_edge(431,2034,weight=1)
G8.add_edge(431,1166,weight=1)
G8.add_edge(431,1793,weight=1)
G8.add_edge(431,2198,weight=1)
G8.add_edge(431,2138,weight=1)
G8.add_edge(431,224,weight=1)
G8.add_edge(431,1021,weight=1)
G8.add_edge(431,525,weight=1)
G8.add_edge(431,1699,weight=1)
G8.add_edge(431,382,weight=1)
G8.add_edge(431,1285,weight=1)
G8.add_edge(431,652,weight=1)
G8.add_edge(431,1208,weight=1)
G8.add_edge(431,2344,weight=1)
G8.add_edge(431,548,weight=1)
G8.add_edge(431,1569,weight=1)
G8.add_edge(431,633,weight=1)
G8.add_edge(431,2558,weight=1)
G8.add_edge(431,209,weight=1)
G8.add_edge(431,687,weight=1)
G8.add_edge(431,173,weight=1)
G8.add_edge(431,178,weight=1)
G8.add_edge(431,90,weight=1)
G8.add_edge(431,1271,weight=1)
G8.add_edge(431,120,weight=1)
G8.add_edge(431,53,weight=1)
G8.add_edge(431,237,weight=1)
G8.add_edge(2343,2594,weight=1)
G8.add_edge(2343,2578,weight=1)
G8.add_edge(2343,2198,weight=1)
G8.add_edge(2343,382,weight=1)
G8.add_edge(2343,498,weight=1)
G8.add_edge(2343,2152,weight=1)
G8.add_edge(2343,1271,weight=1)
G8.add_edge(1336,1025,weight=1)
G8.add_edge(1336,1479,weight=1)
G8.add_edge(1336,945,weight=1)
G8.add_edge(1336,1608,weight=1)
G8.add_edge(1336,1513,weight=1)
G8.add_edge(1336,1224,weight=1)
G8.add_edge(1336,886,weight=1)
G8.add_edge(1336,1271,weight=1)
G8.add_edge(1336,1033,weight=1)
G8.add_edge(1337,1479,weight=1)
G8.add_edge(1337,1028,weight=1)
G8.add_edge(1338,945,weight=1)
G8.add_edge(1338,1021,weight=1)
G8.add_edge(1338,1336,weight=1)
G8.add_edge(1338,2344,weight=1)
G8.add_edge(1338,2301,weight=1)
G8.add_edge(2344,2444,weight=1)
G8.add_edge(1340,435,weight=1)
G8.add_edge(1340,2678,weight=1)
G8.add_edge(1340,1336,weight=1)
G8.add_edge(1340,1112,weight=1)
G8.add_edge(1340,1571,weight=1)
G8.add_edge(1340,2444,weight=1)
G8.add_edge(439,80,weight=1)
G8.add_edge(439,551,weight=1)
G8.add_edge(439,97,weight=1)
G8.add_edge(439,22,weight=1)
G8.add_edge(439,608,weight=1)
G8.add_edge(439,228,weight=1)
G8.add_edge(439,297,weight=1)
G8.add_edge(439,617,weight=1)
G8.add_edge(439,711,weight=1)
G8.add_edge(439,693,weight=1)
G8.add_edge(439,721,weight=1)
G8.add_edge(439,392,weight=1)
G8.add_edge(290,2569,weight=1)
G8.add_edge(290,1499,weight=1)
G8.add_edge(290,1279,weight=1)
G8.add_edge(290,65,weight=1)
G8.add_edge(2277,1865,weight=1)
G8.add_edge(2693,1371,weight=1)
G8.add_edge(2693,2039,weight=1)
G8.add_edge(2693,2173,weight=1)
G8.add_edge(72,930,weight=1)
G8.add_edge(72,678,weight=1)
G8.add_edge(72,2443,weight=1)
G8.add_edge(72,1560,weight=1)
G8.add_edge(72,1924,weight=1)
G8.add_edge(72,662,weight=1)
G8.add_edge(72,1642,weight=1)
G8.add_edge(75,907,weight=1)
G8.add_edge(75,1025,weight=1)
G8.add_edge(75,1018,weight=1)
G8.add_edge(75,1104,weight=1)
G8.add_edge(75,872,weight=1)
G8.add_edge(75,950,weight=1)
G8.add_edge(75,1221,weight=1)
G8.add_edge(75,1635,weight=1)
G8.add_edge(75,681,weight=1)
G8.add_edge(75,605,weight=1)
G8.add_edge(75,2514,weight=1)
G8.add_edge(75,2267,weight=1)
G8.add_edge(75,155,weight=1)
G8.add_edge(75,976,weight=1)
G8.add_edge(75,2655,weight=1)
G8.add_edge(75,2285,weight=1)
G8.add_edge(75,2336,weight=1)
G8.add_edge(75,506,weight=1)
G8.add_edge(75,512,weight=1)
G8.add_edge(75,988,weight=1)
G8.add_edge(75,2014,weight=1)
G8.add_edge(75,253,weight=1)
G8.add_edge(75,346,weight=1)
G8.add_edge(75,1677,weight=1)
G8.add_edge(75,623,weight=1)
G8.add_edge(75,548,weight=1)
G8.add_edge(75,882,weight=1)
G8.add_edge(75,788,weight=1)
G8.add_edge(75,2391,weight=1)
G8.add_edge(75,1929,weight=1)
G8.add_edge(75,264,weight=1)
G8.add_edge(75,72,weight=1)
G8.add_edge(75,662,weight=1)
G8.add_edge(75,2125,weight=1)
G8.add_edge(75,311,weight=1)
G8.add_edge(75,378,weight=1)
G8.add_edge(75,20,weight=1)
G8.add_edge(75,868,weight=1)
G8.add_edge(77,870,weight=1)
G8.add_edge(77,1129,weight=1)
G8.add_edge(77,132,weight=1)
G8.add_edge(77,976,weight=1)
G8.add_edge(77,695,weight=1)
G8.add_edge(77,1935,weight=1)
G8.add_edge(77,346,weight=1)
G8.add_edge(77,182,weight=1)
G8.add_edge(77,2533,weight=1)
G8.add_edge(77,251,weight=1)
G8.add_edge(77,2391,weight=1)
G8.add_edge(77,662,weight=1)
G8.add_edge(77,2328,weight=1)
G8.add_edge(77,1549,weight=1)
G8.add_edge(77,393,weight=1)
G8.add_edge(77,519,weight=1)
G8.add_edge(1152,1351,weight=1)
G8.add_edge(295,84,weight=1)
G8.add_edge(295,1354,weight=1)
G8.add_edge(295,22,weight=1)
G8.add_edge(295,503,weight=1)
G8.add_edge(295,1439,weight=1)
G8.add_edge(295,666,weight=1)
G8.add_edge(295,104,weight=1)
G8.add_edge(295,187,weight=1)
G8.add_edge(295,174,weight=1)
G8.add_edge(295,639,weight=1)
G8.add_edge(295,555,weight=1)
G8.add_edge(295,192,weight=1)
G8.add_edge(295,1673,weight=1)
G8.add_edge(295,808,weight=1)
G8.add_edge(295,253,weight=1)
G8.add_edge(295,700,weight=1)
G8.add_edge(295,86,weight=1)
G8.add_edge(295,634,weight=1)
G8.add_edge(295,637,weight=1)
G8.add_edge(295,2477,weight=1)
G8.add_edge(295,2431,weight=1)
G8.add_edge(295,466,weight=1)
G8.add_edge(295,467,weight=1)
G8.add_edge(295,478,weight=1)
G8.add_edge(295,484,weight=1)
G8.add_edge(295,565,weight=1)
G8.add_edge(295,566,weight=1)
G8.add_edge(295,567,weight=1)
G8.add_edge(295,589,weight=1)
G8.add_edge(295,129,weight=1)
G8.add_edge(295,154,weight=1)
G8.add_edge(295,278,weight=1)
G8.add_edge(2247,2476,weight=1)
G8.add_edge(1260,1945,weight=1)
G8.add_edge(1260,2214,weight=1)
G8.add_edge(1260,1082,weight=1)
G8.add_edge(1260,1083,weight=1)
G8.add_edge(1260,894,weight=1)
G8.add_edge(1260,93,weight=1)
G8.add_edge(1260,2313,weight=1)
G8.add_edge(1260,522,weight=1)
G8.add_edge(1260,137,weight=1)
G8.add_edge(1270,1313,weight=1)
G8.add_edge(1270,2619,weight=1)
G8.add_edge(1270,877,weight=1)
G8.add_edge(1270,1261,weight=1)
G8.add_edge(592,2691,weight=1)
G8.add_edge(592,2156,weight=1)
G8.add_edge(592,288,weight=1)
G8.add_edge(592,1187,weight=1)
G8.add_edge(592,454,weight=1)
G8.add_edge(592,550,weight=1)
G8.add_edge(592,1359,weight=1)
G8.add_edge(592,2476,weight=1)
G8.add_edge(592,2183,weight=1)
G8.add_edge(592,849,weight=1)
G8.add_edge(592,200,weight=1)
G8.add_edge(592,489,weight=1)
G8.add_edge(592,1033,weight=1)
G8.add_edge(592,1106,weight=1)
G8.add_edge(2521,2360,weight=1)
G8.add_edge(2638,2653,weight=1)
G8.add_edge(71,1522,weight=1)
G8.add_edge(71,1693,weight=1)
G8.add_edge(71,256,weight=1)
G8.add_edge(71,932,weight=1)
G8.add_edge(71,1000,weight=1)
G8.add_edge(71,1987,weight=1)
G8.add_edge(76,2123,weight=1)
G8.add_edge(76,2300,weight=1)
G8.add_edge(76,1855,weight=1)
G8.add_edge(76,1856,weight=1)
G8.add_edge(76,289,weight=1)
G8.add_edge(1855,2123,weight=1)
G8.add_edge(1855,1977,weight=1)
G8.add_edge(1855,2049,weight=1)
G8.add_edge(1855,2646,weight=1)
G8.add_edge(1855,2300,weight=1)
G8.add_edge(1855,2483,weight=1)
G8.add_edge(1855,1856,weight=1)
G8.add_edge(1856,2123,weight=1)
G8.add_edge(1856,2483,weight=1)
G8.add_edge(1856,1855,weight=1)
G8.add_edge(1865,2123,weight=1)
G8.add_edge(1865,1855,weight=1)
G8.add_edge(1865,1856,weight=1)
G8.add_edge(85,13,weight=1)
G8.add_edge(85,716,weight=1)
G8.add_edge(85,999,weight=1)
G8.add_edge(85,156,weight=1)
G8.add_edge(85,348,weight=1)
G8.add_edge(85,1241,weight=1)
G8.add_edge(85,28,weight=1)
G8.add_edge(85,951,weight=1)
G8.add_edge(85,1804,weight=1)
G8.add_edge(85,1348,weight=1)
G8.add_edge(85,1551,weight=1)
G8.add_edge(1029,1436,weight=1)
G8.add_edge(1029,1038,weight=1)
G8.add_edge(1029,207,weight=1)
G8.add_edge(1029,1043,weight=1)
G8.add_edge(1029,1611,weight=1)
G8.add_edge(360,204,weight=1)
G8.add_edge(360,2377,weight=1)
G8.add_edge(360,1788,weight=1)
G8.add_edge(360,2568,weight=1)
G8.add_edge(360,2506,weight=1)
G8.add_edge(360,1872,weight=1)
G8.add_edge(360,331,weight=1)
G8.add_edge(360,1321,weight=1)
G8.add_edge(360,2423,weight=1)
G8.add_edge(360,2499,weight=1)
G8.add_edge(360,2500,weight=1)
G8.add_edge(360,2323,weight=1)
G8.add_edge(360,809,weight=1)
G8.add_edge(360,1343,weight=1)
G8.add_edge(360,1069,weight=1)
G8.add_edge(477,606,weight=1)
G8.add_edge(477,1092,weight=1)
G8.add_edge(477,1057,weight=1)
G8.add_edge(2379,2051,weight=1)
G8.add_edge(2379,2336,weight=1)
G8.add_edge(2379,1669,weight=1)
G8.add_edge(2379,612,weight=1)
G8.add_edge(1622,301,weight=1)
G8.add_edge(1622,1449,weight=1)
G8.add_edge(1622,1195,weight=1)
G8.add_edge(1622,595,weight=1)
G8.add_edge(1622,825,weight=1)
G8.add_edge(1936,2341,weight=1)
G8.add_edge(1936,612,weight=1)
G8.add_edge(2058,2466,weight=1)
G8.add_edge(2058,2696,weight=1)
G8.add_edge(2058,791,weight=1)
G8.add_edge(2058,1511,weight=1)
G8.add_edge(229,2466,weight=1)
G8.add_edge(229,2618,weight=1)
G8.add_edge(229,1693,weight=1)
G8.add_edge(229,260,weight=1)
G8.add_edge(229,901,weight=1)
G8.add_edge(229,120,weight=1)
G8.add_edge(229,324,weight=1)
G8.add_edge(229,53,weight=1)
G8.add_edge(457,1297,weight=1)
G8.add_edge(457,2456,weight=1)
G8.add_edge(457,2654,weight=1)
G8.add_edge(457,2100,weight=1)
G8.add_edge(457,1749,weight=1)
G8.add_edge(1367,2514,weight=1)
G8.add_edge(1367,2414,weight=1)
G8.add_edge(1367,2415,weight=1)
G8.add_edge(1367,2240,weight=1)
G8.add_edge(1367,2563,weight=1)
G8.add_edge(466,119,weight=1)
G8.add_edge(466,2028,weight=1)
G8.add_edge(466,104,weight=1)
G8.add_edge(466,2673,weight=1)
G8.add_edge(466,86,weight=1)
G8.add_edge(466,2369,weight=1)
G8.add_edge(466,467,weight=1)
G8.add_edge(466,566,weight=1)
G8.add_edge(466,567,weight=1)
G8.add_edge(2369,2643,weight=1)
G8.add_edge(467,17,weight=1)
G8.add_edge(1379,1441,weight=1)
G8.add_edge(1379,240,weight=1)
G8.add_edge(1379,22,weight=1)
G8.add_edge(1379,560,weight=1)
G8.add_edge(1379,883,weight=1)
G8.add_edge(1379,83,weight=1)
G8.add_edge(1379,1541,weight=1)
G8.add_edge(1379,971,weight=1)
G8.add_edge(472,3,weight=1)
G8.add_edge(472,1046,weight=1)
G8.add_edge(472,104,weight=1)
G8.add_edge(472,1633,weight=1)
G8.add_edge(472,1860,weight=1)
G8.add_edge(472,296,weight=1)
G8.add_edge(478,1646,weight=1)
G8.add_edge(478,1767,weight=1)
G8.add_edge(478,464,weight=1)
G8.add_edge(478,230,weight=1)
G8.add_edge(478,2410,weight=1)
G8.add_edge(478,2659,weight=1)
G8.add_edge(478,428,weight=1)
G8.add_edge(478,334,weight=1)
G8.add_edge(484,464,weight=1)
G8.add_edge(484,934,weight=1)
G8.add_edge(484,750,weight=1)
G8.add_edge(484,368,weight=1)
G8.add_edge(484,651,weight=1)
G8.add_edge(484,278,weight=1)
G8.add_edge(563,1402,weight=1)
G8.add_edge(563,2015,weight=1)
G8.add_edge(563,2019,weight=1)
G8.add_edge(563,253,weight=1)
G8.add_edge(563,441,weight=1)
G8.add_edge(563,1890,weight=1)
G8.add_edge(563,61,weight=1)
G8.add_edge(563,672,weight=1)
G8.add_edge(563,371,weight=1)
G8.add_edge(563,1118,weight=1)
G8.add_edge(563,1184,weight=1)
G8.add_edge(565,464,weight=1)
G8.add_edge(566,479,weight=1)
G8.add_edge(566,549,weight=1)
G8.add_edge(566,1783,weight=1)
G8.add_edge(566,3,weight=1)
G8.add_edge(566,119,weight=1)
G8.add_edge(566,1441,weight=1)
G8.add_edge(566,2028,weight=1)
G8.add_edge(566,306,weight=1)
G8.add_edge(566,1402,weight=1)
G8.add_edge(566,1586,weight=1)
G8.add_edge(566,240,weight=1)
G8.add_edge(566,426,weight=1)
G8.add_edge(566,2443,weight=1)
G8.add_edge(566,1013,weight=1)
G8.add_edge(566,1377,weight=1)
G8.add_edge(566,1617,weight=1)
G8.add_edge(566,503,weight=1)
G8.add_edge(566,1317,weight=1)
G8.add_edge(566,645,weight=1)
G8.add_edge(566,1439,weight=1)
G8.add_edge(566,774,weight=1)
G8.add_edge(566,386,weight=1)
G8.add_edge(566,843,weight=1)
G8.add_edge(566,1958,weight=1)
G8.add_edge(566,1214,weight=1)
G8.add_edge(566,104,weight=1)
G8.add_edge(566,174,weight=1)
G8.add_edge(566,911,weight=1)
G8.add_edge(566,555,weight=1)
G8.add_edge(566,1742,weight=1)
G8.add_edge(566,883,weight=1)
G8.add_edge(566,2673,weight=1)
G8.add_edge(566,35,weight=1)
G8.add_edge(566,1547,weight=1)
G8.add_edge(566,284,weight=1)
G8.add_edge(566,493,weight=1)
G8.add_edge(566,86,weight=1)
G8.add_edge(566,2119,weight=1)
G8.add_edge(566,1860,weight=1)
G8.add_edge(566,83,weight=1)
G8.add_edge(566,1530,weight=1)
G8.add_edge(566,411,weight=1)
G8.add_edge(566,1728,weight=1)
G8.add_edge(566,1541,weight=1)
G8.add_edge(566,622,weight=1)
G8.add_edge(566,634,weight=1)
G8.add_edge(566,739,weight=1)
G8.add_edge(566,58,weight=1)
G8.add_edge(566,61,weight=1)
G8.add_edge(566,2473,weight=1)
G8.add_edge(566,2643,weight=1)
G8.add_edge(566,1677,weight=1)
G8.add_edge(566,245,weight=1)
G8.add_edge(566,1976,weight=1)
G8.add_edge(566,672,weight=1)
G8.add_edge(566,334,weight=1)
G8.add_edge(566,93,weight=1)
G8.add_edge(566,692,weight=1)
G8.add_edge(566,368,weight=1)
G8.add_edge(566,1791,weight=1)
G8.add_edge(566,466,weight=1)
G8.add_edge(566,467,weight=1)
G8.add_edge(566,1379,weight=1)
G8.add_edge(566,478,weight=1)
G8.add_edge(566,565,weight=1)
G8.add_edge(566,567,weight=1)
G8.add_edge(566,2481,weight=1)
G8.add_edge(566,576,weight=1)
G8.add_edge(566,2486,weight=1)
G8.add_edge(566,2629,weight=1)
G8.add_edge(566,2630,weight=1)
G8.add_edge(566,129,weight=1)
G8.add_edge(566,1374,weight=1)
G8.add_edge(566,296,weight=1)
G8.add_edge(566,602,weight=1)
G8.add_edge(566,307,weight=1)
G8.add_edge(567,1783,weight=1)
G8.add_edge(567,473,weight=1)
G8.add_edge(567,1545,weight=1)
G8.add_edge(567,734,weight=1)
G8.add_edge(567,17,weight=1)
G8.add_edge(567,187,weight=1)
G8.add_edge(567,1448,weight=1)
G8.add_edge(567,174,weight=1)
G8.add_edge(567,35,weight=1)
G8.add_edge(567,2222,weight=1)
G8.add_edge(567,700,weight=1)
G8.add_edge(567,689,weight=1)
G8.add_edge(567,2659,weight=1)
G8.add_edge(567,634,weight=1)
G8.add_edge(567,859,weight=1)
G8.add_edge(567,1817,weight=1)
G8.add_edge(567,672,weight=1)
G8.add_edge(567,334,weight=1)
G8.add_edge(567,467,weight=1)
G8.add_edge(567,478,weight=1)
G8.add_edge(567,484,weight=1)
G8.add_edge(567,2481,weight=1)
G8.add_edge(567,2486,weight=1)
G8.add_edge(567,589,weight=1)
G8.add_edge(567,1644,weight=1)
G8.add_edge(567,275,weight=1)
G8.add_edge(567,322,weight=1)
G8.add_edge(567,450,weight=1)
G8.add_edge(2481,654,weight=1)
G8.add_edge(2481,2486,weight=1)
G8.add_edge(568,22,weight=1)
G8.add_edge(568,503,weight=1)
G8.add_edge(568,386,weight=1)
G8.add_edge(568,1829,weight=1)
G8.add_edge(568,971,weight=1)
G8.add_edge(568,278,weight=1)
G8.add_edge(576,883,weight=1)
G8.add_edge(576,1379,weight=1)
G8.add_edge(576,2486,weight=1)
G8.add_edge(2486,750,weight=1)
G8.add_edge(2486,1817,weight=1)
G8.add_edge(2486,1379,weight=1)
G8.add_edge(1492,1391,weight=1)
G8.add_edge(1492,852,weight=1)
G8.add_edge(1492,1606,weight=1)
G8.add_edge(1492,904,weight=1)
G8.add_edge(2584,2311,weight=1)
G8.add_edge(2584,1424,weight=1)
G8.add_edge(2584,2334,weight=1)
G8.add_edge(2584,1174,weight=1)
G8.add_edge(2586,1810,weight=1)
G8.add_edge(2586,2442,weight=1)
G8.add_edge(2586,1831,weight=1)
G8.add_edge(2586,1264,weight=1)
G8.add_edge(687,684,weight=1)
G8.add_edge(687,1274,weight=1)
G8.add_edge(687,117,weight=1)
G8.add_edge(687,226,weight=1)
G8.add_edge(687,761,weight=1)
G8.add_edge(687,224,weight=1)
G8.add_edge(687,45,weight=1)
G8.add_edge(687,1699,weight=1)
G8.add_edge(687,382,weight=1)
G8.add_edge(687,431,weight=1)
G8.add_edge(687,2542,weight=1)
G8.add_edge(687,2558,weight=1)
G8.add_edge(687,120,weight=1)
G8.add_edge(1627,1585,weight=1)
G8.add_edge(1627,1408,weight=1)
G8.add_edge(1627,1699,weight=1)
G8.add_edge(1627,1736,weight=1)
G8.add_edge(1627,1165,weight=1)
G8.add_edge(2598,2257,weight=1)
G8.add_edge(2598,1699,weight=1)
G8.add_edge(691,446,weight=1)
G8.add_edge(691,2458,weight=1)
G8.add_edge(1644,774,weight=1)
G8.add_edge(1644,1190,weight=1)
G8.add_edge(1644,869,weight=1)
G8.add_edge(1650,1229,weight=1)
G8.add_edge(1650,1214,weight=1)
G8.add_edge(1650,1688,weight=1)
G8.add_edge(1650,869,weight=1)
G8.add_edge(1650,692,weight=1)
G8.add_edge(1650,1644,weight=1)
G8.add_edge(1650,149,weight=1)
G8.add_edge(1650,1558,weight=1)
G8.add_edge(925,1481,weight=1)
G8.add_edge(1062,1011,weight=1)
G8.add_edge(1062,1579,weight=1)
G8.add_edge(1062,1652,weight=1)
G8.add_edge(1062,1245,weight=1)
G8.add_edge(1175,1034,weight=1)
G8.add_edge(1175,1075,weight=1)
G8.add_edge(1175,1136,weight=1)
G8.add_edge(1175,894,weight=1)
G8.add_edge(1175,304,weight=1)
G8.add_edge(1175,1298,weight=1)
G8.add_edge(1175,1103,weight=1)
G8.add_edge(1175,903,weight=1)
G8.add_edge(1175,415,weight=1)
G8.add_edge(319,446,weight=1)
G8.add_edge(319,2311,weight=1)
G8.add_edge(319,992,weight=1)
G8.add_edge(319,2334,weight=1)
G8.add_edge(319,687,weight=1)
G8.add_edge(319,2428,weight=1)
G8.add_edge(319,1174,weight=1)
G8.add_edge(323,1028,weight=1)
G8.add_edge(1196,1324,weight=1)
G8.add_edge(1196,1631,weight=1)
G8.add_edge(2181,2578,weight=1)
G8.add_edge(2181,1501,weight=1)
G8.add_edge(1204,1120,weight=1)
G8.add_edge(1204,817,weight=1)
G8.add_edge(1204,1631,weight=1)
G8.add_edge(1204,882,weight=1)
G8.add_edge(1204,780,weight=1)
G8.add_edge(1204,543,weight=1)
G8.add_edge(1204,1412,weight=1)
G8.add_edge(1204,1196,weight=1)
G8.add_edge(326,1202,weight=1)
G8.add_edge(326,212,weight=1)
G8.add_edge(1288,811,weight=1)
G8.add_edge(1288,946,weight=1)
G8.add_edge(1288,977,weight=1)
G8.add_edge(1288,1415,weight=1)
G8.add_edge(1288,1438,weight=1)
G8.add_edge(1288,1398,weight=1)
G8.add_edge(1288,788,weight=1)
G8.add_edge(1288,500,weight=1)
G8.add_edge(1288,906,weight=1)
G8.add_edge(1289,1503,weight=1)
G8.add_edge(1289,1129,weight=1)
G8.add_edge(1289,1092,weight=1)
G8.add_edge(1289,1415,weight=1)
G8.add_edge(1289,788,weight=1)
G8.add_edge(1289,500,weight=1)
G8.add_edge(1289,108,weight=1)
G8.add_edge(1289,109,weight=1)
G8.add_edge(421,11,weight=1)
G8.add_edge(421,2215,weight=1)
G8.add_edge(421,193,weight=1)
G8.add_edge(421,2591,weight=1)
G8.add_edge(421,520,weight=1)
G8.add_edge(421,440,weight=1)
G8.add_edge(520,2609,weight=1)
G8.add_edge(520,11,weight=1)
G8.add_edge(520,191,weight=1)
G8.add_edge(520,706,weight=1)
G8.add_edge(520,511,weight=1)
G8.add_edge(520,2224,weight=1)
G8.add_edge(520,2550,weight=1)
G8.add_edge(520,2591,weight=1)
G8.add_edge(520,281,weight=1)
G8.add_edge(2422,1830,weight=1)
G8.add_edge(2422,2390,weight=1)
G8.add_edge(2422,1857,weight=1)
G8.add_edge(2422,2267,weight=1)
G8.add_edge(2422,2655,weight=1)
G8.add_edge(2422,346,weight=1)
G8.add_edge(2422,264,weight=1)
G8.add_edge(2422,2591,weight=1)
G8.add_edge(2422,75,weight=1)
G8.add_edge(2422,1869,weight=1)
G8.add_edge(522,157,weight=1)
G8.add_edge(522,2313,weight=1)
G8.add_edge(2427,1687,weight=1)
G8.add_edge(2427,2127,weight=1)
G8.add_edge(2427,2283,weight=1)
G8.add_edge(2427,2494,weight=1)
G8.add_edge(2427,2432,weight=1)
G8.add_edge(2427,1920,weight=1)
G8.add_edge(2428,1889,weight=1)
G8.add_edge(2428,2549,weight=1)
G8.add_edge(2428,2592,weight=1)
G8.add_edge(2428,2392,weight=1)
G8.add_edge(2428,508,weight=1)
G8.add_edge(1429,1623,weight=1)
G8.add_edge(1429,2080,weight=1)
G8.add_edge(1429,2618,weight=1)
G8.add_edge(1429,2425,weight=1)
G8.add_edge(1429,2230,weight=1)
G8.add_edge(1429,1313,weight=1)
G8.add_edge(1429,2540,weight=1)
G8.add_edge(1429,2494,weight=1)
G8.add_edge(1430,1209,weight=1)
G8.add_edge(2432,2592,weight=1)
G8.add_edge(1609,950,weight=1)
G8.add_edge(1609,1221,weight=1)
G8.add_edge(1609,988,weight=1)
G8.add_edge(1609,2226,weight=1)
G8.add_edge(1609,432,weight=1)
G8.add_edge(1613,1518,weight=1)
G8.add_edge(2587,2163,weight=1)
G8.add_edge(2587,1648,weight=1)
G8.add_edge(2587,1890,weight=1)
G8.add_edge(2587,2487,weight=1)
G8.add_edge(2587,2114,weight=1)
G8.add_edge(123,1094,weight=1)
G8.add_edge(123,125,weight=1)
G8.add_edge(123,128,weight=1)
G8.add_edge(123,2401,weight=1)
G8.add_edge(125,2401,weight=1)
G8.add_edge(1917,2401,weight=1)
G8.add_edge(128,125,weight=1)
G8.add_edge(128,2229,weight=1)
G8.add_edge(128,2401,weight=1)
G8.add_edge(296,1783,weight=1)
G8.add_edge(296,198,weight=1)
G8.add_edge(296,2079,weight=1)
G8.add_edge(296,1013,weight=1)
G8.add_edge(296,1317,weight=1)
G8.add_edge(296,221,weight=1)
G8.add_edge(296,211,weight=1)
G8.add_edge(2151,2184,weight=1)
G8.add_edge(2151,1193,weight=1)
G8.add_edge(1282,1605,weight=1)
G8.add_edge(1282,949,weight=1)
G8.add_edge(1282,1472,weight=1)
G8.add_edge(1282,1273,weight=1)
G8.add_edge(1282,1283,weight=1)
G8.add_edge(1282,1452,weight=1)
G8.add_edge(1283,1495,weight=1)
G8.add_edge(1283,1023,weight=1)
G8.add_edge(1283,949,weight=1)
G8.add_edge(1283,1472,weight=1)
G8.add_edge(1283,1273,weight=1)
G8.add_edge(1283,1447,weight=1)
G8.add_edge(1283,1282,weight=1)
G8.add_edge(2653,2309,weight=1)
G8.add_edge(2653,241,weight=1)
G8.add_edge(1054,854,weight=1)
G8.add_edge(1054,1025,weight=1)
G8.add_edge(1054,932,weight=1)
G8.add_edge(1054,1464,weight=1)
G8.add_edge(1054,1608,weight=1)
G8.add_edge(1054,1198,weight=1)
G8.add_edge(1054,1569,weight=1)
G8.add_edge(379,961,weight=1)
G8.add_edge(379,336,weight=1)
G8.add_edge(379,2013,weight=1)
G8.add_edge(379,2196,weight=1)
G8.add_edge(379,2043,weight=1)
G8.add_edge(379,1239,weight=1)
G8.add_edge(379,219,weight=1)
G8.add_edge(379,179,weight=1)
G8.add_edge(379,287,weight=1)
G8.add_edge(379,586,weight=1)
G8.add_edge(379,1645,weight=1)
G8.add_edge(379,1291,weight=1)
G8.add_edge(379,1208,weight=1)
G8.add_edge(379,228,weight=1)
G8.add_edge(379,693,weight=1)
G8.add_edge(379,1276,weight=1)
G8.add_edge(379,2291,weight=1)
G8.add_edge(379,2292,weight=1)
G8.add_edge(379,1964,weight=1)
G8.add_edge(379,397,weight=1)
G8.add_edge(379,2448,weight=1)
G8.add_edge(379,2584,weight=1)
G8.add_edge(379,1028,weight=1)
G8.add_edge(379,1062,weight=1)
G8.add_edge(379,1679,weight=1)
G8.add_edge(732,1451,weight=1)
G8.add_edge(732,331,weight=1)
G8.add_edge(732,577,weight=1)
G8.add_edge(732,1779,weight=1)
G8.add_edge(732,809,weight=1)
G8.add_edge(2650,1957,weight=1)
G8.add_edge(2650,250,weight=1)
G8.add_edge(2650,2437,weight=1)
G8.add_edge(2650,2107,weight=1)
G8.add_edge(736,217,weight=1)
G8.add_edge(736,1957,weight=1)
G8.add_edge(736,2528,weight=1)
G8.add_edge(736,308,weight=1)
G8.add_edge(736,196,weight=1)
G8.add_edge(736,87,weight=1)
G8.add_edge(736,2107,weight=1)
G8.add_edge(736,2650,weight=1)
G8.add_edge(827,1274,weight=1)
G8.add_edge(827,1607,weight=1)
G8.add_edge(827,1458,weight=1)
G8.add_edge(827,1423,weight=1)
G8.add_edge(827,1238,weight=1)
G8.add_edge(829,1458,weight=1)
G8.add_edge(829,1423,weight=1)
G8.add_edge(1820,2581,weight=1)
G8.add_edge(1820,225,weight=1)
G8.add_edge(1824,2488,weight=1)
G8.add_edge(1824,2581,weight=1)
G8.add_edge(53,2488,weight=1)
G8.add_edge(53,2581,weight=1)
G8.add_edge(137,2400,weight=1)
G8.add_edge(137,2488,weight=1)
G8.add_edge(137,2581,weight=1)
G8.add_edge(137,1371,weight=1)
G8.add_edge(137,2173,weight=1)
G8.add_edge(137,1480,weight=1)
G8.add_edge(137,2544,weight=1)
G8.add_edge(137,1777,weight=1)
G8.add_edge(137,245,weight=1)
G8.add_edge(137,93,weight=1)
G8.add_edge(137,1414,weight=1)
G8.add_edge(137,160,weight=1)
G8.add_edge(137,1103,weight=1)
G8.add_edge(137,2232,weight=1)
G8.add_edge(137,2636,weight=1)
G8.add_edge(1962,2101,weight=1)
G8.add_edge(237,1618,weight=1)
G8.add_edge(237,157,weight=1)
G8.add_edge(1108,1618,weight=1)
G8.add_edge(1115,1697,weight=1)
G8.add_edge(1115,1589,weight=1)
G8.add_edge(1115,1262,weight=1)
G8.add_edge(1115,1604,weight=1)
G8.add_edge(1115,603,weight=1)
G8.add_edge(1115,1372,weight=1)
G8.add_edge(1115,1118,weight=1)
G8.add_edge(248,2258,weight=1)
G8.add_edge(248,501,weight=1)
G8.add_edge(248,2447,weight=1)
G8.add_edge(248,1974,weight=1)
G8.add_edge(248,564,weight=1)
G8.add_edge(248,983,weight=1)
G8.add_edge(248,1262,weight=1)
G8.add_edge(248,1118,weight=1)
G8.add_edge(2090,358,weight=1)
G8.add_edge(2090,1966,weight=1)
G8.add_edge(2090,1262,weight=1)
G8.add_edge(2090,1604,weight=1)
G8.add_edge(2090,715,weight=1)
G8.add_edge(2090,1531,weight=1)
G8.add_edge(2090,1118,weight=1)
G8.add_edge(1118,1589,weight=1)
G8.add_edge(1118,2333,weight=1)
G8.add_edge(1118,1226,weight=1)
G8.add_edge(1118,1262,weight=1)
G8.add_edge(1118,1604,weight=1)
G8.add_edge(2208,80,weight=1)
G8.add_edge(2208,1787,weight=1)
G8.add_edge(2054,2686,weight=1)
G8.add_edge(2055,2686,weight=1)
G8.add_edge(2055,592,weight=1)
G8.add_edge(2055,2054,weight=1)
G8.add_edge(2071,2603,weight=1)
G8.add_edge(2071,2641,weight=1)
G8.add_edge(2082,2160,weight=1)
G8.add_edge(2082,2367,weight=1)
G8.add_edge(2083,2546,weight=1)
G8.add_edge(2083,2179,weight=1)
G8.add_edge(415,1878,weight=1)
G8.add_edge(415,909,weight=1)
G8.add_edge(415,1443,weight=1)
G8.add_edge(415,1446,weight=1)
G8.add_edge(415,441,weight=1)
G8.add_edge(1310,1187,weight=1)
G8.add_edge(1310,1133,weight=1)
G8.add_edge(1310,925,weight=1)
G8.add_edge(636,684,weight=1)
G8.add_edge(636,723,weight=1)
G8.add_edge(636,443,weight=1)
G8.add_edge(636,661,weight=1)
G8.add_edge(636,257,weight=1)
G8.add_edge(636,382,weight=1)
G8.add_edge(636,475,weight=1)
G8.add_edge(636,652,weight=1)
G8.add_edge(636,431,weight=1)
G8.add_edge(636,633,weight=1)
G8.add_edge(636,635,weight=1)
G8.add_edge(636,209,weight=1)
G8.add_edge(636,265,weight=1)
G8.add_edge(636,687,weight=1)
G8.add_edge(636,173,weight=1)
G8.add_edge(636,178,weight=1)
G8.add_edge(636,90,weight=1)
G8.add_edge(636,437,weight=1)
G8.add_edge(636,376,weight=1)
G8.add_edge(2556,2561,weight=1)
G8.add_edge(641,636,weight=1)
G8.add_edge(641,288,weight=1)
G8.add_edge(641,460,weight=1)
G8.add_edge(641,1364,weight=1)
G8.add_edge(641,550,weight=1)
G8.add_edge(641,1898,weight=1)
G8.add_edge(641,110,weight=1)
G8.add_edge(641,2476,weight=1)
G8.add_edge(641,21,weight=1)
G8.add_edge(641,733,weight=1)
G8.add_edge(641,592,weight=1)
G8.add_edge(641,575,weight=1)
G8.add_edge(641,131,weight=1)
G8.add_edge(641,200,weight=1)
G8.add_edge(641,2055,weight=1)
G8.add_edge(650,2561,weight=1)
G8.add_edge(650,167,weight=1)
G8.add_edge(650,2295,weight=1)
G8.add_edge(650,97,weight=1)
G8.add_edge(650,2582,weight=1)
G8.add_edge(650,458,weight=1)
G8.add_edge(650,2404,weight=1)
G8.add_edge(650,2124,weight=1)
G8.add_edge(650,753,weight=1)
G8.add_edge(650,1644,weight=1)
G8.add_edge(2561,324,weight=1)
G8.add_edge(167,458,weight=1)
G8.add_edge(167,1126,weight=1)
G8.add_edge(987,2485,weight=1)
G8.add_edge(987,863,weight=1)
G8.add_edge(987,1643,weight=1)
G8.add_edge(987,2617,weight=1)
G8.add_edge(987,2606,weight=1)
G8.add_edge(987,137,weight=1)
G8.add_edge(259,8,weight=1)
G8.add_edge(259,401,weight=1)
G8.add_edge(259,545,weight=1)
G8.add_edge(259,1032,weight=1)
G8.add_edge(259,269,weight=1)
G8.add_edge(259,1,weight=1)
G8.add_edge(259,9,weight=1)
G8.add_edge(259,15,weight=1)
G8.add_edge(259,1112,weight=1)
G8.add_edge(259,752,weight=1)
G8.add_edge(259,436,weight=1)
G8.add_edge(479,97,weight=1)
G8.add_edge(479,216,weight=1)
G8.add_edge(479,619,weight=1)
G8.add_edge(479,219,weight=1)
G8.add_edge(479,521,weight=1)
G8.add_edge(479,498,weight=1)
G8.add_edge(479,228,weight=1)
G8.add_edge(479,471,weight=1)
G8.add_edge(479,304,weight=1)
G8.add_edge(479,755,weight=1)
G8.add_edge(479,397,weight=1)
G8.add_edge(479,569,weight=1)
G8.add_edge(479,53,weight=1)
G8.add_edge(479,275,weight=1)
G8.add_edge(480,650,weight=1)
G8.add_edge(480,479,weight=1)
G8.add_edge(480,148,weight=1)
G8.add_edge(480,419,weight=1)
G8.add_edge(480,1355,weight=1)
G8.add_edge(480,1628,weight=1)
G8.add_edge(480,2637,weight=1)
G8.add_edge(480,93,weight=1)
G8.add_edge(480,2301,weight=1)
G8.add_edge(480,137,weight=1)
G8.add_edge(480,450,weight=1)
G8.add_edge(1646,798,weight=1)
G8.add_edge(1646,205,weight=1)
G8.add_edge(872,1592,weight=1)
G8.add_edge(159,185,weight=1)
G8.add_edge(159,599,weight=1)
G8.add_edge(159,434,weight=1)
G8.add_edge(159,708,weight=1)
G8.add_edge(1228,1237,weight=1)
G8.add_edge(1235,978,weight=1)
G8.add_edge(1235,1227,weight=1)
G8.add_edge(1235,1188,weight=1)
G8.add_edge(1235,1264,weight=1)
G8.add_edge(1235,14,weight=1)
G8.add_edge(1235,938,weight=1)
G8.add_edge(1235,1276,weight=1)
G8.add_edge(1235,791,weight=1)
G8.add_edge(1235,1511,weight=1)
G8.add_edge(2211,1809,weight=1)
G8.add_edge(1237,1228,weight=1)
G8.add_edge(1237,862,weight=1)
G8.add_edge(1237,297,weight=1)
G8.add_edge(1237,383,weight=1)
G8.add_edge(1237,1670,weight=1)
G8.add_edge(1239,1501,weight=1)
G8.add_edge(1239,1026,weight=1)
G8.add_edge(1239,1551,weight=1)
G8.add_edge(2351,2169,weight=1)
G8.add_edge(1344,1365,weight=1)
G8.add_edge(1344,1457,weight=1)
G8.add_edge(2355,401,weight=1)
G8.add_edge(2355,557,weight=1)
G8.add_edge(2357,1706,weight=1)
G8.add_edge(2357,2247,weight=1)
G8.add_edge(2357,1106,weight=1)
G8.add_edge(454,2157,weight=1)
G8.add_edge(454,10,weight=1)
G8.add_edge(454,707,weight=1)
G8.add_edge(454,1713,weight=1)
G8.add_edge(454,2060,weight=1)
G8.add_edge(454,2355,weight=1)
G8.add_edge(454,2456,weight=1)
G8.add_edge(454,1567,weight=1)
G8.add_edge(454,1359,weight=1)
G8.add_edge(454,1662,weight=1)
G8.add_edge(454,2093,weight=1)
G8.add_edge(454,2099,weight=1)
G8.add_edge(454,2100,weight=1)
G8.add_edge(454,1450,weight=1)
G8.add_edge(454,515,weight=1)
G8.add_edge(454,2619,weight=1)
G8.add_edge(454,1270,weight=1)
G8.add_edge(454,877,weight=1)
G8.add_edge(454,1026,weight=1)
G8.add_edge(454,2035,weight=1)
G8.add_edge(1361,1096,weight=1)
G8.add_edge(2361,2005,weight=1)
G8.add_edge(2361,401,weight=1)
G8.add_edge(2361,557,weight=1)
G8.add_edge(2361,1387,weight=1)
G8.add_edge(2361,2274,weight=1)
G8.add_edge(2361,2355,weight=1)
G8.add_edge(2361,1007,weight=1)
G8.add_edge(2361,2041,weight=1)
G8.add_edge(460,268,weight=1)
G8.add_edge(460,1706,weight=1)
G8.add_edge(460,131,weight=1)
G8.add_edge(1364,2169,weight=1)
G8.add_edge(1364,1344,weight=1)
G8.add_edge(1364,1365,weight=1)
G8.add_edge(1364,2476,weight=1)
G8.add_edge(1364,2654,weight=1)
G8.add_edge(1364,787,weight=1)
G8.add_edge(1365,1706,weight=1)
G8.add_edge(1365,1470,weight=1)
G8.add_edge(550,288,weight=1)
G8.add_edge(550,454,weight=1)
G8.add_edge(550,592,weight=1)
G8.add_edge(550,508,weight=1)
G8.add_edge(550,457,weight=1)
G8.add_edge(550,575,weight=1)
G8.add_edge(550,489,weight=1)
G8.add_edge(2456,2355,weight=1)
G8.add_edge(556,616,weight=1)
G8.add_edge(556,401,weight=1)
G8.add_edge(556,557,weight=1)
G8.add_edge(556,2355,weight=1)
G8.add_edge(1470,1706,weight=1)
G8.add_edge(1470,1457,weight=1)
G8.add_edge(668,2271,weight=1)
G8.add_edge(668,481,weight=1)
G8.add_edge(668,2073,weight=1)
G8.add_edge(668,1975,weight=1)
G8.add_edge(668,2228,weight=1)
G8.add_edge(668,1357,weight=1)
G8.add_edge(668,1929,weight=1)
G8.add_edge(668,258,weight=1)
G8.add_edge(668,868,weight=1)
G8.add_edge(1593,1735,weight=1)
G8.add_edge(1593,1294,weight=1)
G8.add_edge(1593,406,weight=1)
G8.add_edge(1593,1299,weight=1)
G8.add_edge(1593,1177,weight=1)
G8.add_edge(1593,1162,weight=1)
G8.add_edge(1601,1294,weight=1)
G8.add_edge(1601,947,weight=1)
G8.add_edge(673,2330,weight=1)
G8.add_edge(673,536,weight=1)
G8.add_edge(673,2005,weight=1)
G8.add_edge(673,1387,weight=1)
G8.add_edge(673,1344,weight=1)
G8.add_edge(673,2361,weight=1)
G8.add_edge(673,1364,weight=1)
G8.add_edge(673,550,weight=1)
G8.add_edge(673,2697,weight=1)
G8.add_edge(673,1461,weight=1)
G8.add_edge(673,1294,weight=1)
G8.add_edge(673,1004,weight=1)
G8.add_edge(673,1005,weight=1)
G8.add_edge(673,2654,weight=1)
G8.add_edge(673,592,weight=1)
G8.add_edge(673,457,weight=1)
G8.add_edge(673,925,weight=1)
G8.add_edge(673,575,weight=1)
G8.add_edge(673,131,weight=1)
G8.add_edge(1607,2382,weight=1)
G8.add_edge(1607,1294,weight=1)
G8.add_edge(1607,29,weight=1)
G8.add_edge(1733,854,weight=1)
G8.add_edge(1733,2226,weight=1)
G8.add_edge(1733,1494,weight=1)
G8.add_edge(911,1590,weight=1)
G8.add_edge(911,1674,weight=1)
G8.add_edge(911,1132,weight=1)
G8.add_edge(911,1433,weight=1)
G8.add_edge(911,1591,weight=1)
G8.add_edge(911,1141,weight=1)
G8.add_edge(911,1254,weight=1)
G8.add_edge(911,983,weight=1)
G8.add_edge(911,1574,weight=1)
G8.add_edge(911,1312,weight=1)
G8.add_edge(911,965,weight=1)
G8.add_edge(911,967,weight=1)
G8.add_edge(911,765,weight=1)
G8.add_edge(911,812,weight=1)
G8.add_edge(911,1222,weight=1)
G8.add_edge(911,982,weight=1)
G8.add_edge(911,1482,weight=1)
G8.add_edge(911,903,weight=1)
G8.add_edge(911,1118,weight=1)
G8.add_edge(193,11,weight=1)
G8.add_edge(193,706,weight=1)
G8.add_edge(193,511,weight=1)
G8.add_edge(193,194,weight=1)
G8.add_edge(193,281,weight=1)
G8.add_edge(193,421,weight=1)
G8.add_edge(193,520,weight=1)
G8.add_edge(193,440,weight=1)
G8.add_edge(194,11,weight=1)
G8.add_edge(194,239,weight=1)
G8.add_edge(194,2550,weight=1)
G8.add_edge(194,75,weight=1)
G8.add_edge(194,281,weight=1)
G8.add_edge(194,421,weight=1)
G8.add_edge(194,520,weight=1)
G8.add_edge(653,374,weight=1)
G8.add_edge(2677,1788,weight=1)
G8.add_edge(2677,2386,weight=1)
G8.add_edge(1742,493,weight=1)
G8.add_edge(1742,602,weight=1)
G8.add_edge(4,198,weight=1)
G8.add_edge(4,464,weight=1)
G8.add_edge(4,602,weight=1)
G8.add_edge(1845,1980,weight=1)
G8.add_edge(1845,1846,weight=1)
G8.add_edge(1846,1980,weight=1)
G8.add_edge(1846,980,weight=1)
G8.add_edge(69,1980,weight=1)
G8.add_edge(69,980,weight=1)
G8.add_edge(69,1846,weight=1)
G8.add_edge(69,1633,weight=1)
G8.add_edge(69,2008,weight=1)
G8.add_edge(69,523,weight=1)
G8.add_edge(69,1826,weight=1)
G8.add_edge(880,701,weight=1)
G8.add_edge(880,980,weight=1)
G8.add_edge(899,1688,weight=1)
G8.add_edge(497,2394,weight=1)
G8.add_edge(497,502,weight=1)
G8.add_edge(497,1897,weight=1)
G8.add_edge(2394,1814,weight=1)
G8.add_edge(2394,1953,weight=1)
G8.add_edge(2394,1897,weight=1)
G8.add_edge(2394,2030,weight=1)
G8.add_edge(502,175,weight=1)
G8.add_edge(504,1814,weight=1)
G8.add_edge(504,175,weight=1)
G8.add_edge(504,1953,weight=1)
G8.add_edge(504,502,weight=1)
G8.add_edge(504,2397,weight=1)
G8.add_edge(504,1852,weight=1)
G8.add_edge(504,2030,weight=1)
G8.add_edge(2410,2028,weight=1)
G8.add_edge(2410,175,weight=1)
G8.add_edge(2410,639,weight=1)
G8.add_edge(2410,2673,weight=1)
G8.add_edge(598,435,weight=1)
G8.add_edge(598,107,weight=1)
G8.add_edge(598,225,weight=1)
G8.add_edge(1673,900,weight=1)
G8.add_edge(1673,808,weight=1)
G8.add_edge(1673,1353,weight=1)
G8.add_edge(1673,1565,weight=1)
G8.add_edge(1673,904,weight=1)
G8.add_edge(808,1439,weight=1)
G8.add_edge(36,35,weight=1)
G8.add_edge(40,25,weight=1)
G8.add_edge(40,83,weight=1)
G8.add_edge(41,762,weight=1)
G8.add_edge(1017,1111,weight=1)
G8.add_edge(1017,944,weight=1)
G8.add_edge(1017,1071,weight=1)
G8.add_edge(1017,1206,weight=1)
G8.add_edge(1017,857,weight=1)
G8.add_edge(1017,1730,weight=1)
G8.add_edge(1017,1306,weight=1)
G8.add_edge(1017,728,weight=1)
G8.add_edge(1017,1671,weight=1)
G8.add_edge(1017,729,weight=1)
G8.add_edge(1017,1467,weight=1)
G8.add_edge(1024,1091,weight=1)
G8.add_edge(1030,1091,weight=1)
G8.add_edge(1030,2457,weight=1)
G8.add_edge(1504,1109,weight=1)
G8.add_edge(1504,1277,weight=1)
G8.add_edge(1504,1505,weight=1)
G8.add_edge(1504,1508,weight=1)
G8.add_edge(1504,1526,weight=1)
G8.add_edge(1504,1309,weight=1)
G8.add_edge(1505,1109,weight=1)
G8.add_edge(1505,1277,weight=1)
G8.add_edge(1505,1416,weight=1)
G8.add_edge(1505,1508,weight=1)
G8.add_edge(1505,1526,weight=1)
G8.add_edge(1505,1309,weight=1)
G8.add_edge(1507,1109,weight=1)
G8.add_edge(1507,1277,weight=1)
G8.add_edge(1507,1416,weight=1)
G8.add_edge(1507,1505,weight=1)
G8.add_edge(1507,1508,weight=1)
G8.add_edge(1507,1526,weight=1)
G8.add_edge(1507,1309,weight=1)
G8.add_edge(1508,1109,weight=1)
G8.add_edge(2608,2075,weight=1)
G8.add_edge(2608,2311,weight=1)
G8.add_edge(2608,2370,weight=1)
G8.add_edge(2608,1988,weight=1)
G8.add_edge(1637,2075,weight=1)
G8.add_edge(1637,1074,weight=1)
G8.add_edge(1637,929,weight=1)
G8.add_edge(1637,984,weight=1)
G8.add_edge(1637,1685,weight=1)
G8.add_edge(1637,246,weight=1)
G8.add_edge(2616,2075,weight=1)
G8.add_edge(2616,1648,weight=1)
G8.add_edge(1648,2075,weight=1)
G8.add_edge(1648,1685,weight=1)
G8.add_edge(1063,1324,weight=1)
G8.add_edge(1063,2652,weight=1)
G8.add_edge(1390,1080,weight=1)
G8.add_edge(1390,822,weight=1)
G8.add_edge(110,636,weight=1)
G8.add_edge(110,641,weight=1)
G8.add_edge(110,536,weight=1)
G8.add_edge(110,1297,weight=1)
G8.add_edge(110,1889,weight=1)
G8.add_edge(110,1075,weight=1)
G8.add_edge(110,460,weight=1)
G8.add_edge(110,673,weight=1)
G8.add_edge(110,885,weight=1)
G8.add_edge(110,2316,weight=1)
G8.add_edge(110,779,weight=1)
G8.add_edge(110,1662,weight=1)
G8.add_edge(110,172,weight=1)
G8.add_edge(110,956,weight=1)
G8.add_edge(110,508,weight=1)
G8.add_edge(110,489,weight=1)
G8.add_edge(110,968,weight=1)
G8.add_edge(110,2054,weight=1)
G8.add_edge(110,2055,weight=1)
G8.add_edge(115,172,weight=1)
G8.add_edge(2178,1884,weight=1)
G8.add_edge(540,1666,weight=1)
G8.add_edge(540,1211,weight=1)
G8.add_edge(540,1210,weight=1)
G8.add_edge(540,184,weight=1)
G8.add_edge(540,1454,weight=1)
G8.add_edge(540,455,weight=1)
G8.add_edge(540,542,weight=1)
G8.add_edge(1454,1153,weight=1)
G8.add_edge(1454,1211,weight=1)
G8.add_edge(646,52,weight=1)
G8.add_edge(646,313,weight=1)
G8.add_edge(646,743,weight=1)
G8.add_edge(646,212,weight=1)
G8.add_edge(2560,842,weight=1)
G8.add_edge(2560,52,weight=1)
G8.add_edge(2560,1242,weight=1)
G8.add_edge(2560,2077,weight=1)
G8.add_edge(2560,1748,weight=1)
G8.add_edge(2560,261,weight=1)
G8.add_edge(773,1503,weight=1)
G8.add_edge(773,769,weight=1)
G8.add_edge(773,52,weight=1)
G8.add_edge(773,2167,weight=1)
G8.add_edge(773,2168,weight=1)
G8.add_edge(773,313,weight=1)
G8.add_edge(773,743,weight=1)
G8.add_edge(773,1241,weight=1)
G8.add_edge(773,1242,weight=1)
G8.add_edge(773,2077,weight=1)
G8.add_edge(773,1148,weight=1)
G8.add_edge(1286,771,weight=1)
G8.add_edge(1286,406,weight=1)
G8.add_edge(1286,1299,weight=1)
G8.add_edge(1286,1177,weight=1)
G8.add_edge(1286,1383,weight=1)
G8.add_edge(1286,969,weight=1)
G8.add_edge(1294,2285,weight=1)
G8.add_edge(2307,2595,weight=1)
G8.add_edge(406,754,weight=1)
G8.add_edge(408,513,weight=1)
G8.add_edge(408,2440,weight=1)
G8.add_edge(408,1764,weight=1)
G8.add_edge(408,1299,weight=1)
G8.add_edge(408,1876,weight=1)
G8.add_edge(1299,408,weight=1)
G8.add_edge(410,2583,weight=1)
G8.add_edge(410,754,weight=1)
G8.add_edge(410,1607,weight=1)
G8.add_edge(410,1299,weight=1)
G8.add_edge(412,1892,weight=1)
G8.add_edge(412,513,weight=1)
G8.add_edge(412,2595,weight=1)
G8.add_edge(412,1471,weight=1)
G8.add_edge(412,537,weight=1)
G8.add_edge(412,2583,weight=1)
G8.add_edge(412,1601,weight=1)
G8.add_edge(412,2233,weight=1)
G8.add_edge(42,1734,weight=1)
G8.add_edge(42,890,weight=1)
G8.add_edge(42,95,weight=1)
G8.add_edge(42,1983,weight=1)
G8.add_edge(42,726,weight=1)
G8.add_edge(42,534,weight=1)
G8.add_edge(42,2457,weight=1)
G8.add_edge(42,1714,weight=1)
G8.add_edge(42,893,weight=1)
G8.add_edge(42,353,weight=1)
G8.add_edge(42,24,weight=1)
G8.add_edge(42,116,weight=1)
G8.add_edge(49,1784,weight=1)
G8.add_edge(49,2683,weight=1)
G8.add_edge(49,2457,weight=1)
G8.add_edge(491,7,weight=1)
G8.add_edge(491,221,weight=1)
G8.add_edge(491,6,weight=1)
G8.add_edge(491,493,weight=1)
G8.add_edge(491,735,weight=1)
G8.add_edge(491,46,weight=1)
G8.add_edge(491,214,weight=1)
G8.add_edge(493,2324,weight=1)
G8.add_edge(493,1187,weight=1)
G8.add_edge(493,735,weight=1)
G8.add_edge(493,46,weight=1)
G8.add_edge(493,214,weight=1)
G8.add_edge(514,118,weight=1)
G8.add_edge(514,453,weight=1)
G8.add_edge(2527,2449,weight=1)
G8.add_edge(1688,1229,weight=1)
G8.add_edge(1688,871,weight=1)
G8.add_edge(1688,774,weight=1)
G8.add_edge(1688,1190,weight=1)
G8.add_edge(1688,1633,weight=1)
G8.add_edge(1688,55,weight=1)
G8.add_edge(1688,1138,weight=1)
G8.add_edge(1688,149,weight=1)
G8.add_edge(1688,1558,weight=1)
G8.add_edge(1689,1229,weight=1)
G8.add_edge(1694,1229,weight=1)
G8.add_edge(257,2370,weight=1)
G8.add_edge(257,929,weight=1)
G8.add_edge(257,431,weight=1)
G8.add_edge(257,2226,weight=1)
G8.add_edge(257,635,weight=1)
G8.add_edge(257,984,weight=1)
G8.add_edge(2193,1744,weight=1)
G8.add_edge(2193,178,weight=1)
G8.add_edge(1218,1350,weight=1)
G8.add_edge(836,1249,weight=1)
G8.add_edge(133,347,weight=1)
G8.add_edge(2187,2322,weight=1)
G8.add_edge(2188,2315,weight=1)
G8.add_edge(2188,532,weight=1)
G8.add_edge(2188,2322,weight=1)
G8.add_edge(1313,1623,weight=1)
G8.add_edge(1313,1270,weight=1)
G8.add_edge(1313,1261,weight=1)
G8.add_edge(419,136,weight=1)
G8.add_edge(419,1313,weight=1)
G8.add_edge(419,1270,weight=1)
G8.add_edge(2462,2493,weight=1)
G8.add_edge(2462,1077,weight=1)
G8.add_edge(2462,1079,weight=1)
G8.add_edge(2463,1987,weight=1)
G8.add_edge(2463,402,weight=1)
G8.add_edge(558,424,weight=1)
G8.add_edge(559,2661,weight=1)
G8.add_edge(559,377,weight=1)
G8.add_edge(559,1935,weight=1)
G8.add_edge(559,2464,weight=1)
G8.add_edge(559,558,weight=1)
G8.add_edge(559,1138,weight=1)
G8.add_edge(559,1817,weight=1)
G8.add_edge(559,906,weight=1)
G8.add_edge(559,1642,weight=1)
G8.add_edge(559,1533,weight=1)
G8.add_edge(559,424,weight=1)
G8.add_edge(559,1556,weight=1)
G8.add_edge(559,2035,weight=1)
G8.add_edge(2468,2156,weight=1)
G8.add_edge(2468,2469,weight=1)
G8.add_edge(2468,324,weight=1)
G8.add_edge(642,1740,weight=1)
G8.add_edge(642,377,weight=1)
G8.add_edge(642,485,weight=1)
G8.add_edge(643,485,weight=1)
G8.add_edge(643,698,weight=1)
G8.add_edge(1575,1018,weight=1)
G8.add_edge(1575,485,weight=1)
G8.add_edge(657,774,weight=1)
G8.add_edge(657,658,weight=1)
G8.add_edge(657,149,weight=1)
G8.add_edge(657,651,weight=1)
G8.add_edge(1582,1448,weight=1)
G8.add_edge(1582,1712,weight=1)
G8.add_edge(1582,658,weight=1)
G8.add_edge(1932,2278,weight=1)
G8.add_edge(1932,2403,weight=1)
G8.add_edge(1932,1409,weight=1)
G8.add_edge(1932,1922,weight=1)
G8.add_edge(1199,1297,weight=1)
G8.add_edge(1199,860,weight=1)
G8.add_edge(1199,518,weight=1)
G8.add_edge(1199,419,weight=1)
G8.add_edge(1199,1171,weight=1)
G8.add_edge(1199,1684,weight=1)
G8.add_edge(746,601,weight=1)
G8.add_edge(746,459,weight=1)
G8.add_edge(746,1757,weight=1)
G8.add_edge(746,1764,weight=1)
G8.add_edge(746,667,weight=1)
G8.add_edge(746,1158,weight=1)
G8.add_edge(746,2163,weight=1)
G8.add_edge(746,387,weight=1)
G8.add_edge(746,688,weight=1)
G8.add_edge(746,772,weight=1)
G8.add_edge(746,111,weight=1)
G8.add_edge(746,2166,weight=1)
G8.add_edge(746,73,weight=1)
G8.add_edge(746,1565,weight=1)
G8.add_edge(746,19,weight=1)
G8.add_edge(746,478,weight=1)
G8.add_edge(746,2438,weight=1)
G8.add_edge(746,963,weight=1)
G8.add_edge(746,1933,weight=1)
G8.add_edge(746,1701,weight=1)
G8.add_edge(746,1705,weight=1)
G8.add_edge(746,37,weight=1)
G8.add_edge(746,2326,weight=1)
G8.add_edge(746,125,weight=1)
G8.add_edge(746,128,weight=1)
G8.add_edge(746,356,weight=1)
G8.add_edge(746,206,weight=1)
G8.add_edge(746,1887,weight=1)
G8.add_edge(746,2337,weight=1)
G8.add_edge(746,562,weight=1)
G8.add_edge(746,92,weight=1)
G8.add_edge(767,980,weight=1)
G8.add_edge(767,182,weight=1)
G8.add_edge(767,2533,weight=1)
G8.add_edge(767,79,weight=1)
G8.add_edge(767,519,weight=1)
G8.add_edge(1345,1139,weight=1)
G8.add_edge(1345,1122,weight=1)
G8.add_edge(1345,1048,weight=1)
G8.add_edge(1346,1345,weight=1)
G8.add_edge(1346,1047,weight=1)
G8.add_edge(1346,1048,weight=1)
G8.add_edge(1346,1641,weight=1)
G8.add_edge(1349,1048,weight=1)
G8.add_edge(1349,613,weight=1)
G8.add_edge(1355,1696,weight=1)
G8.add_edge(1355,314,weight=1)
G8.add_edge(1355,1421,weight=1)
G8.add_edge(1355,1090,weight=1)
G8.add_edge(1355,1305,weight=1)
G8.add_edge(1355,1466,weight=1)
G8.add_edge(1355,1040,weight=1)
G8.add_edge(1355,1041,weight=1)
G8.add_edge(1355,1318,weight=1)
G8.add_edge(1355,807,weight=1)
G8.add_edge(1355,993,weight=1)
G8.add_edge(1355,995,weight=1)
G8.add_edge(1355,1002,weight=1)
G8.add_edge(1355,912,weight=1)
G8.add_edge(1355,1124,weight=1)
G8.add_edge(1355,1189,weight=1)
G8.add_edge(1355,1177,weight=1)
G8.add_edge(1355,76,weight=1)
G8.add_edge(1355,1427,weight=1)
G8.add_edge(1355,1675,weight=1)
G8.add_edge(1355,1681,weight=1)
G8.add_edge(1355,1048,weight=1)
G8.add_edge(1860,549,weight=1)
G8.add_edge(1860,1792,weight=1)
G8.add_edge(1861,549,weight=1)
G8.add_edge(1861,907,weight=1)
G8.add_edge(1861,684,weight=1)
G8.add_edge(83,549,weight=1)
G8.add_edge(83,1441,weight=1)
G8.add_edge(83,306,weight=1)
G8.add_edge(83,240,weight=1)
G8.add_edge(83,411,weight=1)
G8.add_edge(83,1554,weight=1)
G8.add_edge(83,296,weight=1)
G8.add_edge(103,405,weight=1)
G8.add_edge(103,2013,weight=1)
G8.add_edge(103,1825,weight=1)
G8.add_edge(103,1894,weight=1)
G8.add_edge(103,1896,weight=1)
G8.add_edge(103,2334,weight=1)
G8.add_edge(103,1965,weight=1)
G8.add_edge(103,2584,weight=1)
G8.add_edge(1894,1896,weight=1)
G8.add_edge(1004,2147,weight=1)
G8.add_edge(1004,2029,weight=1)
G8.add_edge(1004,2484,weight=1)
G8.add_edge(1004,1344,weight=1)
G8.add_edge(1004,1365,weight=1)
G8.add_edge(1004,1470,weight=1)
G8.add_edge(1004,1898,weight=1)
G8.add_edge(1004,1005,weight=1)
G8.add_edge(1004,2183,weight=1)
G8.add_edge(1004,792,weight=1)
G8.add_edge(1004,787,weight=1)
G8.add_edge(1005,2147,weight=1)
G8.add_edge(1005,2484,weight=1)
G8.add_edge(1005,2351,weight=1)
G8.add_edge(1005,1344,weight=1)
G8.add_edge(1005,1365,weight=1)
G8.add_edge(1005,1470,weight=1)
G8.add_edge(1005,1004,weight=1)
G8.add_edge(1005,792,weight=1)
G8.add_edge(1005,787,weight=1)
G8.add_edge(2254,2214,weight=1)
G8.add_edge(2254,2592,weight=1)
G8.add_edge(2283,2218,weight=1)
G8.add_edge(2283,1687,weight=1)
G8.add_edge(486,2515,weight=1)
G8.add_edge(486,712,weight=1)
G8.add_edge(486,2381,weight=1)
G8.add_edge(486,1839,weight=1)
G8.add_edge(486,2562,weight=1)
G8.add_edge(2381,2295,weight=1)
G8.add_edge(2381,216,weight=1)
G8.add_edge(2381,2515,weight=1)
G8.add_edge(2381,1839,weight=1)
G8.add_edge(2381,2562,weight=1)
G8.add_edge(2381,289,weight=1)
G8.add_edge(2409,2146,weight=1)
G8.add_edge(2221,2479,weight=1)
G8.add_edge(2221,2140,weight=1)
G8.add_edge(2221,2601,weight=1)
G8.add_edge(2221,1959,weight=1)
G8.add_edge(2221,1410,weight=1)
G8.add_edge(2221,2564,weight=1)
G8.add_edge(2221,2227,weight=1)
G8.add_edge(2221,1941,weight=1)
G8.add_edge(2221,2253,weight=1)
G8.add_edge(2221,291,weight=1)
G8.add_edge(2221,1951,weight=1)
G8.add_edge(2227,2479,weight=1)
G8.add_edge(2227,2140,weight=1)
G8.add_edge(2227,1959,weight=1)
G8.add_edge(2227,1951,weight=1)
G8.add_edge(487,1598,weight=1)
G8.add_edge(487,232,weight=1)
G8.add_edge(487,1279,weight=1)
G8.add_edge(487,65,weight=1)
G8.add_edge(487,285,weight=1)
G8.add_edge(487,1149,weight=1)
G8.add_edge(487,286,weight=1)
G8.add_edge(487,2277,weight=1)
G8.add_edge(586,336,weight=1)
G8.add_edge(586,912,weight=1)
G8.add_edge(586,1276,weight=1)
G8.add_edge(586,2291,weight=1)
G8.add_edge(586,2292,weight=1)
G8.add_edge(586,1062,weight=1)
G8.add_edge(1645,987,weight=1)
G8.add_edge(1645,1863,weight=1)
G8.add_edge(1645,1864,weight=1)
G8.add_edge(1645,2485,weight=1)
G8.add_edge(1645,702,weight=1)
G8.add_edge(1645,2365,weight=1)
G8.add_edge(1645,823,weight=1)
G8.add_edge(1645,1733,weight=1)
G8.add_edge(1645,2410,weight=1)
G8.add_edge(1645,1257,weight=1)
G8.add_edge(1645,2615,weight=1)
G8.add_edge(1645,2617,weight=1)
G8.add_edge(1645,1329,weight=1)
G8.add_edge(1645,2424,weight=1)
G8.add_edge(1645,1965,weight=1)
G8.add_edge(1645,1602,weight=1)
G8.add_edge(1645,319,weight=1)
G8.add_edge(1645,323,weight=1)
G8.add_edge(1645,463,weight=1)
G8.add_edge(2615,987,weight=1)
G8.add_edge(2615,2485,weight=1)
G8.add_edge(2615,2617,weight=1)
G8.add_edge(2617,987,weight=1)
G8.add_edge(2617,2485,weight=1)
G8.add_edge(218,2356,weight=1)
G8.add_edge(218,686,weight=1)
G8.add_edge(2053,2063,weight=1)
G8.add_edge(2061,2063,weight=1)
G8.add_edge(2062,97,weight=1)
G8.add_edge(2062,216,weight=1)
G8.add_edge(2062,2063,weight=1)
G8.add_edge(2067,2063,weight=1)
G8.add_edge(1085,2036,weight=1)
G8.add_edge(1085,992,weight=1)
G8.add_edge(1085,859,weight=1)
G8.add_edge(465,468,weight=1)
G8.add_edge(465,574,weight=1)
G8.add_edge(468,2001,weight=1)
G8.add_edge(468,271,weight=1)
G8.add_edge(468,277,weight=1)
G8.add_edge(468,496,weight=1)
G8.add_edge(468,730,weight=1)
G8.add_edge(561,1376,weight=1)
G8.add_edge(561,2288,weight=1)
G8.add_edge(561,79,weight=1)
G8.add_edge(561,956,weight=1)
G8.add_edge(561,939,weight=1)
G8.add_edge(689,1249,weight=1)
G8.add_edge(689,253,weight=1)
G8.add_edge(689,836,weight=1)
G8.add_edge(203,772,weight=1)
G8.add_edge(203,205,weight=1)
G8.add_edge(205,1812,weight=1)
G8.add_edge(205,598,weight=1)
G8.add_edge(205,2411,weight=1)
G8.add_edge(205,2651,weight=1)
G8.add_edge(205,203,weight=1)
G8.add_edge(205,742,weight=1)
G8.add_edge(205,275,weight=1)
G8.add_edge(2033,2172,weight=1)
G8.add_edge(1291,2081,weight=1)
G8.add_edge(1291,2330,weight=1)
G8.add_edge(1291,1713,weight=1)
G8.add_edge(1291,1227,weight=1)
G8.add_edge(1291,1239,weight=1)
G8.add_edge(1291,1919,weight=1)
G8.add_edge(1291,219,weight=1)
G8.add_edge(1291,2170,weight=1)
G8.add_edge(1291,1645,weight=1)
G8.add_edge(1291,1192,weight=1)
G8.add_edge(1291,1198,weight=1)
G8.add_edge(1291,894,weight=1)
G8.add_edge(1291,1026,weight=1)
G8.add_edge(1291,424,weight=1)
G8.add_edge(1291,430,weight=1)
G8.add_edge(411,3,weight=1)
G8.add_edge(411,306,weight=1)
G8.add_edge(411,299,weight=1)
G8.add_edge(2708,2345,weight=1)
G8.add_edge(2708,1390,weight=1)
G8.add_edge(2708,775,weight=1)
G8.add_edge(2302,2589,weight=1)
G8.add_edge(2302,1908,weight=1)
G8.add_edge(2302,1902,weight=1)
G8.add_edge(2407,2079,weight=1)
G8.add_edge(158,1018,weight=1)
G8.add_edge(158,2231,weight=1)
G8.add_edge(158,1376,weight=1)
G8.add_edge(158,1638,weight=1)
G8.add_edge(158,1076,weight=1)
G8.add_edge(158,394,weight=1)
G8.add_edge(158,2681,weight=1)
G8.add_edge(158,898,weight=1)
G8.add_edge(158,717,weight=1)
G8.add_edge(158,1290,weight=1)
G8.add_edge(158,2020,weight=1)
G8.add_edge(158,2541,weight=1)
G8.add_edge(158,980,weight=1)
G8.add_edge(158,1039,weight=1)
G8.add_edge(158,215,weight=1)
G8.add_edge(158,767,weight=1)
G8.add_edge(158,2022,weight=1)
G8.add_edge(158,1719,weight=1)
G8.add_edge(158,561,weight=1)
G8.add_edge(158,182,weight=1)
G8.add_edge(158,741,weight=1)
G8.add_edge(158,79,weight=1)
G8.add_edge(158,2096,weight=1)
G8.add_edge(158,917,weight=1)
G8.add_edge(158,1453,weight=1)
G8.add_edge(158,372,weight=1)
G8.add_edge(158,1551,weight=1)
G8.add_edge(158,1341,weight=1)
G8.add_edge(158,2418,weight=1)
G8.add_edge(161,2682,weight=1)
G8.add_edge(161,470,weight=1)
G8.add_edge(161,2541,weight=1)
G8.add_edge(161,126,weight=1)
G8.add_edge(161,1858,weight=1)
G8.add_edge(161,323,weight=1)
G8.add_edge(1284,824,weight=1)
G8.add_edge(1284,1319,weight=1)
G8.add_edge(1284,1548,weight=1)
G8.add_edge(1284,1285,weight=1)
G8.add_edge(1284,231,weight=1)
G8.add_edge(1284,392,weight=1)
G8.add_edge(1285,1056,weight=1)
G8.add_edge(1285,1170,weight=1)
G8.add_edge(1080,1104,weight=1)
G8.add_edge(1080,865,weight=1)
G8.add_edge(1080,976,weight=1)
G8.add_edge(1080,1010,weight=1)
G8.add_edge(1080,1725,weight=1)
G8.add_edge(1080,1719,weight=1)
G8.add_edge(1080,1316,weight=1)
G8.add_edge(1080,612,weight=1)
G8.add_edge(1113,1404,weight=1)
G8.add_edge(1113,1074,weight=1)
G8.add_edge(1113,927,weight=1)
G8.add_edge(1113,2416,weight=1)
G8.add_edge(1113,1513,weight=1)
G8.add_edge(1113,102,weight=1)
G8.add_edge(1119,1541,weight=1)
G8.add_edge(1119,135,weight=1)
G8.add_edge(2091,1261,weight=1)
G8.add_edge(2183,2147,weight=1)
G8.add_edge(2183,2634,weight=1)
G8.add_edge(2183,1836,weight=1)
G8.add_edge(2183,1749,weight=1)
G8.add_edge(656,199,weight=1)
G8.add_edge(231,521,weight=1)
G8.add_edge(231,1285,weight=1)
G8.add_edge(231,407,weight=1)
G8.add_edge(231,392,weight=1)
G8.add_edge(1096,1457,weight=1)
G8.add_edge(1096,1434,weight=1)
G8.add_edge(309,10,weight=1)
G8.add_edge(309,2029,weight=1)
G8.add_edge(309,1457,weight=1)
G8.add_edge(2172,2032,weight=1)
G8.add_edge(320,30,weight=1)
G8.add_edge(320,1274,weight=1)
G8.add_edge(1186,1522,weight=1)
G8.add_edge(1186,1001,weight=1)
G8.add_edge(321,260,weight=1)
G8.add_edge(1192,693,weight=1)
G8.add_edge(1192,1054,weight=1)
G8.add_edge(625,113,weight=1)
G8.add_edge(625,12,weight=1)
G8.add_edge(625,701,weight=1)
G8.add_edge(625,573,weight=1)
G8.add_edge(626,701,weight=1)
G8.add_edge(628,1058,weight=1)
G8.add_edge(628,339,weight=1)
G8.add_edge(628,1563,weight=1)
G8.add_edge(628,1680,weight=1)
G8.add_edge(628,1271,weight=1)
G8.add_edge(1563,1638,weight=1)
G8.add_edge(1563,356,weight=1)
G8.add_edge(2544,2441,weight=1)
G8.add_edge(2544,2410,weight=1)
G8.add_edge(2544,1480,weight=1)
G8.add_edge(2544,1777,weight=1)
G8.add_edge(2544,2675,weight=1)
G8.add_edge(632,1058,weight=1)
G8.add_edge(632,384,weight=1)
G8.add_edge(632,1563,weight=1)
G8.add_edge(733,268,weight=1)
G8.add_edge(733,1898,weight=1)
G8.add_edge(733,21,weight=1)
G8.add_edge(2654,2029,weight=1)
G8.add_edge(2654,1364,weight=1)
G8.add_edge(1698,930,weight=1)
G8.add_edge(1698,1600,weight=1)
G8.add_edge(1698,17,weight=1)
G8.add_edge(1698,1704,weight=1)
G8.add_edge(1698,1560,weight=1)
G8.add_edge(1698,1665,weight=1)
G8.add_edge(1698,1668,weight=1)
G8.add_edge(1698,1684,weight=1)
G8.add_edge(1698,805,weight=1)
G8.add_edge(741,930,weight=1)
G8.add_edge(741,165,weight=1)
G8.add_edge(741,419,weight=1)
G8.add_edge(741,561,weight=1)
G8.add_edge(741,1704,weight=1)
G8.add_edge(741,1560,weight=1)
G8.add_edge(741,1665,weight=1)
G8.add_edge(741,1668,weight=1)
G8.add_edge(741,1684,weight=1)
G8.add_edge(2659,1861,weight=1)
G8.add_edge(2659,1560,weight=1)
G8.add_edge(2659,692,weight=1)
G8.add_edge(1704,1560,weight=1)
G8.add_edge(1020,854,weight=1)
G8.add_edge(1020,907,weight=1)
G8.add_edge(1020,927,weight=1)
G8.add_edge(1020,863,weight=1)
G8.add_edge(1020,928,weight=1)
G8.add_edge(1020,1257,weight=1)
G8.add_edge(1020,1291,weight=1)
G8.add_edge(2300,1855,weight=1)
G8.add_edge(2300,1856,weight=1)
G8.add_edge(1541,1046,weight=1)
G8.add_edge(1541,843,weight=1)
G8.add_edge(1541,1444,weight=1)
G8.add_edge(622,1860,weight=1)
G8.add_edge(622,472,weight=1)
G8.add_edge(2530,2295,weight=1)
G8.add_edge(2530,2522,weight=1)
G8.add_edge(2537,503,weight=1)
G8.add_edge(2537,1958,weight=1)
G8.add_edge(2537,2630,weight=1)
G8.add_edge(1560,930,weight=1)
G8.add_edge(1560,1792,weight=1)
G8.add_edge(1560,1665,weight=1)
G8.add_edge(1560,1684,weight=1)
G8.add_edge(630,222,weight=1)
G8.add_edge(630,613,weight=1)
G8.add_edge(2545,2569,weight=1)
G8.add_edge(634,198,weight=1)
G8.add_edge(637,119,weight=1)
G8.add_edge(637,2028,weight=1)
G8.add_edge(637,2174,weight=1)
G8.add_edge(637,2673,weight=1)
G8.add_edge(637,2175,weight=1)
G8.add_edge(637,2205,weight=1)
G8.add_edge(637,2234,weight=1)
G8.add_edge(637,765,weight=1)
G8.add_edge(1574,1674,weight=1)
G8.add_edge(1574,1433,weight=1)
G8.add_edge(1574,637,weight=1)
G8.add_edge(1574,2175,weight=1)
G8.add_edge(1574,765,weight=1)
G8.add_edge(1574,1482,weight=1)
G8.add_edge(1691,996,weight=1)
G8.add_edge(739,1586,weight=1)
G8.add_edge(739,1377,weight=1)
G8.add_edge(739,1214,weight=1)
G8.add_edge(739,1742,weight=1)
G8.add_edge(739,1728,weight=1)
G8.add_edge(739,622,weight=1)
G8.add_edge(739,211,weight=1)
G8.add_edge(739,565,weight=1)
G8.add_edge(739,576,weight=1)
G8.add_edge(739,2629,weight=1)
G8.add_edge(739,2630,weight=1)
G8.add_edge(739,1184,weight=1)
G8.add_edge(2656,1976,weight=1)
G8.add_edge(2667,452,weight=1)
G8.add_edge(2667,2454,weight=1)
G8.add_edge(2667,2412,weight=1)
G8.add_edge(750,198,weight=1)
G8.add_edge(751,245,weight=1)
G8.add_edge(844,1102,weight=1)
G8.add_edge(844,2659,weight=1)
G8.add_edge(844,569,weight=1)
G8.add_edge(844,576,weight=1)
G8.add_edge(859,2036,weight=1)
G8.add_edge(859,992,weight=1)
G8.add_edge(859,282,weight=1)
G8.add_edge(58,307,weight=1)
G8.add_edge(61,253,weight=1)
G8.add_edge(61,689,weight=1)
G8.add_edge(876,934,weight=1)
G8.add_edge(876,555,weight=1)
G8.add_edge(876,776,weight=1)
G8.add_edge(876,371,weight=1)
G8.add_edge(876,1193,weight=1)
G8.add_edge(1842,1742,weight=1)
G8.add_edge(1842,565,weight=1)
G8.add_edge(64,740,weight=1)
G8.add_edge(64,1547,weight=1)
G8.add_edge(64,211,weight=1)
G8.add_edge(64,154,weight=1)
G8.add_edge(1955,99,weight=1)
G8.add_edge(1955,2184,weight=1)
G8.add_edge(1955,979,weight=1)
G8.add_edge(1955,2223,weight=1)
G8.add_edge(1955,2151,weight=1)
G8.add_edge(1955,322,weight=1)
G8.add_edge(163,174,weight=1)
G8.add_edge(163,776,weight=1)
G8.add_edge(2093,2016,weight=1)
G8.add_edge(2099,2157,weight=1)
G8.add_edge(2099,2100,weight=1)
G8.add_edge(1125,454,weight=1)
G8.add_edge(1125,1567,weight=1)
G8.add_edge(1125,1568,weight=1)
G8.add_edge(2362,2176,weight=1)
G8.add_edge(2362,2217,weight=1)
G8.add_edge(2362,140,weight=1)
G8.add_edge(578,2661,weight=1)
G8.add_edge(578,1503,weight=1)
G8.add_edge(578,390,weight=1)
G8.add_edge(578,313,weight=1)
G8.add_edge(578,2493,weight=1)
G8.add_edge(578,251,weight=1)
G8.add_edge(578,1533,weight=1)
G8.add_edge(578,402,weight=1)
G8.add_edge(578,2035,weight=1)
G8.add_edge(2493,1901,weight=1)
G8.add_edge(2493,2548,weight=1)
G8.add_edge(351,181,weight=1)
G8.add_edge(351,112,weight=1)
G8.add_edge(351,731,weight=1)
G8.add_edge(351,202,weight=1)
G8.add_edge(351,771,weight=1)
G8.add_edge(351,665,weight=1)
G8.add_edge(351,428,weight=1)
G8.add_edge(351,352,weight=1)
G8.add_edge(665,979,weight=1)
G8.add_edge(1594,979,weight=1)
G8.add_edge(1594,1438,weight=1)
G8.add_edge(1594,1117,weight=1)
G8.add_edge(1594,1289,weight=1)
G8.add_edge(2574,1943,weight=1)
G8.add_edge(2025,738,weight=1)
G8.add_edge(2025,428,weight=1)
G8.add_edge(2025,1933,weight=1)
G8.add_edge(2025,1779,weight=1)
G8.add_edge(228,824,weight=1)
G8.add_edge(228,1319,weight=1)
G8.add_edge(228,1548,weight=1)
G8.add_edge(228,2320,weight=1)
G8.add_edge(228,297,weight=1)
G8.add_edge(1171,518,weight=1)
G8.add_edge(1360,1058,weight=1)
G8.add_edge(2368,2578,weight=1)
G8.add_edge(2368,1834,weight=1)
G8.add_edge(1463,1658,weight=1)
G8.add_edge(1463,1458,weight=1)
G8.add_edge(1,545,weight=1)
G8.add_edge(1,9,weight=1)
G8.add_edge(1,436,weight=1)
G8.add_edge(9,436,weight=1)
G8.add_edge(15,259,weight=1)
G8.add_edge(15,438,weight=1)
G8.add_edge(15,401,weight=1)
G8.add_edge(15,394,weight=1)
G8.add_edge(15,233,weight=1)
G8.add_edge(15,545,weight=1)
G8.add_edge(15,747,weight=1)
G8.add_edge(15,243,weight=1)
G8.add_edge(15,709,weight=1)
G8.add_edge(15,32,weight=1)
G8.add_edge(15,737,weight=1)
G8.add_edge(15,269,weight=1)
G8.add_edge(15,385,weight=1)
G8.add_edge(15,1,weight=1)
G8.add_edge(15,9,weight=1)
G8.add_edge(15,614,weight=1)
G8.add_edge(15,752,weight=1)
G8.add_edge(15,436,weight=1)
G8.add_edge(15,444,weight=1)
G8.add_edge(1909,2515,weight=1)
G8.add_edge(1909,443,weight=1)
G8.add_edge(1909,1648,weight=1)
G8.add_edge(1909,1894,weight=1)
G8.add_edge(1909,1839,weight=1)
G8.add_edge(1909,2562,weight=1)
G8.add_edge(1909,1583,weight=1)
G8.add_edge(1910,2678,weight=1)
G8.add_edge(1910,1546,weight=1)
G8.add_edge(1910,633,weight=1)
G8.add_edge(1910,1583,weight=1)
G8.add_edge(1911,2160,weight=1)
G8.add_edge(1911,706,weight=1)
G8.add_edge(1911,1669,weight=1)
G8.add_edge(1911,2417,weight=1)
G8.add_edge(1911,2507,weight=1)
G8.add_edge(1911,2243,weight=1)
G8.add_edge(1911,2376,weight=1)
G8.add_edge(1911,2489,weight=1)
G8.add_edge(1911,2367,weight=1)
G8.add_edge(1911,2576,weight=1)
G8.add_edge(1911,2082,weight=1)
G8.add_edge(400,1251,weight=1)
G8.add_edge(400,1252,weight=1)
G8.add_edge(1426,526,weight=1)
G8.add_edge(1426,685,weight=1)
G8.add_edge(614,233,weight=1)
G8.add_edge(614,2572,weight=1)
G8.add_edge(614,269,weight=1)
G8.add_edge(614,617,weight=1)
G8.add_edge(617,269,weight=1)
G8.add_edge(16,235,weight=1)
G8.add_edge(16,702,weight=1)
G8.add_edge(16,255,weight=1)
G8.add_edge(16,153,weight=1)
G8.add_edge(16,400,weight=1)
G8.add_edge(16,254,weight=1)
G8.add_edge(16,130,weight=1)
G8.add_edge(16,539,weight=1)
G8.add_edge(16,323,weight=1)
G8.add_edge(16,81,weight=1)
G8.add_edge(2016,2328,weight=1)
G8.add_edge(2016,2035,weight=1)
G8.add_edge(1051,1390,weight=1)
G8.add_edge(1051,1307,weight=1)
G8.add_edge(1051,1592,weight=1)
G8.add_edge(1051,1493,weight=1)
G8.add_edge(1051,1500,weight=1)
G8.add_edge(1051,1172,weight=1)
G8.add_edge(623,346,weight=1)
G8.add_edge(623,378,weight=1)
G8.add_edge(2533,2418,weight=1)
G8.add_edge(46,221,weight=1)
G8.add_edge(1394,1503,weight=1)
G8.add_edge(1394,1129,weight=1)
G8.add_edge(1394,1098,weight=1)
G8.add_edge(1394,977,weight=1)
G8.add_edge(1394,1415,weight=1)
G8.add_edge(1394,1438,weight=1)
G8.add_edge(1394,1398,weight=1)
G8.add_edge(1394,906,weight=1)
G8.add_edge(594,301,weight=1)
G8.add_edge(594,595,weight=1)
G8.add_edge(2510,1963,weight=1)
G8.add_edge(2510,518,weight=1)
G8.add_edge(2510,2619,weight=1)
G8.add_edge(699,361,weight=1)
G8.add_edge(699,475,weight=1)
G8.add_edge(699,1432,weight=1)
G8.add_edge(2179,2546,weight=1)
G8.add_edge(2179,2583,weight=1)
G8.add_edge(2179,1789,weight=1)
G8.add_edge(2179,1794,weight=1)
G8.add_edge(2179,1796,weight=1)
G8.add_edge(2182,2144,weight=1)
G8.add_edge(2182,2179,weight=1)
G8.add_edge(2182,1789,weight=1)
G8.add_edge(2182,1794,weight=1)
G8.add_edge(2182,1796,weight=1)
G8.add_edge(344,1275,weight=1)
G8.add_edge(344,1572,weight=1)
G8.add_edge(344,531,weight=1)
G8.add_edge(344,1304,weight=1)
G8.add_edge(344,1099,weight=1)
G8.add_edge(344,1578,weight=1)
G8.add_edge(1304,1275,weight=1)
G8.add_edge(1304,2348,weight=1)
G8.add_edge(1306,1275,weight=1)
G8.add_edge(1306,1741,weight=1)
G8.add_edge(1306,1576,weight=1)
G8.add_edge(1306,1066,weight=1)
G8.add_edge(1306,1099,weight=1)
G8.add_edge(1526,1277,weight=1)
G8.add_edge(1526,1416,weight=1)
G8.add_edge(810,1191,weight=1)
G8.add_edge(810,1230,weight=1)
G8.add_edge(810,1640,weight=1)
G8.add_edge(810,298,weight=1)
G8.add_edge(810,1223,weight=1)
G8.add_edge(810,1045,weight=1)
G8.add_edge(810,909,weight=1)
G8.add_edge(810,1616,weight=1)
G8.add_edge(810,813,weight=1)
G8.add_edge(810,815,weight=1)
G8.add_edge(810,920,weight=1)
G8.add_edge(810,1081,weight=1)
G8.add_edge(810,1366,weight=1)
G8.add_edge(813,1045,weight=1)
G8.add_edge(813,1081,weight=1)
G8.add_edge(813,1366,weight=1)
G8.add_edge(815,1045,weight=1)
G8.add_edge(815,813,weight=1)
G8.add_edge(815,1366,weight=1)
G8.add_edge(126,395,weight=1)
G8.add_edge(126,1858,weight=1)
G8.add_edge(134,395,weight=1)
G8.add_edge(1432,1182,weight=1)
G8.add_edge(1432,1413,weight=1)
G8.add_edge(1432,862,weight=1)
G8.add_edge(2675,2067,weight=1)
G8.add_edge(2675,1353,weight=1)
G8.add_edge(762,536,weight=1)
G8.add_edge(762,10,weight=1)
G8.add_edge(762,41,weight=1)
G8.add_edge(130,2364,weight=1)
G8.add_edge(130,2447,weight=1)
G8.add_edge(130,1974,weight=1)
G8.add_edge(130,1410,weight=1)
G8.add_edge(130,189,weight=1)
G8.add_edge(130,2221,weight=1)
G8.add_edge(130,2166,weight=1)
G8.add_edge(130,2452,weight=1)
G8.add_edge(130,539,weight=1)
G8.add_edge(130,23,weight=1)
G8.add_edge(130,2194,weight=1)
G8.add_edge(130,1205,weight=1)
G8.add_edge(2308,1773,weight=1)
G8.add_edge(2308,407,weight=1)
G8.add_edge(2308,1940,weight=1)
G8.add_edge(407,312,weight=1)
G8.add_edge(541,544,weight=1)
G8.add_edge(541,604,weight=1)
G8.add_edge(541,1122,weight=1)
G8.add_edge(541,1641,weight=1)
G8.add_edge(63,325,weight=1)
G8.add_edge(63,333,weight=1)
G8.add_edge(63,2426,weight=1)
G8.add_edge(63,946,weight=1)
G8.add_edge(63,2475,weight=1)
G8.add_edge(63,2567,weight=1)
G8.add_edge(63,424,weight=1)
G8.add_edge(882,1120,weight=1)
G8.add_edge(882,2296,weight=1)
G8.add_edge(882,113,weight=1)
G8.add_edge(882,817,weight=1)
G8.add_edge(882,2703,weight=1)
G8.add_edge(882,573,weight=1)
G8.add_edge(882,2204,weight=1)
G8.add_edge(882,2406,weight=1)
G8.add_edge(882,1412,weight=1)
G8.add_edge(882,2455,weight=1)
G8.add_edge(1970,2310,weight=1)
G8.add_edge(1970,2364,weight=1)
G8.add_edge(1970,2052,weight=1)
G8.add_edge(1970,2201,weight=1)
G8.add_edge(1976,2690,weight=1)
G8.add_edge(1976,2409,weight=1)
G8.add_edge(1976,2656,weight=1)
G8.add_edge(1117,709,weight=1)
G8.add_edge(1117,32,weight=1)
G8.add_edge(1117,737,weight=1)
G8.add_edge(1117,1594,weight=1)
G8.add_edge(251,171,weight=1)
G8.add_edge(251,695,weight=1)
G8.add_edge(251,709,weight=1)
G8.add_edge(251,32,weight=1)
G8.add_edge(251,737,weight=1)
G8.add_edge(251,640,weight=1)
G8.add_edge(251,499,weight=1)
G8.add_edge(251,500,weight=1)
G8.add_edge(251,585,weight=1)
G8.add_edge(251,305,weight=1)
G8.add_edge(251,108,weight=1)
G8.add_edge(251,109,weight=1)
G8.add_edge(251,34,weight=1)
G8.add_edge(2420,2324,weight=1)
G8.add_edge(2420,493,weight=1)
G8.add_edge(2420,1269,weight=1)
G8.add_edge(2424,854,weight=1)
G8.add_edge(2424,551,weight=1)
G8.add_edge(2424,863,weight=1)
G8.add_edge(2424,1733,weight=1)
G8.add_edge(2424,1645,weight=1)
G8.add_edge(2424,1192,weight=1)
G8.add_edge(2424,2226,weight=1)
G8.add_edge(2424,2628,weight=1)
G8.add_edge(2424,882,weight=1)
G8.add_edge(2424,2658,weight=1)
G8.add_edge(2424,1162,weight=1)
G8.add_edge(2424,1602,weight=1)
G8.add_edge(1423,551,weight=1)
G8.add_edge(1423,1658,weight=1)
G8.add_edge(1423,1274,weight=1)
G8.add_edge(1423,1224,weight=1)
G8.add_edge(1423,1238,weight=1)
G8.add_edge(2429,2043,weight=1)
G8.add_edge(2429,117,weight=1)
G8.add_edge(2429,226,weight=1)
G8.add_edge(2429,761,weight=1)
G8.add_edge(2429,45,weight=1)
G8.add_edge(2429,2022,weight=1)
G8.add_edge(2429,1861,weight=1)
G8.add_edge(2431,1861,weight=1)
G8.add_edge(1112,1102,weight=1)
G8.add_edge(1112,1413,weight=1)
G8.add_edge(1112,2293,weight=1)
G8.add_edge(2342,2426,weight=1)
G8.add_edge(2342,560,weight=1)
G8.add_edge(2342,624,weight=1)
G8.add_edge(2342,789,weight=1)
G8.add_edge(2342,1758,weight=1)
G8.add_edge(2342,1843,weight=1)
G8.add_edge(2216,1878,weight=1)
G8.add_edge(2216,2651,weight=1)
G8.add_edge(2216,345,weight=1)
G8.add_edge(2216,2058,weight=1)
G8.add_edge(2216,676,weight=1)
G8.add_edge(2347,398,weight=1)
G8.add_edge(2347,1968,weight=1)
G8.add_edge(5,171,weight=1)
G8.add_edge(788,1438,weight=1)
G8.add_edge(789,2426,weight=1)
G8.add_edge(789,946,weight=1)
G8.add_edge(789,2324,weight=1)
G8.add_edge(789,1415,weight=1)
G8.add_edge(789,1269,weight=1)
G8.add_edge(416,996,weight=1)
G8.add_edge(416,790,weight=1)
G8.add_edge(416,2665,weight=1)
G8.add_edge(416,2104,weight=1)
G8.add_edge(416,1738,weight=1)
G8.add_edge(416,1833,weight=1)
G8.add_edge(416,1837,weight=1)
G8.add_edge(416,210,weight=1)
G8.add_edge(416,1791,weight=1)
G8.add_edge(416,2354,weight=1)
G8.add_edge(416,713,weight=1)
G8.add_edge(672,1600,weight=1)
G8.add_edge(760,94,weight=1)
G8.add_edge(760,96,weight=1)
G8.add_edge(760,162,weight=1)
G8.add_edge(760,100,weight=1)
G8.add_edge(760,293,weight=1)
G8.add_edge(760,24,weight=1)
G8.add_edge(1007,473,weight=1)
G8.add_edge(1007,243,weight=1)
G8.add_edge(1007,877,weight=1)
G8.add_edge(1007,2041,weight=1)
G8.add_edge(1007,1026,weight=1)
G8.add_edge(1007,651,weight=1)
G8.add_edge(195,674,weight=1)
G8.add_edge(1139,1598,weight=1)
G8.add_edge(1139,1421,weight=1)
G8.add_edge(1139,807,weight=1)
G8.add_edge(285,65,weight=1)
G8.add_edge(1149,1598,weight=1)
G8.add_edge(1149,1356,weight=1)
G8.add_edge(286,232,weight=1)
G8.add_edge(286,1149,weight=1)
G8.add_edge(633,684,weight=1)
G8.add_edge(633,402,weight=1)
G8.add_edge(1570,1074,weight=1)
G8.add_edge(1570,1610,weight=1)
G8.add_edge(1570,1722,weight=1)
G8.add_edge(1570,1727,weight=1)
G8.add_edge(1570,1009,weight=1)
G8.add_edge(1570,1011,weight=1)
G8.add_edge(1570,1012,weight=1)
G8.add_edge(1570,878,weight=1)
G8.add_edge(1570,1487,weight=1)
G8.add_edge(1570,431,weight=1)
G8.add_edge(1570,1588,weight=1)
G8.add_edge(1570,209,weight=1)
G8.add_edge(1570,1328,weight=1)
G8.add_edge(1570,1680,weight=1)
G8.add_edge(1570,1685,weight=1)
G8.add_edge(1570,1183,weight=1)
G8.add_edge(1571,1110,weight=1)
G8.add_edge(1571,1112,weight=1)
G8.add_edge(1571,635,weight=1)
G8.add_edge(1571,1165,weight=1)
G8.add_edge(635,2594,weight=1)
G8.add_edge(635,1514,weight=1)
G8.add_edge(635,431,weight=1)
G8.add_edge(635,687,weight=1)
G8.add_edge(1583,2678,weight=1)
G8.add_edge(1583,1546,weight=1)
G8.add_edge(1583,633,weight=1)
G8.add_edge(1592,1702,weight=1)
G8.add_edge(1592,34,weight=1)
G8.add_edge(1592,37,weight=1)
G8.add_edge(1702,34,weight=1)
G8.add_edge(1702,833,weight=1)
G8.add_edge(168,560,weight=1)
G8.add_edge(1019,1143,weight=1)
G8.add_edge(1528,2421,weight=1)
G8.add_edge(1528,1544,weight=1)
G8.add_edge(1528,2525,weight=1)
G8.add_edge(1528,2526,weight=1)
G8.add_edge(1528,2222,weight=1)
G8.add_edge(1528,1087,weight=1)
G8.add_edge(1528,2167,weight=1)
G8.add_edge(1528,828,weight=1)
G8.add_edge(1528,2562,weight=1)
G8.add_edge(1528,1241,weight=1)
G8.add_edge(1528,910,weight=1)
G8.add_edge(1528,972,weight=1)
G8.add_edge(1528,2077,weight=1)
G8.add_edge(1528,420,weight=1)
G8.add_edge(1528,1748,weight=1)
G8.add_edge(1528,2089,weight=1)
G8.add_edge(1528,261,weight=1)
G8.add_edge(1528,212,weight=1)
G8.add_edge(79,2682,weight=1)
G8.add_edge(79,470,weight=1)
G8.add_edge(1858,2682,weight=1)
G8.add_edge(1868,1755,weight=1)
G8.add_edge(1870,1755,weight=1)
G8.add_edge(1870,2410,weight=1)
G8.add_edge(89,763,weight=1)
G8.add_edge(89,588,weight=1)
G8.add_edge(89,117,weight=1)
G8.add_edge(89,226,weight=1)
G8.add_edge(89,761,weight=1)
G8.add_edge(89,45,weight=1)
G8.add_edge(89,683,weight=1)
G8.add_edge(89,209,weight=1)
G8.add_edge(89,372,weight=1)
G8.add_edge(2507,2417,weight=1)
G8.add_edge(2507,2576,weight=1)
G8.add_edge(2507,2459,weight=1)
G8.add_edge(1813,2365,weight=1)
G8.add_edge(1813,863,weight=1)
G8.add_edge(1813,1514,weight=1)
G8.add_edge(1813,1408,weight=1)
G8.add_edge(1813,1156,weight=1)
G8.add_edge(1813,784,weight=1)
G8.add_edge(1813,786,weight=1)
G8.add_edge(1375,870,weight=1)
G8.add_edge(2606,1402,weight=1)
G8.add_edge(2606,2503,weight=1)
G8.add_edge(2606,952,weight=1)
G8.add_edge(956,1281,weight=1)
G8.add_edge(220,403,weight=1)
G8.add_edge(220,1326,weight=1)
G8.add_edge(220,1739,weight=1)
G8.add_edge(220,39,weight=1)
G8.add_edge(220,2564,weight=1)
G8.add_edge(220,1690,weight=1)
G8.add_edge(220,564,weight=1)
G8.add_edge(220,847,weight=1)
G8.add_edge(220,758,weight=1)
G8.add_edge(220,1405,weight=1)
G8.add_edge(220,713,weight=1)
G8.add_edge(2613,725,weight=1)
G8.add_edge(2613,1904,weight=1)
G8.add_edge(2613,2626,weight=1)
G8.add_edge(2613,27,weight=1)
G8.add_edge(792,1662,weight=1)
G8.add_edge(793,1096,weight=1)
G8.add_edge(799,1434,weight=1)
G8.add_edge(799,1096,weight=1)
G8.add_edge(799,793,weight=1)
G8.add_edge(799,508,weight=1)
G8.add_edge(799,1106,weight=1)
G8.add_edge(1565,900,weight=1)
G8.add_edge(1566,900,weight=1)
G8.add_edge(1566,1089,weight=1)
G8.add_edge(2042,1745,weight=1)
G8.add_edge(2050,2219,weight=1)
G8.add_edge(2050,2411,weight=1)
G8.add_edge(2050,2307,weight=1)
G8.add_edge(727,1419,weight=1)
G8.add_edge(727,1069,weight=1)
G8.add_edge(752,545,weight=1)
G8.add_edge(752,9,weight=1)
G8.add_edge(752,436,weight=1)
G8.add_edge(383,30,weight=1)
G8.add_edge(383,2133,weight=1)
G8.add_edge(383,1838,weight=1)
G8.add_edge(2402,1989,weight=1)
G8.add_edge(728,1800,weight=1)
G8.add_edge(847,782,weight=1)
G8.add_edge(847,2174,weight=1)
G8.add_edge(847,1534,weight=1)
G8.add_edge(847,2205,weight=1)
G8.add_edge(847,220,weight=1)
G8.add_edge(847,758,weight=1)
G8.add_edge(145,189,weight=1)
G8.add_edge(145,445,weight=1)
G8.add_edge(145,2103,weight=1)
G8.add_edge(2635,1877,weight=1)
G8.add_edge(2635,1198,weight=1)
G8.add_edge(848,1403,weight=1)
G8.add_edge(848,1469,weight=1)
G8.add_edge(848,711,weight=1)
G8.add_edge(848,849,weight=1)
G8.add_edge(848,1033,weight=1)
G8.add_edge(849,1389,weight=1)
G8.add_edge(849,1740,weight=1)
G8.add_edge(849,1401,weight=1)
G8.add_edge(849,1469,weight=1)
G8.add_edge(849,1608,weight=1)
G8.add_edge(849,1224,weight=1)
G8.add_edge(849,848,weight=1)
G8.add_edge(849,1033,weight=1)
G8.add_edge(135,1783,weight=1)
G8.add_edge(135,506,weight=1)
G8.add_edge(1322,1042,weight=1)
G8.add_edge(1322,1191,weight=1)
G8.add_edge(1322,298,weight=1)
G8.add_edge(1322,1223,weight=1)
G8.add_edge(1322,815,weight=1)
G8.add_edge(1322,1081,weight=1)
G8.add_edge(2445,2258,weight=1)
G8.add_edge(2445,501,weight=1)
G8.add_edge(539,403,weight=1)
G8.add_edge(539,2310,weight=1)
G8.add_edge(539,488,weight=1)
G8.add_edge(539,1739,weight=1)
G8.add_edge(539,1690,weight=1)
G8.add_edge(539,2166,weight=1)
G8.add_edge(539,2194,weight=1)
G8.add_edge(539,1205,weight=1)
G8.add_edge(539,1037,weight=1)
G8.add_edge(238,330,weight=1)
G8.add_edge(238,1061,weight=1)
G8.add_edge(655,136,weight=1)
G8.add_edge(655,1261,weight=1)
G8.add_edge(662,1192,weight=1)
G8.add_edge(1587,1389,weight=1)
G8.add_edge(1596,1863,weight=1)
G8.add_edge(1596,1864,weight=1)
G8.add_edge(1596,1658,weight=1)
G8.add_edge(1596,1825,weight=1)
G8.add_edge(1596,1577,weight=1)
G8.add_edge(1596,837,weight=1)
G8.add_edge(1596,1302,weight=1)
G8.add_edge(1596,1777,weight=1)
G8.add_edge(1596,2347,weight=1)
G8.add_edge(1596,1965,weight=1)
G8.add_edge(1596,2676,weight=1)
G8.add_edge(1596,152,weight=1)
G8.add_edge(2658,1162,weight=1)
G8.add_edge(1703,1200,weight=1)
G8.add_edge(1703,1162,weight=1)
G8.add_edge(2676,1863,weight=1)
G8.add_edge(2676,1825,weight=1)
G8.add_edge(2676,1257,weight=1)
G8.add_edge(2676,1965,weight=1)
G8.add_edge(2136,2013,weight=1)
G8.add_edge(2136,2024,weight=1)
G8.add_edge(2136,2334,weight=1)
G8.add_edge(2136,135,weight=1)
G8.add_edge(2136,2584,weight=1)
G8.add_edge(2124,2483,weight=1)
G8.add_edge(2244,832,weight=1)
G8.add_edge(2244,2245,weight=1)
G8.add_edge(2244,2246,weight=1)
G8.add_edge(2495,2064,weight=1)
G8.add_edge(1489,207,weight=1)
G8.add_edge(1489,1669,weight=1)
G8.add_edge(1491,207,weight=1)
G8.add_edge(1491,1489,weight=1)
G8.add_edge(1768,2258,weight=1)
G8.add_edge(1768,501,weight=1)
G8.add_edge(23,2258,weight=1)
G8.add_edge(23,1791,weight=1)
G8.add_edge(1769,2258,weight=1)
G8.add_edge(804,358,weight=1)
G8.add_edge(804,54,weight=1)
G8.add_edge(804,1564,weight=1)
G8.add_edge(804,1440,weight=1)
G8.add_edge(19,891,weight=1)
G8.add_edge(19,867,weight=1)
G8.add_edge(19,1933,weight=1)
G8.add_edge(19,128,weight=1)
G8.add_edge(209,2344,weight=1)
G8.add_edge(209,1271,weight=1)
G8.add_edge(2137,2149,weight=1)
G8.add_edge(2149,2137,weight=1)
G8.add_edge(1268,1336,weight=1)
G8.add_edge(1268,1501,weight=1)
G8.add_edge(1276,2618,weight=1)
G8.add_edge(1276,2291,weight=1)
G8.add_edge(1276,2292,weight=1)
G8.add_edge(2291,2214,weight=1)
G8.add_edge(2292,2291,weight=1)
G8.add_edge(108,899,weight=1)
G8.add_edge(108,251,weight=1)
G8.add_edge(108,499,weight=1)
G8.add_edge(108,109,weight=1)
G8.add_edge(108,1938,weight=1)
G8.add_edge(109,899,weight=1)
G8.add_edge(109,499,weight=1)
G8.add_edge(109,305,weight=1)
G8.add_edge(931,1255,weight=1)
G8.add_edge(931,1063,weight=1)
G8.add_edge(931,806,weight=1)
G8.add_edge(1925,2328,weight=1)
G8.add_edge(1542,1396,weight=1)
G8.add_edge(2531,2416,weight=1)
G8.add_edge(2531,2091,weight=1)
G8.add_edge(2666,2396,weight=1)
G8.add_edge(910,828,weight=1)
G8.add_edge(1895,2145,weight=1)
G8.add_edge(1895,2525,weight=1)
G8.add_edge(1895,828,weight=1)
G8.add_edge(1895,1528,weight=1)
G8.add_edge(1895,910,weight=1)
G8.add_edge(914,828,weight=1)
G8.add_edge(914,910,weight=1)
G8.add_edge(2041,2005,weight=1)
G8.add_edge(2041,2361,weight=1)
G8.add_edge(2041,673,weight=1)
G8.add_edge(2041,1925,weight=1)
G8.add_edge(160,2173,weight=1)
G8.add_edge(160,2067,weight=1)
G8.add_edge(160,2544,weight=1)
G8.add_edge(160,2675,weight=1)
G8.add_edge(986,2039,weight=1)
G8.add_edge(986,2067,weight=1)
G8.add_edge(986,160,weight=1)
G8.add_edge(986,1103,weight=1)
G8.add_edge(1103,1371,weight=1)
G8.add_edge(1103,1480,weight=1)
G8.add_edge(1103,1414,weight=1)
G8.add_edge(2085,2616,weight=1)
G8.add_edge(365,591,weight=1)
G8.add_edge(365,367,weight=1)
G8.add_edge(366,591,weight=1)
G8.add_edge(366,2573,weight=1)
G8.add_edge(366,365,weight=1)
G8.add_edge(367,591,weight=1)
G8.add_edge(955,1462,weight=1)
G8.add_edge(955,1592,weight=1)
G8.add_edge(955,451,weight=1)
G8.add_edge(1238,1522,weight=1)
G8.add_edge(1328,1166,weight=1)
G8.add_edge(432,2138,weight=1)
G8.add_edge(432,157,weight=1)
G8.add_edge(888,1532,weight=1)
G8.add_edge(888,984,weight=1)
G8.add_edge(1179,996,weight=1)
G8.add_edge(1179,855,weight=1)
G8.add_edge(1179,957,weight=1)
G8.add_edge(1179,1477,weight=1)
G8.add_edge(1179,1737,weight=1)
G8.add_edge(1179,1738,weight=1)
G8.add_edge(1179,1689,weight=1)
G8.add_edge(1179,1691,weight=1)
G8.add_edge(1179,1231,weight=1)
G8.add_edge(1179,1232,weight=1)
G8.add_edge(1179,1233,weight=1)
G8.add_edge(1179,1363,weight=1)
G8.add_edge(1179,1035,weight=1)
G8.add_edge(1179,1100,weight=1)
G8.add_edge(2176,1746,weight=1)
G8.add_edge(2176,1010,weight=1)
G8.add_edge(2176,1720,weight=1)
G8.add_edge(2176,1941,weight=1)
G8.add_edge(2176,2250,weight=1)
G8.add_edge(2176,2251,weight=1)
G8.add_edge(2176,1363,weight=1)
G8.add_edge(2176,2088,weight=1)
G8.add_edge(2176,2207,weight=1)
G8.add_edge(2176,121,weight=1)
G8.add_edge(2176,2260,weight=1)
G8.add_edge(1185,142,weight=1)
G8.add_edge(1185,782,weight=1)
G8.add_edge(1185,1363,weight=1)
G8.add_edge(1185,1785,weight=1)
G8.add_edge(1185,2200,weight=1)
G8.add_edge(1185,845,weight=1)
G8.add_edge(1185,121,weight=1)
G8.add_edge(1185,861,weight=1)
G8.add_edge(753,1622,weight=1)
G8.add_edge(2114,2624,weight=1)
G8.add_edge(1964,2196,weight=1)
G8.add_edge(1378,314,weight=1)
G8.add_edge(1378,1643,weight=1)
G8.add_edge(1378,289,weight=1)
G8.add_edge(1476,1643,weight=1)
G8.add_edge(692,1558,weight=1)
G8.add_edge(965,1674,weight=1)
G8.add_edge(965,1433,weight=1)
G8.add_edge(965,1141,weight=1)
G8.add_edge(965,966,weight=1)
G8.add_edge(965,967,weight=1)
G8.add_edge(965,1482,weight=1)
G8.add_edge(965,903,weight=1)
G8.add_edge(966,1674,weight=1)
G8.add_edge(966,1433,weight=1)
G8.add_edge(966,1591,weight=1)
G8.add_edge(966,1141,weight=1)
G8.add_edge(966,1254,weight=1)
G8.add_edge(966,967,weight=1)
G8.add_edge(966,982,weight=1)
G8.add_edge(966,178,weight=1)
G8.add_edge(966,1482,weight=1)
G8.add_edge(966,1061,weight=1)
G8.add_edge(967,1674,weight=1)
G8.add_edge(967,1132,weight=1)
G8.add_edge(967,1141,weight=1)
G8.add_edge(967,1482,weight=1)
G8.add_edge(984,1532,weight=1)
G8.add_edge(984,1303,weight=1)
G8.add_edge(984,929,weight=1)
G8.add_edge(984,1003,weight=1)
G8.add_edge(984,1736,weight=1)
G8.add_edge(2688,637,weight=1)
G8.add_edge(2688,2175,weight=1)
G8.add_edge(2688,765,weight=1)
G8.add_edge(765,770,weight=1)
G8.add_edge(765,1582,weight=1)
G8.add_edge(765,637,weight=1)
G8.add_edge(765,2175,weight=1)
G8.add_edge(765,1312,weight=1)
G8.add_edge(2335,1589,weight=1)
G8.add_edge(2335,1918,weight=1)
G8.add_edge(2335,2333,weight=1)
G8.add_edge(2335,2104,weight=1)
G8.add_edge(2335,55,weight=1)
G8.add_edge(2335,1833,weight=1)
G8.add_edge(2335,1837,weight=1)
G8.add_edge(2335,210,weight=1)
G8.add_edge(2335,1791,weight=1)
G8.add_edge(2335,2354,weight=1)
G8.add_edge(2335,603,weight=1)
G8.add_edge(1162,2325,weight=1)
G8.add_edge(2150,1956,weight=1)
G8.add_edge(2150,2189,weight=1)
G8.add_edge(2392,2549,weight=1)
G8.add_edge(2392,508,weight=1)
G8.add_edge(2395,2428,weight=1)
G8.add_edge(508,1889,weight=1)
G8.add_edge(508,2549,weight=1)
G8.add_edge(508,2392,weight=1)
G8.add_edge(87,653,weight=1)
G8.add_edge(87,250,weight=1)
G8.add_edge(87,374,weight=1)
G8.add_edge(2129,2480,weight=1)
G8.add_edge(2129,2031,weight=1)
G8.add_edge(2131,2031,weight=1)
G8.add_edge(2132,2320,weight=1)
G8.add_edge(2132,2031,weight=1)
G8.add_edge(368,441,weight=1)
G8.add_edge(1266,922,weight=1)
G8.add_edge(1266,1726,weight=1)
G8.add_edge(1524,814,weight=1)
G8.add_edge(1524,816,weight=1)
G8.add_edge(1524,1137,weight=1)
G8.add_edge(1524,344,weight=1)
G8.add_edge(1524,881,weight=1)
G8.add_edge(1524,1061,weight=1)
G8.add_edge(1001,932,weight=1)
G8.add_edge(26,1182,weight=1)
G8.add_edge(26,1898,weight=1)
G8.add_edge(26,779,weight=1)
G8.add_edge(26,21,weight=1)
G8.add_edge(26,457,weight=1)
G8.add_edge(121,1960,weight=1)
G8.add_edge(1940,2308,weight=1)
G8.add_edge(143,824,weight=1)
G8.add_edge(143,1319,weight=1)
G8.add_edge(143,1548,weight=1)
G8.add_edge(143,521,weight=1)
G8.add_edge(143,1284,weight=1)
G8.add_edge(143,231,weight=1)
G8.add_edge(143,228,weight=1)
G8.add_edge(143,392,weight=1)
G8.add_edge(2398,2204,weight=1)
G8.add_edge(2398,2406,weight=1)
G8.add_edge(2406,2204,weight=1)
G8.add_edge(1412,2204,weight=1)
G8.add_edge(246,173,weight=1)
G8.add_edge(713,682,weight=1)
G8.add_edge(713,2665,weight=1)
G8.add_edge(713,1966,weight=1)
G8.add_edge(713,55,weight=1)
G8.add_edge(713,1037,weight=1)
G8.add_edge(1663,1037,weight=1)
G8.add_edge(2048,2112,weight=1)
G8.add_edge(1178,1181,weight=1)
G8.add_edge(1178,1628,weight=1)
G8.add_edge(1178,1621,weight=1)
G8.add_edge(2328,2016,weight=1)
G8.add_edge(2328,1925,weight=1)
G8.add_edge(2328,2035,weight=1)
G8.add_edge(2430,2282,weight=1)
G8.add_edge(2430,2625,weight=1)
G8.add_edge(2430,2318,weight=1)
G8.add_edge(2430,2153,weight=1)
G8.add_edge(2430,1144,weight=1)
G8.add_edge(2438,2434,weight=1)
G8.add_edge(2448,2011,weight=1)
G8.add_edge(1549,1292,weight=1)
G8.add_edge(1549,1375,weight=1)
G8.add_edge(850,1721,weight=1)
G8.add_edge(861,1941,weight=1)
G8.add_edge(879,1024,weight=1)
G8.add_edge(879,1473,weight=1)
G8.add_edge(982,1141,weight=1)
G8.add_edge(178,886,weight=1)
G8.add_edge(178,1490,weight=1)
G8.add_edge(178,1960,weight=1)
G8.add_edge(2096,2231,weight=1)
G8.add_edge(2467,2241,weight=1)
G8.add_edge(2467,1831,weight=1)
G8.add_edge(2467,832,weight=1)
G8.add_edge(2474,435,weight=1)
G8.add_edge(2474,1193,weight=1)
G8.add_edge(1602,1156,weight=1)
G8.add_edge(1602,225,weight=1)
G8.add_edge(1602,1429,weight=1)
G8.add_edge(1602,1430,weight=1)
G8.add_edge(676,2340,weight=1)
G8.add_edge(676,1443,weight=1)
G8.add_edge(676,1446,weight=1)
G8.add_edge(676,598,weight=1)
G8.add_edge(676,1430,weight=1)
G8.add_edge(1122,807,weight=1)
G8.add_edge(1122,1139,weight=1)
G8.add_edge(1122,644,weight=1)
G8.add_edge(1122,1047,weight=1)
G8.add_edge(1122,1641,weight=1)
G8.add_edge(335,268,weight=1)
G8.add_edge(335,21,weight=1)
G8.add_edge(456,375,weight=1)
G8.add_edge(552,483,weight=1)
G8.add_edge(552,2216,weight=1)
G8.add_edge(552,2474,weight=1)
G8.add_edge(340,1550,weight=1)
G8.add_edge(340,271,weight=1)
G8.add_edge(340,1603,weight=1)
G8.add_edge(340,496,weight=1)
G8.add_edge(343,1946,weight=1)
G8.add_edge(343,2547,weight=1)
G8.add_edge(343,271,weight=1)
G8.add_edge(343,277,weight=1)
G8.add_edge(343,1994,weight=1)
G8.add_edge(343,468,weight=1)
G8.add_edge(343,340,weight=1)
G8.add_edge(343,777,weight=1)
G8.add_edge(343,496,weight=1)
G8.add_edge(343,730,weight=1)
G8.add_edge(1453,1376,weight=1)
G8.add_edge(1453,1290,weight=1)
G8.add_edge(1453,561,weight=1)
G8.add_edge(1026,1389,weight=1)
G8.add_edge(1026,1713,weight=1)
G8.add_edge(1026,1450,weight=1)
G8.add_edge(1026,877,weight=1)
G8.add_edge(1028,1389,weight=1)
G8.add_edge(1028,57,weight=1)
G8.add_edge(1258,1873,weight=1)
G8.add_edge(1258,1720,weight=1)
G8.add_edge(1258,2176,weight=1)
G8.add_edge(1258,2088,weight=1)
G8.add_edge(1258,2260,weight=1)
G8.add_edge(2260,2272,weight=1)
G8.add_edge(2260,1960,weight=1)
G8.add_edge(2262,1720,weight=1)
G8.add_edge(2262,2088,weight=1)
G8.add_edge(2262,2207,weight=1)
G8.add_edge(2262,2260,weight=1)
G8.add_edge(1271,237,weight=1)
G8.add_edge(1272,998,weight=1)
G8.add_edge(1272,237,weight=1)
G8.add_edge(2512,2084,weight=1)
G8.add_edge(2512,2321,weight=1)
G8.add_edge(2512,2142,weight=1)
G8.add_edge(2512,2419,weight=1)
G8.add_edge(2512,2346,weight=1)
G8.add_edge(1529,1488,weight=1)
G8.add_edge(1529,152,weight=1)
G8.add_edge(1533,1481,weight=1)
G8.add_edge(1533,1488,weight=1)
G8.add_edge(1533,926,weight=1)
G8.add_edge(690,1827,weight=1)
G8.add_edge(690,1581,weight=1)
G8.add_edge(1626,1581,weight=1)
G8.add_edge(34,1702,weight=1)
G8.add_edge(833,1702,weight=1)
G8.add_edge(37,1067,weight=1)
G8.add_edge(37,2073,weight=1)
G8.add_edge(37,1975,weight=1)
G8.add_edge(37,2391,weight=1)
G8.add_edge(37,1702,weight=1)
G8.add_edge(1251,400,weight=1)
G8.add_edge(1251,1252,weight=1)
G8.add_edge(1252,400,weight=1)
G8.add_edge(2261,2502,weight=1)
G8.add_edge(575,1963,weight=1)
G8.add_edge(575,243,weight=1)
G8.add_edge(575,2361,weight=1)
G8.add_edge(575,2476,weight=1)
G8.add_edge(575,1828,weight=1)
G8.add_edge(575,1007,weight=1)
G8.add_edge(575,2041,weight=1)
G8.add_edge(1655,853,weight=1)
G8.add_edge(1655,856,weight=1)
G8.add_edge(1655,1339,weight=1)
G8.add_edge(802,314,weight=1)
G8.add_edge(802,1476,weight=1)
G8.add_edge(2472,2246,weight=1)
G8.add_edge(677,606,weight=1)
G8.add_edge(677,1092,weight=1)
G8.add_edge(677,1057,weight=1)
G8.add_edge(677,1015,weight=1)
G8.add_edge(122,1991,weight=1)
G8.add_edge(122,1826,weight=1)
G8.add_edge(122,732,weight=1)
G8.add_edge(1061,1891,weight=1)
G8.add_edge(1061,2040,weight=1)
G8.add_edge(2040,1891,weight=1)
G8.add_edge(311,2514,weight=1)
G8.add_edge(1308,1658,weight=1)
G8.add_edge(1308,1058,weight=1)
G8.add_edge(1554,902,weight=1)
G8.add_edge(1554,1553,weight=1)
G8.add_edge(1555,902,weight=1)
G8.add_edge(1555,1402,weight=1)
G8.add_edge(1555,952,weight=1)
G8.add_edge(1555,1553,weight=1)
G8.add_edge(1555,1554,weight=1)
G8.add_edge(2648,2659,weight=1)
G8.add_edge(2648,2431,weight=1)
G8.add_edge(1250,1735,weight=1)
G8.add_edge(1250,1055,weight=1)
G8.add_edge(1250,1409,weight=1)
G8.add_edge(1811,1767,weight=1)
G8.add_edge(2349,1784,weight=1)
G8.add_edge(2349,2683,weight=1)
G8.add_edge(663,182,weight=1)
G8.add_edge(663,519,weight=1)
G8.add_edge(1807,2280,weight=1)
G8.add_edge(947,1180,weight=1)
G8.add_edge(2329,1783,weight=1)
G8.add_edge(1850,2164,weight=1)
G8.add_edge(1008,2218,weight=1)
G8.add_edge(1008,2427,weight=1)
G8.add_edge(1008,1920,weight=1)
G8.add_edge(1008,838,weight=1)
G8.add_edge(1309,1505,weight=1)
G8.add_edge(1309,1508,weight=1)
G8.add_edge(424,2426,weight=1)
G8.add_edge(424,1935,weight=1)
G8.add_edge(424,558,weight=1)
G8.add_edge(424,63,weight=1)
G8.add_edge(1327,2145,weight=1)
G8.add_edge(1327,2162,weight=1)
G8.add_edge(1327,2435,weight=1)
G8.add_edge(1327,1603,weight=1)
G8.add_edge(1327,972,weight=1)
G8.add_edge(1327,420,weight=1)
G8.add_edge(1327,1748,weight=1)
G8.add_edge(1327,249,weight=1)
G8.add_edge(1327,777,weight=1)
G8.add_edge(1327,106,weight=1)
G8.add_edge(2557,1905,weight=1)
G8.add_edge(2557,859,weight=1)
G8.add_edge(644,1905,weight=1)
G8.add_edge(644,2490,weight=1)
G8.add_edge(644,223,weight=1)
G8.add_edge(644,50,weight=1)
G8.add_edge(644,51,weight=1)
G8.add_edge(2113,2573,weight=1)
G8.add_edge(2113,366,weight=1)
G8.add_edge(2396,2666,weight=1)
G8.add_edge(1130,1191,weight=1)
G8.add_edge(1130,1223,weight=1)
G8.add_edge(1130,810,weight=1)
G8.add_edge(1130,815,weight=1)
G8.add_edge(1130,920,weight=1)
G8.add_edge(1130,1081,weight=1)
G8.add_edge(496,271,weight=1)
G8.add_edge(496,468,weight=1)
G8.add_edge(574,271,weight=1)
G8.add_edge(574,465,weight=1)
G8.add_edge(1506,2561,weight=1)
G8.add_edge(1506,933,weight=1)
G8.add_edge(1506,252,weight=1)
G8.add_edge(2597,2670,weight=1)
G8.add_edge(2597,2681,weight=1)
G8.add_edge(2597,2687,weight=1)
G8.add_edge(1765,1939,weight=1)
G8.add_edge(1766,1939,weight=1)
G8.add_edge(1771,1939,weight=1)
G8.add_edge(1771,1765,weight=1)
G8.add_edge(1771,1766,weight=1)
G8.add_edge(31,680,weight=1)
G8.add_edge(31,232,weight=1)
G8.add_edge(31,1149,weight=1)
G8.add_edge(31,286,weight=1)
G8.add_edge(31,2277,weight=1)
G8.add_edge(2092,1945,weight=1)
G8.add_edge(2092,2618,weight=1)
G8.add_edge(2092,586,weight=1)
G8.add_edge(2092,304,weight=1)
G8.add_edge(448,631,weight=1)
G8.add_edge(448,1502,weight=1)
G8.add_edge(448,1472,weight=1)
G8.add_edge(805,1586,weight=1)
G8.add_edge(805,1665,weight=1)
G8.add_edge(1780,2008,weight=1)
G8.add_edge(1781,2008,weight=1)
G8.add_edge(809,864,weight=1)
G8.add_edge(809,1451,weight=1)
G8.add_edge(809,1321,weight=1)
G8.add_edge(809,1343,weight=1)
G8.add_edge(315,1074,weight=1)
G8.add_edge(315,1597,weight=1)
G8.add_edge(315,1727,weight=1)
G8.add_edge(315,157,weight=1)
G8.add_edge(315,1245,weight=1)
G8.add_edge(315,1685,weight=1)
G8.add_edge(1183,1074,weight=1)
G8.add_edge(1183,1727,weight=1)
G8.add_edge(1183,1685,weight=1)
G8.add_edge(2177,1915,weight=1)
G8.add_edge(2177,2536,weight=1)
G8.add_edge(1854,2321,weight=1)
G8.add_edge(1854,1862,weight=1)
G8.add_edge(1269,1899,weight=1)
G8.add_edge(1269,2141,weight=1)
G8.add_edge(1269,2324,weight=1)
G8.add_edge(1269,1187,weight=1)
G8.add_edge(1269,977,weight=1)
G8.add_edge(1269,1415,weight=1)
G8.add_edge(1269,789,weight=1)
G8.add_edge(1269,1758,weight=1)
G8.add_edge(1269,1843,weight=1)
G8.add_edge(489,2054,weight=1)
G8.add_edge(720,1134,weight=1)
G8.add_edge(720,610,weight=1)
G8.add_edge(720,521,weight=1)
G8.add_edge(720,1113,weight=1)
G8.add_edge(720,1839,weight=1)
G8.add_edge(720,2562,weight=1)
G8.add_edge(720,88,weight=1)
G8.add_edge(720,1667,weight=1)
G8.add_edge(720,612,weight=1)
G8.add_edge(825,1216,weight=1)
G8.add_edge(825,1708,weight=1)
G8.add_edge(826,1216,weight=1)
G8.add_edge(826,1708,weight=1)
G8.add_edge(1774,2546,weight=1)
G8.add_edge(1776,1772,weight=1)
G8.add_edge(1776,2117,weight=1)
G8.add_edge(1776,2083,weight=1)
G8.add_edge(1794,2179,weight=1)
G8.add_edge(1794,1776,weight=1)
G8.add_edge(1794,1796,weight=1)
G8.add_edge(1794,2083,weight=1)
G8.add_edge(1795,2546,weight=1)
G8.add_edge(1795,2179,weight=1)
G8.add_edge(1795,1796,weight=1)
G8.add_edge(1796,2546,weight=1)
G8.add_edge(1796,2179,weight=1)
G8.add_edge(1796,1776,weight=1)
G8.add_edge(1796,1789,weight=1)
G8.add_edge(1796,2083,weight=1)
G8.add_edge(440,98,weight=1)
G8.add_edge(440,389,weight=1)
G8.add_edge(1551,1625,weight=1)
G8.add_edge(1551,1159,weight=1)
G8.add_edge(1551,1239,weight=1)
G8.add_edge(1551,1200,weight=1)
G8.add_edge(1551,1426,weight=1)
G8.add_edge(1551,1420,weight=1)
G8.add_edge(1551,1556,weight=1)
G8.add_edge(1551,685,weight=1)
G8.add_edge(1556,1159,weight=1)
G8.add_edge(1347,1273,weight=1)
G8.add_edge(1347,1282,weight=1)
G8.add_edge(1347,1452,weight=1)
G8.add_edge(1452,1605,weight=1)
G8.add_edge(1452,1273,weight=1)
G8.add_edge(664,497,weight=1)
G8.add_edge(664,2394,weight=1)
G8.add_edge(664,502,weight=1)
G8.add_edge(651,473,weight=1)
G8.add_edge(651,1558,weight=1)
G8.add_edge(263,267,weight=1)
G8.add_edge(1581,1578,weight=1)
G8.add_edge(659,1578,weight=1)
G8.add_edge(2372,2490,weight=1)
G8.add_edge(2372,1127,weight=1)
G8.add_edge(2373,2399,weight=1)
G8.add_edge(2373,2490,weight=1)
G8.add_edge(2373,1127,weight=1)
G8.add_edge(794,1111,weight=1)
G8.add_edge(794,960,weight=1)
G8.add_edge(794,1335,weight=1)
G8.add_edge(794,1695,weight=1)
G8.add_edge(794,1331,weight=1)
G8.add_edge(794,1735,weight=1)
G8.add_edge(794,944,weight=1)
G8.add_edge(794,1071,weight=1)
G8.add_edge(794,1206,weight=1)
G8.add_edge(794,1128,weight=1)
G8.add_edge(794,857,weight=1)
G8.add_edge(794,1381,weight=1)
G8.add_edge(794,43,weight=1)
G8.add_edge(794,1140,weight=1)
G8.add_edge(794,164,weight=1)
G8.add_edge(794,1070,weight=1)
G8.add_edge(794,1692,weight=1)
G8.add_edge(794,1456,weight=1)
G8.add_edge(794,1716,weight=1)
G8.add_edge(794,1250,weight=1)
G8.add_edge(794,263,weight=1)
G8.add_edge(794,1334,weight=1)
G8.add_edge(794,1468,weight=1)
G8.add_edge(1484,858,weight=1)
G8.add_edge(1484,2053,weight=1)
G8.add_edge(1484,2061,weight=1)
G8.add_edge(1490,838,weight=1)
G8.add_edge(903,911,weight=1)
G8.add_edge(903,965,weight=1)
G8.add_edge(903,966,weight=1)
G8.add_edge(903,967,weight=1)
G8.add_edge(903,982,weight=1)
G8.add_edge(1897,1852,weight=1)
G8.add_edge(206,601,weight=1)
G8.add_edge(1053,601,weight=1)
G8.add_edge(1053,1067,weight=1)
G8.add_edge(1053,1052,weight=1)
G8.add_edge(1053,833,weight=1)
G8.add_edge(430,1515,weight=1)
G8.add_edge(430,30,weight=1)
G8.add_edge(1334,1515,weight=1)
G8.add_edge(1334,1370,weight=1)
G8.add_edge(1334,1226,weight=1)
G8.add_edge(1334,1208,weight=1)
G8.add_edge(1334,1225,weight=1)
G8.add_edge(1334,1709,weight=1)
G8.add_edge(1334,1468,weight=1)
G8.add_edge(208,689,weight=1)
G8.add_edge(208,1677,weight=1)
G8.add_edge(208,1537,weight=1)
G8.add_edge(1147,1677,weight=1)
G8.add_edge(1147,821,weight=1)
G8.add_edge(1154,1612,weight=1)
G8.add_edge(1154,1676,weight=1)
G8.add_edge(1154,943,weight=1)
G8.add_edge(1154,1071,weight=1)
G8.add_edge(1154,1206,weight=1)
G8.add_edge(1154,1128,weight=1)
G8.add_edge(1154,857,weight=1)
G8.add_edge(1154,1064,weight=1)
G8.add_edge(1154,1220,weight=1)
G8.add_edge(1154,1411,weight=1)
G8.add_edge(1154,1456,weight=1)
G8.add_edge(1154,1078,weight=1)
G8.add_edge(1154,1066,weight=1)
G8.add_edge(1154,1114,weight=1)
G8.add_edge(1154,1258,weight=1)
G8.add_edge(1154,1205,weight=1)
G8.add_edge(1267,1577,weight=1)
G8.add_edge(1267,837,weight=1)
G8.add_edge(376,1577,weight=1)
G8.add_edge(376,837,weight=1)
G8.add_edge(1540,925,weight=1)
G8.add_edge(1540,2092,weight=1)
G8.add_edge(1540,379,weight=1)
G8.add_edge(1540,137,weight=1)
G8.add_edge(1540,450,weight=1)
G8.add_edge(482,10,weight=1)
G8.add_edge(1516,937,weight=1)
G8.add_edge(1516,1448,weight=1)
G8.add_edge(1516,1712,weight=1)
G8.add_edge(1516,1582,weight=1)
G8.add_edge(2322,2371,weight=1)
G8.add_edge(2322,2187,weight=1)
G8.add_edge(1922,2403,weight=1)
G8.add_edge(2031,2078,weight=1)
G8.add_edge(2031,2660,weight=1)
G8.add_edge(2031,744,weight=1)
G8.add_edge(2031,745,weight=1)
G8.add_edge(2031,2317,weight=1)
G8.add_edge(1081,815,weight=1)
G8.add_edge(730,1946,weight=1)
G8.add_edge(730,68,weight=1)
G8.add_edge(730,1994,weight=1)
G8.add_edge(730,468,weight=1)
G8.add_edge(2627,2400,weight=1)
G8.add_edge(2627,2657,weight=1)
G8.add_edge(2636,2232,weight=1)
G8.add_edge(1801,1908,weight=1)
G8.add_edge(1801,1902,weight=1)
G8.add_edge(1801,2017,weight=1)
G8.add_edge(1801,1996,weight=1)
G8.add_edge(1801,343,weight=1)
G8.add_edge(968,57,weight=1)
G8.add_edge(968,1395,weight=1)
G8.add_edge(968,1401,weight=1)
G8.add_edge(968,958,weight=1)
G8.add_edge(968,1562,weight=1)
G8.add_edge(968,58,weight=1)
G8.add_edge(968,848,weight=1)
G8.add_edge(968,849,weight=1)
G8.add_edge(968,690,weight=1)
G8.add_edge(968,323,weight=1)
G8.add_edge(968,307,weight=1)
G8.add_edge(968,1033,weight=1)
G8.add_edge(2109,1899,weight=1)
G8.add_edge(2109,2141,weight=1)
G8.add_edge(1366,1045,weight=1)
G8.add_edge(1366,813,weight=1)
G8.add_edge(697,216,weight=1)
G8.add_edge(697,1673,weight=1)
G8.add_edge(697,580,weight=1)
G8.add_edge(970,955,weight=1)
G8.add_edge(685,1426,weight=1)
G8.add_edge(685,526,weight=1)
G8.add_edge(1047,1345,weight=1)
G8.add_edge(1048,1356,weight=1)
G8.add_edge(1048,1421,weight=1)
G8.add_edge(1048,604,weight=1)
G8.add_edge(1048,1345,weight=1)
G8.add_edge(307,1377,weight=1)
G8.add_edge(1341,1076,weight=1)
G8.add_edge(2018,2678,weight=1)
G8.add_edge(2018,1480,weight=1)
G8.add_edge(2018,1583,weight=1)
G8.add_edge(1027,1434,weight=1)
G8.add_edge(1027,1106,weight=1)
G8.add_edge(270,25,weight=1)
G8.add_edge(490,2671,weight=1)
G8.add_edge(490,239,weight=1)
G8.add_edge(490,302,weight=1)
G8.add_edge(490,1816,weight=1)
G8.add_edge(490,47,weight=1)
G8.add_edge(1659,885,weight=1)
G8.add_edge(1920,2432,weight=1)
G8.add_edge(463,188,weight=1)
G8.add_edge(463,2583,weight=1)
G8.add_edge(1621,1181,weight=1)
G8.add_edge(1912,1720,weight=1)
G8.add_edge(1912,291,weight=1)
G8.add_edge(1912,1099,weight=1)
G8.add_edge(1912,2088,weight=1)
G8.add_edge(1912,2207,weight=1)
G8.add_edge(1912,861,weight=1)
G8.add_edge(935,1354,weight=1)
G8.add_edge(935,941,weight=1)
G8.add_edge(935,1010,weight=1)
G8.add_edge(935,1632,weight=1)
G8.add_edge(935,845,weight=1)
G8.add_edge(935,861,weight=1)
G8.add_edge(1641,1356,weight=1)
G8.add_edge(1641,604,weight=1)
G8.add_edge(1649,1356,weight=1)
G8.add_edge(787,792,weight=1)
G8.add_edge(1193,322,weight=1)
G8.add_edge(1194,1184,weight=1)
G8.add_edge(2180,2184,weight=1)
G8.add_edge(2180,2151,weight=1)
G8.add_edge(2180,322,weight=1)
G8.add_edge(1203,1184,weight=1)
G8.add_edge(2304,2036,weight=1)
G8.add_edge(2304,859,weight=1)
G8.add_edge(2304,1952,weight=1)
G8.add_edge(519,2533,weight=1)
G8.add_edge(2418,2533,weight=1)
G8.add_edge(971,1397,weight=1)
G8.add_edge(975,845,weight=1)
G8.add_edge(975,971,weight=1)
G8.add_edge(1679,862,weight=1)
G8.add_edge(1679,1340,weight=1)
G8.add_edge(1679,1571,weight=1)
G8.add_edge(729,244,weight=1)
G8.add_edge(729,944,weight=1)
G8.add_edge(729,1128,weight=1)
G8.add_edge(729,1306,weight=1)
G8.add_edge(729,1467,weight=1)
G8.add_edge(1107,1306,weight=1)
G8.add_edge(1107,1066,weight=1)
G8.add_edge(1467,944,weight=1)
G8.add_edge(1467,1651,weight=1)
G8.add_edge(1468,1334,weight=1)
G8.add_edge(1068,1802,weight=1)
G8.add_edge(1068,835,weight=1)
G8.add_edge(1068,1815,weight=1)
G8.add_edge(1073,840,weight=1)
G8.add_edge(2045,1150,weight=1)
G8.add_edge(2045,2684,weight=1)
G8.add_edge(2045,81,weight=1)
G8.add_edge(33,680,weight=1)
G8.add_edge(244,358,weight=1)
G8.add_edge(244,54,weight=1)
G8.add_edge(1120,2406,weight=1)
G8.add_edge(2399,2400,weight=1)
G8.add_edge(2399,2185,weight=1)
G8.add_edge(615,616,weight=1)
G8.add_edge(616,557,weight=1)
G8.add_edge(616,99,weight=1)
G8.add_edge(616,55,weight=1)
G8.add_edge(2080,2610,weight=1)
G8.add_edge(2081,2610,weight=1)
G8.add_edge(2330,2081,weight=1)
G8.add_edge(2330,1919,weight=1)
G8.add_edge(2330,2590,weight=1)
G8.add_edge(2330,1429,weight=1)
G8.add_edge(2631,1827,weight=1)
G8.add_edge(2686,2054,weight=1)
G8.add_edge(2691,2054,weight=1)
G8.add_edge(1847,1928,weight=1)
G8.add_edge(1848,1847,weight=1)
G8.add_edge(1848,1928,weight=1)
G8.add_edge(631,409,weight=1)
G8.add_edge(631,448,weight=1)
G8.add_edge(660,696,weight=1)
G8.add_edge(660,398,weight=1)
G8.add_edge(660,12,weight=1)
G8.add_edge(660,625,weight=1)
G8.add_edge(660,626,weight=1)
G8.add_edge(1697,1770,weight=1)
G8.add_edge(1697,1335,weight=1)
G8.add_edge(1697,1944,weight=1)
G8.add_edge(1697,1873,weight=1)
G8.add_edge(1697,1972,weight=1)
G8.add_edge(1697,2331,weight=1)
G8.add_edge(1697,2251,weight=1)
G8.add_edge(1697,2176,weight=1)
G8.add_edge(1697,1772,weight=1)
G8.add_edge(2657,1700,weight=1)
G8.add_edge(2657,800,weight=1)
G8.add_edge(1700,800,weight=1)
G8.add_edge(1707,800,weight=1)
G8.add_edge(1707,1386,weight=1)
G8.add_edge(2661,1934,weight=1)
G8.add_edge(749,1934,weight=1)
G8.add_edge(749,139,weight=1)
G8.add_edge(1892,1150,weight=1)
G8.add_edge(1892,1876,weight=1)
G8.add_edge(1892,1775,weight=1)
G8.add_edge(913,1150,weight=1)
G8.add_edge(913,1151,weight=1)
G8.add_edge(930,2443,weight=1)
G8.add_edge(1899,279,weight=1)
G8.add_edge(1899,2141,weight=1)
G8.add_edge(768,256,weight=1)
G8.add_edge(1863,1864,weight=1)
G8.add_edge(1863,1965,weight=1)
G8.add_edge(2271,2044,weight=1)
G8.add_edge(2271,481,weight=1)
G8.add_edge(996,2140,weight=1)
G8.add_edge(996,957,weight=1)
G8.add_edge(996,1972,weight=1)
G8.add_edge(996,1986,weight=1)
G8.add_edge(996,1691,weight=1)
G8.add_edge(996,1231,weight=1)
G8.add_edge(996,1232,weight=1)
G8.add_edge(996,1233,weight=1)
G8.add_edge(996,1496,weight=1)
G8.add_edge(325,327,weight=1)
G8.add_edge(325,279,weight=1)
G8.add_edge(325,390,weight=1)
G8.add_edge(325,186,weight=1)
G8.add_edge(327,325,weight=1)
G8.add_edge(327,279,weight=1)
G8.add_edge(328,325,weight=1)
G8.add_edge(328,327,weight=1)
G8.add_edge(328,946,weight=1)
G8.add_edge(328,1900,weight=1)
G8.add_edge(328,1533,weight=1)
G8.add_edge(333,325,weight=1)
G8.add_edge(333,390,weight=1)
G8.add_edge(333,186,weight=1)
G8.add_edge(333,1065,weight=1)
G8.add_edge(13,156,weight=1)
G8.add_edge(1647,944,weight=1)
G8.add_edge(1647,146,weight=1)
G8.add_edge(818,819,weight=1)
G8.add_edge(818,820,weight=1)
G8.add_edge(819,818,weight=1)
G8.add_edge(819,820,weight=1)
G8.add_edge(1844,2627,weight=1)
G8.add_edge(994,974,weight=1)
G8.add_edge(268,21,weight=1)
G8.add_edge(535,314,weight=1)
G8.add_edge(863,2365,weight=1)
G8.add_edge(863,1514,weight=1)
G8.add_edge(863,1408,weight=1)
G8.add_edge(863,1156,weight=1)
G8.add_edge(863,786,weight=1)
G8.add_edge(1369,992,weight=1)
G8.add_edge(443,402,weight=1)
G8.add_edge(2552,2474,weight=1)
G8.add_edge(505,703,weight=1)
G8.add_edge(2685,2524,weight=1)
G8.add_edge(165,518,weight=1)
G8.add_edge(1009,1012,weight=1)
G8.add_edge(1009,1487,weight=1)
G8.add_edge(1011,1012,weight=1)
G8.add_edge(1011,878,weight=1)
G8.add_edge(1011,1652,weight=1)
G8.add_edge(1011,1183,weight=1)
G8.add_edge(2585,2633,weight=1)
G8.add_edge(1619,1221,weight=1)
G8.add_edge(1619,1113,weight=1)
G8.add_edge(1619,2344,weight=1)
G8.add_edge(2599,855,weight=1)
G8.add_edge(2599,957,weight=1)
G8.add_edge(2599,1632,weight=1)
G8.add_edge(2599,2331,weight=1)
G8.add_edge(2599,1231,weight=1)
G8.add_edge(2599,1233,weight=1)
G8.add_edge(2605,1632,weight=1)
G8.add_edge(2605,2331,weight=1)
G8.add_edge(1632,2331,weight=1)
G8.add_edge(1935,1817,weight=1)
G8.add_edge(1737,1691,weight=1)
G8.add_edge(1738,1737,weight=1)
G8.add_edge(2707,1753,weight=1)
G8.add_edge(2707,1139,weight=1)
G8.add_edge(2707,1047,weight=1)
G8.add_edge(2707,1641,weight=1)
G8.add_edge(2015,2019,weight=1)
G8.add_edge(2015,1829,weight=1)
G8.add_edge(2019,1829,weight=1)
G8.add_edge(2279,2285,weight=1)
G8.add_edge(2284,2285,weight=1)
G8.add_edge(2413,2256,weight=1)
G8.add_edge(2413,1759,weight=1)
G8.add_edge(1819,2074,weight=1)
G8.add_edge(1819,1814,weight=1)
G8.add_edge(1819,2030,weight=1)
G8.add_edge(1953,2397,weight=1)
G8.add_edge(1953,2030,weight=1)
G8.add_edge(2056,2353,weight=1)
G8.add_edge(239,302,weight=1)
G8.add_edge(239,47,weight=1)
G8.add_edge(528,2607,weight=1)
G8.add_edge(2460,554,weight=1)
G8.add_edge(2460,2607,weight=1)
G8.add_edge(1747,1949,weight=1)
G8.add_edge(1255,1465,weight=1)
G8.add_edge(1255,1063,weight=1)
G8.add_edge(1255,931,weight=1)
G8.add_edge(688,1158,weight=1)
G8.add_edge(688,2382,weight=1)
G8.add_edge(688,627,weight=1)
G8.add_edge(688,111,weight=1)
G8.add_edge(688,1565,weight=1)
G8.add_edge(688,2684,weight=1)
G8.add_edge(688,1701,weight=1)
G8.add_edge(688,1705,weight=1)
G8.add_edge(174,1379,weight=1)
G8.add_edge(174,576,weight=1)
G8.add_edge(174,2486,weight=1)
G8.add_edge(2215,2224,weight=1)
G8.add_edge(2215,421,weight=1)
G8.add_edge(361,152,weight=1)
G8.add_edge(1898,2147,weight=1)
G8.add_edge(1898,2183,weight=1)
G8.add_edge(2020,2148,weight=1)
G8.add_edge(2165,2095,weight=1)
G8.add_edge(2165,2522,weight=1)
G8.add_edge(2165,1750,weight=1)
G8.add_edge(2165,6,weight=1)
G8.add_edge(2165,516,weight=1)
G8.add_edge(726,94,weight=1)
G8.add_edge(726,49,weight=1)
G8.add_edge(1805,2290,weight=1)
G8.add_edge(1806,2290,weight=1)
G8.add_edge(1806,1805,weight=1)
G8.add_edge(1927,749,weight=1)
G8.add_edge(1934,749,weight=1)
G8.add_edge(1934,37,weight=1)
G8.add_edge(138,749,weight=1)
G8.add_edge(138,139,weight=1)
G8.add_edge(138,417,weight=1)
G8.add_edge(1942,749,weight=1)
G8.add_edge(1942,1927,weight=1)
G8.add_edge(139,749,weight=1)
G8.add_edge(139,138,weight=1)
G8.add_edge(139,893,weight=1)
G8.add_edge(139,417,weight=1)
G8.add_edge(139,116,weight=1)
G8.add_edge(1141,2295,weight=1)
G8.add_edge(1141,1312,weight=1)
G8.add_edge(1141,812,weight=1)
G8.add_edge(1314,1624,weight=1)
G8.add_edge(1314,1656,weight=1)
G8.add_edge(796,795,weight=1)
G8.add_edge(952,1402,weight=1)
G8.add_edge(2174,2205,weight=1)
G8.add_edge(923,1417,weight=1)
G8.add_edge(923,1234,weight=1)
G8.add_edge(923,924,weight=1)
G8.add_edge(924,1417,weight=1)
G8.add_edge(924,1234,weight=1)
G8.add_edge(1197,841,weight=1)
G8.add_edge(2541,1858,weight=1)
G8.add_edge(731,771,weight=1)
G8.add_edge(731,1890,weight=1)
G8.add_edge(2649,1890,weight=1)
G8.add_edge(979,1826,weight=1)
G8.add_edge(1967,1826,weight=1)
G8.add_edge(980,1826,weight=1)
G8.add_edge(192,449,weight=1)
G8.add_edge(192,23,weight=1)
G8.add_edge(192,396,weight=1)
G8.add_edge(2006,2346,weight=1)
G8.add_edge(1038,1471,weight=1)
G8.add_edge(1038,1043,weight=1)
G8.add_edge(1038,1611,weight=1)
G8.add_edge(201,1892,weight=1)
G8.add_edge(201,537,weight=1)
G8.add_edge(207,1892,weight=1)
G8.add_edge(207,537,weight=1)
G8.add_edge(207,1043,weight=1)
G8.add_edge(1043,1436,weight=1)
G8.add_edge(1044,1436,weight=1)
G8.add_edge(1044,1043,weight=1)
G8.add_edge(287,2196,weight=1)
G8.add_edge(287,179,weight=1)
G8.add_edge(287,1964,weight=1)
G8.add_edge(2135,2645,weight=1)
G8.add_edge(2690,2210,weight=1)
G8.add_edge(883,1013,weight=1)
G8.add_edge(883,1617,weight=1)
G8.add_edge(883,1317,weight=1)
G8.add_edge(101,1612,weight=1)
G8.add_edge(101,266,weight=1)
G8.add_edge(101,759,weight=1)
G8.add_edge(1981,2458,weight=1)
G8.add_edge(1981,759,weight=1)
G8.add_edge(1981,2203,weight=1)
G8.add_edge(1981,263,weight=1)
G8.add_edge(271,277,weight=1)
G8.add_edge(2116,2001,weight=1)
G8.add_edge(2116,277,weight=1)
G8.add_edge(276,465,weight=1)
G8.add_edge(277,2547,weight=1)
G8.add_edge(277,1994,weight=1)
G8.add_edge(280,465,weight=1)
G8.add_edge(1254,1132,weight=1)
G8.add_edge(1254,1312,weight=1)
G8.add_edge(1254,812,weight=1)
G8.add_edge(1657,2358,weight=1)
G8.add_edge(1657,1823,weight=1)
G8.add_edge(1657,528,weight=1)
G8.add_edge(1657,1711,weight=1)
G8.add_edge(1657,732,weight=1)
G8.add_edge(1082,522,weight=1)
G8.add_edge(1083,1082,weight=1)
G8.add_edge(1083,2313,weight=1)
G8.add_edge(1083,522,weight=1)
G8.add_edge(2072,1810,weight=1)
G8.add_edge(413,689,weight=1)
G8.add_edge(1443,1136,weight=1)
G8.add_edge(1445,1443,weight=1)
G8.add_edge(1445,1136,weight=1)
G8.add_edge(1446,1443,weight=1)
G8.add_edge(2017,468,weight=1)
G8.add_edge(2017,343,weight=1)
G8.add_edge(1293,961,weight=1)
G8.add_edge(1293,1086,weight=1)
G8.add_edge(1293,1050,weight=1)
G8.add_edge(1293,1072,weight=1)
G8.add_edge(1302,961,weight=1)
G8.add_edge(1302,2347,weight=1)
G8.add_edge(1302,1267,weight=1)
G8.add_edge(866,2561,weight=1)
G8.add_edge(866,933,weight=1)
G8.add_edge(866,796,weight=1)
G8.add_edge(866,252,weight=1)
G8.add_edge(866,1506,weight=1)
G8.add_edge(156,118,weight=1)
G8.add_edge(156,514,weight=1)
G8.add_edge(162,714,weight=1)
G8.add_edge(162,96,weight=1)
G8.add_edge(162,997,weight=1)
G8.add_edge(162,2312,weight=1)
G8.add_edge(162,1148,weight=1)
G8.add_edge(162,685,weight=1)
G8.add_edge(250,653,weight=1)
G8.add_edge(1580,1104,weight=1)
G8.add_edge(764,242,weight=1)
G8.add_edge(764,2356,weight=1)
G8.add_edge(141,324,weight=1)
G8.add_edge(1465,1471,weight=1)
G8.add_edge(1465,1022,weight=1)
G8.add_edge(1465,201,weight=1)
G8.add_edge(1465,1101,weight=1)
G8.add_edge(1472,1502,weight=1)
G8.add_edge(1720,2364,weight=1)
G8.add_edge(1720,1217,weight=1)
G8.add_edge(1729,1111,weight=1)
G8.add_edge(1729,130,weight=1)
G8.add_edge(1729,169,weight=1)
G8.add_edge(1729,1537,weight=1)
G8.add_edge(1729,1475,weight=1)
G8.add_edge(1729,1250,weight=1)
G8.add_edge(1729,1776,weight=1)
G8.add_edge(1730,1111,weight=1)
G8.add_edge(2021,1858,weight=1)
G8.add_edge(1408,784,weight=1)
G8.add_edge(1994,1946,weight=1)
G8.add_edge(1994,2547,weight=1)
G8.add_edge(1994,277,weight=1)
G8.add_edge(579,1878,weight=1)
G8.add_edge(579,483,weight=1)
G8.add_edge(579,14,weight=1)
G8.add_edge(579,1123,weight=1)
G8.add_edge(579,1334,weight=1)
G8.add_edge(705,377,weight=1)
G8.add_edge(1928,1847,weight=1)
G8.add_edge(302,47,weight=1)
G8.add_edge(1739,1265,weight=1)
G8.add_edge(912,178,weight=1)
G8.add_edge(202,144,weight=1)
G8.add_edge(202,428,weight=1)
G8.add_edge(2320,2480,weight=1)
G8.add_edge(1410,2601,weight=1)
G8.add_edge(1410,2704,weight=1)
G8.add_edge(1410,790,weight=1)
G8.add_edge(1410,959,weight=1)
G8.add_edge(1410,2221,weight=1)
G8.add_edge(1410,1798,weight=1)
G8.add_edge(1039,1197,weight=1)
G8.add_edge(1039,917,weight=1)
G8.add_edge(215,470,weight=1)
G8.add_edge(215,588,weight=1)
G8.add_edge(215,79,weight=1)
G8.add_edge(1156,1408,weight=1)
G8.add_edge(1156,784,weight=1)
G8.add_edge(1156,786,weight=1)
G8.add_edge(380,757,weight=1)
G8.add_edge(380,177,weight=1)
G8.add_edge(2306,2184,weight=1)
G8.add_edge(2306,2151,weight=1)
G8.add_edge(2306,322,weight=1)
G8.add_edge(1725,1539,weight=1)
G8.add_edge(1725,1509,weight=1)
G8.add_edge(1725,1493,weight=1)
G8.add_edge(1207,1545,weight=1)
G8.add_edge(1207,790,weight=1)
G8.add_edge(1442,1639,weight=1)
G8.add_edge(1442,1718,weight=1)
G8.add_edge(2529,1859,weight=1)
G8.add_edge(2222,1758,weight=1)
G8.add_edge(2590,2425,weight=1)
G8.add_edge(2014,2383,weight=1)
G8.add_edge(2696,2199,weight=1)
G8.add_edge(2697,99,weight=1)
G8.add_edge(2697,2696,weight=1)
G8.add_edge(2697,2037,weight=1)
G8.add_edge(1064,1629,weight=1)
G8.add_edge(1064,1066,weight=1)
G8.add_edge(2411,1812,weight=1)
G8.add_edge(2411,205,weight=1)
G8.add_edge(100,1853,weight=1)
G8.add_edge(100,1804,weight=1)
G8.add_edge(100,2579,weight=1)
G8.add_edge(100,1614,weight=1)
G8.add_edge(100,2593,weight=1)
G8.add_edge(100,1620,weight=1)
G8.add_edge(100,1723,weight=1)
G8.add_edge(100,2680,weight=1)
G8.add_edge(100,1256,weight=1)
G8.add_edge(1547,1377,weight=1)
G8.add_edge(2237,2543,weight=1)
G8.add_edge(2237,2436,weight=1)
G8.add_edge(2237,2263,weight=1)
G8.add_edge(2263,2543,weight=1)
G8.add_edge(2263,2436,weight=1)
G8.add_edge(2263,2237,weight=1)
G8.add_edge(1464,932,weight=1)
G8.add_edge(1464,1400,weight=1)
G8.add_edge(1464,1406,weight=1)
G8.add_edge(1464,1407,weight=1)
G8.add_edge(1464,1362,weight=1)
G8.add_edge(1464,1000,weight=1)
G8.add_edge(1782,2664,weight=1)
G8.add_edge(1782,1820,weight=1)
G8.add_edge(1567,1568,weight=1)
G8.add_edge(1568,1567,weight=1)
G8.add_edge(1568,1359,weight=1)
G8.add_edge(1984,2079,weight=1)
G8.add_edge(2228,2073,weight=1)
G8.add_edge(2228,1975,weight=1)
G8.add_edge(2228,2391,weight=1)
G8.add_edge(2228,1924,weight=1)
G8.add_edge(147,18,weight=1)
G8.add_edge(1359,1662,weight=1)
G8.add_edge(1461,925,weight=1)
G8.add_edge(977,906,weight=1)
G8.add_edge(661,636,weight=1)
G8.add_edge(78,338,weight=1)
G8.add_edge(341,530,weight=1)
G8.add_edge(341,342,weight=1)
G8.add_edge(341,710,weight=1)
G8.add_edge(342,530,weight=1)
G8.add_edge(342,710,weight=1)
G8.add_edge(390,186,weight=1)
G8.add_edge(390,1938,weight=1)
G8.add_edge(510,670,weight=1)
G8.add_edge(510,315,weight=1)
G8.add_edge(284,2094,weight=1)
G8.add_edge(2269,2446,weight=1)
G8.add_edge(2269,2276,weight=1)
G8.add_edge(2269,1754,weight=1)
G8.add_edge(2276,2446,weight=1)
G8.add_edge(2276,2523,weight=1)
G8.add_edge(2276,2269,weight=1)
G8.add_edge(2276,2192,weight=1)
G8.add_edge(1087,1546,weight=1)
G8.add_edge(1087,1583,weight=1)
G8.add_edge(1087,1642,weight=1)
G8.add_edge(1087,1077,weight=1)
G8.add_edge(1087,1079,weight=1)
G8.add_edge(124,554,weight=1)
G8.add_edge(266,759,weight=1)
G8.add_edge(422,256,weight=1)
G8.add_edge(884,1666,weight=1)
G8.add_edge(884,1211,weight=1)
G8.add_edge(2012,2634,weight=1)
G8.add_edge(2012,1836,weight=1)
G8.add_edge(2012,2183,weight=1)
G8.add_edge(1021,2578,weight=1)
G8.add_edge(989,1516,weight=1)
G8.add_edge(2224,191,weight=1)
G8.add_edge(2224,511,weight=1)
G8.add_edge(357,191,weight=1)
G8.add_edge(357,511,weight=1)
G8.add_edge(1215,1636,weight=1)
G8.add_edge(1828,1963,weight=1)
G8.add_edge(1828,2084,weight=1)
G8.add_edge(1828,2225,weight=1)
G8.add_edge(1828,2111,weight=1)
G8.add_edge(2464,1935,weight=1)
G8.add_edge(1307,608,weight=1)
G8.add_edge(1307,1500,weight=1)
G8.add_edge(1311,1307,weight=1)
G8.add_edge(1311,1500,weight=1)
G8.add_edge(538,358,weight=1)
G8.add_edge(538,607,weight=1)
G8.add_edge(538,1205,weight=1)
G8.add_edge(1458,1238,weight=1)
G8.add_edge(59,1892,weight=1)
G8.add_edge(59,537,weight=1)
G8.add_edge(59,991,weight=1)
G8.add_edge(59,2496,weight=1)
G8.add_edge(455,1369,weight=1)
G8.add_edge(455,184,weight=1)
G8.add_edge(455,542,weight=1)
G8.add_edge(2076,1856,weight=1)
G8.add_edge(2171,1326,weight=1)
G8.add_edge(1176,1326,weight=1)
G8.add_edge(1176,1629,weight=1)
G8.add_edge(1579,1623,weight=1)
G8.add_edge(1579,1011,weight=1)
G8.add_edge(1579,878,weight=1)
G8.add_edge(1579,1652,weight=1)
G8.add_edge(1579,1183,weight=1)
G8.add_edge(2255,2348,weight=1)
G8.add_edge(2255,2040,weight=1)
G8.add_edge(2266,2348,weight=1)
G8.add_edge(1880,2356,weight=1)
G8.add_edge(1880,1761,weight=1)
G8.add_edge(1880,764,weight=1)
G8.add_edge(694,1502,weight=1)
G8.add_edge(694,649,weight=1)
G8.add_edge(694,1472,weight=1)
G8.add_edge(694,532,weight=1)
G8.add_edge(1634,1472,weight=1)
G8.add_edge(308,599,weight=1)
G8.add_edge(2385,2461,weight=1)
G8.add_edge(521,525,weight=1)
G8.add_edge(525,1455,weight=1)
G8.add_edge(525,1170,weight=1)
G8.add_edge(38,564,weight=1)
G8.add_edge(39,373,weight=1)
G8.add_edge(39,564,weight=1)
G8.add_edge(39,274,weight=1)
G8.add_edge(735,214,weight=1)
G8.add_edge(772,105,weight=1)
G8.add_edge(772,73,weight=1)
G8.add_edge(1990,1883,weight=1)
G8.add_edge(2127,2340,weight=1)
G8.add_edge(2127,2214,weight=1)
G8.add_edge(2127,2592,weight=1)
G8.add_edge(573,113,weight=1)
G8.add_edge(700,589,weight=1)
G8.add_edge(1173,916,weight=1)
G8.add_edge(1173,624,weight=1)
G8.add_edge(1173,1142,weight=1)
G8.add_edge(1173,2342,weight=1)
G8.add_edge(1173,2109,weight=1)
G8.add_edge(2167,2525,weight=1)
G8.add_edge(2167,2168,weight=1)
G8.add_edge(2167,972,weight=1)
G8.add_edge(2167,777,weight=1)
G8.add_edge(2168,2162,weight=1)
G8.add_edge(2170,2421,weight=1)
G8.add_edge(2293,2141,weight=1)
G8.add_edge(2293,1900,weight=1)
G8.add_edge(2294,1938,weight=1)
G8.add_edge(399,420,weight=1)
G8.add_edge(399,249,weight=1)
G8.add_edge(399,777,weight=1)
G8.add_edge(2405,2222,weight=1)
G8.add_edge(516,638,weight=1)
G8.add_edge(2435,2161,weight=1)
G8.add_edge(2435,106,weight=1)
G8.add_edge(624,638,weight=1)
G8.add_edge(624,2141,weight=1)
G8.add_edge(624,1938,weight=1)
G8.add_edge(624,1065,weight=1)
G8.add_edge(1584,901,weight=1)
G8.add_edge(1584,2564,weight=1)
G8.add_edge(1584,2397,weight=1)
G8.add_edge(1690,1739,weight=1)
G8.add_edge(1690,39,weight=1)
G8.add_edge(1422,1418,weight=1)
G8.add_edge(1422,822,weight=1)
G8.add_edge(1608,1224,weight=1)
G8.add_edge(1616,1230,weight=1)
G8.add_edge(1616,1640,weight=1)
G8.add_edge(429,185,weight=1)
G8.add_edge(429,599,weight=1)
G8.add_edge(429,434,weight=1)
G8.add_edge(429,303,weight=1)
G8.add_edge(433,185,weight=1)
G8.add_edge(433,599,weight=1)
G8.add_edge(433,429,weight=1)
G8.add_edge(433,434,weight=1)
G8.add_edge(433,303,weight=1)
G8.add_edge(434,185,weight=1)
G8.add_edge(86,426,weight=1)
G8.add_edge(86,2369,weight=1)
G8.add_edge(1146,1324,weight=1)
G8.add_edge(1150,1324,weight=1)
G8.add_edge(1150,1151,weight=1)
G8.add_edge(1151,1324,weight=1)
G8.add_edge(640,1764,weight=1)
G8.add_edge(640,585,weight=1)
G8.add_edge(2108,2575,weight=1)
G8.add_edge(2108,2572,weight=1)
G8.add_edge(2108,1954,weight=1)
G8.add_edge(2108,1866,weight=1)
G8.add_edge(2108,2047,weight=1)
G8.add_edge(2108,614,weight=1)
G8.add_edge(2108,2497,weight=1)
G8.add_edge(269,233,weight=1)
G8.add_edge(269,2572,weight=1)
G8.add_edge(269,2047,weight=1)
G8.add_edge(269,617,weight=1)
G8.add_edge(282,461,weight=1)
G8.add_edge(878,1652,weight=1)
G8.add_edge(2119,2443,weight=1)
G8.add_edge(2625,2703,weight=1)
G8.add_edge(2625,1932,weight=1)
G8.add_edge(2625,1879,weight=1)
G8.add_edge(2625,2278,weight=1)
G8.add_edge(2625,1205,weight=1)
G8.add_edge(564,501,weight=1)
G8.add_edge(2022,2219,weight=1)
G8.add_edge(2022,88,weight=1)
G8.add_edge(2397,2564,weight=1)
G8.add_edge(718,2503,weight=1)
G8.add_edge(718,1878,weight=1)
G8.add_edge(718,2569,weight=1)
G8.add_edge(718,1123,weight=1)
G8.add_edge(718,524,weight=1)
G8.add_edge(151,51,weight=1)
G8.add_edge(151,644,weight=1)
G8.add_edge(1948,2558,weight=1)
G8.add_edge(153,1384,weight=1)
G8.add_edge(2559,2515,weight=1)
G8.add_edge(2559,451,weight=1)
G8.add_edge(647,451,weight=1)
G8.add_edge(1719,811,weight=1)
G8.add_edge(1719,2583,weight=1)
G8.add_edge(1719,2414,weight=1)
G8.add_edge(1719,2415,weight=1)
G8.add_edge(1719,202,weight=1)
G8.add_edge(1719,29,weight=1)
G8.add_edge(1719,1868,weight=1)
G8.add_edge(1719,2139,weight=1)
G8.add_edge(1719,1374,weight=1)
G8.add_edge(272,719,weight=1)
G8.add_edge(355,331,weight=1)
G8.add_edge(355,840,weight=1)
G8.add_edge(355,1073,weight=1)
G8.add_edge(1253,1635,weight=1)
G8.add_edge(1253,908,weight=1)
G8.add_edge(1253,919,weight=1)
G8.add_edge(1253,821,weight=1)
G8.add_edge(1253,830,weight=1)
G8.add_edge(1253,831,weight=1)
G8.add_edge(1253,1486,weight=1)
G8.add_edge(1253,1116,weight=1)
G8.add_edge(1891,2040,weight=1)
G8.add_edge(2133,1838,weight=1)
G8.add_edge(1643,314,weight=1)
G8.add_edge(1643,1476,weight=1)
G8.add_edge(2316,2288,weight=1)
G8.add_edge(189,142,weight=1)
G8.add_edge(189,722,weight=1)
G8.add_edge(189,756,weight=1)
G8.add_edge(189,718,weight=1)
G8.add_edge(189,43,weight=1)
G8.add_edge(189,164,weight=1)
G8.add_edge(189,416,weight=1)
G8.add_edge(189,758,weight=1)
G8.add_edge(189,381,weight=1)
G8.add_edge(189,140,weight=1)
G8.add_edge(189,121,weight=1)
G8.add_edge(189,263,weight=1)
G8.add_edge(189,524,weight=1)
G8.add_edge(190,2665,weight=1)
G8.add_edge(190,189,weight=1)
G8.add_edge(190,43,weight=1)
G8.add_edge(190,164,weight=1)
G8.add_edge(190,396,weight=1)
G8.add_edge(190,2103,weight=1)
G8.add_edge(1715,1459,weight=1)
G8.add_edge(839,1459,weight=1)
G8.add_edge(1342,1045,weight=1)
G8.add_edge(1342,813,weight=1)
G8.add_edge(1357,1431,weight=1)
G8.add_edge(2047,617,weight=1)
G8.add_edge(418,318,weight=1)
G8.add_edge(418,509,weight=1)
G8.add_edge(2327,318,weight=1)
G8.add_edge(2327,509,weight=1)
G8.add_edge(2327,2325,weight=1)
G8.add_edge(779,57,weight=1)
G8.add_edge(14,483,weight=1)
G8.add_edge(867,891,weight=1)
G8.add_edge(1530,1315,weight=1)
G8.add_edge(1530,883,weight=1)
G8.add_edge(858,2061,weight=1)
G8.add_edge(2621,1973,weight=1)
G8.add_edge(1662,1470,weight=1)
G8.add_edge(1662,1359,weight=1)
G8.add_edge(144,181,weight=1)
G8.add_edge(144,338,weight=1)
G8.add_edge(144,347,weight=1)
G8.add_edge(144,364,weight=1)
G8.add_edge(671,604,weight=1)
G8.add_edge(2206,1800,weight=1)
G8.add_edge(542,184,weight=1)
G8.add_edge(1821,2384,weight=1)
G8.add_edge(1821,2570,weight=1)
G8.add_edge(1534,901,weight=1)
G8.add_edge(1534,1731,weight=1)
G8.add_edge(1534,782,weight=1)
G8.add_edge(1534,1278,weight=1)
G8.add_edge(1534,971,weight=1)
G8.add_edge(1669,2536,weight=1)
G8.add_edge(1997,1998,weight=1)
G8.add_edge(1829,2019,weight=1)
G8.add_edge(1521,1259,weight=1)
G8.add_edge(1521,1527,weight=1)
G8.add_edge(1521,1201,weight=1)
G8.add_edge(1521,921,weight=1)
G8.add_edge(1521,1247,weight=1)
G8.add_edge(1521,355,weight=1)
G8.add_edge(1521,1525,weight=1)
G8.add_edge(1521,1710,weight=1)
G8.add_edge(1521,571,weight=1)
G8.add_edge(1521,835,weight=1)
G8.add_edge(1521,840,weight=1)
G8.add_edge(1521,873,weight=1)
G8.add_edge(1521,874,weight=1)
G8.add_edge(1521,60,weight=1)
G8.add_edge(1521,875,weight=1)
G8.add_edge(1521,1246,weight=1)
G8.add_edge(1521,1212,weight=1)
G8.add_edge(1521,1073,weight=1)
G8.add_edge(1525,1382,weight=1)
G8.add_edge(1525,1527,weight=1)
G8.add_edge(1525,1244,weight=1)
G8.add_edge(1525,1246,weight=1)
G8.add_edge(1991,234,weight=1)
G8.add_edge(1991,2094,weight=1)
G8.add_edge(1991,202,weight=1)
G8.add_edge(1131,960,weight=1)
G8.add_edge(1131,1335,weight=1)
G8.add_edge(196,708,weight=1)
G8.add_edge(197,217,weight=1)
G8.add_edge(197,196,weight=1)
G8.add_edge(197,708,weight=1)
G8.add_edge(1296,887,weight=1)
G8.add_edge(2679,2478,weight=1)
G8.add_edge(2679,2577,weight=1)
G8.add_edge(2190,2189,weight=1)
G8.add_edge(2190,2191,weight=1)
G8.add_edge(2190,2366,weight=1)
G8.add_edge(2191,1956,weight=1)
G8.add_edge(2191,2189,weight=1)
G8.add_edge(2191,2190,weight=1)
G8.add_edge(2191,2150,weight=1)
G8.add_edge(2191,2366,weight=1)
G8.add_edge(2195,1956,weight=1)
G8.add_edge(2195,2189,weight=1)
G8.add_edge(2195,2190,weight=1)
G8.add_edge(2195,2191,weight=1)
G8.add_edge(2195,2150,weight=1)
G8.add_edge(1711,1451,weight=1)
G8.add_edge(1711,577,weight=1)
G8.add_edge(44,577,weight=1)
G8.add_edge(2540,2214,weight=1)
G8.add_edge(2068,1793,weight=1)
G8.add_edge(182,519,weight=1)
G8.add_edge(1003,1736,weight=1)
G8.add_edge(669,1509,weight=1)
G8.add_edge(669,605,weight=1)
G8.add_edge(669,976,weight=1)
G8.add_edge(669,2414,weight=1)
G8.add_edge(669,2415,weight=1)
G8.add_edge(669,264,weight=1)
G8.add_edge(669,2684,weight=1)
G8.add_edge(669,1316,weight=1)
G8.add_edge(669,2563,weight=1)
G8.add_edge(669,311,weight=1)
G8.add_edge(2166,2364,weight=1)
G8.add_edge(2166,539,weight=1)
G8.add_edge(2166,2194,weight=1)
G8.add_edge(2166,1144,weight=1)
G8.add_edge(1728,1377,weight=1)
G8.add_edge(710,530,weight=1)
G8.add_edge(710,342,weight=1)
G8.add_edge(29,587,weight=1)
G8.add_edge(1941,2251,weight=1)
G8.add_edge(2433,1756,weight=1)
G8.add_edge(2331,1770,weight=1)
G8.add_edge(2331,1632,weight=1)
G8.add_edge(1726,922,weight=1)
G8.add_edge(1726,1266,weight=1)
G8.add_edge(56,1906,weight=1)
G8.add_edge(56,362,weight=1)
G8.add_edge(56,1833,weight=1)
G8.add_edge(56,845,weight=1)
G8.add_edge(1835,1906,weight=1)
G8.add_edge(461,149,weight=1)
G8.add_edge(347,338,weight=1)
G8.add_edge(347,364,weight=1)
G8.add_edge(21,268,weight=1)
G8.add_edge(2623,2034,weight=1)
G8.add_edge(2318,1354,weight=1)
G8.add_edge(2318,2625,weight=1)
G8.add_edge(2637,2301,weight=1)
G8.add_edge(247,561,weight=1)
G8.add_edge(445,754,weight=1)
G8.add_edge(2674,996,weight=1)
G8.add_edge(2674,2542,weight=1)
G8.add_edge(1195,595,weight=1)
G8.add_edge(2663,1823,weight=1)
G8.add_edge(2663,2607,weight=1)
G8.add_edge(2663,528,weight=1)
G8.add_edge(2423,2377,weight=1)
G8.add_edge(2352,1773,weight=1)
G8.add_edge(1380,1612,weight=1)
G8.add_edge(1381,1612,weight=1)
G8.add_edge(1381,1718,weight=1)
G8.add_edge(2404,2582,weight=1)
G8.add_edge(91,681,weight=1)
G8.add_edge(1968,696,weight=1)
G8.add_edge(1155,820,weight=1)
G8.add_edge(1155,1532,weight=1)
G8.add_edge(2604,1757,weight=1)
G8.add_edge(2614,1757,weight=1)
G8.add_edge(1664,1757,weight=1)
G8.add_edge(1664,2505,weight=1)
G8.add_edge(1664,1267,weight=1)
G8.add_edge(942,1718,weight=1)
G8.add_edge(942,1442,weight=1)
G8.add_edge(1208,901,weight=1)
G8.add_edge(2450,1882,weight=1)
G8.add_edge(2450,2159,weight=1)
G8.add_edge(2450,2451,weight=1)
G8.add_edge(2451,1882,weight=1)
G8.add_edge(2452,2447,weight=1)
G8.add_edge(2452,1974,weight=1)
G8.add_edge(2550,2609,weight=1)
G8.add_edge(2550,281,weight=1)
G8.add_edge(2550,520,weight=1)
G8.add_edge(2652,2060,weight=1)
G8.add_edge(1969,528,weight=1)
G8.add_edge(1969,1775,weight=1)
G8.add_edge(1969,2620,weight=1)
G8.add_edge(962,1516,weight=1)
G8.add_edge(2439,2092,weight=1)
G8.add_edge(596,234,weight=1)
G8.add_edge(596,122,weight=1)
G8.add_edge(2057,272,weight=1)
G8.add_edge(2057,2096,weight=1)
G8.add_edge(2305,2298,weight=1)
G8.add_edge(2305,909,weight=1)
G8.add_edge(1224,1608,weight=1)
G8.add_edge(43,189,weight=1)
G8.add_edge(1876,2440,weight=1)
G8.add_edge(1059,2455,weight=1)
G8.add_edge(1059,1212,weight=1)
G8.add_edge(2130,2455,weight=1)
G8.add_edge(453,118,weight=1)
G8.add_edge(170,213,weight=1)
G8.add_edge(1816,2671,weight=1)
G8.add_edge(1816,2672,weight=1)
G8.add_edge(1816,239,weight=1)
G8.add_edge(1816,254,weight=1)
G8.add_edge(47,239,weight=1)
G8.add_edge(47,868,weight=1)
G8.add_edge(442,298,weight=1)
G8.add_edge(1060,1056,weight=1)
G8.add_edge(2238,2275,weight=1)
G8.add_edge(2238,2171,weight=1)
G8.add_edge(146,2522,weight=1)
G8.add_edge(146,2287,weight=1)
G8.add_edge(146,391,weight=1)
G8.add_edge(146,1225,weight=1)
G8.add_edge(146,1219,weight=1)
G8.add_edge(146,2249,weight=1)
G8.add_edge(146,2252,weight=1)
G8.add_edge(146,2281,weight=1)
G8.add_edge(146,2297,weight=1)
G8.add_edge(146,1531,weight=1)
G8.add_edge(2482,1859,weight=1)
G8.add_edge(1775,991,weight=1)
G8.add_edge(806,991,weight=1)
G8.add_edge(2660,744,weight=1)
G8.add_edge(2660,2317,weight=1)
G8.add_edge(744,2078,weight=1)
G8.add_edge(744,66,weight=1)
G8.add_edge(744,2317,weight=1)
G8.add_edge(745,2078,weight=1)
G8.add_edge(745,66,weight=1)
G8.add_edge(745,744,weight=1)
G8.add_edge(745,2317,weight=1)
G8.add_edge(1839,2515,weight=1)
G8.add_edge(1839,2381,weight=1)
G8.add_edge(981,1489,weight=1)
G8.add_edge(2102,2471,weight=1)
G8.add_edge(2473,2477,weight=1)
G8.add_edge(1415,946,weight=1)
G8.add_edge(2562,2381,weight=1)
G8.add_edge(757,177,weight=1)
G8.add_edge(1462,1358,weight=1)
G8.add_edge(1462,1594,weight=1)
G8.add_edge(1462,305,weight=1)
G8.add_edge(1462,1289,weight=1)
G8.add_edge(82,648,weight=1)
G8.add_edge(1140,781,weight=1)
G8.add_edge(1140,1573,weight=1)
G8.add_edge(1140,1019,weight=1)
G8.add_edge(1140,1599,weight=1)
G8.add_edge(608,1056,weight=1)
G8.add_edge(608,822,weight=1)
G8.add_edge(1947,1031,weight=1)
G8.add_edge(1947,2517,weight=1)
G8.add_edge(65,285,weight=1)
G8.add_edge(186,390,weight=1)
G8.add_edge(1273,1605,weight=1)
G8.add_edge(241,2309,weight=1)
G8.add_edge(241,62,weight=1)
G8.add_edge(1295,916,weight=1)
G8.add_edge(1435,1209,weight=1)
G8.add_edge(1320,1392,weight=1)
G8.add_edge(515,507,weight=1)
G8.add_edge(388,294,weight=1)
G8.add_edge(73,772,weight=1)
G8.add_edge(73,203,weight=1)
G8.add_edge(990,1419,weight=1)
G8.add_edge(990,1069,weight=1)
G8.add_edge(509,318,weight=1)
G8.add_edge(2620,329,weight=1)
G8.add_edge(2620,1775,weight=1)
G8.add_edge(1573,1599,weight=1)
G8.add_edge(1368,818,weight=1)
G8.add_edge(1494,775,weight=1)
G8.add_edge(1494,868,weight=1)
G8.add_edge(27,1130,weight=1)
G8.add_edge(211,645,weight=1)
G8.add_edge(211,149,weight=1)
G8.add_edge(2323,2499,weight=1)
G8.add_edge(1110,2152,weight=1)
G8.add_edge(254,2671,weight=1)
G8.add_edge(254,1816,weight=1)
G8.add_edge(166,433,weight=1)
G8.add_edge(1631,1151,weight=1)
G8.add_edge(28,533,weight=1)
G8.add_edge(1420,526,weight=1)
G8.add_edge(1420,1551,weight=1)
G8.add_edge(523,526,weight=1)
G8.add_edge(1428,526,weight=1)
G8.add_edge(1428,1551,weight=1)
G8.add_edge(2007,1825,weight=1)
G8.add_edge(2223,2151,weight=1)
G8.add_edge(273,517,weight=1)
G8.add_edge(273,915,weight=1)
G8.add_edge(693,297,weight=1)
G8.add_edge(920,909,weight=1)
G8.add_edge(940,1243,weight=1)
G8.add_edge(332,180,weight=1)
G8.add_edge(332,102,weight=1)
G8.add_edge(332,775,weight=1)
G8.add_edge(2491,1808,weight=1)
G8.add_edge(803,974,weight=1)
G8.add_edge(1049,985,weight=1)
G8.add_edge(1879,2572,weight=1)
G8.add_edge(1543,802,weight=1)
G8.add_edge(1961,1950,weight=1)
G8.add_edge(679,529,weight=1)
G8.add_edge(2483,2646,weight=1)
G8.add_edge(1170,1455,weight=1)
G8.add_edge(2205,2174,weight=1)
G8.add_edge(1965,1863,weight=1)
G8.add_edge(1965,1864,weight=1)
G8.add_edge(292,488,weight=1)
G8.add_edge(1923,2027,weight=1)
G8.add_edge(1929,1924,weight=1)
G8.add_edge(1929,2115,weight=1)
G8.add_edge(1177,2273,weight=1)
G8.add_edge(1177,255,weight=1)
G8.add_edge(1564,358,weight=1)
G8.add_edge(1564,54,weight=1)
G8.add_edge(2664,2230,weight=1)
G8.add_edge(334,464,weight=1)
G8.add_edge(2204,2398,weight=1)
G8.add_edge(258,481,weight=1)
G8.add_edge(258,37,weight=1)
G8.add_edge(1243,940,weight=1)
G8.add_edge(352,731,weight=1)
G8.add_edge(896,1437,weight=1)
G8.add_edge(1447,949,weight=1)
G8.add_edge(1225,2674,weight=1)
G8.add_edge(1849,2484,weight=1)
G8.add_edge(1006,1314,weight=1)
G8.add_edge(1169,803,weight=1)
G8.add_edge(1440,54,weight=1)
G8.add_edge(1440,804,weight=1)
G8.add_edge(532,1491,weight=1)
G8.add_edge(2668,2296,weight=1)
G8.add_edge(2668,2382,weight=1)
G8.add_edge(2668,2505,weight=1)
G8.add_edge(2668,2430,weight=1)
G8.add_edge(1383,1314,weight=1)
G8.add_edge(2632,2602,weight=1)
G8.add_edge(1916,2498,weight=1)
G8.add_edge(1916,1982,weight=1)
G8.add_edge(2591,389,weight=1)
G8.add_edge(1804,1914,weight=1)
G8.add_edge(2449,2312,weight=1)
G8.add_edge(2449,520,weight=1)
G8.add_edge(906,1398,weight=1)
G8.add_edge(892,1438,weight=1)
G8.add_edge(759,266,weight=1)
G8.add_edge(2313,2588,weight=1)
G8.add_edge(1161,1167,weight=1)
G8.add_edge(2239,2272,weight=1)
G8.add_edge(2250,2272,weight=1)
G8.add_edge(2251,2272,weight=1)
G8.add_edge(2253,2272,weight=1)
G8.add_edge(2253,2278,weight=1)
G8.add_edge(783,1533,weight=1)
G8.add_edge(2408,2705,weight=1)
G8.add_edge(2408,1632,weight=1)
G8.add_edge(2201,2704,weight=1)
G8.add_edge(2201,1798,weight=1)
G8.add_edge(1217,1036,weight=1)
G8.add_edge(1217,1572,weight=1)
G8.add_edge(1217,1300,weight=1)
G8.add_edge(1219,1036,weight=1)
G8.add_edge(1219,1572,weight=1)
G8.add_edge(1219,1300,weight=1)
G8.add_edge(1219,1425,weight=1)
G8.add_edge(1219,715,weight=1)
G8.add_edge(1219,1671,weight=1)
G8.add_edge(1219,834,weight=1)
G8.add_edge(1219,1517,weight=1)
G8.add_edge(1219,1531,weight=1)
G8.add_edge(2203,1572,weight=1)
G8.add_edge(2203,1304,weight=1)
G8.add_edge(2203,2639,weight=1)
G8.add_edge(886,832,weight=1)
G8.add_edge(1070,1097,weight=1)
G8.add_edge(1070,1686,weight=1)
G8.add_edge(1070,1456,weight=1)
G8.add_edge(1070,423,weight=1)
G8.add_edge(2567,2532,weight=1)
G8.add_edge(1603,1550,weight=1)
G8.add_edge(2046,2551,weight=1)
G8.add_edge(2038,1585,weight=1)
G8.add_edge(2038,289,weight=1)
G8.add_edge(2038,2152,weight=1)
G8.add_edge(289,2158,weight=1)
G8.add_edge(2152,1585,weight=1)
G8.add_edge(1165,1585,weight=1)
G8.add_edge(2158,2566,weight=1)
G8.add_edge(972,1603,weight=1)
G8.add_edge(1661,1084,weight=1)
G8.add_edge(1661,953,weight=1)
G8.add_edge(2612,1764,weight=1)
G8.add_edge(2612,2607,weight=1)
G8.add_edge(2077,212,weight=1)
G8.add_edge(1325,1544,weight=1)
G8.add_edge(1325,2077,weight=1)
G8.add_edge(1325,1748,weight=1)
G8.add_edge(1325,261,weight=1)
G8.add_edge(2120,2155,weight=1)
G8.add_edge(1077,1079,weight=1)
G8.add_edge(1079,1077,weight=1)
G8.add_edge(791,1511,weight=1)
G8.add_edge(420,1748,weight=1)
G8.add_edge(420,261,weight=1)
G8.add_edge(780,543,weight=1)
G8.add_edge(780,685,weight=1)
G8.add_edge(1348,1148,weight=1)
G8.add_edge(2579,100,weight=1)
G8.add_edge(1614,1615,weight=1)
G8.add_edge(1614,1163,weight=1)
G8.add_edge(1615,1614,weight=1)
G8.add_edge(2593,1853,weight=1)
G8.add_edge(1620,1614,weight=1)
G8.add_edge(1620,1615,weight=1)
G8.add_edge(1723,1620,weight=1)
G8.add_edge(1724,1620,weight=1)
G8.add_edge(1724,1723,weight=1)
G8.add_edge(2680,100,weight=1)
G8.add_edge(2680,2579,weight=1)
G8.add_edge(785,846,weight=1)
G8.add_edge(785,1614,weight=1)
G8.add_edge(785,1339,weight=1)
G8.add_edge(918,851,weight=1)
G8.add_edge(918,785,weight=1)
G8.add_edge(918,1339,weight=1)
G8.add_edge(1016,853,weight=1)
G8.add_edge(1016,997,weight=1)
G8.add_edge(1016,1339,weight=1)
G8.add_edge(1339,856,weight=1)
G8.add_edge(543,780,weight=1)
G8.add_edge(1520,964,weight=1)
G8.add_edge(1520,1178,weight=1)
G8.add_edge(2497,2572,weight=1)
G8.add_edge(2497,614,weight=1)
G8.add_edge(1675,1124,weight=1)
G8.add_edge(1675,1681,weight=1)
G8.add_edge(1681,1124,weight=1)
G8.add_edge(2234,2202,weight=1)
G8.add_edge(364,338,weight=1)
G8.add_edge(758,756,weight=1)
G8.add_edge(758,1966,weight=1)
G8.add_edge(2142,2419,weight=1)
G8.add_edge(300,747,weight=1)
G8.add_edge(300,2084,weight=1)
G8.add_edge(300,597,weight=1)
G8.add_edge(300,1828,weight=1)
G8.add_edge(300,2111,weight=1)
G8.add_edge(1686,1070,weight=1)
G8.add_edge(1686,1456,weight=1)
G8.add_edge(1686,423,weight=1)
G8.add_edge(1851,2555,weight=1)
G8.add_edge(1160,2565,weight=1)
G8.add_edge(1160,1486,weight=1)
G8.add_edge(381,1738,weight=1)
G8.add_edge(381,1785,weight=1)
G8.add_edge(381,2642,weight=1)
G8.add_edge(381,2197,weight=1)
G8.add_edge(381,1474,weight=1)
G8.add_edge(381,1475,weight=1)
G8.add_edge(381,1258,weight=1)
G8.add_edge(381,935,weight=1)
G8.add_edge(1537,1519,weight=1)
G8.add_edge(1537,2505,weight=1)
G8.add_edge(1772,1873,weight=1)
G8.add_edge(1222,2704,weight=1)
G8.add_edge(1222,1798,weight=1)
G8.add_edge(939,1128,weight=1)
G8.add_edge(939,2197,weight=1)
G8.add_edge(2059,756,weight=1)
G8.add_edge(2059,2233,weight=1)
G8.add_edge(2059,758,weight=1)
G8.add_edge(1456,1695,weight=1)
G8.add_edge(1559,2565,weight=1)
G8.add_edge(2565,1786,weight=1)
G8.add_edge(881,816,weight=1)
G8.add_edge(881,1524,weight=1)
G8.add_edge(2299,2519,weight=1)
G8.add_edge(1425,1800,weight=1)
G8.add_edge(1802,1815,weight=1)
G8.add_edge(835,1815,weight=1)
G8.add_edge(840,1527,weight=1)
G8.add_edge(840,1710,weight=1)
G8.add_edge(840,60,weight=1)
G8.add_edge(840,875,weight=1)
G8.add_edge(840,1068,weight=1)
G8.add_edge(1815,835,weight=1)
G8.add_edge(1822,1802,weight=1)
G8.add_edge(1822,835,weight=1)
G8.add_edge(1232,855,weight=1)
G8.add_edge(1232,957,weight=1)
G8.add_edge(1232,1986,weight=1)
G8.add_edge(1232,2087,weight=1)
G8.add_edge(1232,1231,weight=1)
G8.add_edge(1232,1233,weight=1)
G8.add_edge(2212,855,weight=1)
G8.add_edge(2212,1972,weight=1)
G8.add_edge(2212,1233,weight=1)
G8.add_edge(2217,2689,weight=1)
G8.add_edge(2217,2706,weight=1)
G8.add_edge(2471,1231,weight=1)
G8.add_edge(2471,1278,weight=1)
G8.add_edge(2471,2644,weight=1)
G8.add_edge(2471,2647,weight=1)
G8.add_edge(1496,1232,weight=1)
G8.add_edge(1785,2689,weight=1)
G8.add_edge(1785,2706,weight=1)
G8.add_edge(1785,56,weight=1)
G8.add_edge(1785,2200,weight=1)
G8.add_edge(1785,845,weight=1)
G8.add_edge(1937,1944,weight=1)
G8.add_edge(1015,1092,weight=1)
G8.add_edge(1015,1057,weight=1)
G8.add_edge(1261,1105,weight=1)
G8.add_edge(1263,1105,weight=1)
G8.add_edge(1511,791,weight=1)
G8.add_edge(1758,1843,weight=1)
G8.add_edge(349,1597,weight=1)
G8.add_edge(349,1245,weight=1)
G8.add_edge(1245,1597,weight=1)
G8.add_edge(1680,1727,weight=1)
G8.add_edge(1685,1727,weight=1)
G8.add_edge(1832,1833,weight=1)
G8.add_edge(1832,1837,weight=1)
G8.add_edge(1833,1837,weight=1)
G8.add_edge(1837,1833,weight=1)
G8.add_edge(1840,2689,weight=1)
G8.add_edge(1840,2706,weight=1)
G8.add_edge(1840,291,weight=1)
G8.add_edge(1841,2689,weight=1)
G8.add_edge(1841,2706,weight=1)
G8.add_edge(1035,1731,weight=1)
G8.add_edge(210,2705,weight=1)
G8.add_edge(1066,1280,weight=1)
G8.add_edge(2134,1926,weight=1)
G8.add_edge(2134,2200,weight=1)
G8.add_edge(2134,2207,weight=1)
G8.add_edge(2153,1278,weight=1)
G8.add_edge(2268,2197,weight=1)
G8.add_edge(2270,2278,weight=1)
G8.add_edge(1278,1477,weight=1)
G8.add_edge(1278,2278,weight=1)
G8.add_edge(2403,1922,weight=1)
G8.add_edge(1409,1922,weight=1)
G8.add_edge(2642,1746,weight=1)
G8.add_edge(2644,2647,weight=1)
G8.add_edge(2647,2644,weight=1)
G8.add_edge(2207,1926,weight=1)
G8.add_edge(2207,1985,weight=1)
G8.add_edge(2380,1985,weight=1)
G8.add_edge(2380,2207,weight=1)
G8.add_edge(2393,1926,weight=1)
G8.add_edge(1405,756,weight=1)
G8.add_edge(1405,1144,weight=1)
G8.add_edge(2513,1931,weight=1)
G8.add_edge(973,954,weight=1)
G8.add_edge(2066,1986,weight=1)
G8.add_edge(2066,2087,weight=1)
G8.add_edge(1168,1331,weight=1)
G8.add_edge(1168,1332,weight=1)
G8.add_edge(1168,1333,weight=1)
G8.add_edge(1790,2470,weight=1)
G8.add_edge(2103,2665,weight=1)
G8.add_edge(1474,1475,weight=1)
G8.add_edge(1474,1250,weight=1)
G8.add_edge(1475,1250,weight=1)
G8.add_edge(1716,1735,weight=1)
G8.add_edge(1716,1387,weight=1)
G8.add_edge(2622,2106,weight=1)
G8.add_edge(303,599,weight=1)
G8.add_edge(303,429,weight=1)
G8.add_edge(1667,1134,weight=1)
G8.add_edge(2669,2366,weight=1)
G8.add_edge(2122,2213,weight=1)
G8.add_edge(1388,1240,weight=1)
G8.add_edge(1818,2465,weight=1)
G8.add_edge(2698,2701,weight=1)
G8.add_edge(2698,2289,weight=1)
G8.add_edge(2698,2699,weight=1)
G8.add_edge(2698,1760,weight=1)
G8.add_edge(2699,2701,weight=1)
G8.add_edge(2699,1760,weight=1)
G8.add_edge(1760,2289,weight=1)
G8.add_edge(1301,1321,weight=1)
G8.add_edge(1301,1343,weight=1)
G8.add_edge(2455,1343,weight=1)
G8.add_edge(2387,2388,weight=1)
G8.add_edge(589,187,weight=1)
G8.add_edge(589,700,weight=1)
G8.add_edge(2629,2630,weight=1)
G8.add_edge(821,830,weight=1)
G8.add_edge(821,1116,weight=1)
G8.add_edge(129,104,weight=1)
G8.add_edge(963,830,weight=1)
G8.add_edge(2553,2027,weight=1)
G8.add_edge(2577,1052,weight=1)
G8.add_edge(1913,305,weight=1)
G8.add_edge(1913,1933,weight=1)
G8.add_edge(2243,2459,weight=1)
G8.add_edge(2376,2417,weight=1)
G8.add_edge(1682,1683,weight=1)
G8.add_edge(1683,1682,weight=1)
G8.add_edge(2662,2160,weight=1)
G8.add_edge(2662,1911,weight=1)
G8.add_edge(2662,2507,weight=1)
G8.add_edge(2662,2243,weight=1)
G8.add_edge(2662,2376,weight=1)
G8.add_edge(2367,2507,weight=1)
G8.add_edge(2367,2576,weight=1)
G8.add_edge(2367,2459,weight=1)
G8.add_edge(2576,2389,weight=1)
G8.add_edge(2249,2297,weight=1)
G8.add_edge(2252,2297,weight=1)
G8.add_edge(2281,2287,weight=1)
G8.add_edge(1114,1381,weight=1)
G8.add_edge(102,2681,weight=1)
G8.add_edge(102,2687,weight=1)
G8.add_edge(917,2687,weight=1)
G8.add_edge(917,2374,weight=1)
G8.add_edge(917,2375,weight=1)
G8.add_edge(1752,2692,weight=1)
G8.add_edge(889,1732,weight=1)
G8.add_edge(889,897,weight=1)
G8.add_edge(889,42,weight=1)
G8.add_edge(895,1256,weight=1)
G8.add_edge(905,1734,weight=1)
G8.add_edge(905,890,weight=1)
G8.add_edge(905,893,weight=1)
G8.add_edge(905,353,weight=1)
G8.add_edge(905,1373,weight=1)
G8.add_edge(905,116,weight=1)
G8.add_edge(2026,2694,weight=1)
G8.add_edge(2026,2378,weight=1)
G8.add_edge(293,905,weight=1)
G8.add_edge(2154,2700,weight=1)
G8.add_edge(2154,890,weight=1)
G8.add_edge(2154,1978,weight=1)
G8.add_edge(2154,1983,weight=1)
G8.add_edge(2154,1993,weight=1)
G8.add_edge(2154,2026,weight=1)
G8.add_edge(2154,1717,weight=1)
G8.add_edge(2154,2003,weight=1)
G8.add_edge(2154,2220,weight=1)
G8.add_edge(2154,2319,weight=1)
G8.add_edge(2154,116,weight=1)
G8.add_edge(2154,2009,weight=1)
G8.add_edge(2154,2023,weight=1)
G8.add_edge(2248,2702,weight=1)
G8.add_edge(2248,2264,weight=1)
G8.add_edge(2264,2702,weight=1)
G8.add_edge(534,778,weight=1)
G8.add_edge(2457,1743,weight=1)
G8.add_edge(1599,781,weight=1)
G8.add_edge(1714,890,weight=1)
G8.add_edge(1714,1871,weight=1)
G8.add_edge(1714,1979,weight=1)
G8.add_edge(1714,1983,weight=1)
G8.add_edge(1714,1992,weight=1)
G8.add_edge(1714,1993,weight=1)
G8.add_edge(1714,139,weight=1)
G8.add_edge(1714,42,weight=1)
G8.add_edge(1714,2154,weight=1)
G8.add_edge(1714,1717,weight=1)
G8.add_edge(1714,2003,weight=1)
G8.add_edge(1714,893,weight=1)
G8.add_edge(1714,469,weight=1)
G8.add_edge(1714,2511,weight=1)
G8.add_edge(1714,417,weight=1)
G8.add_edge(1714,116,weight=1)
G8.add_edge(1717,1983,weight=1)
G8.add_edge(1717,1992,weight=1)
G8.add_edge(1717,139,weight=1)
G8.add_edge(1717,42,weight=1)
G8.add_edge(1717,1714,weight=1)
G8.add_edge(1717,893,weight=1)
G8.add_edge(2003,1714,weight=1)
G8.add_edge(2286,2312,weight=1)
G8.add_edge(2286,2139,weight=1)
G8.add_edge(1384,1399,weight=1)
G8.add_edge(1385,1384,weight=1)
G8.add_edge(893,890,weight=1)
G8.add_edge(2235,2209,weight=1)
G8.add_edge(2236,2209,weight=1)
G8.add_edge(2363,1871,weight=1)
G8.add_edge(2363,2511,weight=1)
G8.add_edge(1373,897,weight=1)
G8.add_edge(1373,353,weight=1)
G8.add_edge(469,897,weight=1)
G8.add_edge(469,183,weight=1)
G8.add_edge(469,2378,weight=1)
G8.add_edge(469,492,weight=1)
G8.add_edge(492,183,weight=1)
G8.add_edge(492,2378,weight=1)
G8.add_edge(2511,1978,weight=1)
G8.add_edge(2511,1979,weight=1)
G8.add_edge(2511,1992,weight=1)
G8.add_edge(2511,2003,weight=1)
G8.add_edge(2511,417,weight=1)
G8.add_edge(127,95,weight=1)
G8.add_edge(127,2209,weight=1)
G8.add_edge(2209,2314,weight=1)
G8.add_edge(2220,1978,weight=1)
G8.add_edge(2220,2009,weight=1)
G8.add_edge(2220,2023,weight=1)
G8.add_edge(1256,96,weight=1)
G8.add_edge(359,96,weight=1)
G8.add_edge(359,1256,weight=1)
G8.add_edge(1762,2209,weight=1)
G8.add_edge(1914,417,weight=1)
G8.add_edge(2319,1978,weight=1)
G8.add_edge(2319,492,weight=1)
G8.add_edge(417,139,weight=1)
G8.add_edge(116,2422,weight=1)
G8.add_edge(1907,2422,weight=1)
G8.add_edge(2009,2023,weight=1)
G8.add_edge(2023,2009,weight=1)
G8.add_edge(1157,1014,weight=1)
G8.add_edge(715,1036,weight=1)
G8.add_edge(715,1300,weight=1)
G8.add_edge(715,834,weight=1)
G8.add_edge(1797,2002,weight=1)
G8.add_edge(1797,1031,weight=1)
G8.add_edge(1797,1918,weight=1)
G8.add_edge(1797,834,weight=1)
G8.add_edge(834,2002,weight=1)
G8.add_edge(834,1517,weight=1)
G8.add_edge(1393,1300,weight=1)
G8.add_edge(1531,1425,weight=1)
G8.add_edge(590,283,weight=1)
G8.add_edge(590,593,weight=1)
G8.add_edge(593,1145,weight=1)
G8.add_edge(593,283,weight=1)
G8.add_edge(593,590,weight=1)
G8.add_edge(593,1510,weight=1)
G8.add_edge(1510,1145,weight=1)
G8.add_edge(1510,590,weight=1)
G8.add_edge(704,2234,weight=1)
G8.add_edge(1799,2121,weight=1)
G8.add_edge(1701,1158,weight=1)
G8.add_edge(742,1158,weight=1)
G8.add_edge(742,688,weight=1)
G8.add_edge(1705,1158,weight=1)
G8.add_edge(1135,1265,weight=1)
G8.add_edge(1135,2282,weight=1)
G8.add_edge(274,373,weight=1)
G8.add_edge(1748,2161,weight=1)
G8.add_edge(1748,2435,weight=1)
G8.add_edge(2010,2535,weight=1)
G8.add_edge(786,1514,weight=1)
G8.add_edge(2000,2516,weight=1)
G8.add_edge(2000,2004,weight=1)
G8.add_edge(2004,2516,weight=1)
G8.add_edge(2004,2000,weight=1)
G8.add_edge(2265,2520,weight=1)
G8.add_edge(1386,1654,weight=1)
G8.add_edge(1636,1707,weight=1)
G8.add_edge(1874,1999,weight=1)
G8.add_edge(281,11,weight=1)
G8.add_edge(389,2591,weight=1)
G8.add_edge(1982,2492,weight=1)
G8.add_edge(2240,588,weight=1)
G8.add_edge(2240,2508,weight=1)
G8.add_edge(2240,2514,weight=1)
G8.add_edge(2240,2414,weight=1)
G8.add_edge(2240,2415,weight=1)
G8.add_edge(372,2508,weight=1)
G8.add_edge(1374,1367,weight=1)
G8.add_edge(2374,2508,weight=1)
G8.add_edge(2374,2414,weight=1)
G8.add_edge(2374,2415,weight=1)
G8.add_edge(2374,1367,weight=1)
G8.add_edge(2374,2375,weight=1)
G8.add_edge(2375,2508,weight=1)
G8.add_edge(2375,2514,weight=1)
G8.add_edge(2375,1367,weight=1)
G8.add_edge(2375,2374,weight=1)
G8.add_edge(1316,2596,weight=1)
G8.add_edge(2453,2596,weight=1)
G8.add_edge(2453,1316,weight=1)
G8.add_edge(2563,2596,weight=1)
G8.add_edge(1778,2065,weight=1)
G8.add_edge(1803,2065,weight=1)
G8.add_edge(1803,1778,weight=1)
G8.add_edge(1938,1065,weight=1)
G8.add_edge(953,1065,weight=1)
G8.add_edge(1065,1938,weight=1)
G8.add_edge(2539,2326,weight=1)
G8.add_edge(2115,1924,weight=1)
G8.add_edge(2125,1929,weight=1)
G8.add_edge(2125,2115,weight=1)
G8.add_edge(2554,1876,weight=1)
G8.add_edge(2504,2359,weight=1)
G8.add_edge(1512,1358,weight=1)
G8.add_edge(991,1775,weight=1)
G8.add_edge(2419,2142,weight=1)
G8.add_edge(2419,2346,weight=1)
G8.add_edge(874,921,weight=1)
G8.add_edge(60,1710,weight=1)
G8.add_edge(60,571,weight=1)
G8.add_edge(60,873,weight=1)
G8.add_edge(60,875,weight=1)
G8.add_edge(2107,1751,weight=1)
G8.add_edge(1630,797,weight=1)
G8.add_edge(1875,1921,weight=1)
G8.add_edge(2070,2435,weight=1)
G8.add_edge(2089,2525,weight=1)
G8.add_edge(2089,2526,weight=1)
G8.add_edge(2089,2168,weight=1)
G8.add_edge(2089,249,weight=1)
G8.add_edge(2089,777,weight=1)
G8.add_edge(249,420,weight=1)
G8.add_edge(249,106,weight=1)
G8.add_edge(261,620,weight=1)
G8.add_edge(261,2525,weight=1)
G8.add_edge(261,2167,weight=1)
G8.add_edge(261,420,weight=1)
G8.add_edge(261,2070,weight=1)
G8.add_edge(262,627,weight=1)
G8.add_edge(2580,2280,weight=1)
G8.add_edge(2580,2337,weight=1)
G8.add_edge(2600,2163,weight=1)
G8.add_edge(1867,2273,weight=1)
G8.add_edge(2117,2126,weight=1)
G8.add_edge(1121,1678,weight=1)
G8.add_edge(1121,2276,weight=1)
G8.add_edge(1552,273,weight=1)
G8.add_edge(1552,915,weight=1)
G8.add_edge(2192,363,weight=1)
G8.add_edge(2338,2219,weight=1)
G8.add_edge(2338,2607,weight=1)
G8.add_edge(2338,2611,weight=1)
G8.add_edge(915,273,weight=1)
G8.add_edge(1246,1244,weight=1)
G8.add_edge(1930,2350,weight=1)
G8.add_edge(572,2332,weight=1)
G8.add_edge(572,425,weight=1)
G8.add_edge(572,2496,weight=1)
G8.add_edge(572,1763,weight=1)
G8.add_edge(572,20,weight=1)
G8.add_edge(572,2695,weight=1)
G8.add_edge(2496,572,weight=1)
G8.add_edge(2496,1763,weight=1)
G8.add_edge(2496,2695,weight=1)
G8.add_edge(1763,572,weight=1)
G8.add_edge(1763,2695,weight=1)
G8.add_edge(20,572,weight=1)
G8.add_edge(2534,2518,weight=1)
G8.add_edge(393,155,weight=1)
G8.add_edge(1660,1653,weight=1)
G8.add_edge(1660,1094,weight=1)
G8.add_edge(1660,1095,weight=1)
G8.add_edge(2086,1094,weight=1)
G8.add_edge(2086,1095,weight=1)
G8.add_edge(356,1094,weight=1)
G8.add_edge(356,1095,weight=1)
G8.add_edge(356,2069,weight=1)
G8.add_edge(2229,1094,weight=1)
G8.add_edge(2229,1095,weight=1)
G8.add_edge(2229,2069,weight=1)
G8.add_edge(2229,2086,weight=1)
G8.add_edge(2695,2332,weight=1)
G8.add_edge(618,227,weight=1)
G8.add_edge(423,1692,weight=1)
G8.add_edge(2143,2097,weight=1)
G8.add_edge(1478,1253,weight=1)
G8.add_edge(1486,1253,weight=1)
G8.add_edge(2186,2110,weight=1)
G8.add_edge(2118,2640,weight=1)
G8.add_edge(1212,1248,weight=1)
G8.add_edge(1885,746,weight=1)
G8.add_edge(1885,1887,weight=1)
G8.add_edge(1885,1903,weight=1)
G8.add_edge(1886,746,weight=1)
G8.add_edge(1886,1885,weight=1)
G8.add_edge(1886,1887,weight=1)
G8.add_edge(1886,1903,weight=1)
G8.add_edge(1887,746,weight=1)
G8.add_edge(1887,1903,weight=1)
G8.add_edge(1888,2259,weight=1)
G8.add_edge(1903,1888,weight=1)
G8.add_edge(838,1687,weight=1)




#Citas
G9 = nx.DiGraph()
G9.add_edge(1,870,weight=1)
G9.add_edge(2,598,weight=1)
G9.add_edge(2,2207,weight=1)
G9.add_edge(2,2887,weight=1)
G9.add_edge(2,670,weight=1)
G9.add_edge(2,327,weight=1)
G9.add_edge(2,795,weight=1)
G9.add_edge(3,2090,weight=1)
G9.add_edge(3,712,weight=1)
G9.add_edge(3,2929,weight=1)
G9.add_edge(3,2907,weight=1)
G9.add_edge(3,2039,weight=1)
G9.add_edge(3,18,weight=1)
G9.add_edge(3,1560,weight=1)
G9.add_edge(4,452,weight=1)
G9.add_edge(4,930,weight=1)
G9.add_edge(5,627,weight=1)
G9.add_edge(5,774,weight=1)
G9.add_edge(6,642,weight=1)
G9.add_edge(6,443,weight=1)
G9.add_edge(6,687,weight=1)
G9.add_edge(6,446,weight=1)
G9.add_edge(6,2303,weight=1)
G9.add_edge(6,858,weight=1)
G9.add_edge(6,931,weight=1)
G9.add_edge(7,2090,weight=1)
G9.add_edge(7,2686,weight=1)
G9.add_edge(7,2508,weight=1)
G9.add_edge(7,3075,weight=1)
G9.add_edge(8,599,weight=1)
G9.add_edge(8,679,weight=1)
G9.add_edge(8,15,weight=1)
G9.add_edge(8,715,weight=1)
G9.add_edge(8,716,weight=1)
G9.add_edge(8,771,weight=1)
G9.add_edge(8,799,weight=1)
G9.add_edge(8,870,weight=1)
G9.add_edge(8,897,weight=1)
G9.add_edge(8,898,weight=1)
G9.add_edge(9,561,weight=1)
G9.add_edge(9,905,weight=1)
G9.add_edge(10,1196,weight=1)
G9.add_edge(11,354,weight=1)
G9.add_edge(11,554,weight=1)
G9.add_edge(11,184,weight=1)
G9.add_edge(11,720,weight=1)
G9.add_edge(11,735,weight=1)
G9.add_edge(11,201,weight=1)
G9.add_edge(12,172,weight=1)
G9.add_edge(12,2217,weight=1)
G9.add_edge(12,2201,weight=1)
G9.add_edge(12,2532,weight=1)
G9.add_edge(12,2188,weight=1)
G9.add_edge(12,1642,weight=1)
G9.add_edge(12,3258,weight=1)
G9.add_edge(12,3059,weight=1)
G9.add_edge(12,2277,weight=1)
G9.add_edge(12,2298,weight=1)
G9.add_edge(13,39,weight=1)
G9.add_edge(14,749,weight=1)
G9.add_edge(15,715,weight=1)
G9.add_edge(16,170,weight=1)
G9.add_edge(16,3019,weight=1)
G9.add_edge(17,1912,weight=1)
G9.add_edge(18,1542,weight=1)
G9.add_edge(18,1552,weight=1)
G9.add_edge(18,712,weight=1)
G9.add_edge(18,2929,weight=1)
G9.add_edge(18,2907,weight=1)
G9.add_edge(18,2039,weight=1)
G9.add_edge(18,1560,weight=1)
G9.add_edge(19,2521,weight=1)
G9.add_edge(19,2495,weight=1)
G9.add_edge(20,848,weight=1)
G9.add_edge(21,1200,weight=1)
G9.add_edge(21,840,weight=1)
G9.add_edge(22,599,weight=1)
G9.add_edge(22,830,weight=1)
G9.add_edge(23,2089,weight=1)
G9.add_edge(23,1345,weight=1)
G9.add_edge(23,2511,weight=1)
G9.add_edge(24,1170,weight=1)
G9.add_edge(25,601,weight=1)
G9.add_edge(25,1559,weight=1)
G9.add_edge(25,863,weight=1)
G9.add_edge(26,782,weight=1)
G9.add_edge(26,528,weight=1)
G9.add_edge(26,202,weight=1)
G9.add_edge(27,1340,weight=1)
G9.add_edge(27,1550,weight=1)
G9.add_edge(27,1192,weight=1)
G9.add_edge(27,802,weight=1)
G9.add_edge(27,502,weight=1)
G9.add_edge(28,360,weight=1)
G9.add_edge(28,2677,weight=1)
G9.add_edge(29,1191,weight=1)
G9.add_edge(29,367,weight=1)
G9.add_edge(30,1548,weight=1)
G9.add_edge(31,598,weight=1)
G9.add_edge(31,2207,weight=1)
G9.add_edge(31,2887,weight=1)
G9.add_edge(31,670,weight=1)
G9.add_edge(31,327,weight=1)
G9.add_edge(31,795,weight=1)
G9.add_edge(32,2346,weight=1)
G9.add_edge(32,1284,weight=1)
G9.add_edge(32,840,weight=1)
G9.add_edge(32,500,weight=1)
G9.add_edge(32,2366,weight=1)
G9.add_edge(32,959,weight=1)
G9.add_edge(32,2246,weight=1)
G9.add_edge(32,2318,weight=1)
G9.add_edge(32,1686,weight=1)
G9.add_edge(32,2625,weight=1)
G9.add_edge(32,98,weight=1)
G9.add_edge(32,2919,weight=1)
G9.add_edge(33,1975,weight=1)
G9.add_edge(34,170,weight=1)
G9.add_edge(34,3259,weight=1)
G9.add_edge(34,677,weight=1)
G9.add_edge(34,1196,weight=1)
G9.add_edge(34,750,weight=1)
G9.add_edge(34,755,weight=1)
G9.add_edge(34,798,weight=1)
G9.add_edge(34,812,weight=1)
G9.add_edge(34,888,weight=1)
G9.add_edge(34,899,weight=1)
G9.add_edge(34,932,weight=1)
G9.add_edge(35,3072,weight=1)
G9.add_edge(36,2134,weight=1)
G9.add_edge(36,2209,weight=1)
G9.add_edge(36,1474,weight=1)
G9.add_edge(36,2039,weight=1)
G9.add_edge(36,3087,weight=1)
G9.add_edge(36,2508,weight=1)
G9.add_edge(36,1352,weight=1)
G9.add_edge(36,196,weight=1)
G9.add_edge(36,199,weight=1)
G9.add_edge(36,848,weight=1)
G9.add_edge(36,2959,weight=1)
G9.add_edge(36,892,weight=1)
G9.add_edge(37,354,weight=1)
G9.add_edge(37,554,weight=1)
G9.add_edge(37,184,weight=1)
G9.add_edge(37,720,weight=1)
G9.add_edge(37,735,weight=1)
G9.add_edge(37,201,weight=1)
G9.add_edge(38,870,weight=1)
G9.add_edge(39,837,weight=1)
G9.add_edge(40,172,weight=1)
G9.add_edge(40,2217,weight=1)
G9.add_edge(40,2201,weight=1)
G9.add_edge(40,44,weight=1)
G9.add_edge(40,2532,weight=1)
G9.add_edge(40,2188,weight=1)
G9.add_edge(40,1642,weight=1)
G9.add_edge(40,3258,weight=1)
G9.add_edge(40,2448,weight=1)
G9.add_edge(40,3059,weight=1)
G9.add_edge(40,2277,weight=1)
G9.add_edge(40,2298,weight=1)
G9.add_edge(41,735,weight=1)
G9.add_edge(42,795,weight=1)
G9.add_edge(43,444,weight=1)
G9.add_edge(43,887,weight=1)
G9.add_edge(44,172,weight=1)
G9.add_edge(44,2217,weight=1)
G9.add_edge(45,2032,weight=1)
G9.add_edge(45,2670,weight=1)
G9.add_edge(46,1874,weight=1)
G9.add_edge(46,2577,weight=1)
G9.add_edge(47,2579,weight=1)
G9.add_edge(48,1600,weight=1)
G9.add_edge(49,2092,weight=1)
G9.add_edge(49,1269,weight=1)
G9.add_edge(50,3006,weight=1)
G9.add_edge(51,79,weight=1)
G9.add_edge(52,1377,weight=1)
G9.add_edge(53,3154,weight=1)
G9.add_edge(54,61,weight=1)
G9.add_edge(54,68,weight=1)
G9.add_edge(55,2532,weight=1)
G9.add_edge(56,1595,weight=1)
G9.add_edge(56,1377,weight=1)
G9.add_edge(56,2409,weight=1)
G9.add_edge(56,71,weight=1)
G9.add_edge(56,73,weight=1)
G9.add_edge(57,2744,weight=1)
G9.add_edge(57,455,weight=1)
G9.add_edge(57,2553,weight=1)
G9.add_edge(57,2645,weight=1)
G9.add_edge(57,245,weight=1)
G9.add_edge(58,3173,weight=1)
G9.add_edge(58,130,weight=1)
G9.add_edge(59,383,weight=1)
G9.add_edge(59,2175,weight=1)
G9.add_edge(60,145,weight=1)
G9.add_edge(61,2390,weight=1)
G9.add_edge(61,2623,weight=1)
G9.add_edge(62,2412,weight=1)
G9.add_edge(62,1487,weight=1)
G9.add_edge(63,1268,weight=1)
G9.add_edge(64,1850,weight=1)
G9.add_edge(64,2827,weight=1)
G9.add_edge(64,1816,weight=1)
G9.add_edge(64,2575,weight=1)
G9.add_edge(65,1207,weight=1)
G9.add_edge(65,2044,weight=1)
G9.add_edge(66,2634,weight=1)
G9.add_edge(67,2631,weight=1)
G9.add_edge(69,2724,weight=1)
G9.add_edge(69,2564,weight=1)
G9.add_edge(69,166,weight=1)
G9.add_edge(69,2952,weight=1)
G9.add_edge(70,724,weight=1)
G9.add_edge(70,2564,weight=1)
G9.add_edge(70,2952,weight=1)
G9.add_edge(71,2309,weight=1)
G9.add_edge(71,1382,weight=1)
G9.add_edge(71,2891,weight=1)
G9.add_edge(72,1645,weight=1)
G9.add_edge(73,2138,weight=1)
G9.add_edge(73,2774,weight=1)
G9.add_edge(73,1381,weight=1)
G9.add_edge(73,1461,weight=1)
G9.add_edge(73,1488,weight=1)
G9.add_edge(73,3235,weight=1)
G9.add_edge(73,313,weight=1)
G9.add_edge(73,1672,weight=1)
G9.add_edge(74,294,weight=1)
G9.add_edge(75,297,weight=1)
G9.add_edge(75,2295,weight=1)
G9.add_edge(76,2754,weight=1)
G9.add_edge(76,2951,weight=1)
G9.add_edge(77,2572,weight=1)
G9.add_edge(77,993,weight=1)
G9.add_edge(77,994,weight=1)
G9.add_edge(78,2533,weight=1)
G9.add_edge(78,1908,weight=1)
G9.add_edge(79,373,weight=1)
G9.add_edge(79,228,weight=1)
G9.add_edge(79,995,weight=1)
G9.add_edge(80,2579,weight=1)
G9.add_edge(81,2230,weight=1)
G9.add_edge(81,1219,weight=1)
G9.add_edge(81,2906,weight=1)
G9.add_edge(82,1590,weight=1)
G9.add_edge(83,2336,weight=1)
G9.add_edge(84,1707,weight=1)
G9.add_edge(85,1471,weight=1)
G9.add_edge(86,1529,weight=1)
G9.add_edge(86,1242,weight=1)
G9.add_edge(87,111,weight=1)
G9.add_edge(88,100,weight=1)
G9.add_edge(88,268,weight=1)
G9.add_edge(88,1055,weight=1)
G9.add_edge(89,2678,weight=1)
G9.add_edge(90,3199,weight=1)
G9.add_edge(91,1083,weight=1)
G9.add_edge(92,1712,weight=1)
G9.add_edge(93,2875,weight=1)
G9.add_edge(93,1744,weight=1)
G9.add_edge(94,1010,weight=1)
G9.add_edge(94,2080,weight=1)
G9.add_edge(95,2476,weight=1)
G9.add_edge(96,518,weight=1)
G9.add_edge(96,1059,weight=1)
G9.add_edge(96,2639,weight=1)
G9.add_edge(97,2770,weight=1)
G9.add_edge(97,279,weight=1)
G9.add_edge(97,123,weight=1)
G9.add_edge(98,2366,weight=1)
G9.add_edge(98,1030,weight=1)
G9.add_edge(98,1454,weight=1)
G9.add_edge(98,127,weight=1)
G9.add_edge(99,2366,weight=1)
G9.add_edge(99,1454,weight=1)
G9.add_edge(100,268,weight=1)
G9.add_edge(100,1055,weight=1)
G9.add_edge(101,1117,weight=1)
G9.add_edge(102,1023,weight=1)
G9.add_edge(102,1031,weight=1)
G9.add_edge(102,1038,weight=1)
G9.add_edge(102,1039,weight=1)
G9.add_edge(102,1955,weight=1)
G9.add_edge(102,1069,weight=1)
G9.add_edge(102,1750,weight=1)
G9.add_edge(103,499,weight=1)
G9.add_edge(103,500,weight=1)
G9.add_edge(103,887,weight=1)
G9.add_edge(103,2366,weight=1)
G9.add_edge(103,126,weight=1)
G9.add_edge(104,113,weight=1)
G9.add_edge(105,974,weight=1)
G9.add_edge(105,257,weight=1)
G9.add_edge(105,3171,weight=1)
G9.add_edge(106,2786,weight=1)
G9.add_edge(106,2017,weight=1)
G9.add_edge(106,2271,weight=1)
G9.add_edge(107,499,weight=1)
G9.add_edge(107,2924,weight=1)
G9.add_edge(108,2472,weight=1)
G9.add_edge(109,113,weight=1)
G9.add_edge(110,1841,weight=1)
G9.add_edge(112,379,weight=1)
G9.add_edge(112,1838,weight=1)
G9.add_edge(114,1400,weight=1)
G9.add_edge(115,1420,weight=1)
G9.add_edge(116,1079,weight=1)
G9.add_edge(117,427,weight=1)
G9.add_edge(117,3207,weight=1)
G9.add_edge(118,544,weight=1)
G9.add_edge(119,1092,weight=1)
G9.add_edge(120,544,weight=1)
G9.add_edge(121,2193,weight=1)
G9.add_edge(121,2486,weight=1)
G9.add_edge(122,3001,weight=1)
G9.add_edge(123,2500,weight=1)
G9.add_edge(123,279,weight=1)
G9.add_edge(124,1067,weight=1)
G9.add_edge(125,2686,weight=1)
G9.add_edge(125,589,weight=1)
G9.add_edge(127,494,weight=1)
G9.add_edge(127,2366,weight=1)
G9.add_edge(128,3183,weight=1)
G9.add_edge(129,1421,weight=1)
G9.add_edge(130,1693,weight=1)
G9.add_edge(130,1710,weight=1)
G9.add_edge(131,765,weight=1)
G9.add_edge(132,1342,weight=1)
G9.add_edge(132,647,weight=1)
G9.add_edge(132,678,weight=1)
G9.add_edge(132,682,weight=1)
G9.add_edge(132,485,weight=1)
G9.add_edge(132,369,weight=1)
G9.add_edge(133,2683,weight=1)
G9.add_edge(133,2684,weight=1)
G9.add_edge(133,2685,weight=1)
G9.add_edge(133,3072,weight=1)
G9.add_edge(133,586,weight=1)
G9.add_edge(133,1171,weight=1)
G9.add_edge(133,2798,weight=1)
G9.add_edge(133,2201,weight=1)
G9.add_edge(133,3132,weight=1)
G9.add_edge(133,1204,weight=1)
G9.add_edge(134,1565,weight=1)
G9.add_edge(135,2092,weight=1)
G9.add_edge(135,2508,weight=1)
G9.add_edge(135,1918,weight=1)
G9.add_edge(136,765,weight=1)
G9.add_edge(137,809,weight=1)
G9.add_edge(137,881,weight=1)
G9.add_edge(137,564,weight=1)
G9.add_edge(137,2155,weight=1)
G9.add_edge(137,1306,weight=1)
G9.add_edge(137,990,weight=1)
G9.add_edge(138,795,weight=1)
G9.add_edge(138,813,weight=1)
G9.add_edge(139,1473,weight=1)
G9.add_edge(140,1193,weight=1)
G9.add_edge(140,754,weight=1)
G9.add_edge(140,1791,weight=1)
G9.add_edge(140,871,weight=1)
G9.add_edge(140,2222,weight=1)
G9.add_edge(141,675,weight=1)
G9.add_edge(141,446,weight=1)
G9.add_edge(141,365,weight=1)
G9.add_edge(141,790,weight=1)
G9.add_edge(141,800,weight=1)
G9.add_edge(142,2199,weight=1)
G9.add_edge(143,2530,weight=1)
G9.add_edge(143,2064,weight=1)
G9.add_edge(143,3246,weight=1)
G9.add_edge(143,390,weight=1)
G9.add_edge(143,981,weight=1)
G9.add_edge(143,1658,weight=1)
G9.add_edge(144,378,weight=1)
G9.add_edge(144,2439,weight=1)
G9.add_edge(144,1487,weight=1)
G9.add_edge(145,1219,weight=1)
G9.add_edge(146,2227,weight=1)
G9.add_edge(147,2707,weight=1)
G9.add_edge(148,2391,weight=1)
G9.add_edge(148,2892,weight=1)
G9.add_edge(149,947,weight=1)
G9.add_edge(149,1378,weight=1)
G9.add_edge(150,1543,weight=1)
G9.add_edge(150,2376,weight=1)
G9.add_edge(151,1766,weight=1)
G9.add_edge(151,335,weight=1)
G9.add_edge(151,3153,weight=1)
G9.add_edge(151,2537,weight=1)
G9.add_edge(151,1214,weight=1)
G9.add_edge(151,1643,weight=1)
G9.add_edge(151,2554,weight=1)
G9.add_edge(152,459,weight=1)
G9.add_edge(153,2072,weight=1)
G9.add_edge(153,207,weight=1)
G9.add_edge(153,3011,weight=1)
G9.add_edge(153,2389,weight=1)
G9.add_edge(153,3149,weight=1)
G9.add_edge(153,239,weight=1)
G9.add_edge(153,2950,weight=1)
G9.add_edge(154,565,weight=1)
G9.add_edge(154,956,weight=1)
G9.add_edge(154,979,weight=1)
G9.add_edge(154,568,weight=1)
G9.add_edge(155,2418,weight=1)
G9.add_edge(155,463,weight=1)
G9.add_edge(156,2761,weight=1)
G9.add_edge(156,2406,weight=1)
G9.add_edge(156,3005,weight=1)
G9.add_edge(157,1653,weight=1)
G9.add_edge(158,2909,weight=1)
G9.add_edge(158,2322,weight=1)
G9.add_edge(159,1045,weight=1)
G9.add_edge(160,2941,weight=1)
G9.add_edge(161,2090,weight=1)
G9.add_edge(161,2885,weight=1)
G9.add_edge(162,1879,weight=1)
G9.add_edge(163,2839,weight=1)
G9.add_edge(164,2948,weight=1)
G9.add_edge(164,3244,weight=1)
G9.add_edge(164,288,weight=1)
G9.add_edge(165,2028,weight=1)
G9.add_edge(166,1514,weight=1)
G9.add_edge(166,1234,weight=1)
G9.add_edge(166,2952,weight=1)
G9.add_edge(167,433,weight=1)
G9.add_edge(167,2293,weight=1)
G9.add_edge(168,1747,weight=1)
G9.add_edge(168,1246,weight=1)
G9.add_edge(168,2860,weight=1)
G9.add_edge(168,1100,weight=1)
G9.add_edge(168,1101,weight=1)
G9.add_edge(168,1119,weight=1)
G9.add_edge(169,2279,weight=1)
G9.add_edge(169,2613,weight=1)
G9.add_edge(170,1473,weight=1)
G9.add_edge(170,630,weight=1)
G9.add_edge(170,714,weight=1)
G9.add_edge(170,753,weight=1)
G9.add_edge(170,772,weight=1)
G9.add_edge(170,812,weight=1)
G9.add_edge(170,485,weight=1)
G9.add_edge(170,369,weight=1)
G9.add_edge(170,907,weight=1)
G9.add_edge(171,555,weight=1)
G9.add_edge(172,2498,weight=1)
G9.add_edge(172,494,weight=1)
G9.add_edge(173,555,weight=1)
G9.add_edge(174,1789,weight=1)
G9.add_edge(174,2657,weight=1)
G9.add_edge(174,1978,weight=1)
G9.add_edge(174,276,weight=1)
G9.add_edge(175,179,weight=1)
G9.add_edge(175,2091,weight=1)
G9.add_edge(175,2092,weight=1)
G9.add_edge(175,1872,weight=1)
G9.add_edge(175,2994,weight=1)
G9.add_edge(176,1476,weight=1)
G9.add_edge(177,2955,weight=1)
G9.add_edge(177,1358,weight=1)
G9.add_edge(178,919,weight=1)
G9.add_edge(179,1855,weight=1)
G9.add_edge(179,180,weight=1)
G9.add_edge(179,692,weight=1)
G9.add_edge(179,704,weight=1)
G9.add_edge(179,1915,weight=1)
G9.add_edge(179,730,weight=1)
G9.add_edge(179,731,weight=1)
G9.add_edge(179,2619,weight=1)
G9.add_edge(179,1918,weight=1)
G9.add_edge(179,1284,weight=1)
G9.add_edge(179,839,weight=1)
G9.add_edge(179,840,weight=1)
G9.add_edge(179,847,weight=1)
G9.add_edge(179,867,weight=1)
G9.add_edge(180,692,weight=1)
G9.add_edge(181,2134,weight=1)
G9.add_edge(181,491,weight=1)
G9.add_edge(181,1547,weight=1)
G9.add_edge(181,1449,weight=1)
G9.add_edge(181,484,weight=1)
G9.add_edge(181,2059,weight=1)
G9.add_edge(181,1289,weight=1)
G9.add_edge(181,2502,weight=1)
G9.add_edge(181,1343,weight=1)
G9.add_edge(181,2209,weight=1)
G9.add_edge(181,1349,weight=1)
G9.add_edge(181,3015,weight=1)
G9.add_edge(181,2039,weight=1)
G9.add_edge(181,363,weight=1)
G9.add_edge(181,2304,weight=1)
G9.add_edge(181,3087,weight=1)
G9.add_edge(181,1973,weight=1)
G9.add_edge(181,2507,weight=1)
G9.add_edge(181,1352,weight=1)
G9.add_edge(181,1563,weight=1)
G9.add_edge(181,1128,weight=1)
G9.add_edge(181,2912,weight=1)
G9.add_edge(181,3053,weight=1)
G9.add_edge(181,1200,weight=1)
G9.add_edge(181,3222,weight=1)
G9.add_edge(181,2362,weight=1)
G9.add_edge(181,2956,weight=1)
G9.add_edge(181,2127,weight=1)
G9.add_edge(181,196,weight=1)
G9.add_edge(181,1573,weight=1)
G9.add_edge(181,199,weight=1)
G9.add_edge(181,2959,weight=1)
G9.add_edge(181,892,weight=1)
G9.add_edge(181,3167,weight=1)
G9.add_edge(182,3221,weight=1)
G9.add_edge(182,2883,weight=1)
G9.add_edge(183,720,weight=1)
G9.add_edge(184,554,weight=1)
G9.add_edge(184,720,weight=1)
G9.add_edge(184,821,weight=1)
G9.add_edge(185,722,weight=1)
G9.add_edge(185,723,weight=1)
G9.add_edge(185,362,weight=1)
G9.add_edge(185,1201,weight=1)
G9.add_edge(185,502,weight=1)
G9.add_edge(186,724,weight=1)
G9.add_edge(186,362,weight=1)
G9.add_edge(186,790,weight=1)
G9.add_edge(187,870,weight=1)
G9.add_edge(188,2993,weight=1)
G9.add_edge(188,2907,weight=1)
G9.add_edge(188,1978,weight=1)
G9.add_edge(188,1984,weight=1)
G9.add_edge(189,2798,weight=1)
G9.add_edge(190,3098,weight=1)
G9.add_edge(190,2134,weight=1)
G9.add_edge(190,2708,weight=1)
G9.add_edge(190,3087,weight=1)
G9.add_edge(190,290,weight=1)
G9.add_edge(190,291,weight=1)
G9.add_edge(190,1786,weight=1)
G9.add_edge(190,1789,weight=1)
G9.add_edge(190,2360,weight=1)
G9.add_edge(190,2657,weight=1)
G9.add_edge(190,1790,weight=1)
G9.add_edge(190,2641,weight=1)
G9.add_edge(190,1793,weight=1)
G9.add_edge(190,2517,weight=1)
G9.add_edge(190,2218,weight=1)
G9.add_edge(190,3066,weight=1)
G9.add_edge(190,371,weight=1)
G9.add_edge(190,2630,weight=1)
G9.add_edge(190,975,weight=1)
G9.add_edge(190,992,weight=1)
G9.add_edge(190,2494,weight=1)
G9.add_edge(191,1921,weight=1)
G9.add_edge(192,720,weight=1)
G9.add_edge(193,1974,weight=1)
G9.add_edge(193,1919,weight=1)
G9.add_edge(194,322,weight=1)
G9.add_edge(194,2782,weight=1)
G9.add_edge(195,824,weight=1)
G9.add_edge(195,2958,weight=1)
G9.add_edge(196,2686,weight=1)
G9.add_edge(196,1343,weight=1)
G9.add_edge(196,2907,weight=1)
G9.add_edge(196,2041,weight=1)
G9.add_edge(196,2127,weight=1)
G9.add_edge(196,2979,weight=1)
G9.add_edge(196,199,weight=1)
G9.add_edge(196,848,weight=1)
G9.add_edge(197,2520,weight=1)
G9.add_edge(198,625,weight=1)
G9.add_edge(198,735,weight=1)
G9.add_edge(198,837,weight=1)
G9.add_edge(199,1343,weight=1)
G9.add_edge(199,2907,weight=1)
G9.add_edge(199,2039,weight=1)
G9.add_edge(199,2127,weight=1)
G9.add_edge(199,2979,weight=1)
G9.add_edge(199,848,weight=1)
G9.add_edge(199,892,weight=1)
G9.add_edge(200,687,weight=1)
G9.add_edge(200,877,weight=1)
G9.add_edge(200,1029,weight=1)
G9.add_edge(202,591,weight=1)
G9.add_edge(202,1907,weight=1)
G9.add_edge(203,795,weight=1)
G9.add_edge(203,3009,weight=1)
G9.add_edge(203,927,weight=1)
G9.add_edge(203,929,weight=1)
G9.add_edge(204,1368,weight=1)
G9.add_edge(204,1928,weight=1)
G9.add_edge(204,458,weight=1)
G9.add_edge(204,226,weight=1)
G9.add_edge(204,1142,weight=1)
G9.add_edge(204,1668,weight=1)
G9.add_edge(205,1479,weight=1)
G9.add_edge(205,2237,weight=1)
G9.add_edge(205,2406,weight=1)
G9.add_edge(205,2562,weight=1)
G9.add_edge(205,1485,weight=1)
G9.add_edge(205,1674,weight=1)
G9.add_edge(205,464,weight=1)
G9.add_edge(205,2260,weight=1)
G9.add_edge(205,2010,weight=1)
G9.add_edge(206,247,weight=1)
G9.add_edge(207,2950,weight=1)
G9.add_edge(208,2668,weight=1)
G9.add_edge(209,2072,weight=1)
G9.add_edge(209,2695,weight=1)
G9.add_edge(209,3011,weight=1)
G9.add_edge(209,2074,weight=1)
G9.add_edge(209,2075,weight=1)
G9.add_edge(209,2241,weight=1)
G9.add_edge(209,2158,weight=1)
G9.add_edge(209,1388,weight=1)
G9.add_edge(209,243,weight=1)
G9.add_edge(209,2697,weight=1)
G9.add_edge(210,2092,weight=1)
G9.add_edge(210,2962,weight=1)
G9.add_edge(210,1269,weight=1)
G9.add_edge(210,953,weight=1)
G9.add_edge(211,1958,weight=1)
G9.add_edge(212,2716,weight=1)
G9.add_edge(212,2378,weight=1)
G9.add_edge(213,1960,weight=1)
G9.add_edge(213,3227,weight=1)
G9.add_edge(214,505,weight=1)
G9.add_edge(214,1451,weight=1)
G9.add_edge(215,1624,weight=1)
G9.add_edge(216,1216,weight=1)
G9.add_edge(216,992,weight=1)
G9.add_edge(217,1585,weight=1)
G9.add_edge(217,1801,weight=1)
G9.add_edge(218,2229,weight=1)
G9.add_edge(218,2951,weight=1)
G9.add_edge(219,2716,weight=1)
G9.add_edge(219,1597,weight=1)
G9.add_edge(219,1483,weight=1)
G9.add_edge(219,2969,weight=1)
G9.add_edge(220,1662,weight=1)
G9.add_edge(220,2570,weight=1)
G9.add_edge(220,3254,weight=1)
G9.add_edge(221,1662,weight=1)
G9.add_edge(222,1662,weight=1)
G9.add_edge(222,2570,weight=1)
G9.add_edge(222,3254,weight=1)
G9.add_edge(223,3109,weight=1)
G9.add_edge(223,2829,weight=1)
G9.add_edge(223,2931,weight=1)
G9.add_edge(223,2715,weight=1)
G9.add_edge(223,337,weight=1)
G9.add_edge(223,957,weight=1)
G9.add_edge(223,996,weight=1)
G9.add_edge(224,1173,weight=1)
G9.add_edge(224,1559,weight=1)
G9.add_edge(224,1179,weight=1)
G9.add_edge(224,1639,weight=1)
G9.add_edge(224,2422,weight=1)
G9.add_edge(225,934,weight=1)
G9.add_edge(225,334,weight=1)
G9.add_edge(225,1603,weight=1)
G9.add_edge(225,1625,weight=1)
G9.add_edge(225,2077,weight=1)
G9.add_edge(225,2442,weight=1)
G9.add_edge(226,2348,weight=1)
G9.add_edge(226,458,weight=1)
G9.add_edge(226,232,weight=1)
G9.add_edge(227,3185,weight=1)
G9.add_edge(227,3190,weight=1)
G9.add_edge(229,294,weight=1)
G9.add_edge(230,2314,weight=1)
G9.add_edge(231,3212,weight=1)
G9.add_edge(231,1893,weight=1)
G9.add_edge(232,1368,weight=1)
G9.add_edge(232,1928,weight=1)
G9.add_edge(232,458,weight=1)
G9.add_edge(232,1142,weight=1)
G9.add_edge(232,1668,weight=1)
G9.add_edge(233,248,weight=1)
G9.add_edge(234,1371,weight=1)
G9.add_edge(235,3240,weight=1)
G9.add_edge(235,580,weight=1)
G9.add_edge(235,581,weight=1)
G9.add_edge(235,582,weight=1)
G9.add_edge(235,1391,weight=1)
G9.add_edge(235,1669,weight=1)
G9.add_edge(236,2159,weight=1)
G9.add_edge(237,1601,weight=1)
G9.add_edge(238,375,weight=1)
G9.add_edge(238,2559,weight=1)
G9.add_edge(238,1168,weight=1)
G9.add_edge(239,2950,weight=1)
G9.add_edge(240,1874,weight=1)
G9.add_edge(241,2932,weight=1)
G9.add_edge(241,1605,weight=1)
G9.add_edge(242,960,weight=1)
G9.add_edge(243,2716,weight=1)
G9.add_edge(244,298,weight=1)
G9.add_edge(244,2251,weight=1)
G9.add_edge(244,1930,weight=1)
G9.add_edge(244,2846,weight=1)
G9.add_edge(244,398,weight=1)
G9.add_edge(245,455,weight=1)
G9.add_edge(245,1675,weight=1)
G9.add_edge(246,2373,weight=1)
G9.add_edge(246,510,weight=1)
G9.add_edge(246,513,weight=1)
G9.add_edge(247,2801,weight=1)
G9.add_edge(247,297,weight=1)
G9.add_edge(247,2395,weight=1)
G9.add_edge(248,334,weight=1)
G9.add_edge(249,3020,weight=1)
G9.add_edge(249,3243,weight=1)
G9.add_edge(249,2948,weight=1)
G9.add_edge(249,3244,weight=1)
G9.add_edge(250,573,weight=1)
G9.add_edge(250,574,weight=1)
G9.add_edge(251,2617,weight=1)
G9.add_edge(252,1691,weight=1)
G9.add_edge(252,1752,weight=1)
G9.add_edge(253,1716,weight=1)
G9.add_edge(254,2306,weight=1)
G9.add_edge(254,2468,weight=1)
G9.add_edge(255,264,weight=1)
G9.add_edge(255,265,weight=1)
G9.add_edge(256,2275,weight=1)
G9.add_edge(258,1097,weight=1)
G9.add_edge(259,1084,weight=1)
G9.add_edge(260,2588,weight=1)
G9.add_edge(261,2493,weight=1)
G9.add_edge(262,1003,weight=1)
G9.add_edge(263,314,weight=1)
G9.add_edge(263,315,weight=1)
G9.add_edge(264,265,weight=1)
G9.add_edge(265,1050,weight=1)
G9.add_edge(265,1051,weight=1)
G9.add_edge(265,1052,weight=1)
G9.add_edge(266,2051,weight=1)
G9.add_edge(267,2490,weight=1)
G9.add_edge(268,1055,weight=1)
G9.add_edge(269,1418,weight=1)
G9.add_edge(270,2530,weight=1)
G9.add_edge(270,3093,weight=1)
G9.add_edge(271,2859,weight=1)
G9.add_edge(271,1149,weight=1)
G9.add_edge(271,3139,weight=1)
G9.add_edge(272,1404,weight=1)
G9.add_edge(273,3024,weight=1)
G9.add_edge(274,2062,weight=1)
G9.add_edge(275,1864,weight=1)
G9.add_edge(275,1453,weight=1)
G9.add_edge(275,2036,weight=1)
G9.add_edge(275,1431,weight=1)
G9.add_edge(275,439,weight=1)
G9.add_edge(276,3159,weight=1)
G9.add_edge(276,1332,weight=1)
G9.add_edge(277,2717,weight=1)
G9.add_edge(278,2281,weight=1)
G9.add_edge(279,2770,weight=1)
G9.add_edge(279,2144,weight=1)
G9.add_edge(279,2840,weight=1)
G9.add_edge(279,2145,weight=1)
G9.add_edge(279,1745,weight=1)
G9.add_edge(280,1416,weight=1)
G9.add_edge(280,2120,weight=1)
G9.add_edge(281,1723,weight=1)
G9.add_edge(282,344,weight=1)
G9.add_edge(283,2619,weight=1)
G9.add_edge(283,2445,weight=1)
G9.add_edge(283,2491,weight=1)
G9.add_edge(284,2724,weight=1)
G9.add_edge(284,1001,weight=1)
G9.add_edge(285,2193,weight=1)
G9.add_edge(285,2486,weight=1)
G9.add_edge(286,2589,weight=1)
G9.add_edge(286,3166,weight=1)
G9.add_edge(287,2070,weight=1)
G9.add_edge(287,2030,weight=1)
G9.add_edge(288,2948,weight=1)
G9.add_edge(288,3244,weight=1)
G9.add_edge(289,3060,weight=1)
G9.add_edge(289,3062,weight=1)
G9.add_edge(289,3063,weight=1)
G9.add_edge(289,3064,weight=1)
G9.add_edge(289,3065,weight=1)
G9.add_edge(289,1351,weight=1)
G9.add_edge(289,2307,weight=1)
G9.add_edge(289,3222,weight=1)
G9.add_edge(289,3036,weight=1)
G9.add_edge(289,3102,weight=1)
G9.add_edge(289,2630,weight=1)
G9.add_edge(289,2715,weight=1)
G9.add_edge(290,2362,weight=1)
G9.add_edge(291,3060,weight=1)
G9.add_edge(291,3062,weight=1)
G9.add_edge(291,3063,weight=1)
G9.add_edge(291,3064,weight=1)
G9.add_edge(291,3065,weight=1)
G9.add_edge(291,1351,weight=1)
G9.add_edge(291,2307,weight=1)
G9.add_edge(291,3222,weight=1)
G9.add_edge(291,3036,weight=1)
G9.add_edge(291,3102,weight=1)
G9.add_edge(291,371,weight=1)
G9.add_edge(291,2630,weight=1)
G9.add_edge(291,2715,weight=1)
G9.add_edge(291,1298,weight=1)
G9.add_edge(291,1948,weight=1)
G9.add_edge(291,1949,weight=1)
G9.add_edge(292,1326,weight=1)
G9.add_edge(293,3181,weight=1)
G9.add_edge(293,334,weight=1)
G9.add_edge(294,1950,weight=1)
G9.add_edge(294,2551,weight=1)
G9.add_edge(295,2664,weight=1)
G9.add_edge(296,2986,weight=1)
G9.add_edge(298,2414,weight=1)
G9.add_edge(298,2416,weight=1)
G9.add_edge(298,2846,weight=1)
G9.add_edge(298,398,weight=1)
G9.add_edge(299,462,weight=1)
G9.add_edge(300,2622,weight=1)
G9.add_edge(300,1667,weight=1)
G9.add_edge(300,2320,weight=1)
G9.add_edge(301,2981,weight=1)
G9.add_edge(301,1721,weight=1)
G9.add_edge(302,2027,weight=1)
G9.add_edge(303,2284,weight=1)
G9.add_edge(304,305,weight=1)
G9.add_edge(305,2787,weight=1)
G9.add_edge(306,1969,weight=1)
G9.add_edge(306,689,weight=1)
G9.add_edge(306,1499,weight=1)
G9.add_edge(306,2835,weight=1)
G9.add_edge(306,1567,weight=1)
G9.add_edge(306,2040,weight=1)
G9.add_edge(306,308,weight=1)
G9.add_edge(307,870,weight=1)
G9.add_edge(307,2773,weight=1)
G9.add_edge(308,684,weight=1)
G9.add_edge(308,2089,weight=1)
G9.add_edge(308,1345,weight=1)
G9.add_edge(308,1499,weight=1)
G9.add_edge(308,711,weight=1)
G9.add_edge(308,2835,weight=1)
G9.add_edge(309,1367,weight=1)
G9.add_edge(310,505,weight=1)
G9.add_edge(310,2046,weight=1)
G9.add_edge(311,1650,weight=1)
G9.add_edge(312,2946,weight=1)
G9.add_edge(312,532,weight=1)
G9.add_edge(312,973,weight=1)
G9.add_edge(313,1461,weight=1)
G9.add_edge(313,3235,weight=1)
G9.add_edge(313,1672,weight=1)
G9.add_edge(316,2100,weight=1)
G9.add_edge(317,1089,weight=1)
G9.add_edge(318,1817,weight=1)
G9.add_edge(319,692,weight=1)
G9.add_edge(319,588,weight=1)
G9.add_edge(320,321,weight=1)
G9.add_edge(322,649,weight=1)
G9.add_edge(322,702,weight=1)
G9.add_edge(322,330,weight=1)
G9.add_edge(322,869,weight=1)
G9.add_edge(323,838,weight=1)
G9.add_edge(323,933,weight=1)
G9.add_edge(324,562,weight=1)
G9.add_edge(324,826,weight=1)
G9.add_edge(324,331,weight=1)
G9.add_edge(324,932,weight=1)
G9.add_edge(325,3241,weight=1)
G9.add_edge(326,1464,weight=1)
G9.add_edge(327,632,weight=1)
G9.add_edge(327,1907,weight=1)
G9.add_edge(327,328,weight=1)
G9.add_edge(327,483,weight=1)
G9.add_edge(327,880,weight=1)
G9.add_edge(328,329,weight=1)
G9.add_edge(329,632,weight=1)
G9.add_edge(330,702,weight=1)
G9.add_edge(331,823,weight=1)
G9.add_edge(331,562,weight=1)
G9.add_edge(331,826,weight=1)
G9.add_edge(331,907,weight=1)
G9.add_edge(332,2181,weight=1)
G9.add_edge(332,2358,weight=1)
G9.add_edge(332,365,weight=1)
G9.add_edge(332,858,weight=1)
G9.add_edge(333,838,weight=1)
G9.add_edge(333,933,weight=1)
G9.add_edge(334,2953,weight=1)
G9.add_edge(334,1588,weight=1)
G9.add_edge(334,350,weight=1)
G9.add_edge(334,567,weight=1)
G9.add_edge(334,2574,weight=1)
G9.add_edge(335,1766,weight=1)
G9.add_edge(335,1214,weight=1)
G9.add_edge(336,1766,weight=1)
G9.add_edge(337,2980,weight=1)
G9.add_edge(337,1628,weight=1)
G9.add_edge(338,1850,weight=1)
G9.add_edge(338,1992,weight=1)
G9.add_edge(338,2195,weight=1)
G9.add_edge(338,1816,weight=1)
G9.add_edge(338,1437,weight=1)
G9.add_edge(339,2891,weight=1)
G9.add_edge(339,2748,weight=1)
G9.add_edge(340,1891,weight=1)
G9.add_edge(340,1893,weight=1)
G9.add_edge(340,1399,weight=1)
G9.add_edge(341,1705,weight=1)
G9.add_edge(342,1347,weight=1)
G9.add_edge(342,2597,weight=1)
G9.add_edge(343,1720,weight=1)
G9.add_edge(344,2786,weight=1)
G9.add_edge(344,2595,weight=1)
G9.add_edge(344,2271,weight=1)
G9.add_edge(344,1965,weight=1)
G9.add_edge(345,2591,weight=1)
G9.add_edge(345,1111,weight=1)
G9.add_edge(345,2617,weight=1)
G9.add_edge(346,2587,weight=1)
G9.add_edge(347,602,weight=1)
G9.add_edge(347,348,weight=1)
G9.add_edge(347,870,weight=1)
G9.add_edge(348,869,weight=1)
G9.add_edge(348,873,weight=1)
G9.add_edge(349,1434,weight=1)
G9.add_edge(349,2567,weight=1)
G9.add_edge(349,550,weight=1)
G9.add_edge(350,389,weight=1)
G9.add_edge(350,2578,weight=1)
G9.add_edge(351,2985,weight=1)
G9.add_edge(351,1443,weight=1)
G9.add_edge(351,1758,weight=1)
G9.add_edge(352,2567,weight=1)
G9.add_edge(353,2849,weight=1)
G9.add_edge(355,2365,weight=1)
G9.add_edge(356,1542,weight=1)
G9.add_edge(356,491,weight=1)
G9.add_edge(356,1552,weight=1)
G9.add_edge(356,2619,weight=1)
G9.add_edge(356,1510,weight=1)
G9.add_edge(356,1359,weight=1)
G9.add_edge(356,1642,weight=1)
G9.add_edge(356,1512,weight=1)
G9.add_edge(356,2703,weight=1)
G9.add_edge(356,1513,weight=1)
G9.add_edge(357,1869,weight=1)
G9.add_edge(358,2683,weight=1)
G9.add_edge(358,2684,weight=1)
G9.add_edge(358,2685,weight=1)
G9.add_edge(359,2686,weight=1)
G9.add_edge(359,810,weight=1)
G9.add_edge(361,604,weight=1)
G9.add_edge(361,2737,weight=1)
G9.add_edge(362,722,weight=1)
G9.add_edge(362,724,weight=1)
G9.add_edge(362,1201,weight=1)
G9.add_edge(362,2578,weight=1)
G9.add_edge(363,1547,weight=1)
G9.add_edge(363,2686,weight=1)
G9.add_edge(363,2907,weight=1)
G9.add_edge(363,2306,weight=1)
G9.add_edge(363,1976,weight=1)
G9.add_edge(363,1510,weight=1)
G9.add_edge(364,451,weight=1)
G9.add_edge(364,2994,weight=1)
G9.add_edge(365,649,weight=1)
G9.add_edge(366,1570,weight=1)
G9.add_edge(367,1199,weight=1)
G9.add_edge(367,899,weight=1)
G9.add_edge(368,2862,weight=1)
G9.add_edge(368,2153,weight=1)
G9.add_edge(369,2904,weight=1)
G9.add_edge(370,656,weight=1)
G9.add_edge(370,762,weight=1)
G9.add_edge(371,3060,weight=1)
G9.add_edge(371,3062,weight=1)
G9.add_edge(371,2708,weight=1)
G9.add_edge(371,3063,weight=1)
G9.add_edge(371,3064,weight=1)
G9.add_edge(371,2070,weight=1)
G9.add_edge(371,2041,weight=1)
G9.add_edge(371,2517,weight=1)
G9.add_edge(371,2218,weight=1)
G9.add_edge(371,3036,weight=1)
G9.add_edge(371,3102,weight=1)
G9.add_edge(371,3066,weight=1)
G9.add_edge(371,2630,weight=1)
G9.add_edge(372,3098,weight=1)
G9.add_edge(372,1557,weight=1)
G9.add_edge(372,1572,weight=1)
G9.add_edge(374,2138,weight=1)
G9.add_edge(374,1458,weight=1)
G9.add_edge(374,1381,weight=1)
G9.add_edge(375,1934,weight=1)
G9.add_edge(375,3121,weight=1)
G9.add_edge(375,3123,weight=1)
G9.add_edge(376,490,weight=1)
G9.add_edge(376,2093,weight=1)
G9.add_edge(376,2417,weight=1)
G9.add_edge(377,1574,weight=1)
G9.add_edge(377,1310,weight=1)
G9.add_edge(378,2983,weight=1)
G9.add_edge(379,1359,weight=1)
G9.add_edge(380,2524,weight=1)
G9.add_edge(380,2547,weight=1)
G9.add_edge(381,952,weight=1)
G9.add_edge(382,1373,weight=1)
G9.add_edge(383,1373,weight=1)
G9.add_edge(384,3048,weight=1)
G9.add_edge(385,1305,weight=1)
G9.add_edge(386,2688,weight=1)
G9.add_edge(387,1583,weight=1)
G9.add_edge(387,3209,weight=1)
G9.add_edge(387,1994,weight=1)
G9.add_edge(388,2900,weight=1)
G9.add_edge(388,2548,weight=1)
G9.add_edge(388,2872,weight=1)
G9.add_edge(388,2873,weight=1)
G9.add_edge(389,2367,weight=1)
G9.add_edge(389,2371,weight=1)
G9.add_edge(389,2531,weight=1)
G9.add_edge(389,1680,weight=1)
G9.add_edge(389,1685,weight=1)
G9.add_edge(391,2841,weight=1)
G9.add_edge(391,2842,weight=1)
G9.add_edge(392,2496,weight=1)
G9.add_edge(392,3180,weight=1)
G9.add_edge(392,580,weight=1)
G9.add_edge(392,581,weight=1)
G9.add_edge(393,967,weight=1)
G9.add_edge(393,462,weight=1)
G9.add_edge(394,1360,weight=1)
G9.add_edge(394,1138,weight=1)
G9.add_edge(394,1143,weight=1)
G9.add_edge(395,1930,weight=1)
G9.add_edge(396,1541,weight=1)
G9.add_edge(396,1577,weight=1)
G9.add_edge(396,1186,weight=1)
G9.add_edge(396,454,weight=1)
G9.add_edge(396,2096,weight=1)
G9.add_edge(397,2561,weight=1)
G9.add_edge(397,1501,weight=1)
G9.add_edge(398,1812,weight=1)
G9.add_edge(398,1930,weight=1)
G9.add_edge(398,2846,weight=1)
G9.add_edge(399,1800,weight=1)
G9.add_edge(399,2881,weight=1)
G9.add_edge(399,3217,weight=1)
G9.add_edge(399,3091,weight=1)
G9.add_edge(400,490,weight=1)
G9.add_edge(400,1198,weight=1)
G9.add_edge(400,2093,weight=1)
G9.add_edge(400,1359,weight=1)
G9.add_edge(400,1220,weight=1)
G9.add_edge(400,1226,weight=1)
G9.add_edge(400,2573,weight=1)
G9.add_edge(401,1856,weight=1)
G9.add_edge(401,1859,weight=1)
G9.add_edge(402,3187,weight=1)
G9.add_edge(402,3251,weight=1)
G9.add_edge(403,3152,weight=1)
G9.add_edge(404,2586,weight=1)
G9.add_edge(404,1150,weight=1)
G9.add_edge(404,438,weight=1)
G9.add_edge(405,436,weight=1)
G9.add_edge(405,1498,weight=1)
G9.add_edge(406,410,weight=1)
G9.add_edge(406,1535,weight=1)
G9.add_edge(407,2612,weight=1)
G9.add_edge(408,1261,weight=1)
G9.add_edge(408,413,weight=1)
G9.add_edge(409,3013,weight=1)
G9.add_edge(409,2751,weight=1)
G9.add_edge(410,3187,weight=1)
G9.add_edge(410,3251,weight=1)
G9.add_edge(411,1744,weight=1)
G9.add_edge(412,2266,weight=1)
G9.add_edge(413,1261,weight=1)
G9.add_edge(414,1413,weight=1)
G9.add_edge(415,1532,weight=1)
G9.add_edge(416,1836,weight=1)
G9.add_edge(417,2020,weight=1)
G9.add_edge(418,2196,weight=1)
G9.add_edge(418,2329,weight=1)
G9.add_edge(418,2816,weight=1)
G9.add_edge(419,1001,weight=1)
G9.add_edge(419,2850,weight=1)
G9.add_edge(420,2111,weight=1)
G9.add_edge(421,2341,weight=1)
G9.add_edge(422,1719,weight=1)
G9.add_edge(423,2674,weight=1)
G9.add_edge(424,1080,weight=1)
G9.add_edge(424,589,weight=1)
G9.add_edge(425,2277,weight=1)
G9.add_edge(426,2601,weight=1)
G9.add_edge(426,2610,weight=1)
G9.add_edge(427,3207,weight=1)
G9.add_edge(428,3123,weight=1)
G9.add_edge(429,1328,weight=1)
G9.add_edge(429,550,weight=1)
G9.add_edge(429,1330,weight=1)
G9.add_edge(430,2614,weight=1)
G9.add_edge(431,2276,weight=1)
G9.add_edge(432,2016,weight=1)
G9.add_edge(434,569,weight=1)
G9.add_edge(434,2035,weight=1)
G9.add_edge(435,1830,weight=1)
G9.add_edge(435,3123,weight=1)
G9.add_edge(436,1498,weight=1)
G9.add_edge(437,1516,weight=1)
G9.add_edge(438,703,weight=1)
G9.add_edge(439,1717,weight=1)
G9.add_edge(440,1432,weight=1)
G9.add_edge(441,2274,weight=1)
G9.add_edge(442,1783,weight=1)
G9.add_edge(443,609,weight=1)
G9.add_edge(443,669,weight=1)
G9.add_edge(443,446,weight=1)
G9.add_edge(443,447,weight=1)
G9.add_edge(443,858,weight=1)
G9.add_edge(443,890,weight=1)
G9.add_edge(445,2359,weight=1)
G9.add_edge(446,639,weight=1)
G9.add_edge(446,643,weight=1)
G9.add_edge(446,661,weight=1)
G9.add_edge(446,707,weight=1)
G9.add_edge(446,447,weight=1)
G9.add_edge(446,2303,weight=1)
G9.add_edge(446,1196,weight=1)
G9.add_edge(446,2364,weight=1)
G9.add_edge(447,610,weight=1)
G9.add_edge(447,642,weight=1)
G9.add_edge(447,687,weight=1)
G9.add_edge(447,2954,weight=1)
G9.add_edge(447,2303,weight=1)
G9.add_edge(447,858,weight=1)
G9.add_edge(447,931,weight=1)
G9.add_edge(448,687,weight=1)
G9.add_edge(448,2354,weight=1)
G9.add_edge(449,1561,weight=1)
G9.add_edge(450,1975,weight=1)
G9.add_edge(450,3222,weight=1)
G9.add_edge(451,1266,weight=1)
G9.add_edge(451,2865,weight=1)
G9.add_edge(451,1267,weight=1)
G9.add_edge(451,2868,weight=1)
G9.add_edge(453,768,weight=1)
G9.add_edge(453,483,weight=1)
G9.add_edge(454,1577,weight=1)
G9.add_edge(454,1587,weight=1)
G9.add_edge(454,2096,weight=1)
G9.add_edge(455,1724,weight=1)
G9.add_edge(456,2754,weight=1)
G9.add_edge(456,2092,weight=1)
G9.add_edge(456,1310,weight=1)
G9.add_edge(456,1065,weight=1)
G9.add_edge(457,2430,weight=1)
G9.add_edge(457,2006,weight=1)
G9.add_edge(459,2176,weight=1)
G9.add_edge(460,1609,weight=1)
G9.add_edge(460,1611,weight=1)
G9.add_edge(460,1698,weight=1)
G9.add_edge(461,1206,weight=1)
G9.add_edge(461,1210,weight=1)
G9.add_edge(461,1630,weight=1)
G9.add_edge(462,966,weight=1)
G9.add_edge(462,967,weight=1)
G9.add_edge(463,2421,weight=1)
G9.add_edge(463,1254,weight=1)
G9.add_edge(463,2644,weight=1)
G9.add_edge(463,1487,weight=1)
G9.add_edge(465,2571,weight=1)
G9.add_edge(465,2705,weight=1)
G9.add_edge(466,1506,weight=1)
G9.add_edge(467,1698,weight=1)
G9.add_edge(468,2583,weight=1)
G9.add_edge(469,1348,weight=1)
G9.add_edge(469,2600,weight=1)
G9.add_edge(469,1063,weight=1)
G9.add_edge(470,1348,weight=1)
G9.add_edge(470,1979,weight=1)
G9.add_edge(470,2454,weight=1)
G9.add_edge(470,1861,weight=1)
G9.add_edge(471,515,weight=1)
G9.add_edge(472,1504,weight=1)
G9.add_edge(472,2330,weight=1)
G9.add_edge(473,2608,weight=1)
G9.add_edge(474,2596,weight=1)
G9.add_edge(475,1169,weight=1)
G9.add_edge(475,2994,weight=1)
G9.add_edge(476,1233,weight=1)
G9.add_edge(477,1062,weight=1)
G9.add_edge(478,1936,weight=1)
G9.add_edge(479,480,weight=1)
G9.add_edge(479,481,weight=1)
G9.add_edge(479,482,weight=1)
G9.add_edge(479,778,weight=1)
G9.add_edge(479,502,weight=1)
G9.add_edge(479,2769,weight=1)
G9.add_edge(480,482,weight=1)
G9.add_edge(480,496,weight=1)
G9.add_edge(480,498,weight=1)
G9.add_edge(480,802,weight=1)
G9.add_edge(480,502,weight=1)
G9.add_edge(481,482,weight=1)
G9.add_edge(481,496,weight=1)
G9.add_edge(481,498,weight=1)
G9.add_edge(481,802,weight=1)
G9.add_edge(481,502,weight=1)
G9.add_edge(481,2769,weight=1)
G9.add_edge(482,498,weight=1)
G9.add_edge(482,802,weight=1)
G9.add_edge(482,2769,weight=1)
G9.add_edge(483,616,weight=1)
G9.add_edge(483,686,weight=1)
G9.add_edge(483,770,weight=1)
G9.add_edge(483,908,weight=1)
G9.add_edge(483,913,weight=1)
G9.add_edge(484,2907,weight=1)
G9.add_edge(484,2039,weight=1)
G9.add_edge(484,2507,weight=1)
G9.add_edge(484,3075,weight=1)
G9.add_edge(484,1580,weight=1)
G9.add_edge(486,1132,weight=1)
G9.add_edge(487,510,weight=1)
G9.add_edge(487,2725,weight=1)
G9.add_edge(488,1559,weight=1)
G9.add_edge(488,2061,weight=1)
G9.add_edge(489,2332,weight=1)
G9.add_edge(490,2092,weight=1)
G9.add_edge(490,1562,weight=1)
G9.add_edge(490,2230,weight=1)
G9.add_edge(490,1207,weight=1)
G9.add_edge(490,1630,weight=1)
G9.add_edge(490,2417,weight=1)
G9.add_edge(490,1664,weight=1)
G9.add_edge(490,1675,weight=1)
G9.add_edge(491,558,weight=1)
G9.add_edge(491,2907,weight=1)
G9.add_edge(492,1869,weight=1)
G9.add_edge(492,2091,weight=1)
G9.add_edge(492,2930,weight=1)
G9.add_edge(492,2092,weight=1)
G9.add_edge(492,1975,weight=1)
G9.add_edge(492,1198,weight=1)
G9.add_edge(492,1574,weight=1)
G9.add_edge(492,2962,weight=1)
G9.add_edge(492,1207,weight=1)
G9.add_edge(492,1630,weight=1)
G9.add_edge(492,1220,weight=1)
G9.add_edge(492,1664,weight=1)
G9.add_edge(493,684,weight=1)
G9.add_edge(493,711,weight=1)
G9.add_edge(493,1558,weight=1)
G9.add_edge(493,1560,weight=1)
G9.add_edge(493,2818,weight=1)
G9.add_edge(494,1283,weight=1)
G9.add_edge(494,2498,weight=1)
G9.add_edge(494,2091,weight=1)
G9.add_edge(494,1284,weight=1)
G9.add_edge(494,2201,weight=1)
G9.add_edge(494,2366,weight=1)
G9.add_edge(494,2129,weight=1)
G9.add_edge(495,548,weight=1)
G9.add_edge(495,732,weight=1)
G9.add_edge(496,1339,weight=1)
G9.add_edge(496,662,weight=1)
G9.add_edge(496,825,weight=1)
G9.add_edge(496,502,weight=1)
G9.add_edge(497,1467,weight=1)
G9.add_edge(497,1193,weight=1)
G9.add_edge(497,2222,weight=1)
G9.add_edge(498,631,weight=1)
G9.add_edge(498,802,weight=1)
G9.add_edge(498,1514,weight=1)
G9.add_edge(498,502,weight=1)
G9.add_edge(498,503,weight=1)
G9.add_edge(499,1200,weight=1)
G9.add_edge(499,1220,weight=1)
G9.add_edge(499,2438,weight=1)
G9.add_edge(499,3230,weight=1)
G9.add_edge(500,2088,weight=1)
G9.add_edge(500,558,weight=1)
G9.add_edge(500,811,weight=1)
G9.add_edge(500,3222,weight=1)
G9.add_edge(500,1574,weight=1)
G9.add_edge(500,1927,weight=1)
G9.add_edge(500,2438,weight=1)
G9.add_edge(500,2000,weight=1)
G9.add_edge(500,2972,weight=1)
G9.add_edge(501,747,weight=1)
G9.add_edge(501,813,weight=1)
G9.add_edge(502,2347,weight=1)
G9.add_edge(502,776,weight=1)
G9.add_edge(502,802,weight=1)
G9.add_edge(502,2730,weight=1)
G9.add_edge(503,1192,weight=1)
G9.add_edge(503,802,weight=1)
G9.add_edge(504,2528,weight=1)
G9.add_edge(504,1884,weight=1)
G9.add_edge(504,1383,weight=1)
G9.add_edge(504,1528,weight=1)
G9.add_edge(504,2560,weight=1)
G9.add_edge(505,1451,weight=1)
G9.add_edge(506,2154,weight=1)
G9.add_edge(506,3103,weight=1)
G9.add_edge(506,1212,weight=1)
G9.add_edge(506,1384,weight=1)
G9.add_edge(507,1609,weight=1)
G9.add_edge(507,1611,weight=1)
G9.add_edge(508,2096,weight=1)
G9.add_edge(509,1465,weight=1)
G9.add_edge(509,1466,weight=1)
G9.add_edge(509,1467,weight=1)
G9.add_edge(509,1468,weight=1)
G9.add_edge(509,1469,weight=1)
G9.add_edge(509,1470,weight=1)
G9.add_edge(509,3178,weight=1)
G9.add_edge(509,3011,weight=1)
G9.add_edge(509,2548,weight=1)
G9.add_edge(510,1599,weight=1)
G9.add_edge(510,2388,weight=1)
G9.add_edge(510,2725,weight=1)
G9.add_edge(510,513,weight=1)
G9.add_edge(510,2951,weight=1)
G9.add_edge(511,2804,weight=1)
G9.add_edge(511,2188,weight=1)
G9.add_edge(512,3082,weight=1)
G9.add_edge(512,2546,weight=1)
G9.add_edge(513,2775,weight=1)
G9.add_edge(513,2634,weight=1)
G9.add_edge(514,954,weight=1)
G9.add_edge(516,1047,weight=1)
G9.add_edge(516,1109,weight=1)
G9.add_edge(517,2957,weight=1)
G9.add_edge(517,3012,weight=1)
G9.add_edge(519,1413,weight=1)
G9.add_edge(520,1321,weight=1)
G9.add_edge(521,1080,weight=1)
G9.add_edge(522,1073,weight=1)
G9.add_edge(523,2957,weight=1)
G9.add_edge(523,3012,weight=1)
G9.add_edge(524,1688,weight=1)
G9.add_edge(524,1695,weight=1)
G9.add_edge(524,3245,weight=1)
G9.add_edge(524,2733,weight=1)
G9.add_edge(524,1413,weight=1)
G9.add_edge(525,2793,weight=1)
G9.add_edge(526,1022,weight=1)
G9.add_edge(527,3178,weight=1)
G9.add_edge(527,1845,weight=1)
G9.add_edge(528,611,weight=1)
G9.add_edge(528,686,weight=1)
G9.add_edge(528,933,weight=1)
G9.add_edge(529,2946,weight=1)
G9.add_edge(529,1764,weight=1)
G9.add_edge(529,532,weight=1)
G9.add_edge(529,549,weight=1)
G9.add_edge(529,973,weight=1)
G9.add_edge(529,983,weight=1)
G9.add_edge(530,2946,weight=1)
G9.add_edge(530,1764,weight=1)
G9.add_edge(530,973,weight=1)
G9.add_edge(531,2836,weight=1)
G9.add_edge(531,1678,weight=1)
G9.add_edge(532,534,weight=1)
G9.add_edge(533,2199,weight=1)
G9.add_edge(535,1308,weight=1)
G9.add_edge(535,1399,weight=1)
G9.add_edge(536,1224,weight=1)
G9.add_edge(537,1230,weight=1)
G9.add_edge(538,2217,weight=1)
G9.add_edge(538,2448,weight=1)
G9.add_edge(538,1103,weight=1)
G9.add_edge(539,547,weight=1)
G9.add_edge(540,2130,weight=1)
G9.add_edge(541,1048,weight=1)
G9.add_edge(542,1008,weight=1)
G9.add_edge(543,1240,weight=1)
G9.add_edge(544,1046,weight=1)
G9.add_edge(545,722,weight=1)
G9.add_edge(545,724,weight=1)
G9.add_edge(545,546,weight=1)
G9.add_edge(547,723,weight=1)
G9.add_edge(547,1080,weight=1)
G9.add_edge(548,732,weight=1)
G9.add_edge(548,814,weight=1)
G9.add_edge(548,875,weight=1)
G9.add_edge(548,893,weight=1)
G9.add_edge(548,956,weight=1)
G9.add_edge(549,949,weight=1)
G9.add_edge(549,962,weight=1)
G9.add_edge(551,956,weight=1)
G9.add_edge(551,1099,weight=1)
G9.add_edge(552,2144,weight=1)
G9.add_edge(552,1282,weight=1)
G9.add_edge(552,2145,weight=1)
G9.add_edge(553,555,weight=1)
G9.add_edge(553,556,weight=1)
G9.add_edge(554,2301,weight=1)
G9.add_edge(554,720,weight=1)
G9.add_edge(554,798,weight=1)
G9.add_edge(555,1122,weight=1)
G9.add_edge(555,790,weight=1)
G9.add_edge(555,800,weight=1)
G9.add_edge(555,806,weight=1)
G9.add_edge(555,828,weight=1)
G9.add_edge(555,899,weight=1)
G9.add_edge(555,919,weight=1)
G9.add_edge(555,924,weight=1)
G9.add_edge(556,1191,weight=1)
G9.add_edge(556,890,weight=1)
G9.add_edge(556,919,weight=1)
G9.add_edge(557,2348,weight=1)
G9.add_edge(557,1357,weight=1)
G9.add_edge(558,623,weight=1)
G9.add_edge(558,2088,weight=1)
G9.add_edge(558,2090,weight=1)
G9.add_edge(558,1127,weight=1)
G9.add_edge(558,2092,weight=1)
G9.add_edge(558,1352,weight=1)
G9.add_edge(558,2093,weight=1)
G9.add_edge(558,1509,weight=1)
G9.add_edge(559,3241,weight=1)
G9.add_edge(560,784,weight=1)
G9.add_edge(560,786,weight=1)
G9.add_edge(560,790,weight=1)
G9.add_edge(561,716,weight=1)
G9.add_edge(561,909,weight=1)
G9.add_edge(562,823,weight=1)
G9.add_edge(562,870,weight=1)
G9.add_edge(562,903,weight=1)
G9.add_edge(563,1472,weight=1)
G9.add_edge(563,1477,weight=1)
G9.add_edge(563,1478,weight=1)
G9.add_edge(564,1517,weight=1)
G9.add_edge(564,2843,weight=1)
G9.add_edge(564,613,weight=1)
G9.add_edge(564,685,weight=1)
G9.add_edge(564,1518,weight=1)
G9.add_edge(564,703,weight=1)
G9.add_edge(564,2302,weight=1)
G9.add_edge(564,739,weight=1)
G9.add_edge(564,753,weight=1)
G9.add_edge(564,791,weight=1)
G9.add_edge(564,881,weight=1)
G9.add_edge(564,894,weight=1)
G9.add_edge(565,2240,weight=1)
G9.add_edge(565,1219,weight=1)
G9.add_edge(566,2577,weight=1)
G9.add_edge(568,979,weight=1)
G9.add_edge(570,576,weight=1)
G9.add_edge(571,2704,weight=1)
G9.add_edge(572,577,weight=1)
G9.add_edge(575,1032,weight=1)
G9.add_edge(575,1320,weight=1)
G9.add_edge(575,1092,weight=1)
G9.add_edge(575,1109,weight=1)
G9.add_edge(578,579,weight=1)
G9.add_edge(580,3180,weight=1)
G9.add_edge(580,2524,weight=1)
G9.add_edge(580,2380,weight=1)
G9.add_edge(580,582,weight=1)
G9.add_edge(580,2156,weight=1)
G9.add_edge(581,3180,weight=1)
G9.add_edge(581,2524,weight=1)
G9.add_edge(581,2380,weight=1)
G9.add_edge(581,582,weight=1)
G9.add_edge(581,2156,weight=1)
G9.add_edge(582,3180,weight=1)
G9.add_edge(582,2524,weight=1)
G9.add_edge(582,2434,weight=1)
G9.add_edge(583,1160,weight=1)
G9.add_edge(584,1165,weight=1)
G9.add_edge(585,2767,weight=1)
G9.add_edge(586,735,weight=1)
G9.add_edge(587,921,weight=1)
G9.add_edge(588,692,weight=1)
G9.add_edge(590,1765,weight=1)
G9.add_edge(591,611,weight=1)
G9.add_edge(591,686,weight=1)
G9.add_edge(591,1907,weight=1)
G9.add_edge(592,1907,weight=1)
G9.add_edge(593,2309,weight=1)
G9.add_edge(593,1487,weight=1)
G9.add_edge(594,596,weight=1)
G9.add_edge(595,2286,weight=1)
G9.add_edge(597,912,weight=1)
G9.add_edge(598,603,weight=1)
G9.add_edge(598,1199,weight=1)
G9.add_edge(599,645,weight=1)
G9.add_edge(599,682,weight=1)
G9.add_edge(599,3002,weight=1)
G9.add_edge(599,916,weight=1)
G9.add_edge(600,716,weight=1)
G9.add_edge(600,905,weight=1)
G9.add_edge(601,1907,weight=1)
G9.add_edge(601,1199,weight=1)
G9.add_edge(601,861,weight=1)
G9.add_edge(602,870,weight=1)
G9.add_edge(603,1199,weight=1)
G9.add_edge(605,682,weight=1)
G9.add_edge(606,3226,weight=1)
G9.add_edge(606,747,weight=1)
G9.add_edge(606,795,weight=1)
G9.add_edge(606,854,weight=1)
G9.add_edge(607,795,weight=1)
G9.add_edge(608,1907,weight=1)
G9.add_edge(608,933,weight=1)
G9.add_edge(609,644,weight=1)
G9.add_edge(609,653,weight=1)
G9.add_edge(609,727,weight=1)
G9.add_edge(609,740,weight=1)
G9.add_edge(609,784,weight=1)
G9.add_edge(609,788,weight=1)
G9.add_edge(609,801,weight=1)
G9.add_edge(609,870,weight=1)
G9.add_edge(609,872,weight=1)
G9.add_edge(610,2132,weight=1)
G9.add_edge(611,1907,weight=1)
G9.add_edge(612,870,weight=1)
G9.add_edge(613,791,weight=1)
G9.add_edge(613,809,weight=1)
G9.add_edge(614,824,weight=1)
G9.add_edge(614,2958,weight=1)
G9.add_edge(615,870,weight=1)
G9.add_edge(617,2659,weight=1)
G9.add_edge(617,924,weight=1)
G9.add_edge(618,790,weight=1)
G9.add_edge(618,800,weight=1)
G9.add_edge(618,829,weight=1)
G9.add_edge(619,657,weight=1)
G9.add_edge(619,902,weight=1)
G9.add_edge(620,807,weight=1)
G9.add_edge(621,870,weight=1)
G9.add_edge(622,828,weight=1)
G9.add_edge(622,924,weight=1)
G9.add_edge(623,1855,weight=1)
G9.add_edge(623,2088,weight=1)
G9.add_edge(623,2619,weight=1)
G9.add_edge(623,2835,weight=1)
G9.add_edge(623,1523,weight=1)
G9.add_edge(623,772,weight=1)
G9.add_edge(623,2168,weight=1)
G9.add_edge(623,840,weight=1)
G9.add_edge(623,3100,weight=1)
G9.add_edge(623,2783,weight=1)
G9.add_edge(624,763,weight=1)
G9.add_edge(624,764,weight=1)
G9.add_edge(624,844,weight=1)
G9.add_edge(625,735,weight=1)
G9.add_edge(625,837,weight=1)
G9.add_edge(626,762,weight=1)
G9.add_edge(627,636,weight=1)
G9.add_edge(627,3259,weight=1)
G9.add_edge(627,641,weight=1)
G9.add_edge(628,699,weight=1)
G9.add_edge(628,871,weight=1)
G9.add_edge(629,771,weight=1)
G9.add_edge(633,726,weight=1)
G9.add_edge(634,714,weight=1)
G9.add_edge(634,852,weight=1)
G9.add_edge(635,795,weight=1)
G9.add_edge(635,854,weight=1)
G9.add_edge(635,856,weight=1)
G9.add_edge(635,880,weight=1)
G9.add_edge(636,774,weight=1)
G9.add_edge(637,699,weight=1)
G9.add_edge(637,1363,weight=1)
G9.add_edge(638,795,weight=1)
G9.add_edge(638,851,weight=1)
G9.add_edge(638,854,weight=1)
G9.add_edge(639,700,weight=1)
G9.add_edge(640,743,weight=1)
G9.add_edge(640,870,weight=1)
G9.add_edge(641,774,weight=1)
G9.add_edge(642,931,weight=1)
G9.add_edge(643,738,weight=1)
G9.add_edge(643,745,weight=1)
G9.add_edge(644,669,weight=1)
G9.add_edge(645,766,weight=1)
G9.add_edge(645,770,weight=1)
G9.add_edge(645,916,weight=1)
G9.add_edge(646,777,weight=1)
G9.add_edge(646,830,weight=1)
G9.add_edge(646,874,weight=1)
G9.add_edge(646,920,weight=1)
G9.add_edge(648,878,weight=1)
G9.add_edge(649,690,weight=1)
G9.add_edge(649,1346,weight=1)
G9.add_edge(649,756,weight=1)
G9.add_edge(649,869,weight=1)
G9.add_edge(650,748,weight=1)
G9.add_edge(650,857,weight=1)
G9.add_edge(651,726,weight=1)
G9.add_edge(651,749,weight=1)
G9.add_edge(652,2708,weight=1)
G9.add_edge(652,3063,weight=1)
G9.add_edge(652,2907,weight=1)
G9.add_edge(652,1772,weight=1)
G9.add_edge(652,3102,weight=1)
G9.add_edge(652,3066,weight=1)
G9.add_edge(653,669,weight=1)
G9.add_edge(654,870,weight=1)
G9.add_edge(655,870,weight=1)
G9.add_edge(655,914,weight=1)
G9.add_edge(656,760,weight=1)
G9.add_edge(656,854,weight=1)
G9.add_edge(658,781,weight=1)
G9.add_edge(659,833,weight=1)
G9.add_edge(659,834,weight=1)
G9.add_edge(659,835,weight=1)
G9.add_edge(660,796,weight=1)
G9.add_edge(660,888,weight=1)
G9.add_edge(662,670,weight=1)
G9.add_edge(662,802,weight=1)
G9.add_edge(663,706,weight=1)
G9.add_edge(663,793,weight=1)
G9.add_edge(664,800,weight=1)
G9.add_edge(665,799,weight=1)
G9.add_edge(665,877,weight=1)
G9.add_edge(666,2307,weight=1)
G9.add_edge(667,790,weight=1)
G9.add_edge(667,924,weight=1)
G9.add_edge(668,788,weight=1)
G9.add_edge(669,727,weight=1)
G9.add_edge(669,740,weight=1)
G9.add_edge(669,784,weight=1)
G9.add_edge(669,788,weight=1)
G9.add_edge(670,746,weight=1)
G9.add_edge(670,795,weight=1)
G9.add_edge(670,1570,weight=1)
G9.add_edge(671,715,weight=1)
G9.add_edge(671,771,weight=1)
G9.add_edge(671,783,weight=1)
G9.add_edge(671,799,weight=1)
G9.add_edge(671,846,weight=1)
G9.add_edge(671,897,weight=1)
G9.add_edge(672,726,weight=1)
G9.add_edge(672,790,weight=1)
G9.add_edge(672,800,weight=1)
G9.add_edge(673,858,weight=1)
G9.add_edge(673,870,weight=1)
G9.add_edge(673,919,weight=1)
G9.add_edge(674,820,weight=1)
G9.add_edge(674,932,weight=1)
G9.add_edge(675,2363,weight=1)
G9.add_edge(675,898,weight=1)
G9.add_edge(676,717,weight=1)
G9.add_edge(677,2925,weight=1)
G9.add_edge(679,799,weight=1)
G9.add_edge(680,735,weight=1)
G9.add_edge(681,2659,weight=1)
G9.add_edge(683,821,weight=1)
G9.add_edge(683,855,weight=1)
G9.add_edge(684,1969,weight=1)
G9.add_edge(684,1545,weight=1)
G9.add_edge(684,2903,weight=1)
G9.add_edge(684,1851,weight=1)
G9.add_edge(684,2913,weight=1)
G9.add_edge(684,1567,weight=1)
G9.add_edge(686,1907,weight=1)
G9.add_edge(686,795,weight=1)
G9.add_edge(686,812,weight=1)
G9.add_edge(687,771,weight=1)
G9.add_edge(687,858,weight=1)
G9.add_edge(688,790,weight=1)
G9.add_edge(689,1969,weight=1)
G9.add_edge(689,735,weight=1)
G9.add_edge(689,1567,weight=1)
G9.add_edge(689,2961,weight=1)
G9.add_edge(690,3178,weight=1)
G9.add_edge(691,870,weight=1)
G9.add_edge(692,2575,weight=1)
G9.add_edge(693,696,weight=1)
G9.add_edge(694,696,weight=1)
G9.add_edge(694,795,weight=1)
G9.add_edge(694,802,weight=1)
G9.add_edge(694,873,weight=1)
G9.add_edge(695,696,weight=1)
G9.add_edge(695,795,weight=1)
G9.add_edge(695,802,weight=1)
G9.add_edge(697,726,weight=1)
G9.add_edge(697,785,weight=1)
G9.add_edge(698,3221,weight=1)
G9.add_edge(698,844,weight=1)
G9.add_edge(699,751,weight=1)
G9.add_edge(699,752,weight=1)
G9.add_edge(699,753,weight=1)
G9.add_edge(700,800,weight=1)
G9.add_edge(701,718,weight=1)
G9.add_edge(701,831,weight=1)
G9.add_edge(703,1371,weight=1)
G9.add_edge(703,2085,weight=1)
G9.add_edge(704,1043,weight=1)
G9.add_edge(705,876,weight=1)
G9.add_edge(706,817,weight=1)
G9.add_edge(707,1348,weight=1)
G9.add_edge(707,2600,weight=1)
G9.add_edge(708,1907,weight=1)
G9.add_edge(709,1907,weight=1)
G9.add_edge(710,2359,weight=1)
G9.add_edge(710,798,weight=1)
G9.add_edge(711,1969,weight=1)
G9.add_edge(711,1545,weight=1)
G9.add_edge(711,2903,weight=1)
G9.add_edge(711,1851,weight=1)
G9.add_edge(711,1567,weight=1)
G9.add_edge(711,2961,weight=1)
G9.add_edge(712,2059,weight=1)
G9.add_edge(712,2088,weight=1)
G9.add_edge(712,2929,weight=1)
G9.add_edge(712,1560,weight=1)
G9.add_edge(712,1030,weight=1)
G9.add_edge(712,2475,weight=1)
G9.add_edge(712,2942,weight=1)
G9.add_edge(713,809,weight=1)
G9.add_edge(713,907,weight=1)
G9.add_edge(714,779,weight=1)
G9.add_edge(714,780,weight=1)
G9.add_edge(714,810,weight=1)
G9.add_edge(714,1202,weight=1)
G9.add_edge(715,771,weight=1)
G9.add_edge(715,897,weight=1)
G9.add_edge(716,733,weight=1)
G9.add_edge(716,797,weight=1)
G9.add_edge(717,769,weight=1)
G9.add_edge(717,808,weight=1)
G9.add_edge(718,1121,weight=1)
G9.add_edge(718,2181,weight=1)
G9.add_edge(718,769,weight=1)
G9.add_edge(718,771,weight=1)
G9.add_edge(718,792,weight=1)
G9.add_edge(718,831,weight=1)
G9.add_edge(718,900,weight=1)
G9.add_edge(718,917,weight=1)
G9.add_edge(719,748,weight=1)
G9.add_edge(719,870,weight=1)
G9.add_edge(719,1883,weight=1)
G9.add_edge(719,2737,weight=1)
G9.add_edge(721,906,weight=1)
G9.add_edge(722,723,weight=1)
G9.add_edge(722,724,weight=1)
G9.add_edge(722,1201,weight=1)
G9.add_edge(724,1201,weight=1)
G9.add_edge(724,952,weight=1)
G9.add_edge(725,2757,weight=1)
G9.add_edge(725,870,weight=1)
G9.add_edge(726,727,weight=1)
G9.add_edge(726,767,weight=1)
G9.add_edge(726,785,weight=1)
G9.add_edge(726,918,weight=1)
G9.add_edge(727,790,weight=1)
G9.add_edge(728,858,weight=1)
G9.add_edge(729,2303,weight=1)
G9.add_edge(729,852,weight=1)
G9.add_edge(729,858,weight=1)
G9.add_edge(730,731,weight=1)
G9.add_edge(731,1915,weight=1)
G9.add_edge(732,893,weight=1)
G9.add_edge(733,748,weight=1)
G9.add_edge(734,870,weight=1)
G9.add_edge(735,1172,weight=1)
G9.add_edge(735,915,weight=1)
G9.add_edge(736,852,weight=1)
G9.add_edge(737,857,weight=1)
G9.add_edge(738,1464,weight=1)
G9.add_edge(738,858,weight=1)
G9.add_edge(741,870,weight=1)
G9.add_edge(742,743,weight=1)
G9.add_edge(744,1195,weight=1)
G9.add_edge(744,890,weight=1)
G9.add_edge(744,919,weight=1)
G9.add_edge(745,777,weight=1)
G9.add_edge(748,819,weight=1)
G9.add_edge(748,868,weight=1)
G9.add_edge(748,870,weight=1)
G9.add_edge(748,1883,weight=1)
G9.add_edge(748,902,weight=1)
G9.add_edge(749,870,weight=1)
G9.add_edge(749,871,weight=1)
G9.add_edge(753,2756,weight=1)
G9.add_edge(753,2758,weight=1)
G9.add_edge(753,1350,weight=1)
G9.add_edge(754,1193,weight=1)
G9.add_edge(754,1194,weight=1)
G9.add_edge(754,1791,weight=1)
G9.add_edge(754,871,weight=1)
G9.add_edge(755,854,weight=1)
G9.add_edge(755,870,weight=1)
G9.add_edge(756,870,weight=1)
G9.add_edge(757,870,weight=1)
G9.add_edge(758,870,weight=1)
G9.add_edge(759,870,weight=1)
G9.add_edge(760,761,weight=1)
G9.add_edge(760,852,weight=1)
G9.add_edge(762,1464,weight=1)
G9.add_edge(762,870,weight=1)
G9.add_edge(763,764,weight=1)
G9.add_edge(763,844,weight=1)
G9.add_edge(764,924,weight=1)
G9.add_edge(765,841,weight=1)
G9.add_edge(765,918,weight=1)
G9.add_edge(767,1195,weight=1)
G9.add_edge(767,800,weight=1)
G9.add_edge(768,2564,weight=1)
G9.add_edge(768,2492,weight=1)
G9.add_edge(769,830,weight=1)
G9.add_edge(769,844,weight=1)
G9.add_edge(770,870,weight=1)
G9.add_edge(773,850,weight=1)
G9.add_edge(774,3259,weight=1)
G9.add_edge(775,776,weight=1)
G9.add_edge(775,870,weight=1)
G9.add_edge(776,2213,weight=1)
G9.add_edge(776,870,weight=1)
G9.add_edge(778,802,weight=1)
G9.add_edge(779,2500,weight=1)
G9.add_edge(779,2770,weight=1)
G9.add_edge(779,1870,weight=1)
G9.add_edge(779,2522,weight=1)
G9.add_edge(780,1970,weight=1)
G9.add_edge(781,872,weight=1)
G9.add_edge(782,908,weight=1)
G9.add_edge(784,786,weight=1)
G9.add_edge(784,787,weight=1)
G9.add_edge(785,921,weight=1)
G9.add_edge(786,787,weight=1)
G9.add_edge(786,788,weight=1)
G9.add_edge(787,788,weight=1)
G9.add_edge(789,1346,weight=1)
G9.add_edge(789,1194,weight=1)
G9.add_edge(789,1791,weight=1)
G9.add_edge(790,1195,weight=1)
G9.add_edge(790,829,weight=1)
G9.add_edge(790,919,weight=1)
G9.add_edge(790,920,weight=1)
G9.add_edge(790,925,weight=1)
G9.add_edge(790,928,weight=1)
G9.add_edge(790,932,weight=1)
G9.add_edge(793,1342,weight=1)
G9.add_edge(793,795,weight=1)
G9.add_edge(793,817,weight=1)
G9.add_edge(794,924,weight=1)
G9.add_edge(795,1339,weight=1)
G9.add_edge(795,1346,weight=1)
G9.add_edge(795,802,weight=1)
G9.add_edge(795,1199,weight=1)
G9.add_edge(795,849,weight=1)
G9.add_edge(795,850,weight=1)
G9.add_edge(795,854,weight=1)
G9.add_edge(795,867,weight=1)
G9.add_edge(795,880,weight=1)
G9.add_edge(796,888,weight=1)
G9.add_edge(798,895,weight=1)
G9.add_edge(798,924,weight=1)
G9.add_edge(799,877,weight=1)
G9.add_edge(799,897,weight=1)
G9.add_edge(800,1122,weight=1)
G9.add_edge(800,801,weight=1)
G9.add_edge(800,820,weight=1)
G9.add_edge(800,829,weight=1)
G9.add_edge(800,883,weight=1)
G9.add_edge(800,919,weight=1)
G9.add_edge(800,920,weight=1)
G9.add_edge(800,924,weight=1)
G9.add_edge(800,932,weight=1)
G9.add_edge(801,3133,weight=1)
G9.add_edge(801,924,weight=1)
G9.add_edge(802,2757,weight=1)
G9.add_edge(802,1192,weight=1)
G9.add_edge(802,807,weight=1)
G9.add_edge(802,822,weight=1)
G9.add_edge(802,825,weight=1)
G9.add_edge(802,864,weight=1)
G9.add_edge(802,865,weight=1)
G9.add_edge(802,1797,weight=1)
G9.add_edge(802,883,weight=1)
G9.add_edge(802,891,weight=1)
G9.add_edge(803,883,weight=1)
G9.add_edge(804,2346,weight=1)
G9.add_edge(804,1284,weight=1)
G9.add_edge(804,2151,weight=1)
G9.add_edge(804,2152,weight=1)
G9.add_edge(804,839,weight=1)
G9.add_edge(804,2129,weight=1)
G9.add_edge(805,806,weight=1)
G9.add_edge(807,1943,weight=1)
G9.add_edge(808,1087,weight=1)
G9.add_edge(809,881,weight=1)
G9.add_edge(809,1306,weight=1)
G9.add_edge(809,990,weight=1)
G9.add_edge(810,3019,weight=1)
G9.add_edge(810,2367,weight=1)
G9.add_edge(811,2346,weight=1)
G9.add_edge(811,1284,weight=1)
G9.add_edge(811,840,weight=1)
G9.add_edge(811,2366,weight=1)
G9.add_edge(811,2318,weight=1)
G9.add_edge(811,2919,weight=1)
G9.add_edge(814,956,weight=1)
G9.add_edge(815,829,weight=1)
G9.add_edge(815,870,weight=1)
G9.add_edge(815,899,weight=1)
G9.add_edge(816,883,weight=1)
G9.add_edge(818,834,weight=1)
G9.add_edge(819,820,weight=1)
G9.add_edge(820,2203,weight=1)
G9.add_edge(820,843,weight=1)
G9.add_edge(821,1473,weight=1)
G9.add_edge(821,838,weight=1)
G9.add_edge(821,854,weight=1)
G9.add_edge(821,856,weight=1)
G9.add_edge(821,870,weight=1)
G9.add_edge(823,870,weight=1)
G9.add_edge(823,932,weight=1)
G9.add_edge(824,875,weight=1)
G9.add_edge(825,2347,weight=1)
G9.add_edge(825,2730,weight=1)
G9.add_edge(827,870,weight=1)
G9.add_edge(827,932,weight=1)
G9.add_edge(828,919,weight=1)
G9.add_edge(829,919,weight=1)
G9.add_edge(829,924,weight=1)
G9.add_edge(830,1780,weight=1)
G9.add_edge(830,2351,weight=1)
G9.add_edge(830,2359,weight=1)
G9.add_edge(830,2342,weight=1)
G9.add_edge(830,2983,weight=1)
G9.add_edge(830,853,weight=1)
G9.add_edge(830,2309,weight=1)
G9.add_edge(830,882,weight=1)
G9.add_edge(832,833,weight=1)
G9.add_edge(835,870,weight=1)
G9.add_edge(836,870,weight=1)
G9.add_edge(837,2926,weight=1)
G9.add_edge(837,2929,weight=1)
G9.add_edge(837,3141,weight=1)
G9.add_edge(837,848,weight=1)
G9.add_edge(839,2151,weight=1)
G9.add_edge(839,2152,weight=1)
G9.add_edge(839,2366,weight=1)
G9.add_edge(839,2318,weight=1)
G9.add_edge(840,1284,weight=1)
G9.add_edge(840,847,weight=1)
G9.add_edge(840,2829,weight=1)
G9.add_edge(840,2931,weight=1)
G9.add_edge(840,2915,weight=1)
G9.add_edge(840,996,weight=1)
G9.add_edge(840,2919,weight=1)
G9.add_edge(841,870,weight=1)
G9.add_edge(841,876,weight=1)
G9.add_edge(842,854,weight=1)
G9.add_edge(844,870,weight=1)
G9.add_edge(844,3133,weight=1)
G9.add_edge(844,899,weight=1)
G9.add_edge(845,1907,weight=1)
G9.add_edge(848,1352,weight=1)
G9.add_edge(848,1978,weight=1)
G9.add_edge(848,2127,weight=1)
G9.add_edge(849,851,weight=1)
G9.add_edge(849,854,weight=1)
G9.add_edge(850,851,weight=1)
G9.add_edge(850,854,weight=1)
G9.add_edge(852,1280,weight=1)
G9.add_edge(852,858,weight=1)
G9.add_edge(852,889,weight=1)
G9.add_edge(854,856,weight=1)
G9.add_edge(854,870,weight=1)
G9.add_edge(854,871,weight=1)
G9.add_edge(854,880,weight=1)
G9.add_edge(854,2712,weight=1)
G9.add_edge(856,870,weight=1)
G9.add_edge(856,871,weight=1)
G9.add_edge(856,880,weight=1)
G9.add_edge(857,931,weight=1)
G9.add_edge(858,2349,weight=1)
G9.add_edge(858,1464,weight=1)
G9.add_edge(858,2354,weight=1)
G9.add_edge(858,2358,weight=1)
G9.add_edge(858,2364,weight=1)
G9.add_edge(859,916,weight=1)
G9.add_edge(860,1199,weight=1)
G9.add_edge(860,863,weight=1)
G9.add_edge(860,885,weight=1)
G9.add_edge(862,1907,weight=1)
G9.add_edge(863,1907,weight=1)
G9.add_edge(863,913,weight=1)
G9.add_edge(864,866,weight=1)
G9.add_edge(865,866,weight=1)
G9.add_edge(868,870,weight=1)
G9.add_edge(869,871,weight=1)
G9.add_edge(869,873,weight=1)
G9.add_edge(870,2501,weight=1)
G9.add_edge(870,2757,weight=1)
G9.add_edge(870,2213,weight=1)
G9.add_edge(870,914,weight=1)
G9.add_edge(870,920,weight=1)
G9.add_edge(870,922,weight=1)
G9.add_edge(870,923,weight=1)
G9.add_edge(870,929,weight=1)
G9.add_edge(871,1346,weight=1)
G9.add_edge(871,920,weight=1)
G9.add_edge(871,922,weight=1)
G9.add_edge(871,923,weight=1)
G9.add_edge(875,1570,weight=1)
G9.add_edge(875,973,weight=1)
G9.add_edge(875,1004,weight=1)
G9.add_edge(875,1024,weight=1)
G9.add_edge(875,1057,weight=1)
G9.add_edge(875,1966,weight=1)
G9.add_edge(875,1099,weight=1)
G9.add_edge(876,878,weight=1)
G9.add_edge(878,879,weight=1)
G9.add_edge(878,2946,weight=1)
G9.add_edge(881,1306,weight=1)
G9.add_edge(882,1019,weight=1)
G9.add_edge(882,1102,weight=1)
G9.add_edge(883,1576,weight=1)
G9.add_edge(883,901,weight=1)
G9.add_edge(884,919,weight=1)
G9.add_edge(885,933,weight=1)
G9.add_edge(886,2621,weight=1)
G9.add_edge(887,1972,weight=1)
G9.add_edge(891,1797,weight=1)
G9.add_edge(892,1343,weight=1)
G9.add_edge(892,2127,weight=1)
G9.add_edge(892,2979,weight=1)
G9.add_edge(894,2302,weight=1)
G9.add_edge(896,902,weight=1)
G9.add_edge(897,2709,weight=1)
G9.add_edge(898,916,weight=1)
G9.add_edge(899,919,weight=1)
G9.add_edge(899,924,weight=1)
G9.add_edge(899,928,weight=1)
G9.add_edge(900,901,weight=1)
G9.add_edge(900,1091,weight=1)
G9.add_edge(902,1883,weight=1)
G9.add_edge(902,2737,weight=1)
G9.add_edge(904,2659,weight=1)
G9.add_edge(904,924,weight=1)
G9.add_edge(906,2352,weight=1)
G9.add_edge(906,2359,weight=1)
G9.add_edge(908,1907,weight=1)
G9.add_edge(910,2308,weight=1)
G9.add_edge(911,924,weight=1)
G9.add_edge(912,1101,weight=1)
G9.add_edge(913,1907,weight=1)
G9.add_edge(917,1121,weight=1)
G9.add_edge(919,920,weight=1)
G9.add_edge(920,924,weight=1)
G9.add_edge(920,927,weight=1)
G9.add_edge(921,1346,weight=1)
G9.add_edge(924,3133,weight=1)
G9.add_edge(924,928,weight=1)
G9.add_edge(926,927,weight=1)
G9.add_edge(929,2958,weight=1)
G9.add_edge(929,3009,weight=1)
G9.add_edge(933,1907,weight=1)
G9.add_edge(935,961,weight=1)
G9.add_edge(936,2398,weight=1)
G9.add_edge(937,2576,weight=1)
G9.add_edge(938,2033,weight=1)
G9.add_edge(939,2373,weight=1)
G9.add_edge(939,2441,weight=1)
G9.add_edge(940,2398,weight=1)
G9.add_edge(941,1963,weight=1)
G9.add_edge(942,2259,weight=1)
G9.add_edge(943,1609,weight=1)
G9.add_edge(943,1611,weight=1)
G9.add_edge(943,1213,weight=1)
G9.add_edge(944,2949,weight=1)
G9.add_edge(944,2564,weight=1)
G9.add_edge(944,1672,weight=1)
G9.add_edge(945,2246,weight=1)
G9.add_edge(945,1686,weight=1)
G9.add_edge(945,2625,weight=1)
G9.add_edge(946,983,weight=1)
G9.add_edge(947,2091,weight=1)
G9.add_edge(947,2092,weight=1)
G9.add_edge(947,2394,weight=1)
G9.add_edge(948,2964,weight=1)
G9.add_edge(948,1216,weight=1)
G9.add_edge(948,1218,weight=1)
G9.add_edge(950,1270,weight=1)
G9.add_edge(951,2951,weight=1)
G9.add_edge(952,986,weight=1)
G9.add_edge(953,3192,weight=1)
G9.add_edge(953,1269,weight=1)
G9.add_edge(954,958,weight=1)
G9.add_edge(954,989,weight=1)
G9.add_edge(955,2567,weight=1)
G9.add_edge(956,1057,weight=1)
G9.add_edge(956,1966,weight=1)
G9.add_edge(957,1260,weight=1)
G9.add_edge(958,989,weight=1)
G9.add_edge(959,2246,weight=1)
G9.add_edge(959,1686,weight=1)
G9.add_edge(959,2625,weight=1)
G9.add_edge(961,977,weight=1)
G9.add_edge(962,1004,weight=1)
G9.add_edge(963,2381,weight=1)
G9.add_edge(964,1485,weight=1)
G9.add_edge(965,1371,weight=1)
G9.add_edge(968,2567,weight=1)
G9.add_edge(969,2529,weight=1)
G9.add_edge(969,2986,weight=1)
G9.add_edge(969,1257,weight=1)
G9.add_edge(970,1820,weight=1)
G9.add_edge(971,3126,weight=1)
G9.add_edge(972,1510,weight=1)
G9.add_edge(972,1608,weight=1)
G9.add_edge(972,3215,weight=1)
G9.add_edge(972,1512,weight=1)
G9.add_edge(972,1740,weight=1)
G9.add_edge(972,2337,weight=1)
G9.add_edge(974,2796,weight=1)
G9.add_edge(974,2138,weight=1)
G9.add_edge(975,2686,weight=1)
G9.add_edge(975,2915,weight=1)
G9.add_edge(975,2919,weight=1)
G9.add_edge(976,2854,weight=1)
G9.add_edge(978,1371,weight=1)
G9.add_edge(980,1816,weight=1)
G9.add_edge(980,1437,weight=1)
G9.add_edge(981,3246,weight=1)
G9.add_edge(981,2639,weight=1)
G9.add_edge(982,3005,weight=1)
G9.add_edge(983,984,weight=1)
G9.add_edge(984,1099,weight=1)
G9.add_edge(985,2086,weight=1)
G9.add_edge(987,1219,weight=1)
G9.add_edge(988,2949,weight=1)
G9.add_edge(988,2564,weight=1)
G9.add_edge(988,1672,weight=1)
G9.add_edge(990,1306,weight=1)
G9.add_edge(991,1863,weight=1)
G9.add_edge(991,2086,weight=1)
G9.add_edge(996,2829,weight=1)
G9.add_edge(997,1238,weight=1)
G9.add_edge(998,1054,weight=1)
G9.add_edge(999,2989,weight=1)
G9.add_edge(1000,1320,weight=1)
G9.add_edge(1000,1092,weight=1)
G9.add_edge(1000,1109,weight=1)
G9.add_edge(1002,1098,weight=1)
G9.add_edge(1004,2113,weight=1)
G9.add_edge(1005,1689,weight=1)
G9.add_edge(1006,1094,weight=1)
G9.add_edge(1007,1104,weight=1)
G9.add_edge(1007,1118,weight=1)
G9.add_edge(1008,1099,weight=1)
G9.add_edge(1009,3038,weight=1)
G9.add_edge(1011,1012,weight=1)
G9.add_edge(1011,1068,weight=1)
G9.add_edge(1013,1071,weight=1)
G9.add_edge(1014,1117,weight=1)
G9.add_edge(1015,1231,weight=1)
G9.add_edge(1015,1237,weight=1)
G9.add_edge(1016,2083,weight=1)
G9.add_edge(1017,2110,weight=1)
G9.add_edge(1018,1082,weight=1)
G9.add_edge(1019,1102,weight=1)
G9.add_edge(1020,1062,weight=1)
G9.add_edge(1021,1847,weight=1)
G9.add_edge(1022,1568,weight=1)
G9.add_edge(1022,1696,weight=1)
G9.add_edge(1022,1117,weight=1)
G9.add_edge(1023,1031,weight=1)
G9.add_edge(1023,1069,weight=1)
G9.add_edge(1023,1750,weight=1)
G9.add_edge(1024,1057,weight=1)
G9.add_edge(1024,1966,weight=1)
G9.add_edge(1025,1847,weight=1)
G9.add_edge(1026,1104,weight=1)
G9.add_edge(1026,1105,weight=1)
G9.add_edge(1026,1118,weight=1)
G9.add_edge(1027,1058,weight=1)
G9.add_edge(1028,1085,weight=1)
G9.add_edge(1028,1086,weight=1)
G9.add_edge(1028,1087,weight=1)
G9.add_edge(1029,1075,weight=1)
G9.add_edge(1029,1076,weight=1)
G9.add_edge(1031,1069,weight=1)
G9.add_edge(1032,1110,weight=1)
G9.add_edge(1033,2817,weight=1)
G9.add_edge(1034,2446,weight=1)
G9.add_edge(1035,1078,weight=1)
G9.add_edge(1035,2291,weight=1)
G9.add_edge(1035,1108,weight=1)
G9.add_edge(1036,3200,weight=1)
G9.add_edge(1037,2911,weight=1)
G9.add_edge(1037,2464,weight=1)
G9.add_edge(1038,1040,weight=1)
G9.add_edge(1038,1091,weight=1)
G9.add_edge(1039,1091,weight=1)
G9.add_edge(1040,1091,weight=1)
G9.add_edge(1041,1090,weight=1)
G9.add_edge(1042,2288,weight=1)
G9.add_edge(1042,2489,weight=1)
G9.add_edge(1043,1147,weight=1)
G9.add_edge(1044,1120,weight=1)
G9.add_edge(1047,2766,weight=1)
G9.add_edge(1048,1049,weight=1)
G9.add_edge(1050,1051,weight=1)
G9.add_edge(1050,1053,weight=1)
G9.add_edge(1051,1052,weight=1)
G9.add_edge(1052,1053,weight=1)
G9.add_edge(1056,2850,weight=1)
G9.add_edge(1057,1966,weight=1)
G9.add_edge(1059,1117,weight=1)
G9.add_edge(1060,3224,weight=1)
G9.add_edge(1061,1093,weight=1)
G9.add_edge(1062,2363,weight=1)
G9.add_edge(1063,1348,weight=1)
G9.add_edge(1064,1115,weight=1)
G9.add_edge(1065,2754,weight=1)
G9.add_edge(1065,1406,weight=1)
G9.add_edge(1066,2593,weight=1)
G9.add_edge(1067,2724,weight=1)
G9.add_edge(1070,1422,weight=1)
G9.add_edge(1071,1072,weight=1)
G9.add_edge(1072,1966,weight=1)
G9.add_edge(1074,2168,weight=1)
G9.add_edge(1074,2173,weight=1)
G9.add_edge(1077,2770,weight=1)
G9.add_edge(1077,1970,weight=1)
G9.add_edge(1077,2839,weight=1)
G9.add_edge(1080,1117,weight=1)
G9.add_edge(1081,3032,weight=1)
G9.add_edge(1081,1106,weight=1)
G9.add_edge(1082,1847,weight=1)
G9.add_edge(1085,1086,weight=1)
G9.add_edge(1085,1088,weight=1)
G9.add_edge(1086,1087,weight=1)
G9.add_edge(1087,1088,weight=1)
G9.add_edge(1090,1757,weight=1)
G9.add_edge(1090,1239,weight=1)
G9.add_edge(1090,2611,weight=1)
G9.add_edge(1090,1765,weight=1)
G9.add_edge(1092,1320,weight=1)
G9.add_edge(1094,2468,weight=1)
G9.add_edge(1095,1096,weight=1)
G9.add_edge(1099,2113,weight=1)
G9.add_edge(1100,1747,weight=1)
G9.add_edge(1100,2860,weight=1)
G9.add_edge(1101,1231,weight=1)
G9.add_edge(1104,1105,weight=1)
G9.add_edge(1104,1118,weight=1)
G9.add_edge(1105,1118,weight=1)
G9.add_edge(1107,3204,weight=1)
G9.add_edge(1107,3050,weight=1)
G9.add_edge(1112,2591,weight=1)
G9.add_edge(1113,2591,weight=1)
G9.add_edge(1113,2617,weight=1)
G9.add_edge(1114,1847,weight=1)
G9.add_edge(1114,1116,weight=1)
G9.add_edge(1120,2309,weight=1)
G9.add_edge(1123,1229,weight=1)
G9.add_edge(1124,2469,weight=1)
G9.add_edge(1125,1126,weight=1)
G9.add_edge(1125,2397,weight=1)
G9.add_edge(1125,1693,weight=1)
G9.add_edge(1126,1542,weight=1)
G9.add_edge(1126,1552,weight=1)
G9.add_edge(1126,2907,weight=1)
G9.add_edge(1126,1128,weight=1)
G9.add_edge(1127,1542,weight=1)
G9.add_edge(1127,2208,weight=1)
G9.add_edge(1127,1552,weight=1)
G9.add_edge(1127,1563,weight=1)
G9.add_edge(1127,2657,weight=1)
G9.add_edge(1127,2041,weight=1)
G9.add_edge(1128,1551,weight=1)
G9.add_edge(1128,2907,weight=1)
G9.add_edge(1129,1437,weight=1)
G9.add_edge(1129,3250,weight=1)
G9.add_edge(1130,2204,weight=1)
G9.add_edge(1130,1306,weight=1)
G9.add_edge(1131,2372,weight=1)
G9.add_edge(1131,1134,weight=1)
G9.add_edge(1131,1140,weight=1)
G9.add_edge(1133,1140,weight=1)
G9.add_edge(1134,2369,weight=1)
G9.add_edge(1134,2401,weight=1)
G9.add_edge(1134,2432,weight=1)
G9.add_edge(1135,2760,weight=1)
G9.add_edge(1136,1144,weight=1)
G9.add_edge(1137,2899,weight=1)
G9.add_edge(1137,2523,weight=1)
G9.add_edge(1137,1756,weight=1)
G9.add_edge(1137,2407,weight=1)
G9.add_edge(1139,2432,weight=1)
G9.add_edge(1140,1141,weight=1)
G9.add_edge(1140,1145,weight=1)
G9.add_edge(1141,2035,weight=1)
G9.add_edge(1143,2971,weight=1)
G9.add_edge(1146,1148,weight=1)
G9.add_edge(1148,2035,weight=1)
G9.add_edge(1151,1314,weight=1)
G9.add_edge(1151,1427,weight=1)
G9.add_edge(1152,2453,weight=1)
G9.add_edge(1153,2679,weight=1)
G9.add_edge(1154,1771,weight=1)
G9.add_edge(1154,2641,weight=1)
G9.add_edge(1155,2798,weight=1)
G9.add_edge(1156,1159,weight=1)
G9.add_edge(1156,2434,weight=1)
G9.add_edge(1157,1219,weight=1)
G9.add_edge(1157,1487,weight=1)
G9.add_edge(1158,3082,weight=1)
G9.add_edge(1160,3020,weight=1)
G9.add_edge(1161,1166,weight=1)
G9.add_edge(1162,2331,weight=1)
G9.add_edge(1162,1167,weight=1)
G9.add_edge(1163,1708,weight=1)
G9.add_edge(1164,1682,weight=1)
G9.add_edge(1164,2331,weight=1)
G9.add_edge(1164,1167,weight=1)
G9.add_edge(1167,2331,weight=1)
G9.add_edge(1169,2092,weight=1)
G9.add_edge(1169,1411,weight=1)
G9.add_edge(1169,1416,weight=1)
G9.add_edge(1170,2907,weight=1)
G9.add_edge(1172,2907,weight=1)
G9.add_edge(1173,2683,weight=1)
G9.add_edge(1173,2684,weight=1)
G9.add_edge(1173,2685,weight=1)
G9.add_edge(1173,1559,weight=1)
G9.add_edge(1173,1639,weight=1)
G9.add_edge(1174,1181,weight=1)
G9.add_edge(1175,1806,weight=1)
G9.add_edge(1175,2435,weight=1)
G9.add_edge(1176,1585,weight=1)
G9.add_edge(1176,1801,weight=1)
G9.add_edge(1176,2805,weight=1)
G9.add_edge(1176,1177,weight=1)
G9.add_edge(1176,3116,weight=1)
G9.add_edge(1177,1766,weight=1)
G9.add_edge(1177,3153,weight=1)
G9.add_edge(1177,2534,weight=1)
G9.add_edge(1177,2537,weight=1)
G9.add_edge(1177,2654,weight=1)
G9.add_edge(1177,2233,weight=1)
G9.add_edge(1177,2392,weight=1)
G9.add_edge(1177,2076,weight=1)
G9.add_edge(1177,2550,weight=1)
G9.add_edge(1177,2061,weight=1)
G9.add_edge(1177,1643,weight=1)
G9.add_edge(1177,2317,weight=1)
G9.add_edge(1177,1305,weight=1)
G9.add_edge(1177,2554,weight=1)
G9.add_edge(1177,2555,weight=1)
G9.add_edge(1177,2104,weight=1)
G9.add_edge(1177,1221,weight=1)
G9.add_edge(1177,2435,weight=1)
G9.add_edge(1177,1396,weight=1)
G9.add_edge(1177,2579,weight=1)
G9.add_edge(1178,2622,weight=1)
G9.add_edge(1178,1636,weight=1)
G9.add_edge(1179,1559,weight=1)
G9.add_edge(1179,1636,weight=1)
G9.add_edge(1179,1639,weight=1)
G9.add_edge(1179,2908,weight=1)
G9.add_edge(1179,1181,weight=1)
G9.add_edge(1180,2789,weight=1)
G9.add_edge(1182,1183,weight=1)
G9.add_edge(1184,2211,weight=1)
G9.add_edge(1184,1577,weight=1)
G9.add_edge(1184,1627,weight=1)
G9.add_edge(1185,2255,weight=1)
G9.add_edge(1186,1577,weight=1)
G9.add_edge(1186,1627,weight=1)
G9.add_edge(1186,1661,weight=1)
G9.add_edge(1187,2211,weight=1)
G9.add_edge(1187,1577,weight=1)
G9.add_edge(1187,2255,weight=1)
G9.add_edge(1187,1189,weight=1)
G9.add_edge(1188,1219,weight=1)
G9.add_edge(1188,1489,weight=1)
G9.add_edge(1189,2255,weight=1)
G9.add_edge(1190,1327,weight=1)
G9.add_edge(1190,1570,weight=1)
G9.add_edge(1193,1469,weight=1)
G9.add_edge(1193,3178,weight=1)
G9.add_edge(1193,1791,weight=1)
G9.add_edge(1197,1920,weight=1)
G9.add_edge(1197,1315,weight=1)
G9.add_edge(1198,2754,weight=1)
G9.add_edge(1198,1786,weight=1)
G9.add_edge(1198,2201,weight=1)
G9.add_edge(1198,1207,weight=1)
G9.add_edge(1198,1630,weight=1)
G9.add_edge(1198,1654,weight=1)
G9.add_edge(1198,1675,weight=1)
G9.add_edge(1198,2906,weight=1)
G9.add_edge(1199,1356,weight=1)
G9.add_edge(1200,2088,weight=1)
G9.add_edge(1200,2686,weight=1)
G9.add_edge(1200,1556,weight=1)
G9.add_edge(1200,1557,weight=1)
G9.add_edge(1200,2907,weight=1)
G9.add_edge(1200,1972,weight=1)
G9.add_edge(1200,1563,weight=1)
G9.add_edge(1200,2516,weight=1)
G9.add_edge(1200,1572,weight=1)
G9.add_edge(1200,2220,weight=1)
G9.add_edge(1200,1981,weight=1)
G9.add_edge(1200,2366,weight=1)
G9.add_edge(1202,3161,weight=1)
G9.add_edge(1203,2038,weight=1)
G9.add_edge(1203,2868,weight=1)
G9.add_edge(1204,2754,weight=1)
G9.add_edge(1204,1574,weight=1)
G9.add_edge(1205,2932,weight=1)
G9.add_edge(1206,1210,weight=1)
G9.add_edge(1207,2754,weight=1)
G9.add_edge(1207,2093,weight=1)
G9.add_edge(1207,2044,weight=1)
G9.add_edge(1208,2764,weight=1)
G9.add_edge(1209,1371,weight=1)
G9.add_edge(1210,2092,weight=1)
G9.add_edge(1210,2137,weight=1)
G9.add_edge(1210,1630,weight=1)
G9.add_edge(1210,1223,weight=1)
G9.add_edge(1211,1371,weight=1)
G9.add_edge(1212,2568,weight=1)
G9.add_edge(1213,2405,weight=1)
G9.add_edge(1214,1766,weight=1)
G9.add_edge(1214,3153,weight=1)
G9.add_edge(1214,2537,weight=1)
G9.add_edge(1214,2076,weight=1)
G9.add_edge(1214,1643,weight=1)
G9.add_edge(1214,2317,weight=1)
G9.add_edge(1214,3134,weight=1)
G9.add_edge(1214,2554,weight=1)
G9.add_edge(1214,2555,weight=1)
G9.add_edge(1215,1585,weight=1)
G9.add_edge(1215,1801,weight=1)
G9.add_edge(1215,1589,weight=1)
G9.add_edge(1215,2805,weight=1)
G9.add_edge(1215,1305,weight=1)
G9.add_edge(1216,1297,weight=1)
G9.add_edge(1217,2541,weight=1)
G9.add_edge(1219,2385,weight=1)
G9.add_edge(1219,1996,weight=1)
G9.add_edge(1219,2199,weight=1)
G9.add_edge(1219,2045,weight=1)
G9.add_edge(1219,2433,weight=1)
G9.add_edge(1219,1826,weight=1)
G9.add_edge(1219,1828,weight=1)
G9.add_edge(1219,2838,weight=1)
G9.add_edge(1219,2098,weight=1)
G9.add_edge(1219,2582,weight=1)
G9.add_edge(1219,1489,weight=1)
G9.add_edge(1220,2093,weight=1)
G9.add_edge(1220,1617,weight=1)
G9.add_edge(1220,1253,weight=1)
G9.add_edge(1220,1634,weight=1)
G9.add_edge(1220,1664,weight=1)
G9.add_edge(1220,1675,weight=1)
G9.add_edge(1220,3230,weight=1)
G9.add_edge(1222,1588,weight=1)
G9.add_edge(1223,2059,weight=1)
G9.add_edge(1225,3114,weight=1)
G9.add_edge(1226,1253,weight=1)
G9.add_edge(1227,3190,weight=1)
G9.add_edge(1228,3258,weight=1)
G9.add_edge(1232,3001,weight=1)
G9.add_edge(1234,2952,weight=1)
G9.add_edge(1235,1498,weight=1)
G9.add_edge(1236,1248,weight=1)
G9.add_edge(1238,2607,weight=1)
G9.add_edge(1238,2886,weight=1)
G9.add_edge(1241,1315,weight=1)
G9.add_edge(1243,2269,weight=1)
G9.add_edge(1244,3236,weight=1)
G9.add_edge(1244,1899,weight=1)
G9.add_edge(1245,1276,weight=1)
G9.add_edge(1246,1690,weight=1)
G9.add_edge(1247,1414,weight=1)
G9.add_edge(1249,2089,weight=1)
G9.add_edge(1249,1345,weight=1)
G9.add_edge(1250,1299,weight=1)
G9.add_edge(1250,1818,weight=1)
G9.add_edge(1250,2891,weight=1)
G9.add_edge(1250,2436,weight=1)
G9.add_edge(1250,2192,weight=1)
G9.add_edge(1250,1824,weight=1)
G9.add_edge(1250,2624,weight=1)
G9.add_edge(1250,2176,weight=1)
G9.add_edge(1251,2746,weight=1)
G9.add_edge(1252,1299,weight=1)
G9.add_edge(1252,1818,weight=1)
G9.add_edge(1252,2891,weight=1)
G9.add_edge(1252,2436,weight=1)
G9.add_edge(1252,2192,weight=1)
G9.add_edge(1252,1824,weight=1)
G9.add_edge(1252,2624,weight=1)
G9.add_edge(1252,2176,weight=1)
G9.add_edge(1254,1487,weight=1)
G9.add_edge(1255,1808,weight=1)
G9.add_edge(1256,3210,weight=1)
G9.add_edge(1256,3148,weight=1)
G9.add_edge(1257,2992,weight=1)
G9.add_edge(1257,2529,weight=1)
G9.add_edge(1257,2986,weight=1)
G9.add_edge(1258,3037,weight=1)
G9.add_edge(1259,3210,weight=1)
G9.add_edge(1259,1810,weight=1)
G9.add_edge(1259,1818,weight=1)
G9.add_edge(1259,2891,weight=1)
G9.add_edge(1259,2667,weight=1)
G9.add_edge(1259,2192,weight=1)
G9.add_edge(1260,2719,weight=1)
G9.add_edge(1262,1700,weight=1)
G9.add_edge(1263,3119,weight=1)
G9.add_edge(1264,2736,weight=1)
G9.add_edge(1264,2366,weight=1)
G9.add_edge(1264,1273,weight=1)
G9.add_edge(1265,2068,weight=1)
G9.add_edge(1266,1267,weight=1)
G9.add_edge(1266,1268,weight=1)
G9.add_edge(1266,1270,weight=1)
G9.add_edge(1266,2128,weight=1)
G9.add_edge(1267,2346,weight=1)
G9.add_edge(1267,3008,weight=1)
G9.add_edge(1267,3178,weight=1)
G9.add_edge(1267,1284,weight=1)
G9.add_edge(1267,1268,weight=1)
G9.add_edge(1267,1270,weight=1)
G9.add_edge(1267,2128,weight=1)
G9.add_edge(1268,2841,weight=1)
G9.add_edge(1268,2842,weight=1)
G9.add_edge(1268,1312,weight=1)
G9.add_edge(1269,2962,weight=1)
G9.add_edge(1270,2128,weight=1)
G9.add_edge(1270,1271,weight=1)
G9.add_edge(1272,1416,weight=1)
G9.add_edge(1273,2800,weight=1)
G9.add_edge(1274,1681,weight=1)
G9.add_edge(1275,3178,weight=1)
G9.add_edge(1275,1835,weight=1)
G9.add_edge(1276,1278,weight=1)
G9.add_edge(1276,1279,weight=1)
G9.add_edge(1277,1278,weight=1)
G9.add_edge(1280,2598,weight=1)
G9.add_edge(1281,2907,weight=1)
G9.add_edge(1283,2498,weight=1)
G9.add_edge(1283,2499,weight=1)
G9.add_edge(1283,1574,weight=1)
G9.add_edge(1283,2093,weight=1)
G9.add_edge(1283,2366,weight=1)
G9.add_edge(1283,1927,weight=1)
G9.add_edge(1283,2000,weight=1)
G9.add_edge(1283,2972,weight=1)
G9.add_edge(1283,2258,weight=1)
G9.add_edge(1284,3191,weight=1)
G9.add_edge(1284,3194,weight=1)
G9.add_edge(1284,2619,weight=1)
G9.add_edge(1284,3222,weight=1)
G9.add_edge(1284,3239,weight=1)
G9.add_edge(1284,1442,weight=1)
G9.add_edge(1284,2366,weight=1)
G9.add_edge(1285,2106,weight=1)
G9.add_edge(1286,2865,weight=1)
G9.add_edge(1287,1562,weight=1)
G9.add_edge(1287,1291,weight=1)
G9.add_edge(1287,2201,weight=1)
G9.add_edge(1288,2187,weight=1)
G9.add_edge(1289,2907,weight=1)
G9.add_edge(1289,2039,weight=1)
G9.add_edge(1289,2507,weight=1)
G9.add_edge(1289,3075,weight=1)
G9.add_edge(1289,1580,weight=1)
G9.add_edge(1290,3072,weight=1)
G9.add_edge(1291,1562,weight=1)
G9.add_edge(1291,1564,weight=1)
G9.add_edge(1291,2201,weight=1)
G9.add_edge(1291,1577,weight=1)
G9.add_edge(1292,1465,weight=1)
G9.add_edge(1292,1466,weight=1)
G9.add_edge(1292,1467,weight=1)
G9.add_edge(1292,1468,weight=1)
G9.add_edge(1292,1469,weight=1)
G9.add_edge(1292,1470,weight=1)
G9.add_edge(1292,1787,weight=1)
G9.add_edge(1293,2806,weight=1)
G9.add_edge(1293,2810,weight=1)
G9.add_edge(1294,1949,weight=1)
G9.add_edge(1295,1481,weight=1)
G9.add_edge(1295,1436,weight=1)
G9.add_edge(1295,1818,weight=1)
G9.add_edge(1295,2624,weight=1)
G9.add_edge(1295,1488,weight=1)
G9.add_edge(1296,2129,weight=1)
G9.add_edge(1298,2692,weight=1)
G9.add_edge(1298,2742,weight=1)
G9.add_edge(1298,1947,weight=1)
G9.add_edge(1298,2393,weight=1)
G9.add_edge(1298,1948,weight=1)
G9.add_edge(1298,2552,weight=1)
G9.add_edge(1298,2645,weight=1)
G9.add_edge(1298,1670,weight=1)
G9.add_edge(1298,1675,weight=1)
G9.add_edge(1300,2382,weight=1)
G9.add_edge(1301,1593,weight=1)
G9.add_edge(1301,1623,weight=1)
G9.add_edge(1301,1952,weight=1)
G9.add_edge(1302,1502,weight=1)
G9.add_edge(1303,3190,weight=1)
G9.add_edge(1304,1862,weight=1)
G9.add_edge(1305,1801,weight=1)
G9.add_edge(1306,2060,weight=1)
G9.add_edge(1306,1885,weight=1)
G9.add_edge(1306,3249,weight=1)
G9.add_edge(1306,2162,weight=1)
G9.add_edge(1307,2808,weight=1)
G9.add_edge(1308,1456,weight=1)
G9.add_edge(1309,2804,weight=1)
G9.add_edge(1309,2968,weight=1)
G9.add_edge(1310,2972,weight=1)
G9.add_edge(1311,2986,weight=1)
G9.add_edge(1313,1859,weight=1)
G9.add_edge(1313,1941,weight=1)
G9.add_edge(1313,1942,weight=1)
G9.add_edge(1314,1761,weight=1)
G9.add_edge(1314,1427,weight=1)
G9.add_edge(1316,2602,weight=1)
G9.add_edge(1317,2919,weight=1)
G9.add_edge(1318,2140,weight=1)
G9.add_edge(1319,1792,weight=1)
G9.add_edge(1319,2062,weight=1)
G9.add_edge(1320,2451,weight=1)
G9.add_edge(1322,2019,weight=1)
G9.add_edge(1323,1490,weight=1)
G9.add_edge(1324,1832,weight=1)
G9.add_edge(1324,1325,weight=1)
G9.add_edge(1327,1763,weight=1)
G9.add_edge(1327,2350,weight=1)
G9.add_edge(1328,2887,weight=1)
G9.add_edge(1328,2503,weight=1)
G9.add_edge(1328,1330,weight=1)
G9.add_edge(1328,1334,weight=1)
G9.add_edge(1328,1335,weight=1)
G9.add_edge(1329,2353,weight=1)
G9.add_edge(1329,1524,weight=1)
G9.add_edge(1329,2411,weight=1)
G9.add_edge(1331,1333,weight=1)
G9.add_edge(1331,1334,weight=1)
G9.add_edge(1334,1335,weight=1)
G9.add_edge(1336,1980,weight=1)
G9.add_edge(1337,1840,weight=1)
G9.add_edge(1338,1554,weight=1)
G9.add_edge(1340,1550,weight=1)
G9.add_edge(1341,2907,weight=1)
G9.add_edge(1341,2041,weight=1)
G9.add_edge(1342,2496,weight=1)
G9.add_edge(1343,2197,weight=1)
G9.add_edge(1343,2059,weight=1)
G9.add_edge(1343,2930,weight=1)
G9.add_edge(1343,2039,weight=1)
G9.add_edge(1344,2039,weight=1)
G9.add_edge(1344,2149,weight=1)
G9.add_edge(1344,2852,weight=1)
G9.add_edge(1344,3205,weight=1)
G9.add_edge(1345,1969,weight=1)
G9.add_edge(1345,1545,weight=1)
G9.add_edge(1345,2092,weight=1)
G9.add_edge(1345,3076,weight=1)
G9.add_edge(1345,2511,weight=1)
G9.add_edge(1345,1567,weight=1)
G9.add_edge(1345,1509,weight=1)
G9.add_edge(1345,2221,weight=1)
G9.add_edge(1345,2310,weight=1)
G9.add_edge(1347,2955,weight=1)
G9.add_edge(1347,1348,weight=1)
G9.add_edge(1348,2955,weight=1)
G9.add_edge(1348,2872,weight=1)
G9.add_edge(1348,1831,weight=1)
G9.add_edge(1348,3046,weight=1)
G9.add_edge(1348,2600,weight=1)
G9.add_edge(1348,1861,weight=1)
G9.add_edge(1349,2039,weight=1)
G9.add_edge(1349,2852,weight=1)
G9.add_edge(1349,1872,weight=1)
G9.add_edge(1349,2911,weight=1)
G9.add_edge(1350,2756,weight=1)
G9.add_edge(1350,2758,weight=1)
G9.add_edge(1350,2759,weight=1)
G9.add_edge(1351,2187,weight=1)
G9.add_edge(1351,1558,weight=1)
G9.add_edge(1351,1788,weight=1)
G9.add_edge(1351,3099,weight=1)
G9.add_edge(1351,3160,weight=1)
G9.add_edge(1351,3222,weight=1)
G9.add_edge(1352,2134,weight=1)
G9.add_edge(1352,1969,weight=1)
G9.add_edge(1352,2686,weight=1)
G9.add_edge(1352,2907,weight=1)
G9.add_edge(1352,2740,weight=1)
G9.add_edge(1352,2649,weight=1)
G9.add_edge(1352,2039,weight=1)
G9.add_edge(1352,2507,weight=1)
G9.add_edge(1352,2041,weight=1)
G9.add_edge(1352,2127,weight=1)
G9.add_edge(1352,2979,weight=1)
G9.add_edge(1352,2960,weight=1)
G9.add_edge(1352,2853,weight=1)
G9.add_edge(1352,1872,weight=1)
G9.add_edge(1353,1358,weight=1)
G9.add_edge(1354,1871,weight=1)
G9.add_edge(1355,2843,weight=1)
G9.add_edge(1355,2825,weight=1)
G9.add_edge(1355,2509,weight=1)
G9.add_edge(1356,1979,weight=1)
G9.add_edge(1358,3191,weight=1)
G9.add_edge(1358,1473,weight=1)
G9.add_edge(1358,2210,weight=1)
G9.add_edge(1358,1787,weight=1)
G9.add_edge(1358,2865,weight=1)
G9.add_edge(1359,1976,weight=1)
G9.add_edge(1359,1607,weight=1)
G9.add_edge(1359,3013,weight=1)
G9.add_edge(1359,2569,weight=1)
G9.add_edge(1361,1988,weight=1)
G9.add_edge(1361,2254,weight=1)
G9.add_edge(1361,1660,weight=1)
G9.add_edge(1361,2732,weight=1)
G9.add_edge(1362,3208,weight=1)
G9.add_edge(1363,2316,weight=1)
G9.add_edge(1364,2496,weight=1)
G9.add_edge(1364,3240,weight=1)
G9.add_edge(1365,1775,weight=1)
G9.add_edge(1366,1648,weight=1)
G9.add_edge(1368,2803,weight=1)
G9.add_edge(1369,2810,weight=1)
G9.add_edge(1370,1888,weight=1)
G9.add_edge(1370,1616,weight=1)
G9.add_edge(1372,1397,weight=1)
G9.add_edge(1372,2851,weight=1)
G9.add_edge(1373,2643,weight=1)
G9.add_edge(1373,2870,weight=1)
G9.add_edge(1374,1776,weight=1)
G9.add_edge(1374,1394,weight=1)
G9.add_edge(1375,1645,weight=1)
G9.add_edge(1376,2370,weight=1)
G9.add_edge(1376,2419,weight=1)
G9.add_edge(1376,1823,weight=1)
G9.add_edge(1376,2194,weight=1)
G9.add_edge(1377,1481,weight=1)
G9.add_edge(1377,2624,weight=1)
G9.add_edge(1377,1488,weight=1)
G9.add_edge(1378,1602,weight=1)
G9.add_edge(1379,2764,weight=1)
G9.add_edge(1379,1767,weight=1)
G9.add_edge(1380,2764,weight=1)
G9.add_edge(1381,1460,weight=1)
G9.add_edge(1383,3010,weight=1)
G9.add_edge(1383,1528,weight=1)
G9.add_edge(1385,2723,weight=1)
G9.add_edge(1385,1932,weight=1)
G9.add_edge(1386,2828,weight=1)
G9.add_edge(1387,2370,weight=1)
G9.add_edge(1387,3083,weight=1)
G9.add_edge(1389,1435,weight=1)
G9.add_edge(1389,2396,weight=1)
G9.add_edge(1390,1640,weight=1)
G9.add_edge(1390,1890,weight=1)
G9.add_edge(1390,2429,weight=1)
G9.add_edge(1392,2372,weight=1)
G9.add_edge(1392,3190,weight=1)
G9.add_edge(1393,2386,weight=1)
G9.add_edge(1393,2415,weight=1)
G9.add_edge(1394,3212,weight=1)
G9.add_edge(1394,1776,weight=1)
G9.add_edge(1394,1827,weight=1)
G9.add_edge(1394,1893,weight=1)
G9.add_edge(1395,1958,weight=1)
G9.add_edge(1397,2368,weight=1)
G9.add_edge(1397,2716,weight=1)
G9.add_edge(1398,1489,weight=1)
G9.add_edge(1401,1414,weight=1)
G9.add_edge(1401,1739,weight=1)
G9.add_edge(1402,2960,weight=1)
G9.add_edge(1402,1727,weight=1)
G9.add_edge(1402,2296,weight=1)
G9.add_edge(1403,1409,weight=1)
G9.add_edge(1405,1723,weight=1)
G9.add_edge(1405,1738,weight=1)
G9.add_edge(1406,2313,weight=1)
G9.add_edge(1407,2261,weight=1)
G9.add_edge(1408,1433,weight=1)
G9.add_edge(1410,1770,weight=1)
G9.add_edge(1412,3205,weight=1)
G9.add_edge(1413,1446,weight=1)
G9.add_edge(1414,2069,weight=1)
G9.add_edge(1414,1607,weight=1)
G9.add_edge(1414,2753,weight=1)
G9.add_edge(1415,2326,weight=1)
G9.add_edge(1416,2754,weight=1)
G9.add_edge(1416,1850,weight=1)
G9.add_edge(1416,2868,weight=1)
G9.add_edge(1416,2994,weight=1)
G9.add_edge(1417,2604,weight=1)
G9.add_edge(1419,3023,weight=1)
G9.add_edge(1423,1840,weight=1)
G9.add_edge(1424,1731,weight=1)
G9.add_edge(1425,2344,weight=1)
G9.add_edge(1426,3255,weight=1)
G9.add_edge(1426,2327,weight=1)
G9.add_edge(1426,2037,weight=1)
G9.add_edge(1426,2478,weight=1)
G9.add_edge(1427,1761,weight=1)
G9.add_edge(1427,2701,weight=1)
G9.add_edge(1428,1736,weight=1)
G9.add_edge(1429,2522,weight=1)
G9.add_edge(1429,1680,weight=1)
G9.add_edge(1429,1685,weight=1)
G9.add_edge(1429,2626,weight=1)
G9.add_edge(1430,1770,weight=1)
G9.add_edge(1431,2036,weight=1)
G9.add_edge(1435,2177,weight=1)
G9.add_edge(1437,1816,weight=1)
G9.add_edge(1437,2248,weight=1)
G9.add_edge(1437,2814,weight=1)
G9.add_edge(1438,1440,weight=1)
G9.add_edge(1439,3045,weight=1)
G9.add_edge(1441,2149,weight=1)
G9.add_edge(1444,2374,weight=1)
G9.add_edge(1444,1807,weight=1)
G9.add_edge(1444,2240,weight=1)
G9.add_edge(1444,2655,weight=1)
G9.add_edge(1444,2623,weight=1)
G9.add_edge(1444,1893,weight=1)
G9.add_edge(1445,2833,weight=1)
G9.add_edge(1445,2646,weight=1)
G9.add_edge(1447,2053,weight=1)
G9.add_edge(1448,2859,weight=1)
G9.add_edge(1448,1533,weight=1)
G9.add_edge(1449,2907,weight=1)
G9.add_edge(1449,1789,weight=1)
G9.add_edge(1450,2234,weight=1)
G9.add_edge(1450,1668,weight=1)
G9.add_edge(1452,1726,weight=1)
G9.add_edge(1453,1864,weight=1)
G9.add_edge(1453,1867,weight=1)
G9.add_edge(1455,2907,weight=1)
G9.add_edge(1455,1850,weight=1)
G9.add_edge(1455,2514,weight=1)
G9.add_edge(1455,2722,weight=1)
G9.add_edge(1456,1459,weight=1)
G9.add_edge(1457,1610,weight=1)
G9.add_edge(1457,1461,weight=1)
G9.add_edge(1457,1488,weight=1)
G9.add_edge(1457,1672,weight=1)
G9.add_edge(1458,2342,weight=1)
G9.add_edge(1458,2382,weight=1)
G9.add_edge(1458,1459,weight=1)
G9.add_edge(1458,1460,weight=1)
G9.add_edge(1458,1461,weight=1)
G9.add_edge(1459,2382,weight=1)
G9.add_edge(1459,1488,weight=1)
G9.add_edge(1460,2138,weight=1)
G9.add_edge(1462,1463,weight=1)
G9.add_edge(1464,2205,weight=1)
G9.add_edge(1465,1466,weight=1)
G9.add_edge(1465,1468,weight=1)
G9.add_edge(1465,1469,weight=1)
G9.add_edge(1465,2222,weight=1)
G9.add_edge(1465,2124,weight=1)
G9.add_edge(1466,1469,weight=1)
G9.add_edge(1466,2222,weight=1)
G9.add_edge(1466,2124,weight=1)
G9.add_edge(1467,1469,weight=1)
G9.add_edge(1467,2124,weight=1)
G9.add_edge(1468,1469,weight=1)
G9.add_edge(1468,2222,weight=1)
G9.add_edge(1468,2124,weight=1)
G9.add_edge(1469,2620,weight=1)
G9.add_edge(1469,2222,weight=1)
G9.add_edge(1469,2124,weight=1)
G9.add_edge(1470,2222,weight=1)
G9.add_edge(1470,2124,weight=1)
G9.add_edge(1472,2518,weight=1)
G9.add_edge(1472,1477,weight=1)
G9.add_edge(1472,1478,weight=1)
G9.add_edge(1472,2796,weight=1)
G9.add_edge(1474,2134,weight=1)
G9.add_edge(1474,1969,weight=1)
G9.add_edge(1474,2090,weight=1)
G9.add_edge(1474,2686,weight=1)
G9.add_edge(1474,2209,weight=1)
G9.add_edge(1474,2907,weight=1)
G9.add_edge(1474,2127,weight=1)
G9.add_edge(1474,1760,weight=1)
G9.add_edge(1475,2907,weight=1)
G9.add_edge(1475,1754,weight=1)
G9.add_edge(1476,2219,weight=1)
G9.add_edge(1476,1492,weight=1)
G9.add_edge(1476,1493,weight=1)
G9.add_edge(1477,1478,weight=1)
G9.add_edge(1478,2518,weight=1)
G9.add_edge(1478,2796,weight=1)
G9.add_edge(1480,1482,weight=1)
G9.add_edge(1480,1650,weight=1)
G9.add_edge(1481,2831,weight=1)
G9.add_edge(1481,2624,weight=1)
G9.add_edge(1481,1488,weight=1)
G9.add_edge(1484,3034,weight=1)
G9.add_edge(1484,3035,weight=1)
G9.add_edge(1485,1985,weight=1)
G9.add_edge(1485,2525,weight=1)
G9.add_edge(1486,1487,weight=1)
G9.add_edge(1487,2831,weight=1)
G9.add_edge(1487,2668,weight=1)
G9.add_edge(1487,3027,weight=1)
G9.add_edge(1487,2257,weight=1)
G9.add_edge(1487,1488,weight=1)
G9.add_edge(1487,2894,weight=1)
G9.add_edge(1487,1489,weight=1)
G9.add_edge(1488,2309,weight=1)
G9.add_edge(1488,2624,weight=1)
G9.add_edge(1488,3027,weight=1)
G9.add_edge(1488,2257,weight=1)
G9.add_edge(1488,1672,weight=1)
G9.add_edge(1489,2004,weight=1)
G9.add_edge(1490,2189,weight=1)
G9.add_edge(1491,1732,weight=1)
G9.add_edge(1492,1493,weight=1)
G9.add_edge(1494,2455,weight=1)
G9.add_edge(1495,2673,weight=1)
G9.add_edge(1496,3203,weight=1)
G9.add_edge(1497,1769,weight=1)
G9.add_edge(1500,1502,weight=1)
G9.add_edge(1501,3000,weight=1)
G9.add_edge(1501,2058,weight=1)
G9.add_edge(1501,2828,weight=1)
G9.add_edge(1503,1505,weight=1)
G9.add_edge(1506,1507,weight=1)
G9.add_edge(1508,2306,weight=1)
G9.add_edge(1508,1975,weight=1)
G9.add_edge(1509,2089,weight=1)
G9.add_edge(1510,2682,weight=1)
G9.add_edge(1510,1542,weight=1)
G9.add_edge(1510,1552,weight=1)
G9.add_edge(1510,1976,weight=1)
G9.add_edge(1510,1608,weight=1)
G9.add_edge(1510,2313,weight=1)
G9.add_edge(1510,1679,weight=1)
G9.add_edge(1510,1512,weight=1)
G9.add_edge(1510,1740,weight=1)
G9.add_edge(1511,2682,weight=1)
G9.add_edge(1511,2306,weight=1)
G9.add_edge(1511,1975,weight=1)
G9.add_edge(1512,2313,weight=1)
G9.add_edge(1512,1679,weight=1)
G9.add_edge(1512,1740,weight=1)
G9.add_edge(1512,2337,weight=1)
G9.add_edge(1512,1513,weight=1)
G9.add_edge(1513,1976,weight=1)
G9.add_edge(1513,1679,weight=1)
G9.add_edge(1513,1740,weight=1)
G9.add_edge(1514,2850,weight=1)
G9.add_edge(1515,2538,weight=1)
G9.add_edge(1517,3249,weight=1)
G9.add_edge(1518,3249,weight=1)
G9.add_edge(1519,1520,weight=1)
G9.add_edge(1521,2126,weight=1)
G9.add_edge(1522,2898,weight=1)
G9.add_edge(1522,1875,weight=1)
G9.add_edge(1522,2202,weight=1)
G9.add_edge(1522,3158,weight=1)
G9.add_edge(1523,2070,weight=1)
G9.add_edge(1523,2619,weight=1)
G9.add_edge(1524,2411,weight=1)
G9.add_edge(1525,1992,weight=1)
G9.add_edge(1525,2195,weight=1)
G9.add_edge(1526,1621,weight=1)
G9.add_edge(1527,2944,weight=1)
G9.add_edge(1528,2375,weight=1)
G9.add_edge(1528,3010,weight=1)
G9.add_edge(1529,2033,weight=1)
G9.add_edge(1530,1538,weight=1)
G9.add_edge(1531,1833,weight=1)
G9.add_edge(1531,2079,weight=1)
G9.add_edge(1534,2031,weight=1)
G9.add_edge(1536,1850,weight=1)
G9.add_edge(1536,1816,weight=1)
G9.add_edge(1536,3007,weight=1)
G9.add_edge(1537,1538,weight=1)
G9.add_edge(1539,1903,weight=1)
G9.add_edge(1540,1956,weight=1)
G9.add_edge(1541,3065,weight=1)
G9.add_edge(1541,2201,weight=1)
G9.add_edge(1542,2312,weight=1)
G9.add_edge(1542,1642,weight=1)
G9.add_edge(1542,2596,weight=1)
G9.add_edge(1543,2953,weight=1)
G9.add_edge(1543,1562,weight=1)
G9.add_edge(1543,2422,weight=1)
G9.add_edge(1543,2054,weight=1)
G9.add_edge(1544,2907,weight=1)
G9.add_edge(1544,2041,weight=1)
G9.add_edge(1545,2089,weight=1)
G9.add_edge(1545,1851,weight=1)
G9.add_edge(1546,3015,weight=1)
G9.add_edge(1547,2686,weight=1)
G9.add_edge(1547,2907,weight=1)
G9.add_edge(1547,2039,weight=1)
G9.add_edge(1547,2073,weight=1)
G9.add_edge(1547,1566,weight=1)
G9.add_edge(1547,2852,weight=1)
G9.add_edge(1549,1568,weight=1)
G9.add_edge(1552,2836,weight=1)
G9.add_edge(1552,2312,weight=1)
G9.add_edge(1552,1642,weight=1)
G9.add_edge(1552,2596,weight=1)
G9.add_edge(1553,3178,weight=1)
G9.add_edge(1553,2222,weight=1)
G9.add_edge(1555,2826,weight=1)
G9.add_edge(1557,1769,weight=1)
G9.add_edge(1558,3060,weight=1)
G9.add_edge(1558,3061,weight=1)
G9.add_edge(1558,3062,weight=1)
G9.add_edge(1558,3063,weight=1)
G9.add_edge(1558,3064,weight=1)
G9.add_edge(1558,3065,weight=1)
G9.add_edge(1558,2070,weight=1)
G9.add_edge(1558,1788,weight=1)
G9.add_edge(1558,3099,weight=1)
G9.add_edge(1558,3160,weight=1)
G9.add_edge(1558,2517,weight=1)
G9.add_edge(1558,3102,weight=1)
G9.add_edge(1558,2666,weight=1)
G9.add_edge(1559,1906,weight=1)
G9.add_edge(1559,2201,weight=1)
G9.add_edge(1559,2527,weight=1)
G9.add_edge(1559,1639,weight=1)
G9.add_edge(1559,1641,weight=1)
G9.add_edge(1560,2090,weight=1)
G9.add_edge(1560,2686,weight=1)
G9.add_edge(1560,2929,weight=1)
G9.add_edge(1560,2907,weight=1)
G9.add_edge(1560,2039,weight=1)
G9.add_edge(1562,1969,weight=1)
G9.add_edge(1562,2953,weight=1)
G9.add_edge(1562,2510,weight=1)
G9.add_edge(1562,2201,weight=1)
G9.add_edge(1563,2059,weight=1)
G9.add_edge(1563,2907,weight=1)
G9.add_edge(1563,2039,weight=1)
G9.add_edge(1563,2912,weight=1)
G9.add_edge(1563,1982,weight=1)
G9.add_edge(1564,2201,weight=1)
G9.add_edge(1565,1567,weight=1)
G9.add_edge(1566,2073,weight=1)
G9.add_edge(1566,2214,weight=1)
G9.add_edge(1566,2852,weight=1)
G9.add_edge(1567,2089,weight=1)
G9.add_edge(1567,2305,weight=1)
G9.add_edge(1567,1771,weight=1)
G9.add_edge(1567,2961,weight=1)
G9.add_edge(1568,1696,weight=1)
G9.add_edge(1569,2907,weight=1)
G9.add_edge(1569,2039,weight=1)
G9.add_edge(1569,2507,weight=1)
G9.add_edge(1570,2087,weight=1)
G9.add_edge(1570,2958,weight=1)
G9.add_edge(1570,3009,weight=1)
G9.add_edge(1571,2039,weight=1)
G9.add_edge(1571,1851,weight=1)
G9.add_edge(1571,2041,weight=1)
G9.add_edge(1572,3143,weight=1)
G9.add_edge(1572,1852,weight=1)
G9.add_edge(1573,1967,weight=1)
G9.add_edge(1573,2852,weight=1)
G9.add_edge(1574,2754,weight=1)
G9.add_edge(1574,2783,weight=1)
G9.add_edge(1574,1634,weight=1)
G9.add_edge(1574,1654,weight=1)
G9.add_edge(1574,1679,weight=1)
G9.add_edge(1574,1740,weight=1)
G9.add_edge(1575,2219,weight=1)
G9.add_edge(1575,2220,weight=1)
G9.add_edge(1577,2211,weight=1)
G9.add_edge(1577,2201,weight=1)
G9.add_edge(1577,2096,weight=1)
G9.add_edge(1577,1661,weight=1)
G9.add_edge(1578,2187,weight=1)
G9.add_edge(1578,2907,weight=1)
G9.add_edge(1579,2307,weight=1)
G9.add_edge(1579,1872,weight=1)
G9.add_edge(1579,1873,weight=1)
G9.add_edge(1580,2502,weight=1)
G9.add_edge(1580,2686,weight=1)
G9.add_edge(1580,2507,weight=1)
G9.add_edge(1580,3167,weight=1)
G9.add_edge(1580,1984,weight=1)
G9.add_edge(1581,1919,weight=1)
G9.add_edge(1582,3231,weight=1)
G9.add_edge(1582,3260,weight=1)
G9.add_edge(1582,2161,weight=1)
G9.add_edge(1583,2620,weight=1)
G9.add_edge(1583,2222,weight=1)
G9.add_edge(1584,2148,weight=1)
G9.add_edge(1585,1801,weight=1)
G9.add_edge(1585,2805,weight=1)
G9.add_edge(1585,1635,weight=1)
G9.add_edge(1586,1616,weight=1)
G9.add_edge(1588,1603,weight=1)
G9.add_edge(1589,1643,weight=1)
G9.add_edge(1589,1663,weight=1)
G9.add_edge(1591,1640,weight=1)
G9.add_edge(1592,2557,weight=1)
G9.add_edge(1592,1646,weight=1)
G9.add_edge(1594,1657,weight=1)
G9.add_edge(1596,1815,weight=1)
G9.add_edge(1597,2150,weight=1)
G9.add_edge(1597,1615,weight=1)
G9.add_edge(1597,3021,weight=1)
G9.add_edge(1597,1860,weight=1)
G9.add_edge(1598,2159,weight=1)
G9.add_edge(1599,2725,weight=1)
G9.add_edge(1601,2402,weight=1)
G9.add_edge(1601,2935,weight=1)
G9.add_edge(1601,2571,weight=1)
G9.add_edge(1602,2394,weight=1)
G9.add_edge(1602,1774,weight=1)
G9.add_edge(1602,3042,weight=1)
G9.add_edge(1602,1827,weight=1)
G9.add_edge(1603,2219,weight=1)
G9.add_edge(1603,2226,weight=1)
G9.add_edge(1603,1641,weight=1)
G9.add_edge(1603,2033,weight=1)
G9.add_edge(1603,2707,weight=1)
G9.add_edge(1604,3173,weight=1)
G9.add_edge(1604,2227,weight=1)
G9.add_edge(1604,2544,weight=1)
G9.add_edge(1604,1931,weight=1)
G9.add_edge(1604,2459,weight=1)
G9.add_edge(1606,3173,weight=1)
G9.add_edge(1606,2227,weight=1)
G9.add_edge(1606,2544,weight=1)
G9.add_edge(1606,1931,weight=1)
G9.add_edge(1606,2459,weight=1)
G9.add_edge(1607,2543,weight=1)
G9.add_edge(1608,2313,weight=1)
G9.add_edge(1608,1632,weight=1)
G9.add_edge(1608,3214,weight=1)
G9.add_edge(1612,2565,weight=1)
G9.add_edge(1613,1645,weight=1)
G9.add_edge(1613,1659,weight=1)
G9.add_edge(1614,2553,weight=1)
G9.add_edge(1614,2707,weight=1)
G9.add_edge(1616,1888,weight=1)
G9.add_edge(1616,1889,weight=1)
G9.add_edge(1616,1619,weight=1)
G9.add_edge(1618,1673,weight=1)
G9.add_edge(1619,1888,weight=1)
G9.add_edge(1619,2252,weight=1)
G9.add_edge(1620,1906,weight=1)
G9.add_edge(1620,1639,weight=1)
G9.add_edge(1621,2408,weight=1)
G9.add_edge(1622,1645,weight=1)
G9.add_edge(1626,2682,weight=1)
G9.add_edge(1626,2182,weight=1)
G9.add_edge(1626,2183,weight=1)
G9.add_edge(1628,1788,weight=1)
G9.add_edge(1629,2255,weight=1)
G9.add_edge(1630,2913,weight=1)
G9.add_edge(1630,2093,weight=1)
G9.add_edge(1631,2231,weight=1)
G9.add_edge(1633,1653,weight=1)
G9.add_edge(1633,1657,weight=1)
G9.add_edge(1634,2093,weight=1)
G9.add_edge(1634,1664,weight=1)
G9.add_edge(1635,2762,weight=1)
G9.add_edge(1635,2805,weight=1)
G9.add_edge(1635,1643,weight=1)
G9.add_edge(1635,2579,weight=1)
G9.add_edge(1637,1651,weight=1)
G9.add_edge(1637,1665,weight=1)
G9.add_edge(1637,1666,weight=1)
G9.add_edge(1638,3155,weight=1)
G9.add_edge(1639,2527,weight=1)
G9.add_edge(1640,2761,weight=1)
G9.add_edge(1640,2107,weight=1)
G9.add_edge(1641,2522,weight=1)
G9.add_edge(1641,1656,weight=1)
G9.add_edge(1642,2312,weight=1)
G9.add_edge(1642,1995,weight=1)
G9.add_edge(1643,1766,weight=1)
G9.add_edge(1643,2762,weight=1)
G9.add_edge(1643,3153,weight=1)
G9.add_edge(1643,2537,weight=1)
G9.add_edge(1643,2654,weight=1)
G9.add_edge(1643,2076,weight=1)
G9.add_edge(1643,2550,weight=1)
G9.add_edge(1643,2317,weight=1)
G9.add_edge(1643,3134,weight=1)
G9.add_edge(1643,2554,weight=1)
G9.add_edge(1643,2555,weight=1)
G9.add_edge(1643,1663,weight=1)
G9.add_edge(1644,1663,weight=1)
G9.add_edge(1645,2820,weight=1)
G9.add_edge(1645,1659,weight=1)
G9.add_edge(1645,2097,weight=1)
G9.add_edge(1646,2557,weight=1)
G9.add_edge(1647,1948,weight=1)
G9.add_edge(1647,3135,weight=1)
G9.add_edge(1649,2402,weight=1)
G9.add_edge(1649,2935,weight=1)
G9.add_edge(1649,2571,weight=1)
G9.add_edge(1650,2247,weight=1)
G9.add_edge(1651,1665,weight=1)
G9.add_edge(1652,1657,weight=1)
G9.add_edge(1653,1656,weight=1)
G9.add_edge(1654,2093,weight=1)
G9.add_edge(1654,1664,weight=1)
G9.add_edge(1655,2311,weight=1)
G9.add_edge(1655,3074,weight=1)
G9.add_edge(1656,1657,weight=1)
G9.add_edge(1658,1696,weight=1)
G9.add_edge(1660,3054,weight=1)
G9.add_edge(1661,2366,weight=1)
G9.add_edge(1662,2570,weight=1)
G9.add_edge(1663,3242,weight=1)
G9.add_edge(1663,2076,weight=1)
G9.add_edge(1663,2317,weight=1)
G9.add_edge(1664,2093,weight=1)
G9.add_edge(1668,3173,weight=1)
G9.add_edge(1668,2802,weight=1)
G9.add_edge(1668,2695,weight=1)
G9.add_edge(1668,2809,weight=1)
G9.add_edge(1668,2723,weight=1)
G9.add_edge(1668,2697,weight=1)
G9.add_edge(1669,1900,weight=1)
G9.add_edge(1669,2440,weight=1)
G9.add_edge(1671,2821,weight=1)
G9.add_edge(1672,2624,weight=1)
G9.add_edge(1672,2008,weight=1)
G9.add_edge(1673,3041,weight=1)
G9.add_edge(1673,2109,weight=1)
G9.add_edge(1675,2093,weight=1)
G9.add_edge(1675,2645,weight=1)
G9.add_edge(1676,3136,weight=1)
G9.add_edge(1677,2836,weight=1)
G9.add_edge(1677,1678,weight=1)
G9.add_edge(1678,2836,weight=1)
G9.add_edge(1679,2754,weight=1)
G9.add_edge(1680,1685,weight=1)
G9.add_edge(1683,2333,weight=1)
G9.add_edge(1684,3031,weight=1)
G9.add_edge(1686,2366,weight=1)
G9.add_edge(1686,2335,weight=1)
G9.add_edge(1687,3163,weight=1)
G9.add_edge(1688,2264,weight=1)
G9.add_edge(1690,2860,weight=1)
G9.add_edge(1692,2938,weight=1)
G9.add_edge(1693,2619,weight=1)
G9.add_edge(1693,1710,weight=1)
G9.add_edge(1694,1696,weight=1)
G9.add_edge(1695,2264,weight=1)
G9.add_edge(1697,2998,weight=1)
G9.add_edge(1699,2921,weight=1)
G9.add_edge(1701,1733,weight=1)
G9.add_edge(1702,2122,weight=1)
G9.add_edge(1703,1733,weight=1)
G9.add_edge(1704,3137,weight=1)
G9.add_edge(1706,2592,weight=1)
G9.add_edge(1708,2277,weight=1)
G9.add_edge(1708,2133,weight=1)
G9.add_edge(1709,2019,weight=1)
G9.add_edge(1711,3237,weight=1)
G9.add_edge(1712,1737,weight=1)
G9.add_edge(1712,2485,weight=1)
G9.add_edge(1713,1714,weight=1)
G9.add_edge(1715,3043,weight=1)
G9.add_edge(1715,1751,weight=1)
G9.add_edge(1718,2626,weight=1)
G9.add_edge(1722,2706,weight=1)
G9.add_edge(1724,3198,weight=1)
G9.add_edge(1725,1742,weight=1)
G9.add_edge(1727,2296,weight=1)
G9.add_edge(1728,2083,weight=1)
G9.add_edge(1729,2025,weight=1)
G9.add_edge(1730,2590,weight=1)
G9.add_edge(1734,2449,weight=1)
G9.add_edge(1735,2466,weight=1)
G9.add_edge(1735,2190,weight=1)
G9.add_edge(1740,2754,weight=1)
G9.add_edge(1741,2778,weight=1)
G9.add_edge(1741,2779,weight=1)
G9.add_edge(1743,2466,weight=1)
G9.add_edge(1743,2190,weight=1)
G9.add_edge(1746,2463,weight=1)
G9.add_edge(1747,2860,weight=1)
G9.add_edge(1748,2815,weight=1)
G9.add_edge(1749,2114,weight=1)
G9.add_edge(1750,1955,weight=1)
G9.add_edge(1753,2824,weight=1)
G9.add_edge(1754,2485,weight=1)
G9.add_edge(1755,1756,weight=1)
G9.add_edge(1756,1882,weight=1)
G9.add_edge(1758,3182,weight=1)
G9.add_edge(1758,2889,weight=1)
G9.add_edge(1759,2901,weight=1)
G9.add_edge(1759,2283,weight=1)
G9.add_edge(1759,3225,weight=1)
G9.add_edge(1760,2907,weight=1)
G9.add_edge(1760,1916,weight=1)
G9.add_edge(1760,2073,weight=1)
G9.add_edge(1760,2956,weight=1)
G9.add_edge(1762,3106,weight=1)
G9.add_edge(1763,2350,weight=1)
G9.add_edge(1768,2419,weight=1)
G9.add_edge(1769,2672,weight=1)
G9.add_edge(1770,3145,weight=1)
G9.add_edge(1771,3206,weight=1)
G9.add_edge(1772,1788,weight=1)
G9.add_edge(1773,1774,weight=1)
G9.add_edge(1773,1775,weight=1)
G9.add_edge(1773,1776,weight=1)
G9.add_edge(1774,2394,weight=1)
G9.add_edge(1775,1776,weight=1)
G9.add_edge(1777,2339,weight=1)
G9.add_edge(1777,2788,weight=1)
G9.add_edge(1778,2756,weight=1)
G9.add_edge(1778,2758,weight=1)
G9.add_edge(1778,2759,weight=1)
G9.add_edge(1778,1815,weight=1)
G9.add_edge(1778,2713,weight=1)
G9.add_edge(1779,2073,weight=1)
G9.add_edge(1781,2758,weight=1)
G9.add_edge(1781,2759,weight=1)
G9.add_edge(1781,1815,weight=1)
G9.add_edge(1781,2713,weight=1)
G9.add_edge(1782,2620,weight=1)
G9.add_edge(1782,3178,weight=1)
G9.add_edge(1782,1794,weight=1)
G9.add_edge(1783,2264,weight=1)
G9.add_edge(1784,2686,weight=1)
G9.add_edge(1784,2907,weight=1)
G9.add_edge(1785,2513,weight=1)
G9.add_edge(1786,2360,weight=1)
G9.add_edge(1786,2657,weight=1)
G9.add_edge(1786,1790,weight=1)
G9.add_edge(1786,1978,weight=1)
G9.add_edge(1788,2197,weight=1)
G9.add_edge(1788,3064,weight=1)
G9.add_edge(1788,2630,weight=1)
G9.add_edge(1789,3222,weight=1)
G9.add_edge(1789,1978,weight=1)
G9.add_edge(1790,1946,weight=1)
G9.add_edge(1790,1978,weight=1)
G9.add_edge(1790,2041,weight=1)
G9.add_edge(1792,3169,weight=1)
G9.add_edge(1792,3232,weight=1)
G9.add_edge(1792,3233,weight=1)
G9.add_edge(1793,2070,weight=1)
G9.add_edge(1793,2686,weight=1)
G9.add_edge(1793,2907,weight=1)
G9.add_edge(1793,1978,weight=1)
G9.add_edge(1794,2620,weight=1)
G9.add_edge(1795,2996,weight=1)
G9.add_edge(1796,2641,weight=1)
G9.add_edge(1797,2690,weight=1)
G9.add_edge(1798,2513,weight=1)
G9.add_edge(1799,1978,weight=1)
G9.add_edge(1800,2956,weight=1)
G9.add_edge(1800,3217,weight=1)
G9.add_edge(1800,2884,weight=1)
G9.add_edge(1801,2805,weight=1)
G9.add_edge(1802,2525,weight=1)
G9.add_edge(1803,1948,weight=1)
G9.add_edge(1804,1856,weight=1)
G9.add_edge(1804,1859,weight=1)
G9.add_edge(1804,2723,weight=1)
G9.add_edge(1804,1932,weight=1)
G9.add_edge(1805,1948,weight=1)
G9.add_edge(1806,2435,weight=1)
G9.add_edge(1806,3156,weight=1)
G9.add_edge(1808,2402,weight=1)
G9.add_edge(1809,1819,weight=1)
G9.add_edge(1811,2536,weight=1)
G9.add_edge(1811,1825,weight=1)
G9.add_edge(1813,2888,weight=1)
G9.add_edge(1813,2889,weight=1)
G9.add_edge(1814,1923,weight=1)
G9.add_edge(1815,1963,weight=1)
G9.add_edge(1815,2713,weight=1)
G9.add_edge(1816,2661,weight=1)
G9.add_edge(1816,2704,weight=1)
G9.add_edge(1816,2679,weight=1)
G9.add_edge(1817,2423,weight=1)
G9.add_edge(1818,2891,weight=1)
G9.add_edge(1820,2425,weight=1)
G9.add_edge(1821,1959,weight=1)
G9.add_edge(1821,2883,weight=1)
G9.add_edge(1821,2436,weight=1)
G9.add_edge(1821,2437,weight=1)
G9.add_edge(1822,2950,weight=1)
G9.add_edge(1823,2387,weight=1)
G9.add_edge(1824,2176,weight=1)
G9.add_edge(1825,2536,weight=1)
G9.add_edge(1825,2997,weight=1)
G9.add_edge(1829,2744,weight=1)
G9.add_edge(1829,2553,weight=1)
G9.add_edge(1830,3123,weight=1)
G9.add_edge(1831,1911,weight=1)
G9.add_edge(1831,2065,weight=1)
G9.add_edge(1831,2066,weight=1)
G9.add_edge(1831,2674,weight=1)
G9.add_edge(1832,2591,weight=1)
G9.add_edge(1834,2834,weight=1)
G9.add_edge(1835,2782,weight=1)
G9.add_edge(1837,2918,weight=1)
G9.add_edge(1839,2675,weight=1)
G9.add_edge(1841,2191,weight=1)
G9.add_edge(1842,1937,weight=1)
G9.add_edge(1843,3223,weight=1)
G9.add_edge(1844,2480,weight=1)
G9.add_edge(1846,2166,weight=1)
G9.add_edge(1848,2728,weight=1)
G9.add_edge(1849,2105,weight=1)
G9.add_edge(1850,2514,weight=1)
G9.add_edge(1850,2722,weight=1)
G9.add_edge(1850,2695,weight=1)
G9.add_edge(1850,2399,weight=1)
G9.add_edge(1850,2723,weight=1)
G9.add_edge(1850,1932,weight=1)
G9.add_edge(1850,2450,weight=1)
G9.add_edge(1851,2903,weight=1)
G9.add_edge(1851,2305,weight=1)
G9.add_edge(1851,1977,weight=1)
G9.add_edge(1851,1872,weight=1)
G9.add_edge(1851,2961,weight=1)
G9.add_edge(1852,3084,weight=1)
G9.add_edge(1853,3017,weight=1)
G9.add_edge(1854,2714,weight=1)
G9.add_edge(1855,1859,weight=1)
G9.add_edge(1856,2665,weight=1)
G9.add_edge(1856,2565,weight=1)
G9.add_edge(1856,2580,weight=1)
G9.add_edge(1857,2695,weight=1)
G9.add_edge(1857,2253,weight=1)
G9.add_edge(1857,2697,weight=1)
G9.add_edge(1858,2827,weight=1)
G9.add_edge(1858,2575,weight=1)
G9.add_edge(1859,2827,weight=1)
G9.add_edge(1859,2665,weight=1)
G9.add_edge(1859,2565,weight=1)
G9.add_edge(1859,2580,weight=1)
G9.add_edge(1859,2586,weight=1)
G9.add_edge(1860,2072,weight=1)
G9.add_edge(1860,2695,weight=1)
G9.add_edge(1860,2158,weight=1)
G9.add_edge(1860,3079,weight=1)
G9.add_edge(1861,2454,weight=1)
G9.add_edge(1863,2086,weight=1)
G9.add_edge(1864,1867,weight=1)
G9.add_edge(1865,2698,weight=1)
G9.add_edge(1866,2698,weight=1)
G9.add_edge(1868,2698,weight=1)
G9.add_edge(1869,2093,weight=1)
G9.add_edge(1870,2500,weight=1)
G9.add_edge(1871,2895,weight=1)
G9.add_edge(1871,2652,weight=1)
G9.add_edge(1872,3098,weight=1)
G9.add_edge(1872,2209,weight=1)
G9.add_edge(1872,3087,weight=1)
G9.add_edge(1872,2903,weight=1)
G9.add_edge(1872,2307,weight=1)
G9.add_edge(1872,2220,weight=1)
G9.add_edge(1872,2221,weight=1)
G9.add_edge(1872,2804,weight=1)
G9.add_edge(1872,2830,weight=1)
G9.add_edge(1872,2047,weight=1)
G9.add_edge(1873,2134,weight=1)
G9.add_edge(1873,3087,weight=1)
G9.add_edge(1873,1919,weight=1)
G9.add_edge(1873,2182,weight=1)
G9.add_edge(1873,2804,weight=1)
G9.add_edge(1873,2183,weight=1)
G9.add_edge(1873,2941,weight=1)
G9.add_edge(1874,2496,weight=1)
G9.add_edge(1874,3175,weight=1)
G9.add_edge(1875,2514,weight=1)
G9.add_edge(1875,2722,weight=1)
G9.add_edge(1875,2723,weight=1)
G9.add_edge(1875,2055,weight=1)
G9.add_edge(1876,3176,weight=1)
G9.add_edge(1877,2936,weight=1)
G9.add_edge(1878,2583,weight=1)
G9.add_edge(1880,2899,weight=1)
G9.add_edge(1880,1924,weight=1)
G9.add_edge(1881,2790,weight=1)
G9.add_edge(1882,2899,weight=1)
G9.add_edge(1882,1924,weight=1)
G9.add_edge(1882,2523,weight=1)
G9.add_edge(1882,3240,weight=1)
G9.add_edge(1882,2395,weight=1)
G9.add_edge(1883,2737,weight=1)
G9.add_edge(1883,3247,weight=1)
G9.add_edge(1884,3010,weight=1)
G9.add_edge(1884,2560,weight=1)
G9.add_edge(1886,2178,weight=1)
G9.add_edge(1887,2125,weight=1)
G9.add_edge(1888,1889,weight=1)
G9.add_edge(1890,2765,weight=1)
G9.add_edge(1890,3208,weight=1)
G9.add_edge(1891,1892,weight=1)
G9.add_edge(1892,3182,weight=1)
G9.add_edge(1893,2747,weight=1)
G9.add_edge(1893,1895,weight=1)
G9.add_edge(1894,1896,weight=1)
G9.add_edge(1896,2743,weight=1)
G9.add_edge(1897,2138,weight=1)
G9.add_edge(1897,2421,weight=1)
G9.add_edge(1898,3138,weight=1)
G9.add_edge(1898,2627,weight=1)
G9.add_edge(1900,3175,weight=1)
G9.add_edge(1901,2506,weight=1)
G9.add_edge(1901,2215,weight=1)
G9.add_edge(1902,2687,weight=1)
G9.add_edge(1904,3045,weight=1)
G9.add_edge(1905,3016,weight=1)
G9.add_edge(1906,2201,weight=1)
G9.add_edge(1908,2533,weight=1)
G9.add_edge(1909,2878,weight=1)
G9.add_edge(1910,2262,weight=1)
G9.add_edge(1910,3058,weight=1)
G9.add_edge(1910,2780,weight=1)
G9.add_edge(1911,2973,weight=1)
G9.add_edge(1912,2907,weight=1)
G9.add_edge(1913,2073,weight=1)
G9.add_edge(1914,3257,weight=1)
G9.add_edge(1914,3015,weight=1)
G9.add_edge(1914,2907,weight=1)
G9.add_edge(1914,2852,weight=1)
G9.add_edge(1916,2945,weight=1)
G9.add_edge(1917,2146,weight=1)
G9.add_edge(1917,2907,weight=1)
G9.add_edge(1917,2999,weight=1)
G9.add_edge(1918,2498,weight=1)
G9.add_edge(1918,2092,weight=1)
G9.add_edge(1918,2736,weight=1)
G9.add_edge(1918,3132,weight=1)
G9.add_edge(1918,2220,weight=1)
G9.add_edge(1919,2206,weight=1)
G9.add_edge(1919,3018,weight=1)
G9.add_edge(1919,2907,weight=1)
G9.add_edge(1919,2039,weight=1)
G9.add_edge(1919,1974,weight=1)
G9.add_edge(1922,2862,weight=1)
G9.add_edge(1922,2041,weight=1)
G9.add_edge(1922,1983,weight=1)
G9.add_edge(1924,3184,weight=1)
G9.add_edge(1924,3185,weight=1)
G9.add_edge(1925,2057,weight=1)
G9.add_edge(1926,2746,weight=1)
G9.add_edge(1927,2246,weight=1)
G9.add_edge(1927,2096,weight=1)
G9.add_edge(1927,2000,weight=1)
G9.add_edge(1927,2972,weight=1)
G9.add_edge(1928,2809,weight=1)
G9.add_edge(1929,2832,weight=1)
G9.add_edge(1929,2488,weight=1)
G9.add_edge(1930,2563,weight=1)
G9.add_edge(1930,2846,weight=1)
G9.add_edge(1931,3173,weight=1)
G9.add_edge(1932,2723,weight=1)
G9.add_edge(1932,2055,weight=1)
G9.add_edge(1933,2631,weight=1)
G9.add_edge(1933,2633,weight=1)
G9.add_edge(1934,3123,weight=1)
G9.add_edge(1935,2047,weight=1)
G9.add_edge(1938,2939,weight=1)
G9.add_edge(1939,3193,weight=1)
G9.add_edge(1939,2284,weight=1)
G9.add_edge(1940,2648,weight=1)
G9.add_edge(1941,2948,weight=1)
G9.add_edge(1941,3244,weight=1)
G9.add_edge(1944,2064,weight=1)
G9.add_edge(1945,2101,weight=1)
G9.add_edge(1945,3172,weight=1)
G9.add_edge(1946,2657,weight=1)
G9.add_edge(1946,2307,weight=1)
G9.add_edge(1946,1947,weight=1)
G9.add_edge(1948,2692,weight=1)
G9.add_edge(1948,2694,weight=1)
G9.add_edge(1948,1949,weight=1)
G9.add_edge(1948,2688,weight=1)
G9.add_edge(1948,2645,weight=1)
G9.add_edge(1949,2688,weight=1)
G9.add_edge(1950,1953,weight=1)
G9.add_edge(1950,2669,weight=1)
G9.add_edge(1951,2104,weight=1)
G9.add_edge(1953,3068,weight=1)
G9.add_edge(1954,3218,weight=1)
G9.add_edge(1957,3008,weight=1)
G9.add_edge(1957,2782,weight=1)
G9.add_edge(1958,3117,weight=1)
G9.add_edge(1958,3258,weight=1)
G9.add_edge(1959,2426,weight=1)
G9.add_edge(1959,2436,weight=1)
G9.add_edge(1960,3256,weight=1)
G9.add_edge(1960,2987,weight=1)
G9.add_edge(1960,3227,weight=1)
G9.add_edge(1961,2539,weight=1)
G9.add_edge(1962,2403,weight=1)
G9.add_edge(1964,2833,weight=1)
G9.add_edge(1964,2646,weight=1)
G9.add_edge(1965,2786,weight=1)
G9.add_edge(1965,2595,weight=1)
G9.add_edge(1967,2305,weight=1)
G9.add_edge(1967,2556,weight=1)
G9.add_edge(1968,2907,weight=1)
G9.add_edge(1969,2089,weight=1)
G9.add_edge(1969,2209,weight=1)
G9.add_edge(1969,2835,weight=1)
G9.add_edge(1971,2682,weight=1)
G9.add_edge(1971,2689,weight=1)
G9.add_edge(1971,2041,weight=1)
G9.add_edge(1971,2691,weight=1)
G9.add_edge(1973,2134,weight=1)
G9.add_edge(1973,2686,weight=1)
G9.add_edge(1973,2907,weight=1)
G9.add_edge(1973,2039,weight=1)
G9.add_edge(1973,3087,weight=1)
G9.add_edge(1973,3222,weight=1)
G9.add_edge(1975,2682,weight=1)
G9.add_edge(1975,2498,weight=1)
G9.add_edge(1975,2907,weight=1)
G9.add_edge(1975,3075,weight=1)
G9.add_edge(1975,2360,weight=1)
G9.add_edge(1978,2689,weight=1)
G9.add_edge(1978,2862,weight=1)
G9.add_edge(1978,2147,weight=1)
G9.add_edge(1978,2907,weight=1)
G9.add_edge(1978,3087,weight=1)
G9.add_edge(1978,2360,weight=1)
G9.add_edge(1978,2657,weight=1)
G9.add_edge(1978,2307,weight=1)
G9.add_edge(1978,2641,weight=1)
G9.add_edge(1978,3222,weight=1)
G9.add_edge(1978,2691,weight=1)
G9.add_edge(1978,2362,weight=1)
G9.add_edge(1980,2907,weight=1)
G9.add_edge(1982,2039,weight=1)
G9.add_edge(1983,2090,weight=1)
G9.add_edge(1983,2907,weight=1)
G9.add_edge(1983,2945,weight=1)
G9.add_edge(1983,2073,weight=1)
G9.add_edge(1983,2979,weight=1)
G9.add_edge(1983,3216,weight=1)
G9.add_edge(1984,2993,weight=1)
G9.add_edge(1984,2686,weight=1)
G9.add_edge(1986,2858,weight=1)
G9.add_edge(1986,2026,weight=1)
G9.add_edge(1987,2417,weight=1)
G9.add_edge(1987,2123,weight=1)
G9.add_edge(1989,3185,weight=1)
G9.add_edge(1989,2043,weight=1)
G9.add_edge(1990,2095,weight=1)
G9.add_edge(1990,2033,weight=1)
G9.add_edge(1991,2854,weight=1)
G9.add_edge(1992,2368,weight=1)
G9.add_edge(1992,2716,weight=1)
G9.add_edge(1992,2195,weight=1)
G9.add_edge(1992,2814,weight=1)
G9.add_edge(1993,3155,weight=1)
G9.add_edge(1993,2823,weight=1)
G9.add_edge(1994,2997,weight=1)
G9.add_edge(1997,2403,weight=1)
G9.add_edge(1998,2553,weight=1)
G9.add_edge(1999,2159,weight=1)
G9.add_edge(2000,2096,weight=1)
G9.add_edge(2001,2820,weight=1)
G9.add_edge(2001,2097,weight=1)
G9.add_edge(2002,2078,weight=1)
G9.add_edge(2003,2007,weight=1)
G9.add_edge(2005,2806,weight=1)
G9.add_edge(2005,2810,weight=1)
G9.add_edge(2005,2256,weight=1)
G9.add_edge(2009,3091,weight=1)
G9.add_edge(2010,2011,weight=1)
G9.add_edge(2012,3196,weight=1)
G9.add_edge(2013,2324,weight=1)
G9.add_edge(2014,2024,weight=1)
G9.add_edge(2014,2029,weight=1)
G9.add_edge(2015,2024,weight=1)
G9.add_edge(2017,2271,weight=1)
G9.add_edge(2018,3146,weight=1)
G9.add_edge(2021,3201,weight=1)
G9.add_edge(2022,2023,weight=1)
G9.add_edge(2024,2699,weight=1)
G9.add_edge(2024,3044,weight=1)
G9.add_edge(2024,2029,weight=1)
G9.add_edge(2024,2338,weight=1)
G9.add_edge(2027,2575,weight=1)
G9.add_edge(2031,3051,weight=1)
G9.add_edge(2033,2549,weight=1)
G9.add_edge(2033,2095,weight=1)
G9.add_edge(2034,2974,weight=1)
G9.add_edge(2034,2452,weight=1)
G9.add_edge(2036,2658,weight=1)
G9.add_edge(2037,3255,weight=1)
G9.add_edge(2037,3128,weight=1)
G9.add_edge(2037,2327,weight=1)
G9.add_edge(2037,3129,weight=1)
G9.add_edge(2038,3191,weight=1)
G9.add_edge(2039,2134,weight=1)
G9.add_edge(2039,2197,weight=1)
G9.add_edge(2039,3231,weight=1)
G9.add_edge(2039,2502,weight=1)
G9.add_edge(2039,2209,weight=1)
G9.add_edge(2039,3015,weight=1)
G9.add_edge(2039,2198,weight=1)
G9.add_edge(2039,2930,weight=1)
G9.add_edge(2039,3087,weight=1)
G9.add_edge(2039,2507,weight=1)
G9.add_edge(2039,2912,weight=1)
G9.add_edge(2039,2149,weight=1)
G9.add_edge(2039,2641,weight=1)
G9.add_edge(2039,3206,weight=1)
G9.add_edge(2039,3222,weight=1)
G9.add_edge(2039,3167,weight=1)
G9.add_edge(2040,2686,weight=1)
G9.add_edge(2041,2187,weight=1)
G9.add_edge(2041,2682,weight=1)
G9.add_edge(2041,3062,weight=1)
G9.add_edge(2041,2689,weight=1)
G9.add_edge(2041,3063,weight=1)
G9.add_edge(2041,3064,weight=1)
G9.add_edge(2041,3065,weight=1)
G9.add_edge(2041,2861,weight=1)
G9.add_edge(2041,2059,weight=1)
G9.add_edge(2041,2307,weight=1)
G9.add_edge(2041,2641,weight=1)
G9.add_edge(2041,3206,weight=1)
G9.add_edge(2041,3222,weight=1)
G9.add_edge(2041,2691,weight=1)
G9.add_edge(2041,2362,weight=1)
G9.add_edge(2041,2517,weight=1)
G9.add_edge(2041,2218,weight=1)
G9.add_edge(2041,3101,weight=1)
G9.add_edge(2041,3102,weight=1)
G9.add_edge(2041,2630,weight=1)
G9.add_edge(2042,3213,weight=1)
G9.add_edge(2043,3240,weight=1)
G9.add_edge(2043,3190,weight=1)
G9.add_edge(2044,2754,weight=1)
G9.add_edge(2044,2556,weight=1)
G9.add_edge(2045,3213,weight=1)
G9.add_edge(2046,2645,weight=1)
G9.add_edge(2048,2338,weight=1)
G9.add_edge(2048,3262,weight=1)
G9.add_edge(2049,2741,weight=1)
G9.add_edge(2049,2278,weight=1)
G9.add_edge(2050,2278,weight=1)
G9.add_edge(2052,2675,weight=1)
G9.add_edge(2054,2637,weight=1)
G9.add_edge(2055,2723,weight=1)
G9.add_edge(2055,3007,weight=1)
G9.add_edge(2055,2143,weight=1)
G9.add_edge(2056,2057,weight=1)
G9.add_edge(2058,3000,weight=1)
G9.add_edge(2059,2930,weight=1)
G9.add_edge(2059,2641,weight=1)
G9.add_edge(2059,3206,weight=1)
G9.add_edge(2059,2127,weight=1)
G9.add_edge(2060,2843,weight=1)
G9.add_edge(2063,2809,weight=1)
G9.add_edge(2067,2702,weight=1)
G9.add_edge(2069,2220,weight=1)
G9.add_edge(2070,3062,weight=1)
G9.add_edge(2070,3064,weight=1)
G9.add_edge(2070,3099,weight=1)
G9.add_edge(2070,3222,weight=1)
G9.add_edge(2070,2517,weight=1)
G9.add_edge(2070,2630,weight=1)
G9.add_edge(2070,2715,weight=1)
G9.add_edge(2070,2688,weight=1)
G9.add_edge(2071,2073,weight=1)
G9.add_edge(2071,2700,weight=1)
G9.add_edge(2071,2082,weight=1)
G9.add_edge(2072,2955,weight=1)
G9.add_edge(2072,3011,weight=1)
G9.add_edge(2073,2907,weight=1)
G9.add_edge(2073,2361,weight=1)
G9.add_edge(2073,2149,weight=1)
G9.add_edge(2073,2956,weight=1)
G9.add_edge(2074,2075,weight=1)
G9.add_edge(2076,3153,weight=1)
G9.add_edge(2077,2953,weight=1)
G9.add_edge(2081,2571,weight=1)
G9.add_edge(2081,3096,weight=1)
G9.add_edge(2082,2700,weight=1)
G9.add_edge(2084,2316,weight=1)
G9.add_edge(2085,3186,weight=1)
G9.add_edge(2085,2245,weight=1)
G9.add_edge(2085,2560,weight=1)
G9.add_edge(2086,2170,weight=1)
G9.add_edge(2088,2092,weight=1)
G9.add_edge(2088,2999,weight=1)
G9.add_edge(2088,2093,weight=1)
G9.add_edge(2088,3100,weight=1)
G9.add_edge(2089,2092,weight=1)
G9.add_edge(2089,3076,weight=1)
G9.add_edge(2089,2511,weight=1)
G9.add_edge(2089,2221,weight=1)
G9.add_edge(2089,2310,weight=1)
G9.add_edge(2090,2092,weight=1)
G9.add_edge(2090,3222,weight=1)
G9.add_edge(2091,2754,weight=1)
G9.add_edge(2091,2497,weight=1)
G9.add_edge(2091,2508,weight=1)
G9.add_edge(2091,2999,weight=1)
G9.add_edge(2091,3100,weight=1)
G9.add_edge(2091,2220,weight=1)
G9.add_edge(2091,2942,weight=1)
G9.add_edge(2092,2754,weight=1)
G9.add_edge(2092,2497,weight=1)
G9.add_edge(2092,2508,weight=1)
G9.add_edge(2092,2999,weight=1)
G9.add_edge(2092,2306,weight=1)
G9.add_edge(2092,3222,weight=1)
G9.add_edge(2092,2201,weight=1)
G9.add_edge(2092,2093,weight=1)
G9.add_edge(2092,3100,weight=1)
G9.add_edge(2092,2220,weight=1)
G9.add_edge(2092,2962,weight=1)
G9.add_edge(2092,2607,weight=1)
G9.add_edge(2092,2942,weight=1)
G9.add_edge(2093,2201,weight=1)
G9.add_edge(2093,2783,weight=1)
G9.add_edge(2093,2230,weight=1)
G9.add_edge(2093,2417,weight=1)
G9.add_edge(2094,2748,weight=1)
G9.add_edge(2096,2438,weight=1)
G9.add_edge(2096,2972,weight=1)
G9.add_edge(2097,2820,weight=1)
G9.add_edge(2099,2102,weight=1)
G9.add_edge(2103,3219,weight=1)
G9.add_edge(2107,2761,weight=1)
G9.add_edge(2108,2934,weight=1)
G9.add_edge(2109,2496,weight=1)
G9.add_edge(2109,3180,weight=1)
G9.add_edge(2112,2115,weight=1)
G9.add_edge(2116,2745,weight=1)
G9.add_edge(2117,2590,weight=1)
G9.add_edge(2118,2119,weight=1)
G9.add_edge(2121,2787,weight=1)
G9.add_edge(2123,2417,weight=1)
G9.add_edge(2124,2848,weight=1)
G9.add_edge(2127,2930,weight=1)
G9.add_edge(2127,2959,weight=1)
G9.add_edge(2129,2366,weight=1)
G9.add_edge(2129,2318,weight=1)
G9.add_edge(2129,3230,weight=1)
G9.add_edge(2131,2994,weight=1)
G9.add_edge(2132,2896,weight=1)
G9.add_edge(2134,2863,weight=1)
G9.add_edge(2134,2209,weight=1)
G9.add_edge(2134,2907,weight=1)
G9.add_edge(2134,2930,weight=1)
G9.add_edge(2134,2304,weight=1)
G9.add_edge(2134,3087,weight=1)
G9.add_edge(2134,2507,weight=1)
G9.add_edge(2134,2149,weight=1)
G9.add_edge(2135,2201,weight=1)
G9.add_edge(2136,2319,weight=1)
G9.add_edge(2139,2215,weight=1)
G9.add_edge(2139,2881,weight=1)
G9.add_edge(2139,2884,weight=1)
G9.add_edge(2141,2681,weight=1)
G9.add_edge(2142,2676,weight=1)
G9.add_edge(2143,2851,weight=1)
G9.add_edge(2146,3000,weight=1)
G9.add_edge(2147,2187,weight=1)
G9.add_edge(2147,2862,weight=1)
G9.add_edge(2147,2907,weight=1)
G9.add_edge(2147,2153,weight=1)
G9.add_edge(2148,2187,weight=1)
G9.add_edge(2148,2907,weight=1)
G9.add_edge(2149,2863,weight=1)
G9.add_edge(2149,2907,weight=1)
G9.add_edge(2149,2507,weight=1)
G9.add_edge(2150,2945,weight=1)
G9.add_edge(2150,3079,weight=1)
G9.add_edge(2151,2907,weight=1)
G9.add_edge(2151,2152,weight=1)
G9.add_edge(2151,2224,weight=1)
G9.add_edge(2153,2360,weight=1)
G9.add_edge(2155,2400,weight=1)
G9.add_edge(2155,2162,weight=1)
G9.add_edge(2157,3109,weight=1)
G9.add_edge(2157,2580,weight=1)
G9.add_edge(2160,2316,weight=1)
G9.add_edge(2161,3260,weight=1)
G9.add_edge(2163,2585,weight=1)
G9.add_edge(2164,2165,weight=1)
G9.add_edge(2167,2364,weight=1)
G9.add_edge(2167,3049,weight=1)
G9.add_edge(2168,2173,weight=1)
G9.add_edge(2169,2170,weight=1)
G9.add_edge(2169,2171,weight=1)
G9.add_edge(2170,2171,weight=1)
G9.add_edge(2170,2316,weight=1)
G9.add_edge(2172,2477,weight=1)
G9.add_edge(2174,3049,weight=1)
G9.add_edge(2175,2477,weight=1)
G9.add_edge(2176,2192,weight=1)
G9.add_edge(2176,2194,weight=1)
G9.add_edge(2179,2180,weight=1)
G9.add_edge(2183,2804,weight=1)
G9.add_edge(2184,2185,weight=1)
G9.add_edge(2186,2785,weight=1)
G9.add_edge(2187,2864,weight=1)
G9.add_edge(2187,2907,weight=1)
G9.add_edge(2187,2867,weight=1)
G9.add_edge(2187,2641,weight=1)
G9.add_edge(2188,2754,weight=1)
G9.add_edge(2191,2707,weight=1)
G9.add_edge(2193,2486,weight=1)
G9.add_edge(2195,2368,weight=1)
G9.add_edge(2195,2716,weight=1)
G9.add_edge(2197,2862,weight=1)
G9.add_edge(2197,2907,weight=1)
G9.add_edge(2197,2198,weight=1)
G9.add_edge(2197,3222,weight=1)
G9.add_edge(2198,2907,weight=1)
G9.add_edge(2199,2741,weight=1)
G9.add_edge(2199,2744,weight=1)
G9.add_edge(2199,2553,weight=1)
G9.add_edge(2199,2970,weight=1)
G9.add_edge(2200,2269,weight=1)
G9.add_edge(2201,2754,weight=1)
G9.add_edge(2201,2356,weight=1)
G9.add_edge(2201,2366,weight=1)
G9.add_edge(2202,2660,weight=1)
G9.add_edge(2202,3158,weight=1)
G9.add_edge(2204,2693,weight=1)
G9.add_edge(2206,2246,weight=1)
G9.add_edge(2207,2741,weight=1)
G9.add_edge(2208,3075,weight=1)
G9.add_edge(2208,2641,weight=1)
G9.add_edge(2208,3206,weight=1)
G9.add_edge(2209,2907,weight=1)
G9.add_edge(2209,2507,weight=1)
G9.add_edge(2209,2979,weight=1)
G9.add_edge(2209,2960,weight=1)
G9.add_edge(2209,2853,weight=1)
G9.add_edge(2212,2686,weight=1)
G9.add_edge(2212,2912,weight=1)
G9.add_edge(2212,3075,weight=1)
G9.add_edge(2214,2686,weight=1)
G9.add_edge(2214,2852,weight=1)
G9.add_edge(2215,2884,weight=1)
G9.add_edge(2216,2928,weight=1)
G9.add_edge(2216,2686,weight=1)
G9.add_edge(2216,2907,weight=1)
G9.add_edge(2216,2507,weight=1)
G9.add_edge(2216,3115,weight=1)
G9.add_edge(2216,2515,weight=1)
G9.add_edge(2216,2852,weight=1)
G9.add_edge(2217,2448,weight=1)
G9.add_edge(2218,3060,weight=1)
G9.add_edge(2218,3061,weight=1)
G9.add_edge(2218,3062,weight=1)
G9.add_edge(2218,2708,weight=1)
G9.add_edge(2218,3063,weight=1)
G9.add_edge(2218,3064,weight=1)
G9.add_edge(2218,3065,weight=1)
G9.add_edge(2218,3222,weight=1)
G9.add_edge(2218,2517,weight=1)
G9.add_edge(2218,3036,weight=1)
G9.add_edge(2218,3102,weight=1)
G9.add_edge(2218,3066,weight=1)
G9.add_edge(2218,2630,weight=1)
G9.add_edge(2218,2688,weight=1)
G9.add_edge(2219,2220,weight=1)
G9.add_edge(2220,2907,weight=1)
G9.add_edge(2220,2641,weight=1)
G9.add_edge(2220,2366,weight=1)
G9.add_edge(2221,3076,weight=1)
G9.add_edge(2221,2310,weight=1)
G9.add_edge(2222,2300,weight=1)
G9.add_edge(2223,2907,weight=1)
G9.add_edge(2224,2367,weight=1)
G9.add_edge(2225,2299,weight=1)
G9.add_edge(2225,3241,weight=1)
G9.add_edge(2225,2340,weight=1)
G9.add_edge(2227,2814,weight=1)
G9.add_edge(2228,2624,weight=1)
G9.add_edge(2228,2259,weight=1)
G9.add_edge(2229,2951,weight=1)
G9.add_edge(2230,2235,weight=1)
G9.add_edge(2230,2668,weight=1)
G9.add_edge(2230,2582,weight=1)
G9.add_edge(2232,3154,weight=1)
G9.add_edge(2235,2582,weight=1)
G9.add_edge(2236,2582,weight=1)
G9.add_edge(2238,2915,weight=1)
G9.add_edge(2239,2947,weight=1)
G9.add_edge(2242,2316,weight=1)
G9.add_edge(2243,2404,weight=1)
G9.add_edge(2244,3174,weight=1)
G9.add_edge(2244,2249,weight=1)
G9.add_edge(2245,2915,weight=1)
G9.add_edge(2245,2253,weight=1)
G9.add_edge(2246,2972,weight=1)
G9.add_edge(2246,2258,weight=1)
G9.add_edge(2248,2827,weight=1)
G9.add_edge(2248,2665,weight=1)
G9.add_edge(2249,2512,weight=1)
G9.add_edge(2249,2529,weight=1)
G9.add_edge(2249,2382,weight=1)
G9.add_edge(2249,2774,weight=1)
G9.add_edge(2250,2642,weight=1)
G9.add_edge(2251,2414,weight=1)
G9.add_edge(2251,2846,weight=1)
G9.add_edge(2253,3039,weight=1)
G9.add_edge(2253,2695,weight=1)
G9.add_edge(2254,3054,weight=1)
G9.add_edge(2255,2406,weight=1)
G9.add_edge(2255,2668,weight=1)
G9.add_edge(2257,2997,weight=1)
G9.add_edge(2257,2624,weight=1)
G9.add_edge(2257,3027,weight=1)
G9.add_edge(2261,3095,weight=1)
G9.add_edge(2262,3058,weight=1)
G9.add_edge(2263,2495,weight=1)
G9.add_edge(2264,3245,weight=1)
G9.add_edge(2265,2446,weight=1)
G9.add_edge(2267,3264,weight=1)
G9.add_edge(2268,2584,weight=1)
G9.add_edge(2269,2647,weight=1)
G9.add_edge(2270,2952,weight=1)
G9.add_edge(2271,2786,weight=1)
G9.add_edge(2272,2726,weight=1)
G9.add_edge(2272,2910,weight=1)
G9.add_edge(2273,3264,weight=1)
G9.add_edge(2274,2874,weight=1)
G9.add_edge(2274,2974,weight=1)
G9.add_edge(2274,2295,weight=1)
G9.add_edge(2277,2792,weight=1)
G9.add_edge(2280,3070,weight=1)
G9.add_edge(2280,2988,weight=1)
G9.add_edge(2282,3177,weight=1)
G9.add_edge(2283,2901,weight=1)
G9.add_edge(2283,3220,weight=1)
G9.add_edge(2283,3225,weight=1)
G9.add_edge(2285,2340,weight=1)
G9.add_edge(2287,2638,weight=1)
G9.add_edge(2287,2298,weight=1)
G9.add_edge(2288,3164,weight=1)
G9.add_edge(2289,2876,weight=1)
G9.add_edge(2290,2328,weight=1)
G9.add_edge(2292,2479,weight=1)
G9.add_edge(2294,2781,weight=1)
G9.add_edge(2294,2952,weight=1)
G9.add_edge(2295,2974,weight=1)
G9.add_edge(2297,2937,weight=1)
G9.add_edge(2297,2609,weight=1)
G9.add_edge(2298,2638,weight=1)
G9.add_edge(2299,3241,weight=1)
G9.add_edge(2299,3028,weight=1)
G9.add_edge(2299,2340,weight=1)
G9.add_edge(2304,2686,weight=1)
G9.add_edge(2304,2907,weight=1)
G9.add_edge(2306,2468,weight=1)
G9.add_edge(2307,2689,weight=1)
G9.add_edge(2307,2907,weight=1)
G9.add_edge(2307,2691,weight=1)
G9.add_edge(2310,3076,weight=1)
G9.add_edge(2315,3003,weight=1)
G9.add_edge(2320,2622,weight=1)
G9.add_edge(2321,2933,weight=1)
G9.add_edge(2322,2428,weight=1)
G9.add_edge(2323,3260,weight=1)
G9.add_edge(2325,2922,weight=1)
G9.add_edge(2329,2816,weight=1)
G9.add_edge(2330,2674,weight=1)
G9.add_edge(2334,2487,weight=1)
G9.add_edge(2335,2736,weight=1)
G9.add_edge(2335,2366,weight=1)
G9.add_edge(2335,2625,weight=1)
G9.add_edge(2337,3215,weight=1)
G9.add_edge(2339,2770,weight=1)
G9.add_edge(2340,3028,weight=1)
G9.add_edge(2342,2382,weight=1)
G9.add_edge(2343,2653,weight=1)
G9.add_edge(2345,2798,weight=1)
G9.add_edge(2346,3222,weight=1)
G9.add_edge(2346,3239,weight=1)
G9.add_edge(2347,2730,weight=1)
G9.add_edge(2353,2709,weight=1)
G9.add_edge(2354,2355,weight=1)
G9.add_edge(2357,2945,weight=1)
G9.add_edge(2360,2862,weight=1)
G9.add_edge(2360,3075,weight=1)
G9.add_edge(2360,2657,weight=1)
G9.add_edge(2360,3222,weight=1)
G9.add_edge(2362,2907,weight=1)
G9.add_edge(2362,2641,weight=1)
G9.add_edge(2362,2852,weight=1)
G9.add_edge(2366,3222,weight=1)
G9.add_edge(2366,2829,weight=1)
G9.add_edge(2366,2931,weight=1)
G9.add_edge(2366,3230,weight=1)
G9.add_edge(2366,2919,weight=1)
G9.add_edge(2368,2695,weight=1)
G9.add_edge(2369,2827,weight=1)
G9.add_edge(2369,3130,weight=1)
G9.add_edge(2370,3083,weight=1)
G9.add_edge(2370,2427,weight=1)
G9.add_edge(2371,2531,weight=1)
G9.add_edge(2373,2441,weight=1)
G9.add_edge(2374,3212,weight=1)
G9.add_edge(2377,2633,weight=1)
G9.add_edge(2379,3086,weight=1)
G9.add_edge(2383,2413,weight=1)
G9.add_edge(2384,2806,weight=1)
G9.add_edge(2384,2810,weight=1)
G9.add_edge(2384,3047,weight=1)
G9.add_edge(2385,3037,weight=1)
G9.add_edge(2386,2424,weight=1)
G9.add_edge(2391,2662,weight=1)
G9.add_edge(2391,2892,weight=1)
G9.add_edge(2391,2667,weight=1)
G9.add_edge(2394,3055,weight=1)
G9.add_edge(2397,2580,weight=1)
G9.add_edge(2399,2723,weight=1)
G9.add_edge(2399,3151,weight=1)
G9.add_edge(2402,3086,weight=1)
G9.add_edge(2405,2763,weight=1)
G9.add_edge(2407,2558,weight=1)
G9.add_edge(2410,2496,weight=1)
G9.add_edge(2410,3180,weight=1)
G9.add_edge(2410,3175,weight=1)
G9.add_edge(2415,2424,weight=1)
G9.add_edge(2418,2419,weight=1)
G9.add_edge(2419,2741,weight=1)
G9.add_edge(2420,3022,weight=1)
G9.add_edge(2427,3083,weight=1)
G9.add_edge(2429,3140,weight=1)
G9.add_edge(2431,2432,weight=1)
G9.add_edge(2432,2434,weight=1)
G9.add_edge(2436,3089,weight=1)
G9.add_edge(2436,2437,weight=1)
G9.add_edge(2437,2985,weight=1)
G9.add_edge(2439,2983,weight=1)
G9.add_edge(2440,2496,weight=1)
G9.add_edge(2443,2915,weight=1)
G9.add_edge(2444,2642,weight=1)
G9.add_edge(2445,3239,weight=1)
G9.add_edge(2445,2703,weight=1)
G9.add_edge(2446,3107,weight=1)
G9.add_edge(2447,3171,weight=1)
G9.add_edge(2450,2459,weight=1)
G9.add_edge(2450,3151,weight=1)
G9.add_edge(2454,2460,weight=1)
G9.add_edge(2455,2859,weight=1)
G9.add_edge(2456,2458,weight=1)
G9.add_edge(2457,2458,weight=1)
G9.add_edge(2459,3151,weight=1)
G9.add_edge(2461,2734,weight=1)
G9.add_edge(2462,2940,weight=1)
G9.add_edge(2465,2644,weight=1)
G9.add_edge(2465,2488,weight=1)
G9.add_edge(2467,2750,weight=1)
G9.add_edge(2469,2677,weight=1)
G9.add_edge(2470,3157,weight=1)
G9.add_edge(2471,2782,weight=1)
G9.add_edge(2473,2877,weight=1)
G9.add_edge(2474,3171,weight=1)
G9.add_edge(2475,2942,weight=1)
G9.add_edge(2475,2616,weight=1)
G9.add_edge(2480,2923,weight=1)
G9.add_edge(2481,3120,weight=1)
G9.add_edge(2482,2977,weight=1)
G9.add_edge(2483,2484,weight=1)
G9.add_edge(2483,3111,weight=1)
G9.add_edge(2484,3111,weight=1)
G9.add_edge(2488,2644,weight=1)
G9.add_edge(2491,2619,weight=1)
G9.add_edge(2492,3105,weight=1)
G9.add_edge(2494,3094,weight=1)
G9.add_edge(2496,2547,weight=1)
G9.add_edge(2496,3175,weight=1)
G9.add_edge(2497,3191,weight=1)
G9.add_edge(2498,2686,weight=1)
G9.add_edge(2499,2686,weight=1)
G9.add_edge(2499,2907,weight=1)
G9.add_edge(2499,2619,weight=1)
G9.add_edge(2499,2507,weight=1)
G9.add_edge(2500,2770,weight=1)
G9.add_edge(2500,2839,weight=1)
G9.add_edge(2502,2907,weight=1)
G9.add_edge(2502,2507,weight=1)
G9.add_edge(2502,3075,weight=1)
G9.add_edge(2503,2729,weight=1)
G9.add_edge(2504,2656,weight=1)
G9.add_edge(2505,3064,weight=1)
G9.add_edge(2507,3257,weight=1)
G9.add_edge(2507,2907,weight=1)
G9.add_edge(2507,3087,weight=1)
G9.add_edge(2507,2912,weight=1)
G9.add_edge(2507,2956,weight=1)
G9.add_edge(2507,3167,weight=1)
G9.add_edge(2508,2907,weight=1)
G9.add_edge(2509,2825,weight=1)
G9.add_edge(2512,3221,weight=1)
G9.add_edge(2512,3069,weight=1)
G9.add_edge(2513,3029,weight=1)
G9.add_edge(2515,3115,weight=1)
G9.add_edge(2516,2907,weight=1)
G9.add_edge(2516,2649,weight=1)
G9.add_edge(2516,2800,weight=1)
G9.add_edge(2517,3060,weight=1)
G9.add_edge(2517,3062,weight=1)
G9.add_edge(2517,2708,weight=1)
G9.add_edge(2517,3063,weight=1)
G9.add_edge(2517,3064,weight=1)
G9.add_edge(2517,3065,weight=1)
G9.add_edge(2517,3036,weight=1)
G9.add_edge(2517,3102,weight=1)
G9.add_edge(2517,2630,weight=1)
G9.add_edge(2519,2798,weight=1)
G9.add_edge(2523,2899,weight=1)
G9.add_edge(2524,2547,weight=1)
G9.add_edge(2526,2564,weight=1)
G9.add_edge(2528,3010,weight=1)
G9.add_edge(2528,2560,weight=1)
G9.add_edge(2533,2905,weight=1)
G9.add_edge(2535,2810,weight=1)
G9.add_edge(2537,3134,weight=1)
G9.add_edge(2537,2554,weight=1)
G9.add_edge(2537,2555,weight=1)
G9.add_edge(2538,3185,weight=1)
G9.add_edge(2540,3067,weight=1)
G9.add_edge(2542,3240,weight=1)
G9.add_edge(2543,3005,weight=1)
G9.add_edge(2544,3173,weight=1)
G9.add_edge(2544,3151,weight=1)
G9.add_edge(2545,2934,weight=1)
G9.add_edge(2553,2735,weight=1)
G9.add_edge(2553,2696,weight=1)
G9.add_edge(2554,3153,weight=1)
G9.add_edge(2554,3134,weight=1)
G9.add_edge(2554,2555,weight=1)
G9.add_edge(2555,3153,weight=1)
G9.add_edge(2555,3134,weight=1)
G9.add_edge(2559,3119,weight=1)
G9.add_edge(2559,3123,weight=1)
G9.add_edge(2559,2902,weight=1)
G9.add_edge(2560,3010,weight=1)
G9.add_edge(2565,3142,weight=1)
G9.add_edge(2565,2566,weight=1)
G9.add_edge(2566,3036,weight=1)
G9.add_edge(2575,2827,weight=1)
G9.add_edge(2575,3109,weight=1)
G9.add_edge(2577,2668,weight=1)
G9.add_edge(2579,2762,weight=1)
G9.add_edge(2580,3142,weight=1)
G9.add_edge(2580,2661,weight=1)
G9.add_edge(2580,2665,weight=1)
G9.add_edge(2581,2650,weight=1)
G9.add_edge(2581,2635,weight=1)
G9.add_edge(2594,2652,weight=1)
G9.add_edge(2598,3025,weight=1)
G9.add_edge(2599,3188,weight=1)
G9.add_edge(2600,3046,weight=1)
G9.add_edge(2603,3238,weight=1)
G9.add_edge(2605,2784,weight=1)
G9.add_edge(2606,2901,weight=1)
G9.add_edge(2607,2754,weight=1)
G9.add_edge(2609,2937,weight=1)
G9.add_edge(2609,2943,weight=1)
G9.add_edge(2615,2895,weight=1)
G9.add_edge(2615,2652,weight=1)
G9.add_edge(2616,2942,weight=1)
G9.add_edge(2618,2879,weight=1)
G9.add_edge(2619,2783,weight=1)
G9.add_edge(2620,3209,weight=1)
G9.add_edge(2624,3027,weight=1)
G9.add_edge(2627,3138,weight=1)
G9.add_edge(2628,2629,weight=1)
G9.add_edge(2628,3099,weight=1)
G9.add_edge(2629,3099,weight=1)
G9.add_edge(2630,2708,weight=1)
G9.add_edge(2630,2907,weight=1)
G9.add_edge(2630,3036,weight=1)
G9.add_edge(2630,3066,weight=1)
G9.add_edge(2631,2633,weight=1)
G9.add_edge(2632,2743,weight=1)
G9.add_edge(2632,3013,weight=1)
G9.add_edge(2635,3150,weight=1)
G9.add_edge(2636,3195,weight=1)
G9.add_edge(2639,3246,weight=1)
G9.add_edge(2640,2680,weight=1)
G9.add_edge(2641,2907,weight=1)
G9.add_edge(2642,3004,weight=1)
G9.add_edge(2642,2738,weight=1)
G9.add_edge(2642,2668,weight=1)
G9.add_edge(2646,2833,weight=1)
G9.add_edge(2649,2930,weight=1)
G9.add_edge(2651,2897,weight=1)
G9.add_edge(2656,2721,weight=1)
G9.add_edge(2656,2907,weight=1)
G9.add_edge(2657,3222,weight=1)
G9.add_edge(2658,2995,weight=1)
G9.add_edge(2660,2898,weight=1)
G9.add_edge(2663,2845,weight=1)
G9.add_edge(2666,2791,weight=1)
G9.add_edge(2669,3068,weight=1)
G9.add_edge(2671,2854,weight=1)
G9.add_edge(2674,3202,weight=1)
G9.add_edge(2675,2935,weight=1)
G9.add_edge(2682,2862,weight=1)
G9.add_edge(2682,3222,weight=1)
G9.add_edge(2686,2708,weight=1)
G9.add_edge(2686,3064,weight=1)
G9.add_edge(2686,3065,weight=1)
G9.add_edge(2686,2857,weight=1)
G9.add_edge(2686,2903,weight=1)
G9.add_edge(2689,2907,weight=1)
G9.add_edge(2689,3222,weight=1)
G9.add_edge(2691,2907,weight=1)
G9.add_edge(2691,3222,weight=1)
G9.add_edge(2692,2871,weight=1)
G9.add_edge(2695,2722,weight=1)
G9.add_edge(2695,2802,weight=1)
G9.add_edge(2695,2760,weight=1)
G9.add_edge(2695,3112,weight=1)
G9.add_edge(2695,3113,weight=1)
G9.add_edge(2695,2844,weight=1)
G9.add_edge(2695,3248,weight=1)
G9.add_edge(2695,2809,weight=1)
G9.add_edge(2695,2814,weight=1)
G9.add_edge(2697,2722,weight=1)
G9.add_edge(2698,3137,weight=1)
G9.add_edge(2708,3062,weight=1)
G9.add_edge(2708,3063,weight=1)
G9.add_edge(2708,3066,weight=1)
G9.add_edge(2710,2987,weight=1)
G9.add_edge(2710,3165,weight=1)
G9.add_edge(2711,2978,weight=1)
G9.add_edge(2711,3081,weight=1)
G9.add_edge(2716,2858,weight=1)
G9.add_edge(2716,2837,weight=1)
G9.add_edge(2718,3033,weight=1)
G9.add_edge(2720,2739,weight=1)
G9.add_edge(2724,2730,weight=1)
G9.add_edge(2725,3114,weight=1)
G9.add_edge(2725,2917,weight=1)
G9.add_edge(2727,2752,weight=1)
G9.add_edge(2731,3149,weight=1)
G9.add_edge(2732,3054,weight=1)
G9.add_edge(2735,2744,weight=1)
G9.add_edge(2736,3132,weight=1)
G9.add_edge(2748,3263,weight=1)
G9.add_edge(2748,2880,weight=1)
G9.add_edge(2749,2894,weight=1)
G9.add_edge(2755,2914,weight=1)
G9.add_edge(2756,2759,weight=1)
G9.add_edge(2758,2759,weight=1)
G9.add_edge(2763,2764,weight=1)
G9.add_edge(2768,2818,weight=1)
G9.add_edge(2771,3097,weight=1)
G9.add_edge(2772,2862,weight=1)
G9.add_edge(2776,3080,weight=1)
G9.add_edge(2777,2942,weight=1)
G9.add_edge(2778,2779,weight=1)
G9.add_edge(2792,3059,weight=1)
G9.add_edge(2794,3222,weight=1)
G9.add_edge(2795,2945,weight=1)
G9.add_edge(2797,2907,weight=1)
G9.add_edge(2798,3090,weight=1)
G9.add_edge(2798,2799,weight=1)
G9.add_edge(2803,2966,weight=1)
G9.add_edge(2803,2809,weight=1)
G9.add_edge(2807,2944,weight=1)
G9.add_edge(2809,3112,weight=1)
G9.add_edge(2809,3113,weight=1)
G9.add_edge(2809,2966,weight=1)
G9.add_edge(2809,2822,weight=1)
G9.add_edge(2811,3078,weight=1)
G9.add_edge(2812,3143,weight=1)
G9.add_edge(2813,3170,weight=1)
G9.add_edge(2813,3256,weight=1)
G9.add_edge(2814,3173,weight=1)
G9.add_edge(2819,2869,weight=1)
G9.add_edge(2823,3155,weight=1)
G9.add_edge(2828,3000,weight=1)
G9.add_edge(2830,2890,weight=1)
G9.add_edge(2831,2847,weight=1)
G9.add_edge(2832,3088,weight=1)
G9.add_edge(2836,3074,weight=1)
G9.add_edge(2852,2907,weight=1)
G9.add_edge(2852,3115,weight=1)
G9.add_edge(2852,3092,weight=1)
G9.add_edge(2855,2907,weight=1)
G9.add_edge(2855,2866,weight=1)
G9.add_edge(2856,2865,weight=1)
G9.add_edge(2857,2907,weight=1)
G9.add_edge(2857,2866,weight=1)
G9.add_edge(2861,2907,weight=1)
G9.add_edge(2862,3063,weight=1)
G9.add_edge(2862,2907,weight=1)
G9.add_edge(2862,3102,weight=1)
G9.add_edge(2863,2907,weight=1)
G9.add_edge(2863,3087,weight=1)
G9.add_edge(2866,2867,weight=1)
G9.add_edge(2872,2900,weight=1)
G9.add_edge(2872,2873,weight=1)
G9.add_edge(2874,2974,weight=1)
G9.add_edge(2882,3091,weight=1)
G9.add_edge(2892,2897,weight=1)
G9.add_edge(2893,3118,weight=1)
G9.add_edge(2898,3191,weight=1)
G9.add_edge(2901,3225,weight=1)
G9.add_edge(2902,3125,weight=1)
G9.add_edge(2903,2961,weight=1)
G9.add_edge(2907,3098,weight=1)
G9.add_edge(2907,3060,weight=1)
G9.add_edge(2907,3062,weight=1)
G9.add_edge(2907,2927,weight=1)
G9.add_edge(2907,3064,weight=1)
G9.add_edge(2907,2993,weight=1)
G9.add_edge(2907,3065,weight=1)
G9.add_edge(2907,3231,weight=1)
G9.add_edge(2907,3169,weight=1)
G9.add_edge(2907,3232,weight=1)
G9.add_edge(2907,3233,weight=1)
G9.add_edge(2907,2996,weight=1)
G9.add_edge(2907,3015,weight=1)
G9.add_edge(2907,2945,weight=1)
G9.add_edge(2907,3087,weight=1)
G9.add_edge(2907,2912,weight=1)
G9.add_edge(2907,3053,weight=1)
G9.add_edge(2907,3181,weight=1)
G9.add_edge(2907,3144,weight=1)
G9.add_edge(2907,3206,weight=1)
G9.add_edge(2907,3222,weight=1)
G9.add_edge(2907,2956,weight=1)
G9.add_edge(2907,2979,weight=1)
G9.add_edge(2907,3101,weight=1)
G9.add_edge(2907,2959,weight=1)
G9.add_edge(2908,3073,weight=1)
G9.add_edge(2915,3186,weight=1)
G9.add_edge(2916,3000,weight=1)
G9.add_edge(2920,3123,weight=1)
G9.add_edge(2927,3144,weight=1)
G9.add_edge(2936,3234,weight=1)
G9.add_edge(2944,2964,weight=1)
G9.add_edge(2951,3114,weight=1)
G9.add_edge(2956,2979,weight=1)
G9.add_edge(2957,3221,weight=1)
G9.add_edge(2958,3009,weight=1)
G9.add_edge(2962,3192,weight=1)
G9.add_edge(2963,3039,weight=1)
G9.add_edge(2963,3186,weight=1)
G9.add_edge(2965,2970,weight=1)
G9.add_edge(2967,2968,weight=1)
G9.add_edge(2973,2976,weight=1)
G9.add_edge(2975,3123,weight=1)
G9.add_edge(2978,3081,weight=1)
G9.add_edge(2980,3209,weight=1)
G9.add_edge(2982,3002,weight=1)
G9.add_edge(2984,3252,weight=1)
G9.add_edge(2984,3253,weight=1)
G9.add_edge(2987,3170,weight=1)
G9.add_edge(2987,3165,weight=1)
G9.add_edge(2990,3097,weight=1)
G9.add_edge(2991,3111,weight=1)
G9.add_edge(2997,3027,weight=1)
G9.add_edge(3008,3191,weight=1)
G9.add_edge(3014,3104,weight=1)
G9.add_edge(3026,3261,weight=1)
G9.add_edge(3029,3083,weight=1)
G9.add_edge(3030,3057,weight=1)
G9.add_edge(3036,3062,weight=1)
G9.add_edge(3036,3160,weight=1)
G9.add_edge(3040,3130,weight=1)
G9.add_edge(3052,3147,weight=1)
G9.add_edge(3056,3168,weight=1)
G9.add_edge(3060,3061,weight=1)
G9.add_edge(3060,3062,weight=1)
G9.add_edge(3060,3063,weight=1)
G9.add_edge(3060,3064,weight=1)
G9.add_edge(3060,3065,weight=1)
G9.add_edge(3060,3160,weight=1)
G9.add_edge(3060,3102,weight=1)
G9.add_edge(3061,3064,weight=1)
G9.add_edge(3062,3066,weight=1)
G9.add_edge(3063,3066,weight=1)
G9.add_edge(3064,3066,weight=1)
G9.add_edge(3069,3221,weight=1)
G9.add_edge(3069,3070,weight=1)
G9.add_edge(3071,3132,weight=1)
G9.add_edge(3072,3131,weight=1)
G9.add_edge(3075,3222,weight=1)
G9.add_edge(3077,3078,weight=1)
G9.add_edge(3085,3089,weight=1)
G9.add_edge(3088,3213,weight=1)
G9.add_edge(3108,3162,weight=1)
G9.add_edge(3110,3126,weight=1)
G9.add_edge(3118,3211,weight=1)
G9.add_edge(3119,3125,weight=1)
G9.add_edge(3121,3123,weight=1)
G9.add_edge(3122,3123,weight=1)
G9.add_edge(3123,3124,weight=1)
G9.add_edge(3126,3127,weight=1)
G9.add_edge(3134,3153,weight=1)
G9.add_edge(3165,3256,weight=1)
G9.add_edge(3170,3256,weight=1)
G9.add_edge(3175,3180,weight=1)
G9.add_edge(3175,3240,weight=1)
G9.add_edge(3179,3252,weight=1)
G9.add_edge(3179,3253,weight=1)
G9.add_edge(3184,3185,weight=1)
G9.add_edge(3184,3190,weight=1)
G9.add_edge(3185,3190,weight=1)
G9.add_edge(3189,3250,weight=1)
G9.add_edge(3197,3198,weight=1)
G9.add_edge(3228,3229,weight=1)
G9.add_edge(3243,3244,weight=1)



#PowerGrill
G10 = nx.DiGraph()
G10.add_edge(1,2,weight=1)
G10.add_edge(1,3,weight=1)
G10.add_edge(1,4,weight=1)
G10.add_edge(2,428,weight=1)
G10.add_edge(2,435,weight=1)
G10.add_edge(2,436,weight=1)
G10.add_edge(2,437,weight=1)
G10.add_edge(2,438,weight=1)
G10.add_edge(3,432,weight=1)
G10.add_edge(3,458,weight=1)
G10.add_edge(3,459,weight=1)
G10.add_edge(3,460,weight=1)
G10.add_edge(4,478,weight=1)
G10.add_edge(4,523,weight=1)
G10.add_edge(5,6,weight=1)
G10.add_edge(5,7,weight=1)
G10.add_edge(5,8,weight=1)
G10.add_edge(5,9,weight=1)
G10.add_edge(6,3644,weight=1)
G10.add_edge(6,3680,weight=1)
G10.add_edge(7,3397,weight=1)
G10.add_edge(8,3397,weight=1)
G10.add_edge(9,3751,weight=1)
G10.add_edge(9,3830,weight=1)
G10.add_edge(10,11,weight=1)
G10.add_edge(11,3454,weight=1)
G10.add_edge(11,3535,weight=1)
G10.add_edge(12,13,weight=1)
G10.add_edge(13,4935,weight=1)
G10.add_edge(13,4936,weight=1)
G10.add_edge(14,15,weight=1)
G10.add_edge(15,115,weight=1)
G10.add_edge(15,119,weight=1)
G10.add_edge(15,125,weight=1)
G10.add_edge(15,171,weight=1)
G10.add_edge(15,173,weight=1)
G10.add_edge(15,174,weight=1)
G10.add_edge(15,175,weight=1)
G10.add_edge(16,17,weight=1)
G10.add_edge(16,18,weight=1)
G10.add_edge(17,32,weight=1)
G10.add_edge(17,33,weight=1)
G10.add_edge(17,34,weight=1)
G10.add_edge(17,35,weight=1)
G10.add_edge(18,47,weight=1)
G10.add_edge(19,20,weight=1)
G10.add_edge(20,21,weight=1)
G10.add_edge(20,22,weight=1)
G10.add_edge(22,23,weight=1)
G10.add_edge(22,24,weight=1)
G10.add_edge(22,25,weight=1)
G10.add_edge(22,26,weight=1)
G10.add_edge(22,27,weight=1)
G10.add_edge(23,28,weight=1)
G10.add_edge(24,131,weight=1)
G10.add_edge(24,132,weight=1)
G10.add_edge(25,106,weight=1)
G10.add_edge(26,28,weight=1)
G10.add_edge(26,67,weight=1)
G10.add_edge(26,131,weight=1)
G10.add_edge(26,198,weight=1)
G10.add_edge(26,205,weight=1)
G10.add_edge(26,226,weight=1)
G10.add_edge(27,31,weight=1)
G10.add_edge(28,61,weight=1)
G10.add_edge(29,30,weight=1)
G10.add_edge(29,31,weight=1)
G10.add_edge(30,59,weight=1)
G10.add_edge(30,60,weight=1)
G10.add_edge(30,99,weight=1)
G10.add_edge(30,177,weight=1)
G10.add_edge(30,178,weight=1)
G10.add_edge(30,179,weight=1)
G10.add_edge(31,67,weight=1)
G10.add_edge(33,36,weight=1)
G10.add_edge(33,37,weight=1)
G10.add_edge(33,38,weight=1)
G10.add_edge(33,39,weight=1)
G10.add_edge(35,96,weight=1)
G10.add_edge(35,167,weight=1)
G10.add_edge(38,44,weight=1)
G10.add_edge(38,68,weight=1)
G10.add_edge(39,117,weight=1)
G10.add_edge(40,41,weight=1)
G10.add_edge(40,42,weight=1)
G10.add_edge(41,43,weight=1)
G10.add_edge(41,44,weight=1)
G10.add_edge(42,190,weight=1)
G10.add_edge(43,45,weight=1)
G10.add_edge(45,163,weight=1)
G10.add_edge(45,245,weight=1)
G10.add_edge(45,255,weight=1)
G10.add_edge(46,47,weight=1)
G10.add_edge(46,48,weight=1)
G10.add_edge(47,111,weight=1)
G10.add_edge(48,65,weight=1)
G10.add_edge(49,50,weight=1)
G10.add_edge(50,110,weight=1)
G10.add_edge(50,127,weight=1)
G10.add_edge(51,52,weight=1)
G10.add_edge(52,55,weight=1)
G10.add_edge(52,153,weight=1)
G10.add_edge(52,168,weight=1)
G10.add_edge(52,530,weight=1)
G10.add_edge(52,540,weight=1)
G10.add_edge(53,54,weight=1)
G10.add_edge(53,55,weight=1)
G10.add_edge(54,116,weight=1)
G10.add_edge(55,127,weight=1)
G10.add_edge(55,168,weight=1)
G10.add_edge(55,218,weight=1)
G10.add_edge(55,4523,weight=1)
G10.add_edge(56,57,weight=1)
G10.add_edge(56,58,weight=1)
G10.add_edge(57,93,weight=1)
G10.add_edge(57,95,weight=1)
G10.add_edge(57,128,weight=1)
G10.add_edge(58,222,weight=1)
G10.add_edge(59,60,weight=1)
G10.add_edge(59,61,weight=1)
G10.add_edge(59,62,weight=1)
G10.add_edge(60,61,weight=1)
G10.add_edge(61,71,weight=1)
G10.add_edge(61,226,weight=1)
G10.add_edge(63,64,weight=1)
G10.add_edge(63,65,weight=1)
G10.add_edge(64,231,weight=1)
G10.add_edge(66,67,weight=1)
G10.add_edge(67,94,weight=1)
G10.add_edge(67,126,weight=1)
G10.add_edge(67,154,weight=1)
G10.add_edge(67,166,weight=1)
G10.add_edge(67,184,weight=1)
G10.add_edge(67,185,weight=1)
G10.add_edge(69,70,weight=1)
G10.add_edge(69,71,weight=1)
G10.add_edge(70,71,weight=1)
G10.add_edge(70,107,weight=1)
G10.add_edge(70,177,weight=1)
G10.add_edge(70,210,weight=1)
G10.add_edge(71,84,weight=1)
G10.add_edge(71,189,weight=1)
G10.add_edge(71,196,weight=1)
G10.add_edge(71,226,weight=1)
G10.add_edge(72,73,weight=1)
G10.add_edge(72,74,weight=1)
G10.add_edge(73,130,weight=1)
G10.add_edge(74,77,weight=1)
G10.add_edge(74,193,weight=1)
G10.add_edge(75,76,weight=1)
G10.add_edge(75,77,weight=1)
G10.add_edge(76,130,weight=1)
G10.add_edge(77,177,weight=1)
G10.add_edge(77,235,weight=1)
G10.add_edge(78,79,weight=1)
G10.add_edge(79,80,weight=1)
G10.add_edge(79,81,weight=1)
G10.add_edge(80,82,weight=1)
G10.add_edge(80,83,weight=1)
G10.add_edge(80,84,weight=1)
G10.add_edge(80,85,weight=1)
G10.add_edge(81,91,weight=1)
G10.add_edge(82,86,weight=1)
G10.add_edge(82,87,weight=1)
G10.add_edge(82,88,weight=1)
G10.add_edge(82,89,weight=1)
G10.add_edge(82,90,weight=1)
G10.add_edge(83,134,weight=1)
G10.add_edge(83,137,weight=1)
G10.add_edge(83,139,weight=1)
G10.add_edge(83,142,weight=1)
G10.add_edge(83,143,weight=1)
G10.add_edge(83,144,weight=1)
G10.add_edge(84,85,weight=1)
G10.add_edge(89,101,weight=1)
G10.add_edge(89,102,weight=1)
G10.add_edge(89,103,weight=1)
G10.add_edge(89,105,weight=1)
G10.add_edge(90,130,weight=1)
G10.add_edge(90,192,weight=1)
G10.add_edge(90,236,weight=1)
G10.add_edge(91,92,weight=1)
G10.add_edge(92,227,weight=1)
G10.add_edge(93,94,weight=1)
G10.add_edge(93,95,weight=1)
G10.add_edge(93,96,weight=1)
G10.add_edge(93,97,weight=1)
G10.add_edge(94,98,weight=1)
G10.add_edge(96,263,weight=1)
G10.add_edge(97,211,weight=1)
G10.add_edge(98,228,weight=1)
G10.add_edge(99,100,weight=1)
G10.add_edge(100,160,weight=1)
G10.add_edge(100,237,weight=1)
G10.add_edge(100,239,weight=1)
G10.add_edge(103,104,weight=1)
G10.add_edge(104,163,weight=1)
G10.add_edge(104,194,weight=1)
G10.add_edge(104,248,weight=1)
G10.add_edge(104,249,weight=1)
G10.add_edge(104,250,weight=1)
G10.add_edge(104,251,weight=1)
G10.add_edge(105,159,weight=1)
G10.add_edge(105,179,weight=1)
G10.add_edge(106,107,weight=1)
G10.add_edge(106,108,weight=1)
G10.add_edge(107,188,weight=1)
G10.add_edge(108,196,weight=1)
G10.add_edge(109,110,weight=1)
G10.add_edge(110,127,weight=1)
G10.add_edge(110,169,weight=1)
G10.add_edge(110,602,weight=1)
G10.add_edge(110,611,weight=1)
G10.add_edge(110,4510,weight=1)
G10.add_edge(111,191,weight=1)
G10.add_edge(111,211,weight=1)
G10.add_edge(112,113,weight=1)
G10.add_edge(112,114,weight=1)
G10.add_edge(112,115,weight=1)
G10.add_edge(113,183,weight=1)
G10.add_edge(113,200,weight=1)
G10.add_edge(113,201,weight=1)
G10.add_edge(114,121,weight=1)
G10.add_edge(114,124,weight=1)
G10.add_edge(114,183,weight=1)
G10.add_edge(116,150,weight=1)
G10.add_edge(117,118,weight=1)
G10.add_edge(118,161,weight=1)
G10.add_edge(119,120,weight=1)
G10.add_edge(120,174,weight=1)
G10.add_edge(121,122,weight=1)
G10.add_edge(121,123,weight=1)
G10.add_edge(121,124,weight=1)
G10.add_edge(121,125,weight=1)
G10.add_edge(122,125,weight=1)
G10.add_edge(123,125,weight=1)
G10.add_edge(125,250,weight=1)
G10.add_edge(125,255,weight=1)
G10.add_edge(126,127,weight=1)
G10.add_edge(127,165,weight=1)
G10.add_edge(127,219,weight=1)
G10.add_edge(127,2567,weight=1)
G10.add_edge(128,129,weight=1)
G10.add_edge(129,130,weight=1)
G10.add_edge(130,192,weight=1)
G10.add_edge(130,212,weight=1)
G10.add_edge(130,234,weight=1)
G10.add_edge(133,134,weight=1)
G10.add_edge(134,135,weight=1)
G10.add_edge(134,136,weight=1)
G10.add_edge(134,140,weight=1)
G10.add_edge(134,141,weight=1)
G10.add_edge(138,139,weight=1)
G10.add_edge(139,145,weight=1)
G10.add_edge(140,1048,weight=1)
G10.add_edge(140,1115,weight=1)
G10.add_edge(141,1147,weight=1)
G10.add_edge(141,4896,weight=1)
G10.add_edge(141,4901,weight=1)
G10.add_edge(141,4916,weight=1)
G10.add_edge(142,194,weight=1)
G10.add_edge(142,1079,weight=1)
G10.add_edge(142,1096,weight=1)
G10.add_edge(142,1142,weight=1)
G10.add_edge(142,1143,weight=1)
G10.add_edge(142,1144,weight=1)
G10.add_edge(142,1145,weight=1)
G10.add_edge(142,1146,weight=1)
G10.add_edge(142,1147,weight=1)
G10.add_edge(143,1041,weight=1)
G10.add_edge(143,1062,weight=1)
G10.add_edge(143,1079,weight=1)
G10.add_edge(143,1148,weight=1)
G10.add_edge(144,2451,weight=1)
G10.add_edge(145,197,weight=1)
G10.add_edge(145,198,weight=1)
G10.add_edge(145,199,weight=1)
G10.add_edge(146,147,weight=1)
G10.add_edge(146,148,weight=1)
G10.add_edge(147,155,weight=1)
G10.add_edge(147,157,weight=1)
G10.add_edge(148,223,weight=1)
G10.add_edge(148,264,weight=1)
G10.add_edge(149,150,weight=1)
G10.add_edge(149,151,weight=1)
G10.add_edge(149,152,weight=1)
G10.add_edge(150,153,weight=1)
G10.add_edge(151,152,weight=1)
G10.add_edge(151,202,weight=1)
G10.add_edge(152,259,weight=1)
G10.add_edge(153,176,weight=1)
G10.add_edge(155,156,weight=1)
G10.add_edge(155,157,weight=1)
G10.add_edge(155,158,weight=1)
G10.add_edge(156,159,weight=1)
G10.add_edge(156,160,weight=1)
G10.add_edge(158,242,weight=1)
G10.add_edge(159,240,weight=1)
G10.add_edge(159,241,weight=1)
G10.add_edge(160,228,weight=1)
G10.add_edge(161,162,weight=1)
G10.add_edge(162,163,weight=1)
G10.add_edge(163,164,weight=1)
G10.add_edge(164,251,weight=1)
G10.add_edge(164,1095,weight=1)
G10.add_edge(165,166,weight=1)
G10.add_edge(167,262,weight=1)
G10.add_edge(167,265,weight=1)
G10.add_edge(169,170,weight=1)
G10.add_edge(170,227,weight=1)
G10.add_edge(171,172,weight=1)
G10.add_edge(174,244,weight=1)
G10.add_edge(175,201,weight=1)
G10.add_edge(176,260,weight=1)
G10.add_edge(176,535,weight=1)
G10.add_edge(176,547,weight=1)
G10.add_edge(179,180,weight=1)
G10.add_edge(180,198,weight=1)
G10.add_edge(180,202,weight=1)
G10.add_edge(180,213,weight=1)
G10.add_edge(180,214,weight=1)
G10.add_edge(180,215,weight=1)
G10.add_edge(180,216,weight=1)
G10.add_edge(180,217,weight=1)
G10.add_edge(181,182,weight=1)
G10.add_edge(181,183,weight=1)
G10.add_edge(182,200,weight=1)
G10.add_edge(184,186,weight=1)
G10.add_edge(185,221,weight=1)
G10.add_edge(186,187,weight=1)
G10.add_edge(187,602,weight=1)
G10.add_edge(188,256,weight=1)
G10.add_edge(188,257,weight=1)
G10.add_edge(188,258,weight=1)
G10.add_edge(190,191,weight=1)
G10.add_edge(190,192,weight=1)
G10.add_edge(191,231,weight=1)
G10.add_edge(192,211,weight=1)
G10.add_edge(192,222,weight=1)
G10.add_edge(192,232,weight=1)
G10.add_edge(192,233,weight=1)
G10.add_edge(193,220,weight=1)
G10.add_edge(194,195,weight=1)
G10.add_edge(195,1045,weight=1)
G10.add_edge(195,1117,weight=1)
G10.add_edge(197,198,weight=1)
G10.add_edge(197,206,weight=1)
G10.add_edge(197,207,weight=1)
G10.add_edge(197,208,weight=1)
G10.add_edge(198,247,weight=1)
G10.add_edge(199,208,weight=1)
G10.add_edge(199,651,weight=1)
G10.add_edge(199,964,weight=1)
G10.add_edge(199,4257,weight=1)
G10.add_edge(200,236,weight=1)
G10.add_edge(202,203,weight=1)
G10.add_edge(203,549,weight=1)
G10.add_edge(203,3892,weight=1)
G10.add_edge(204,205,weight=1)
G10.add_edge(208,571,weight=1)
G10.add_edge(208,611,weight=1)
G10.add_edge(209,210,weight=1)
G10.add_edge(211,212,weight=1)
G10.add_edge(217,542,weight=1)
G10.add_edge(217,4245,weight=1)
G10.add_edge(220,221,weight=1)
G10.add_edge(221,268,weight=1)
G10.add_edge(223,224,weight=1)
G10.add_edge(223,225,weight=1)
G10.add_edge(224,243,weight=1)
G10.add_edge(225,240,weight=1)
G10.add_edge(225,266,weight=1)
G10.add_edge(226,227,weight=1)
G10.add_edge(226,228,weight=1)
G10.add_edge(226,229,weight=1)
G10.add_edge(227,230,weight=1)
G10.add_edge(229,4897,weight=1)
G10.add_edge(230,267,weight=1)
G10.add_edge(232,235,weight=1)
G10.add_edge(237,238,weight=1)
G10.add_edge(241,242,weight=1)
G10.add_edge(241,243,weight=1)
G10.add_edge(242,252,weight=1)
G10.add_edge(242,253,weight=1)
G10.add_edge(244,245,weight=1)
G10.add_edge(244,246,weight=1)
G10.add_edge(245,247,weight=1)
G10.add_edge(251,1061,weight=1)
G10.add_edge(251,1100,weight=1)
G10.add_edge(251,1105,weight=1)
G10.add_edge(252,261,weight=1)
G10.add_edge(253,266,weight=1)
G10.add_edge(254,255,weight=1)
G10.add_edge(259,260,weight=1)
G10.add_edge(261,262,weight=1)
G10.add_edge(269,270,weight=1)
G10.add_edge(269,271,weight=1)
G10.add_edge(269,272,weight=1)
G10.add_edge(270,310,weight=1)
G10.add_edge(270,311,weight=1)
G10.add_edge(270,315,weight=1)
G10.add_edge(270,316,weight=1)
G10.add_edge(270,317,weight=1)
G10.add_edge(271,377,weight=1)
G10.add_edge(271,399,weight=1)
G10.add_edge(273,274,weight=1)
G10.add_edge(273,275,weight=1)
G10.add_edge(273,276,weight=1)
G10.add_edge(274,277,weight=1)
G10.add_edge(274,278,weight=1)
G10.add_edge(274,279,weight=1)
G10.add_edge(275,410,weight=1)
G10.add_edge(276,393,weight=1)
G10.add_edge(276,421,weight=1)
G10.add_edge(277,279,weight=1)
G10.add_edge(277,322,weight=1)
G10.add_edge(277,323,weight=1)
G10.add_edge(277,330,weight=1)
G10.add_edge(277,388,weight=1)
G10.add_edge(277,396,weight=1)
G10.add_edge(278,405,weight=1)
G10.add_edge(278,406,weight=1)
G10.add_edge(279,315,weight=1)
G10.add_edge(279,377,weight=1)
G10.add_edge(280,281,weight=1)
G10.add_edge(280,282,weight=1)
G10.add_edge(281,283,weight=1)
G10.add_edge(281,284,weight=1)
G10.add_edge(281,285,weight=1)
G10.add_edge(281,286,weight=1)
G10.add_edge(281,287,weight=1)
G10.add_edge(282,334,weight=1)
G10.add_edge(282,363,weight=1)
G10.add_edge(282,411,weight=1)
G10.add_edge(283,287,weight=1)
G10.add_edge(283,306,weight=1)
G10.add_edge(283,319,weight=1)
G10.add_edge(283,322,weight=1)
G10.add_edge(283,369,weight=1)
G10.add_edge(283,372,weight=1)
G10.add_edge(283,373,weight=1)
G10.add_edge(286,373,weight=1)
G10.add_edge(288,289,weight=1)
G10.add_edge(288,290,weight=1)
G10.add_edge(289,290,weight=1)
G10.add_edge(289,365,weight=1)
G10.add_edge(289,386,weight=1)
G10.add_edge(290,393,weight=1)
G10.add_edge(290,398,weight=1)
G10.add_edge(290,410,weight=1)
G10.add_edge(291,292,weight=1)
G10.add_edge(291,293,weight=1)
G10.add_edge(292,298,weight=1)
G10.add_edge(292,367,weight=1)
G10.add_edge(292,412,weight=1)
G10.add_edge(293,354,weight=1)
G10.add_edge(293,423,weight=1)
G10.add_edge(294,295,weight=1)
G10.add_edge(295,296,weight=1)
G10.add_edge(295,301,weight=1)
G10.add_edge(296,297,weight=1)
G10.add_edge(296,298,weight=1)
G10.add_edge(296,299,weight=1)
G10.add_edge(296,300,weight=1)
G10.add_edge(297,309,weight=1)
G10.add_edge(297,325,weight=1)
G10.add_edge(297,326,weight=1)
G10.add_edge(297,327,weight=1)
G10.add_edge(297,328,weight=1)
G10.add_edge(298,323,weight=1)
G10.add_edge(298,367,weight=1)
G10.add_edge(298,381,weight=1)
G10.add_edge(298,386,weight=1)
G10.add_edge(299,300,weight=1)
G10.add_edge(300,367,weight=1)
G10.add_edge(300,424,weight=1)
G10.add_edge(301,321,weight=1)
G10.add_edge(301,407,weight=1)
G10.add_edge(302,303,weight=1)
G10.add_edge(303,304,weight=1)
G10.add_edge(304,305,weight=1)
G10.add_edge(304,306,weight=1)
G10.add_edge(304,307,weight=1)
G10.add_edge(304,308,weight=1)
G10.add_edge(304,309,weight=1)
G10.add_edge(305,306,weight=1)
G10.add_edge(305,307,weight=1)
G10.add_edge(305,308,weight=1)
G10.add_edge(305,309,weight=1)
G10.add_edge(305,368,weight=1)
G10.add_edge(306,308,weight=1)
G10.add_edge(306,330,weight=1)
G10.add_edge(306,390,weight=1)
G10.add_edge(307,308,weight=1)
G10.add_edge(308,368,weight=1)
G10.add_edge(308,369,weight=1)
G10.add_edge(308,370,weight=1)
G10.add_edge(308,371,weight=1)
G10.add_edge(309,328,weight=1)
G10.add_edge(310,311,weight=1)
G10.add_edge(310,312,weight=1)
G10.add_edge(310,313,weight=1)
G10.add_edge(310,314,weight=1)
G10.add_edge(312,314,weight=1)
G10.add_edge(312,316,weight=1)
G10.add_edge(312,401,weight=1)
G10.add_edge(313,314,weight=1)
G10.add_edge(315,318,weight=1)
G10.add_edge(318,4466,weight=1)
G10.add_edge(319,320,weight=1)
G10.add_edge(319,321,weight=1)
G10.add_edge(320,322,weight=1)
G10.add_edge(320,323,weight=1)
G10.add_edge(322,330,weight=1)
G10.add_edge(322,374,weight=1)
G10.add_edge(323,329,weight=1)
G10.add_edge(323,354,weight=1)
G10.add_edge(324,325,weight=1)
G10.add_edge(326,327,weight=1)
G10.add_edge(326,331,weight=1)
G10.add_edge(329,330,weight=1)
G10.add_edge(329,331,weight=1)
G10.add_edge(331,341,weight=1)
G10.add_edge(331,342,weight=1)
G10.add_edge(332,333,weight=1)
G10.add_edge(332,334,weight=1)
G10.add_edge(332,335,weight=1)
G10.add_edge(332,336,weight=1)
G10.add_edge(332,337,weight=1)
G10.add_edge(333,338,weight=1)
G10.add_edge(333,339,weight=1)
G10.add_edge(333,340,weight=1)
G10.add_edge(335,336,weight=1)
G10.add_edge(335,350,weight=1)
G10.add_edge(335,418,weight=1)
G10.add_edge(338,341,weight=1)
G10.add_edge(338,342,weight=1)
G10.add_edge(339,350,weight=1)
G10.add_edge(340,346,weight=1)
G10.add_edge(340,347,weight=1)
G10.add_edge(340,411,weight=1)
G10.add_edge(343,344,weight=1)
G10.add_edge(343,345,weight=1)
G10.add_edge(344,401,weight=1)
G10.add_edge(344,402,weight=1)
G10.add_edge(345,4440,weight=1)
G10.add_edge(345,4485,weight=1)
G10.add_edge(346,347,weight=1)
G10.add_edge(348,349,weight=1)
G10.add_edge(349,384,weight=1)
G10.add_edge(349,385,weight=1)
G10.add_edge(349,391,weight=1)
G10.add_edge(351,352,weight=1)
G10.add_edge(352,353,weight=1)
G10.add_edge(352,354,weight=1)
G10.add_edge(353,415,weight=1)
G10.add_edge(354,356,weight=1)
G10.add_edge(354,380,weight=1)
G10.add_edge(355,356,weight=1)
G10.add_edge(356,357,weight=1)
G10.add_edge(356,358,weight=1)
G10.add_edge(356,360,weight=1)
G10.add_edge(356,361,weight=1)
G10.add_edge(356,362,weight=1)
G10.add_edge(358,359,weight=1)
G10.add_edge(362,380,weight=1)
G10.add_edge(362,403,weight=1)
G10.add_edge(363,364,weight=1)
G10.add_edge(365,366,weight=1)
G10.add_edge(366,396,weight=1)
G10.add_edge(366,397,weight=1)
G10.add_edge(368,369,weight=1)
G10.add_edge(368,370,weight=1)
G10.add_edge(368,371,weight=1)
G10.add_edge(369,370,weight=1)
G10.add_edge(369,371,weight=1)
G10.add_edge(370,371,weight=1)
G10.add_edge(370,373,weight=1)
G10.add_edge(371,387,weight=1)
G10.add_edge(374,1268,weight=1)
G10.add_edge(374,1422,weight=1)
G10.add_edge(375,376,weight=1)
G10.add_edge(376,377,weight=1)
G10.add_edge(376,378,weight=1)
G10.add_edge(377,413,weight=1)
G10.add_edge(378,4494,weight=1)
G10.add_edge(378,4495,weight=1)
G10.add_edge(378,4499,weight=1)
G10.add_edge(378,4505,weight=1)
G10.add_edge(379,380,weight=1)
G10.add_edge(382,383,weight=1)
G10.add_edge(383,384,weight=1)
G10.add_edge(384,385,weight=1)
G10.add_edge(385,408,weight=1)
G10.add_edge(385,415,weight=1)
G10.add_edge(388,389,weight=1)
G10.add_edge(392,393,weight=1)
G10.add_edge(392,394,weight=1)
G10.add_edge(392,395,weight=1)
G10.add_edge(393,410,weight=1)
G10.add_edge(393,419,weight=1)
G10.add_edge(393,420,weight=1)
G10.add_edge(395,4504,weight=1)
G10.add_edge(396,398,weight=1)
G10.add_edge(397,404,weight=1)
G10.add_edge(399,400,weight=1)
G10.add_edge(400,1161,weight=1)
G10.add_edge(400,1167,weight=1)
G10.add_edge(400,1259,weight=1)
G10.add_edge(400,1355,weight=1)
G10.add_edge(400,1357,weight=1)
G10.add_edge(402,4443,weight=1)
G10.add_edge(402,4444,weight=1)
G10.add_edge(407,422,weight=1)
G10.add_edge(408,409,weight=1)
G10.add_edge(411,416,weight=1)
G10.add_edge(411,417,weight=1)
G10.add_edge(413,414,weight=1)
G10.add_edge(421,4500,weight=1)
G10.add_edge(425,426,weight=1)
G10.add_edge(425,427,weight=1)
G10.add_edge(425,428,weight=1)
G10.add_edge(425,429,weight=1)
G10.add_edge(425,430,weight=1)
G10.add_edge(425,431,weight=1)
G10.add_edge(426,432,weight=1)
G10.add_edge(426,433,weight=1)
G10.add_edge(426,434,weight=1)
G10.add_edge(427,437,weight=1)
G10.add_edge(427,508,weight=1)
G10.add_edge(429,430,weight=1)
G10.add_edge(429,508,weight=1)
G10.add_edge(430,489,weight=1)
G10.add_edge(431,485,weight=1)
G10.add_edge(432,434,weight=1)
G10.add_edge(432,454,weight=1)
G10.add_edge(432,455,weight=1)
G10.add_edge(433,434,weight=1)
G10.add_edge(433,452,weight=1)
G10.add_edge(433,465,weight=1)
G10.add_edge(433,504,weight=1)
G10.add_edge(433,505,weight=1)
G10.add_edge(438,2457,weight=1)
G10.add_edge(439,440,weight=1)
G10.add_edge(440,449,weight=1)
G10.add_edge(440,492,weight=1)
G10.add_edge(440,496,weight=1)
G10.add_edge(440,509,weight=1)
G10.add_edge(441,442,weight=1)
G10.add_edge(441,443,weight=1)
G10.add_edge(442,478,weight=1)
G10.add_edge(443,467,weight=1)
G10.add_edge(443,892,weight=1)
G10.add_edge(443,2294,weight=1)
G10.add_edge(443,2393,weight=1)
G10.add_edge(443,2399,weight=1)
G10.add_edge(443,2406,weight=1)
G10.add_edge(443,2424,weight=1)
G10.add_edge(443,2434,weight=1)
G10.add_edge(443,2436,weight=1)
G10.add_edge(443,2437,weight=1)
G10.add_edge(444,445,weight=1)
G10.add_edge(445,446,weight=1)
G10.add_edge(445,447,weight=1)
G10.add_edge(445,448,weight=1)
G10.add_edge(445,449,weight=1)
G10.add_edge(446,450,weight=1)
G10.add_edge(446,451,weight=1)
G10.add_edge(446,452,weight=1)
G10.add_edge(446,453,weight=1)
G10.add_edge(447,456,weight=1)
G10.add_edge(447,457,weight=1)
G10.add_edge(448,475,weight=1)
G10.add_edge(450,453,weight=1)
G10.add_edge(450,468,weight=1)
G10.add_edge(450,469,weight=1)
G10.add_edge(450,470,weight=1)
G10.add_edge(450,471,weight=1)
G10.add_edge(450,472,weight=1)
G10.add_edge(451,474,weight=1)
G10.add_edge(452,486,weight=1)
G10.add_edge(452,502,weight=1)
G10.add_edge(453,510,weight=1)
G10.add_edge(454,461,weight=1)
G10.add_edge(454,462,weight=1)
G10.add_edge(454,463,weight=1)
G10.add_edge(454,464,weight=1)
G10.add_edge(454,465,weight=1)
G10.add_edge(454,466,weight=1)
G10.add_edge(454,467,weight=1)
G10.add_edge(455,2315,weight=1)
G10.add_edge(455,2316,weight=1)
G10.add_edge(455,2317,weight=1)
G10.add_edge(455,2328,weight=1)
G10.add_edge(455,2329,weight=1)
G10.add_edge(456,509,weight=1)
G10.add_edge(456,518,weight=1)
G10.add_edge(457,474,weight=1)
G10.add_edge(457,476,weight=1)
G10.add_edge(458,479,weight=1)
G10.add_edge(458,498,weight=1)
G10.add_edge(459,903,weight=1)
G10.add_edge(459,2294,weight=1)
G10.add_edge(460,2326,weight=1)
G10.add_edge(460,2439,weight=1)
G10.add_edge(460,2495,weight=1)
G10.add_edge(465,467,weight=1)
G10.add_edge(465,498,weight=1)
G10.add_edge(466,2299,weight=1)
G10.add_edge(466,2309,weight=1)
G10.add_edge(466,2341,weight=1)
G10.add_edge(467,2434,weight=1)
G10.add_edge(470,480,weight=1)
G10.add_edge(470,481,weight=1)
G10.add_edge(471,513,weight=1)
G10.add_edge(471,514,weight=1)
G10.add_edge(471,515,weight=1)
G10.add_edge(472,499,weight=1)
G10.add_edge(472,516,weight=1)
G10.add_edge(473,474,weight=1)
G10.add_edge(474,475,weight=1)
G10.add_edge(474,476,weight=1)
G10.add_edge(475,510,weight=1)
G10.add_edge(475,511,weight=1)
G10.add_edge(475,512,weight=1)
G10.add_edge(477,478,weight=1)
G10.add_edge(478,479,weight=1)
G10.add_edge(479,521,weight=1)
G10.add_edge(481,1517,weight=1)
G10.add_edge(481,1548,weight=1)
G10.add_edge(482,483,weight=1)
G10.add_edge(482,484,weight=1)
G10.add_edge(483,516,weight=1)
G10.add_edge(484,1599,weight=1)
G10.add_edge(485,486,weight=1)
G10.add_edge(485,487,weight=1)
G10.add_edge(485,488,weight=1)
G10.add_edge(485,489,weight=1)
G10.add_edge(485,490,weight=1)
G10.add_edge(485,491,weight=1)
G10.add_edge(488,507,weight=1)
G10.add_edge(489,507,weight=1)
G10.add_edge(490,496,weight=1)
G10.add_edge(490,503,weight=1)
G10.add_edge(491,495,weight=1)
G10.add_edge(491,503,weight=1)
G10.add_edge(492,493,weight=1)
G10.add_edge(492,494,weight=1)
G10.add_edge(492,495,weight=1)
G10.add_edge(492,496,weight=1)
G10.add_edge(494,502,weight=1)
G10.add_edge(497,498,weight=1)
G10.add_edge(499,500,weight=1)
G10.add_edge(499,501,weight=1)
G10.add_edge(500,1448,weight=1)
G10.add_edge(500,1808,weight=1)
G10.add_edge(502,503,weight=1)
G10.add_edge(504,506,weight=1)
G10.add_edge(505,4094,weight=1)
G10.add_edge(506,1368,weight=1)
G10.add_edge(509,517,weight=1)
G10.add_edge(512,516,weight=1)
G10.add_edge(512,526,weight=1)
G10.add_edge(514,525,weight=1)
G10.add_edge(515,1604,weight=1)
G10.add_edge(515,1663,weight=1)
G10.add_edge(519,520,weight=1)
G10.add_edge(520,521,weight=1)
G10.add_edge(520,522,weight=1)
G10.add_edge(522,524,weight=1)
G10.add_edge(523,524,weight=1)
G10.add_edge(527,528,weight=1)
G10.add_edge(527,529,weight=1)
G10.add_edge(528,537,weight=1)
G10.add_edge(528,541,weight=1)
G10.add_edge(528,542,weight=1)
G10.add_edge(529,542,weight=1)
G10.add_edge(529,4247,weight=1)
G10.add_edge(530,531,weight=1)
G10.add_edge(531,4246,weight=1)
G10.add_edge(532,533,weight=1)
G10.add_edge(533,534,weight=1)
G10.add_edge(533,535,weight=1)
G10.add_edge(534,537,weight=1)
G10.add_edge(534,540,weight=1)
G10.add_edge(535,540,weight=1)
G10.add_edge(535,544,weight=1)
G10.add_edge(535,548,weight=1)
G10.add_edge(536,537,weight=1)
G10.add_edge(536,538,weight=1)
G10.add_edge(536,539,weight=1)
G10.add_edge(538,540,weight=1)
G10.add_edge(538,551,weight=1)
G10.add_edge(539,544,weight=1)
G10.add_edge(541,552,weight=1)
G10.add_edge(542,4242,weight=1)
G10.add_edge(542,4243,weight=1)
G10.add_edge(542,4244,weight=1)
G10.add_edge(543,544,weight=1)
G10.add_edge(543,545,weight=1)
G10.add_edge(543,546,weight=1)
G10.add_edge(543,547,weight=1)
G10.add_edge(544,545,weight=1)
G10.add_edge(544,550,weight=1)
G10.add_edge(544,551,weight=1)
G10.add_edge(546,550,weight=1)
G10.add_edge(548,549,weight=1)
G10.add_edge(549,695,weight=1)
G10.add_edge(553,554,weight=1)
G10.add_edge(553,555,weight=1)
G10.add_edge(554,1490,weight=1)
G10.add_edge(555,1434,weight=1)
G10.add_edge(555,1479,weight=1)
G10.add_edge(555,1480,weight=1)
G10.add_edge(556,557,weight=1)
G10.add_edge(556,558,weight=1)
G10.add_edge(556,559,weight=1)
G10.add_edge(556,560,weight=1)
G10.add_edge(556,561,weight=1)
G10.add_edge(556,562,weight=1)
G10.add_edge(556,563,weight=1)
G10.add_edge(556,564,weight=1)
G10.add_edge(556,565,weight=1)
G10.add_edge(556,566,weight=1)
G10.add_edge(556,567,weight=1)
G10.add_edge(557,2514,weight=1)
G10.add_edge(557,2515,weight=1)
G10.add_edge(558,564,weight=1)
G10.add_edge(558,565,weight=1)
G10.add_edge(558,566,weight=1)
G10.add_edge(558,2762,weight=1)
G10.add_edge(558,2797,weight=1)
G10.add_edge(558,2800,weight=1)
G10.add_edge(558,2818,weight=1)
G10.add_edge(558,2819,weight=1)
G10.add_edge(558,2820,weight=1)
G10.add_edge(558,2821,weight=1)
G10.add_edge(558,2822,weight=1)
G10.add_edge(558,2823,weight=1)
G10.add_edge(559,563,weight=1)
G10.add_edge(559,2633,weight=1)
G10.add_edge(559,2634,weight=1)
G10.add_edge(559,2685,weight=1)
G10.add_edge(559,2824,weight=1)
G10.add_edge(560,2514,weight=1)
G10.add_edge(560,2684,weight=1)
G10.add_edge(560,2807,weight=1)
G10.add_edge(560,2809,weight=1)
G10.add_edge(560,2835,weight=1)
G10.add_edge(560,2836,weight=1)
G10.add_edge(560,2837,weight=1)
G10.add_edge(560,2838,weight=1)
G10.add_edge(561,3060,weight=1)
G10.add_edge(561,3061,weight=1)
G10.add_edge(562,3041,weight=1)
G10.add_edge(562,3042,weight=1)
G10.add_edge(562,3182,weight=1)
G10.add_edge(562,3189,weight=1)
G10.add_edge(562,3190,weight=1)
G10.add_edge(563,3214,weight=1)
G10.add_edge(564,3211,weight=1)
G10.add_edge(565,3212,weight=1)
G10.add_edge(566,3213,weight=1)
G10.add_edge(567,3293,weight=1)
G10.add_edge(568,569,weight=1)
G10.add_edge(569,570,weight=1)
G10.add_edge(569,571,weight=1)
G10.add_edge(569,572,weight=1)
G10.add_edge(570,580,weight=1)
G10.add_edge(571,624,weight=1)
G10.add_edge(571,637,weight=1)
G10.add_edge(571,651,weight=1)
G10.add_edge(572,616,weight=1)
G10.add_edge(573,574,weight=1)
G10.add_edge(574,575,weight=1)
G10.add_edge(574,576,weight=1)
G10.add_edge(574,577,weight=1)
G10.add_edge(574,578,weight=1)
G10.add_edge(576,614,weight=1)
G10.add_edge(576,615,weight=1)
G10.add_edge(576,616,weight=1)
G10.add_edge(577,616,weight=1)
G10.add_edge(577,644,weight=1)
G10.add_edge(578,616,weight=1)
G10.add_edge(578,647,weight=1)
G10.add_edge(578,648,weight=1)
G10.add_edge(578,649,weight=1)
G10.add_edge(579,580,weight=1)
G10.add_edge(579,581,weight=1)
G10.add_edge(580,581,weight=1)
G10.add_edge(580,616,weight=1)
G10.add_edge(580,625,weight=1)
G10.add_edge(580,641,weight=1)
G10.add_edge(580,643,weight=1)
G10.add_edge(581,595,weight=1)
G10.add_edge(581,599,weight=1)
G10.add_edge(581,600,weight=1)
G10.add_edge(582,583,weight=1)
G10.add_edge(582,584,weight=1)
G10.add_edge(582,585,weight=1)
G10.add_edge(583,586,weight=1)
G10.add_edge(583,632,weight=1)
G10.add_edge(583,633,weight=1)
G10.add_edge(583,634,weight=1)
G10.add_edge(583,635,weight=1)
G10.add_edge(583,636,weight=1)
G10.add_edge(584,586,weight=1)
G10.add_edge(584,628,weight=1)
G10.add_edge(584,638,weight=1)
G10.add_edge(584,639,weight=1)
G10.add_edge(584,640,weight=1)
G10.add_edge(585,588,weight=1)
G10.add_edge(585,635,weight=1)
G10.add_edge(586,587,weight=1)
G10.add_edge(587,588,weight=1)
G10.add_edge(587,636,weight=1)
G10.add_edge(589,590,weight=1)
G10.add_edge(589,591,weight=1)
G10.add_edge(589,592,weight=1)
G10.add_edge(590,635,weight=1)
G10.add_edge(590,636,weight=1)
G10.add_edge(593,594,weight=1)
G10.add_edge(594,635,weight=1)
G10.add_edge(594,636,weight=1)
G10.add_edge(594,646,weight=1)
G10.add_edge(595,596,weight=1)
G10.add_edge(595,597,weight=1)
G10.add_edge(595,598,weight=1)
G10.add_edge(595,599,weight=1)
G10.add_edge(595,600,weight=1)
G10.add_edge(598,599,weight=1)
G10.add_edge(598,600,weight=1)
G10.add_edge(600,634,weight=1)
G10.add_edge(601,602,weight=1)
G10.add_edge(602,611,weight=1)
G10.add_edge(602,637,weight=1)
G10.add_edge(602,921,weight=1)
G10.add_edge(602,927,weight=1)
G10.add_edge(602,931,weight=1)
G10.add_edge(602,968,weight=1)
G10.add_edge(602,998,weight=1)
G10.add_edge(602,2566,weight=1)
G10.add_edge(602,4257,weight=1)
G10.add_edge(602,4514,weight=1)
G10.add_edge(602,4517,weight=1)
G10.add_edge(602,4519,weight=1)
G10.add_edge(602,4520,weight=1)
G10.add_edge(602,4521,weight=1)
G10.add_edge(602,4522,weight=1)
G10.add_edge(603,604,weight=1)
G10.add_edge(604,606,weight=1)
G10.add_edge(605,606,weight=1)
G10.add_edge(606,607,weight=1)
G10.add_edge(606,608,weight=1)
G10.add_edge(606,610,weight=1)
G10.add_edge(608,609,weight=1)
G10.add_edge(609,2481,weight=1)
G10.add_edge(609,4116,weight=1)
G10.add_edge(610,2328,weight=1)
G10.add_edge(610,2404,weight=1)
G10.add_edge(610,2459,weight=1)
G10.add_edge(611,612,weight=1)
G10.add_edge(611,613,weight=1)
G10.add_edge(612,923,weight=1)
G10.add_edge(612,927,weight=1)
G10.add_edge(613,932,weight=1)
G10.add_edge(614,617,weight=1)
G10.add_edge(615,618,weight=1)
G10.add_edge(616,619,weight=1)
G10.add_edge(616,624,weight=1)
G10.add_edge(616,625,weight=1)
G10.add_edge(617,618,weight=1)
G10.add_edge(617,622,weight=1)
G10.add_edge(618,623,weight=1)
G10.add_edge(619,620,weight=1)
G10.add_edge(619,621,weight=1)
G10.add_edge(621,4290,weight=1)
G10.add_edge(625,650,weight=1)
G10.add_edge(626,627,weight=1)
G10.add_edge(626,628,weight=1)
G10.add_edge(626,629,weight=1)
G10.add_edge(627,630,weight=1)
G10.add_edge(627,631,weight=1)
G10.add_edge(631,640,weight=1)
G10.add_edge(631,644,weight=1)
G10.add_edge(632,637,weight=1)
G10.add_edge(633,637,weight=1)
G10.add_edge(638,642,weight=1)
G10.add_edge(639,642,weight=1)
G10.add_edge(640,645,weight=1)
G10.add_edge(641,642,weight=1)
G10.add_edge(642,643,weight=1)
G10.add_edge(644,645,weight=1)
G10.add_edge(649,4190,weight=1)
G10.add_edge(649,4249,weight=1)
G10.add_edge(649,4276,weight=1)
G10.add_edge(651,964,weight=1)
G10.add_edge(651,2824,weight=1)
G10.add_edge(651,4258,weight=1)
G10.add_edge(651,4318,weight=1)
G10.add_edge(651,4319,weight=1)
G10.add_edge(652,653,weight=1)
G10.add_edge(652,654,weight=1)
G10.add_edge(652,655,weight=1)
G10.add_edge(652,656,weight=1)
G10.add_edge(653,691,weight=1)
G10.add_edge(653,710,weight=1)
G10.add_edge(653,714,weight=1)
G10.add_edge(653,718,weight=1)
G10.add_edge(653,741,weight=1)
G10.add_edge(657,658,weight=1)
G10.add_edge(657,659,weight=1)
G10.add_edge(657,660,weight=1)
G10.add_edge(658,674,weight=1)
G10.add_edge(658,675,weight=1)
G10.add_edge(658,676,weight=1)
G10.add_edge(658,677,weight=1)
G10.add_edge(660,676,weight=1)
G10.add_edge(660,758,weight=1)
G10.add_edge(661,662,weight=1)
G10.add_edge(662,663,weight=1)
G10.add_edge(662,664,weight=1)
G10.add_edge(663,692,weight=1)
G10.add_edge(663,699,weight=1)
G10.add_edge(663,700,weight=1)
G10.add_edge(663,701,weight=1)
G10.add_edge(663,702,weight=1)
G10.add_edge(664,716,weight=1)
G10.add_edge(664,718,weight=1)
G10.add_edge(664,721,weight=1)
G10.add_edge(665,666,weight=1)
G10.add_edge(665,667,weight=1)
G10.add_edge(666,775,weight=1)
G10.add_edge(667,723,weight=1)
G10.add_edge(667,756,weight=1)
G10.add_edge(667,780,weight=1)
G10.add_edge(667,796,weight=1)
G10.add_edge(667,797,weight=1)
G10.add_edge(668,669,weight=1)
G10.add_edge(668,670,weight=1)
G10.add_edge(669,670,weight=1)
G10.add_edge(669,781,weight=1)
G10.add_edge(669,804,weight=1)
G10.add_edge(669,806,weight=1)
G10.add_edge(669,807,weight=1)
G10.add_edge(670,728,weight=1)
G10.add_edge(670,729,weight=1)
G10.add_edge(670,734,weight=1)
G10.add_edge(670,744,weight=1)
G10.add_edge(670,767,weight=1)
G10.add_edge(670,769,weight=1)
G10.add_edge(670,781,weight=1)
G10.add_edge(670,810,weight=1)
G10.add_edge(671,672,weight=1)
G10.add_edge(671,673,weight=1)
G10.add_edge(672,675,weight=1)
G10.add_edge(672,682,weight=1)
G10.add_edge(673,724,weight=1)
G10.add_edge(675,678,weight=1)
G10.add_edge(675,679,weight=1)
G10.add_edge(675,680,weight=1)
G10.add_edge(675,681,weight=1)
G10.add_edge(676,722,weight=1)
G10.add_edge(676,723,weight=1)
G10.add_edge(677,763,weight=1)
G10.add_edge(677,764,weight=1)
G10.add_edge(678,681,weight=1)
G10.add_edge(678,734,weight=1)
G10.add_edge(678,769,weight=1)
G10.add_edge(679,681,weight=1)
G10.add_edge(679,731,weight=1)
G10.add_edge(679,769,weight=1)
G10.add_edge(680,751,weight=1)
G10.add_edge(680,778,weight=1)
G10.add_edge(680,779,weight=1)
G10.add_edge(681,756,weight=1)
G10.add_edge(681,769,weight=1)
G10.add_edge(681,783,weight=1)
G10.add_edge(681,784,weight=1)
G10.add_edge(681,785,weight=1)
G10.add_edge(682,722,weight=1)
G10.add_edge(682,724,weight=1)
G10.add_edge(682,725,weight=1)
G10.add_edge(682,726,weight=1)
G10.add_edge(683,684,weight=1)
G10.add_edge(684,685,weight=1)
G10.add_edge(684,686,weight=1)
G10.add_edge(684,687,weight=1)
G10.add_edge(684,688,weight=1)
G10.add_edge(684,689,weight=1)
G10.add_edge(685,730,weight=1)
G10.add_edge(685,731,weight=1)
G10.add_edge(686,731,weight=1)
G10.add_edge(686,736,weight=1)
G10.add_edge(686,737,weight=1)
G10.add_edge(686,738,weight=1)
G10.add_edge(687,737,weight=1)
G10.add_edge(688,756,weight=1)
G10.add_edge(688,782,weight=1)
G10.add_edge(690,691,weight=1)
G10.add_edge(691,696,weight=1)
G10.add_edge(691,703,weight=1)
G10.add_edge(691,704,weight=1)
G10.add_edge(691,705,weight=1)
G10.add_edge(691,706,weight=1)
G10.add_edge(691,707,weight=1)
G10.add_edge(691,708,weight=1)
G10.add_edge(692,693,weight=1)
G10.add_edge(692,694,weight=1)
G10.add_edge(692,695,weight=1)
G10.add_edge(695,715,weight=1)
G10.add_edge(695,769,weight=1)
G10.add_edge(695,773,weight=1)
G10.add_edge(695,791,weight=1)
G10.add_edge(696,697,weight=1)
G10.add_edge(696,698,weight=1)
G10.add_edge(698,740,weight=1)
G10.add_edge(698,741,weight=1)
G10.add_edge(701,714,weight=1)
G10.add_edge(701,739,weight=1)
G10.add_edge(709,710,weight=1)
G10.add_edge(710,711,weight=1)
G10.add_edge(710,712,weight=1)
G10.add_edge(710,713,weight=1)
G10.add_edge(711,714,weight=1)
G10.add_edge(711,715,weight=1)
G10.add_edge(713,720,weight=1)
G10.add_edge(713,790,weight=1)
G10.add_edge(714,759,weight=1)
G10.add_edge(714,771,weight=1)
G10.add_edge(714,772,weight=1)
G10.add_edge(714,773,weight=1)
G10.add_edge(715,776,weight=1)
G10.add_edge(717,718,weight=1)
G10.add_edge(718,719,weight=1)
G10.add_edge(718,720,weight=1)
G10.add_edge(720,774,weight=1)
G10.add_edge(721,772,weight=1)
G10.add_edge(721,808,weight=1)
G10.add_edge(721,809,weight=1)
G10.add_edge(725,748,weight=1)
G10.add_edge(725,749,weight=1)
G10.add_edge(726,780,weight=1)
G10.add_edge(727,728,weight=1)
G10.add_edge(728,729,weight=1)
G10.add_edge(729,734,weight=1)
G10.add_edge(729,743,weight=1)
G10.add_edge(729,755,weight=1)
G10.add_edge(729,756,weight=1)
G10.add_edge(729,757,weight=1)
G10.add_edge(731,743,weight=1)
G10.add_edge(731,746,weight=1)
G10.add_edge(732,733,weight=1)
G10.add_edge(733,734,weight=1)
G10.add_edge(733,735,weight=1)
G10.add_edge(734,757,weight=1)
G10.add_edge(734,768,weight=1)
G10.add_edge(735,797,weight=1)
G10.add_edge(735,815,weight=1)
G10.add_edge(737,745,weight=1)
G10.add_edge(737,770,weight=1)
G10.add_edge(738,756,weight=1)
G10.add_edge(741,793,weight=1)
G10.add_edge(742,743,weight=1)
G10.add_edge(743,744,weight=1)
G10.add_edge(743,745,weight=1)
G10.add_edge(743,746,weight=1)
G10.add_edge(744,747,weight=1)
G10.add_edge(745,788,weight=1)
G10.add_edge(745,789,weight=1)
G10.add_edge(746,805,weight=1)
G10.add_edge(749,761,weight=1)
G10.add_edge(749,765,weight=1)
G10.add_edge(750,751,weight=1)
G10.add_edge(750,752,weight=1)
G10.add_edge(750,753,weight=1)
G10.add_edge(750,754,weight=1)
G10.add_edge(751,753,weight=1)
G10.add_edge(752,798,weight=1)
G10.add_edge(752,800,weight=1)
G10.add_edge(753,764,weight=1)
G10.add_edge(753,802,weight=1)
G10.add_edge(754,794,weight=1)
G10.add_edge(754,812,weight=1)
G10.add_edge(756,757,weight=1)
G10.add_edge(756,762,weight=1)
G10.add_edge(757,786,weight=1)
G10.add_edge(760,761,weight=1)
G10.add_edge(761,762,weight=1)
G10.add_edge(762,787,weight=1)
G10.add_edge(764,777,weight=1)
G10.add_edge(766,767,weight=1)
G10.add_edge(769,791,weight=1)
G10.add_edge(769,811,weight=1)
G10.add_edge(772,801,weight=1)
G10.add_edge(773,816,weight=1)
G10.add_edge(789,813,weight=1)
G10.add_edge(791,792,weight=1)
G10.add_edge(794,795,weight=1)
G10.add_edge(797,814,weight=1)
G10.add_edge(798,799,weight=1)
G10.add_edge(803,804,weight=1)
G10.add_edge(811,3887,weight=1)
G10.add_edge(811,3907,weight=1)
G10.add_edge(811,3913,weight=1)
G10.add_edge(811,3915,weight=1)
G10.add_edge(811,3916,weight=1)
G10.add_edge(811,3917,weight=1)
G10.add_edge(817,818,weight=1)
G10.add_edge(817,819,weight=1)
G10.add_edge(817,820,weight=1)
G10.add_edge(818,821,weight=1)
G10.add_edge(818,822,weight=1)
G10.add_edge(818,823,weight=1)
G10.add_edge(818,824,weight=1)
G10.add_edge(819,843,weight=1)
G10.add_edge(819,844,weight=1)
G10.add_edge(820,824,weight=1)
G10.add_edge(820,834,weight=1)
G10.add_edge(820,863,weight=1)
G10.add_edge(821,822,weight=1)
G10.add_edge(821,841,weight=1)
G10.add_edge(821,842,weight=1)
G10.add_edge(821,843,weight=1)
G10.add_edge(822,823,weight=1)
G10.add_edge(822,845,weight=1)
G10.add_edge(822,846,weight=1)
G10.add_edge(823,853,weight=1)
G10.add_edge(823,854,weight=1)
G10.add_edge(823,856,weight=1)
G10.add_edge(823,857,weight=1)
G10.add_edge(824,826,weight=1)
G10.add_edge(824,914,weight=1)
G10.add_edge(825,826,weight=1)
G10.add_edge(826,828,weight=1)
G10.add_edge(826,836,weight=1)
G10.add_edge(826,837,weight=1)
G10.add_edge(826,838,weight=1)
G10.add_edge(827,828,weight=1)
G10.add_edge(827,829,weight=1)
G10.add_edge(827,830,weight=1)
G10.add_edge(827,831,weight=1)
G10.add_edge(828,832,weight=1)
G10.add_edge(828,833,weight=1)
G10.add_edge(828,834,weight=1)
G10.add_edge(828,835,weight=1)
G10.add_edge(829,830,weight=1)
G10.add_edge(829,833,weight=1)
G10.add_edge(829,862,weight=1)
G10.add_edge(830,840,weight=1)
G10.add_edge(830,862,weight=1)
G10.add_edge(830,864,weight=1)
G10.add_edge(830,865,weight=1)
G10.add_edge(831,865,weight=1)
G10.add_edge(831,884,weight=1)
G10.add_edge(833,863,weight=1)
G10.add_edge(834,883,weight=1)
G10.add_edge(835,884,weight=1)
G10.add_edge(835,895,weight=1)
G10.add_edge(835,906,weight=1)
G10.add_edge(837,903,weight=1)
G10.add_edge(838,889,weight=1)
G10.add_edge(838,906,weight=1)
G10.add_edge(838,1574,weight=1)
G10.add_edge(839,840,weight=1)
G10.add_edge(840,851,weight=1)
G10.add_edge(840,852,weight=1)
G10.add_edge(840,865,weight=1)
G10.add_edge(840,880,weight=1)
G10.add_edge(840,885,weight=1)
G10.add_edge(840,886,weight=1)
G10.add_edge(844,847,weight=1)
G10.add_edge(845,848,weight=1)
G10.add_edge(846,879,weight=1)
G10.add_edge(846,882,weight=1)
G10.add_edge(847,848,weight=1)
G10.add_edge(848,2410,weight=1)
G10.add_edge(848,2432,weight=1)
G10.add_edge(848,2505,weight=1)
G10.add_edge(849,850,weight=1)
G10.add_edge(849,851,weight=1)
G10.add_edge(849,852,weight=1)
G10.add_edge(850,866,weight=1)
G10.add_edge(850,867,weight=1)
G10.add_edge(851,857,weight=1)
G10.add_edge(851,889,weight=1)
G10.add_edge(852,855,weight=1)
G10.add_edge(852,857,weight=1)
G10.add_edge(852,869,weight=1)
G10.add_edge(852,887,weight=1)
G10.add_edge(852,904,weight=1)
G10.add_edge(852,905,weight=1)
G10.add_edge(854,855,weight=1)
G10.add_edge(855,857,weight=1)
G10.add_edge(856,858,weight=1)
G10.add_edge(856,859,weight=1)
G10.add_edge(856,860,weight=1)
G10.add_edge(856,861,weight=1)
G10.add_edge(858,912,weight=1)
G10.add_edge(859,913,weight=1)
G10.add_edge(860,873,weight=1)
G10.add_edge(861,873,weight=1)
G10.add_edge(862,863,weight=1)
G10.add_edge(865,890,weight=1)
G10.add_edge(866,880,weight=1)
G10.add_edge(866,881,weight=1)
G10.add_edge(867,916,weight=1)
G10.add_edge(868,869,weight=1)
G10.add_edge(870,871,weight=1)
G10.add_edge(871,873,weight=1)
G10.add_edge(871,875,weight=1)
G10.add_edge(871,878,weight=1)
G10.add_edge(871,879,weight=1)
G10.add_edge(872,873,weight=1)
G10.add_edge(873,874,weight=1)
G10.add_edge(875,876,weight=1)
G10.add_edge(875,877,weight=1)
G10.add_edge(876,879,weight=1)
G10.add_edge(881,915,weight=1)
G10.add_edge(882,4788,weight=1)
G10.add_edge(882,4933,weight=1)
G10.add_edge(883,891,weight=1)
G10.add_edge(884,898,weight=1)
G10.add_edge(884,899,weight=1)
G10.add_edge(886,900,weight=1)
G10.add_edge(887,888,weight=1)
G10.add_edge(888,4937,weight=1)
G10.add_edge(888,4939,weight=1)
G10.add_edge(889,901,weight=1)
G10.add_edge(889,902,weight=1)
G10.add_edge(891,892,weight=1)
G10.add_edge(892,2455,weight=1)
G10.add_edge(893,894,weight=1)
G10.add_edge(894,895,weight=1)
G10.add_edge(894,896,weight=1)
G10.add_edge(894,897,weight=1)
G10.add_edge(896,909,weight=1)
G10.add_edge(896,910,weight=1)
G10.add_edge(896,911,weight=1)
G10.add_edge(897,1442,weight=1)
G10.add_edge(899,908,weight=1)
G10.add_edge(902,906,weight=1)
G10.add_edge(902,1258,weight=1)
G10.add_edge(902,1561,weight=1)
G10.add_edge(902,1611,weight=1)
G10.add_edge(902,1612,weight=1)
G10.add_edge(904,907,weight=1)
G10.add_edge(908,909,weight=1)
G10.add_edge(909,911,weight=1)
G10.add_edge(911,1366,weight=1)
G10.add_edge(911,1367,weight=1)
G10.add_edge(912,1574,weight=1)
G10.add_edge(913,1574,weight=1)
G10.add_edge(915,916,weight=1)
G10.add_edge(916,917,weight=1)
G10.add_edge(917,4940,weight=1)
G10.add_edge(918,919,weight=1)
G10.add_edge(918,920,weight=1)
G10.add_edge(919,921,weight=1)
G10.add_edge(919,922,weight=1)
G10.add_edge(920,969,weight=1)
G10.add_edge(920,970,weight=1)
G10.add_edge(921,928,weight=1)
G10.add_edge(921,929,weight=1)
G10.add_edge(922,931,weight=1)
G10.add_edge(922,4511,weight=1)
G10.add_edge(922,4512,weight=1)
G10.add_edge(923,924,weight=1)
G10.add_edge(923,925,weight=1)
G10.add_edge(923,926,weight=1)
G10.add_edge(924,926,weight=1)
G10.add_edge(924,938,weight=1)
G10.add_edge(925,1002,weight=1)
G10.add_edge(927,955,weight=1)
G10.add_edge(927,956,weight=1)
G10.add_edge(927,957,weight=1)
G10.add_edge(928,946,weight=1)
G10.add_edge(929,952,weight=1)
G10.add_edge(930,931,weight=1)
G10.add_edge(932,933,weight=1)
G10.add_edge(932,934,weight=1)
G10.add_edge(932,935,weight=1)
G10.add_edge(932,936,weight=1)
G10.add_edge(932,937,weight=1)
G10.add_edge(932,938,weight=1)
G10.add_edge(932,939,weight=1)
G10.add_edge(932,940,weight=1)
G10.add_edge(932,941,weight=1)
G10.add_edge(932,942,weight=1)
G10.add_edge(932,943,weight=1)
G10.add_edge(932,944,weight=1)
G10.add_edge(932,945,weight=1)
G10.add_edge(933,947,weight=1)
G10.add_edge(933,972,weight=1)
G10.add_edge(933,986,weight=1)
G10.add_edge(933,998,weight=1)
G10.add_edge(936,937,weight=1)
G10.add_edge(936,1006,weight=1)
G10.add_edge(937,954,weight=1)
G10.add_edge(937,1010,weight=1)
G10.add_edge(937,1013,weight=1)
G10.add_edge(946,947,weight=1)
G10.add_edge(946,948,weight=1)
G10.add_edge(946,949,weight=1)
G10.add_edge(946,950,weight=1)
G10.add_edge(946,951,weight=1)
G10.add_edge(947,951,weight=1)
G10.add_edge(947,966,weight=1)
G10.add_edge(947,967,weight=1)
G10.add_edge(947,990,weight=1)
G10.add_edge(947,992,weight=1)
G10.add_edge(947,996,weight=1)
G10.add_edge(947,997,weight=1)
G10.add_edge(948,965,weight=1)
G10.add_edge(950,1002,weight=1)
G10.add_edge(950,1010,weight=1)
G10.add_edge(950,1011,weight=1)
G10.add_edge(951,1003,weight=1)
G10.add_edge(952,953,weight=1)
G10.add_edge(952,954,weight=1)
G10.add_edge(953,954,weight=1)
G10.add_edge(953,1014,weight=1)
G10.add_edge(955,958,weight=1)
G10.add_edge(955,993,weight=1)
G10.add_edge(956,1002,weight=1)
G10.add_edge(956,1007,weight=1)
G10.add_edge(957,1008,weight=1)
G10.add_edge(958,959,weight=1)
G10.add_edge(959,960,weight=1)
G10.add_edge(960,961,weight=1)
G10.add_edge(961,2404,weight=1)
G10.add_edge(962,963,weight=1)
G10.add_edge(963,964,weight=1)
G10.add_edge(964,4330,weight=1)
G10.add_edge(964,4331,weight=1)
G10.add_edge(965,966,weight=1)
G10.add_edge(965,967,weight=1)
G10.add_edge(965,968,weight=1)
G10.add_edge(969,971,weight=1)
G10.add_edge(969,972,weight=1)
G10.add_edge(969,973,weight=1)
G10.add_edge(969,974,weight=1)
G10.add_edge(969,975,weight=1)
G10.add_edge(969,976,weight=1)
G10.add_edge(969,977,weight=1)
G10.add_edge(969,978,weight=1)
G10.add_edge(970,993,weight=1)
G10.add_edge(971,979,weight=1)
G10.add_edge(971,980,weight=1)
G10.add_edge(972,996,weight=1)
G10.add_edge(974,977,weight=1)
G10.add_edge(975,1004,weight=1)
G10.add_edge(976,1001,weight=1)
G10.add_edge(976,1013,weight=1)
G10.add_edge(977,993,weight=1)
G10.add_edge(978,1017,weight=1)
G10.add_edge(978,1018,weight=1)
G10.add_edge(979,981,weight=1)
G10.add_edge(979,982,weight=1)
G10.add_edge(979,983,weight=1)
G10.add_edge(979,984,weight=1)
G10.add_edge(979,985,weight=1)
G10.add_edge(986,987,weight=1)
G10.add_edge(986,988,weight=1)
G10.add_edge(986,989,weight=1)
G10.add_edge(990,991,weight=1)
G10.add_edge(993,994,weight=1)
G10.add_edge(993,995,weight=1)
G10.add_edge(994,1009,weight=1)
G10.add_edge(995,999,weight=1)
G10.add_edge(995,1007,weight=1)
G10.add_edge(995,1008,weight=1)
G10.add_edge(995,1015,weight=1)
G10.add_edge(995,1016,weight=1)
G10.add_edge(996,997,weight=1)
G10.add_edge(996,1003,weight=1)
G10.add_edge(996,1004,weight=1)
G10.add_edge(997,1004,weight=1)
G10.add_edge(997,1005,weight=1)
G10.add_edge(999,1000,weight=1)
G10.add_edge(999,1001,weight=1)
G10.add_edge(1000,1004,weight=1)
G10.add_edge(1000,1015,weight=1)
G10.add_edge(1001,1012,weight=1)
G10.add_edge(1002,1007,weight=1)
G10.add_edge(1003,1012,weight=1)
G10.add_edge(1005,1023,weight=1)
G10.add_edge(1005,1024,weight=1)
G10.add_edge(1006,1009,weight=1)
G10.add_edge(1007,1008,weight=1)
G10.add_edge(1007,1011,weight=1)
G10.add_edge(1007,1014,weight=1)
G10.add_edge(1008,1014,weight=1)
G10.add_edge(1008,1016,weight=1)
G10.add_edge(1009,1014,weight=1)
G10.add_edge(1012,1013,weight=1)
G10.add_edge(1012,1014,weight=1)
G10.add_edge(1017,1019,weight=1)
G10.add_edge(1017,1020,weight=1)
G10.add_edge(1018,1021,weight=1)
G10.add_edge(1018,1022,weight=1)
G10.add_edge(1023,1025,weight=1)
G10.add_edge(1023,1026,weight=1)
G10.add_edge(1024,1027,weight=1)
G10.add_edge(1024,1028,weight=1)
G10.add_edge(1029,1030,weight=1)
G10.add_edge(1029,1031,weight=1)
G10.add_edge(1030,1036,weight=1)
G10.add_edge(1030,1057,weight=1)
G10.add_edge(1032,1033,weight=1)
G10.add_edge(1032,1034,weight=1)
G10.add_edge(1032,1035,weight=1)
G10.add_edge(1033,1035,weight=1)
G10.add_edge(1034,1036,weight=1)
G10.add_edge(1035,1060,weight=1)
G10.add_edge(1035,1101,weight=1)
G10.add_edge(1036,1050,weight=1)
G10.add_edge(1036,1051,weight=1)
G10.add_edge(1037,1038,weight=1)
G10.add_edge(1037,1039,weight=1)
G10.add_edge(1037,1040,weight=1)
G10.add_edge(1037,1041,weight=1)
G10.add_edge(1039,1041,weight=1)
G10.add_edge(1039,1086,weight=1)
G10.add_edge(1040,1080,weight=1)
G10.add_edge(1040,1083,weight=1)
G10.add_edge(1040,1085,weight=1)
G10.add_edge(1040,1120,weight=1)
G10.add_edge(1041,1044,weight=1)
G10.add_edge(1041,1046,weight=1)
G10.add_edge(1041,1049,weight=1)
G10.add_edge(1041,1081,weight=1)
G10.add_edge(1041,1082,weight=1)
G10.add_edge(1041,1136,weight=1)
G10.add_edge(1041,1137,weight=1)
G10.add_edge(1042,1043,weight=1)
G10.add_edge(1042,1044,weight=1)
G10.add_edge(1042,1045,weight=1)
G10.add_edge(1042,1046,weight=1)
G10.add_edge(1043,1047,weight=1)
G10.add_edge(1043,1048,weight=1)
G10.add_edge(1043,1049,weight=1)
G10.add_edge(1044,1088,weight=1)
G10.add_edge(1044,1089,weight=1)
G10.add_edge(1046,1119,weight=1)
G10.add_edge(1047,1089,weight=1)
G10.add_edge(1047,1134,weight=1)
G10.add_edge(1050,1052,weight=1)
G10.add_edge(1050,1053,weight=1)
G10.add_edge(1051,1124,weight=1)
G10.add_edge(1051,1126,weight=1)
G10.add_edge(1052,1063,weight=1)
G10.add_edge(1053,1061,weight=1)
G10.add_edge(1053,1090,weight=1)
G10.add_edge(1054,1055,weight=1)
G10.add_edge(1054,1056,weight=1)
G10.add_edge(1054,1057,weight=1)
G10.add_edge(1054,1058,weight=1)
G10.add_edge(1055,1058,weight=1)
G10.add_edge(1055,1059,weight=1)
G10.add_edge(1055,1060,weight=1)
G10.add_edge(1056,1129,weight=1)
G10.add_edge(1057,1061,weight=1)
G10.add_edge(1057,1076,weight=1)
G10.add_edge(1057,1092,weight=1)
G10.add_edge(1057,1104,weight=1)
G10.add_edge(1057,1107,weight=1)
G10.add_edge(1057,1128,weight=1)
G10.add_edge(1057,1131,weight=1)
G10.add_edge(1057,1132,weight=1)
G10.add_edge(1057,1133,weight=1)
G10.add_edge(1058,1060,weight=1)
G10.add_edge(1058,1130,weight=1)
G10.add_edge(1059,1061,weight=1)
G10.add_edge(1059,1062,weight=1)
G10.add_edge(1061,1103,weight=1)
G10.add_edge(1061,1105,weight=1)
G10.add_edge(1064,1065,weight=1)
G10.add_edge(1064,1066,weight=1)
G10.add_edge(1064,1067,weight=1)
G10.add_edge(1064,1068,weight=1)
G10.add_edge(1064,1069,weight=1)
G10.add_edge(1065,1070,weight=1)
G10.add_edge(1065,1071,weight=1)
G10.add_edge(1065,1072,weight=1)
G10.add_edge(1066,1070,weight=1)
G10.add_edge(1066,1073,weight=1)
G10.add_edge(1066,1074,weight=1)
G10.add_edge(1067,1099,weight=1)
G10.add_edge(1068,1113,weight=1)
G10.add_edge(1068,1114,weight=1)
G10.add_edge(1069,1112,weight=1)
G10.add_edge(1070,1075,weight=1)
G10.add_edge(1070,1076,weight=1)
G10.add_edge(1070,1077,weight=1)
G10.add_edge(1071,1091,weight=1)
G10.add_edge(1072,1104,weight=1)
G10.add_edge(1073,1075,weight=1)
G10.add_edge(1073,1106,weight=1)
G10.add_edge(1074,1112,weight=1)
G10.add_edge(1074,1125,weight=1)
G10.add_edge(1074,1150,weight=1)
G10.add_edge(1075,1104,weight=1)
G10.add_edge(1075,1107,weight=1)
G10.add_edge(1077,1104,weight=1)
G10.add_edge(1078,1079,weight=1)
G10.add_edge(1078,1080,weight=1)
G10.add_edge(1078,1081,weight=1)
G10.add_edge(1078,1082,weight=1)
G10.add_edge(1078,1083,weight=1)
G10.add_edge(1079,1084,weight=1)
G10.add_edge(1079,1085,weight=1)
G10.add_edge(1080,1111,weight=1)
G10.add_edge(1081,1109,weight=1)
G10.add_edge(1081,1140,weight=1)
G10.add_edge(1082,1109,weight=1)
G10.add_edge(1082,1140,weight=1)
G10.add_edge(1083,1151,weight=1)
G10.add_edge(1084,1087,weight=1)
G10.add_edge(1085,1097,weight=1)
G10.add_edge(1086,1094,weight=1)
G10.add_edge(1086,1101,weight=1)
G10.add_edge(1086,1108,weight=1)
G10.add_edge(1089,1117,weight=1)
G10.add_edge(1090,1091,weight=1)
G10.add_edge(1090,1092,weight=1)
G10.add_edge(1091,1092,weight=1)
G10.add_edge(1091,1114,weight=1)
G10.add_edge(1092,1150,weight=1)
G10.add_edge(1093,1094,weight=1)
G10.add_edge(1093,1095,weight=1)
G10.add_edge(1094,1100,weight=1)
G10.add_edge(1094,1101,weight=1)
G10.add_edge(1095,1123,weight=1)
G10.add_edge(1096,1097,weight=1)
G10.add_edge(1096,1098,weight=1)
G10.add_edge(1097,1111,weight=1)
G10.add_edge(1098,1121,weight=1)
G10.add_edge(1099,1104,weight=1)
G10.add_edge(1101,1130,weight=1)
G10.add_edge(1101,1139,weight=1)
G10.add_edge(1102,1103,weight=1)
G10.add_edge(1102,1104,weight=1)
G10.add_edge(1103,1105,weight=1)
G10.add_edge(1104,1129,weight=1)
G10.add_edge(1104,1141,weight=1)
G10.add_edge(1106,1127,weight=1)
G10.add_edge(1109,1110,weight=1)
G10.add_edge(1110,1136,weight=1)
G10.add_edge(1110,1138,weight=1)
G10.add_edge(1110,1148,weight=1)
G10.add_edge(1113,1114,weight=1)
G10.add_edge(1114,1125,weight=1)
G10.add_edge(1115,1116,weight=1)
G10.add_edge(1116,4895,weight=1)
G10.add_edge(1117,1118,weight=1)
G10.add_edge(1120,1121,weight=1)
G10.add_edge(1120,1122,weight=1)
G10.add_edge(1121,1122,weight=1)
G10.add_edge(1122,1151,weight=1)
G10.add_edge(1127,1128,weight=1)
G10.add_edge(1135,1136,weight=1)
G10.add_edge(1135,1137,weight=1)
G10.add_edge(1136,1138,weight=1)
G10.add_edge(1137,1138,weight=1)
G10.add_edge(1137,1140,weight=1)
G10.add_edge(1138,1140,weight=1)
G10.add_edge(1143,1149,weight=1)
G10.add_edge(1149,4906,weight=1)
G10.add_edge(1149,4924,weight=1)
G10.add_edge(1151,1152,weight=1)
G10.add_edge(1153,1154,weight=1)
G10.add_edge(1153,1155,weight=1)
G10.add_edge(1153,1156,weight=1)
G10.add_edge(1153,1157,weight=1)
G10.add_edge(1153,1158,weight=1)
G10.add_edge(1154,1159,weight=1)
G10.add_edge(1155,1159,weight=1)
G10.add_edge(1156,1891,weight=1)
G10.add_edge(1156,1892,weight=1)
G10.add_edge(1157,2010,weight=1)
G10.add_edge(1158,1891,weight=1)
G10.add_edge(1159,1400,weight=1)
G10.add_edge(1159,1766,weight=1)
G10.add_edge(1160,1161,weight=1)
G10.add_edge(1160,1162,weight=1)
G10.add_edge(1160,1163,weight=1)
G10.add_edge(1160,1164,weight=1)
G10.add_edge(1160,1165,weight=1)
G10.add_edge(1161,1168,weight=1)
G10.add_edge(1162,1359,weight=1)
G10.add_edge(1162,1456,weight=1)
G10.add_edge(1163,1626,weight=1)
G10.add_edge(1164,2019,weight=1)
G10.add_edge(1165,2022,weight=1)
G10.add_edge(1166,1167,weight=1)
G10.add_edge(1167,1260,weight=1)
G10.add_edge(1168,1234,weight=1)
G10.add_edge(1168,1240,weight=1)
G10.add_edge(1168,1242,weight=1)
G10.add_edge(1168,1256,weight=1)
G10.add_edge(1168,1257,weight=1)
G10.add_edge(1168,1258,weight=1)
G10.add_edge(1168,1259,weight=1)
G10.add_edge(1168,1260,weight=1)
G10.add_edge(1168,1261,weight=1)
G10.add_edge(1169,1170,weight=1)
G10.add_edge(1169,1171,weight=1)
G10.add_edge(1169,1172,weight=1)
G10.add_edge(1170,1761,weight=1)
G10.add_edge(1171,2166,weight=1)
G10.add_edge(1172,1809,weight=1)
G10.add_edge(1173,1174,weight=1)
G10.add_edge(1173,1175,weight=1)
G10.add_edge(1173,1176,weight=1)
G10.add_edge(1174,1177,weight=1)
G10.add_edge(1175,1177,weight=1)
G10.add_edge(1175,1803,weight=1)
G10.add_edge(1175,1804,weight=1)
G10.add_edge(1176,1341,weight=1)
G10.add_edge(1176,1999,weight=1)
G10.add_edge(1177,1344,weight=1)
G10.add_edge(1177,1345,weight=1)
G10.add_edge(1177,1619,weight=1)
G10.add_edge(1177,1652,weight=1)
G10.add_edge(1177,1805,weight=1)
G10.add_edge(1177,1806,weight=1)
G10.add_edge(1178,1179,weight=1)
G10.add_edge(1178,1180,weight=1)
G10.add_edge(1178,1181,weight=1)
G10.add_edge(1178,1182,weight=1)
G10.add_edge(1178,1183,weight=1)
G10.add_edge(1178,1184,weight=1)
G10.add_edge(1178,1185,weight=1)
G10.add_edge(1179,1186,weight=1)
G10.add_edge(1179,1187,weight=1)
G10.add_edge(1180,1181,weight=1)
G10.add_edge(1181,1220,weight=1)
G10.add_edge(1182,1397,weight=1)
G10.add_edge(1182,1651,weight=1)
G10.add_edge(1182,1672,weight=1)
G10.add_edge(1183,1620,weight=1)
G10.add_edge(1184,1619,weight=1)
G10.add_edge(1185,1935,weight=1)
G10.add_edge(1185,1936,weight=1)
G10.add_edge(1186,1618,weight=1)
G10.add_edge(1186,1623,weight=1)
G10.add_edge(1187,1477,weight=1)
G10.add_edge(1187,1754,weight=1)
G10.add_edge(1187,1765,weight=1)
G10.add_edge(1187,1766,weight=1)
G10.add_edge(1187,1767,weight=1)
G10.add_edge(1187,1768,weight=1)
G10.add_edge(1188,1189,weight=1)
G10.add_edge(1188,1190,weight=1)
G10.add_edge(1188,1191,weight=1)
G10.add_edge(1189,1900,weight=1)
G10.add_edge(1190,2009,weight=1)
G10.add_edge(1191,1677,weight=1)
G10.add_edge(1192,1193,weight=1)
G10.add_edge(1192,1194,weight=1)
G10.add_edge(1192,1195,weight=1)
G10.add_edge(1192,1196,weight=1)
G10.add_edge(1192,1197,weight=1)
G10.add_edge(1192,1198,weight=1)
G10.add_edge(1192,1199,weight=1)
G10.add_edge(1193,1200,weight=1)
G10.add_edge(1193,1201,weight=1)
G10.add_edge(1194,1200,weight=1)
G10.add_edge(1194,1497,weight=1)
G10.add_edge(1194,1508,weight=1)
G10.add_edge(1194,1509,weight=1)
G10.add_edge(1195,1538,weight=1)
G10.add_edge(1195,1539,weight=1)
G10.add_edge(1196,1641,weight=1)
G10.add_edge(1196,1642,weight=1)
G10.add_edge(1196,1643,weight=1)
G10.add_edge(1197,1559,weight=1)
G10.add_edge(1197,1690,weight=1)
G10.add_edge(1198,1374,weight=1)
G10.add_edge(1199,1915,weight=1)
G10.add_edge(1199,1916,weight=1)
G10.add_edge(1199,1917,weight=1)
G10.add_edge(1199,1918,weight=1)
G10.add_edge(1199,1919,weight=1)
G10.add_edge(1199,1920,weight=1)
G10.add_edge(1199,1921,weight=1)
G10.add_edge(1199,1922,weight=1)
G10.add_edge(1200,1379,weight=1)
G10.add_edge(1201,1211,weight=1)
G10.add_edge(1201,1324,weight=1)
G10.add_edge(1201,1364,weight=1)
G10.add_edge(1201,1623,weight=1)
G10.add_edge(1201,1641,weight=1)
G10.add_edge(1201,1689,weight=1)
G10.add_edge(1202,1203,weight=1)
G10.add_edge(1202,1204,weight=1)
G10.add_edge(1202,1205,weight=1)
G10.add_edge(1203,1206,weight=1)
G10.add_edge(1204,1205,weight=1)
G10.add_edge(1204,1257,weight=1)
G10.add_edge(1204,1389,weight=1)
G10.add_edge(1204,1434,weight=1)
G10.add_edge(1204,1451,weight=1)
G10.add_edge(1204,1455,weight=1)
G10.add_edge(1205,1217,weight=1)
G10.add_edge(1205,1517,weight=1)
G10.add_edge(1205,1596,weight=1)
G10.add_edge(1205,1776,weight=1)
G10.add_edge(1205,1868,weight=1)
G10.add_edge(1205,1869,weight=1)
G10.add_edge(1205,1870,weight=1)
G10.add_edge(1206,1207,weight=1)
G10.add_edge(1206,1208,weight=1)
G10.add_edge(1207,1549,weight=1)
G10.add_edge(1208,1451,weight=1)
G10.add_edge(1209,1210,weight=1)
G10.add_edge(1210,1223,weight=1)
G10.add_edge(1210,1231,weight=1)
G10.add_edge(1210,1564,weight=1)
G10.add_edge(1210,1565,weight=1)
G10.add_edge(1210,1566,weight=1)
G10.add_edge(1210,1567,weight=1)
G10.add_edge(1210,1568,weight=1)
G10.add_edge(1210,1569,weight=1)
G10.add_edge(1210,1570,weight=1)
G10.add_edge(1210,1571,weight=1)
G10.add_edge(1210,1572,weight=1)
G10.add_edge(1211,1212,weight=1)
G10.add_edge(1211,1213,weight=1)
G10.add_edge(1211,1214,weight=1)
G10.add_edge(1211,1215,weight=1)
G10.add_edge(1213,1214,weight=1)
G10.add_edge(1213,1324,weight=1)
G10.add_edge(1213,1325,weight=1)
G10.add_edge(1213,1421,weight=1)
G10.add_edge(1213,1596,weight=1)
G10.add_edge(1214,1662,weight=1)
G10.add_edge(1214,1674,weight=1)
G10.add_edge(1214,1675,weight=1)
G10.add_edge(1215,1324,weight=1)
G10.add_edge(1215,1332,weight=1)
G10.add_edge(1215,1364,weight=1)
G10.add_edge(1215,1711,weight=1)
G10.add_edge(1216,1217,weight=1)
G10.add_edge(1216,1218,weight=1)
G10.add_edge(1217,1279,weight=1)
G10.add_edge(1217,1473,weight=1)
G10.add_edge(1217,1584,weight=1)
G10.add_edge(1217,1715,weight=1)
G10.add_edge(1217,1718,weight=1)
G10.add_edge(1217,1719,weight=1)
G10.add_edge(1217,1720,weight=1)
G10.add_edge(1217,1721,weight=1)
G10.add_edge(1217,1722,weight=1)
G10.add_edge(1218,1886,weight=1)
G10.add_edge(1219,1220,weight=1)
G10.add_edge(1219,1221,weight=1)
G10.add_edge(1219,1222,weight=1)
G10.add_edge(1221,1929,weight=1)
G10.add_edge(1222,1375,weight=1)
G10.add_edge(1223,1224,weight=1)
G10.add_edge(1223,1225,weight=1)
G10.add_edge(1223,1226,weight=1)
G10.add_edge(1224,1792,weight=1)
G10.add_edge(1224,1872,weight=1)
G10.add_edge(1225,1938,weight=1)
G10.add_edge(1227,1228,weight=1)
G10.add_edge(1227,1229,weight=1)
G10.add_edge(1227,1230,weight=1)
G10.add_edge(1229,1247,weight=1)
G10.add_edge(1229,1248,weight=1)
G10.add_edge(1229,1265,weight=1)
G10.add_edge(1229,1269,weight=1)
G10.add_edge(1229,1822,weight=1)
G10.add_edge(1231,1232,weight=1)
G10.add_edge(1232,1279,weight=1)
G10.add_edge(1232,1280,weight=1)
G10.add_edge(1232,1281,weight=1)
G10.add_edge(1232,1282,weight=1)
G10.add_edge(1232,1283,weight=1)
G10.add_edge(1232,1284,weight=1)
G10.add_edge(1232,1285,weight=1)
G10.add_edge(1233,1234,weight=1)
G10.add_edge(1233,1235,weight=1)
G10.add_edge(1233,1236,weight=1)
G10.add_edge(1233,1237,weight=1)
G10.add_edge(1233,1238,weight=1)
G10.add_edge(1233,1239,weight=1)
G10.add_edge(1234,1240,weight=1)
G10.add_edge(1234,1241,weight=1)
G10.add_edge(1235,1240,weight=1)
G10.add_edge(1235,1242,weight=1)
G10.add_edge(1235,1243,weight=1)
G10.add_edge(1235,1244,weight=1)
G10.add_edge(1235,1245,weight=1)
G10.add_edge(1236,1255,weight=1)
G10.add_edge(1237,1245,weight=1)
G10.add_edge(1237,1760,weight=1)
G10.add_edge(1237,1761,weight=1)
G10.add_edge(1238,1560,weight=1)
G10.add_edge(1239,1503,weight=1)
G10.add_edge(1239,1531,weight=1)
G10.add_edge(1239,1575,weight=1)
G10.add_edge(1239,1603,weight=1)
G10.add_edge(1239,1853,weight=1)
G10.add_edge(1240,1246,weight=1)
G10.add_edge(1241,1274,weight=1)
G10.add_edge(1241,1277,weight=1)
G10.add_edge(1241,1278,weight=1)
G10.add_edge(1242,1250,weight=1)
G10.add_edge(1242,1251,weight=1)
G10.add_edge(1242,1252,weight=1)
G10.add_edge(1242,1253,weight=1)
G10.add_edge(1242,1254,weight=1)
G10.add_edge(1242,1255,weight=1)
G10.add_edge(1243,1560,weight=1)
G10.add_edge(1244,1563,weight=1)
G10.add_edge(1246,1371,weight=1)
G10.add_edge(1246,1761,weight=1)
G10.add_edge(1247,1248,weight=1)
G10.add_edge(1247,1249,weight=1)
G10.add_edge(1248,1685,weight=1)
G10.add_edge(1248,1686,weight=1)
G10.add_edge(1248,1687,weight=1)
G10.add_edge(1248,1688,weight=1)
G10.add_edge(1249,1303,weight=1)
G10.add_edge(1250,1252,weight=1)
G10.add_edge(1250,1326,weight=1)
G10.add_edge(1251,1257,weight=1)
G10.add_edge(1251,1468,weight=1)
G10.add_edge(1251,1469,weight=1)
G10.add_edge(1251,1470,weight=1)
G10.add_edge(1252,1866,weight=1)
G10.add_edge(1253,2023,weight=1)
G10.add_edge(1254,2049,weight=1)
G10.add_edge(1255,2237,weight=1)
G10.add_edge(1256,1262,weight=1)
G10.add_edge(1257,1261,weight=1)
G10.add_edge(1257,1417,weight=1)
G10.add_edge(1257,1421,weight=1)
G10.add_edge(1257,1471,weight=1)
G10.add_edge(1257,1472,weight=1)
G10.add_edge(1257,1473,weight=1)
G10.add_edge(1257,1474,weight=1)
G10.add_edge(1258,1278,weight=1)
G10.add_edge(1258,1371,weight=1)
G10.add_edge(1258,1602,weight=1)
G10.add_edge(1258,1741,weight=1)
G10.add_edge(1258,1742,weight=1)
G10.add_edge(1259,1809,weight=1)
G10.add_edge(1261,1563,weight=1)
G10.add_edge(1262,1525,weight=1)
G10.add_edge(1262,1574,weight=1)
G10.add_edge(1262,1611,weight=1)
G10.add_edge(1263,1264,weight=1)
G10.add_edge(1263,1265,weight=1)
G10.add_edge(1263,1266,weight=1)
G10.add_edge(1263,1267,weight=1)
G10.add_edge(1264,1268,weight=1)
G10.add_edge(1264,1269,weight=1)
G10.add_edge(1265,1270,weight=1)
G10.add_edge(1265,1271,weight=1)
G10.add_edge(1265,1272,weight=1)
G10.add_edge(1265,1273,weight=1)
G10.add_edge(1266,1267,weight=1)
G10.add_edge(1266,1270,weight=1)
G10.add_edge(1266,2258,weight=1)
G10.add_edge(1268,1491,weight=1)
G10.add_edge(1268,1492,weight=1)
G10.add_edge(1269,1354,weight=1)
G10.add_edge(1269,1685,weight=1)
G10.add_edge(1269,1739,weight=1)
G10.add_edge(1270,1273,weight=1)
G10.add_edge(1270,1492,weight=1)
G10.add_edge(1271,2249,weight=1)
G10.add_edge(1271,2250,weight=1)
G10.add_edge(1272,2258,weight=1)
G10.add_edge(1273,2268,weight=1)
G10.add_edge(1273,2269,weight=1)
G10.add_edge(1274,1275,weight=1)
G10.add_edge(1274,1276,weight=1)
G10.add_edge(1275,1650,weight=1)
G10.add_edge(1276,1775,weight=1)
G10.add_edge(1277,1604,weight=1)
G10.add_edge(1277,1733,weight=1)
G10.add_edge(1278,1773,weight=1)
G10.add_edge(1280,1510,weight=1)
G10.add_edge(1281,1758,weight=1)
G10.add_edge(1282,1715,weight=1)
G10.add_edge(1282,2086,weight=1)
G10.add_edge(1283,1572,weight=1)
G10.add_edge(1283,2120,weight=1)
G10.add_edge(1284,1887,weight=1)
G10.add_edge(1285,2033,weight=1)
G10.add_edge(1286,1287,weight=1)
G10.add_edge(1286,1288,weight=1)
G10.add_edge(1286,1289,weight=1)
G10.add_edge(1286,1290,weight=1)
G10.add_edge(1286,1291,weight=1)
G10.add_edge(1286,1292,weight=1)
G10.add_edge(1287,1293,weight=1)
G10.add_edge(1287,1294,weight=1)
G10.add_edge(1288,1293,weight=1)
G10.add_edge(1288,1432,weight=1)
G10.add_edge(1288,1635,weight=1)
G10.add_edge(1288,1636,weight=1)
G10.add_edge(1288,1637,weight=1)
G10.add_edge(1289,1390,weight=1)
G10.add_edge(1289,1428,weight=1)
G10.add_edge(1289,1430,weight=1)
G10.add_edge(1289,1431,weight=1)
G10.add_edge(1289,1543,weight=1)
G10.add_edge(1289,1544,weight=1)
G10.add_edge(1289,1847,weight=1)
G10.add_edge(1289,1890,weight=1)
G10.add_edge(1290,1842,weight=1)
G10.add_edge(1291,1543,weight=1)
G10.add_edge(1292,1432,weight=1)
G10.add_edge(1293,1638,weight=1)
G10.add_edge(1294,1432,weight=1)
G10.add_edge(1294,1435,weight=1)
G10.add_edge(1294,1480,weight=1)
G10.add_edge(1295,1296,weight=1)
G10.add_edge(1295,1297,weight=1)
G10.add_edge(1295,1298,weight=1)
G10.add_edge(1295,1299,weight=1)
G10.add_edge(1295,1300,weight=1)
G10.add_edge(1296,1301,weight=1)
G10.add_edge(1296,1302,weight=1)
G10.add_edge(1297,1373,weight=1)
G10.add_edge(1297,1573,weight=1)
G10.add_edge(1298,1320,weight=1)
G10.add_edge(1298,1728,weight=1)
G10.add_edge(1299,1595,weight=1)
G10.add_edge(1299,1693,weight=1)
G10.add_edge(1299,1771,weight=1)
G10.add_edge(1299,1807,weight=1)
G10.add_edge(1300,1941,weight=1)
G10.add_edge(1300,1942,weight=1)
G10.add_edge(1301,1310,weight=1)
G10.add_edge(1301,1409,weight=1)
G10.add_edge(1301,1410,weight=1)
G10.add_edge(1302,1500,weight=1)
G10.add_edge(1302,1538,weight=1)
G10.add_edge(1302,1689,weight=1)
G10.add_edge(1302,1819,weight=1)
G10.add_edge(1302,1820,weight=1)
G10.add_edge(1302,1821,weight=1)
G10.add_edge(1303,1304,weight=1)
G10.add_edge(1303,1305,weight=1)
G10.add_edge(1303,1306,weight=1)
G10.add_edge(1303,1307,weight=1)
G10.add_edge(1303,1308,weight=1)
G10.add_edge(1304,1351,weight=1)
G10.add_edge(1304,1829,weight=1)
G10.add_edge(1304,1830,weight=1)
G10.add_edge(1304,1831,weight=1)
G10.add_edge(1304,1832,weight=1)
G10.add_edge(1304,1833,weight=1)
G10.add_edge(1304,1834,weight=1)
G10.add_edge(1304,1835,weight=1)
G10.add_edge(1304,1836,weight=1)
G10.add_edge(1305,2126,weight=1)
G10.add_edge(1306,1465,weight=1)
G10.add_edge(1307,2242,weight=1)
G10.add_edge(1308,2218,weight=1)
G10.add_edge(1309,1310,weight=1)
G10.add_edge(1309,1311,weight=1)
G10.add_edge(1309,1312,weight=1)
G10.add_edge(1309,1313,weight=1)
G10.add_edge(1309,1314,weight=1)
G10.add_edge(1310,1315,weight=1)
G10.add_edge(1310,1316,weight=1)
G10.add_edge(1310,1317,weight=1)
G10.add_edge(1310,1318,weight=1)
G10.add_edge(1310,1319,weight=1)
G10.add_edge(1310,1320,weight=1)
G10.add_edge(1310,1321,weight=1)
G10.add_edge(1310,1322,weight=1)
G10.add_edge(1311,1413,weight=1)
G10.add_edge(1311,1414,weight=1)
G10.add_edge(1313,2029,weight=1)
G10.add_edge(1314,1857,weight=1)
G10.add_edge(1315,1323,weight=1)
G10.add_edge(1315,1324,weight=1)
G10.add_edge(1315,1325,weight=1)
G10.add_edge(1316,1323,weight=1)
G10.add_edge(1316,1394,weight=1)
G10.add_edge(1316,1395,weight=1)
G10.add_edge(1318,1600,weight=1)
G10.add_edge(1319,1693,weight=1)
G10.add_edge(1319,1694,weight=1)
G10.add_edge(1320,1385,weight=1)
G10.add_edge(1320,1704,weight=1)
G10.add_edge(1320,1705,weight=1)
G10.add_edge(1320,1706,weight=1)
G10.add_edge(1320,1707,weight=1)
G10.add_edge(1321,1584,weight=1)
G10.add_edge(1322,1346,weight=1)
G10.add_edge(1324,1365,weight=1)
G10.add_edge(1324,1617,weight=1)
G10.add_edge(1325,1623,weight=1)
G10.add_edge(1325,1704,weight=1)
G10.add_edge(1325,1757,weight=1)
G10.add_edge(1327,1328,weight=1)
G10.add_edge(1327,1329,weight=1)
G10.add_edge(1327,1330,weight=1)
G10.add_edge(1328,1542,weight=1)
G10.add_edge(1328,1543,weight=1)
G10.add_edge(1328,1544,weight=1)
G10.add_edge(1329,1750,weight=1)
G10.add_edge(1329,1845,weight=1)
G10.add_edge(1329,1847,weight=1)
G10.add_edge(1330,1756,weight=1)
G10.add_edge(1330,1765,weight=1)
G10.add_edge(1331,1332,weight=1)
G10.add_edge(1333,1334,weight=1)
G10.add_edge(1333,1335,weight=1)
G10.add_edge(1334,1448,weight=1)
G10.add_edge(1334,1949,weight=1)
G10.add_edge(1335,1950,weight=1)
G10.add_edge(1336,1337,weight=1)
G10.add_edge(1337,1344,weight=1)
G10.add_edge(1337,1345,weight=1)
G10.add_edge(1337,1346,weight=1)
G10.add_edge(1337,1347,weight=1)
G10.add_edge(1338,1339,weight=1)
G10.add_edge(1338,1340,weight=1)
G10.add_edge(1339,2183,weight=1)
G10.add_edge(1340,2209,weight=1)
G10.add_edge(1341,1342,weight=1)
G10.add_edge(1341,1343,weight=1)
G10.add_edge(1342,1893,weight=1)
G10.add_edge(1343,1978,weight=1)
G10.add_edge(1344,1825,weight=1)
G10.add_edge(1345,1826,weight=1)
G10.add_edge(1346,1593,weight=1)
G10.add_edge(1346,1660,weight=1)
G10.add_edge(1346,1757,weight=1)
G10.add_edge(1347,1718,weight=1)
G10.add_edge(1348,1349,weight=1)
G10.add_edge(1348,1350,weight=1)
G10.add_edge(1348,1351,weight=1)
G10.add_edge(1348,1352,weight=1)
G10.add_edge(1348,1353,weight=1)
G10.add_edge(1348,1354,weight=1)
G10.add_edge(1349,1505,weight=1)
G10.add_edge(1349,1506,weight=1)
G10.add_edge(1350,1464,weight=1)
G10.add_edge(1350,1480,weight=1)
G10.add_edge(1350,1731,weight=1)
G10.add_edge(1351,1506,weight=1)
G10.add_edge(1351,1732,weight=1)
G10.add_edge(1351,1739,weight=1)
G10.add_edge(1351,1837,weight=1)
G10.add_edge(1351,1838,weight=1)
G10.add_edge(1352,1639,weight=1)
G10.add_edge(1352,1731,weight=1)
G10.add_edge(1352,1839,weight=1)
G10.add_edge(1353,1956,weight=1)
G10.add_edge(1353,1957,weight=1)
G10.add_edge(1353,1958,weight=1)
G10.add_edge(1353,1959,weight=1)
G10.add_edge(1353,1960,weight=1)
G10.add_edge(1353,1961,weight=1)
G10.add_edge(1355,1356,weight=1)
G10.add_edge(1356,1358,weight=1)
G10.add_edge(1357,4506,weight=1)
G10.add_edge(1357,4507,weight=1)
G10.add_edge(1358,1359,weight=1)
G10.add_edge(1360,1361,weight=1)
G10.add_edge(1361,1362,weight=1)
G10.add_edge(1361,1363,weight=1)
G10.add_edge(1362,1458,weight=1)
G10.add_edge(1362,1462,weight=1)
G10.add_edge(1363,1425,weight=1)
G10.add_edge(1363,1634,weight=1)
G10.add_edge(1364,1365,weight=1)
G10.add_edge(1365,1377,weight=1)
G10.add_edge(1365,1378,weight=1)
G10.add_edge(1365,1594,weight=1)
G10.add_edge(1365,1595,weight=1)
G10.add_edge(1366,1774,weight=1)
G10.add_edge(1367,1773,weight=1)
G10.add_edge(1368,1378,weight=1)
G10.add_edge(1368,1594,weight=1)
G10.add_edge(1369,1370,weight=1)
G10.add_edge(1369,1371,weight=1)
G10.add_edge(1370,1372,weight=1)
G10.add_edge(1372,2188,weight=1)
G10.add_edge(1373,1374,weight=1)
G10.add_edge(1375,1376,weight=1)
G10.add_edge(1376,1862,weight=1)
G10.add_edge(1377,1378,weight=1)
G10.add_edge(1377,1379,weight=1)
G10.add_edge(1377,1380,weight=1)
G10.add_edge(1378,1634,weight=1)
G10.add_edge(1378,1678,weight=1)
G10.add_edge(1379,1587,weight=1)
G10.add_edge(1380,2625,weight=1)
G10.add_edge(1380,2795,weight=1)
G10.add_edge(1381,1382,weight=1)
G10.add_edge(1381,1383,weight=1)
G10.add_edge(1381,1384,weight=1)
G10.add_edge(1382,1392,weight=1)
G10.add_edge(1382,1393,weight=1)
G10.add_edge(1383,1860,weight=1)
G10.add_edge(1384,1992,weight=1)
G10.add_edge(1384,1993,weight=1)
G10.add_edge(1384,1994,weight=1)
G10.add_edge(1385,1386,weight=1)
G10.add_edge(1385,1387,weight=1)
G10.add_edge(1386,1590,weight=1)
G10.add_edge(1386,1591,weight=1)
G10.add_edge(1386,1592,weight=1)
G10.add_edge(1386,1593,weight=1)
G10.add_edge(1387,1591,weight=1)
G10.add_edge(1387,2001,weight=1)
G10.add_edge(1387,2002,weight=1)
G10.add_edge(1388,1389,weight=1)
G10.add_edge(1388,1390,weight=1)
G10.add_edge(1388,1391,weight=1)
G10.add_edge(1389,1455,weight=1)
G10.add_edge(1389,1512,weight=1)
G10.add_edge(1389,1798,weight=1)
G10.add_edge(1389,1799,weight=1)
G10.add_edge(1389,1801,weight=1)
G10.add_edge(1390,1434,weight=1)
G10.add_edge(1390,1483,weight=1)
G10.add_edge(1390,1485,weight=1)
G10.add_edge(1392,1620,weight=1)
G10.add_edge(1392,1705,weight=1)
G10.add_edge(1392,1769,weight=1)
G10.add_edge(1392,1823,weight=1)
G10.add_edge(1392,1824,weight=1)
G10.add_edge(1396,1397,weight=1)
G10.add_edge(1396,1398,weight=1)
G10.add_edge(1396,1399,weight=1)
G10.add_edge(1396,1400,weight=1)
G10.add_edge(1396,1401,weight=1)
G10.add_edge(1396,1402,weight=1)
G10.add_edge(1396,1403,weight=1)
G10.add_edge(1396,1404,weight=1)
G10.add_edge(1398,1399,weight=1)
G10.add_edge(1398,1696,weight=1)
G10.add_edge(1399,1482,weight=1)
G10.add_edge(1399,1737,weight=1)
G10.add_edge(1400,1485,weight=1)
G10.add_edge(1400,1553,weight=1)
G10.add_edge(1400,1628,weight=1)
G10.add_edge(1400,1749,weight=1)
G10.add_edge(1400,1754,weight=1)
G10.add_edge(1400,1755,weight=1)
G10.add_edge(1401,1696,weight=1)
G10.add_edge(1401,1828,weight=1)
G10.add_edge(1402,1434,weight=1)
G10.add_edge(1402,1483,weight=1)
G10.add_edge(1403,2014,weight=1)
G10.add_edge(1404,2012,weight=1)
G10.add_edge(1404,2013,weight=1)
G10.add_edge(1405,1406,weight=1)
G10.add_edge(1406,1407,weight=1)
G10.add_edge(1406,1408,weight=1)
G10.add_edge(1407,1647,weight=1)
G10.add_edge(1407,1851,weight=1)
G10.add_edge(1407,1852,weight=1)
G10.add_edge(1408,1799,weight=1)
G10.add_edge(1409,1813,weight=1)
G10.add_edge(1410,1910,weight=1)
G10.add_edge(1410,1971,weight=1)
G10.add_edge(1410,2015,weight=1)
G10.add_edge(1410,2016,weight=1)
G10.add_edge(1411,1412,weight=1)
G10.add_edge(1412,1562,weight=1)
G10.add_edge(1413,1857,weight=1)
G10.add_edge(1413,1858,weight=1)
G10.add_edge(1413,1859,weight=1)
G10.add_edge(1414,2127,weight=1)
G10.add_edge(1415,1416,weight=1)
G10.add_edge(1416,1417,weight=1)
G10.add_edge(1416,1421,weight=1)
G10.add_edge(1416,1422,weight=1)
G10.add_edge(1416,1423,weight=1)
G10.add_edge(1417,1418,weight=1)
G10.add_edge(1417,1419,weight=1)
G10.add_edge(1417,1420,weight=1)
G10.add_edge(1418,1532,weight=1)
G10.add_edge(1419,1837,weight=1)
G10.add_edge(1420,1838,weight=1)
G10.add_edge(1421,1475,weight=1)
G10.add_edge(1422,1684,weight=1)
G10.add_edge(1422,1729,weight=1)
G10.add_edge(1423,1475,weight=1)
G10.add_edge(1423,1512,weight=1)
G10.add_edge(1424,1425,weight=1)
G10.add_edge(1424,1426,weight=1)
G10.add_edge(1425,1427,weight=1)
G10.add_edge(1426,2065,weight=1)
G10.add_edge(1427,1644,weight=1)
G10.add_edge(1427,1646,weight=1)
G10.add_edge(1428,1429,weight=1)
G10.add_edge(1428,1430,weight=1)
G10.add_edge(1428,1431,weight=1)
G10.add_edge(1428,1432,weight=1)
G10.add_edge(1429,1430,weight=1)
G10.add_edge(1429,1723,weight=1)
G10.add_edge(1430,1432,weight=1)
G10.add_edge(1430,1435,weight=1)
G10.add_edge(1430,1640,weight=1)
G10.add_edge(1430,1723,weight=1)
G10.add_edge(1430,1743,weight=1)
G10.add_edge(1431,1850,weight=1)
G10.add_edge(1432,1635,weight=1)
G10.add_edge(1432,1640,weight=1)
G10.add_edge(1432,1682,weight=1)
G10.add_edge(1433,1434,weight=1)
G10.add_edge(1433,1435,weight=1)
G10.add_edge(1433,1436,weight=1)
G10.add_edge(1434,1436,weight=1)
G10.add_edge(1434,1476,weight=1)
G10.add_edge(1434,1478,weight=1)
G10.add_edge(1434,1479,weight=1)
G10.add_edge(1434,1480,weight=1)
G10.add_edge(1436,1483,weight=1)
G10.add_edge(1436,1484,weight=1)
G10.add_edge(1436,1790,weight=1)
G10.add_edge(1436,1856,weight=1)
G10.add_edge(1437,1438,weight=1)
G10.add_edge(1437,1439,weight=1)
G10.add_edge(1438,1693,weight=1)
G10.add_edge(1439,1599,weight=1)
G10.add_edge(1440,1441,weight=1)
G10.add_edge(1440,1442,weight=1)
G10.add_edge(1440,1443,weight=1)
G10.add_edge(1441,1444,weight=1)
G10.add_edge(1441,1445,weight=1)
G10.add_edge(1441,1446,weight=1)
G10.add_edge(1443,2227,weight=1)
G10.add_edge(1444,1446,weight=1)
G10.add_edge(1444,1457,weight=1)
G10.add_edge(1445,1561,weight=1)
G10.add_edge(1446,1613,weight=1)
G10.add_edge(1447,1448,weight=1)
G10.add_edge(1447,1449,weight=1)
G10.add_edge(1447,1450,weight=1)
G10.add_edge(1448,1564,weight=1)
G10.add_edge(1448,1600,weight=1)
G10.add_edge(1448,1708,weight=1)
G10.add_edge(1448,1711,weight=1)
G10.add_edge(1448,1712,weight=1)
G10.add_edge(1448,1713,weight=1)
G10.add_edge(1448,1714,weight=1)
G10.add_edge(1449,1875,weight=1)
G10.add_edge(1450,1709,weight=1)
G10.add_edge(1450,2149,weight=1)
G10.add_edge(1451,1452,weight=1)
G10.add_edge(1451,1453,weight=1)
G10.add_edge(1451,1454,weight=1)
G10.add_edge(1452,1597,weight=1)
G10.add_edge(1452,1700,weight=1)
G10.add_edge(1452,1701,weight=1)
G10.add_edge(1453,1647,weight=1)
G10.add_edge(1454,1545,weight=1)
G10.add_edge(1456,2035,weight=1)
G10.add_edge(1457,1601,weight=1)
G10.add_edge(1458,1459,weight=1)
G10.add_edge(1458,1460,weight=1)
G10.add_edge(1458,1461,weight=1)
G10.add_edge(1459,1589,weight=1)
G10.add_edge(1460,1489,weight=1)
G10.add_edge(1462,1587,weight=1)
G10.add_edge(1462,1667,weight=1)
G10.add_edge(1462,1670,weight=1)
G10.add_edge(1463,1464,weight=1)
G10.add_edge(1463,1465,weight=1)
G10.add_edge(1463,1466,weight=1)
G10.add_edge(1463,1467,weight=1)
G10.add_edge(1464,1466,weight=1)
G10.add_edge(1464,1635,weight=1)
G10.add_edge(1464,1636,weight=1)
G10.add_edge(1467,1636,weight=1)
G10.add_edge(1468,2128,weight=1)
G10.add_edge(1469,1534,weight=1)
G10.add_edge(1470,2048,weight=1)
G10.add_edge(1472,1719,weight=1)
G10.add_edge(1472,1797,weight=1)
G10.add_edge(1473,1817,weight=1)
G10.add_edge(1474,1755,weight=1)
G10.add_edge(1475,1477,weight=1)
G10.add_edge(1475,1596,weight=1)
G10.add_edge(1476,1477,weight=1)
G10.add_edge(1477,1478,weight=1)
G10.add_edge(1477,1684,weight=1)
G10.add_edge(1477,1790,weight=1)
G10.add_edge(1479,1490,weight=1)
G10.add_edge(1480,1540,weight=1)
G10.add_edge(1480,1639,weight=1)
G10.add_edge(1480,1679,weight=1)
G10.add_edge(1480,1680,weight=1)
G10.add_edge(1480,1681,weight=1)
G10.add_edge(1480,1682,weight=1)
G10.add_edge(1480,1683,weight=1)
G10.add_edge(1481,1482,weight=1)
G10.add_edge(1482,2276,weight=1)
G10.add_edge(1483,1484,weight=1)
G10.add_edge(1483,1485,weight=1)
G10.add_edge(1483,1486,weight=1)
G10.add_edge(1484,1843,weight=1)
G10.add_edge(1484,1844,weight=1)
G10.add_edge(1484,1845,weight=1)
G10.add_edge(1486,1844,weight=1)
G10.add_edge(1486,2046,weight=1)
G10.add_edge(1486,2047,weight=1)
G10.add_edge(1487,1488,weight=1)
G10.add_edge(1487,1489,weight=1)
G10.add_edge(1488,1499,weight=1)
G10.add_edge(1490,1540,weight=1)
G10.add_edge(1490,1680,weight=1)
G10.add_edge(1490,1682,weight=1)
G10.add_edge(1493,1494,weight=1)
G10.add_edge(1493,1495,weight=1)
G10.add_edge(1494,1744,weight=1)
G10.add_edge(1495,1846,weight=1)
G10.add_edge(1496,1497,weight=1)
G10.add_edge(1496,1498,weight=1)
G10.add_edge(1497,1507,weight=1)
G10.add_edge(1498,1668,weight=1)
G10.add_edge(1499,2588,weight=1)
G10.add_edge(1499,2589,weight=1)
G10.add_edge(1499,2590,weight=1)
G10.add_edge(1499,2591,weight=1)
G10.add_edge(1501,1502,weight=1)
G10.add_edge(1501,1503,weight=1)
G10.add_edge(1501,1504,weight=1)
G10.add_edge(1502,1563,weight=1)
G10.add_edge(1504,2019,weight=1)
G10.add_edge(1506,1578,weight=1)
G10.add_edge(1508,1576,weight=1)
G10.add_edge(1509,1555,weight=1)
G10.add_edge(1510,1608,weight=1)
G10.add_edge(1510,1886,weight=1)
G10.add_edge(1510,1887,weight=1)
G10.add_edge(1510,1888,weight=1)
G10.add_edge(1510,1889,weight=1)
G10.add_edge(1511,1512,weight=1)
G10.add_edge(1511,1513,weight=1)
G10.add_edge(1512,1514,weight=1)
G10.add_edge(1513,1597,weight=1)
G10.add_edge(1514,1880,weight=1)
G10.add_edge(1514,1881,weight=1)
G10.add_edge(1515,1516,weight=1)
G10.add_edge(1515,1517,weight=1)
G10.add_edge(1516,1520,weight=1)
G10.add_edge(1516,1604,weight=1)
G10.add_edge(1517,1875,weight=1)
G10.add_edge(1517,1876,weight=1)
G10.add_edge(1518,1519,weight=1)
G10.add_edge(1518,1520,weight=1)
G10.add_edge(1518,1521,weight=1)
G10.add_edge(1519,1522,weight=1)
G10.add_edge(1520,1530,weight=1)
G10.add_edge(1520,1665,weight=1)
G10.add_edge(1520,1666,weight=1)
G10.add_edge(1521,1873,weight=1)
G10.add_edge(1522,2066,weight=1)
G10.add_edge(1523,1524,weight=1)
G10.add_edge(1523,1525,weight=1)
G10.add_edge(1524,1526,weight=1)
G10.add_edge(1525,1527,weight=1)
G10.add_edge(1526,1664,weight=1)
G10.add_edge(1526,1733,weight=1)
G10.add_edge(1527,1604,weight=1)
G10.add_edge(1527,1605,weight=1)
G10.add_edge(1528,1529,weight=1)
G10.add_edge(1528,1530,weight=1)
G10.add_edge(1528,1531,weight=1)
G10.add_edge(1529,1658,weight=1)
G10.add_edge(1530,1734,weight=1)
G10.add_edge(1532,1533,weight=1)
G10.add_edge(1532,1534,weight=1)
G10.add_edge(1532,1535,weight=1)
G10.add_edge(1532,1536,weight=1)
G10.add_edge(1532,1537,weight=1)
G10.add_edge(1533,1746,weight=1)
G10.add_edge(1533,1747,weight=1)
G10.add_edge(1533,1748,weight=1)
G10.add_edge(1537,2090,weight=1)
G10.add_edge(1538,1539,weight=1)
G10.add_edge(1539,2171,weight=1)
G10.add_edge(1540,1541,weight=1)
G10.add_edge(1541,1966,weight=1)
G10.add_edge(1545,1546,weight=1)
G10.add_edge(1546,1547,weight=1)
G10.add_edge(1547,1715,weight=1)
G10.add_edge(1547,1738,weight=1)
G10.add_edge(1549,1717,weight=1)
G10.add_edge(1549,1783,weight=1)
G10.add_edge(1550,1551,weight=1)
G10.add_edge(1550,1552,weight=1)
G10.add_edge(1550,1553,weight=1)
G10.add_edge(1550,1554,weight=1)
G10.add_edge(1551,1598,weight=1)
G10.add_edge(1552,1780,weight=1)
G10.add_edge(1553,1629,weight=1)
G10.add_edge(1554,2061,weight=1)
G10.add_edge(1554,2083,weight=1)
G10.add_edge(1554,2096,weight=1)
G10.add_edge(1554,2097,weight=1)
G10.add_edge(1555,1556,weight=1)
G10.add_edge(1555,1557,weight=1)
G10.add_edge(1555,1558,weight=1)
G10.add_edge(1555,1559,weight=1)
G10.add_edge(1556,1802,weight=1)
G10.add_edge(1557,1930,weight=1)
G10.add_edge(1557,1932,weight=1)
G10.add_edge(1557,2040,weight=1)
G10.add_edge(1557,2098,weight=1)
G10.add_edge(1558,2038,weight=1)
G10.add_edge(1560,1603,weight=1)
G10.add_edge(1562,1575,weight=1)
G10.add_edge(1563,1575,weight=1)
G10.add_edge(1563,1782,weight=1)
G10.add_edge(1566,1615,weight=1)
G10.add_edge(1566,1810,weight=1)
G10.add_edge(1567,1873,weight=1)
G10.add_edge(1568,2045,weight=1)
G10.add_edge(1569,2094,weight=1)
G10.add_edge(1571,1616,weight=1)
G10.add_edge(1573,2052,weight=1)
G10.add_edge(1573,2115,weight=1)
G10.add_edge(1573,2116,weight=1)
G10.add_edge(1576,1577,weight=1)
G10.add_edge(1577,1585,weight=1)
G10.add_edge(1577,1587,weight=1)
G10.add_edge(1577,1588,weight=1)
G10.add_edge(1579,1580,weight=1)
G10.add_edge(1579,1581,weight=1)
G10.add_edge(1579,1582,weight=1)
G10.add_edge(1579,1583,weight=1)
G10.add_edge(1580,1584,weight=1)
G10.add_edge(1581,1716,weight=1)
G10.add_edge(1582,1764,weight=1)
G10.add_edge(1583,1937,weight=1)
G10.add_edge(1585,1586,weight=1)
G10.add_edge(1586,1667,weight=1)
G10.add_edge(1586,1671,weight=1)
G10.add_edge(1587,1588,weight=1)
G10.add_edge(1589,1669,weight=1)
G10.add_edge(1590,1659,weight=1)
G10.add_edge(1590,1660,weight=1)
G10.add_edge(1591,2129,weight=1)
G10.add_edge(1591,2130,weight=1)
G10.add_edge(1592,2131,weight=1)
G10.add_edge(1592,2132,weight=1)
G10.add_edge(1593,1914,weight=1)
G10.add_edge(1594,1772,weight=1)
G10.add_edge(1598,1780,weight=1)
G10.add_edge(1600,1610,weight=1)
G10.add_edge(1601,1602,weight=1)
G10.add_edge(1602,1654,weight=1)
G10.add_edge(1602,1655,weight=1)
G10.add_edge(1602,1657,weight=1)
G10.add_edge(1603,1782,weight=1)
G10.add_edge(1605,1662,weight=1)
G10.add_edge(1605,1673,weight=1)
G10.add_edge(1606,1607,weight=1)
G10.add_edge(1606,1608,weight=1)
G10.add_edge(1606,1609,weight=1)
G10.add_edge(1607,1937,weight=1)
G10.add_edge(1609,1791,weight=1)
G10.add_edge(1610,2155,weight=1)
G10.add_edge(1612,1742,weight=1)
G10.add_edge(1614,1615,weight=1)
G10.add_edge(1618,1619,weight=1)
G10.add_edge(1618,1620,weight=1)
G10.add_edge(1618,1621,weight=1)
G10.add_edge(1618,1622,weight=1)
G10.add_edge(1619,1795,weight=1)
G10.add_edge(1620,1848,weight=1)
G10.add_edge(1620,1849,weight=1)
G10.add_edge(1621,1992,weight=1)
G10.add_edge(1621,2101,weight=1)
G10.add_edge(1621,2107,weight=1)
G10.add_edge(1621,2162,weight=1)
G10.add_edge(1621,2163,weight=1)
G10.add_edge(1622,2277,weight=1)
G10.add_edge(1623,1769,weight=1)
G10.add_edge(1624,1625,weight=1)
G10.add_edge(1625,1626,weight=1)
G10.add_edge(1627,1628,weight=1)
G10.add_edge(1627,1629,weight=1)
G10.add_edge(1627,1630,weight=1)
G10.add_edge(1627,1631,weight=1)
G10.add_edge(1627,1632,weight=1)
G10.add_edge(1627,1633,weight=1)
G10.add_edge(1630,1933,weight=1)
G10.add_edge(1631,1633,weight=1)
G10.add_edge(1631,1964,weight=1)
G10.add_edge(1632,2020,weight=1)
G10.add_edge(1633,2020,weight=1)
G10.add_edge(1633,2207,weight=1)
G10.add_edge(1634,1676,weight=1)
G10.add_edge(1634,1677,weight=1)
G10.add_edge(1635,1639,weight=1)
G10.add_edge(1635,1640,weight=1)
G10.add_edge(1636,1637,weight=1)
G10.add_edge(1637,1840,weight=1)
G10.add_edge(1637,1841,weight=1)
G10.add_edge(1637,1842,weight=1)
G10.add_edge(1638,2282,weight=1)
G10.add_edge(1642,1864,weight=1)
G10.add_edge(1642,1883,weight=1)
G10.add_edge(1642,1884,weight=1)
G10.add_edge(1643,1940,weight=1)
G10.add_edge(1643,2063,weight=1)
G10.add_edge(1643,2104,weight=1)
G10.add_edge(1643,2168,weight=1)
G10.add_edge(1644,1645,weight=1)
G10.add_edge(1645,1972,weight=1)
G10.add_edge(1646,1693,weight=1)
G10.add_edge(1646,1772,weight=1)
G10.add_edge(1647,1648,weight=1)
G10.add_edge(1647,1649,weight=1)
G10.add_edge(1648,1784,weight=1)
G10.add_edge(1648,1885,weight=1)
G10.add_edge(1650,1871,weight=1)
G10.add_edge(1651,1652,weight=1)
G10.add_edge(1651,1653,weight=1)
G10.add_edge(1652,1855,weight=1)
G10.add_edge(1653,1672,weight=1)
G10.add_edge(1653,1997,weight=1)
G10.add_edge(1655,1656,weight=1)
G10.add_edge(1656,2172,weight=1)
G10.add_edge(1657,2188,weight=1)
G10.add_edge(1658,1794,weight=1)
G10.add_edge(1659,2088,weight=1)
G10.add_edge(1660,1914,weight=1)
G10.add_edge(1661,1662,weight=1)
G10.add_edge(1663,1664,weight=1)
G10.add_edge(1663,1665,weight=1)
G10.add_edge(1663,1666,weight=1)
G10.add_edge(1665,1785,weight=1)
G10.add_edge(1667,1668,weight=1)
G10.add_edge(1667,1669,weight=1)
G10.add_edge(1667,1670,weight=1)
G10.add_edge(1667,1671,weight=1)
G10.add_edge(1669,1671,weight=1)
G10.add_edge(1672,1989,weight=1)
G10.add_edge(1672,2071,weight=1)
G10.add_edge(1675,1711,weight=1)
G10.add_edge(1675,1811,weight=1)
G10.add_edge(1676,2174,weight=1)
G10.add_edge(1677,2244,weight=1)
G10.add_edge(1678,2943,weight=1)
G10.add_edge(1678,2944,weight=1)
G10.add_edge(1679,1684,weight=1)
G10.add_edge(1680,1692,weight=1)
G10.add_edge(1681,1801,weight=1)
G10.add_edge(1683,2278,weight=1)
G10.add_edge(1687,1688,weight=1)
G10.add_edge(1687,1822,weight=1)
G10.add_edge(1688,2272,weight=1)
G10.add_edge(1688,2273,weight=1)
G10.add_edge(1690,1691,weight=1)
G10.add_edge(1691,2080,weight=1)
G10.add_edge(1691,2082,weight=1)
G10.add_edge(1692,1967,weight=1)
G10.add_edge(1692,2177,weight=1)
G10.add_edge(1694,1924,weight=1)
G10.add_edge(1695,1696,weight=1)
G10.add_edge(1697,1698,weight=1)
G10.add_edge(1697,1699,weight=1)
G10.add_edge(1698,1784,weight=1)
G10.add_edge(1698,1817,weight=1)
G10.add_edge(1699,1797,weight=1)
G10.add_edge(1700,1701,weight=1)
G10.add_edge(1700,1702,weight=1)
G10.add_edge(1700,1703,weight=1)
G10.add_edge(1701,1796,weight=1)
G10.add_edge(1702,1799,weight=1)
G10.add_edge(1702,1800,weight=1)
G10.add_edge(1703,1800,weight=1)
G10.add_edge(1703,2126,weight=1)
G10.add_edge(1705,1769,weight=1)
G10.add_edge(1706,1979,weight=1)
G10.add_edge(1706,2001,weight=1)
G10.add_edge(1706,2099,weight=1)
G10.add_edge(1707,1979,weight=1)
G10.add_edge(1707,2100,weight=1)
G10.add_edge(1707,2129,weight=1)
G10.add_edge(1708,1709,weight=1)
G10.add_edge(1708,1710,weight=1)
G10.add_edge(1710,1938,weight=1)
G10.add_edge(1713,1821,weight=1)
G10.add_edge(1714,1805,weight=1)
G10.add_edge(1715,1716,weight=1)
G10.add_edge(1715,1717,weight=1)
G10.add_edge(1718,1762,weight=1)
G10.add_edge(1719,1783,weight=1)
G10.add_edge(1720,1789,weight=1)
G10.add_edge(1721,1776,weight=1)
G10.add_edge(1721,1867,weight=1)
G10.add_edge(1722,1788,weight=1)
G10.add_edge(1722,1869,weight=1)
G10.add_edge(1724,1725,weight=1)
G10.add_edge(1725,1726,weight=1)
G10.add_edge(1725,1727,weight=1)
G10.add_edge(1726,2022,weight=1)
G10.add_edge(1727,2023,weight=1)
G10.add_edge(1728,1982,weight=1)
G10.add_edge(1728,2181,weight=1)
G10.add_edge(1729,1730,weight=1)
G10.add_edge(1729,1731,weight=1)
G10.add_edge(1730,1732,weight=1)
G10.add_edge(1733,1734,weight=1)
G10.add_edge(1733,1735,weight=1)
G10.add_edge(1733,1736,weight=1)
G10.add_edge(1735,1871,weight=1)
G10.add_edge(1738,1867,weight=1)
G10.add_edge(1739,1740,weight=1)
G10.add_edge(1740,2184,weight=1)
G10.add_edge(1740,2185,weight=1)
G10.add_edge(1740,2186,weight=1)
G10.add_edge(1743,2085,weight=1)
G10.add_edge(1744,1745,weight=1)
G10.add_edge(1745,1852,weight=1)
G10.add_edge(1746,2128,weight=1)
G10.add_edge(1747,1965,weight=1)
G10.add_edge(1748,2239,weight=1)
G10.add_edge(1749,1750,weight=1)
G10.add_edge(1749,1751,weight=1)
G10.add_edge(1749,1752,weight=1)
G10.add_edge(1749,1753,weight=1)
G10.add_edge(1750,1756,weight=1)
G10.add_edge(1751,1753,weight=1)
G10.add_edge(1752,1892,weight=1)
G10.add_edge(1752,2092,weight=1)
G10.add_edge(1753,1933,weight=1)
G10.add_edge(1753,2061,weight=1)
G10.add_edge(1758,1759,weight=1)
G10.add_edge(1759,1784,weight=1)
G10.add_edge(1759,1878,weight=1)
G10.add_edge(1759,1879,weight=1)
G10.add_edge(1760,1781,weight=1)
G10.add_edge(1761,1773,weight=1)
G10.add_edge(1761,1781,weight=1)
G10.add_edge(1762,1763,weight=1)
G10.add_edge(1762,1764,weight=1)
G10.add_edge(1763,2202,weight=1)
G10.add_edge(1770,1771,weight=1)
G10.add_edge(1772,1778,weight=1)
G10.add_edge(1773,1774,weight=1)
G10.add_edge(1773,1775,weight=1)
G10.add_edge(1776,1777,weight=1)
G10.add_edge(1777,2190,weight=1)
G10.add_edge(1777,2198,weight=1)
G10.add_edge(1778,1779,weight=1)
G10.add_edge(1779,1786,weight=1)
G10.add_edge(1780,2199,weight=1)
G10.add_edge(1783,1784,weight=1)
G10.add_edge(1784,1818,weight=1)
G10.add_edge(1786,1787,weight=1)
G10.add_edge(1791,1792,weight=1)
G10.add_edge(1791,1793,weight=1)
G10.add_edge(1793,2226,weight=1)
G10.add_edge(1794,1846,weight=1)
G10.add_edge(1795,1909,weight=1)
G10.add_edge(1795,2133,weight=1)
G10.add_edge(1799,1800,weight=1)
G10.add_edge(1802,1969,weight=1)
G10.add_edge(1802,2117,weight=1)
G10.add_edge(1803,1825,weight=1)
G10.add_edge(1803,1826,weight=1)
G10.add_edge(1803,1827,weight=1)
G10.add_edge(1804,2179,weight=1)
G10.add_edge(1806,2163,weight=1)
G10.add_edge(1808,2149,weight=1)
G10.add_edge(1810,1811,weight=1)
G10.add_edge(1810,1812,weight=1)
G10.add_edge(1813,1896,weight=1)
G10.add_edge(1813,1903,weight=1)
G10.add_edge(1813,1963,weight=1)
G10.add_edge(1813,1971,weight=1)
G10.add_edge(1813,2208,weight=1)
G10.add_edge(1814,1815,weight=1)
G10.add_edge(1814,1816,weight=1)
G10.add_edge(1815,2166,weight=1)
G10.add_edge(1816,2188,weight=1)
G10.add_edge(1819,1864,weight=1)
G10.add_edge(1820,1895,weight=1)
G10.add_edge(1823,1987,weight=1)
G10.add_edge(1823,2069,weight=1)
G10.add_edge(1824,1981,weight=1)
G10.add_edge(1824,2069,weight=1)
G10.add_edge(1824,2240,weight=1)
G10.add_edge(1827,1977,weight=1)
G10.add_edge(1827,1978,weight=1)
G10.add_edge(1828,2182,weight=1)
G10.add_edge(1829,2251,weight=1)
G10.add_edge(1830,2248,weight=1)
G10.add_edge(1831,2241,weight=1)
G10.add_edge(1832,2185,weight=1)
G10.add_edge(1833,2242,weight=1)
G10.add_edge(1834,2251,weight=1)
G10.add_edge(1835,2257,weight=1)
G10.add_edge(1836,2256,weight=1)
G10.add_edge(1839,2217,weight=1)
G10.add_edge(1839,2218,weight=1)
G10.add_edge(1839,2219,weight=1)
G10.add_edge(1839,2220,weight=1)
G10.add_edge(1839,2221,weight=1)
G10.add_edge(1839,2222,weight=1)
G10.add_edge(1843,2146,weight=1)
G10.add_edge(1844,2007,weight=1)
G10.add_edge(1844,2059,weight=1)
G10.add_edge(1848,2194,weight=1)
G10.add_edge(1848,2264,weight=1)
G10.add_edge(1849,2193,weight=1)
G10.add_edge(1849,2214,weight=1)
G10.add_edge(1850,2197,weight=1)
G10.add_edge(1854,1855,weight=1)
G10.add_edge(1856,2007,weight=1)
G10.add_edge(1856,2046,weight=1)
G10.add_edge(1856,2196,weight=1)
G10.add_edge(1856,2262,weight=1)
G10.add_edge(1858,1893,weight=1)
G10.add_edge(1858,2153,weight=1)
G10.add_edge(1858,2154,weight=1)
G10.add_edge(1860,1861,weight=1)
G10.add_edge(1860,1862,weight=1)
G10.add_edge(1860,1863,weight=1)
G10.add_edge(1861,1934,weight=1)
G10.add_edge(1863,2108,weight=1)
G10.add_edge(1864,1865,weight=1)
G10.add_edge(1866,2229,weight=1)
G10.add_edge(1866,2235,weight=1)
G10.add_edge(1867,2189,weight=1)
G10.add_edge(1867,2190,weight=1)
G10.add_edge(1867,2213,weight=1)
G10.add_edge(1867,2234,weight=1)
G10.add_edge(1870,2281,weight=1)
G10.add_edge(1873,1874,weight=1)
G10.add_edge(1874,1876,weight=1)
G10.add_edge(1874,2113,weight=1)
G10.add_edge(1874,2224,weight=1)
G10.add_edge(1875,1877,weight=1)
G10.add_edge(1876,2067,weight=1)
G10.add_edge(1876,2178,weight=1)
G10.add_edge(1876,2195,weight=1)
G10.add_edge(1879,2205,weight=1)
G10.add_edge(1881,1882,weight=1)
G10.add_edge(1882,1965,weight=1)
G10.add_edge(1883,2204,weight=1)
G10.add_edge(1884,2106,weight=1)
G10.add_edge(1884,2175,weight=1)
G10.add_edge(1884,2231,weight=1)
G10.add_edge(1885,2205,weight=1)
G10.add_edge(1888,2105,weight=1)
G10.add_edge(1889,2226,weight=1)
G10.add_edge(1891,2043,weight=1)
G10.add_edge(1891,2044,weight=1)
G10.add_edge(1894,1895,weight=1)
G10.add_edge(1894,1896,weight=1)
G10.add_edge(1895,1902,weight=1)
G10.add_edge(1895,1903,weight=1)
G10.add_edge(1895,1904,weight=1)
G10.add_edge(1895,1905,weight=1)
G10.add_edge(1895,1906,weight=1)
G10.add_edge(1897,1898,weight=1)
G10.add_edge(1897,1899,weight=1)
G10.add_edge(1898,1940,weight=1)
G10.add_edge(1898,1945,weight=1)
G10.add_edge(1899,2058,weight=1)
G10.add_edge(1900,1901,weight=1)
G10.add_edge(1904,2051,weight=1)
G10.add_edge(1905,2135,weight=1)
G10.add_edge(1906,2115,weight=1)
G10.add_edge(1907,1908,weight=1)
G10.add_edge(1907,1909,weight=1)
G10.add_edge(1908,2054,weight=1)
G10.add_edge(1908,2136,weight=1)
G10.add_edge(1908,2165,weight=1)
G10.add_edge(1910,1911,weight=1)
G10.add_edge(1912,1913,weight=1)
G10.add_edge(1912,1914,weight=1)
G10.add_edge(1913,2026,weight=1)
G10.add_edge(1914,2055,weight=1)
G10.add_edge(1915,1976,weight=1)
G10.add_edge(1915,2056,weight=1)
G10.add_edge(1915,2057,weight=1)
G10.add_edge(1915,2058,weight=1)
G10.add_edge(1916,1940,weight=1)
G10.add_edge(1916,2077,weight=1)
G10.add_edge(1918,1939,weight=1)
G10.add_edge(1919,2076,weight=1)
G10.add_edge(1919,2157,weight=1)
G10.add_edge(1920,2075,weight=1)
G10.add_edge(1922,2259,weight=1)
G10.add_edge(1923,1924,weight=1)
G10.add_edge(1923,1925,weight=1)
G10.add_edge(1925,2030,weight=1)
G10.add_edge(1926,1927,weight=1)
G10.add_edge(1926,1928,weight=1)
G10.add_edge(1927,2119,weight=1)
G10.add_edge(1928,2059,weight=1)
G10.add_edge(1929,2003,weight=1)
G10.add_edge(1930,1931,weight=1)
G10.add_edge(1930,1932,weight=1)
G10.add_edge(1931,2118,weight=1)
G10.add_edge(1934,1962,weight=1)
G10.add_edge(1939,1940,weight=1)
G10.add_edge(1940,2074,weight=1)
G10.add_edge(1940,2095,weight=1)
G10.add_edge(1941,1943,weight=1)
G10.add_edge(1941,1944,weight=1)
G10.add_edge(1942,2208,weight=1)
G10.add_edge(1943,1982,weight=1)
G10.add_edge(1943,2015,weight=1)
G10.add_edge(1944,2015,weight=1)
G10.add_edge(1945,2216,weight=1)
G10.add_edge(1946,1947,weight=1)
G10.add_edge(1946,1948,weight=1)
G10.add_edge(1947,1995,weight=1)
G10.add_edge(1948,2232,weight=1)
G10.add_edge(1948,2233,weight=1)
G10.add_edge(1950,1951,weight=1)
G10.add_edge(1952,1953,weight=1)
G10.add_edge(1952,1954,weight=1)
G10.add_edge(1952,1955,weight=1)
G10.add_edge(1953,1954,weight=1)
G10.add_edge(1953,2099,weight=1)
G10.add_edge(1955,2151,weight=1)
G10.add_edge(1956,1961,weight=1)
G10.add_edge(1956,1966,weight=1)
G10.add_edge(1957,1959,weight=1)
G10.add_edge(1957,1960,weight=1)
G10.add_edge(1958,1959,weight=1)
G10.add_edge(1959,1961,weight=1)
G10.add_edge(1960,1966,weight=1)
G10.add_edge(1962,1963,weight=1)
G10.add_edge(1963,2070,weight=1)
G10.add_edge(1964,2020,weight=1)
G10.add_edge(1964,2097,weight=1)
G10.add_edge(1966,1967,weight=1)
G10.add_edge(1967,2177,weight=1)
G10.add_edge(1968,1969,weight=1)
G10.add_edge(1968,1970,weight=1)
G10.add_edge(1969,2125,weight=1)
G10.add_edge(1970,2125,weight=1)
G10.add_edge(1972,1973,weight=1)
G10.add_edge(1974,1975,weight=1)
G10.add_edge(1974,1976,weight=1)
G10.add_edge(1975,2160,weight=1)
G10.add_edge(1976,2201,weight=1)
G10.add_edge(1976,2232,weight=1)
G10.add_edge(1979,1980,weight=1)
G10.add_edge(1979,1981,weight=1)
G10.add_edge(1980,1982,weight=1)
G10.add_edge(1980,1983,weight=1)
G10.add_edge(1981,1987,weight=1)
G10.add_edge(1983,2228,weight=1)
G10.add_edge(1984,1985,weight=1)
G10.add_edge(1984,1986,weight=1)
G10.add_edge(1984,1987,weight=1)
G10.add_edge(1985,2137,weight=1)
G10.add_edge(1985,2215,weight=1)
G10.add_edge(1986,2133,weight=1)
G10.add_edge(1986,2264,weight=1)
G10.add_edge(1987,2215,weight=1)
G10.add_edge(1987,2240,weight=1)
G10.add_edge(1988,1989,weight=1)
G10.add_edge(1988,1990,weight=1)
G10.add_edge(1988,1991,weight=1)
G10.add_edge(1990,1996,weight=1)
G10.add_edge(1991,2027,weight=1)
G10.add_edge(1992,2041,weight=1)
G10.add_edge(1993,1994,weight=1)
G10.add_edge(1993,2068,weight=1)
G10.add_edge(1996,1997,weight=1)
G10.add_edge(1998,1999,weight=1)
G10.add_edge(1998,2000,weight=1)
G10.add_edge(2002,2129,weight=1)
G10.add_edge(2003,2004,weight=1)
G10.add_edge(2003,2005,weight=1)
G10.add_edge(2004,2006,weight=1)
G10.add_edge(2005,2021,weight=1)
G10.add_edge(2006,2152,weight=1)
G10.add_edge(2006,2173,weight=1)
G10.add_edge(2007,2008,weight=1)
G10.add_edge(2008,2046,weight=1)
G10.add_edge(2008,2059,weight=1)
G10.add_edge(2009,2065,weight=1)
G10.add_edge(2010,2011,weight=1)
G10.add_edge(2011,2092,weight=1)
G10.add_edge(2012,2013,weight=1)
G10.add_edge(2014,2152,weight=1)
G10.add_edge(2015,2017,weight=1)
G10.add_edge(2015,2018,weight=1)
G10.add_edge(2016,2101,weight=1)
G10.add_edge(2017,2042,weight=1)
G10.add_edge(2017,2068,weight=1)
G10.add_edge(2017,2070,weight=1)
G10.add_edge(2018,2236,weight=1)
G10.add_edge(2021,2071,weight=1)
G10.add_edge(2024,2025,weight=1)
G10.add_edge(2024,2026,weight=1)
G10.add_edge(2024,2027,weight=1)
G10.add_edge(2025,2165,weight=1)
G10.add_edge(2027,2163,weight=1)
G10.add_edge(2028,2029,weight=1)
G10.add_edge(2028,2030,weight=1)
G10.add_edge(2030,2109,weight=1)
G10.add_edge(2031,2032,weight=1)
G10.add_edge(2031,2033,weight=1)
G10.add_edge(2032,2034,weight=1)
G10.add_edge(2032,2139,weight=1)
G10.add_edge(2035,2036,weight=1)
G10.add_edge(2037,2038,weight=1)
G10.add_edge(2037,2039,weight=1)
G10.add_edge(2039,2040,weight=1)
G10.add_edge(2041,2042,weight=1)
G10.add_edge(2042,2192,weight=1)
G10.add_edge(2044,2152,weight=1)
G10.add_edge(2048,2049,weight=1)
G10.add_edge(2050,2051,weight=1)
G10.add_edge(2050,2052,weight=1)
G10.add_edge(2051,2111,weight=1)
G10.add_edge(2051,2112,weight=1)
G10.add_edge(2051,2171,weight=1)
G10.add_edge(2053,2054,weight=1)
G10.add_edge(2053,2055,weight=1)
G10.add_edge(2056,2142,weight=1)
G10.add_edge(2056,2145,weight=1)
G10.add_edge(2057,2078,weight=1)
G10.add_edge(2058,2073,weight=1)
G10.add_edge(2060,2061,weight=1)
G10.add_edge(2061,2062,weight=1)
G10.add_edge(2061,2083,weight=1)
G10.add_edge(2063,2064,weight=1)
G10.add_edge(2064,2160,weight=1)
G10.add_edge(2066,2067,weight=1)
G10.add_edge(2067,2178,weight=1)
G10.add_edge(2068,2069,weight=1)
G10.add_edge(2071,2124,weight=1)
G10.add_edge(2072,2073,weight=1)
G10.add_edge(2072,2074,weight=1)
G10.add_edge(2075,2076,weight=1)
G10.add_edge(2077,2078,weight=1)
G10.add_edge(2078,2150,weight=1)
G10.add_edge(2079,2080,weight=1)
G10.add_edge(2079,2081,weight=1)
G10.add_edge(2084,2085,weight=1)
G10.add_edge(2086,2087,weight=1)
G10.add_edge(2087,2203,weight=1)
G10.add_edge(2088,2089,weight=1)
G10.add_edge(2089,2136,weight=1)
G10.add_edge(2089,2137,weight=1)
G10.add_edge(2089,2138,weight=1)
G10.add_edge(2090,2091,weight=1)
G10.add_edge(2093,2094,weight=1)
G10.add_edge(2095,2161,weight=1)
G10.add_edge(2098,2121,weight=1)
G10.add_edge(2099,2100,weight=1)
G10.add_edge(2101,2191,weight=1)
G10.add_edge(2102,2103,weight=1)
G10.add_edge(2102,2104,weight=1)
G10.add_edge(2103,2176,weight=1)
G10.add_edge(2107,2108,weight=1)
G10.add_edge(2109,2164,weight=1)
G10.add_edge(2110,2111,weight=1)
G10.add_edge(2110,2112,weight=1)
G10.add_edge(2113,2114,weight=1)
G10.add_edge(2114,2180,weight=1)
G10.add_edge(2117,2118,weight=1)
G10.add_edge(2119,2146,weight=1)
G10.add_edge(2120,2187,weight=1)
G10.add_edge(2121,2140,weight=1)
G10.add_edge(2122,2123,weight=1)
G10.add_edge(2122,2124,weight=1)
G10.add_edge(2123,2134,weight=1)
G10.add_edge(2123,2223,weight=1)
G10.add_edge(2129,2214,weight=1)
G10.add_edge(2129,2215,weight=1)
G10.add_edge(2130,2131,weight=1)
G10.add_edge(2130,2206,weight=1)
G10.add_edge(2131,2151,weight=1)
G10.add_edge(2132,2138,weight=1)
G10.add_edge(2132,2206,weight=1)
G10.add_edge(2133,2134,weight=1)
G10.add_edge(2135,2161,weight=1)
G10.add_edge(2140,2167,weight=1)
G10.add_edge(2141,2142,weight=1)
G10.add_edge(2141,2143,weight=1)
G10.add_edge(2141,2144,weight=1)
G10.add_edge(2143,2144,weight=1)
G10.add_edge(2143,2169,weight=1)
G10.add_edge(2143,2170,weight=1)
G10.add_edge(2144,2238,weight=1)
G10.add_edge(2145,2267,weight=1)
G10.add_edge(2145,2270,weight=1)
G10.add_edge(2146,2147,weight=1)
G10.add_edge(2146,2148,weight=1)
G10.add_edge(2148,2275,weight=1)
G10.add_edge(2149,2195,weight=1)
G10.add_edge(2150,2158,weight=1)
G10.add_edge(2153,2154,weight=1)
G10.add_edge(2156,2157,weight=1)
G10.add_edge(2158,2159,weight=1)
G10.add_edge(2159,2168,weight=1)
G10.add_edge(2159,2216,weight=1)
G10.add_edge(2160,2201,weight=1)
G10.add_edge(2163,2223,weight=1)
G10.add_edge(2167,2204,weight=1)
G10.add_edge(2172,2183,weight=1)
G10.add_edge(2175,2176,weight=1)
G10.add_edge(2178,2195,weight=1)
G10.add_edge(2180,2224,weight=1)
G10.add_edge(2181,2260,weight=1)
G10.add_edge(2184,2248,weight=1)
G10.add_edge(2186,2248,weight=1)
G10.add_edge(2188,2209,weight=1)
G10.add_edge(2188,2210,weight=1)
G10.add_edge(2189,2190,weight=1)
G10.add_edge(2191,2246,weight=1)
G10.add_edge(2191,2263,weight=1)
G10.add_edge(2192,2193,weight=1)
G10.add_edge(2192,2194,weight=1)
G10.add_edge(2196,2197,weight=1)
G10.add_edge(2197,2262,weight=1)
G10.add_edge(2197,2265,weight=1)
G10.add_edge(2198,2213,weight=1)
G10.add_edge(2199,2200,weight=1)
G10.add_edge(2200,2211,weight=1)
G10.add_edge(2202,2234,weight=1)
G10.add_edge(2204,2231,weight=1)
G10.add_edge(2205,2212,weight=1)
G10.add_edge(2207,2275,weight=1)
G10.add_edge(2214,2240,weight=1)
G10.add_edge(2217,2243,weight=1)
G10.add_edge(2218,2219,weight=1)
G10.add_edge(2218,2252,weight=1)
G10.add_edge(2219,2247,weight=1)
G10.add_edge(2220,2254,weight=1)
G10.add_edge(2221,2247,weight=1)
G10.add_edge(2222,2247,weight=1)
G10.add_edge(2224,2225,weight=1)
G10.add_edge(2228,2261,weight=1)
G10.add_edge(2228,2263,weight=1)
G10.add_edge(2229,2230,weight=1)
G10.add_edge(2231,2274,weight=1)
G10.add_edge(2233,2270,weight=1)
G10.add_edge(2236,2260,weight=1)
G10.add_edge(2241,2242,weight=1)
G10.add_edge(2242,2255,weight=1)
G10.add_edge(2243,2253,weight=1)
G10.add_edge(2243,2254,weight=1)
G10.add_edge(2244,2245,weight=1)
G10.add_edge(2248,2251,weight=1)
G10.add_edge(2252,2253,weight=1)
G10.add_edge(2252,2254,weight=1)
G10.add_edge(2253,2255,weight=1)
G10.add_edge(2256,2257,weight=1)
G10.add_edge(2258,2271,weight=1)
G10.add_edge(2260,2261,weight=1)
G10.add_edge(2261,2266,weight=1)
G10.add_edge(2279,2280,weight=1)
G10.add_edge(2279,2281,weight=1)
G10.add_edge(2280,2282,weight=1)
G10.add_edge(2283,2284,weight=1)
G10.add_edge(2283,2285,weight=1)
G10.add_edge(2284,2294,weight=1)
G10.add_edge(2285,2294,weight=1)
G10.add_edge(2286,2287,weight=1)
G10.add_edge(2287,2329,weight=1)
G10.add_edge(2287,2413,weight=1)
G10.add_edge(2287,2417,weight=1)
G10.add_edge(2287,2442,weight=1)
G10.add_edge(2287,2443,weight=1)
G10.add_edge(2287,2448,weight=1)
G10.add_edge(2287,2476,weight=1)
G10.add_edge(2287,2482,weight=1)
G10.add_edge(2287,2489,weight=1)
G10.add_edge(2287,2490,weight=1)
G10.add_edge(2287,2491,weight=1)
G10.add_edge(2287,2492,weight=1)
G10.add_edge(2288,2289,weight=1)
G10.add_edge(2288,2290,weight=1)
G10.add_edge(2289,2310,weight=1)
G10.add_edge(2289,2311,weight=1)
G10.add_edge(2290,2491,weight=1)
G10.add_edge(2291,2292,weight=1)
G10.add_edge(2291,2293,weight=1)
G10.add_edge(2292,2367,weight=1)
G10.add_edge(2293,2501,weight=1)
G10.add_edge(2295,2296,weight=1)
G10.add_edge(2295,2297,weight=1)
G10.add_edge(2296,2353,weight=1)
G10.add_edge(2296,2496,weight=1)
G10.add_edge(2297,2409,weight=1)
G10.add_edge(2297,4861,weight=1)
G10.add_edge(2297,4876,weight=1)
G10.add_edge(2298,2299,weight=1)
G10.add_edge(2298,2300,weight=1)
G10.add_edge(2299,2340,weight=1)
G10.add_edge(2299,2474,weight=1)
G10.add_edge(2300,2369,weight=1)
G10.add_edge(2301,2302,weight=1)
G10.add_edge(2301,2303,weight=1)
G10.add_edge(2302,2341,weight=1)
G10.add_edge(2302,2367,weight=1)
G10.add_edge(2303,2369,weight=1)
G10.add_edge(2303,2440,weight=1)
G10.add_edge(2303,2493,weight=1)
G10.add_edge(2304,2305,weight=1)
G10.add_edge(2304,2306,weight=1)
G10.add_edge(2305,2441,weight=1)
G10.add_edge(2306,2308,weight=1)
G10.add_edge(2307,2308,weight=1)
G10.add_edge(2307,2309,weight=1)
G10.add_edge(2308,2391,weight=1)
G10.add_edge(2310,2311,weight=1)
G10.add_edge(2310,2312,weight=1)
G10.add_edge(2310,2313,weight=1)
G10.add_edge(2311,2312,weight=1)
G10.add_edge(2311,2428,weight=1)
G10.add_edge(2311,2448,weight=1)
G10.add_edge(2311,2460,weight=1)
G10.add_edge(2312,2313,weight=1)
G10.add_edge(2313,2492,weight=1)
G10.add_edge(2313,2497,weight=1)
G10.add_edge(2313,2498,weight=1)
G10.add_edge(2314,2315,weight=1)
G10.add_edge(2315,2317,weight=1)
G10.add_edge(2316,2317,weight=1)
G10.add_edge(2316,2318,weight=1)
G10.add_edge(2316,2319,weight=1)
G10.add_edge(2316,2320,weight=1)
G10.add_edge(2316,2321,weight=1)
G10.add_edge(2316,2322,weight=1)
G10.add_edge(2316,2323,weight=1)
G10.add_edge(2316,2324,weight=1)
G10.add_edge(2317,2325,weight=1)
G10.add_edge(2317,2326,weight=1)
G10.add_edge(2317,2327,weight=1)
G10.add_edge(2318,2324,weight=1)
G10.add_edge(2320,2324,weight=1)
G10.add_edge(2321,2323,weight=1)
G10.add_edge(2321,2484,weight=1)
G10.add_edge(2322,2484,weight=1)
G10.add_edge(2324,2439,weight=1)
G10.add_edge(2324,2457,weight=1)
G10.add_edge(2325,2326,weight=1)
G10.add_edge(2325,2330,weight=1)
G10.add_edge(2326,2332,weight=1)
G10.add_edge(2326,2430,weight=1)
G10.add_edge(2326,2461,weight=1)
G10.add_edge(2326,2462,weight=1)
G10.add_edge(2326,2463,weight=1)
G10.add_edge(2326,2464,weight=1)
G10.add_edge(2327,2489,weight=1)
G10.add_edge(2327,2490,weight=1)
G10.add_edge(2328,2329,weight=1)
G10.add_edge(2328,2347,weight=1)
G10.add_edge(2328,2348,weight=1)
G10.add_edge(2328,2349,weight=1)
G10.add_edge(2328,2350,weight=1)
G10.add_edge(2328,2351,weight=1)
G10.add_edge(2329,2349,weight=1)
G10.add_edge(2329,2350,weight=1)
G10.add_edge(2330,2471,weight=1)
G10.add_edge(2331,2332,weight=1)
G10.add_edge(2331,2333,weight=1)
G10.add_edge(2332,2338,weight=1)
G10.add_edge(2333,2474,weight=1)
G10.add_edge(2334,2335,weight=1)
G10.add_edge(2335,2339,weight=1)
G10.add_edge(2336,2337,weight=1)
G10.add_edge(2336,2338,weight=1)
G10.add_edge(2337,2340,weight=1)
G10.add_edge(2339,2346,weight=1)
G10.add_edge(2340,2405,weight=1)
G10.add_edge(2341,2441,weight=1)
G10.add_edge(2342,2343,weight=1)
G10.add_edge(2342,2344,weight=1)
G10.add_edge(2342,2345,weight=1)
G10.add_edge(2343,2368,weight=1)
G10.add_edge(2344,2432,weight=1)
G10.add_edge(2345,2499,weight=1)
G10.add_edge(2345,2500,weight=1)
G10.add_edge(2345,2501,weight=1)
G10.add_edge(2345,2502,weight=1)
G10.add_edge(2345,2503,weight=1)
G10.add_edge(2345,2504,weight=1)
G10.add_edge(2346,2485,weight=1)
G10.add_edge(2347,2348,weight=1)
G10.add_edge(2347,2401,weight=1)
G10.add_edge(2347,2402,weight=1)
G10.add_edge(2347,2403,weight=1)
G10.add_edge(2347,2404,weight=1)
G10.add_edge(2348,2351,weight=1)
G10.add_edge(2348,2449,weight=1)
G10.add_edge(2348,2450,weight=1)
G10.add_edge(2348,2451,weight=1)
G10.add_edge(2349,2377,weight=1)
G10.add_edge(2350,2397,weight=1)
G10.add_edge(2350,2468,weight=1)
G10.add_edge(2351,2356,weight=1)
G10.add_edge(2352,2353,weight=1)
G10.add_edge(2353,2354,weight=1)
G10.add_edge(2353,2355,weight=1)
G10.add_edge(2353,2356,weight=1)
G10.add_edge(2355,2373,weight=1)
G10.add_edge(2356,2427,weight=1)
G10.add_edge(2356,2428,weight=1)
G10.add_edge(2356,2444,weight=1)
G10.add_edge(2357,2358,weight=1)
G10.add_edge(2357,2359,weight=1)
G10.add_edge(2358,2407,weight=1)
G10.add_edge(2358,2438,weight=1)
G10.add_edge(2358,2439,weight=1)
G10.add_edge(2360,2361,weight=1)
G10.add_edge(2360,2362,weight=1)
G10.add_edge(2361,2370,weight=1)
G10.add_edge(2361,2465,weight=1)
G10.add_edge(2361,2466,weight=1)
G10.add_edge(2362,2461,weight=1)
G10.add_edge(2363,2364,weight=1)
G10.add_edge(2363,2365,weight=1)
G10.add_edge(2363,2366,weight=1)
G10.add_edge(2364,2367,weight=1)
G10.add_edge(2364,2368,weight=1)
G10.add_edge(2364,2369,weight=1)
G10.add_edge(2365,2384,weight=1)
G10.add_edge(2365,4529,weight=1)
G10.add_edge(2366,2385,weight=1)
G10.add_edge(2367,2382,weight=1)
G10.add_edge(2367,2386,weight=1)
G10.add_edge(2367,2387,weight=1)
G10.add_edge(2367,2388,weight=1)
G10.add_edge(2367,2389,weight=1)
G10.add_edge(2367,2390,weight=1)
G10.add_edge(2369,2483,weight=1)
G10.add_edge(2370,2371,weight=1)
G10.add_edge(2371,2431,weight=1)
G10.add_edge(2371,2471,weight=1)
G10.add_edge(2372,2373,weight=1)
G10.add_edge(2372,2374,weight=1)
G10.add_edge(2374,2506,weight=1)
G10.add_edge(2375,2376,weight=1)
G10.add_edge(2375,2377,weight=1)
G10.add_edge(2375,2378,weight=1)
G10.add_edge(2376,2377,weight=1)
G10.add_edge(2376,2458,weight=1)
G10.add_edge(2377,2469,weight=1)
G10.add_edge(2377,2470,weight=1)
G10.add_edge(2377,2475,weight=1)
G10.add_edge(2377,2482,weight=1)
G10.add_edge(2378,2397,weight=1)
G10.add_edge(2379,2380,weight=1)
G10.add_edge(2379,2381,weight=1)
G10.add_edge(2380,2398,weight=1)
G10.add_edge(2381,2456,weight=1)
G10.add_edge(2381,2488,weight=1)
G10.add_edge(2382,2383,weight=1)
G10.add_edge(2382,2384,weight=1)
G10.add_edge(2382,2385,weight=1)
G10.add_edge(2384,4669,weight=1)
G10.add_edge(2388,2391,weight=1)
G10.add_edge(2389,2390,weight=1)
G10.add_edge(2389,4553,weight=1)
G10.add_edge(2389,4728,weight=1)
G10.add_edge(2390,2500,weight=1)
G10.add_edge(2390,4544,weight=1)
G10.add_edge(2390,4667,weight=1)
G10.add_edge(2390,4752,weight=1)
G10.add_edge(2392,2393,weight=1)
G10.add_edge(2392,2394,weight=1)
G10.add_edge(2393,2435,weight=1)
G10.add_edge(2394,2488,weight=1)
G10.add_edge(2395,2396,weight=1)
G10.add_edge(2395,2397,weight=1)
G10.add_edge(2396,2427,weight=1)
G10.add_edge(2397,2460,weight=1)
G10.add_edge(2397,2468,weight=1)
G10.add_edge(2397,2469,weight=1)
G10.add_edge(2397,2470,weight=1)
G10.add_edge(2398,2446,weight=1)
G10.add_edge(2398,2456,weight=1)
G10.add_edge(2399,2400,weight=1)
G10.add_edge(2400,2423,weight=1)
G10.add_edge(2400,2452,weight=1)
G10.add_edge(2400,2453,weight=1)
G10.add_edge(2404,2478,weight=1)
G10.add_edge(2405,2408,weight=1)
G10.add_edge(2406,2407,weight=1)
G10.add_edge(2408,2409,weight=1)
G10.add_edge(2409,4899,weight=1)
G10.add_edge(2410,2411,weight=1)
G10.add_edge(2411,2420,weight=1)
G10.add_edge(2412,2413,weight=1)
G10.add_edge(2412,2414,weight=1)
G10.add_edge(2412,2415,weight=1)
G10.add_edge(2412,2416,weight=1)
G10.add_edge(2412,2417,weight=1)
G10.add_edge(2413,2418,weight=1)
G10.add_edge(2413,2419,weight=1)
G10.add_edge(2416,2417,weight=1)
G10.add_edge(2419,2458,weight=1)
G10.add_edge(2420,2440,weight=1)
G10.add_edge(2421,2422,weight=1)
G10.add_edge(2422,2423,weight=1)
G10.add_edge(2423,2424,weight=1)
G10.add_edge(2424,2454,weight=1)
G10.add_edge(2424,2455,weight=1)
G10.add_edge(2425,2426,weight=1)
G10.add_edge(2425,2427,weight=1)
G10.add_edge(2425,2428,weight=1)
G10.add_edge(2427,2444,weight=1)
G10.add_edge(2429,2430,weight=1)
G10.add_edge(2429,2431,weight=1)
G10.add_edge(2431,2465,weight=1)
G10.add_edge(2433,2434,weight=1)
G10.add_edge(2435,2447,weight=1)
G10.add_edge(2435,2472,weight=1)
G10.add_edge(2435,2473,weight=1)
G10.add_edge(2436,2455,weight=1)
G10.add_edge(2437,2473,weight=1)
G10.add_edge(2438,2439,weight=1)
G10.add_edge(2439,2495,weight=1)
G10.add_edge(2442,2443,weight=1)
G10.add_edge(2445,2446,weight=1)
G10.add_edge(2445,2447,weight=1)
G10.add_edge(2452,2453,weight=1)
G10.add_edge(2459,4554,weight=1)
G10.add_edge(2459,4876,weight=1)
G10.add_edge(2466,2471,weight=1)
G10.add_edge(2467,2468,weight=1)
G10.add_edge(2473,2488,weight=1)
G10.add_edge(2475,2476,weight=1)
G10.add_edge(2477,2478,weight=1)
G10.add_edge(2477,2479,weight=1)
G10.add_edge(2478,2480,weight=1)
G10.add_edge(2478,2485,weight=1)
G10.add_edge(2478,2486,weight=1)
G10.add_edge(2480,2481,weight=1)
G10.add_edge(2483,4757,weight=1)
G10.add_edge(2486,2487,weight=1)
G10.add_edge(2487,4898,weight=1)
G10.add_edge(2487,4900,weight=1)
G10.add_edge(2487,4902,weight=1)
G10.add_edge(2493,4564,weight=1)
G10.add_edge(2493,4587,weight=1)
G10.add_edge(2493,4613,weight=1)
G10.add_edge(2493,4775,weight=1)
G10.add_edge(2494,2495,weight=1)
G10.add_edge(2496,4876,weight=1)
G10.add_edge(2500,4699,weight=1)
G10.add_edge(2500,4700,weight=1)
G10.add_edge(2502,2504,weight=1)
G10.add_edge(2502,4699,weight=1)
G10.add_edge(2502,4773,weight=1)
G10.add_edge(2502,4774,weight=1)
G10.add_edge(2503,4825,weight=1)
G10.add_edge(2503,4826,weight=1)
G10.add_edge(2503,4828,weight=1)
G10.add_edge(2504,4825,weight=1)
G10.add_edge(2505,4738,weight=1)
G10.add_edge(2505,4787,weight=1)
G10.add_edge(2505,4788,weight=1)
G10.add_edge(2507,2508,weight=1)
G10.add_edge(2507,2509,weight=1)
G10.add_edge(2508,4582,weight=1)
G10.add_edge(2508,4583,weight=1)
G10.add_edge(2508,4584,weight=1)
G10.add_edge(2508,4585,weight=1)
G10.add_edge(2509,4714,weight=1)
G10.add_edge(2510,2511,weight=1)
G10.add_edge(2511,2512,weight=1)
G10.add_edge(2511,2513,weight=1)
G10.add_edge(2512,2616,weight=1)
G10.add_edge(2512,2622,weight=1)
G10.add_edge(2512,2625,weight=1)
G10.add_edge(2512,2626,weight=1)
G10.add_edge(2512,2627,weight=1)
G10.add_edge(2513,2625,weight=1)
G10.add_edge(2513,2627,weight=1)
G10.add_edge(2513,2659,weight=1)
G10.add_edge(2513,2745,weight=1)
G10.add_edge(2513,2757,weight=1)
G10.add_edge(2513,2760,weight=1)
G10.add_edge(2514,2604,weight=1)
G10.add_edge(2514,2680,weight=1)
G10.add_edge(2514,2681,weight=1)
G10.add_edge(2514,2682,weight=1)
G10.add_edge(2514,2683,weight=1)
G10.add_edge(2514,2684,weight=1)
G10.add_edge(2515,2712,weight=1)
G10.add_edge(2515,3019,weight=1)
G10.add_edge(2515,3020,weight=1)
G10.add_edge(2515,3021,weight=1)
G10.add_edge(2515,3022,weight=1)
G10.add_edge(2516,2517,weight=1)
G10.add_edge(2516,2518,weight=1)
G10.add_edge(2517,2695,weight=1)
G10.add_edge(2517,3126,weight=1)
G10.add_edge(2518,2549,weight=1)
G10.add_edge(2518,3127,weight=1)
G10.add_edge(2519,2520,weight=1)
G10.add_edge(2519,2521,weight=1)
G10.add_edge(2519,2522,weight=1)
G10.add_edge(2520,2521,weight=1)
G10.add_edge(2520,2572,weight=1)
G10.add_edge(2520,2687,weight=1)
G10.add_edge(2520,2691,weight=1)
G10.add_edge(2520,2692,weight=1)
G10.add_edge(2520,2693,weight=1)
G10.add_edge(2521,2533,weight=1)
G10.add_edge(2521,2536,weight=1)
G10.add_edge(2521,2538,weight=1)
G10.add_edge(2521,2582,weight=1)
G10.add_edge(2521,2607,weight=1)
G10.add_edge(2521,2629,weight=1)
G10.add_edge(2521,2926,weight=1)
G10.add_edge(2521,2933,weight=1)
G10.add_edge(2522,3023,weight=1)
G10.add_edge(2522,3024,weight=1)
G10.add_edge(2523,2524,weight=1)
G10.add_edge(2523,2525,weight=1)
G10.add_edge(2524,2592,weight=1)
G10.add_edge(2524,2783,weight=1)
G10.add_edge(2524,2829,weight=1)
G10.add_edge(2524,2830,weight=1)
G10.add_edge(2524,2831,weight=1)
G10.add_edge(2524,2832,weight=1)
G10.add_edge(2524,2833,weight=1)
G10.add_edge(2524,2834,weight=1)
G10.add_edge(2525,2564,weight=1)
G10.add_edge(2525,2615,weight=1)
G10.add_edge(2525,2616,weight=1)
G10.add_edge(2525,2620,weight=1)
G10.add_edge(2525,2621,weight=1)
G10.add_edge(2525,2736,weight=1)
G10.add_edge(2525,2767,weight=1)
G10.add_edge(2525,2830,weight=1)
G10.add_edge(2525,2969,weight=1)
G10.add_edge(2525,2970,weight=1)
G10.add_edge(2525,2971,weight=1)
G10.add_edge(2526,2527,weight=1)
G10.add_edge(2527,2713,weight=1)
G10.add_edge(2527,2753,weight=1)
G10.add_edge(2527,2754,weight=1)
G10.add_edge(2527,2761,weight=1)
G10.add_edge(2527,2762,weight=1)
G10.add_edge(2527,2763,weight=1)
G10.add_edge(2527,2801,weight=1)
G10.add_edge(2527,2945,weight=1)
G10.add_edge(2528,2529,weight=1)
G10.add_edge(2528,2530,weight=1)
G10.add_edge(2529,2531,weight=1)
G10.add_edge(2531,2629,weight=1)
G10.add_edge(2531,2941,weight=1)
G10.add_edge(2532,2533,weight=1)
G10.add_edge(2532,2534,weight=1)
G10.add_edge(2533,2922,weight=1)
G10.add_edge(2535,2536,weight=1)
G10.add_edge(2535,2537,weight=1)
G10.add_edge(2535,2538,weight=1)
G10.add_edge(2535,2539,weight=1)
G10.add_edge(2535,2540,weight=1)
G10.add_edge(2535,2541,weight=1)
G10.add_edge(2535,2542,weight=1)
G10.add_edge(2535,2543,weight=1)
G10.add_edge(2535,2544,weight=1)
G10.add_edge(2535,2545,weight=1)
G10.add_edge(2535,2546,weight=1)
G10.add_edge(2535,2547,weight=1)
G10.add_edge(2536,2552,weight=1)
G10.add_edge(2537,2649,weight=1)
G10.add_edge(2537,2650,weight=1)
G10.add_edge(2538,2773,weight=1)
G10.add_edge(2539,2651,weight=1)
G10.add_edge(2539,2652,weight=1)
G10.add_edge(2539,2919,weight=1)
G10.add_edge(2539,2938,weight=1)
G10.add_edge(2540,2650,weight=1)
G10.add_edge(2540,2972,weight=1)
G10.add_edge(2541,2866,weight=1)
G10.add_edge(2541,2977,weight=1)
G10.add_edge(2542,2866,weight=1)
G10.add_edge(2542,2978,weight=1)
G10.add_edge(2543,2544,weight=1)
G10.add_edge(2543,2773,weight=1)
G10.add_edge(2543,2958,weight=1)
G10.add_edge(2543,3032,weight=1)
G10.add_edge(2543,3033,weight=1)
G10.add_edge(2543,3034,weight=1)
G10.add_edge(2543,3035,weight=1)
G10.add_edge(2543,3036,weight=1)
G10.add_edge(2543,3037,weight=1)
G10.add_edge(2543,3038,weight=1)
G10.add_edge(2543,3039,weight=1)
G10.add_edge(2544,3040,weight=1)
G10.add_edge(2545,3093,weight=1)
G10.add_edge(2546,2805,weight=1)
G10.add_edge(2547,2741,weight=1)
G10.add_edge(2547,3096,weight=1)
G10.add_edge(2548,2549,weight=1)
G10.add_edge(2548,2550,weight=1)
G10.add_edge(2548,2551,weight=1)
G10.add_edge(2549,2694,weight=1)
G10.add_edge(2549,2695,weight=1)
G10.add_edge(2549,2696,weight=1)
G10.add_edge(2549,2697,weight=1)
G10.add_edge(2550,2741,weight=1)
G10.add_edge(2550,2991,weight=1)
G10.add_edge(2551,2752,weight=1)
G10.add_edge(2551,2763,weight=1)
G10.add_edge(2551,3044,weight=1)
G10.add_edge(2551,3045,weight=1)
G10.add_edge(2552,2927,weight=1)
G10.add_edge(2552,3052,weight=1)
G10.add_edge(2553,2554,weight=1)
G10.add_edge(2554,2555,weight=1)
G10.add_edge(2554,2556,weight=1)
G10.add_edge(2554,2557,weight=1)
G10.add_edge(2554,2558,weight=1)
G10.add_edge(2554,2559,weight=1)
G10.add_edge(2554,2560,weight=1)
G10.add_edge(2554,2561,weight=1)
G10.add_edge(2554,2562,weight=1)
G10.add_edge(2554,2563,weight=1)
G10.add_edge(2554,2564,weight=1)
G10.add_edge(2554,2565,weight=1)
G10.add_edge(2556,2594,weight=1)
G10.add_edge(2556,2595,weight=1)
G10.add_edge(2557,2783,weight=1)
G10.add_edge(2557,2784,weight=1)
G10.add_edge(2557,2785,weight=1)
G10.add_edge(2557,2786,weight=1)
G10.add_edge(2558,2647,weight=1)
G10.add_edge(2558,2783,weight=1)
G10.add_edge(2558,2790,weight=1)
G10.add_edge(2558,2925,weight=1)
G10.add_edge(2558,2946,weight=1)
G10.add_edge(2558,2947,weight=1)
G10.add_edge(2558,2948,weight=1)
G10.add_edge(2559,2988,weight=1)
G10.add_edge(2559,2989,weight=1)
G10.add_edge(2560,2867,weight=1)
G10.add_edge(2561,2647,weight=1)
G10.add_edge(2564,3257,weight=1)
G10.add_edge(2565,2631,weight=1)
G10.add_edge(2565,3317,weight=1)
G10.add_edge(2566,2567,weight=1)
G10.add_edge(2566,2568,weight=1)
G10.add_edge(2567,4270,weight=1)
G10.add_edge(2567,4271,weight=1)
G10.add_edge(2568,4246,weight=1)
G10.add_edge(2568,4293,weight=1)
G10.add_edge(2569,2570,weight=1)
G10.add_edge(2570,2571,weight=1)
G10.add_edge(2570,2572,weight=1)
G10.add_edge(2570,2573,weight=1)
G10.add_edge(2570,2574,weight=1)
G10.add_edge(2572,2586,weight=1)
G10.add_edge(2572,2764,weight=1)
G10.add_edge(2572,2765,weight=1)
G10.add_edge(2573,2747,weight=1)
G10.add_edge(2573,2967,weight=1)
G10.add_edge(2573,2968,weight=1)
G10.add_edge(2574,2586,weight=1)
G10.add_edge(2574,2652,weight=1)
G10.add_edge(2574,2747,weight=1)
G10.add_edge(2574,2979,weight=1)
G10.add_edge(2574,2980,weight=1)
G10.add_edge(2575,2576,weight=1)
G10.add_edge(2575,2577,weight=1)
G10.add_edge(2575,2578,weight=1)
G10.add_edge(2575,2579,weight=1)
G10.add_edge(2575,2580,weight=1)
G10.add_edge(2577,2581,weight=1)
G10.add_edge(2577,3062,weight=1)
G10.add_edge(2579,2580,weight=1)
G10.add_edge(2579,2582,weight=1)
G10.add_edge(2579,2954,weight=1)
G10.add_edge(2581,2582,weight=1)
G10.add_edge(2581,2583,weight=1)
G10.add_edge(2582,2608,weight=1)
G10.add_edge(2582,2733,weight=1)
G10.add_edge(2582,2734,weight=1)
G10.add_edge(2582,2953,weight=1)
G10.add_edge(2582,2954,weight=1)
G10.add_edge(2584,2585,weight=1)
G10.add_edge(2584,2586,weight=1)
G10.add_edge(2584,2587,weight=1)
G10.add_edge(2585,2661,weight=1)
G10.add_edge(2585,2663,weight=1)
G10.add_edge(2585,2704,weight=1)
G10.add_edge(2585,2729,weight=1)
G10.add_edge(2585,2730,weight=1)
G10.add_edge(2585,2731,weight=1)
G10.add_edge(2586,2658,weight=1)
G10.add_edge(2586,2660,weight=1)
G10.add_edge(2586,2872,weight=1)
G10.add_edge(2586,2873,weight=1)
G10.add_edge(2588,2589,weight=1)
G10.add_edge(2588,2616,weight=1)
G10.add_edge(2588,2617,weight=1)
G10.add_edge(2588,2618,weight=1)
G10.add_edge(2588,2619,weight=1)
G10.add_edge(2589,3098,weight=1)
G10.add_edge(2590,2591,weight=1)
G10.add_edge(2590,3098,weight=1)
G10.add_edge(2590,3175,weight=1)
G10.add_edge(2590,3200,weight=1)
G10.add_edge(2591,2619,weight=1)
G10.add_edge(2591,3121,weight=1)
G10.add_edge(2591,3171,weight=1)
G10.add_edge(2592,2593,weight=1)
G10.add_edge(2593,2785,weight=1)
G10.add_edge(2593,2865,weight=1)
G10.add_edge(2594,2906,weight=1)
G10.add_edge(2594,3065,weight=1)
G10.add_edge(2594,3066,weight=1)
G10.add_edge(2594,3067,weight=1)
G10.add_edge(2594,3068,weight=1)
G10.add_edge(2594,3069,weight=1)
G10.add_edge(2595,2969,weight=1)
G10.add_edge(2595,3366,weight=1)
G10.add_edge(2596,2597,weight=1)
G10.add_edge(2596,2598,weight=1)
G10.add_edge(2596,2599,weight=1)
G10.add_edge(2596,2600,weight=1)
G10.add_edge(2596,2601,weight=1)
G10.add_edge(2598,2601,weight=1)
G10.add_edge(2598,2687,weight=1)
G10.add_edge(2598,2688,weight=1)
G10.add_edge(2598,2917,weight=1)
G10.add_edge(2598,2918,weight=1)
G10.add_edge(2599,2600,weight=1)
G10.add_edge(2599,2637,weight=1)
G10.add_edge(2599,2642,weight=1)
G10.add_edge(2599,2643,weight=1)
G10.add_edge(2599,2926,weight=1)
G10.add_edge(2599,2929,weight=1)
G10.add_edge(2599,2935,weight=1)
G10.add_edge(2599,3110,weight=1)
G10.add_edge(2599,3111,weight=1)
G10.add_edge(2602,2603,weight=1)
G10.add_edge(2602,2604,weight=1)
G10.add_edge(2602,2605,weight=1)
G10.add_edge(2603,2719,weight=1)
G10.add_edge(2603,2780,weight=1)
G10.add_edge(2603,2811,weight=1)
G10.add_edge(2603,2845,weight=1)
G10.add_edge(2604,2644,weight=1)
G10.add_edge(2604,2645,weight=1)
G10.add_edge(2604,2707,weight=1)
G10.add_edge(2604,2811,weight=1)
G10.add_edge(2604,2875,weight=1)
G10.add_edge(2604,2876,weight=1)
G10.add_edge(2604,2877,weight=1)
G10.add_edge(2605,3080,weight=1)
G10.add_edge(2605,3081,weight=1)
G10.add_edge(2605,3082,weight=1)
G10.add_edge(2605,3083,weight=1)
G10.add_edge(2606,2607,weight=1)
G10.add_edge(2607,2608,weight=1)
G10.add_edge(2607,2609,weight=1)
G10.add_edge(2608,2874,weight=1)
G10.add_edge(2609,3087,weight=1)
G10.add_edge(2609,3088,weight=1)
G10.add_edge(2609,3089,weight=1)
G10.add_edge(2609,3090,weight=1)
G10.add_edge(2609,3091,weight=1)
G10.add_edge(2609,3092,weight=1)
G10.add_edge(2610,2611,weight=1)
G10.add_edge(2610,2612,weight=1)
G10.add_edge(2610,2613,weight=1)
G10.add_edge(2610,2614,weight=1)
G10.add_edge(2611,2615,weight=1)
G10.add_edge(2611,3094,weight=1)
G10.add_edge(2612,2614,weight=1)
G10.add_edge(2612,3113,weight=1)
G10.add_edge(2612,3116,weight=1)
G10.add_edge(2612,3117,weight=1)
G10.add_edge(2612,3118,weight=1)
G10.add_edge(2613,3113,weight=1)
G10.add_edge(2613,3341,weight=1)
G10.add_edge(2615,2616,weight=1)
G10.add_edge(2616,2620,weight=1)
G10.add_edge(2616,2621,weight=1)
G10.add_edge(2616,2622,weight=1)
G10.add_edge(2616,2623,weight=1)
G10.add_edge(2616,2624,weight=1)
G10.add_edge(2618,3046,weight=1)
G10.add_edge(2618,3047,weight=1)
G10.add_edge(2618,3321,weight=1)
G10.add_edge(2619,3047,weight=1)
G10.add_edge(2620,2686,weight=1)
G10.add_edge(2622,2890,weight=1)
G10.add_edge(2622,2893,weight=1)
G10.add_edge(2622,2899,weight=1)
G10.add_edge(2622,2942,weight=1)
G10.add_edge(2622,2943,weight=1)
G10.add_edge(2623,2885,weight=1)
G10.add_edge(2623,3056,weight=1)
G10.add_edge(2624,3085,weight=1)
G10.add_edge(2624,3086,weight=1)
G10.add_edge(2624,3097,weight=1)
G10.add_edge(2624,3098,weight=1)
G10.add_edge(2625,2659,weight=1)
G10.add_edge(2626,2659,weight=1)
G10.add_edge(2626,2670,weight=1)
G10.add_edge(2627,2949,weight=1)
G10.add_edge(2628,2629,weight=1)
G10.add_edge(2630,2631,weight=1)
G10.add_edge(2631,2632,weight=1)
G10.add_edge(2632,2956,weight=1)
G10.add_edge(2632,3018,weight=1)
G10.add_edge(2633,2634,weight=1)
G10.add_edge(2633,2635,weight=1)
G10.add_edge(2633,2636,weight=1)
G10.add_edge(2634,2682,weight=1)
G10.add_edge(2634,2685,weight=1)
G10.add_edge(2637,2638,weight=1)
G10.add_edge(2637,2639,weight=1)
G10.add_edge(2637,2640,weight=1)
G10.add_edge(2637,2641,weight=1)
G10.add_edge(2638,2640,weight=1)
G10.add_edge(2638,2641,weight=1)
G10.add_edge(2639,3288,weight=1)
G10.add_edge(2639,3289,weight=1)
G10.add_edge(2639,3290,weight=1)
G10.add_edge(2640,2641,weight=1)
G10.add_edge(2644,2645,weight=1)
G10.add_edge(2644,2646,weight=1)
G10.add_edge(2645,2776,weight=1)
G10.add_edge(2645,2777,weight=1)
G10.add_edge(2645,2778,weight=1)
G10.add_edge(2645,2779,weight=1)
G10.add_edge(2647,2648,weight=1)
G10.add_edge(2648,2848,weight=1)
G10.add_edge(2648,3066,weight=1)
G10.add_edge(2650,2963,weight=1)
G10.add_edge(2651,2652,weight=1)
G10.add_edge(2651,2653,weight=1)
G10.add_edge(2652,2700,weight=1)
G10.add_edge(2652,2703,weight=1)
G10.add_edge(2652,2704,weight=1)
G10.add_edge(2652,2705,weight=1)
G10.add_edge(2652,2706,weight=1)
G10.add_edge(2654,2655,weight=1)
G10.add_edge(2654,2656,weight=1)
G10.add_edge(2654,2657,weight=1)
G10.add_edge(2655,2658,weight=1)
G10.add_edge(2655,2664,weight=1)
G10.add_edge(2656,2658,weight=1)
G10.add_edge(2656,2665,weight=1)
G10.add_edge(2657,2729,weight=1)
G10.add_edge(2657,2996,weight=1)
G10.add_edge(2657,2999,weight=1)
G10.add_edge(2657,3000,weight=1)
G10.add_edge(2657,3001,weight=1)
G10.add_edge(2658,2659,weight=1)
G10.add_edge(2658,2660,weight=1)
G10.add_edge(2658,2661,weight=1)
G10.add_edge(2658,2662,weight=1)
G10.add_edge(2659,2663,weight=1)
G10.add_edge(2660,2674,weight=1)
G10.add_edge(2661,2846,weight=1)
G10.add_edge(2663,2732,weight=1)
G10.add_edge(2664,2665,weight=1)
G10.add_edge(2666,2667,weight=1)
G10.add_edge(2666,2668,weight=1)
G10.add_edge(2667,2668,weight=1)
G10.add_edge(2667,2754,weight=1)
G10.add_edge(2667,2755,weight=1)
G10.add_edge(2667,2984,weight=1)
G10.add_edge(2667,3100,weight=1)
G10.add_edge(2667,3101,weight=1)
G10.add_edge(2668,3236,weight=1)
G10.add_edge(2668,3237,weight=1)
G10.add_edge(2668,3238,weight=1)
G10.add_edge(2669,2670,weight=1)
G10.add_edge(2669,2671,weight=1)
G10.add_edge(2669,2672,weight=1)
G10.add_edge(2670,2673,weight=1)
G10.add_edge(2674,2987,weight=1)
G10.add_edge(2675,2676,weight=1)
G10.add_edge(2675,2677,weight=1)
G10.add_edge(2675,2678,weight=1)
G10.add_edge(2675,2679,weight=1)
G10.add_edge(2676,2736,weight=1)
G10.add_edge(2676,2737,weight=1)
G10.add_edge(2676,2738,weight=1)
G10.add_edge(2676,2739,weight=1)
G10.add_edge(2677,2768,weight=1)
G10.add_edge(2677,3130,weight=1)
G10.add_edge(2677,3133,weight=1)
G10.add_edge(2677,3134,weight=1)
G10.add_edge(2678,3070,weight=1)
G10.add_edge(2678,3337,weight=1)
G10.add_edge(2679,3071,weight=1)
G10.add_edge(2679,3339,weight=1)
G10.add_edge(2680,2697,weight=1)
G10.add_edge(2680,2712,weight=1)
G10.add_edge(2681,3136,weight=1)
G10.add_edge(2681,3140,weight=1)
G10.add_edge(2682,3141,weight=1)
G10.add_edge(2683,2697,weight=1)
G10.add_edge(2683,2797,weight=1)
G10.add_edge(2685,2778,weight=1)
G10.add_edge(2685,2780,weight=1)
G10.add_edge(2685,2781,weight=1)
G10.add_edge(2685,2782,weight=1)
G10.add_edge(2686,2990,weight=1)
G10.add_edge(2687,2688,weight=1)
G10.add_edge(2687,2689,weight=1)
G10.add_edge(2687,2690,weight=1)
G10.add_edge(2689,2690,weight=1)
G10.add_edge(2691,2817,weight=1)
G10.add_edge(2691,2924,weight=1)
G10.add_edge(2692,2866,weight=1)
G10.add_edge(2693,2951,weight=1)
G10.add_edge(2694,2709,weight=1)
G10.add_edge(2694,2710,weight=1)
G10.add_edge(2694,2711,weight=1)
G10.add_edge(2695,2715,weight=1)
G10.add_edge(2695,2717,weight=1)
G10.add_edge(2696,2806,weight=1)
G10.add_edge(2697,2707,weight=1)
G10.add_edge(2697,2717,weight=1)
G10.add_edge(2697,2796,weight=1)
G10.add_edge(2697,2875,weight=1)
G10.add_edge(2698,2699,weight=1)
G10.add_edge(2699,2803,weight=1)
G10.add_edge(2700,2701,weight=1)
G10.add_edge(2700,2702,weight=1)
G10.add_edge(2701,2729,weight=1)
G10.add_edge(2701,2994,weight=1)
G10.add_edge(2701,2995,weight=1)
G10.add_edge(2702,2996,weight=1)
G10.add_edge(2702,3002,weight=1)
G10.add_edge(2703,2919,weight=1)
G10.add_edge(2703,3329,weight=1)
G10.add_edge(2704,3330,weight=1)
G10.add_edge(2707,2708,weight=1)
G10.add_edge(2708,2752,weight=1)
G10.add_edge(2708,2876,weight=1)
G10.add_edge(2708,3015,weight=1)
G10.add_edge(2708,3019,weight=1)
G10.add_edge(2708,3101,weight=1)
G10.add_edge(2708,3138,weight=1)
G10.add_edge(2708,3160,weight=1)
G10.add_edge(2712,3019,weight=1)
G10.add_edge(2712,3161,weight=1)
G10.add_edge(2713,2714,weight=1)
G10.add_edge(2713,2715,weight=1)
G10.add_edge(2713,2716,weight=1)
G10.add_edge(2714,3162,weight=1)
G10.add_edge(2715,3163,weight=1)
G10.add_edge(2717,2752,weight=1)
G10.add_edge(2718,2719,weight=1)
G10.add_edge(2718,2720,weight=1)
G10.add_edge(2718,2721,weight=1)
G10.add_edge(2719,2720,weight=1)
G10.add_edge(2719,2783,weight=1)
G10.add_edge(2719,2810,weight=1)
G10.add_edge(2719,2811,weight=1)
G10.add_edge(2719,2812,weight=1)
G10.add_edge(2719,2813,weight=1)
G10.add_edge(2719,2814,weight=1)
G10.add_edge(2720,2812,weight=1)
G10.add_edge(2720,2825,weight=1)
G10.add_edge(2720,2826,weight=1)
G10.add_edge(2720,2827,weight=1)
G10.add_edge(2720,2828,weight=1)
G10.add_edge(2722,2723,weight=1)
G10.add_edge(2722,2724,weight=1)
G10.add_edge(2722,2725,weight=1)
G10.add_edge(2722,2726,weight=1)
G10.add_edge(2722,2727,weight=1)
G10.add_edge(2722,2728,weight=1)
G10.add_edge(2723,2724,weight=1)
G10.add_edge(2723,2727,weight=1)
G10.add_edge(2723,2787,weight=1)
G10.add_edge(2723,2788,weight=1)
G10.add_edge(2723,2789,weight=1)
G10.add_edge(2723,2790,weight=1)
G10.add_edge(2723,2791,weight=1)
G10.add_edge(2724,2727,weight=1)
G10.add_edge(2724,2923,weight=1)
G10.add_edge(2732,2772,weight=1)
G10.add_edge(2732,2782,weight=1)
G10.add_edge(2732,2956,weight=1)
G10.add_edge(2732,2964,weight=1)
G10.add_edge(2732,2965,weight=1)
G10.add_edge(2733,2734,weight=1)
G10.add_edge(2733,2735,weight=1)
G10.add_edge(2734,2961,weight=1)
G10.add_edge(2736,2766,weight=1)
G10.add_edge(2736,2767,weight=1)
G10.add_edge(2736,2768,weight=1)
G10.add_edge(2736,2769,weight=1)
G10.add_edge(2736,2770,weight=1)
G10.add_edge(2736,2771,weight=1)
G10.add_edge(2737,2738,weight=1)
G10.add_edge(2737,3172,weight=1)
G10.add_edge(2737,3173,weight=1)
G10.add_edge(2738,3172,weight=1)
G10.add_edge(2739,2766,weight=1)
G10.add_edge(2739,3332,weight=1)
G10.add_edge(2740,2741,weight=1)
G10.add_edge(2740,2742,weight=1)
G10.add_edge(2740,2743,weight=1)
G10.add_edge(2741,2976,weight=1)
G10.add_edge(2744,2745,weight=1)
G10.add_edge(2745,2746,weight=1)
G10.add_edge(2747,2748,weight=1)
G10.add_edge(2747,2749,weight=1)
G10.add_edge(2750,2751,weight=1)
G10.add_edge(2751,2968,weight=1)
G10.add_edge(2752,3015,weight=1)
G10.add_edge(2753,2754,weight=1)
G10.add_edge(2754,2755,weight=1)
G10.add_edge(2754,2756,weight=1)
G10.add_edge(2757,2758,weight=1)
G10.add_edge(2757,2759,weight=1)
G10.add_edge(2761,2762,weight=1)
G10.add_edge(2761,2763,weight=1)
G10.add_edge(2762,2763,weight=1)
G10.add_edge(2762,2796,weight=1)
G10.add_edge(2762,2797,weight=1)
G10.add_edge(2762,2798,weight=1)
G10.add_edge(2762,2799,weight=1)
G10.add_edge(2762,2800,weight=1)
G10.add_edge(2762,2801,weight=1)
G10.add_edge(2763,2945,weight=1)
G10.add_edge(2763,3044,weight=1)
G10.add_edge(2763,3045,weight=1)
G10.add_edge(2763,3322,weight=1)
G10.add_edge(2764,2919,weight=1)
G10.add_edge(2764,2993,weight=1)
G10.add_edge(2766,2865,weight=1)
G10.add_edge(2766,2906,weight=1)
G10.add_edge(2766,2916,weight=1)
G10.add_edge(2767,2966,weight=1)
G10.add_edge(2768,3134,weight=1)
G10.add_edge(2768,3194,weight=1)
G10.add_edge(2769,3195,weight=1)
G10.add_edge(2770,3342,weight=1)
G10.add_edge(2771,3281,weight=1)
G10.add_edge(2771,3345,weight=1)
G10.add_edge(2773,2972,weight=1)
G10.add_edge(2773,3032,weight=1)
G10.add_edge(2773,3197,weight=1)
G10.add_edge(2773,3198,weight=1)
G10.add_edge(2773,3199,weight=1)
G10.add_edge(2774,2775,weight=1)
G10.add_edge(2775,2940,weight=1)
G10.add_edge(2775,2968,weight=1)
G10.add_edge(2776,2950,weight=1)
G10.add_edge(2777,2879,weight=1)
G10.add_edge(2778,3193,weight=1)
G10.add_edge(2779,3160,weight=1)
G10.add_edge(2779,3201,weight=1)
G10.add_edge(2779,3202,weight=1)
G10.add_edge(2779,3203,weight=1)
G10.add_edge(2780,2810,weight=1)
G10.add_edge(2780,2839,weight=1)
G10.add_edge(2780,2840,weight=1)
G10.add_edge(2781,2782,weight=1)
G10.add_edge(2781,2810,weight=1)
G10.add_edge(2781,2944,weight=1)
G10.add_edge(2781,2955,weight=1)
G10.add_edge(2781,2956,weight=1)
G10.add_edge(2781,2957,weight=1)
G10.add_edge(2782,2795,weight=1)
G10.add_edge(2783,2852,weight=1)
G10.add_edge(2783,2853,weight=1)
G10.add_edge(2783,2854,weight=1)
G10.add_edge(2783,2865,weight=1)
G10.add_edge(2783,2866,weight=1)
G10.add_edge(2783,2867,weight=1)
G10.add_edge(2784,2785,weight=1)
G10.add_edge(2784,2786,weight=1)
G10.add_edge(2784,2847,weight=1)
G10.add_edge(2784,3247,weight=1)
G10.add_edge(2785,2786,weight=1)
G10.add_edge(2786,3349,weight=1)
G10.add_edge(2786,3350,weight=1)
G10.add_edge(2786,3351,weight=1)
G10.add_edge(2786,3369,weight=1)
G10.add_edge(2787,2792,weight=1)
G10.add_edge(2787,2793,weight=1)
G10.add_edge(2787,2794,weight=1)
G10.add_edge(2788,2789,weight=1)
G10.add_edge(2788,2825,weight=1)
G10.add_edge(2788,3079,weight=1)
G10.add_edge(2789,2790,weight=1)
G10.add_edge(2789,3025,weight=1)
G10.add_edge(2789,3271,weight=1)
G10.add_edge(2790,2791,weight=1)
G10.add_edge(2790,2855,weight=1)
G10.add_edge(2790,2946,weight=1)
G10.add_edge(2790,2947,weight=1)
G10.add_edge(2790,3025,weight=1)
G10.add_edge(2790,3271,weight=1)
G10.add_edge(2792,2948,weight=1)
G10.add_edge(2793,2794,weight=1)
G10.add_edge(2793,3119,weight=1)
G10.add_edge(2794,3119,weight=1)
G10.add_edge(2796,2798,weight=1)
G10.add_edge(2796,2799,weight=1)
G10.add_edge(2796,2802,weight=1)
G10.add_edge(2796,2803,weight=1)
G10.add_edge(2797,2800,weight=1)
G10.add_edge(2798,3208,weight=1)
G10.add_edge(2799,3209,weight=1)
G10.add_edge(2802,2883,weight=1)
G10.add_edge(2802,2884,weight=1)
G10.add_edge(2804,2805,weight=1)
G10.add_edge(2805,2806,weight=1)
G10.add_edge(2806,2984,weight=1)
G10.add_edge(2806,2985,weight=1)
G10.add_edge(2807,2808,weight=1)
G10.add_edge(2807,2809,weight=1)
G10.add_edge(2808,3137,weight=1)
G10.add_edge(2808,3217,weight=1)
G10.add_edge(2811,2841,weight=1)
G10.add_edge(2811,2842,weight=1)
G10.add_edge(2811,2843,weight=1)
G10.add_edge(2811,2844,weight=1)
G10.add_edge(2813,2861,weight=1)
G10.add_edge(2814,2861,weight=1)
G10.add_edge(2815,2816,weight=1)
G10.add_edge(2816,2817,weight=1)
G10.add_edge(2816,3132,weight=1)
G10.add_edge(2818,2819,weight=1)
G10.add_edge(2818,2823,weight=1)
G10.add_edge(2818,3124,weight=1)
G10.add_edge(2818,3125,weight=1)
G10.add_edge(2819,3020,weight=1)
G10.add_edge(2819,3125,weight=1)
G10.add_edge(2819,3184,weight=1)
G10.add_edge(2819,3207,weight=1)
G10.add_edge(2819,3295,weight=1)
G10.add_edge(2820,3181,weight=1)
G10.add_edge(2820,3182,weight=1)
G10.add_edge(2820,3186,weight=1)
G10.add_edge(2820,3294,weight=1)
G10.add_edge(2822,3137,weight=1)
G10.add_edge(2822,3282,weight=1)
G10.add_edge(2822,3363,weight=1)
G10.add_edge(2822,3365,weight=1)
G10.add_edge(2824,4185,weight=1)
G10.add_edge(2825,2828,weight=1)
G10.add_edge(2826,2847,weight=1)
G10.add_edge(2826,2850,weight=1)
G10.add_edge(2826,2855,weight=1)
G10.add_edge(2826,2860,weight=1)
G10.add_edge(2828,3267,weight=1)
G10.add_edge(2829,2865,weight=1)
G10.add_edge(2831,3066,weight=1)
G10.add_edge(2831,3233,weight=1)
G10.add_edge(2832,3066,weight=1)
G10.add_edge(2832,3234,weight=1)
G10.add_edge(2833,3066,weight=1)
G10.add_edge(2833,3235,weight=1)
G10.add_edge(2834,2865,weight=1)
G10.add_edge(2837,3137,weight=1)
G10.add_edge(2841,3223,weight=1)
G10.add_edge(2842,3223,weight=1)
G10.add_edge(2845,3223,weight=1)
G10.add_edge(2845,3226,weight=1)
G10.add_edge(2845,3227,weight=1)
G10.add_edge(2845,3228,weight=1)
G10.add_edge(2845,3229,weight=1)
G10.add_edge(2845,3230,weight=1)
G10.add_edge(2845,3231,weight=1)
G10.add_edge(2847,2848,weight=1)
G10.add_edge(2847,2849,weight=1)
G10.add_edge(2847,2850,weight=1)
G10.add_edge(2847,2851,weight=1)
G10.add_edge(2847,2852,weight=1)
G10.add_edge(2847,2853,weight=1)
G10.add_edge(2847,2854,weight=1)
G10.add_edge(2847,2855,weight=1)
G10.add_edge(2847,2856,weight=1)
G10.add_edge(2847,2857,weight=1)
G10.add_edge(2847,2858,weight=1)
G10.add_edge(2847,2859,weight=1)
G10.add_edge(2847,2860,weight=1)
G10.add_edge(2847,2861,weight=1)
G10.add_edge(2847,2862,weight=1)
G10.add_edge(2847,2863,weight=1)
G10.add_edge(2847,2864,weight=1)
G10.add_edge(2849,2856,weight=1)
G10.add_edge(2849,2858,weight=1)
G10.add_edge(2849,2864,weight=1)
G10.add_edge(2851,2857,weight=1)
G10.add_edge(2852,3244,weight=1)
G10.add_edge(2853,3245,weight=1)
G10.add_edge(2854,3246,weight=1)
G10.add_edge(2855,2860,weight=1)
G10.add_edge(2855,2925,weight=1)
G10.add_edge(2855,3025,weight=1)
G10.add_edge(2855,3026,weight=1)
G10.add_edge(2856,3274,weight=1)
G10.add_edge(2857,2973,weight=1)
G10.add_edge(2857,3107,weight=1)
G10.add_edge(2857,3108,weight=1)
G10.add_edge(2861,2863,weight=1)
G10.add_edge(2861,2864,weight=1)
G10.add_edge(2861,3334,weight=1)
G10.add_edge(2861,3335,weight=1)
G10.add_edge(2861,3348,weight=1)
G10.add_edge(2861,3357,weight=1)
G10.add_edge(2861,3358,weight=1)
G10.add_edge(2862,2864,weight=1)
G10.add_edge(2864,3334,weight=1)
G10.add_edge(2864,3335,weight=1)
G10.add_edge(2865,2902,weight=1)
G10.add_edge(2865,2910,weight=1)
G10.add_edge(2865,2911,weight=1)
G10.add_edge(2865,2912,weight=1)
G10.add_edge(2865,2913,weight=1)
G10.add_edge(2865,2914,weight=1)
G10.add_edge(2865,2915,weight=1)
G10.add_edge(2865,2916,weight=1)
G10.add_edge(2866,2880,weight=1)
G10.add_edge(2866,2925,weight=1)
G10.add_edge(2866,2951,weight=1)
G10.add_edge(2866,2956,weight=1)
G10.add_edge(2866,2957,weight=1)
G10.add_edge(2866,2960,weight=1)
G10.add_edge(2867,3368,weight=1)
G10.add_edge(2868,2869,weight=1)
G10.add_edge(2868,2870,weight=1)
G10.add_edge(2868,2871,weight=1)
G10.add_edge(2869,2871,weight=1)
G10.add_edge(2869,2974,weight=1)
G10.add_edge(2869,2975,weight=1)
G10.add_edge(2870,2974,weight=1)
G10.add_edge(2870,4178,weight=1)
G10.add_edge(2870,4220,weight=1)
G10.add_edge(2871,4222,weight=1)
G10.add_edge(2871,4243,weight=1)
G10.add_edge(2871,4265,weight=1)
G10.add_edge(2871,4269,weight=1)
G10.add_edge(2871,4318,weight=1)
G10.add_edge(2874,2931,weight=1)
G10.add_edge(2874,3087,weight=1)
G10.add_edge(2876,2877,weight=1)
G10.add_edge(2876,2984,weight=1)
G10.add_edge(2876,3019,weight=1)
G10.add_edge(2876,3101,weight=1)
G10.add_edge(2876,3138,weight=1)
G10.add_edge(2876,3160,weight=1)
G10.add_edge(2876,3248,weight=1)
G10.add_edge(2877,3265,weight=1)
G10.add_edge(2878,2879,weight=1)
G10.add_edge(2878,2880,weight=1)
G10.add_edge(2878,2881,weight=1)
G10.add_edge(2878,2882,weight=1)
G10.add_edge(2879,2973,weight=1)
G10.add_edge(2881,3255,weight=1)
G10.add_edge(2882,3255,weight=1)
G10.add_edge(2883,3028,weight=1)
G10.add_edge(2885,2886,weight=1)
G10.add_edge(2885,2887,weight=1)
G10.add_edge(2885,2888,weight=1)
G10.add_edge(2886,3204,weight=1)
G10.add_edge(2887,3259,weight=1)
G10.add_edge(2888,2890,weight=1)
G10.add_edge(2888,3291,weight=1)
G10.add_edge(2889,2890,weight=1)
G10.add_edge(2891,2892,weight=1)
G10.add_edge(2892,2893,weight=1)
G10.add_edge(2893,2894,weight=1)
G10.add_edge(2894,3043,weight=1)
G10.add_edge(2895,2896,weight=1)
G10.add_edge(2896,2942,weight=1)
G10.add_edge(2896,3263,weight=1)
G10.add_edge(2897,2898,weight=1)
G10.add_edge(2898,2899,weight=1)
G10.add_edge(2899,2900,weight=1)
G10.add_edge(2900,2901,weight=1)
G10.add_edge(2902,2903,weight=1)
G10.add_edge(2902,2904,weight=1)
G10.add_edge(2902,2905,weight=1)
G10.add_edge(2902,2906,weight=1)
G10.add_edge(2902,2907,weight=1)
G10.add_edge(2902,2908,weight=1)
G10.add_edge(2902,2909,weight=1)
G10.add_edge(2903,2906,weight=1)
G10.add_edge(2903,2907,weight=1)
G10.add_edge(2903,3075,weight=1)
G10.add_edge(2903,3076,weight=1)
G10.add_edge(2903,3077,weight=1)
G10.add_edge(2906,2907,weight=1)
G10.add_edge(2906,3066,weight=1)
G10.add_edge(2906,3221,weight=1)
G10.add_edge(2906,3283,weight=1)
G10.add_edge(2906,3284,weight=1)
G10.add_edge(2906,3285,weight=1)
G10.add_edge(2908,2909,weight=1)
G10.add_edge(2909,3370,weight=1)
G10.add_edge(2911,2956,weight=1)
G10.add_edge(2911,3128,weight=1)
G10.add_edge(2912,2956,weight=1)
G10.add_edge(2912,3177,weight=1)
G10.add_edge(2916,3304,weight=1)
G10.add_edge(2917,2918,weight=1)
G10.add_edge(2919,2920,weight=1)
G10.add_edge(2919,2921,weight=1)
G10.add_edge(2920,3324,weight=1)
G10.add_edge(2926,2927,weight=1)
G10.add_edge(2926,2928,weight=1)
G10.add_edge(2926,2929,weight=1)
G10.add_edge(2926,2930,weight=1)
G10.add_edge(2926,2931,weight=1)
G10.add_edge(2926,2932,weight=1)
G10.add_edge(2926,2933,weight=1)
G10.add_edge(2926,2934,weight=1)
G10.add_edge(2926,2935,weight=1)
G10.add_edge(2926,2936,weight=1)
G10.add_edge(2926,2937,weight=1)
G10.add_edge(2927,2928,weight=1)
G10.add_edge(2927,2937,weight=1)
G10.add_edge(2927,3053,weight=1)
G10.add_edge(2928,2936,weight=1)
G10.add_edge(2928,2937,weight=1)
G10.add_edge(2930,2931,weight=1)
G10.add_edge(2930,2932,weight=1)
G10.add_edge(2930,2970,weight=1)
G10.add_edge(2930,3112,weight=1)
G10.add_edge(2931,2932,weight=1)
G10.add_edge(2931,2934,weight=1)
G10.add_edge(2931,3252,weight=1)
G10.add_edge(2931,3253,weight=1)
G10.add_edge(2931,3254,weight=1)
G10.add_edge(2932,3112,weight=1)
G10.add_edge(2939,2940,weight=1)
G10.add_edge(2940,2968,weight=1)
G10.add_edge(2942,3099,weight=1)
G10.add_edge(2943,3266,weight=1)
G10.add_edge(2944,2953,weight=1)
G10.add_edge(2944,2955,weight=1)
G10.add_edge(2946,3279,weight=1)
G10.add_edge(2947,3280,weight=1)
G10.add_edge(2951,2952,weight=1)
G10.add_edge(2952,2978,weight=1)
G10.add_edge(2953,3296,weight=1)
G10.add_edge(2954,3103,weight=1)
G10.add_edge(2955,2971,weight=1)
G10.add_edge(2956,2958,weight=1)
G10.add_edge(2956,2959,weight=1)
G10.add_edge(2957,3302,weight=1)
G10.add_edge(2958,3034,weight=1)
G10.add_edge(2958,3205,weight=1)
G10.add_edge(2958,3297,weight=1)
G10.add_edge(2958,3298,weight=1)
G10.add_edge(2958,3299,weight=1)
G10.add_edge(2958,3300,weight=1)
G10.add_edge(2958,3301,weight=1)
G10.add_edge(2959,2989,weight=1)
G10.add_edge(2959,3016,weight=1)
G10.add_edge(2959,3367,weight=1)
G10.add_edge(2960,3014,weight=1)
G10.add_edge(2961,3303,weight=1)
G10.add_edge(2962,2963,weight=1)
G10.add_edge(2963,2992,weight=1)
G10.add_edge(2964,3013,weight=1)
G10.add_edge(2964,3372,weight=1)
G10.add_edge(2965,3013,weight=1)
G10.add_edge(2965,3373,weight=1)
G10.add_edge(2966,3305,weight=1)
G10.add_edge(2969,3258,weight=1)
G10.add_edge(2970,3306,weight=1)
G10.add_edge(2971,3307,weight=1)
G10.add_edge(2972,3197,weight=1)
G10.add_edge(2972,3239,weight=1)
G10.add_edge(2973,3107,weight=1)
G10.add_edge(2973,3309,weight=1)
G10.add_edge(2973,3310,weight=1)
G10.add_edge(2974,4182,weight=1)
G10.add_edge(2974,4183,weight=1)
G10.add_edge(2974,4185,weight=1)
G10.add_edge(2974,4249,weight=1)
G10.add_edge(2974,4301,weight=1)
G10.add_edge(2974,4306,weight=1)
G10.add_edge(2975,4386,weight=1)
G10.add_edge(2976,3319,weight=1)
G10.add_edge(2976,3320,weight=1)
G10.add_edge(2977,3084,weight=1)
G10.add_edge(2977,3311,weight=1)
G10.add_edge(2981,2982,weight=1)
G10.add_edge(2981,2983,weight=1)
G10.add_edge(2982,3192,weight=1)
G10.add_edge(2983,3061,weight=1)
G10.add_edge(2983,3316,weight=1)
G10.add_edge(2984,3160,weight=1)
G10.add_edge(2984,3248,weight=1)
G10.add_edge(2984,3315,weight=1)
G10.add_edge(2985,3238,weight=1)
G10.add_edge(2986,2987,weight=1)
G10.add_edge(2992,3269,weight=1)
G10.add_edge(2994,2996,weight=1)
G10.add_edge(2994,2997,weight=1)
G10.add_edge(2994,2998,weight=1)
G10.add_edge(2996,3003,weight=1)
G10.add_edge(2996,3010,weight=1)
G10.add_edge(2999,3003,weight=1)
G10.add_edge(2999,3004,weight=1)
G10.add_edge(2999,3005,weight=1)
G10.add_edge(2999,3006,weight=1)
G10.add_edge(2999,3007,weight=1)
G10.add_edge(2999,3008,weight=1)
G10.add_edge(2999,3009,weight=1)
G10.add_edge(3003,3011,weight=1)
G10.add_edge(3003,3012,weight=1)
G10.add_edge(3016,3017,weight=1)
G10.add_edge(3019,3136,weight=1)
G10.add_edge(3019,3137,weight=1)
G10.add_edge(3019,3138,weight=1)
G10.add_edge(3019,3139,weight=1)
G10.add_edge(3020,3022,weight=1)
G10.add_edge(3020,3184,weight=1)
G10.add_edge(3020,3207,weight=1)
G10.add_edge(3020,3210,weight=1)
G10.add_edge(3025,3026,weight=1)
G10.add_edge(3025,3027,weight=1)
G10.add_edge(3026,3027,weight=1)
G10.add_edge(3026,3078,weight=1)
G10.add_edge(3026,3079,weight=1)
G10.add_edge(3029,3030,weight=1)
G10.add_edge(3029,3031,weight=1)
G10.add_edge(3030,3331,weight=1)
G10.add_edge(3031,3331,weight=1)
G10.add_edge(3034,3035,weight=1)
G10.add_edge(3034,3036,weight=1)
G10.add_edge(3034,3039,weight=1)
G10.add_edge(3034,3205,weight=1)
G10.add_edge(3034,3206,weight=1)
G10.add_edge(3035,3036,weight=1)
G10.add_edge(3035,3037,weight=1)
G10.add_edge(3035,3038,weight=1)
G10.add_edge(3035,3073,weight=1)
G10.add_edge(3035,3206,weight=1)
G10.add_edge(3036,3039,weight=1)
G10.add_edge(3036,3206,weight=1)
G10.add_edge(3036,3292,weight=1)
G10.add_edge(3037,3038,weight=1)
G10.add_edge(3044,3045,weight=1)
G10.add_edge(3044,3325,weight=1)
G10.add_edge(3044,3326,weight=1)
G10.add_edge(3046,3047,weight=1)
G10.add_edge(3046,3048,weight=1)
G10.add_edge(3046,3049,weight=1)
G10.add_edge(3046,3050,weight=1)
G10.add_edge(3046,3051,weight=1)
G10.add_edge(3047,3121,weight=1)
G10.add_edge(3047,3164,weight=1)
G10.add_edge(3048,3050,weight=1)
G10.add_edge(3048,3117,weight=1)
G10.add_edge(3048,3178,weight=1)
G10.add_edge(3048,3179,weight=1)
G10.add_edge(3049,3051,weight=1)
G10.add_edge(3049,3121,weight=1)
G10.add_edge(3049,3167,weight=1)
G10.add_edge(3054,3055,weight=1)
G10.add_edge(3055,3223,weight=1)
G10.add_edge(3055,3225,weight=1)
G10.add_edge(3055,3228,weight=1)
G10.add_edge(3055,3229,weight=1)
G10.add_edge(3055,3268,weight=1)
G10.add_edge(3057,3058,weight=1)
G10.add_edge(3057,3059,weight=1)
G10.add_edge(3058,3059,weight=1)
G10.add_edge(3059,3073,weight=1)
G10.add_edge(3059,3074,weight=1)
G10.add_edge(3061,3314,weight=1)
G10.add_edge(3062,3063,weight=1)
G10.add_edge(3062,3064,weight=1)
G10.add_edge(3063,3170,weight=1)
G10.add_edge(3064,3168,weight=1)
G10.add_edge(3064,3169,weight=1)
G10.add_edge(3065,3069,weight=1)
G10.add_edge(3066,3220,weight=1)
G10.add_edge(3066,3221,weight=1)
G10.add_edge(3066,3222,weight=1)
G10.add_edge(3067,3068,weight=1)
G10.add_edge(3072,3073,weight=1)
G10.add_edge(3073,3074,weight=1)
G10.add_edge(3075,3076,weight=1)
G10.add_edge(3078,3079,weight=1)
G10.add_edge(3079,3371,weight=1)
G10.add_edge(3085,3086,weight=1)
G10.add_edge(3086,3174,weight=1)
G10.add_edge(3086,3175,weight=1)
G10.add_edge(3086,3176,weight=1)
G10.add_edge(3087,3091,weight=1)
G10.add_edge(3087,3249,weight=1)
G10.add_edge(3087,3250,weight=1)
G10.add_edge(3087,3251,weight=1)
G10.add_edge(3089,3090,weight=1)
G10.add_edge(3089,3092,weight=1)
G10.add_edge(3090,3092,weight=1)
G10.add_edge(3094,3095,weight=1)
G10.add_edge(3097,3098,weight=1)
G10.add_edge(3100,3101,weight=1)
G10.add_edge(3101,3160,weight=1)
G10.add_edge(3101,3216,weight=1)
G10.add_edge(3102,3103,weight=1)
G10.add_edge(3102,3104,weight=1)
G10.add_edge(3102,3105,weight=1)
G10.add_edge(3102,3106,weight=1)
G10.add_edge(3103,3106,weight=1)
G10.add_edge(3107,3108,weight=1)
G10.add_edge(3107,3109,weight=1)
G10.add_edge(3110,4087,weight=1)
G10.add_edge(3111,4085,weight=1)
G10.add_edge(3113,3114,weight=1)
G10.add_edge(3113,3115,weight=1)
G10.add_edge(3113,3116,weight=1)
G10.add_edge(3113,3117,weight=1)
G10.add_edge(3113,3118,weight=1)
G10.add_edge(3115,3118,weight=1)
G10.add_edge(3116,3117,weight=1)
G10.add_edge(3116,3118,weight=1)
G10.add_edge(3116,3133,weight=1)
G10.add_edge(3117,3179,weight=1)
G10.add_edge(3117,3215,weight=1)
G10.add_edge(3118,3232,weight=1)
G10.add_edge(3120,3121,weight=1)
G10.add_edge(3120,3122,weight=1)
G10.add_edge(3120,3123,weight=1)
G10.add_edge(3121,3122,weight=1)
G10.add_edge(3121,3164,weight=1)
G10.add_edge(3121,3171,weight=1)
G10.add_edge(3122,3327,weight=1)
G10.add_edge(3122,3328,weight=1)
G10.add_edge(3129,3130,weight=1)
G10.add_edge(3130,3133,weight=1)
G10.add_edge(3131,3132,weight=1)
G10.add_edge(3133,3135,weight=1)
G10.add_edge(3134,3194,weight=1)
G10.add_edge(3137,3138,weight=1)
G10.add_edge(3137,3139,weight=1)
G10.add_edge(3137,3217,weight=1)
G10.add_edge(3137,3282,weight=1)
G10.add_edge(3138,3139,weight=1)
G10.add_edge(3138,3340,weight=1)
G10.add_edge(3142,3143,weight=1)
G10.add_edge(3143,3354,weight=1)
G10.add_edge(3144,3145,weight=1)
G10.add_edge(3145,3337,weight=1)
G10.add_edge(3146,3147,weight=1)
G10.add_edge(3147,3356,weight=1)
G10.add_edge(3148,3149,weight=1)
G10.add_edge(3149,3352,weight=1)
G10.add_edge(3150,3151,weight=1)
G10.add_edge(3151,3356,weight=1)
G10.add_edge(3152,3153,weight=1)
G10.add_edge(3153,3331,weight=1)
G10.add_edge(3154,3155,weight=1)
G10.add_edge(3155,3339,weight=1)
G10.add_edge(3156,3157,weight=1)
G10.add_edge(3157,3343,weight=1)
G10.add_edge(3158,3159,weight=1)
G10.add_edge(3159,3353,weight=1)
G10.add_edge(3160,3203,weight=1)
G10.add_edge(3160,3248,weight=1)
G10.add_edge(3164,3165,weight=1)
G10.add_edge(3166,3167,weight=1)
G10.add_edge(3168,3169,weight=1)
G10.add_edge(3169,3170,weight=1)
G10.add_edge(3170,3196,weight=1)
G10.add_edge(3170,3218,weight=1)
G10.add_edge(3170,3219,weight=1)
G10.add_edge(3175,3176,weight=1)
G10.add_edge(3175,3200,weight=1)
G10.add_edge(3175,3308,weight=1)
G10.add_edge(3179,3215,weight=1)
G10.add_edge(3180,3181,weight=1)
G10.add_edge(3180,3182,weight=1)
G10.add_edge(3180,3183,weight=1)
G10.add_edge(3181,3182,weight=1)
G10.add_edge(3181,3184,weight=1)
G10.add_edge(3181,3185,weight=1)
G10.add_edge(3181,3186,weight=1)
G10.add_edge(3181,3187,weight=1)
G10.add_edge(3181,3188,weight=1)
G10.add_edge(3182,3184,weight=1)
G10.add_edge(3182,3185,weight=1)
G10.add_edge(3182,3186,weight=1)
G10.add_edge(3182,3187,weight=1)
G10.add_edge(3182,3188,weight=1)
G10.add_edge(3182,3191,weight=1)
G10.add_edge(3184,3207,weight=1)
G10.add_edge(3198,3199,weight=1)
G10.add_edge(3200,3312,weight=1)
G10.add_edge(3200,3313,weight=1)
G10.add_edge(3220,3221,weight=1)
G10.add_edge(3220,3222,weight=1)
G10.add_edge(3221,3222,weight=1)
G10.add_edge(3223,3224,weight=1)
G10.add_edge(3223,3225,weight=1)
G10.add_edge(3223,3226,weight=1)
G10.add_edge(3223,3227,weight=1)
G10.add_edge(3223,3228,weight=1)
G10.add_edge(3223,3229,weight=1)
G10.add_edge(3224,3228,weight=1)
G10.add_edge(3224,3275,weight=1)
G10.add_edge(3224,3277,weight=1)
G10.add_edge(3225,3229,weight=1)
G10.add_edge(3226,3227,weight=1)
G10.add_edge(3228,3362,weight=1)
G10.add_edge(3229,3230,weight=1)
G10.add_edge(3236,3238,weight=1)
G10.add_edge(3238,3361,weight=1)
G10.add_edge(3240,3241,weight=1)
G10.add_edge(3241,3345,weight=1)
G10.add_edge(3242,3243,weight=1)
G10.add_edge(3243,3344,weight=1)
G10.add_edge(3249,3250,weight=1)
G10.add_edge(3249,3251,weight=1)
G10.add_edge(3250,3251,weight=1)
G10.add_edge(3251,3359,weight=1)
G10.add_edge(3251,3360,weight=1)
G10.add_edge(3252,3253,weight=1)
G10.add_edge(3252,3254,weight=1)
G10.add_edge(3253,3254,weight=1)
G10.add_edge(3253,3318,weight=1)
G10.add_edge(3253,3347,weight=1)
G10.add_edge(3253,3364,weight=1)
G10.add_edge(3255,3256,weight=1)
G10.add_edge(3259,3260,weight=1)
G10.add_edge(3259,3261,weight=1)
G10.add_edge(3259,3262,weight=1)
G10.add_edge(3260,3262,weight=1)
G10.add_edge(3263,3264,weight=1)
G10.add_edge(3269,3270,weight=1)
G10.add_edge(3271,3278,weight=1)
G10.add_edge(3272,3273,weight=1)
G10.add_edge(3273,3352,weight=1)
G10.add_edge(3275,3276,weight=1)
G10.add_edge(3275,3277,weight=1)
G10.add_edge(3277,3286,weight=1)
G10.add_edge(3277,3287,weight=1)
G10.add_edge(3283,3284,weight=1)
G10.add_edge(3290,4150,weight=1)
G10.add_edge(3297,3298,weight=1)
G10.add_edge(3297,3300,weight=1)
G10.add_edge(3298,3300,weight=1)
G10.add_edge(3299,3301,weight=1)
G10.add_edge(3312,3313,weight=1)
G10.add_edge(3319,3320,weight=1)
G10.add_edge(3323,3324,weight=1)
G10.add_edge(3325,3326,weight=1)
G10.add_edge(3331,3344,weight=1)
G10.add_edge(3332,3333,weight=1)
G10.add_edge(3334,3335,weight=1)
G10.add_edge(3335,3348,weight=1)
G10.add_edge(3335,3358,weight=1)
G10.add_edge(3336,3337,weight=1)
G10.add_edge(3338,3339,weight=1)
G10.add_edge(3342,3343,weight=1)
G10.add_edge(3342,3344,weight=1)
G10.add_edge(3343,3356,weight=1)
G10.add_edge(3345,3346,weight=1)
G10.add_edge(3346,3353,weight=1)
G10.add_edge(3346,3354,weight=1)
G10.add_edge(3349,3350,weight=1)
G10.add_edge(3349,3351,weight=1)
G10.add_edge(3350,3351,weight=1)
G10.add_edge(3352,3353,weight=1)
G10.add_edge(3354,3355,weight=1)
G10.add_edge(3357,3358,weight=1)
G10.add_edge(3359,3360,weight=1)
G10.add_edge(3374,3375,weight=1)
G10.add_edge(3374,3376,weight=1)
G10.add_edge(3374,3377,weight=1)
G10.add_edge(3375,3659,weight=1)
G10.add_edge(3375,3661,weight=1)
G10.add_edge(3375,3662,weight=1)
G10.add_edge(3375,3665,weight=1)
G10.add_edge(3375,3666,weight=1)
G10.add_edge(3375,3667,weight=1)
G10.add_edge(3376,3506,weight=1)
G10.add_edge(3377,3440,weight=1)
G10.add_edge(3378,3379,weight=1)
G10.add_edge(3378,3380,weight=1)
G10.add_edge(3378,3381,weight=1)
G10.add_edge(3378,3382,weight=1)
G10.add_edge(3379,3383,weight=1)
G10.add_edge(3379,3384,weight=1)
G10.add_edge(3380,3517,weight=1)
G10.add_edge(3380,3518,weight=1)
G10.add_edge(3382,3429,weight=1)
G10.add_edge(3383,3395,weight=1)
G10.add_edge(3383,3448,weight=1)
G10.add_edge(3383,3450,weight=1)
G10.add_edge(3384,4549,weight=1)
G10.add_edge(3384,4674,weight=1)
G10.add_edge(3385,3386,weight=1)
G10.add_edge(3385,3387,weight=1)
G10.add_edge(3385,3388,weight=1)
G10.add_edge(3385,3389,weight=1)
G10.add_edge(3385,3390,weight=1)
G10.add_edge(3386,3391,weight=1)
G10.add_edge(3386,3392,weight=1)
G10.add_edge(3386,3393,weight=1)
G10.add_edge(3387,3496,weight=1)
G10.add_edge(3388,3692,weight=1)
G10.add_edge(3389,3514,weight=1)
G10.add_edge(3389,3757,weight=1)
G10.add_edge(3390,3804,weight=1)
G10.add_edge(3390,3805,weight=1)
G10.add_edge(3391,3393,weight=1)
G10.add_edge(3391,3430,weight=1)
G10.add_edge(3391,3434,weight=1)
G10.add_edge(3391,3435,weight=1)
G10.add_edge(3391,3436,weight=1)
G10.add_edge(3391,3437,weight=1)
G10.add_edge(3391,3438,weight=1)
G10.add_edge(3392,3497,weight=1)
G10.add_edge(3393,3440,weight=1)
G10.add_edge(3393,3513,weight=1)
G10.add_edge(3393,3516,weight=1)
G10.add_edge(3394,3395,weight=1)
G10.add_edge(3394,3396,weight=1)
G10.add_edge(3395,3397,weight=1)
G10.add_edge(3395,3398,weight=1)
G10.add_edge(3395,3399,weight=1)
G10.add_edge(3395,3400,weight=1)
G10.add_edge(3396,3782,weight=1)
G10.add_edge(3396,3784,weight=1)
G10.add_edge(3397,3400,weight=1)
G10.add_edge(3397,3503,weight=1)
G10.add_edge(3397,3504,weight=1)
G10.add_edge(3397,3505,weight=1)
G10.add_edge(3397,3506,weight=1)
G10.add_edge(3397,3507,weight=1)
G10.add_edge(3401,3402,weight=1)
G10.add_edge(3402,3403,weight=1)
G10.add_edge(3402,3707,weight=1)
G10.add_edge(3403,3404,weight=1)
G10.add_edge(3403,3405,weight=1)
G10.add_edge(3404,3492,weight=1)
G10.add_edge(3405,3792,weight=1)
G10.add_edge(3405,3847,weight=1)
G10.add_edge(3406,3407,weight=1)
G10.add_edge(3406,3408,weight=1)
G10.add_edge(3406,3409,weight=1)
G10.add_edge(3408,3515,weight=1)
G10.add_edge(3408,3826,weight=1)
G10.add_edge(3409,3521,weight=1)
G10.add_edge(3410,3411,weight=1)
G10.add_edge(3411,3412,weight=1)
G10.add_edge(3411,3413,weight=1)
G10.add_edge(3411,3415,weight=1)
G10.add_edge(3411,3419,weight=1)
G10.add_edge(3411,3420,weight=1)
G10.add_edge(3411,3628,weight=1)
G10.add_edge(3411,3668,weight=1)
G10.add_edge(3411,3674,weight=1)
G10.add_edge(3411,3703,weight=1)
G10.add_edge(3411,3724,weight=1)
G10.add_edge(3411,3725,weight=1)
G10.add_edge(3411,3726,weight=1)
G10.add_edge(3411,3727,weight=1)
G10.add_edge(3414,3415,weight=1)
G10.add_edge(3415,3462,weight=1)
G10.add_edge(3415,3672,weight=1)
G10.add_edge(3415,3698,weight=1)
G10.add_edge(3415,3723,weight=1)
G10.add_edge(3416,3417,weight=1)
G10.add_edge(3416,3418,weight=1)
G10.add_edge(3416,3419,weight=1)
G10.add_edge(3416,3420,weight=1)
G10.add_edge(3416,3421,weight=1)
G10.add_edge(3416,3422,weight=1)
G10.add_edge(3416,3423,weight=1)
G10.add_edge(3417,3505,weight=1)
G10.add_edge(3417,3538,weight=1)
G10.add_edge(3417,3539,weight=1)
G10.add_edge(3418,3679,weight=1)
G10.add_edge(3421,3467,weight=1)
G10.add_edge(3422,3506,weight=1)
G10.add_edge(3422,3678,weight=1)
G10.add_edge(3423,3463,weight=1)
G10.add_edge(3424,3425,weight=1)
G10.add_edge(3424,3426,weight=1)
G10.add_edge(3424,3427,weight=1)
G10.add_edge(3424,3428,weight=1)
G10.add_edge(3424,3429,weight=1)
G10.add_edge(3426,3508,weight=1)
G10.add_edge(3426,3633,weight=1)
G10.add_edge(3427,3816,weight=1)
G10.add_edge(3428,3509,weight=1)
G10.add_edge(3428,3785,weight=1)
G10.add_edge(3429,3845,weight=1)
G10.add_edge(3430,3431,weight=1)
G10.add_edge(3430,3432,weight=1)
G10.add_edge(3430,3433,weight=1)
G10.add_edge(3431,3722,weight=1)
G10.add_edge(3432,3719,weight=1)
G10.add_edge(3432,3721,weight=1)
G10.add_edge(3433,3515,weight=1)
G10.add_edge(3433,3836,weight=1)
G10.add_edge(3434,3580,weight=1)
G10.add_edge(3438,3440,weight=1)
G10.add_edge(3438,3670,weight=1)
G10.add_edge(3438,3695,weight=1)
G10.add_edge(3439,3440,weight=1)
G10.add_edge(3439,3441,weight=1)
G10.add_edge(3440,3442,weight=1)
G10.add_edge(3440,3443,weight=1)
G10.add_edge(3440,3444,weight=1)
G10.add_edge(3440,3445,weight=1)
G10.add_edge(3440,3446,weight=1)
G10.add_edge(3440,3447,weight=1)
G10.add_edge(3441,3608,weight=1)
G10.add_edge(3442,3446,weight=1)
G10.add_edge(3442,3483,weight=1)
G10.add_edge(3442,3484,weight=1)
G10.add_edge(3442,3485,weight=1)
G10.add_edge(3443,3444,weight=1)
G10.add_edge(3443,3457,weight=1)
G10.add_edge(3443,3519,weight=1)
G10.add_edge(3443,3520,weight=1)
G10.add_edge(3444,3501,weight=1)
G10.add_edge(3444,3554,weight=1)
G10.add_edge(3444,3555,weight=1)
G10.add_edge(3444,3559,weight=1)
G10.add_edge(3444,3560,weight=1)
G10.add_edge(3444,3561,weight=1)
G10.add_edge(3444,3562,weight=1)
G10.add_edge(3445,3504,weight=1)
G10.add_edge(3445,3584,weight=1)
G10.add_edge(3445,3586,weight=1)
G10.add_edge(3447,3491,weight=1)
G10.add_edge(3447,3755,weight=1)
G10.add_edge(3448,3449,weight=1)
G10.add_edge(3449,3816,weight=1)
G10.add_edge(3450,3491,weight=1)
G10.add_edge(3451,3452,weight=1)
G10.add_edge(3451,3453,weight=1)
G10.add_edge(3451,3454,weight=1)
G10.add_edge(3451,3455,weight=1)
G10.add_edge(3452,3536,weight=1)
G10.add_edge(3452,3572,weight=1)
G10.add_edge(3452,3573,weight=1)
G10.add_edge(3452,3574,weight=1)
G10.add_edge(3452,3575,weight=1)
G10.add_edge(3452,3576,weight=1)
G10.add_edge(3452,3577,weight=1)
G10.add_edge(3452,3578,weight=1)
G10.add_edge(3452,3579,weight=1)
G10.add_edge(3453,3612,weight=1)
G10.add_edge(3454,3641,weight=1)
G10.add_edge(3455,3537,weight=1)
G10.add_edge(3455,3639,weight=1)
G10.add_edge(3455,3640,weight=1)
G10.add_edge(3456,3457,weight=1)
G10.add_edge(3456,3458,weight=1)
G10.add_edge(3456,3459,weight=1)
G10.add_edge(3456,3460,weight=1)
G10.add_edge(3456,3461,weight=1)
G10.add_edge(3456,3462,weight=1)
G10.add_edge(3457,3463,weight=1)
G10.add_edge(3457,3464,weight=1)
G10.add_edge(3457,3465,weight=1)
G10.add_edge(3457,3466,weight=1)
G10.add_edge(3457,3467,weight=1)
G10.add_edge(3459,3733,weight=1)
G10.add_edge(3460,3732,weight=1)
G10.add_edge(3460,3776,weight=1)
G10.add_edge(3461,3770,weight=1)
G10.add_edge(3463,3503,weight=1)
G10.add_edge(3463,3563,weight=1)
G10.add_edge(3463,3564,weight=1)
G10.add_edge(3463,3565,weight=1)
G10.add_edge(3463,3566,weight=1)
G10.add_edge(3466,3560,weight=1)
G10.add_edge(3468,3469,weight=1)
G10.add_edge(3468,3470,weight=1)
G10.add_edge(3469,3728,weight=1)
G10.add_edge(3470,4887,weight=1)
G10.add_edge(3470,4906,weight=1)
G10.add_edge(3471,3472,weight=1)
G10.add_edge(3471,3473,weight=1)
G10.add_edge(3471,3474,weight=1)
G10.add_edge(3471,3475,weight=1)
G10.add_edge(3471,3476,weight=1)
G10.add_edge(3472,3643,weight=1)
G10.add_edge(3472,3667,weight=1)
G10.add_edge(3473,3669,weight=1)
G10.add_edge(3473,3713,weight=1)
G10.add_edge(3473,3744,weight=1)
G10.add_edge(3473,3745,weight=1)
G10.add_edge(3473,3747,weight=1)
G10.add_edge(3473,3751,weight=1)
G10.add_edge(3474,3535,weight=1)
G10.add_edge(3475,3535,weight=1)
G10.add_edge(3476,3659,weight=1)
G10.add_edge(3477,3478,weight=1)
G10.add_edge(3477,3479,weight=1)
G10.add_edge(3477,3480,weight=1)
G10.add_edge(3477,3481,weight=1)
G10.add_edge(3477,3482,weight=1)
G10.add_edge(3478,3543,weight=1)
G10.add_edge(3478,3563,weight=1)
G10.add_edge(3478,3589,weight=1)
G10.add_edge(3478,3590,weight=1)
G10.add_edge(3479,3626,weight=1)
G10.add_edge(3479,3781,weight=1)
G10.add_edge(3482,3544,weight=1)
G10.add_edge(3482,3823,weight=1)
G10.add_edge(3483,3498,weight=1)
G10.add_edge(3483,3501,weight=1)
G10.add_edge(3483,3502,weight=1)
G10.add_edge(3484,3559,weight=1)
G10.add_edge(3485,3561,weight=1)
G10.add_edge(3486,3487,weight=1)
G10.add_edge(3486,3488,weight=1)
G10.add_edge(3486,3489,weight=1)
G10.add_edge(3486,3490,weight=1)
G10.add_edge(3487,3491,weight=1)
G10.add_edge(3487,3492,weight=1)
G10.add_edge(3488,3714,weight=1)
G10.add_edge(3488,3716,weight=1)
G10.add_edge(3489,3839,weight=1)
G10.add_edge(3490,3512,weight=1)
G10.add_edge(3490,3716,weight=1)
G10.add_edge(3491,3508,weight=1)
G10.add_edge(3491,3511,weight=1)
G10.add_edge(3491,3512,weight=1)
G10.add_edge(3492,3541,weight=1)
G10.add_edge(3492,3542,weight=1)
G10.add_edge(3493,3494,weight=1)
G10.add_edge(3493,3495,weight=1)
G10.add_edge(3494,3516,weight=1)
G10.add_edge(3494,3530,weight=1)
G10.add_edge(3494,3531,weight=1)
G10.add_edge(3494,3534,weight=1)
G10.add_edge(3495,3570,weight=1)
G10.add_edge(3495,3571,weight=1)
G10.add_edge(3495,3693,weight=1)
G10.add_edge(3495,3764,weight=1)
G10.add_edge(3496,3692,weight=1)
G10.add_edge(3496,3807,weight=1)
G10.add_edge(3497,3804,weight=1)
G10.add_edge(3497,3805,weight=1)
G10.add_edge(3497,3809,weight=1)
G10.add_edge(3497,3810,weight=1)
G10.add_edge(3498,3499,weight=1)
G10.add_edge(3498,3500,weight=1)
G10.add_edge(3499,3848,weight=1)
G10.add_edge(3500,3624,weight=1)
G10.add_edge(3500,3704,weight=1)
G10.add_edge(3502,3752,weight=1)
G10.add_edge(3503,3579,weight=1)
G10.add_edge(3506,3752,weight=1)
G10.add_edge(3507,3565,weight=1)
G10.add_edge(3508,3509,weight=1)
G10.add_edge(3508,3510,weight=1)
G10.add_edge(3510,3690,weight=1)
G10.add_edge(3510,3846,weight=1)
G10.add_edge(3512,4865,weight=1)
G10.add_edge(3513,3514,weight=1)
G10.add_edge(3513,3515,weight=1)
G10.add_edge(3515,3580,weight=1)
G10.add_edge(3515,3582,weight=1)
G10.add_edge(3515,3827,weight=1)
G10.add_edge(3515,3829,weight=1)
G10.add_edge(3516,3533,weight=1)
G10.add_edge(3516,3551,weight=1)
G10.add_edge(3516,4559,weight=1)
G10.add_edge(3517,3784,weight=1)
G10.add_edge(3518,4656,weight=1)
G10.add_edge(3520,4555,weight=1)
G10.add_edge(3520,4557,weight=1)
G10.add_edge(3520,4731,weight=1)
G10.add_edge(3520,4750,weight=1)
G10.add_edge(3521,3522,weight=1)
G10.add_edge(3521,3523,weight=1)
G10.add_edge(3521,3524,weight=1)
G10.add_edge(3521,3525,weight=1)
G10.add_edge(3522,3545,weight=1)
G10.add_edge(3522,3552,weight=1)
G10.add_edge(3523,3689,weight=1)
G10.add_edge(3523,3690,weight=1)
G10.add_edge(3524,3817,weight=1)
G10.add_edge(3525,4874,weight=1)
G10.add_edge(3526,3527,weight=1)
G10.add_edge(3527,3828,weight=1)
G10.add_edge(3528,3529,weight=1)
G10.add_edge(3529,3835,weight=1)
G10.add_edge(3531,3532,weight=1)
G10.add_edge(3531,3533,weight=1)
G10.add_edge(3532,3693,weight=1)
G10.add_edge(3533,3601,weight=1)
G10.add_edge(3534,3670,weight=1)
G10.add_edge(3535,3536,weight=1)
G10.add_edge(3535,3537,weight=1)
G10.add_edge(3535,3538,weight=1)
G10.add_edge(3535,3539,weight=1)
G10.add_edge(3535,3540,weight=1)
G10.add_edge(3540,3674,weight=1)
G10.add_edge(3541,3651,weight=1)
G10.add_edge(3541,3652,weight=1)
G10.add_edge(3542,4867,weight=1)
G10.add_edge(3542,4891,weight=1)
G10.add_edge(3542,4921,weight=1)
G10.add_edge(3543,3544,weight=1)
G10.add_edge(3544,3638,weight=1)
G10.add_edge(3545,3546,weight=1)
G10.add_edge(3545,3547,weight=1)
G10.add_edge(3545,3548,weight=1)
G10.add_edge(3545,3549,weight=1)
G10.add_edge(3545,3550,weight=1)
G10.add_edge(3546,3551,weight=1)
G10.add_edge(3547,3646,weight=1)
G10.add_edge(3548,3648,weight=1)
G10.add_edge(3548,3655,weight=1)
G10.add_edge(3548,3837,weight=1)
G10.add_edge(3549,3832,weight=1)
G10.add_edge(3550,3844,weight=1)
G10.add_edge(3551,4889,weight=1)
G10.add_edge(3552,3654,weight=1)
G10.add_edge(3552,3797,weight=1)
G10.add_edge(3553,3554,weight=1)
G10.add_edge(3553,3555,weight=1)
G10.add_edge(3553,3556,weight=1)
G10.add_edge(3553,3557,weight=1)
G10.add_edge(3553,3558,weight=1)
G10.add_edge(3557,3766,weight=1)
G10.add_edge(3558,3766,weight=1)
G10.add_edge(3562,3777,weight=1)
G10.add_edge(3563,3591,weight=1)
G10.add_edge(3564,3592,weight=1)
G10.add_edge(3565,3795,weight=1)
G10.add_edge(3566,4582,weight=1)
G10.add_edge(3566,4628,weight=1)
G10.add_edge(3567,3568,weight=1)
G10.add_edge(3567,3569,weight=1)
G10.add_edge(3567,3570,weight=1)
G10.add_edge(3567,3571,weight=1)
G10.add_edge(3568,3682,weight=1)
G10.add_edge(3568,3683,weight=1)
G10.add_edge(3569,3686,weight=1)
G10.add_edge(3569,3695,weight=1)
G10.add_edge(3569,3734,weight=1)
G10.add_edge(3569,3735,weight=1)
G10.add_edge(3570,3763,weight=1)
G10.add_edge(3573,3615,weight=1)
G10.add_edge(3573,3691,weight=1)
G10.add_edge(3574,3696,weight=1)
G10.add_edge(3575,3699,weight=1)
G10.add_edge(3576,3772,weight=1)
G10.add_edge(3576,3806,weight=1)
G10.add_edge(3580,3581,weight=1)
G10.add_edge(3580,3582,weight=1)
G10.add_edge(3580,3583,weight=1)
G10.add_edge(3581,3700,weight=1)
G10.add_edge(3581,3812,weight=1)
G10.add_edge(3582,3827,weight=1)
G10.add_edge(3582,3842,weight=1)
G10.add_edge(3583,3842,weight=1)
G10.add_edge(3584,3585,weight=1)
G10.add_edge(3584,3586,weight=1)
G10.add_edge(3584,3587,weight=1)
G10.add_edge(3584,3588,weight=1)
G10.add_edge(3587,3811,weight=1)
G10.add_edge(3587,3814,weight=1)
G10.add_edge(3587,3838,weight=1)
G10.add_edge(3588,3739,weight=1)
G10.add_edge(3588,3815,weight=1)
G10.add_edge(3590,3591,weight=1)
G10.add_edge(3590,4709,weight=1)
G10.add_edge(3590,4714,weight=1)
G10.add_edge(3591,3592,weight=1)
G10.add_edge(3592,3835,weight=1)
G10.add_edge(3592,4544,weight=1)
G10.add_edge(3592,4551,weight=1)
G10.add_edge(3592,4552,weight=1)
G10.add_edge(3593,3594,weight=1)
G10.add_edge(3593,3595,weight=1)
G10.add_edge(3595,4577,weight=1)
G10.add_edge(3595,4619,weight=1)
G10.add_edge(3596,3597,weight=1)
G10.add_edge(3596,3598,weight=1)
G10.add_edge(3597,3800,weight=1)
G10.add_edge(3598,4635,weight=1)
G10.add_edge(3598,4725,weight=1)
G10.add_edge(3598,4726,weight=1)
G10.add_edge(3598,4727,weight=1)
G10.add_edge(3599,3600,weight=1)
G10.add_edge(3600,3601,weight=1)
G10.add_edge(3600,3602,weight=1)
G10.add_edge(3602,3603,weight=1)
G10.add_edge(3603,3688,weight=1)
G10.add_edge(3604,3605,weight=1)
G10.add_edge(3604,3606,weight=1)
G10.add_edge(3604,3607,weight=1)
G10.add_edge(3604,3608,weight=1)
G10.add_edge(3606,3694,weight=1)
G10.add_edge(3606,3695,weight=1)
G10.add_edge(3607,3687,weight=1)
G10.add_edge(3608,3756,weight=1)
G10.add_edge(3609,3610,weight=1)
G10.add_edge(3610,3632,weight=1)
G10.add_edge(3610,3791,weight=1)
G10.add_edge(3610,4872,weight=1)
G10.add_edge(3610,4878,weight=1)
G10.add_edge(3610,4906,weight=1)
G10.add_edge(3610,4917,weight=1)
G10.add_edge(3610,4918,weight=1)
G10.add_edge(3610,4919,weight=1)
G10.add_edge(3611,3612,weight=1)
G10.add_edge(3612,3613,weight=1)
G10.add_edge(3612,3614,weight=1)
G10.add_edge(3612,3675,weight=1)
G10.add_edge(3612,3676,weight=1)
G10.add_edge(3612,3691,weight=1)
G10.add_edge(3616,3617,weight=1)
G10.add_edge(3617,3730,weight=1)
G10.add_edge(3617,3731,weight=1)
G10.add_edge(3618,3619,weight=1)
G10.add_edge(3618,3620,weight=1)
G10.add_edge(3618,3621,weight=1)
G10.add_edge(3621,4525,weight=1)
G10.add_edge(3621,4555,weight=1)
G10.add_edge(3622,3623,weight=1)
G10.add_edge(3623,3629,weight=1)
G10.add_edge(3623,4892,weight=1)
G10.add_edge(3623,4893,weight=1)
G10.add_edge(3623,4894,weight=1)
G10.add_edge(3625,3626,weight=1)
G10.add_edge(3626,3801,weight=1)
G10.add_edge(3627,3628,weight=1)
G10.add_edge(3628,3712,weight=1)
G10.add_edge(3630,3631,weight=1)
G10.add_edge(3631,3705,weight=1)
G10.add_edge(3631,3706,weight=1)
G10.add_edge(3631,3707,weight=1)
G10.add_edge(3634,3635,weight=1)
G10.add_edge(3635,3845,weight=1)
G10.add_edge(3636,3637,weight=1)
G10.add_edge(3637,3691,weight=1)
G10.add_edge(3642,3643,weight=1)
G10.add_edge(3642,3644,weight=1)
G10.add_edge(3643,3644,weight=1)
G10.add_edge(3644,3731,weight=1)
G10.add_edge(3644,3739,weight=1)
G10.add_edge(3645,3646,weight=1)
G10.add_edge(3645,3647,weight=1)
G10.add_edge(3645,3648,weight=1)
G10.add_edge(3646,3649,weight=1)
G10.add_edge(3646,3650,weight=1)
G10.add_edge(3648,3798,weight=1)
G10.add_edge(3649,3797,weight=1)
G10.add_edge(3650,3657,weight=1)
G10.add_edge(3651,3652,weight=1)
G10.add_edge(3653,3654,weight=1)
G10.add_edge(3653,3655,weight=1)
G10.add_edge(3654,3717,weight=1)
G10.add_edge(3654,3718,weight=1)
G10.add_edge(3656,3657,weight=1)
G10.add_edge(3658,3659,weight=1)
G10.add_edge(3659,3660,weight=1)
G10.add_edge(3659,3663,weight=1)
G10.add_edge(3659,3664,weight=1)
G10.add_edge(3663,3769,weight=1)
G10.add_edge(3664,3848,weight=1)
G10.add_edge(3665,3767,weight=1)
G10.add_edge(3665,3768,weight=1)
G10.add_edge(3668,3669,weight=1)
G10.add_edge(3671,3672,weight=1)
G10.add_edge(3671,3673,weight=1)
G10.add_edge(3673,3766,weight=1)
G10.add_edge(3675,3676,weight=1)
G10.add_edge(3675,3677,weight=1)
G10.add_edge(3679,3777,weight=1)
G10.add_edge(3680,3681,weight=1)
G10.add_edge(3683,3684,weight=1)
G10.add_edge(3683,3685,weight=1)
G10.add_edge(3683,3686,weight=1)
G10.add_edge(3685,3765,weight=1)
G10.add_edge(3687,3688,weight=1)
G10.add_edge(3694,3695,weight=1)
G10.add_edge(3696,3725,weight=1)
G10.add_edge(3697,3698,weight=1)
G10.add_edge(3697,3699,weight=1)
G10.add_edge(3701,3702,weight=1)
G10.add_edge(3701,3703,weight=1)
G10.add_edge(3707,3737,weight=1)
G10.add_edge(3707,3849,weight=1)
G10.add_edge(3708,3709,weight=1)
G10.add_edge(3708,3710,weight=1)
G10.add_edge(3709,3737,weight=1)
G10.add_edge(3709,3793,weight=1)
G10.add_edge(3709,3794,weight=1)
G10.add_edge(3710,3729,weight=1)
G10.add_edge(3711,3712,weight=1)
G10.add_edge(3714,3715,weight=1)
G10.add_edge(3720,3721,weight=1)
G10.add_edge(3721,3722,weight=1)
G10.add_edge(3723,3753,weight=1)
G10.add_edge(3726,3813,weight=1)
G10.add_edge(3727,3813,weight=1)
G10.add_edge(3728,3792,weight=1)
G10.add_edge(3733,3771,weight=1)
G10.add_edge(3734,3736,weight=1)
G10.add_edge(3735,3764,weight=1)
G10.add_edge(3736,3840,weight=1)
G10.add_edge(3736,3841,weight=1)
G10.add_edge(3738,3739,weight=1)
G10.add_edge(3740,3741,weight=1)
G10.add_edge(3741,3797,weight=1)
G10.add_edge(3742,3743,weight=1)
G10.add_edge(3742,3744,weight=1)
G10.add_edge(3742,3745,weight=1)
G10.add_edge(3744,3746,weight=1)
G10.add_edge(3745,3749,weight=1)
G10.add_edge(3746,3747,weight=1)
G10.add_edge(3746,3748,weight=1)
G10.add_edge(3747,3749,weight=1)
G10.add_edge(3749,3750,weight=1)
G10.add_edge(3751,3773,weight=1)
G10.add_edge(3753,3766,weight=1)
G10.add_edge(3754,3755,weight=1)
G10.add_edge(3754,3756,weight=1)
G10.add_edge(3758,3759,weight=1)
G10.add_edge(3759,3764,weight=1)
G10.add_edge(3760,3761,weight=1)
G10.add_edge(3761,3763,weight=1)
G10.add_edge(3762,3763,weight=1)
G10.add_edge(3763,3764,weight=1)
G10.add_edge(3763,3765,weight=1)
G10.add_edge(3768,3811,weight=1)
G10.add_edge(3771,3772,weight=1)
G10.add_edge(3774,3775,weight=1)
G10.add_edge(3774,3776,weight=1)
G10.add_edge(3778,3779,weight=1)
G10.add_edge(3779,3796,weight=1)
G10.add_edge(3779,3847,weight=1)
G10.add_edge(3780,3781,weight=1)
G10.add_edge(3782,3783,weight=1)
G10.add_edge(3786,3787,weight=1)
G10.add_edge(3786,3788,weight=1)
G10.add_edge(3786,3789,weight=1)
G10.add_edge(3787,3821,weight=1)
G10.add_edge(3789,3839,weight=1)
G10.add_edge(3790,3791,weight=1)
G10.add_edge(3791,4904,weight=1)
G10.add_edge(3794,3825,weight=1)
G10.add_edge(3798,3799,weight=1)
G10.add_edge(3799,3819,weight=1)
G10.add_edge(3799,3822,weight=1)
G10.add_edge(3802,3803,weight=1)
G10.add_edge(3803,3817,weight=1)
G10.add_edge(3803,3818,weight=1)
G10.add_edge(3807,3808,weight=1)
G10.add_edge(3819,3820,weight=1)
G10.add_edge(3821,3824,weight=1)
G10.add_edge(3824,3825,weight=1)
G10.add_edge(3825,3834,weight=1)
G10.add_edge(3828,3829,weight=1)
G10.add_edge(3831,3832,weight=1)
G10.add_edge(3833,3834,weight=1)
G10.add_edge(3834,4864,weight=1)
G10.add_edge(3834,4905,weight=1)
G10.add_edge(3834,4921,weight=1)
G10.add_edge(3835,4552,weight=1)
G10.add_edge(3835,4725,weight=1)
G10.add_edge(3843,3844,weight=1)
G10.add_edge(3850,3851,weight=1)
G10.add_edge(3850,3852,weight=1)
G10.add_edge(3851,3864,weight=1)
G10.add_edge(3851,3891,weight=1)
G10.add_edge(3852,3854,weight=1)
G10.add_edge(3853,3854,weight=1)
G10.add_edge(3853,3855,weight=1)
G10.add_edge(3853,3856,weight=1)
G10.add_edge(3854,3857,weight=1)
G10.add_edge(3854,3877,weight=1)
G10.add_edge(3854,3878,weight=1)
G10.add_edge(3854,3879,weight=1)
G10.add_edge(3854,3880,weight=1)
G10.add_edge(3855,3888,weight=1)
G10.add_edge(3856,3857,weight=1)
G10.add_edge(3856,3934,weight=1)
G10.add_edge(3856,3935,weight=1)
G10.add_edge(3858,3859,weight=1)
G10.add_edge(3858,3860,weight=1)
G10.add_edge(3858,3861,weight=1)
G10.add_edge(3859,3898,weight=1)
G10.add_edge(3860,3936,weight=1)
G10.add_edge(3860,3951,weight=1)
G10.add_edge(3861,3936,weight=1)
G10.add_edge(3862,3863,weight=1)
G10.add_edge(3862,3864,weight=1)
G10.add_edge(3862,3865,weight=1)
G10.add_edge(3863,3866,weight=1)
G10.add_edge(3865,3899,weight=1)
G10.add_edge(3865,3900,weight=1)
G10.add_edge(3866,3876,weight=1)
G10.add_edge(3866,3895,weight=1)
G10.add_edge(3866,3899,weight=1)
G10.add_edge(3866,3915,weight=1)
G10.add_edge(3867,3868,weight=1)
G10.add_edge(3868,3869,weight=1)
G10.add_edge(3868,3870,weight=1)
G10.add_edge(3868,3871,weight=1)
G10.add_edge(3869,3872,weight=1)
G10.add_edge(3869,3901,weight=1)
G10.add_edge(3869,3906,weight=1)
G10.add_edge(3869,3932,weight=1)
G10.add_edge(3869,3933,weight=1)
G10.add_edge(3870,3871,weight=1)
G10.add_edge(3870,3904,weight=1)
G10.add_edge(3872,3873,weight=1)
G10.add_edge(3872,3874,weight=1)
G10.add_edge(3874,4037,weight=1)
G10.add_edge(3875,3876,weight=1)
G10.add_edge(3876,3877,weight=1)
G10.add_edge(3877,3880,weight=1)
G10.add_edge(3877,3930,weight=1)
G10.add_edge(3881,3882,weight=1)
G10.add_edge(3881,3883,weight=1)
G10.add_edge(3881,3884,weight=1)
G10.add_edge(3881,3885,weight=1)
G10.add_edge(3881,3886,weight=1)
G10.add_edge(3883,3887,weight=1)
G10.add_edge(3883,3888,weight=1)
G10.add_edge(3883,3889,weight=1)
G10.add_edge(3884,3923,weight=1)
G10.add_edge(3884,3930,weight=1)
G10.add_edge(3886,3915,weight=1)
G10.add_edge(3886,3924,weight=1)
G10.add_edge(3886,3950,weight=1)
G10.add_edge(3886,4226,weight=1)
G10.add_edge(3886,4227,weight=1)
G10.add_edge(3886,4261,weight=1)
G10.add_edge(3886,4354,weight=1)
G10.add_edge(3886,4355,weight=1)
G10.add_edge(3887,3949,weight=1)
G10.add_edge(3888,3890,weight=1)
G10.add_edge(3888,3955,weight=1)
G10.add_edge(3888,3957,weight=1)
G10.add_edge(3888,3968,weight=1)
G10.add_edge(3888,4003,weight=1)
G10.add_edge(3888,4004,weight=1)
G10.add_edge(3888,4005,weight=1)
G10.add_edge(3888,4006,weight=1)
G10.add_edge(3889,3950,weight=1)
G10.add_edge(3889,4033,weight=1)
G10.add_edge(3890,3891,weight=1)
G10.add_edge(3891,4045,weight=1)
G10.add_edge(3892,3917,weight=1)
G10.add_edge(3893,3894,weight=1)
G10.add_edge(3893,3895,weight=1)
G10.add_edge(3893,3896,weight=1)
G10.add_edge(3893,3897,weight=1)
G10.add_edge(3895,3915,weight=1)
G10.add_edge(3895,3918,weight=1)
G10.add_edge(3895,3919,weight=1)
G10.add_edge(3895,3920,weight=1)
G10.add_edge(3895,3921,weight=1)
G10.add_edge(3895,3922,weight=1)
G10.add_edge(3896,3986,weight=1)
G10.add_edge(3896,3987,weight=1)
G10.add_edge(3897,4011,weight=1)
G10.add_edge(3897,4012,weight=1)
G10.add_edge(3898,3934,weight=1)
G10.add_edge(3898,3936,weight=1)
G10.add_edge(3899,3902,weight=1)
G10.add_edge(3899,3903,weight=1)
G10.add_edge(3899,3931,weight=1)
G10.add_edge(3899,3937,weight=1)
G10.add_edge(3899,3945,weight=1)
G10.add_edge(3900,3909,weight=1)
G10.add_edge(3900,3953,weight=1)
G10.add_edge(3900,3954,weight=1)
G10.add_edge(3900,3988,weight=1)
G10.add_edge(3900,3999,weight=1)
G10.add_edge(3900,4002,weight=1)
G10.add_edge(3901,3902,weight=1)
G10.add_edge(3901,3903,weight=1)
G10.add_edge(3901,3904,weight=1)
G10.add_edge(3901,3905,weight=1)
G10.add_edge(3901,3906,weight=1)
G10.add_edge(3904,3905,weight=1)
G10.add_edge(3904,3926,weight=1)
G10.add_edge(3904,3962,weight=1)
G10.add_edge(3906,3985,weight=1)
G10.add_edge(3907,3908,weight=1)
G10.add_edge(3907,3909,weight=1)
G10.add_edge(3907,3910,weight=1)
G10.add_edge(3907,3911,weight=1)
G10.add_edge(3907,3912,weight=1)
G10.add_edge(3908,3976,weight=1)
G10.add_edge(3909,4001,weight=1)
G10.add_edge(3910,3984,weight=1)
G10.add_edge(3910,3999,weight=1)
G10.add_edge(3911,3943,weight=1)
G10.add_edge(3911,4028,weight=1)
G10.add_edge(3912,3982,weight=1)
G10.add_edge(3912,4028,weight=1)
G10.add_edge(3913,3914,weight=1)
G10.add_edge(3914,3931,weight=1)
G10.add_edge(3915,3916,weight=1)
G10.add_edge(3915,3923,weight=1)
G10.add_edge(3915,3924,weight=1)
G10.add_edge(3916,3946,weight=1)
G10.add_edge(3916,3947,weight=1)
G10.add_edge(3916,3948,weight=1)
G10.add_edge(3917,4027,weight=1)
G10.add_edge(3918,3949,weight=1)
G10.add_edge(3918,4002,weight=1)
G10.add_edge(3920,3986,weight=1)
G10.add_edge(3921,4012,weight=1)
G10.add_edge(3922,3999,weight=1)
G10.add_edge(3923,3925,weight=1)
G10.add_edge(3924,3935,weight=1)
G10.add_edge(3925,3926,weight=1)
G10.add_edge(3925,3927,weight=1)
G10.add_edge(3925,3928,weight=1)
G10.add_edge(3925,3929,weight=1)
G10.add_edge(3928,4022,weight=1)
G10.add_edge(3929,3981,weight=1)
G10.add_edge(3930,3992,weight=1)
G10.add_edge(3930,3993,weight=1)
G10.add_edge(3930,3994,weight=1)
G10.add_edge(3930,3995,weight=1)
G10.add_edge(3930,3998,weight=1)
G10.add_edge(3930,4009,weight=1)
G10.add_edge(3930,4013,weight=1)
G10.add_edge(3930,4014,weight=1)
G10.add_edge(3930,4015,weight=1)
G10.add_edge(3930,4026,weight=1)
G10.add_edge(3930,4030,weight=1)
G10.add_edge(3933,3982,weight=1)
G10.add_edge(3933,3983,weight=1)
G10.add_edge(3935,3961,weight=1)
G10.add_edge(3935,4024,weight=1)
G10.add_edge(3935,4025,weight=1)
G10.add_edge(3935,4032,weight=1)
G10.add_edge(3935,4038,weight=1)
G10.add_edge(3935,4039,weight=1)
G10.add_edge(3936,3950,weight=1)
G10.add_edge(3936,3951,weight=1)
G10.add_edge(3936,3952,weight=1)
G10.add_edge(3937,3938,weight=1)
G10.add_edge(3937,3939,weight=1)
G10.add_edge(3937,3940,weight=1)
G10.add_edge(3937,3941,weight=1)
G10.add_edge(3937,3942,weight=1)
G10.add_edge(3937,3943,weight=1)
G10.add_edge(3937,3944,weight=1)
G10.add_edge(3939,4019,weight=1)
G10.add_edge(3940,3941,weight=1)
G10.add_edge(3941,3943,weight=1)
G10.add_edge(3942,3978,weight=1)
G10.add_edge(3942,4019,weight=1)
G10.add_edge(3943,3983,weight=1)
G10.add_edge(3943,4036,weight=1)
G10.add_edge(3949,3970,weight=1)
G10.add_edge(3949,3988,weight=1)
G10.add_edge(3949,4029,weight=1)
G10.add_edge(3949,4042,weight=1)
G10.add_edge(3952,3989,weight=1)
G10.add_edge(3953,3954,weight=1)
G10.add_edge(3954,3964,weight=1)
G10.add_edge(3955,3956,weight=1)
G10.add_edge(3956,3957,weight=1)
G10.add_edge(3956,3958,weight=1)
G10.add_edge(3957,3969,weight=1)
G10.add_edge(3958,4005,weight=1)
G10.add_edge(3958,4034,weight=1)
G10.add_edge(3959,3960,weight=1)
G10.add_edge(3959,3961,weight=1)
G10.add_edge(3960,4032,weight=1)
G10.add_edge(3960,4033,weight=1)
G10.add_edge(3963,3964,weight=1)
G10.add_edge(3963,3965,weight=1)
G10.add_edge(3964,3966,weight=1)
G10.add_edge(3965,3973,weight=1)
G10.add_edge(3966,3971,weight=1)
G10.add_edge(3966,3975,weight=1)
G10.add_edge(3967,3968,weight=1)
G10.add_edge(3967,3969,weight=1)
G10.add_edge(3967,3970,weight=1)
G10.add_edge(3969,4004,weight=1)
G10.add_edge(3970,4041,weight=1)
G10.add_edge(3971,3972,weight=1)
G10.add_edge(3972,3988,weight=1)
G10.add_edge(3972,4034,weight=1)
G10.add_edge(3973,3974,weight=1)
G10.add_edge(3973,3975,weight=1)
G10.add_edge(3976,3977,weight=1)
G10.add_edge(3976,3978,weight=1)
G10.add_edge(3978,4040,weight=1)
G10.add_edge(3979,3980,weight=1)
G10.add_edge(3980,4034,weight=1)
G10.add_edge(3982,3984,weight=1)
G10.add_edge(3982,3985,weight=1)
G10.add_edge(3983,4035,weight=1)
G10.add_edge(3987,4015,weight=1)
G10.add_edge(3987,4021,weight=1)
G10.add_edge(3987,4022,weight=1)
G10.add_edge(3987,4030,weight=1)
G10.add_edge(3989,3990,weight=1)
G10.add_edge(3989,3991,weight=1)
G10.add_edge(3991,4017,weight=1)
G10.add_edge(3992,3993,weight=1)
G10.add_edge(3992,3994,weight=1)
G10.add_edge(3992,3995,weight=1)
G10.add_edge(3993,3996,weight=1)
G10.add_edge(3995,4043,weight=1)
G10.add_edge(3996,3997,weight=1)
G10.add_edge(3997,3998,weight=1)
G10.add_edge(3999,4000,weight=1)
G10.add_edge(3999,4001,weight=1)
G10.add_edge(4003,4007,weight=1)
G10.add_edge(4006,4025,weight=1)
G10.add_edge(4007,4041,weight=1)
G10.add_edge(4007,4044,weight=1)
G10.add_edge(4008,4009,weight=1)
G10.add_edge(4008,4010,weight=1)
G10.add_edge(4009,4026,weight=1)
G10.add_edge(4009,4029,weight=1)
G10.add_edge(4010,4030,weight=1)
G10.add_edge(4010,4031,weight=1)
G10.add_edge(4012,4020,weight=1)
G10.add_edge(4013,4014,weight=1)
G10.add_edge(4013,4015,weight=1)
G10.add_edge(4016,4017,weight=1)
G10.add_edge(4017,4018,weight=1)
G10.add_edge(4018,4023,weight=1)
G10.add_edge(4019,4040,weight=1)
G10.add_edge(4021,4022,weight=1)
G10.add_edge(4023,4039,weight=1)
G10.add_edge(4024,4025,weight=1)
G10.add_edge(4035,4036,weight=1)
G10.add_edge(4036,4037,weight=1)
G10.add_edge(4038,4039,weight=1)
G10.add_edge(4041,4042,weight=1)
G10.add_edge(4046,4047,weight=1)
G10.add_edge(4047,4065,weight=1)
G10.add_edge(4047,4106,weight=1)
G10.add_edge(4048,4049,weight=1)
G10.add_edge(4048,4050,weight=1)
G10.add_edge(4048,4051,weight=1)
G10.add_edge(4049,4072,weight=1)
G10.add_edge(4049,4073,weight=1)
G10.add_edge(4050,4075,weight=1)
G10.add_edge(4051,4104,weight=1)
G10.add_edge(4051,4136,weight=1)
G10.add_edge(4052,4053,weight=1)
G10.add_edge(4052,4054,weight=1)
G10.add_edge(4053,4055,weight=1)
G10.add_edge(4054,4132,weight=1)
G10.add_edge(4055,4056,weight=1)
G10.add_edge(4055,4109,weight=1)
G10.add_edge(4056,4057,weight=1)
G10.add_edge(4057,4075,weight=1)
G10.add_edge(4058,4059,weight=1)
G10.add_edge(4059,4095,weight=1)
G10.add_edge(4059,4138,weight=1)
G10.add_edge(4060,4061,weight=1)
G10.add_edge(4060,4062,weight=1)
G10.add_edge(4061,4062,weight=1)
G10.add_edge(4061,4114,weight=1)
G10.add_edge(4062,4102,weight=1)
G10.add_edge(4062,4112,weight=1)
G10.add_edge(4062,4133,weight=1)
G10.add_edge(4063,4064,weight=1)
G10.add_edge(4063,4065,weight=1)
G10.add_edge(4063,4066,weight=1)
G10.add_edge(4063,4067,weight=1)
G10.add_edge(4064,4067,weight=1)
G10.add_edge(4064,4094,weight=1)
G10.add_edge(4064,4095,weight=1)
G10.add_edge(4064,4096,weight=1)
G10.add_edge(4065,4130,weight=1)
G10.add_edge(4066,4167,weight=1)
G10.add_edge(4068,4069,weight=1)
G10.add_edge(4068,4070,weight=1)
G10.add_edge(4068,4071,weight=1)
G10.add_edge(4069,4097,weight=1)
G10.add_edge(4069,4098,weight=1)
G10.add_edge(4069,4099,weight=1)
G10.add_edge(4070,4164,weight=1)
G10.add_edge(4071,4137,weight=1)
G10.add_edge(4072,4074,weight=1)
G10.add_edge(4072,4075,weight=1)
G10.add_edge(4072,4076,weight=1)
G10.add_edge(4072,4077,weight=1)
G10.add_edge(4072,4078,weight=1)
G10.add_edge(4073,4081,weight=1)
G10.add_edge(4074,4092,weight=1)
G10.add_edge(4074,4093,weight=1)
G10.add_edge(4075,4080,weight=1)
G10.add_edge(4075,4110,weight=1)
G10.add_edge(4075,4111,weight=1)
G10.add_edge(4075,4112,weight=1)
G10.add_edge(4075,4113,weight=1)
G10.add_edge(4077,4080,weight=1)
G10.add_edge(4078,4102,weight=1)
G10.add_edge(4078,4134,weight=1)
G10.add_edge(4079,4080,weight=1)
G10.add_edge(4079,4081,weight=1)
G10.add_edge(4079,4082,weight=1)
G10.add_edge(4080,4083,weight=1)
G10.add_edge(4080,4084,weight=1)
G10.add_edge(4082,4131,weight=1)
G10.add_edge(4083,4131,weight=1)
G10.add_edge(4084,4145,weight=1)
G10.add_edge(4085,4086,weight=1)
G10.add_edge(4085,4087,weight=1)
G10.add_edge(4086,4088,weight=1)
G10.add_edge(4086,4090,weight=1)
G10.add_edge(4086,4091,weight=1)
G10.add_edge(4087,4124,weight=1)
G10.add_edge(4087,4128,weight=1)
G10.add_edge(4087,4154,weight=1)
G10.add_edge(4088,4089,weight=1)
G10.add_edge(4089,4142,weight=1)
G10.add_edge(4089,4158,weight=1)
G10.add_edge(4090,4135,weight=1)
G10.add_edge(4091,4093,weight=1)
G10.add_edge(4091,4144,weight=1)
G10.add_edge(4091,4152,weight=1)
G10.add_edge(4093,4112,weight=1)
G10.add_edge(4094,4106,weight=1)
G10.add_edge(4094,4117,weight=1)
G10.add_edge(4095,4134,weight=1)
G10.add_edge(4095,4155,weight=1)
G10.add_edge(4095,4156,weight=1)
G10.add_edge(4095,4157,weight=1)
G10.add_edge(4096,4166,weight=1)
G10.add_edge(4099,4100,weight=1)
G10.add_edge(4099,4102,weight=1)
G10.add_edge(4099,4103,weight=1)
G10.add_edge(4100,4101,weight=1)
G10.add_edge(4101,4104,weight=1)
G10.add_edge(4101,4105,weight=1)
G10.add_edge(4102,4136,weight=1)
G10.add_edge(4102,4146,weight=1)
G10.add_edge(4102,4151,weight=1)
G10.add_edge(4102,4152,weight=1)
G10.add_edge(4103,4170,weight=1)
G10.add_edge(4104,4125,weight=1)
G10.add_edge(4104,4126,weight=1)
G10.add_edge(4105,4136,weight=1)
G10.add_edge(4107,4108,weight=1)
G10.add_edge(4108,4117,weight=1)
G10.add_edge(4108,4174,weight=1)
G10.add_edge(4109,4129,weight=1)
G10.add_edge(4112,4149,weight=1)
G10.add_edge(4114,4115,weight=1)
G10.add_edge(4115,4135,weight=1)
G10.add_edge(4115,4141,weight=1)
G10.add_edge(4115,4146,weight=1)
G10.add_edge(4115,4152,weight=1)
G10.add_edge(4116,4129,weight=1)
G10.add_edge(4118,4119,weight=1)
G10.add_edge(4118,4120,weight=1)
G10.add_edge(4119,4137,weight=1)
G10.add_edge(4120,4138,weight=1)
G10.add_edge(4121,4122,weight=1)
G10.add_edge(4121,4123,weight=1)
G10.add_edge(4122,4124,weight=1)
G10.add_edge(4123,4147,weight=1)
G10.add_edge(4127,4128,weight=1)
G10.add_edge(4128,4147,weight=1)
G10.add_edge(4128,4150,weight=1)
G10.add_edge(4130,4157,weight=1)
G10.add_edge(4130,4166,weight=1)
G10.add_edge(4132,4159,weight=1)
G10.add_edge(4133,4134,weight=1)
G10.add_edge(4134,4153,weight=1)
G10.add_edge(4135,4144,weight=1)
G10.add_edge(4135,4152,weight=1)
G10.add_edge(4135,4153,weight=1)
G10.add_edge(4137,4138,weight=1)
G10.add_edge(4139,4140,weight=1)
G10.add_edge(4139,4141,weight=1)
G10.add_edge(4139,4142,weight=1)
G10.add_edge(4139,4143,weight=1)
G10.add_edge(4140,4141,weight=1)
G10.add_edge(4143,4152,weight=1)
G10.add_edge(4147,4148,weight=1)
G10.add_edge(4148,4154,weight=1)
G10.add_edge(4159,4160,weight=1)
G10.add_edge(4160,4161,weight=1)
G10.add_edge(4161,4162,weight=1)
G10.add_edge(4161,4163,weight=1)
G10.add_edge(4162,4169,weight=1)
G10.add_edge(4163,4169,weight=1)
G10.add_edge(4165,4166,weight=1)
G10.add_edge(4165,4167,weight=1)
G10.add_edge(4166,4168,weight=1)
G10.add_edge(4169,4206,weight=1)
G10.add_edge(4169,4208,weight=1)
G10.add_edge(4170,4171,weight=1)
G10.add_edge(4170,4172,weight=1)
G10.add_edge(4170,4173,weight=1)
G10.add_edge(4175,4176,weight=1)
G10.add_edge(4176,4178,weight=1)
G10.add_edge(4176,4180,weight=1)
G10.add_edge(4176,4181,weight=1)
G10.add_edge(4176,4182,weight=1)
G10.add_edge(4177,4178,weight=1)
G10.add_edge(4178,4179,weight=1)
G10.add_edge(4178,4180,weight=1)
G10.add_edge(4180,4192,weight=1)
G10.add_edge(4180,4193,weight=1)
G10.add_edge(4180,4194,weight=1)
G10.add_edge(4182,4239,weight=1)
G10.add_edge(4182,4279,weight=1)
G10.add_edge(4182,4301,weight=1)
G10.add_edge(4182,4315,weight=1)
G10.add_edge(4183,4184,weight=1)
G10.add_edge(4183,4185,weight=1)
G10.add_edge(4183,4186,weight=1)
G10.add_edge(4184,4191,weight=1)
G10.add_edge(4184,4199,weight=1)
G10.add_edge(4184,4322,weight=1)
G10.add_edge(4184,4323,weight=1)
G10.add_edge(4185,4190,weight=1)
G10.add_edge(4185,4306,weight=1)
G10.add_edge(4185,4364,weight=1)
G10.add_edge(4185,4375,weight=1)
G10.add_edge(4187,4188,weight=1)
G10.add_edge(4187,4189,weight=1)
G10.add_edge(4188,4190,weight=1)
G10.add_edge(4188,4191,weight=1)
G10.add_edge(4189,4398,weight=1)
G10.add_edge(4189,4399,weight=1)
G10.add_edge(4190,4191,weight=1)
G10.add_edge(4190,4249,weight=1)
G10.add_edge(4190,4274,weight=1)
G10.add_edge(4190,4332,weight=1)
G10.add_edge(4190,4344,weight=1)
G10.add_edge(4191,4345,weight=1)
G10.add_edge(4192,4220,weight=1)
G10.add_edge(4192,4259,weight=1)
G10.add_edge(4192,4260,weight=1)
G10.add_edge(4192,4261,weight=1)
G10.add_edge(4193,4194,weight=1)
G10.add_edge(4193,4226,weight=1)
G10.add_edge(4193,4314,weight=1)
G10.add_edge(4194,4226,weight=1)
G10.add_edge(4195,4196,weight=1)
G10.add_edge(4196,4197,weight=1)
G10.add_edge(4196,4198,weight=1)
G10.add_edge(4196,4199,weight=1)
G10.add_edge(4198,4323,weight=1)
G10.add_edge(4200,4201,weight=1)
G10.add_edge(4200,4202,weight=1)
G10.add_edge(4201,4228,weight=1)
G10.add_edge(4201,4229,weight=1)
G10.add_edge(4201,4230,weight=1)
G10.add_edge(4202,4257,weight=1)
G10.add_edge(4203,4204,weight=1)
G10.add_edge(4203,4205,weight=1)
G10.add_edge(4204,4228,weight=1)
G10.add_edge(4204,4231,weight=1)
G10.add_edge(4204,4258,weight=1)
G10.add_edge(4204,4298,weight=1)
G10.add_edge(4205,4389,weight=1)
G10.add_edge(4205,4390,weight=1)
G10.add_edge(4205,4391,weight=1)
G10.add_edge(4205,4392,weight=1)
G10.add_edge(4207,4208,weight=1)
G10.add_edge(4208,4216,weight=1)
G10.add_edge(4208,4234,weight=1)
G10.add_edge(4208,4235,weight=1)
G10.add_edge(4208,4236,weight=1)
G10.add_edge(4209,4210,weight=1)
G10.add_edge(4209,4211,weight=1)
G10.add_edge(4209,4212,weight=1)
G10.add_edge(4209,4213,weight=1)
G10.add_edge(4209,4214,weight=1)
G10.add_edge(4210,4212,weight=1)
G10.add_edge(4212,4243,weight=1)
G10.add_edge(4214,4384,weight=1)
G10.add_edge(4214,4385,weight=1)
G10.add_edge(4215,4216,weight=1)
G10.add_edge(4215,4217,weight=1)
G10.add_edge(4215,4218,weight=1)
G10.add_edge(4215,4219,weight=1)
G10.add_edge(4216,4228,weight=1)
G10.add_edge(4216,4291,weight=1)
G10.add_edge(4216,4292,weight=1)
G10.add_edge(4220,4239,weight=1)
G10.add_edge(4220,4240,weight=1)
G10.add_edge(4220,4241,weight=1)
G10.add_edge(4221,4222,weight=1)
G10.add_edge(4221,4223,weight=1)
G10.add_edge(4221,4224,weight=1)
G10.add_edge(4221,4225,weight=1)
G10.add_edge(4222,4226,weight=1)
G10.add_edge(4222,4227,weight=1)
G10.add_edge(4226,4319,weight=1)
G10.add_edge(4228,4285,weight=1)
G10.add_edge(4228,4295,weight=1)
G10.add_edge(4228,4296,weight=1)
G10.add_edge(4228,4297,weight=1)
G10.add_edge(4229,4296,weight=1)
G10.add_edge(4229,4402,weight=1)
G10.add_edge(4231,4232,weight=1)
G10.add_edge(4231,4233,weight=1)
G10.add_edge(4234,4237,weight=1)
G10.add_edge(4235,4238,weight=1)
G10.add_edge(4236,4290,weight=1)
G10.add_edge(4237,4342,weight=1)
G10.add_edge(4239,4255,weight=1)
G10.add_edge(4239,4277,weight=1)
G10.add_edge(4239,4280,weight=1)
G10.add_edge(4239,4284,weight=1)
G10.add_edge(4240,4255,weight=1)
G10.add_edge(4240,4275,weight=1)
G10.add_edge(4240,4302,weight=1)
G10.add_edge(4240,4306,weight=1)
G10.add_edge(4241,4383,weight=1)
G10.add_edge(4242,4243,weight=1)
G10.add_edge(4242,4265,weight=1)
G10.add_edge(4243,4265,weight=1)
G10.add_edge(4243,4376,weight=1)
G10.add_edge(4245,4319,weight=1)
G10.add_edge(4246,4247,weight=1)
G10.add_edge(4246,4248,weight=1)
G10.add_edge(4247,4294,weight=1)
G10.add_edge(4250,4251,weight=1)
G10.add_edge(4250,4252,weight=1)
G10.add_edge(4250,4253,weight=1)
G10.add_edge(4251,4254,weight=1)
G10.add_edge(4251,4255,weight=1)
G10.add_edge(4251,4256,weight=1)
G10.add_edge(4254,4262,weight=1)
G10.add_edge(4254,4263,weight=1)
G10.add_edge(4255,4256,weight=1)
G10.add_edge(4255,4299,weight=1)
G10.add_edge(4255,4301,weight=1)
G10.add_edge(4257,4258,weight=1)
G10.add_edge(4258,4317,weight=1)
G10.add_edge(4259,4287,weight=1)
G10.add_edge(4259,4288,weight=1)
G10.add_edge(4259,4289,weight=1)
G10.add_edge(4260,4261,weight=1)
G10.add_edge(4261,4356,weight=1)
G10.add_edge(4264,4265,weight=1)
G10.add_edge(4265,4266,weight=1)
G10.add_edge(4265,4267,weight=1)
G10.add_edge(4265,4269,weight=1)
G10.add_edge(4267,4268,weight=1)
G10.add_edge(4269,4343,weight=1)
G10.add_edge(4272,4273,weight=1)
G10.add_edge(4273,4274,weight=1)
G10.add_edge(4274,4324,weight=1)
G10.add_edge(4274,4332,weight=1)
G10.add_edge(4274,4350,weight=1)
G10.add_edge(4275,4276,weight=1)
G10.add_edge(4277,4278,weight=1)
G10.add_edge(4277,4279,weight=1)
G10.add_edge(4279,4311,weight=1)
G10.add_edge(4279,4312,weight=1)
G10.add_edge(4279,4313,weight=1)
G10.add_edge(4280,4281,weight=1)
G10.add_edge(4280,4282,weight=1)
G10.add_edge(4280,4283,weight=1)
G10.add_edge(4281,4307,weight=1)
G10.add_edge(4281,4308,weight=1)
G10.add_edge(4281,4309,weight=1)
G10.add_edge(4281,4310,weight=1)
G10.add_edge(4284,4377,weight=1)
G10.add_edge(4284,4378,weight=1)
G10.add_edge(4284,4379,weight=1)
G10.add_edge(4284,4380,weight=1)
G10.add_edge(4284,4381,weight=1)
G10.add_edge(4284,4382,weight=1)
G10.add_edge(4285,4286,weight=1)
G10.add_edge(4286,4328,weight=1)
G10.add_edge(4286,4329,weight=1)
G10.add_edge(4291,4292,weight=1)
G10.add_edge(4291,4327,weight=1)
G10.add_edge(4292,4362,weight=1)
G10.add_edge(4292,4363,weight=1)
G10.add_edge(4295,4365,weight=1)
G10.add_edge(4295,4366,weight=1)
G10.add_edge(4295,4367,weight=1)
G10.add_edge(4295,4368,weight=1)
G10.add_edge(4295,4369,weight=1)
G10.add_edge(4297,4317,weight=1)
G10.add_edge(4298,4320,weight=1)
G10.add_edge(4298,4321,weight=1)
G10.add_edge(4299,4300,weight=1)
G10.add_edge(4301,4346,weight=1)
G10.add_edge(4301,4347,weight=1)
G10.add_edge(4301,4348,weight=1)
G10.add_edge(4301,4349,weight=1)
G10.add_edge(4302,4303,weight=1)
G10.add_edge(4302,4304,weight=1)
G10.add_edge(4302,4305,weight=1)
G10.add_edge(4315,4316,weight=1)
G10.add_edge(4318,4319,weight=1)
G10.add_edge(4322,4335,weight=1)
G10.add_edge(4322,4336,weight=1)
G10.add_edge(4322,4337,weight=1)
G10.add_edge(4322,4338,weight=1)
G10.add_edge(4322,4339,weight=1)
G10.add_edge(4323,4374,weight=1)
G10.add_edge(4324,4325,weight=1)
G10.add_edge(4324,4326,weight=1)
G10.add_edge(4332,4333,weight=1)
G10.add_edge(4332,4334,weight=1)
G10.add_edge(4333,4340,weight=1)
G10.add_edge(4333,4341,weight=1)
G10.add_edge(4335,4370,weight=1)
G10.add_edge(4335,4371,weight=1)
G10.add_edge(4335,4372,weight=1)
G10.add_edge(4335,4373,weight=1)
G10.add_edge(4344,4357,weight=1)
G10.add_edge(4344,4358,weight=1)
G10.add_edge(4344,4359,weight=1)
G10.add_edge(4344,4360,weight=1)
G10.add_edge(4344,4361,weight=1)
G10.add_edge(4345,4393,weight=1)
G10.add_edge(4345,4394,weight=1)
G10.add_edge(4350,4351,weight=1)
G10.add_edge(4350,4352,weight=1)
G10.add_edge(4350,4353,weight=1)
G10.add_edge(4374,4387,weight=1)
G10.add_edge(4374,4388,weight=1)
G10.add_edge(4375,4400,weight=1)
G10.add_edge(4375,4401,weight=1)
G10.add_edge(4392,4395,weight=1)
G10.add_edge(4392,4396,weight=1)
G10.add_edge(4392,4397,weight=1)
G10.add_edge(4403,4404,weight=1)
G10.add_edge(4403,4405,weight=1)
G10.add_edge(4403,4406,weight=1)
G10.add_edge(4403,4407,weight=1)
G10.add_edge(4403,4408,weight=1)
G10.add_edge(4404,4405,weight=1)
G10.add_edge(4404,4419,weight=1)
G10.add_edge(4404,4436,weight=1)
G10.add_edge(4404,4441,weight=1)
G10.add_edge(4404,4445,weight=1)
G10.add_edge(4404,4446,weight=1)
G10.add_edge(4404,4447,weight=1)
G10.add_edge(4404,4448,weight=1)
G10.add_edge(4405,4422,weight=1)
G10.add_edge(4405,4446,weight=1)
G10.add_edge(4407,4408,weight=1)
G10.add_edge(4407,4415,weight=1)
G10.add_edge(4407,4443,weight=1)
G10.add_edge(4407,4444,weight=1)
G10.add_edge(4407,4452,weight=1)
G10.add_edge(4407,4453,weight=1)
G10.add_edge(4407,4480,weight=1)
G10.add_edge(4407,4481,weight=1)
G10.add_edge(4407,4483,weight=1)
G10.add_edge(4408,4447,weight=1)
G10.add_edge(4408,4470,weight=1)
G10.add_edge(4409,4410,weight=1)
G10.add_edge(4409,4411,weight=1)
G10.add_edge(4410,4412,weight=1)
G10.add_edge(4410,4416,weight=1)
G10.add_edge(4411,4413,weight=1)
G10.add_edge(4411,4416,weight=1)
G10.add_edge(4411,4419,weight=1)
G10.add_edge(4411,4422,weight=1)
G10.add_edge(4411,4431,weight=1)
G10.add_edge(4411,4436,weight=1)
G10.add_edge(4411,4458,weight=1)
G10.add_edge(4411,4459,weight=1)
G10.add_edge(4411,4460,weight=1)
G10.add_edge(4412,4413,weight=1)
G10.add_edge(4413,4431,weight=1)
G10.add_edge(4413,4467,weight=1)
G10.add_edge(4414,4415,weight=1)
G10.add_edge(4415,4427,weight=1)
G10.add_edge(4415,4428,weight=1)
G10.add_edge(4415,4451,weight=1)
G10.add_edge(4415,4452,weight=1)
G10.add_edge(4415,4453,weight=1)
G10.add_edge(4415,4454,weight=1)
G10.add_edge(4416,4431,weight=1)
G10.add_edge(4417,4418,weight=1)
G10.add_edge(4417,4419,weight=1)
G10.add_edge(4417,4420,weight=1)
G10.add_edge(4417,4421,weight=1)
G10.add_edge(4417,4422,weight=1)
G10.add_edge(4417,4423,weight=1)
G10.add_edge(4417,4424,weight=1)
G10.add_edge(4417,4425,weight=1)
G10.add_edge(4417,4426,weight=1)
G10.add_edge(4417,4427,weight=1)
G10.add_edge(4417,4428,weight=1)
G10.add_edge(4418,4421,weight=1)
G10.add_edge(4418,4422,weight=1)
G10.add_edge(4418,4425,weight=1)
G10.add_edge(4418,4427,weight=1)
G10.add_edge(4418,4435,weight=1)
G10.add_edge(4419,4421,weight=1)
G10.add_edge(4419,4422,weight=1)
G10.add_edge(4419,4434,weight=1)
G10.add_edge(4419,4436,weight=1)
G10.add_edge(4419,4437,weight=1)
G10.add_edge(4419,4438,weight=1)
G10.add_edge(4420,4426,weight=1)
G10.add_edge(4420,4429,weight=1)
G10.add_edge(4420,4431,weight=1)
G10.add_edge(4420,4434,weight=1)
G10.add_edge(4420,4442,weight=1)
G10.add_edge(4421,4422,weight=1)
G10.add_edge(4421,4425,weight=1)
G10.add_edge(4421,4427,weight=1)
G10.add_edge(4421,4436,weight=1)
G10.add_edge(4422,4425,weight=1)
G10.add_edge(4422,4427,weight=1)
G10.add_edge(4422,4436,weight=1)
G10.add_edge(4422,4452,weight=1)
G10.add_edge(4422,4454,weight=1)
G10.add_edge(4423,4428,weight=1)
G10.add_edge(4423,4451,weight=1)
G10.add_edge(4423,4453,weight=1)
G10.add_edge(4424,4429,weight=1)
G10.add_edge(4424,4430,weight=1)
G10.add_edge(4424,4433,weight=1)
G10.add_edge(4424,4434,weight=1)
G10.add_edge(4424,4468,weight=1)
G10.add_edge(4424,4476,weight=1)
G10.add_edge(4424,4488,weight=1)
G10.add_edge(4424,4489,weight=1)
G10.add_edge(4425,4427,weight=1)
G10.add_edge(4425,4452,weight=1)
G10.add_edge(4427,4439,weight=1)
G10.add_edge(4427,4440,weight=1)
G10.add_edge(4427,4453,weight=1)
G10.add_edge(4427,4462,weight=1)
G10.add_edge(4428,4451,weight=1)
G10.add_edge(4428,4452,weight=1)
G10.add_edge(4428,4453,weight=1)
G10.add_edge(4428,4454,weight=1)
G10.add_edge(4429,4430,weight=1)
G10.add_edge(4429,4431,weight=1)
G10.add_edge(4429,4432,weight=1)
G10.add_edge(4429,4433,weight=1)
G10.add_edge(4429,4434,weight=1)
G10.add_edge(4430,4471,weight=1)
G10.add_edge(4431,4459,weight=1)
G10.add_edge(4431,4472,weight=1)
G10.add_edge(4432,4433,weight=1)
G10.add_edge(4432,4464,weight=1)
G10.add_edge(4432,4465,weight=1)
G10.add_edge(4432,4473,weight=1)
G10.add_edge(4432,4474,weight=1)
G10.add_edge(4432,4475,weight=1)
G10.add_edge(4432,4476,weight=1)
G10.add_edge(4432,4477,weight=1)
G10.add_edge(4432,4478,weight=1)
G10.add_edge(4433,4434,weight=1)
G10.add_edge(4433,4471,weight=1)
G10.add_edge(4433,4474,weight=1)
G10.add_edge(4434,4436,weight=1)
G10.add_edge(4434,4437,weight=1)
G10.add_edge(4434,4438,weight=1)
G10.add_edge(4434,4468,weight=1)
G10.add_edge(4434,4476,weight=1)
G10.add_edge(4434,4490,weight=1)
G10.add_edge(4436,4437,weight=1)
G10.add_edge(4436,4438,weight=1)
G10.add_edge(4436,4445,weight=1)
G10.add_edge(4436,4446,weight=1)
G10.add_edge(4436,4449,weight=1)
G10.add_edge(4436,4455,weight=1)
G10.add_edge(4436,4456,weight=1)
G10.add_edge(4436,4457,weight=1)
G10.add_edge(4437,4438,weight=1)
G10.add_edge(4437,4449,weight=1)
G10.add_edge(4437,4455,weight=1)
G10.add_edge(4437,4457,weight=1)
G10.add_edge(4437,4460,weight=1)
G10.add_edge(4437,4476,weight=1)
G10.add_edge(4437,4479,weight=1)
G10.add_edge(4438,4445,weight=1)
G10.add_edge(4438,4449,weight=1)
G10.add_edge(4438,4468,weight=1)
G10.add_edge(4438,4469,weight=1)
G10.add_edge(4438,4474,weight=1)
G10.add_edge(4438,4476,weight=1)
G10.add_edge(4438,4479,weight=1)
G10.add_edge(4439,4440,weight=1)
G10.add_edge(4440,4453,weight=1)
G10.add_edge(4443,4444,weight=1)
G10.add_edge(4443,4452,weight=1)
G10.add_edge(4443,4453,weight=1)
G10.add_edge(4443,4480,weight=1)
G10.add_edge(4443,4481,weight=1)
G10.add_edge(4443,4483,weight=1)
G10.add_edge(4444,4452,weight=1)
G10.add_edge(4444,4453,weight=1)
G10.add_edge(4444,4483,weight=1)
G10.add_edge(4445,4446,weight=1)
G10.add_edge(4445,4450,weight=1)
G10.add_edge(4445,4476,weight=1)
G10.add_edge(4446,4447,weight=1)
G10.add_edge(4446,4448,weight=1)
G10.add_edge(4446,4483,weight=1)
G10.add_edge(4447,4470,weight=1)
G10.add_edge(4447,4483,weight=1)
G10.add_edge(4447,4492,weight=1)
G10.add_edge(4449,4450,weight=1)
G10.add_edge(4450,4487,weight=1)
G10.add_edge(4451,4452,weight=1)
G10.add_edge(4451,4453,weight=1)
G10.add_edge(4451,4454,weight=1)
G10.add_edge(4452,4453,weight=1)
G10.add_edge(4452,4454,weight=1)
G10.add_edge(4452,4466,weight=1)
G10.add_edge(4453,4454,weight=1)
G10.add_edge(4456,4458,weight=1)
G10.add_edge(4459,4461,weight=1)
G10.add_edge(4463,4464,weight=1)
G10.add_edge(4464,4465,weight=1)
G10.add_edge(4465,4473,weight=1)
G10.add_edge(4465,4475,weight=1)
G10.add_edge(4465,4477,weight=1)
G10.add_edge(4466,4480,weight=1)
G10.add_edge(4466,4481,weight=1)
G10.add_edge(4467,4468,weight=1)
G10.add_edge(4467,4469,weight=1)
G10.add_edge(4468,4491,weight=1)
G10.add_edge(4473,4474,weight=1)
G10.add_edge(4474,4475,weight=1)
G10.add_edge(4474,4476,weight=1)
G10.add_edge(4475,4476,weight=1)
G10.add_edge(4475,4477,weight=1)
G10.add_edge(4475,4484,weight=1)
G10.add_edge(4477,4486,weight=1)
G10.add_edge(4482,4483,weight=1)
G10.add_edge(4484,4487,weight=1)
G10.add_edge(4493,4494,weight=1)
G10.add_edge(4494,4495,weight=1)
G10.add_edge(4494,4496,weight=1)
G10.add_edge(4495,4496,weight=1)
G10.add_edge(4495,4506,weight=1)
G10.add_edge(4496,4503,weight=1)
G10.add_edge(4496,4507,weight=1)
G10.add_edge(4497,4498,weight=1)
G10.add_edge(4497,4499,weight=1)
G10.add_edge(4501,4502,weight=1)
G10.add_edge(4501,4503,weight=1)
G10.add_edge(4506,4507,weight=1)
G10.add_edge(4506,4508,weight=1)
G10.add_edge(4507,4509,weight=1)
G10.add_edge(4513,4514,weight=1)
G10.add_edge(4514,4515,weight=1)
G10.add_edge(4516,4517,weight=1)
G10.add_edge(4517,4518,weight=1)
G10.add_edge(4524,4525,weight=1)
G10.add_edge(4524,4526,weight=1)
G10.add_edge(4525,4527,weight=1)
G10.add_edge(4525,4556,weight=1)
G10.add_edge(4525,4565,weight=1)
G10.add_edge(4525,4566,weight=1)
G10.add_edge(4525,4567,weight=1)
G10.add_edge(4526,4594,weight=1)
G10.add_edge(4526,4701,weight=1)
G10.add_edge(4527,4528,weight=1)
G10.add_edge(4528,4622,weight=1)
G10.add_edge(4529,4530,weight=1)
G10.add_edge(4529,4531,weight=1)
G10.add_edge(4529,4532,weight=1)
G10.add_edge(4529,4533,weight=1)
G10.add_edge(4530,4658,weight=1)
G10.add_edge(4530,4659,weight=1)
G10.add_edge(4531,4664,weight=1)
G10.add_edge(4531,4715,weight=1)
G10.add_edge(4531,4716,weight=1)
G10.add_edge(4531,4732,weight=1)
G10.add_edge(4531,4747,weight=1)
G10.add_edge(4532,4588,weight=1)
G10.add_edge(4532,4762,weight=1)
G10.add_edge(4534,4535,weight=1)
G10.add_edge(4534,4536,weight=1)
G10.add_edge(4534,4537,weight=1)
G10.add_edge(4535,4768,weight=1)
G10.add_edge(4536,4661,weight=1)
G10.add_edge(4536,4766,weight=1)
G10.add_edge(4538,4539,weight=1)
G10.add_edge(4538,4540,weight=1)
G10.add_edge(4539,4550,weight=1)
G10.add_edge(4540,4558,weight=1)
G10.add_edge(4540,4740,weight=1)
G10.add_edge(4541,4542,weight=1)
G10.add_edge(4542,4603,weight=1)
G10.add_edge(4543,4544,weight=1)
G10.add_edge(4543,4545,weight=1)
G10.add_edge(4543,4546,weight=1)
G10.add_edge(4543,4547,weight=1)
G10.add_edge(4543,4548,weight=1)
G10.add_edge(4544,4549,weight=1)
G10.add_edge(4545,4616,weight=1)
G10.add_edge(4545,4617,weight=1)
G10.add_edge(4545,4618,weight=1)
G10.add_edge(4546,4748,weight=1)
G10.add_edge(4546,4749,weight=1)
G10.add_edge(4547,4607,weight=1)
G10.add_edge(4548,4688,weight=1)
G10.add_edge(4549,4676,weight=1)
G10.add_edge(4549,4883,weight=1)
G10.add_edge(4549,4907,weight=1)
G10.add_edge(4549,4908,weight=1)
G10.add_edge(4549,4909,weight=1)
G10.add_edge(4551,4553,weight=1)
G10.add_edge(4551,4554,weight=1)
G10.add_edge(4552,4586,weight=1)
G10.add_edge(4552,4636,weight=1)
G10.add_edge(4553,4729,weight=1)
G10.add_edge(4553,4730,weight=1)
G10.add_edge(4553,4731,weight=1)
G10.add_edge(4554,4866,weight=1)
G10.add_edge(4554,4867,weight=1)
G10.add_edge(4556,4557,weight=1)
G10.add_edge(4558,4559,weight=1)
G10.add_edge(4558,4560,weight=1)
G10.add_edge(4559,4561,weight=1)
G10.add_edge(4561,4580,weight=1)
G10.add_edge(4562,4563,weight=1)
G10.add_edge(4562,4564,weight=1)
G10.add_edge(4563,4572,weight=1)
G10.add_edge(4563,4686,weight=1)
G10.add_edge(4565,4596,weight=1)
G10.add_edge(4566,4662,weight=1)
G10.add_edge(4567,4653,weight=1)
G10.add_edge(4568,4569,weight=1)
G10.add_edge(4568,4570,weight=1)
G10.add_edge(4568,4571,weight=1)
G10.add_edge(4569,4572,weight=1)
G10.add_edge(4569,4573,weight=1)
G10.add_edge(4570,4598,weight=1)
G10.add_edge(4571,4696,weight=1)
G10.add_edge(4572,4738,weight=1)
G10.add_edge(4572,4739,weight=1)
G10.add_edge(4573,4599,weight=1)
G10.add_edge(4573,4792,weight=1)
G10.add_edge(4574,4575,weight=1)
G10.add_edge(4574,4576,weight=1)
G10.add_edge(4575,4596,weight=1)
G10.add_edge(4576,4710,weight=1)
G10.add_edge(4576,4712,weight=1)
G10.add_edge(4576,4713,weight=1)
G10.add_edge(4577,4578,weight=1)
G10.add_edge(4578,4618,weight=1)
G10.add_edge(4579,4580,weight=1)
G10.add_edge(4579,4581,weight=1)
G10.add_edge(4580,4601,weight=1)
G10.add_edge(4580,4604,weight=1)
G10.add_edge(4580,4605,weight=1)
G10.add_edge(4581,4693,weight=1)
G10.add_edge(4581,4705,weight=1)
G10.add_edge(4582,4586,weight=1)
G10.add_edge(4584,4697,weight=1)
G10.add_edge(4585,4630,weight=1)
G10.add_edge(4587,4588,weight=1)
G10.add_edge(4587,4589,weight=1)
G10.add_edge(4587,4590,weight=1)
G10.add_edge(4589,4757,weight=1)
G10.add_edge(4589,4758,weight=1)
G10.add_edge(4591,4592,weight=1)
G10.add_edge(4591,4593,weight=1)
G10.add_edge(4592,4666,weight=1)
G10.add_edge(4593,4620,weight=1)
G10.add_edge(4594,4595,weight=1)
G10.add_edge(4595,4628,weight=1)
G10.add_edge(4595,4654,weight=1)
G10.add_edge(4596,4846,weight=1)
G10.add_edge(4597,4598,weight=1)
G10.add_edge(4597,4599,weight=1)
G10.add_edge(4597,4600,weight=1)
G10.add_edge(4598,4599,weight=1)
G10.add_edge(4600,4857,weight=1)
G10.add_edge(4601,4602,weight=1)
G10.add_edge(4602,4603,weight=1)
G10.add_edge(4603,4772,weight=1)
G10.add_edge(4606,4607,weight=1)
G10.add_edge(4607,4608,weight=1)
G10.add_edge(4608,4707,weight=1)
G10.add_edge(4608,4711,weight=1)
G10.add_edge(4609,4610,weight=1)
G10.add_edge(4609,4611,weight=1)
G10.add_edge(4610,4634,weight=1)
G10.add_edge(4610,4645,weight=1)
G10.add_edge(4610,4650,weight=1)
G10.add_edge(4611,4631,weight=1)
G10.add_edge(4611,4698,weight=1)
G10.add_edge(4612,4613,weight=1)
G10.add_edge(4612,4614,weight=1)
G10.add_edge(4613,4615,weight=1)
G10.add_edge(4614,4696,weight=1)
G10.add_edge(4615,4691,weight=1)
G10.add_edge(4616,4663,weight=1)
G10.add_edge(4616,4688,weight=1)
G10.add_edge(4616,4689,weight=1)
G10.add_edge(4617,4732,weight=1)
G10.add_edge(4619,4761,weight=1)
G10.add_edge(4619,4776,weight=1)
G10.add_edge(4620,4621,weight=1)
G10.add_edge(4621,4707,weight=1)
G10.add_edge(4621,4752,weight=1)
G10.add_edge(4621,4756,weight=1)
G10.add_edge(4622,4623,weight=1)
G10.add_edge(4622,4624,weight=1)
G10.add_edge(4622,4625,weight=1)
G10.add_edge(4623,4637,weight=1)
G10.add_edge(4623,4638,weight=1)
G10.add_edge(4623,4639,weight=1)
G10.add_edge(4624,4750,weight=1)
G10.add_edge(4624,4751,weight=1)
G10.add_edge(4626,4627,weight=1)
G10.add_edge(4626,4628,weight=1)
G10.add_edge(4626,4629,weight=1)
G10.add_edge(4629,4655,weight=1)
G10.add_edge(4630,4631,weight=1)
G10.add_edge(4631,4737,weight=1)
G10.add_edge(4632,4633,weight=1)
G10.add_edge(4632,4634,weight=1)
G10.add_edge(4632,4635,weight=1)
G10.add_edge(4633,4636,weight=1)
G10.add_edge(4634,4702,weight=1)
G10.add_edge(4634,4703,weight=1)
G10.add_edge(4636,4733,weight=1)
G10.add_edge(4637,4768,weight=1)
G10.add_edge(4637,4785,weight=1)
G10.add_edge(4639,4778,weight=1)
G10.add_edge(4640,4641,weight=1)
G10.add_edge(4640,4642,weight=1)
G10.add_edge(4641,4643,weight=1)
G10.add_edge(4641,4644,weight=1)
G10.add_edge(4641,4645,weight=1)
G10.add_edge(4641,4646,weight=1)
G10.add_edge(4641,4647,weight=1)
G10.add_edge(4641,4648,weight=1)
G10.add_edge(4641,4649,weight=1)
G10.add_edge(4642,4677,weight=1)
G10.add_edge(4644,4704,weight=1)
G10.add_edge(4644,4744,weight=1)
G10.add_edge(4645,4760,weight=1)
G10.add_edge(4646,4789,weight=1)
G10.add_edge(4646,4791,weight=1)
G10.add_edge(4650,4845,weight=1)
G10.add_edge(4651,4652,weight=1)
G10.add_edge(4651,4653,weight=1)
G10.add_edge(4652,4661,weight=1)
G10.add_edge(4653,4756,weight=1)
G10.add_edge(4653,4765,weight=1)
G10.add_edge(4654,4655,weight=1)
G10.add_edge(4655,4690,weight=1)
G10.add_edge(4655,4704,weight=1)
G10.add_edge(4656,4657,weight=1)
G10.add_edge(4657,4674,weight=1)
G10.add_edge(4657,4722,weight=1)
G10.add_edge(4657,4754,weight=1)
G10.add_edge(4657,4784,weight=1)
G10.add_edge(4660,4661,weight=1)
G10.add_edge(4662,4740,weight=1)
G10.add_edge(4663,4664,weight=1)
G10.add_edge(4663,4665,weight=1)
G10.add_edge(4664,4745,weight=1)
G10.add_edge(4666,4667,weight=1)
G10.add_edge(4666,4668,weight=1)
G10.add_edge(4667,4668,weight=1)
G10.add_edge(4667,4670,weight=1)
G10.add_edge(4667,4723,weight=1)
G10.add_edge(4667,4743,weight=1)
G10.add_edge(4668,4780,weight=1)
G10.add_edge(4669,4670,weight=1)
G10.add_edge(4669,4671,weight=1)
G10.add_edge(4669,4672,weight=1)
G10.add_edge(4669,4673,weight=1)
G10.add_edge(4670,4687,weight=1)
G10.add_edge(4672,4779,weight=1)
G10.add_edge(4673,4734,weight=1)
G10.add_edge(4673,4743,weight=1)
G10.add_edge(4674,4675,weight=1)
G10.add_edge(4674,4676,weight=1)
G10.add_edge(4675,4721,weight=1)
G10.add_edge(4675,4722,weight=1)
G10.add_edge(4676,4861,weight=1)
G10.add_edge(4676,4863,weight=1)
G10.add_edge(4677,4678,weight=1)
G10.add_edge(4677,4679,weight=1)
G10.add_edge(4677,4680,weight=1)
G10.add_edge(4677,4681,weight=1)
G10.add_edge(4678,4781,weight=1)
G10.add_edge(4678,4784,weight=1)
G10.add_edge(4682,4683,weight=1)
G10.add_edge(4682,4684,weight=1)
G10.add_edge(4682,4685,weight=1)
G10.add_edge(4683,4721,weight=1)
G10.add_edge(4687,4859,weight=1)
G10.add_edge(4688,4689,weight=1)
G10.add_edge(4691,4708,weight=1)
G10.add_edge(4691,4775,weight=1)
G10.add_edge(4692,4693,weight=1)
G10.add_edge(4692,4694,weight=1)
G10.add_edge(4693,4695,weight=1)
G10.add_edge(4697,4698,weight=1)
G10.add_edge(4699,4763,weight=1)
G10.add_edge(4700,4799,weight=1)
G10.add_edge(4700,4813,weight=1)
G10.add_edge(4701,4713,weight=1)
G10.add_edge(4705,4706,weight=1)
G10.add_edge(4706,4768,weight=1)
G10.add_edge(4706,4778,weight=1)
G10.add_edge(4709,4710,weight=1)
G10.add_edge(4710,4735,weight=1)
G10.add_edge(4715,4716,weight=1)
G10.add_edge(4715,4717,weight=1)
G10.add_edge(4715,4718,weight=1)
G10.add_edge(4715,4719,weight=1)
G10.add_edge(4716,4720,weight=1)
G10.add_edge(4720,4747,weight=1)
G10.add_edge(4720,4769,weight=1)
G10.add_edge(4722,4777,weight=1)
G10.add_edge(4723,4724,weight=1)
G10.add_edge(4724,4749,weight=1)
G10.add_edge(4724,4767,weight=1)
G10.add_edge(4726,4733,weight=1)
G10.add_edge(4726,4761,weight=1)
G10.add_edge(4727,4843,weight=1)
G10.add_edge(4727,4844,weight=1)
G10.add_edge(4732,4746,weight=1)
G10.add_edge(4732,4860,weight=1)
G10.add_edge(4735,4736,weight=1)
G10.add_edge(4738,4786,weight=1)
G10.add_edge(4739,4854,weight=1)
G10.add_edge(4741,4742,weight=1)
G10.add_edge(4741,4743,weight=1)
G10.add_edge(4747,4770,weight=1)
G10.add_edge(4749,4759,weight=1)
G10.add_edge(4750,4752,weight=1)
G10.add_edge(4750,4753,weight=1)
G10.add_edge(4751,4766,weight=1)
G10.add_edge(4753,4768,weight=1)
G10.add_edge(4754,4755,weight=1)
G10.add_edge(4757,4783,weight=1)
G10.add_edge(4758,4855,weight=1)
G10.add_edge(4762,4851,weight=1)
G10.add_edge(4763,4798,weight=1)
G10.add_edge(4763,4803,weight=1)
G10.add_edge(4763,4834,weight=1)
G10.add_edge(4763,4835,weight=1)
G10.add_edge(4764,4765,weight=1)
G10.add_edge(4770,4771,weight=1)
G10.add_edge(4773,4820,weight=1)
G10.add_edge(4773,4822,weight=1)
G10.add_edge(4773,4838,weight=1)
G10.add_edge(4774,4815,weight=1)
G10.add_edge(4777,4848,weight=1)
G10.add_edge(4779,4858,weight=1)
G10.add_edge(4781,4782,weight=1)
G10.add_edge(4789,4790,weight=1)
G10.add_edge(4792,4853,weight=1)
G10.add_edge(4792,4854,weight=1)
G10.add_edge(4793,4794,weight=1)
G10.add_edge(4793,4795,weight=1)
G10.add_edge(4794,4810,weight=1)
G10.add_edge(4795,4803,weight=1)
G10.add_edge(4796,4797,weight=1)
G10.add_edge(4796,4798,weight=1)
G10.add_edge(4797,4808,weight=1)
G10.add_edge(4799,4800,weight=1)
G10.add_edge(4800,4801,weight=1)
G10.add_edge(4801,4802,weight=1)
G10.add_edge(4802,4813,weight=1)
G10.add_edge(4803,4804,weight=1)
G10.add_edge(4803,4805,weight=1)
G10.add_edge(4803,4806,weight=1)
G10.add_edge(4804,4807,weight=1)
G10.add_edge(4804,4811,weight=1)
G10.add_edge(4804,4816,weight=1)
G10.add_edge(4804,4831,weight=1)
G10.add_edge(4805,4807,weight=1)
G10.add_edge(4808,4809,weight=1)
G10.add_edge(4809,4810,weight=1)
G10.add_edge(4811,4812,weight=1)
G10.add_edge(4812,4814,weight=1)
G10.add_edge(4812,4821,weight=1)
G10.add_edge(4812,4824,weight=1)
G10.add_edge(4812,4829,weight=1)
G10.add_edge(4812,4830,weight=1)
G10.add_edge(4814,4815,weight=1)
G10.add_edge(4815,4822,weight=1)
G10.add_edge(4815,4832,weight=1)
G10.add_edge(4815,4837,weight=1)
G10.add_edge(4816,4817,weight=1)
G10.add_edge(4817,4821,weight=1)
G10.add_edge(4818,4819,weight=1)
G10.add_edge(4818,4820,weight=1)
G10.add_edge(4819,4833,weight=1)
G10.add_edge(4822,4823,weight=1)
G10.add_edge(4823,4841,weight=1)
G10.add_edge(4823,4842,weight=1)
G10.add_edge(4824,4825,weight=1)
G10.add_edge(4826,4827,weight=1)
G10.add_edge(4826,4828,weight=1)
G10.add_edge(4828,4850,weight=1)
G10.add_edge(4831,4832,weight=1)
G10.add_edge(4833,4836,weight=1)
G10.add_edge(4834,4839,weight=1)
G10.add_edge(4836,4837,weight=1)
G10.add_edge(4838,4840,weight=1)
G10.add_edge(4839,4840,weight=1)
G10.add_edge(4846,4847,weight=1)
G10.add_edge(4848,4849,weight=1)
G10.add_edge(4852,4853,weight=1)
G10.add_edge(4854,4856,weight=1)
G10.add_edge(4861,4862,weight=1)
G10.add_edge(4862,4864,weight=1)
G10.add_edge(4862,4876,weight=1)
G10.add_edge(4863,4864,weight=1)
G10.add_edge(4865,4870,weight=1)
G10.add_edge(4865,4909,weight=1)
G10.add_edge(4866,4883,weight=1)
G10.add_edge(4866,4886,weight=1)
G10.add_edge(4866,4887,weight=1)
G10.add_edge(4867,4887,weight=1)
G10.add_edge(4868,4869,weight=1)
G10.add_edge(4868,4870,weight=1)
G10.add_edge(4869,4923,weight=1)
G10.add_edge(4870,4923,weight=1)
G10.add_edge(4871,4872,weight=1)
G10.add_edge(4871,4873,weight=1)
G10.add_edge(4871,4874,weight=1)
G10.add_edge(4872,4889,weight=1)
G10.add_edge(4874,4931,weight=1)
G10.add_edge(4875,4876,weight=1)
G10.add_edge(4877,4878,weight=1)
G10.add_edge(4877,4879,weight=1)
G10.add_edge(4877,4880,weight=1)
G10.add_edge(4879,4913,weight=1)
G10.add_edge(4880,4881,weight=1)
G10.add_edge(4881,4882,weight=1)
G10.add_edge(4882,4914,weight=1)
G10.add_edge(4883,4884,weight=1)
G10.add_edge(4883,4885,weight=1)
G10.add_edge(4887,4921,weight=1)
G10.add_edge(4888,4889,weight=1)
G10.add_edge(4889,4890,weight=1)
G10.add_edge(4889,4891,weight=1)
G10.add_edge(4890,4913,weight=1)
G10.add_edge(4890,4916,weight=1)
G10.add_edge(4891,4911,weight=1)
G10.add_edge(4892,4927,weight=1)
G10.add_edge(4892,4928,weight=1)
G10.add_edge(4893,4924,weight=1)
G10.add_edge(4894,4915,weight=1)
G10.add_edge(4895,4896,weight=1)
G10.add_edge(4897,4898,weight=1)
G10.add_edge(4898,4903,weight=1)
G10.add_edge(4900,4901,weight=1)
G10.add_edge(4904,4905,weight=1)
G10.add_edge(4909,4922,weight=1)
G10.add_edge(4910,4911,weight=1)
G10.add_edge(4910,4912,weight=1)
G10.add_edge(4911,4920,weight=1)
G10.add_edge(4912,4932,weight=1)
G10.add_edge(4913,4914,weight=1)
G10.add_edge(4913,4915,weight=1)
G10.add_edge(4917,4929,weight=1)
G10.add_edge(4919,4929,weight=1)
G10.add_edge(4919,4932,weight=1)
G10.add_edge(4924,4925,weight=1)
G10.add_edge(4924,4926,weight=1)
G10.add_edge(4925,4927,weight=1)
G10.add_edge(4926,4930,weight=1)
G10.add_edge(4933,4934,weight=1)
G10.add_edge(4934,4941,weight=1)
G10.add_edge(4936,4938,weight=1)
G10.add_edge(4937,4938,weight=1)
G10.add_edge(4939,4940,weight=1)

In [ ]:
def _find_sheet_case_insensitive(xls, wanted):
    """Devuelve el nombre real de la hoja."""
    wanted_low = wanted.lower()
    for s in xls.sheet_names:
        if s.lower() == wanted_low:
            return s
    raise ValueError(f"No encuentro la hoja '{wanted}'. Hojas disponibles: {xls.sheet_names}")

def partitions_from_excel_row(row, node_ids):
    """
    Convierte una fila tipo: V1..Vn = id_comunidad  (estilo tu Excel)
    en lista de sets [{nodos_com1}, {nodos_com2}, ...]
    """
    # columnas que representan nodos (V1, V2, ...)
    node_cols = [c for c in row.index if isinstance(c, str) and c.lower().startswith("v")]
    # ordenar por id del nodo
    def node_id_from_col(c):
        m = re.match(r"(?i)v(\d+)$", c.strip())
        return int(m.group(1)) if m else None
    node_cols = [c for c in node_cols if node_id_from_col(c) is not None]
    node_cols = sorted(node_cols, key=node_id_from_col)

    # construir etiqueta por nodo
    label_by_node = {}
    for c in node_cols:
        nid = node_id_from_col(c)
        if nid in node_ids:
            lab = row[c]
            if pd.isna(lab):
                continue
            label_by_node[nid] = int(lab)

    # pasar a sets
    comms = defaultdict(set)
    for nid, lab in label_by_node.items():
        comms[lab].add(nid)
    # devolver lista de sets, orden estable por etiqueta
    return [comms[k] for k in sorted(comms.keys())]

def read_fast_greedy_partitions(excel_path, graph):
    """
    Lee todas las particiones (por alpha) desde la hoja 'Fast Greedy' del Excel.
    Devuelve: dict alpha -> partition_sets (lista de sets)
    """
    xls = pd.ExcelFile(excel_path)
    sheet = _find_sheet_case_insensitive(xls, "Fast Greedy")
    dfp = pd.read_excel(xls, sheet_name=sheet)

    if "Alpha" not in dfp.columns:
        raise ValueError(f"En {excel_path} no hay columna 'Alpha' en la hoja '{sheet}'.")

    node_ids = set(graph.nodes())
    out = {}
    for _, row in dfp.iterrows():
        alpha = float(row["Alpha"])
        part_sets = partitions_from_excel_row(row, node_ids)
        if len(part_sets) == 0:
            continue
        out[alpha] = part_sets
    return out

# ========= EJECUCIÓN MASIVA (Excel 1..10) =========
DIR = r"C:\FMF"  # <-- carpeta de los excel (colocar los resultados de FMF de las redes populares)

# Mapeo: substring del nombre de archivo -> grafo
GRAPH_BY_FILEKEY = {
    "zachary": G1,
    "dolphins": G2,
    "lesmis": G3,
    "polbooks": G4,
    "football": G5,
    "jazz": G6,
    "celegans_metabolic": G7,
    "cora": G8,
    "citeseer": G9,
    "powergrid": G10,
}

def pick_graph_for_excel(filename):
    low = filename.lower()
    for key, G in GRAPH_BY_FILEKEY.items():
        if key in low:
            return key, G
    raise ValueError(f"No sé qué grafo usar para '{filename}'. Añade una regla a GRAPH_BY_FILEKEY.")

rows = []

for i in range(1, 11):
    # coge el primer excel que empiece por "i."
    candidates = sorted(glob.glob(str(Path(DIR) / f"{i}.*.xlsx")))
    if not candidates:
        print(f"[AVISO] No encuentro Excel para i={i} (patrón {i}.*.xlsx) en {DIR}")
        continue

    excel_path = candidates[0]
    file_key, G = pick_graph_for_excel(Path(excel_path).name)

    parts_by_alpha = read_fast_greedy_partitions(excel_path, G)

    for alpha, teacher_part in sorted(parts_by_alpha.items(), key=lambda x: -x[0]):  # alpha desc
        mod_teacher = calculate_modularity_igraph(G, teacher_part)
        mod_csea, _, _, _ = run_csea_with_teacher(G, teacher_part, latent_dim=10, num_epochs=50, seed=SEED)

        rows.append({
        "excel": Path(excel_path).name,
        "file_key": file_key,
        "alpha": alpha,
        "mod_teacher_fast_greedy": mod_teacher,
        "mod_csea_entrenado_en_fast_greedy": mod_csea,
        })


df_mods = pd.DataFrame(rows).sort_values(["excel", "alpha"], ascending=[True, False])



# --- Resúmenes pedidos ---

print("=== Modularidades CSEA entrenado con Fast Greedy ===")
display(df_mods[["excel","file_key","alpha","mod_csea_entrenado_en_fast_greedy"]])


In [ ]:
# ===== Guardar resultados a Excel =====
SAL = r"C:\ResultadosFMF" 
out_path = str(Path(SAL) / "resultados_csea_fast_greedy.xlsx")

with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    df_mods.to_excel(writer, sheet_name="todos", index=False)
 

print("Guardado en:", out_path)